# Paper 3 — UTW-RG-AdaFedResidual

**Unified Temporal Windows + Relation-Guided Adaptive Federated Residual Learning**

This notebook is a new, independent Paper 3 experiment. It does not import a
Paper 2 model or reuse Paper 2 test results.

Primary protocol:

- Stations: Kadhimiya, Karama, Resafa, Tarmyya.
- Targets: Alum and Chlorine.
- Forecast horizons: H+1, H+2, H+3 = next available observations.
- Unified temporal-window candidates: **W = {1, 3, 5, 7, 10}**.
- One single window is selected using **Train + Validation only**.
- The Test partition is excluded from window selection.
- After selection, the same frozen window is used for Centralized, Local,
  all federated baselines, and UTW-RG-AdaFedResidual.
- Scaling is estimated from training data only.
- Every temporal feature is causal: current raw measurements and historical
  station observations only; no future observation is used as an input.
- The final Test set is used only after the unified window and federated method
  have been selected from Validation.

The window-selection stage uses a fixed Ridge probe and macro-averaged
station/task validation performance. It is deliberately separated from the
proposed model so that the temporal window is not selected specifically to
favor UTW-RG-AdaFedResidual.


In [ ]:
# ============================================================
# Cell 1 — Upload and Validate All Excel Files
# Paper 3 — UTW-RG-AdaFedResidual Experiment
# ============================================================

import os, re, sys, time, subprocess, importlib.util
from pathlib import Path

# Notebook-wide wall-clock timer used by final Cell 29
NOTEBOOK_RUN_START_TIME = time.time()
CELL_START = NOTEBOOK_RUN_START_TIME

if importlib.util.find_spec("openpyxl") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl", "-q"])

STATION_FILE_RULES = {
    "Kadhimiya": [["kadhimiya", "final2"], ["kadhmyya", "final2"]],
    "Karama":    [["karama", "final2"], ["krama", "final2"]],
    "Resafa":    [["resafa", "final2"]],
    "Tarmyya":   [["tarmyya", "final2"], ["tarmiya", "final2"]],
}

SUPPORT_FILE_RULES = {
    "capacity": [["capacity"]],
    "jar_test": [["jar", "test"]],
    "chlorine_target": [["chlorine", "dose", "target"]],
    "alum_polymer_doses": [["alum", "polymer", "doses"]],
    "pumps": [["small", "large", "dosing", "pumps"]],
    "standards": [["who", "epa", "drinking", "water", "standards"]],
}

SEARCH_DIRS = [Path("/content"), Path("/mnt/data"), Path.cwd()]

def normalize_filename(name):
    s = str(name).lower().replace("–", "-").replace("—", "-")
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def collect_excel_files():
    files = []
    seen = set()
    for folder in SEARCH_DIRS:
        if folder.exists():
            for p in folder.glob("*.xlsx"):
                try:
                    key = str(p.resolve())
                except Exception:
                    key = str(p)
                if key not in seen:
                    files.append(p)
                    seen.add(key)
    return files

def match_file(files, alternatives):
    for kws in alternatives:
        for p in files:
            n = normalize_filename(p.name)
            if all(k.lower() in n for k in kws):
                return p
    return None

def build_paths(files):
    station_paths = {k: match_file(files, v) for k, v in STATION_FILE_RULES.items()}
    support_paths = {k: match_file(files, v) for k, v in SUPPORT_FILE_RULES.items()}
    return station_paths, support_paths

def missing(paths):
    return [k for k, v in paths.items() if v is None]

excel_files = collect_excel_files()
STATION_PATHS, SUPPORT_PATHS = build_paths(excel_files)

if missing(STATION_PATHS) or missing(SUPPORT_PATHS):
    try:
        from google.colab import files
        print("Please upload the 10 core project Excel files. Keep the 5 prepared external-generalization workbooks in /content before Cells 22A–22E.")
        files.upload()
        excel_files = collect_excel_files()
        STATION_PATHS, SUPPORT_PATHS = build_paths(excel_files)
    except Exception as exc:
        print("Colab upload tool was not available:", exc)

if missing(STATION_PATHS) or missing(SUPPORT_PATHS):
    raise FileNotFoundError(
        f"Missing station files={missing(STATION_PATHS)}; "
        f"missing support files={missing(SUPPORT_PATHS)}"
    )

STATION_PATHS = {k: str(v) for k, v in STATION_PATHS.items()}
SUPPORT_PATHS = {k: str(v) for k, v in SUPPORT_PATHS.items()}
EXPECTED_STATIONS = ["Kadhimiya", "Karama", "Resafa", "Tarmyya"]

print("="*100)
print("ALL REQUIRED EXCEL FILES FOUND")
print("="*100)
for k, v in STATION_PATHS.items():
    print(f"Station  {k:10s}: {Path(v).name}")
for k, v in SUPPORT_PATHS.items():
    print(f"Support  {k:18s}: {Path(v).name}")
print(f"Cell 1 completed in {time.time()-CELL_START:.2f} seconds")


In [ ]:
# ============================================================
# Cell 2 — Read Station Excel Files
# ============================================================

import time
import pandas as pd

CELL_START = time.time()
STATION_RAW = {}
rows = []

for station in EXPECTED_STATIONS:
    path = STATION_PATHS[station]
    df = pd.read_excel(path)
    df.columns = (
        df.columns.astype(str).str.strip()
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )
    df["Station"] = station
    STATION_RAW[station] = df
    rows.append({"Station": station, "Rows": len(df), "Columns": df.shape[1], "File": Path(path).name})

STATION_LOAD_SUMMARY_DF = pd.DataFrame(rows)
print("="*90)
print("STATION EXCEL FILES LOADED")
print("="*90)
display(STATION_LOAD_SUMMARY_DF)
print(f"Cell 2 completed in {time.time()-CELL_START:.2f} seconds")


In [ ]:
# ============================================================
# Cell 3 — Clean and Validate Station Data Chronologically
# ============================================================

import time, numpy as np, pandas as pd
CELL_START = time.time()

def parse_date_safely(s):
    a = pd.to_datetime(s, errors="coerce")
    b = pd.to_datetime(s, errors="coerce", dayfirst=True)
    return a.fillna(b)

def clean_numeric_column(s):
    return pd.to_numeric(
        s.astype(str).str.replace(",", "", regex=False).str.strip()
        .replace(["nan", "None", "", "-", "--"], np.nan),
        errors="coerce"
    )

STATION_CLEAN = {}
summary = []
for station, raw in STATION_RAW.items():
    df = raw.copy()
    df.columns = (
        df.columns.astype(str).str.strip()
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )
    date_candidates = [c for c in df.columns if "date" in c.lower()]
    if not date_candidates:
        raise ValueError(f"No date column found in {station}")
    date_col = date_candidates[0]
    df[date_col] = parse_date_safely(df[date_col])
    if date_col != "Date":
        df = df.rename(columns={date_col: "Date"})
    raw_n = len(df)
    invalid_dates = int(df["Date"].isna().sum())
    df = df.dropna(subset=["Date"]).copy()
    before_dup = len(df)
    df = df.sort_values("Date").drop_duplicates("Date", keep="last").reset_index(drop=True)
    duplicate_count = before_dup - len(df)
    for c in df.columns:
        if c not in ["Date", "Station"]:
            df[c] = clean_numeric_column(df[c])
    df["Station"] = station
    chronological = bool(df["Date"].is_monotonic_increasing)
    STATION_CLEAN[station] = df
    summary.append({
        "Station": station, "Rows_Raw": raw_n, "Rows_Clean": len(df),
        "Removed_Invalid_Dates": invalid_dates, "Removed_Duplicate_Dates": duplicate_count,
        "Missing_Raw": int(raw.isna().sum().sum()),
        "Missing_After_Cleaning": int(df.isna().sum().sum()),
        "Start_Date": df["Date"].min(), "End_Date": df["Date"].max(),
        "Chronological": chronological
    })

STATION_CLEANING_SUMMARY_DF = pd.DataFrame(summary)
print("="*90)
print("STATION DATA CLEANING AND CHRONOLOGICAL VALIDATION")
print("="*90)
display(STATION_CLEANING_SUMMARY_DF)
print(f"Cell 3 completed in {time.time()-CELL_START:.2f} seconds")


In [ ]:
# ============================================================
# Cell 4 — Read and Validate the Supporting Excel Files
# ============================================================

import time, pandas as pd
CELL_START = time.time()
SUPPORT_RAW, SUPPORT_SHEETS = {}, {}
rows = []

for key, path in SUPPORT_PATHS.items():
    book = pd.ExcelFile(path)
    sheet = book.sheet_names[0]
    df = pd.read_excel(path, sheet_name=sheet)
    before_r, before_c = df.shape
    df = df.dropna(how="all").dropna(axis=1, how="all").copy()
    df.columns = (
        df.columns.astype(str).str.strip()
        .str.replace("\n", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )
    SUPPORT_RAW[key] = df
    SUPPORT_SHEETS[key] = book.sheet_names
    rows.append({
        "File_Key": key, "File_Name": Path(path).name, "Used_Sheet": sheet,
        "Available_Sheets": ", ".join(book.sheet_names), "Rows": len(df), "Columns": df.shape[1],
        "Removed_Empty_Rows": before_r-len(df), "Removed_Empty_Columns": before_c-df.shape[1],
        "Missing_Values": int(df.isna().sum().sum())
    })

SUPPORT_SUMMARY_DF = pd.DataFrame(rows)
print("="*90)
print("SUPPORTING DATASETS LOADED AND VALIDATED")
print("="*90)
display(SUPPORT_SUMMARY_DF)
print(f"Cell 4 completed in {time.time()-CELL_START:.2f} seconds")


In [ ]:
# ============================================================
# Cell 5 — Generate Alum and Chlorine Dose Targets
# Historical target-generation logic restored
# ============================================================

import time
import re
import numpy as np
import pandas as pd

CELL_START = time.time()

if "SUPPORT_RAW" not in globals():
    raise NameError("SUPPORT_RAW was not found. Run Cell 4 first.")

if "STATION_CLEAN" not in globals():
    raise NameError("STATION_CLEAN was not found. Run Cell 3 first.")

def to_numeric_safe(series):
    return pd.to_numeric(series, errors="coerce")

def parse_range_text(value):
    text = (
        str(value)
        .strip()
        .replace("–", "-")
        .replace("—", "-")
        .replace("≤", "<=")
        .replace("≥", ">=")
    )

    nums = re.findall(r"\d+\.?\d*", text)

    if text.startswith("<=") or text.startswith("<"):
        return -np.inf, float(nums[0])

    if text.startswith(">=") or text.startswith(">"):
        return float(nums[0]), np.inf

    if len(nums) >= 2:
        return float(nums[0]), float(nums[1])

    if len(nums) == 1:
        value = float(nums[0])
        return value, value

    return np.nan, np.nan


# ------------------------------------------------------------
# Alum target from the Jar Test table
# ------------------------------------------------------------

jar_df = SUPPORT_RAW["jar_test"].copy()
jar_df.columns = jar_df.columns.astype(str).str.strip()

jar_turb_col = [c for c in jar_df.columns if "turb" in c.lower()][0]
jar_alum_col = [
    c for c in jar_df.columns
    if "alum" in c.lower() and "dose" in c.lower()
][0]

jar_df[jar_turb_col] = to_numeric_safe(jar_df[jar_turb_col])
jar_df[jar_alum_col] = to_numeric_safe(jar_df[jar_alum_col])

jar_df = (
    jar_df[[jar_turb_col, jar_alum_col]]
    .dropna()
    .sort_values(jar_turb_col)
    .drop_duplicates(subset=[jar_turb_col])
)

JAR_TURBIDITY_VALUES = jar_df[jar_turb_col].to_numpy(dtype=float)
JAR_ALUM_VALUES = jar_df[jar_alum_col].to_numpy(dtype=float)

def estimate_alum_dose(turbidity):
    if pd.isna(turbidity):
        return np.nan

    clipped = np.clip(
        float(turbidity),
        JAR_TURBIDITY_VALUES.min(),
        JAR_TURBIDITY_VALUES.max(),
    )

    return float(
        np.interp(
            clipped,
            JAR_TURBIDITY_VALUES,
            JAR_ALUM_VALUES,
        )
    )


# ------------------------------------------------------------
# Chlorine target from the historical operational rule table
# ------------------------------------------------------------

chlorine_df = SUPPORT_RAW["chlorine_target"].copy()
chlorine_df.columns = chlorine_df.columns.astype(str).str.strip()

temp_col = [c for c in chlorine_df.columns if "temperature" in c.lower()][0]
turb_col = [c for c in chlorine_df.columns if "turbidity" in c.lower()][0]

ph_low_col = [c for c in chlorine_df.columns if "≤" in c or "<=" in c][0]
ph_mid_col = [c for c in chlorine_df.columns if "7.6" in c or "8.0" in c][0]
ph_high_col = [c for c in chlorine_df.columns if ">" in c and "8" in c][0]

chlorine_rules = []

for _, row in chlorine_df.iterrows():
    temp_low, temp_high = parse_range_text(row[temp_col])
    turb_low, turb_high = parse_range_text(row[turb_col])

    chlorine_rules.append({
        "temp_low": temp_low,
        "temp_high": temp_high,
        "temp_mid": np.mean([temp_low, temp_high]),
        "turb_low": turb_low,
        "turb_high": turb_high,
        "turb_label": str(row[turb_col]),
        "ph_low_dose": row[ph_low_col],
        "ph_mid_dose": row[ph_mid_col],
        "ph_high_dose": row[ph_high_col],
    })

CHLORINE_RULES_DF = pd.DataFrame(chlorine_rules)

for col in ["ph_low_dose", "ph_mid_dose", "ph_high_dose"]:
    CHLORINE_RULES_DF[col] = to_numeric_safe(CHLORINE_RULES_DF[col])

def estimate_chlorine_dose(temp, turbidity, ph):
    if pd.isna(temp) or pd.isna(turbidity) or pd.isna(ph):
        return np.nan

    matched_temp = CHLORINE_RULES_DF[
        (CHLORINE_RULES_DF["temp_low"] <= temp)
        & (temp <= CHLORINE_RULES_DF["temp_high"])
    ]

    if matched_temp.empty:
        nearest_temp_mid = (
            CHLORINE_RULES_DF["temp_mid"] - temp
        ).abs().idxmin()

        temp_low = CHLORINE_RULES_DF.loc[
            nearest_temp_mid,
            "temp_low",
        ]

        temp_high = CHLORINE_RULES_DF.loc[
            nearest_temp_mid,
            "temp_high",
        ]

        matched_temp = CHLORINE_RULES_DF[
            (CHLORINE_RULES_DF["temp_low"] == temp_low)
            & (CHLORINE_RULES_DF["temp_high"] == temp_high)
        ]

    # IMPORTANT:
    # This reproduces the historical July rule-selection logic exactly.
    if turbidity <= 1:
        matched_rule = matched_temp[
            matched_temp["turb_label"]
            .astype(str)
            .str.contains("≤|<=", regex=True)
        ]
    else:
        matched_rule = matched_temp[
            matched_temp["turb_label"]
            .astype(str)
            .str.contains("1", regex=False)
        ]

    if matched_rule.empty:
        matched_rule = matched_temp.iloc[[0]]

    rule = matched_rule.iloc[0]

    if ph <= 7.5:
        return float(rule["ph_low_dose"])
    elif ph <= 8.0:
        return float(rule["ph_mid_dose"])
    else:
        return float(rule["ph_high_dose"])


# ------------------------------------------------------------
# Additional operational alum/polymer targets
# ------------------------------------------------------------

alum_polymer_df = SUPPORT_RAW["alum_polymer_doses"].copy()
alum_polymer_df.columns = alum_polymer_df.columns.astype(str).str.strip()

range_col = [
    c for c in alum_polymer_df.columns
    if "turbidity" in c.lower()
][0]

def get_alum_polymer_rule(turbidity, target_column):
    if pd.isna(turbidity):
        return np.nan

    for _, row in alum_polymer_df.iterrows():
        low, high = parse_range_text(row[range_col])

        if low <= turbidity <= high:
            return row[target_column]

    return np.nan


# ------------------------------------------------------------
# Generate targets station by station
# ------------------------------------------------------------

STATION_TARGETS = {}

for station, source_df in STATION_CLEAN.items():
    temp = source_df.copy()

    if "Date" in temp.columns:
        temp["Date"] = pd.to_datetime(
            temp["Date"],
            errors="coerce",
        )

    numeric_columns = [
        "(°C)_raw",
        "Turb(NTU)_raw",
        "pH_raw",
        "Turb(NTU)_final",
        "pH_final",
        "Chlorine_final",
        "Al_final",
    ]

    for col in numeric_columns:
        if col in temp.columns:
            temp[col] = to_numeric_safe(temp[col])

    temp["Alum_Dose_Target"] = (
        temp["Turb(NTU)_raw"]
        .apply(estimate_alum_dose)
    )

    temp["Chlorine_Dose_Target"] = temp.apply(
        lambda row: estimate_chlorine_dose(
            row.get("(°C)_raw", np.nan),
            row.get("Turb(NTU)_final", np.nan),
            row.get("pH_final", np.nan),
        ),
        axis=1,
    )

    for dose_col in alum_polymer_df.columns:
        if dose_col == range_col:
            continue

        clean_dose_col = (
            str(dose_col)
            .replace(" ", "_")
            .replace("/", "_per_")
            .replace("(", "")
            .replace(")", "")
        )

        temp[
            clean_dose_col + "_Target"
        ] = temp["Turb(NTU)_raw"].apply(
            lambda x, dc=dose_col:
                get_alum_polymer_rule(x, dc)
        )

    STATION_TARGETS[station] = temp


ALL_DATA_TARGETS = pd.concat(
    [STATION_TARGETS[s] for s in EXPECTED_STATIONS],
    axis=0,
    ignore_index=True,
)


# ------------------------------------------------------------
# Target summary and historical fingerprint
# ------------------------------------------------------------

summary_rows = []

for station in EXPECTED_STATIONS:
    df = STATION_TARGETS[station]

    summary_rows.append({
        "Station": station,
        "Rows": int(len(df)),
        "Alum_Target_Missing":
            int(df["Alum_Dose_Target"].isna().sum()),
        "Chlorine_Target_Missing":
            int(df["Chlorine_Dose_Target"].isna().sum()),
        "Alum_Target_Min":
            float(df["Alum_Dose_Target"].min()),
        "Alum_Target_Max":
            float(df["Alum_Dose_Target"].max()),
        "Alum_Target_Mean":
            float(df["Alum_Dose_Target"].mean()),
        "Chlorine_Target_Min":
            float(df["Chlorine_Dose_Target"].min()),
        "Chlorine_Target_Max":
            float(df["Chlorine_Dose_Target"].max()),
        "Chlorine_Target_Mean":
            float(df["Chlorine_Dose_Target"].mean()),
    })

TARGET_SUMMARY_DF = pd.DataFrame(summary_rows)
target_summary_df = TARGET_SUMMARY_DF

EXPECTED_TARGET_FINGERPRINT = {
    "Kadhimiya": {
        "Rows": 1224,
        "Alum_Target_Max": 57.5,
        "Chlorine_Target_Max": 2.2,
    },
    "Karama": {
        "Rows": 1130,
        "Alum_Target_Max": 68.6,
        "Chlorine_Target_Max": 2.7,
    },
    "Resafa": {
        "Rows": 1250,
        "Alum_Target_Max": 68.0,
        "Chlorine_Target_Max": 2.9,
    },
    "Tarmyya": {
        "Rows": 1206,
        "Alum_Target_Max": 74.0,
        "Chlorine_Target_Max": 3.2,
    },
}

fingerprint_rows = []

for station, expected in EXPECTED_TARGET_FINGERPRINT.items():
    observed = TARGET_SUMMARY_DF[
        TARGET_SUMMARY_DF["Station"].eq(station)
    ].iloc[0]

    row_ok = (
        int(observed["Rows"])
        == int(expected["Rows"])
    )

    alum_ok = np.isclose(
        float(observed["Alum_Target_Max"]),
        float(expected["Alum_Target_Max"]),
        atol=1e-10,
    )

    chlorine_ok = np.isclose(
        float(observed["Chlorine_Target_Max"]),
        float(expected["Chlorine_Target_Max"]),
        atol=1e-10,
    )

    fingerprint_rows.append({
        "Station": station,
        "Rows_Observed": int(observed["Rows"]),
        "Rows_Expected": int(expected["Rows"]),
        "Alum_Max_Observed":
            float(observed["Alum_Target_Max"]),
        "Alum_Max_Expected":
            float(expected["Alum_Target_Max"]),
        "Chlorine_Max_Observed":
            float(observed["Chlorine_Target_Max"]),
        "Chlorine_Max_Expected":
            float(expected["Chlorine_Target_Max"]),
        "Fingerprint_OK":
            bool(row_ok and alum_ok and chlorine_ok),
    })

TARGET_GENERATION_FINGERPRINT_DF = pd.DataFrame(
    fingerprint_rows
)

if not bool(
    TARGET_GENERATION_FINGERPRINT_DF[
        "Fingerprint_OK"
    ].all()
):
    raise RuntimeError(
        "Historical target-generation fingerprint failed. "
        "Do not continue because the input files or target rules "
        "do not match the accepted July experiment."
    )

print("=" * 100)
print("ALUM AND CHLORINE TARGETS GENERATED SUCCESSFULLY")
print("=" * 100)

print("\nJar Test interpolation table:")
display(jar_df)

print("\nChlorine operational rules:")
display(
    CHLORINE_RULES_DF.rename(columns={
        "temp_low": "Temperature_Low",
        "temp_high": "Temperature_High",
        "temp_mid": "Temperature_Midpoint",
        "turb_low": "Turbidity_Low",
        "turb_high": "Turbidity_High",
        "turb_label": "Turbidity_Label",
        "ph_low_dose": "pH_Low_Dose",
        "ph_mid_dose": "pH_Middle_Dose",
        "ph_high_dose": "pH_High_Dose",
    })
)

print("\nTarget summary by station:")
display(TARGET_SUMMARY_DF.round(4))

print("\nHistorical target-generation fingerprint:")
display(TARGET_GENERATION_FINGERPRINT_DF)

print(
    "\nHistorical target fingerprint: PASSED"
)

print(
    f"Cell 5 completed in "
    f"{time.time() - CELL_START:.2f} seconds"
)


In [ ]:
# ============================================================
# Cell 6 — Unified Modeling Dataset and Multi-Horizon Tasks
# ============================================================

import time, numpy as np, pandas as pd
CELL_START = time.time()

RAW_FEATURE_COLUMNS = [
    "(°C)_raw", "Turb(NTU)_raw", "Al_raw", "TDS_raw",
    "pH_raw", "Alkalinity_raw", "Cond(us/cm)_raw"
]
TARGET_COLUMNS = ["Alum_Dose_Target", "Chlorine_Dose_Target"]
for c in RAW_FEATURE_COLUMNS + TARGET_COLUMNS:
    if c not in ALL_DATA_TARGETS.columns:
        raise KeyError(f"Required modeling column not found: {c}")

MODEL_BASE = ALL_DATA_TARGETS[["Date", "Station"] + RAW_FEATURE_COLUMNS + TARGET_COLUMNS].copy()
MODEL_BASE["Date"] = pd.to_datetime(MODEL_BASE["Date"], errors="coerce")
for c in RAW_FEATURE_COLUMNS + TARGET_COLUMNS:
    MODEL_BASE[c] = pd.to_numeric(MODEL_BASE[c], errors="coerce")
MODEL_BASE = MODEL_BASE.dropna(subset=["Date"] + TARGET_COLUMNS).sort_values(["Station","Date"]).reset_index(drop=True)

TASK_DEFINITIONS = []
for target in TARGET_COLUMNS:
    prefix = "Alum" if target.startswith("Alum") else "Chlorine"
    for h in [1,2,3]:
        tcol = f"{target}_t_plus_{h}"
        dcol = f"Future_Date_t_plus_{h}"
        MODEL_BASE[tcol] = MODEL_BASE.groupby("Station")[target].shift(-h)
        if dcol not in MODEL_BASE.columns:
            MODEL_BASE[dcol] = MODEL_BASE.groupby("Station")["Date"].shift(-h)
        TASK_DEFINITIONS.append({
            "Task": f"{prefix}_H{h}", "Task_ID": len(TASK_DEFINITIONS)+1,
            "Target": target, "Horizon": h, "Target_Column": tcol, "Future_Date_Column": dcol
        })

UNIFIED_TASKS_DF = pd.DataFrame(TASK_DEFINITIONS)
UNIFIED_TASKS_DF["Valid_Rows"] = UNIFIED_TASKS_DF["Target_Column"].map(lambda c: int(MODEL_BASE[c].notna().sum()))
UNIFIED_TASKS_DF["Missing_Rows"] = len(MODEL_BASE) - UNIFIED_TASKS_DF["Valid_Rows"]

availability=[]
for td in TASK_DEFINITIONS:
    for station in EXPECTED_STATIONS:
        m = MODEL_BASE["Station"].eq(station)
        availability.append({"Station":station,"Target":td["Target"],"Horizon":td["Horizon"],
                             "Target_Column":td["Target_Column"],"Valid_Rows":int(MODEL_BASE.loc[m,td["Target_Column"]].notna().sum())})
TASK_AVAILABILITY_DF = pd.DataFrame(availability)

calendar_rows=[]
for h in [1,2,3]:
    dcol=f"Future_Date_t_plus_{h}"
    valid=MODEL_BASE[dcol].notna()
    gaps=(MODEL_BASE.loc[valid,dcol]-MODEL_BASE.loc[valid,"Date"]).dt.days.astype(float)
    calendar_rows.append({"Horizon":h,"Valid_Pairs":len(gaps),"Exact_Calendar_Day_Matches":int((gaps==h).sum()),
                          "Exact_Match_Rate_Percent":100*float((gaps==h).mean()),
                          "Minimum_Calendar_Gap_Days":float(gaps.min()),"Median_Calendar_Gap_Days":float(gaps.median()),
                          "Maximum_Calendar_Gap_Days":float(gaps.max())})
CALENDAR_GAP_DIAGNOSTIC_DF=pd.DataFrame(calendar_rows)

print("="*95)
print("UNIFIED MULTI-HORIZON MODELING DATASET CREATED")
print("="*95)
print(f"Base dataset rows: {len(MODEL_BASE)} | Raw input features: {len(RAW_FEATURE_COLUMNS)} | Tasks: {len(TASK_DEFINITIONS)}")
display(UNIFIED_TASKS_DF)
print("Task availability by station:")
display(TASK_AVAILABILITY_DF)
print("Calendar-gap diagnostic:")
display(CALENDAR_GAP_DIAGNOSTIC_DF.round(2))
print(f"Cell 6 completed in {time.time()-CELL_START:.2f} seconds")


In [ ]:
# ============================================================
# Cell 7 — Transparent Multi-Split / Multi-Task UTW Study
# Paper 3
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

CELL_START = time.time()

# ============================================================
# Fixed configuration
# ============================================================

UTW_CANDIDATES = [1, 3, 5, 7, 10]

FIXED_ALPHA = 1.0

SPLIT_EXPERIMENTS = {

     "S55_V15": {"train_ratio": 0.55, "validation_ratio": 0.15},
}


# ============================================================
# Helpers
# ============================================================

def utw_safe_std(values):

    values = np.asarray(values, dtype=float)

    std = np.std(
        values,
        axis=0,
        ddof=0,
    )

    bad = (
        (~np.isfinite(std))
        | (std < 1e-12)
    )

    std[bad] = 1.0

    return std


def utw_assign_split(
    frame,
    train_ratio,
    validation_ratio,
):

    if (
        train_ratio <= 0
        or validation_ratio <= 0
        or train_ratio + validation_ratio >= 1
    ):
        raise ValueError(
            "Invalid split ratios."
        )

    out = frame.copy()

    out["Split"] = "Test"

    for station in EXPECTED_STATIONS:

        idx = out.index[
            out["Station"].eq(station)
        ].to_numpy()

        n = len(idx)

        train_end = int(
            n * train_ratio
        )

        validation_end = int(
            n * (
                train_ratio
                + validation_ratio
            )
        )

        out.loc[
            idx[:train_end],
            "Split",
        ] = "Train"

        out.loc[
            idx[train_end:validation_end],
            "Split",
        ] = "Validation"

        out.loc[
            idx[validation_end:],
            "Split",
        ] = "Test"

    return out


def utw_temporal_slope(values):

    values = np.asarray(
        values,
        dtype=float,
    )

    n_rows = values.shape[0]
    n_steps = values.shape[1]

    result = np.full(
        n_rows,
        np.nan,
        dtype=float,
    )

    valid = np.isfinite(
        values
    ).all(axis=1)

    if not np.any(valid):
        return result

    if n_steps <= 1:

        result[valid] = 0.0

        return result

    x = np.arange(
        n_steps,
        dtype=float,
    )

    x_centered = (
        x - x.mean()
    )

    denominator = np.sum(
        x_centered ** 2
    )

    v = values[valid]

    v_centered = (
        v
        - np.mean(
            v,
            axis=1,
            keepdims=True,
        )
    )

    result[valid] = (
        v_centered
        @ x_centered
    ) / denominator

    return result


def make_descriptors(matrix):

    matrix = np.asarray(
        matrix,
        dtype=float,
    )

    n_rows = matrix.shape[0]
    n_steps = matrix.shape[1]

    output = {
        "mean": np.full(n_rows, np.nan),
        "std": np.full(n_rows, np.nan),
        "min": np.full(n_rows, np.nan),
        "max": np.full(n_rows, np.nan),
        "slope": np.full(n_rows, np.nan),
        "recent_change": np.full(n_rows, np.nan),
    }

    valid = (
        np.isfinite(matrix)
        .all(axis=1)
    )

    if not np.any(valid):
        return output

    v = matrix[valid]

    output["mean"][valid] = np.mean(
        v,
        axis=1,
    )

    output["std"][valid] = np.std(
        v,
        axis=1,
        ddof=0,
    )

    output["min"][valid] = np.min(
        v,
        axis=1,
    )

    output["max"][valid] = np.max(
        v,
        axis=1,
    )

    output["slope"] = (
        utw_temporal_slope(
            matrix
        )
    )

    if n_steps >= 2:

        output[
            "recent_change"
        ][valid] = (
            v[:, -1]
            - v[:, -2]
        )

    else:

        output[
            "recent_change"
        ][valid] = 0.0

    return output


# ============================================================
# Build UTW features
# ============================================================

def build_unified_temporal_window(
    base_df,
    window,
):

    window = int(window)

    df = (
        base_df
        .copy()
        .sort_values(
            [
                "Station",
                "Date",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    new_columns = {}

    # --------------------------------------------------------
    # Calendar
    # --------------------------------------------------------

    new_columns["Year"] = (
        df["Date"].dt.year.to_numpy()
    )

    new_columns["Month"] = (
        df["Date"].dt.month.to_numpy()
    )

    new_columns["Day"] = (
        df["Date"].dt.day.to_numpy()
    )

    new_columns["DayOfYear"] = (
        df["Date"].dt.dayofyear.to_numpy()
    )

    new_columns["DayOfWeek"] = (
        df["Date"].dt.dayofweek.to_numpy()
    )

    new_columns["Quarter"] = (
        df["Date"].dt.quarter.to_numpy()
    )

    new_columns["WeekOfYear"] = (
        df["Date"]
        .dt.isocalendar()
        .week
        .astype(int)
        .to_numpy()
    )

    new_columns["Month_sin"] = np.sin(
        2 * np.pi
        * new_columns["Month"]
        / 12
    )

    new_columns["Month_cos"] = np.cos(
        2 * np.pi
        * new_columns["Month"]
        / 12
    )

    new_columns[
        "DayOfYear_sin"
    ] = np.sin(
        2 * np.pi
        * new_columns[
            "DayOfYear"
        ]
        / 365.25
    )

    new_columns[
        "DayOfYear_cos"
    ] = np.cos(
        2 * np.pi
        * new_columns[
            "DayOfYear"
        ]
        / 365.25
    )

    time_columns = [
        "Year",
        "Month",
        "Day",
        "DayOfYear",
        "DayOfWeek",
        "Quarter",
        "WeekOfYear",
        "Month_sin",
        "Month_cos",
        "DayOfYear_sin",
        "DayOfYear_cos",
    ]

    # --------------------------------------------------------
    # Station dummies
    # --------------------------------------------------------

    station_columns = []

    for station in EXPECTED_STATIONS:

        name = (
            f"Station_{station}"
        )

        new_columns[name] = (
            df["Station"]
            .eq(station)
            .astype(int)
            .to_numpy()
        )

        station_columns.append(
            name
        )

    grouped = df.groupby(
        "Station",
        sort=False,
    )

    # --------------------------------------------------------
    # Raw temporal windows
    # --------------------------------------------------------

    raw_window_columns = []
    raw_summary_columns = []

    for column in RAW_FEATURE_COLUMNS:

        lag_names = []

        for offset in range(window):

            name = (
                f"{column}"
                f"__UTW_tminus_{offset}"
            )

            if offset == 0:

                values = (
                    df[column]
                    .to_numpy(
                        dtype=float
                    )
                )

            else:

                values = (
                    grouped[column]
                    .shift(offset)
                    .to_numpy(
                        dtype=float
                    )
                )

            new_columns[name] = values

            lag_names.append(name)

            raw_window_columns.append(
                name
            )

        matrix = np.column_stack(
            [
                new_columns[name]
                for name in lag_names
            ]
        )

        chronological = (
            matrix[:, ::-1]
        )

        descriptors = (
            make_descriptors(
                chronological
            )
        )

        names = {
            "mean":
                f"{column}__UTW_mean",
            "std":
                f"{column}__UTW_std",
            "min":
                f"{column}__UTW_min",
            "max":
                f"{column}__UTW_max",
            "slope":
                f"{column}__UTW_slope",
            "recent_change":
                f"{column}__UTW_recent_change",
        }

        for key, name in names.items():

            new_columns[name] = (
                descriptors[key]
            )

            raw_summary_columns.append(
                name
            )

    # --------------------------------------------------------
    # Target history — past only
    # --------------------------------------------------------

    target_history_columns = []
    target_summary_columns = []

    for target in TARGET_COLUMNS:

        history_names = []

        for lag in range(
            1,
            window + 1,
        ):

            name = (
                f"{target}"
                f"__UTW_history_{lag}"
            )

            new_columns[name] = (
                grouped[target]
                .shift(lag)
                .to_numpy(
                    dtype=float
                )
            )

            history_names.append(name)

            target_history_columns.append(
                name
            )

        matrix = np.column_stack(
            [
                new_columns[name]
                for name
                in history_names
            ]
        )

        chronological = (
            matrix[:, ::-1]
        )

        descriptors = (
            make_descriptors(
                chronological
            )
        )

        names = {
            "mean":
                f"{target}__UTW_hist_mean",

            "std":
                f"{target}__UTW_hist_std",

            "min":
                f"{target}__UTW_hist_min",

            "max":
                f"{target}__UTW_hist_max",

            "slope":
                f"{target}__UTW_hist_slope",

            "recent_change":
                f"{target}__UTW_hist_recent_change",
        }

        for key, name in names.items():

            new_columns[name] = (
                descriptors[key]
            )

            target_summary_columns.append(
                name
            )

    # --------------------------------------------------------
    # Single concat
    # --------------------------------------------------------

    feature_df = pd.DataFrame(
        new_columns,
        index=df.index,
    )

    df = pd.concat(
        [
            df,
            feature_df,
        ],
        axis=1,
    ).copy()

    feature_columns = (
        raw_window_columns
        + raw_summary_columns
        + target_history_columns
        + target_summary_columns
        + time_columns
        + station_columns
    )

    df = (
        df
        .dropna(
            subset=feature_columns
        )
        .reset_index(
            drop=True
        )
    )

    return (
        df,
        feature_columns,
    )


# ============================================================
# Cache each window
# ============================================================

UTW_FEATURE_CACHE = {}

for window in UTW_CANDIDATES:

    (
        window_df,
        window_features,
    ) = build_unified_temporal_window(
        MODEL_BASE,
        window,
    )

    UTW_FEATURE_CACHE[
        window
    ] = {
        "Frame":
            window_df,

        "Feature_Columns":
            window_features,
    }


# ============================================================
# Evaluate all
# split × window × task × station
# ============================================================

rows = []

for split_name, cfg in (
    SPLIT_EXPERIMENTS.items()
):

    train_ratio = float(
        cfg["train_ratio"]
    )

    validation_ratio = float(
        cfg[
            "validation_ratio"
        ]
    )

    for window in UTW_CANDIDATES:

        window_df = (
            UTW_FEATURE_CACHE[
                window
            ]["Frame"]
        )

        feature_columns = (
            UTW_FEATURE_CACHE[
                window
            ][
                "Feature_Columns"
            ]
        )

        for task_meta in (
            TASK_DEFINITIONS
        ):

            task = (
                task_meta["Task"]
            )

            target_column = (
                task_meta[
                    "Target_Column"
                ]
            )

            future_date_column = (
                task_meta[
                    "Future_Date_Column"
                ]
            )

            task_frame = (
                window_df
                .dropna(
                    subset=[
                        target_column,
                        future_date_column,
                    ]
                )
                .sort_values(
                    [
                        "Station",
                        "Date",
                    ]
                )
                .reset_index(
                    drop=True
                )
            )

            task_frame = (
                utw_assign_split(
                    task_frame,
                    train_ratio,
                    validation_ratio,
                )
            )

            train_df = (
                task_frame[
                    task_frame[
                        "Split"
                    ].eq(
                        "Train"
                    )
                ]
                .copy()
            )

            val_df = (
                task_frame[
                    task_frame[
                        "Split"
                    ].eq(
                        "Validation"
                    )
                ]
                .copy()
            )

            # ------------------------------------------------
            # Active features based on train only
            # ------------------------------------------------

            nunique = (
                train_df[
                    feature_columns
                ]
                .nunique(
                    dropna=True
                )
            )

            active_features = [
                c
                for c
                in feature_columns
                if nunique[c] > 1
            ]

            # ------------------------------------------------
            # Standardization based on train only
            # ------------------------------------------------

            mean = (
                train_df[
                    active_features
                ]
                .mean()
                .to_numpy(
                    dtype=float
                )
            )

            std = utw_safe_std(
                train_df[
                    active_features
                ]
                .to_numpy(
                    dtype=float
                )
            )

            X_train = (
                train_df[
                    active_features
                ]
                .to_numpy(
                    dtype=float
                )
                - mean
            ) / std

            X_val = (
                val_df[
                    active_features
                ]
                .to_numpy(
                    dtype=float
                )
                - mean
            ) / std

            y_train = (
                train_df[
                    target_column
                ]
                .to_numpy(
                    dtype=float
                )
            )

            model = Ridge(
                alpha=FIXED_ALPHA,
                fit_intercept=True,
            )

            model.fit(
                X_train,
                y_train,
            )

            val_pred = (
                model.predict(
                    X_val
                )
            )

            val_eval = (
                val_df.copy()
                .reset_index(
                    drop=True
                )
            )

            val_eval[
                "_Prediction"
            ] = val_pred

            # ------------------------------------------------
            # Evaluate each station
            # ------------------------------------------------

            for station in (
                EXPECTED_STATIONS
            ):

                part = (
                    val_eval[
                        val_eval[
                            "Station"
                        ].eq(
                            station
                        )
                    ]
                )

                if len(part) == 0:
                    continue

                y = (
                    part[
                        target_column
                    ]
                    .to_numpy(
                        dtype=float
                    )
                )

                p = (
                    part[
                        "_Prediction"
                    ]
                    .to_numpy(
                        dtype=float
                    )
                )

                mae = float(
                    mean_absolute_error(
                        y,
                        p,
                    )
                )

                rmse = float(
                    np.sqrt(
                        mean_squared_error(
                            y,
                            p,
                        )
                    )
                )

                if (
                    len(y) >= 2
                    and
                    np.var(y) > 1e-12
                ):

                    r2 = float(
                        r2_score(
                            y,
                            p,
                        )
                    )

                else:

                    r2 = np.nan

                rows.append(
                    {
                        "Split":
                            split_name,

                        "Train_Ratio":
                            train_ratio,

                        "Validation_Ratio":
                            validation_ratio,

                        "Window":
                            window,

                        "Task":
                            task,

                        "Target":
                            task_meta[
                                "Target"
                            ],

                        "Horizon":
                            int(
                                task_meta[
                                    "Horizon"
                                ]
                            ),

                        "Station":
                            station,

                        "MAE":
                            mae,

                        "RMSE":
                            rmse,

                        "R2":
                            r2,

                        "Test_Used":
                            False,
                    }
                )


# ============================================================
# Station-level results
# ============================================================

UTW_TRANSPARENT_STATION_DF = (
    pd.DataFrame(rows)
)


# ============================================================
# Task-level results
# Equal station weighting
# ============================================================

UTW_TRANSPARENT_TASK_DF = (
    UTW_TRANSPARENT_STATION_DF
    .groupby(
        [
            "Split",
            "Train_Ratio",
            "Validation_Ratio",
            "Window",
            "Task",
            "Target",
            "Horizon",
        ],
        as_index=False,
    )
    .agg(
        MAE=(
            "MAE",
            "mean",
        ),
        RMSE=(
            "RMSE",
            "mean",
        ),
        R2=(
            "R2",
            "mean",
        ),
    )
)


# ============================================================
# Wide transparent R2 table
# One row = Split + W
# Each task shown separately
# ============================================================

UTW_R2_WIDE_DF = (
    UTW_TRANSPARENT_TASK_DF
    .pivot_table(
        index=[
            "Split",
            "Train_Ratio",
            "Validation_Ratio",
            "Window",
        ],
        columns="Task",
        values="R2",
    )
    .reset_index()
)

UTW_R2_WIDE_DF.columns.name = None


task_columns = [
    c
    for c
    in UTW_R2_WIDE_DF.columns
    if c
    not in [
        "Split",
        "Train_Ratio",
        "Validation_Ratio",
        "Window",
    ]
]


UTW_R2_WIDE_DF[
    "Mean_R2"
] = (
    UTW_R2_WIDE_DF[
        task_columns
    ]
    .mean(
        axis=1
    )
)


UTW_R2_WIDE_DF[
    "Min_R2"
] = (
    UTW_R2_WIDE_DF[
        task_columns
    ]
    .min(
        axis=1
    )
)


UTW_R2_WIDE_DF[
    "Positive_R2_Tasks"
] = (
    UTW_R2_WIDE_DF[
        task_columns
    ]
    .gt(0)
    .sum(
        axis=1
    )
)


# ============================================================
# MAE wide table
# ============================================================

UTW_MAE_WIDE_DF = (
    UTW_TRANSPARENT_TASK_DF
    .pivot_table(
        index=[
            "Split",
            "Train_Ratio",
            "Validation_Ratio",
            "Window",
        ],
        columns="Task",
        values="MAE",
    )
    .reset_index()
)

UTW_MAE_WIDE_DF.columns.name = None

mae_task_columns = [
    c
    for c
    in UTW_MAE_WIDE_DF.columns
    if c
    not in [
        "Split",
        "Train_Ratio",
        "Validation_Ratio",
        "Window",
    ]
]

UTW_MAE_WIDE_DF[
    "Mean_MAE"
] = (
    UTW_MAE_WIDE_DF[
        mae_task_columns
    ]
    .mean(
        axis=1
    )
)


# ============================================================
# RMSE wide table
# ============================================================

UTW_RMSE_WIDE_DF = (
    UTW_TRANSPARENT_TASK_DF
    .pivot_table(
        index=[
            "Split",
            "Train_Ratio",
            "Validation_Ratio",
            "Window",
        ],
        columns="Task",
        values="RMSE",
    )
    .reset_index()
)

UTW_RMSE_WIDE_DF.columns.name = None

rmse_task_columns = [
    c
    for c
    in UTW_RMSE_WIDE_DF.columns
    if c
    not in [
        "Split",
        "Train_Ratio",
        "Validation_Ratio",
        "Window",
    ]
]

UTW_RMSE_WIDE_DF[
    "Mean_RMSE"
] = (
    UTW_RMSE_WIDE_DF[
        rmse_task_columns
    ]
    .mean(
        axis=1
    )
)


# ============================================================
# Best W per split
# Primary:
#   1) most positive R2 tasks
#   2) highest Mean R2
#   3) highest Min R2
# ============================================================

best_rows = []

for split_name in (
    SPLIT_EXPERIMENTS.keys()
):

    part = (
        UTW_R2_WIDE_DF[
            UTW_R2_WIDE_DF[
                "Split"
            ].eq(
                split_name
            )
        ]
        .copy()
        .sort_values(
            [
                "Positive_R2_Tasks",
                "Mean_R2",
                "Min_R2",
                "Window",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )

    best_rows.append(
        part.iloc[0]
    )


BEST_W_PER_SPLIT_DF = (
    pd.DataFrame(
        best_rows
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# Final transparent split ranking
# ============================================================

TRANSPARENT_SPLIT_RANKING_DF = (
    BEST_W_PER_SPLIT_DF
    .sort_values(
        [
            "Positive_R2_Tasks",
            "Mean_R2",
            "Min_R2",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


BEST_SPLIT_NAME = (
    TRANSPARENT_SPLIT_RANKING_DF
    .iloc[0][
        "Split"
    ]
)

BEST_TRAIN_RATIO = float(
    TRANSPARENT_SPLIT_RANKING_DF
    .iloc[0][
        "Train_Ratio"
    ]
)

BEST_VALIDATION_RATIO = float(
    TRANSPARENT_SPLIT_RANKING_DF
    .iloc[0][
        "Validation_Ratio"
    ]
)

SELECTED_UTW = int(
    TRANSPARENT_SPLIT_RANKING_DF
    .iloc[0][
        "Window"
    ]
)


# ============================================================
# Safety
# ============================================================

if bool(
    UTW_TRANSPARENT_STATION_DF[
        "Test_Used"
    ].any()
):

    raise RuntimeError(
        "Test leakage detected."
    )


# ============================================================
# Output
# ============================================================

print(
    "=" * 120
)

print(
    "CELL 7 — TRANSPARENT MULTI-TASK UTW STUDY COMPLETED"
)

print(
    "=" * 120
)

print(
    f"Fixed alpha = {FIXED_ALPHA}"
)

print(
    f"Candidate windows = {UTW_CANDIDATES}"
)

print(
    f"Number of splits = "
    f"{len(SPLIT_EXPERIMENTS)}"
)


print(
    "\nR2 — every task separately:"
)

display(
    UTW_R2_WIDE_DF
    .sort_values(
        "Mean_R2",
        ascending=False,
    )
    .round(6)
)


print(
    "\nBest W inside each split — transparent task-level R2:"
)

display(
    BEST_W_PER_SPLIT_DF
    .sort_values(
        "Mean_R2",
        ascending=False,
    )
    .round(6)
)


print(
    "\nFinal transparent split ranking:"
)

display(
    TRANSPARENT_SPLIT_RANKING_DF
    .round(6)
)


print(
    "\nMAE — every task separately:"
)

display(
    UTW_MAE_WIDE_DF
    .sort_values(
        "Mean_MAE",
        ascending=True,
    )
    .round(6)
)


print(
    "\nRMSE — every task separately:"
)

display(
    UTW_RMSE_WIDE_DF
    .sort_values(
        "Mean_RMSE",
        ascending=True,
    )
    .round(6)
)


print(
    "\n" + "=" * 120
)

print(
    f"BEST SPLIT = {BEST_SPLIT_NAME}"
)

print(
    f"TRAIN_RATIO = {BEST_TRAIN_RATIO}"
)

print(
    f"VALIDATION_RATIO = {BEST_VALIDATION_RATIO}"
)

print(
    f"SELECTED_UTW = {SELECTED_UTW}"
)

print(
    "=" * 120
)

print(
    "\nNo Test data used."
)

print(
    f"\nCell 7 completed in "
    f"{time.time() - CELL_START:.2f} seconds"
)

In [ ]:
# CELL 7A — Chronological Train/Validation Split-Ratio Sweep
# Place after Cell 7 and before Cell 8.
# The final chronological 30% test tail is identical for every candidate.
# Selection uses validation only; test labels are not scored here.

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

_REQUIRED_7A = [
    "UTW_FEATURE_CACHE",
    "UTW_CANDIDATES",
    "TASK_DEFINITIONS",
    "EXPECTED_STATIONS",
    "utw_safe_std",
    "BEST_TRAIN_RATIO",
    "BEST_VALIDATION_RATIO",
    "SELECTED_UTW",
]
_MISSING_7A = [name for name in _REQUIRED_7A if name not in globals()]
if _MISSING_7A:
    raise RuntimeError(
        "Run Cell 7 first. Missing: " + ", ".join(_MISSING_7A)
    )

TEST_RATIO_FIXED = 0.30
TRAIN_VALIDATION_TOTAL = 1.0 - TEST_RATIO_FIXED

# Ratios are percentages of each station's full chronological series.
# The existing 55/15/30 protocol is included as the baseline.
SPLIT_SWEEP_CONFIGS = {
    "S40_V30_T30": {"train_ratio": 0.40, "validation_ratio": 0.30},
    "S45_V25_T30": {"train_ratio": 0.45, "validation_ratio": 0.25},
    "S50_V20_T30": {"train_ratio": 0.50, "validation_ratio": 0.20},
    "S55_V15_T30": {"train_ratio": 0.55, "validation_ratio": 0.15},
    "S60_V10_T30": {"train_ratio": 0.60, "validation_ratio": 0.10},
}

for _name, _cfg in SPLIT_SWEEP_CONFIGS.items():
    if not np.isclose(
        _cfg["train_ratio"] + _cfg["validation_ratio"],
        TRAIN_VALIDATION_TOTAL,
        atol=1e-10,
    ):
        raise ValueError(f"{_name} does not preserve the fixed 30% test tail.")

# Use the same test boundary for every candidate and in downstream Cell 8.
def utw_assign_split_fixed_test(frame, train_ratio, validation_ratio):
    if not np.isclose(
        train_ratio + validation_ratio,
        TRAIN_VALIDATION_TOTAL,
        atol=1e-10,
    ):
        raise ValueError("Train + validation must equal 70%.")

    out = frame.copy()
    out["Split"] = "Test"

    for station in EXPECTED_STATIONS:
        idx = (
            out.loc[out["Station"].eq(station)]
            .sort_values("Date")
            .index.to_numpy()
        )
        n = len(idx)
        test_start = int(np.floor(n * TRAIN_VALIDATION_TOTAL))
        train_end = int(np.floor(n * train_ratio))

        if (
            train_end < 2
            or test_start - train_end < 2
            or n - test_start < 2
        ):
            raise ValueError(
                f"Too few chronological rows for {station}: "
                f"n={n}, train={train_end}, "
                f"validation={test_start-train_end}, test={n-test_start}"
            )

        out.loc[idx[:train_end], "Split"] = "Train"
        out.loc[idx[train_end:test_start], "Split"] = "Validation"

    return out

# Keep Cell 8's final split consistent with this sweep.
utw_assign_split = utw_assign_split_fixed_test

_sweep_rows = []
_sweep_excluded = []

for _split_name, _cfg in SPLIT_SWEEP_CONFIGS.items():
    _train_ratio = float(_cfg["train_ratio"])
    _validation_ratio = float(_cfg["validation_ratio"])

    for _window in UTW_CANDIDATES:
        _window_df = UTW_FEATURE_CACHE[_window]["Frame"]
        _feature_columns = UTW_FEATURE_CACHE[_window]["Feature_Columns"]
        _window_rows = []
        _invalid_reason = None

        for _task_meta in TASK_DEFINITIONS:
            _task = _task_meta["Task"]
            _target_column = _task_meta["Target_Column"]
            _future_date_column = _task_meta["Future_Date_Column"]

            _task_frame = (
                _window_df
                .dropna(subset=[_target_column, _future_date_column])
                .sort_values(["Station", "Date"])
                .reset_index(drop=True)
            )

            _task_frame = utw_assign_split_fixed_test(
                _task_frame,
                _train_ratio,
                _validation_ratio,
            )

            _train_df = _task_frame[_task_frame["Split"].eq("Train")].copy()
            _val_df = _task_frame[_task_frame["Split"].eq("Validation")].copy()

            _train_counts = (
                _train_df.groupby("Station").size()
                .reindex(EXPECTED_STATIONS, fill_value=0)
            )
            _val_counts = (
                _val_df.groupby("Station").size()
                .reindex(EXPECTED_STATIONS, fill_value=0)
            )

            if (_train_counts < 2).any() or (_val_counts < 2).any():
                _invalid_reason = (
                    f"Insufficient train/validation rows for task {_task}"
                )
                break

            # Feature filtering and scaling use training rows only.
            _nunique = _train_df[_feature_columns].nunique(dropna=True)
            _active_features = [
                col for col in _feature_columns if _nunique[col] > 1
            ]
            if not _active_features:
                _invalid_reason = f"No active training features for task {_task}"
                break

            _train_mean = (
                _train_df[_active_features].mean().to_numpy(dtype=float)
            )
            _train_std = utw_safe_std(
                _train_df[_active_features].to_numpy(dtype=float)
            )

            _X_train = (
                _train_df[_active_features].to_numpy(dtype=float) - _train_mean
            ) / _train_std
            _X_val = (
                _val_df[_active_features].to_numpy(dtype=float) - _train_mean
            ) / _train_std
            _y_train = _train_df[_target_column].to_numpy(dtype=float)

            _model = Ridge(alpha=FIXED_ALPHA, fit_intercept=True)
            _model.fit(_X_train, _y_train)
            _val_pred = _model.predict(_X_val)

            _val_eval = _val_df.copy().reset_index(drop=True)
            _val_eval["_Prediction"] = _val_pred

            for _station in EXPECTED_STATIONS:
                _part = _val_eval[_val_eval["Station"].eq(_station)]
                _y = _part[_target_column].to_numpy(dtype=float)
                _pred = _part["_Prediction"].to_numpy(dtype=float)

                _r2 = (
                    float(r2_score(_y, _pred))
                    if len(_y) >= 2 and np.var(_y) > 1e-12
                    else np.nan
                )

                _window_rows.append({
                    "Split": _split_name,
                    "Train_Ratio": _train_ratio,
                    "Validation_Ratio": _validation_ratio,
                    "Test_Ratio": TEST_RATIO_FIXED,
                    "Window": int(_window),
                    "Task": _task,
                    "Target": _task_meta["Target"],
                    "Horizon": int(_task_meta["Horizon"]),
                    "Station": _station,
                    "Train_N": int(_train_counts.loc[_station]),
                    "Validation_N": int(_val_counts.loc[_station]),
                    "MAE": float(mean_absolute_error(_y, _pred)),
                    "RMSE": float(np.sqrt(mean_squared_error(_y, _pred))),
                    "R2": _r2,
                    "Test_Used": False,
                })

        if _invalid_reason:
            _sweep_excluded.append({
                "Split": _split_name,
                "Window": int(_window),
                "Reason": _invalid_reason,
            })
        else:
            _sweep_rows.extend(_window_rows)

if not _sweep_rows:
    raise RuntimeError(
        "No split/window candidate had enough data for a complete comparison."
    )

UTW_TRANSPARENT_STATION_DF = pd.DataFrame(_sweep_rows)
SPLIT_SWEEP_EXCLUDED_DF = pd.DataFrame(_sweep_excluded)

# Equal station weighting within each task, matching Cell 7's evaluation logic.
UTW_TRANSPARENT_TASK_DF = (
    UTW_TRANSPARENT_STATION_DF
    .groupby(
        [
            "Split", "Train_Ratio", "Validation_Ratio", "Test_Ratio",
            "Window", "Task", "Target", "Horizon",
        ],
        as_index=False,
    )
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        R2=("R2", "mean"),
        Scored_Stations=("R2", "count"),
    )
)

_expected_tasks = list(dict.fromkeys(x["Task"] for x in TASK_DEFINITIONS))
_meta_cols = [
    "Split", "Train_Ratio", "Validation_Ratio", "Test_Ratio", "Window"
]

def _sweep_wide_table(metric):
    _wide = UTW_TRANSPARENT_TASK_DF.pivot(
        index=_meta_cols,
        columns="Task",
        values=metric,
    )
    _wide = _wide.reindex(columns=_expected_tasks).reset_index()
    _wide.columns.name = None
    return _wide

UTW_R2_WIDE_DF = _sweep_wide_table("R2")
UTW_MAE_WIDE_DF = _sweep_wide_table("MAE")
UTW_RMSE_WIDE_DF = _sweep_wide_table("RMSE")

UTW_R2_WIDE_DF["Scored_Tasks"] = (
    UTW_R2_WIDE_DF[_expected_tasks].notna().sum(axis=1)
)
UTW_R2_WIDE_DF["Mean_R2"] = UTW_R2_WIDE_DF[_expected_tasks].mean(axis=1)
UTW_R2_WIDE_DF["Min_R2"] = UTW_R2_WIDE_DF[_expected_tasks].min(axis=1)
UTW_R2_WIDE_DF["Positive_R2_Tasks"] = (
    UTW_R2_WIDE_DF[_expected_tasks].gt(0).sum(axis=1)
)

UTW_MAE_WIDE_DF["Mean_MAE"] = UTW_MAE_WIDE_DF[_expected_tasks].mean(axis=1)
UTW_RMSE_WIDE_DF["Mean_RMSE"] = UTW_RMSE_WIDE_DF[_expected_tasks].mean(axis=1)

_sweep_coverage = (
    UTW_TRANSPARENT_TASK_DF
    .groupby(["Split", "Window"], as_index=False)
    .agg(
        Task_Count=("Task", "nunique"),
        All_Stations_Scored=(
            "Scored_Stations",
            lambda s: bool(s.eq(len(EXPECTED_STATIONS)).all()),
        ),
    )
)
_sweep_coverage["Complete_Coverage"] = (
    _sweep_coverage["Task_Count"].eq(len(_expected_tasks))
    & _sweep_coverage["All_Stations_Scored"]
)

_sweep_ranking_base = (
    UTW_R2_WIDE_DF
    .merge(
        UTW_MAE_WIDE_DF[_meta_cols + ["Mean_MAE"]],
        on=_meta_cols,
        how="left",
    )
    .merge(
        UTW_RMSE_WIDE_DF[_meta_cols + ["Mean_RMSE"]],
        on=_meta_cols,
        how="left",
    )
    .merge(_sweep_coverage, on=["Split", "Window"], how="left")
)

_sweep_eligible = _sweep_ranking_base[
    _sweep_ranking_base["Scored_Tasks"].eq(len(_expected_tasks))
    & _sweep_ranking_base["Complete_Coverage"].eq(True)
    & np.isfinite(_sweep_ranking_base["Mean_R2"])
].copy()

if _sweep_eligible.empty:
    raise RuntimeError(
        "No candidate had complete station/task validation coverage. "
        "Review SPLIT_SWEEP_EXCLUDED_DF and the R2 coverage."
    )

# Select the best window within each split using validation metrics only.
BEST_W_PER_SPLIT_DF = (
    _sweep_eligible
    .sort_values(
        [
            "Split", "Positive_R2_Tasks", "Mean_R2", "Min_R2",
            "Mean_MAE", "Mean_RMSE", "Window",
        ],
        ascending=[True, False, False, False, True, True, True],
    )
    .groupby("Split", as_index=False)
    .first()
)

# Rank split protocols by the same validation criterion.
# Exact ties favor the larger validation block for a more stable estimate.
SPLIT_SWEEP_RANKING_DF = BEST_W_PER_SPLIT_DF.sort_values(
    [
        "Positive_R2_Tasks", "Mean_R2", "Min_R2",
        "Mean_MAE", "Mean_RMSE", "Validation_Ratio",
    ],
    ascending=[False, False, False, True, True, False],
).reset_index(drop=True)

TRANSPARENT_SPLIT_RANKING_DF = SPLIT_SWEEP_RANKING_DF.copy()

BEST_SPLIT_NAME = str(SPLIT_SWEEP_RANKING_DF.iloc[0]["Split"])
BEST_TRAIN_RATIO = float(SPLIT_SWEEP_RANKING_DF.iloc[0]["Train_Ratio"])
BEST_VALIDATION_RATIO = float(
    SPLIT_SWEEP_RANKING_DF.iloc[0]["Validation_Ratio"]
)
SELECTED_UTW = int(SPLIT_SWEEP_RANKING_DF.iloc[0]["Window"])

# Leave Cell 8 and downstream cells with the selected configuration.
SPLIT_EXPERIMENTS = {
    BEST_SPLIT_NAME: {
        "train_ratio": BEST_TRAIN_RATIO,
        "validation_ratio": BEST_VALIDATION_RATIO,
    }
}

print("SPLIT-RATIO SWEEP — VALIDATION ONLY")
print(f"Common untouched test tail: {TEST_RATIO_FIXED:.0%}")
print(f"Selected split: {BEST_SPLIT_NAME}")
print(
    f"Train/Validation/Test = "
    f"{BEST_TRAIN_RATIO:.0%}/"
    f"{BEST_VALIDATION_RATIO:.0%}/"
    f"{TEST_RATIO_FIXED:.0%}"
)
print(f"Selected UTW: W={SELECTED_UTW}")

display(SPLIT_SWEEP_RANKING_DF.round(6))
display(
    UTW_R2_WIDE_DF
    .sort_values(
        ["Positive_R2_Tasks", "Mean_R2"],
        ascending=[False, False],
    )
    .round(6)
)
if not SPLIT_SWEEP_EXCLUDED_DF.empty:
    print("Excluded split/window candidates and reasons:")
    display(SPLIT_SWEEP_EXCLUDED_DF)

In [ ]:
# ============================================================
# Cell 8 — Freeze Selected UTW, Split, and Normalize
# Compatible with Transparent Cell 7
# Training-only feature filtering and scaling
# ============================================================

import time
import numpy as np
import pandas as pd

CELL_START = time.time()

# ============================================================
# Required outputs from Cell 7
# ============================================================

required_cell7_objects = [
    "SELECTED_UTW",
    "BEST_TRAIN_RATIO",
    "BEST_VALIDATION_RATIO",
    "UTW_CANDIDATES",
    "UTW_FEATURE_CACHE",
]

missing_objects = [
    name
    for name in required_cell7_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing Cell 7 outputs: "
        + ", ".join(missing_objects)
        + ". Run Cell 7 first."
    )


# ============================================================
# Freeze final configuration selected in Cell 7
# ============================================================

FINAL_UTW = int(SELECTED_UTW)

FINAL_TRAIN_RATIO = float(BEST_TRAIN_RATIO)
FINAL_VALIDATION_RATIO = float(BEST_VALIDATION_RATIO)

FINAL_TEST_RATIO = (
    1.0
    - FINAL_TRAIN_RATIO
    - FINAL_VALIDATION_RATIO
)


if FINAL_UTW not in UTW_CANDIDATES:
    raise RuntimeError(
        "Selected unified window is outside "
        "the predeclared UTW candidates."
    )


if FINAL_TEST_RATIO <= 0:
    raise RuntimeError(
        "Invalid final split ratios."
    )


# ============================================================
# Retrieve the already-built selected UTW
# Do NOT rebuild with a different definition
# ============================================================

MODEL_FEATURES = (
    UTW_FEATURE_CACHE[
        FINAL_UTW
    ]["Frame"]
    .copy()
)

FEATURE_COLUMNS = (
    UTW_FEATURE_CACHE[
        FINAL_UTW
    ]["Feature_Columns"]
    .copy()
)


# ============================================================
# Feature-group diagnostics
# ============================================================

def classify_utw_feature(column):

    if "__UTW_tminus_" in column:
        return "Raw temporal window"

    if "__UTW_hist_" in column:
        return "Target-history descriptor"

    if "__UTW_history_" in column:
        return "Target history"

    if "__UTW_" in column:
        return "Raw temporal descriptor"

    if column.startswith("Station_"):
        return "Station encoding"

    if column in [
        "Year",
        "Month",
        "Day",
        "DayOfYear",
        "DayOfWeek",
        "Quarter",
        "WeekOfYear",
        "Month_sin",
        "Month_cos",
        "DayOfYear_sin",
        "DayOfYear_cos",
    ]:
        return "Calendar"

    return "Other"


UTW_SELECTED_FEATURE_GROUPS_DF = (
    pd.DataFrame(
        {
            "Feature": FEATURE_COLUMNS,
            "Feature_Group": [
                classify_utw_feature(c)
                for c in FEATURE_COLUMNS
            ],
        }
    )
    .groupby(
        "Feature_Group",
        as_index=False,
    )
    .agg(
        Feature_Count=(
            "Feature",
            "count",
        )
    )
    .sort_values(
        "Feature_Count",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


UTW_SELECTED_WINDOW_DIAGNOSTIC = {
    "Selected_Unified_Window":
        FINAL_UTW,

    "Train_Ratio":
        FINAL_TRAIN_RATIO,

    "Validation_Ratio":
        FINAL_VALIDATION_RATIO,

    "Test_Ratio":
        FINAL_TEST_RATIO,

    "Generated_Features":
        len(FEATURE_COLUMNS),
}


# ============================================================
# Final split function
# Uses ONLY ratios frozen by Cell 7
# ============================================================

def final_assign_split(frame):

    return utw_assign_split(
        frame,
        FINAL_TRAIN_RATIO,
        FINAL_VALIDATION_RATIO,
    )


# ============================================================
# Training-only structural feature filtering
# ============================================================

feature_split_frame = (
    MODEL_FEATURES[
        ["Station", "Date"]
        + FEATURE_COLUMNS
    ]
    .copy()
)

feature_split_frame = (
    final_assign_split(
        feature_split_frame
    )
)


feature_training_frame = (
    feature_split_frame[
        feature_split_frame[
            "Split"
        ].eq("Train")
    ]
    .copy()
)


feature_nunique = (
    feature_training_frame[
        FEATURE_COLUMNS
    ]
    .nunique(
        dropna=True
    )
)


REMOVED_CONSTANT_FEATURES = (
    feature_nunique[
        feature_nunique <= 1
    ]
    .index
    .tolist()
)


ACTIVE_FEATURE_COLUMNS = [
    column
    for column in FEATURE_COLUMNS
    if column
    not in REMOVED_CONSTANT_FEATURES
]


REMOVED_CONSTANT_FEATURES_DF = (
    pd.DataFrame(
        {
            "Removed_Constant_Feature":
                REMOVED_CONSTANT_FEATURES
        }
    )
)


# ============================================================
# Numerical helper
#
# 1e-12 is NOT a tuning hyperparameter.
# It is only a numerical-stability tolerance.
# ============================================================

NUMERICAL_EPS = 1e-12


def safe_std(a):

    a = np.asarray(
        a,
        dtype=float,
    )

    s = np.nanstd(
        a,
        axis=0,
        ddof=0,
    )

    return np.where(
        (~np.isfinite(s))
        | (s < NUMERICAL_EPS),
        1.0,
        s,
    )


# ============================================================
# Create final task datasets and scalers
# ============================================================

TASK_DATASETS = {}

TASK_CLIENT_DATASETS = {}

TASK_SCALERS = {}


overall_rows = []

station_rows = []

date_rows = []

scaler_rows = []


for td in TASK_DEFINITIONS:

    task = td["Task"]

    target_col = (
        td["Target_Column"]
    )

    future_date_col = (
        td["Future_Date_Column"]
    )


    # --------------------------------------------------------
    # Valid task-specific frame
    # --------------------------------------------------------

    task_df = (
        MODEL_FEATURES
        .dropna(
            subset=[
                target_col,
                future_date_col,
            ]
        )
        .copy()
        .sort_values(
            [
                "Station",
                "Date",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Same frozen chronological split
    # --------------------------------------------------------

    task_df = (
        final_assign_split(
            task_df
        )
    )


    train = (
        task_df[
            task_df[
                "Split"
            ].eq("Train")
        ]
        .copy()
    )


    if len(train) == 0:
        raise RuntimeError(
            f"No training observations "
            f"for task {task}."
        )


    # --------------------------------------------------------
    # Training-only feature scaling
    # --------------------------------------------------------

    x_mean = (
        train[
            ACTIVE_FEATURE_COLUMNS
        ]
        .mean()
        .to_numpy(
            dtype=float
        )
    )


    x_std = safe_std(
        train[
            ACTIVE_FEATURE_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )


    # --------------------------------------------------------
    # Training-only target statistics
    # --------------------------------------------------------

    y_mean = float(
        train[
            target_col
        ].mean()
    )


    y_std = float(
        train[
            target_col
        ].std(
            ddof=0
        )
    )


    if (
        not np.isfinite(y_std)
        or y_std < NUMERICAL_EPS
    ):
        y_std = 1.0


    # --------------------------------------------------------
    # Save task scaler
    # --------------------------------------------------------

    TASK_SCALERS[
        task
    ] = {

        "x_mean":
            x_mean,

        "x_std":
            x_std,

        "y_mean":
            y_mean,

        "y_std":
            y_std,

        "features":
            ACTIVE_FEATURE_COLUMNS.copy(),

        "Unified_Window":
            FINAL_UTW,

        "Train_Ratio":
            FINAL_TRAIN_RATIO,

        "Validation_Ratio":
            FINAL_VALIDATION_RATIO,

        "Test_Ratio":
            FINAL_TEST_RATIO,
    }


    # ========================================================
    # Global task datasets
    # ========================================================

    split_dict = {}

    TASK_CLIENT_DATASETS[
        task
    ] = {}


    for split in [
        "Train",
        "Validation",
        "Test",
    ]:

        sdf = (
            task_df[
                task_df[
                    "Split"
                ].eq(split)
            ]
            .copy()
        )


        X = (
            sdf[
                ACTIVE_FEATURE_COLUMNS
            ]
            .to_numpy(
                dtype=float
            )
            - x_mean
        ) / x_std


        y = (
            sdf[
                target_col
            ]
            .to_numpy(
                dtype=float
            )
        )


        split_dict[
            split
        ] = {

            "X":
                X,

            "y":
                y,

            "Frame":
                sdf.reset_index(
                    drop=True
                ),

            "Date":
                sdf[
                    "Date"
                ].to_numpy(),

            "Target_Date":
                sdf[
                    future_date_col
                ].to_numpy(),
        }


        overall_rows.append(
            {
                "Task":
                    task,

                "Target":
                    td["Target"],

                "Horizon":
                    td["Horizon"],

                "Split":
                    split,

                "Rows":
                    len(sdf),

                "Unified_Window":
                    FINAL_UTW,

                "Train_Ratio":
                    FINAL_TRAIN_RATIO,

                "Validation_Ratio":
                    FINAL_VALIDATION_RATIO,

                "Test_Ratio":
                    FINAL_TEST_RATIO,
            }
        )


    TASK_DATASETS[
        task
    ] = {

        "Meta":
            td.copy(),

        **split_dict,

        "Frame":
            task_df,
    }


    # ========================================================
    # Station/client-specific datasets
    # ========================================================

    for station in EXPECTED_STATIONS:

        TASK_CLIENT_DATASETS[
            task
        ][station] = {}


        station_frame = (
            task_df[
                task_df[
                    "Station"
                ].eq(station)
            ]
        )


        date_record = {

            "Task":
                task,

            "Station":
                station,

            "Unified_Window":
                FINAL_UTW,
        }


        for split in [
            "Train",
            "Validation",
            "Test",
        ]:

            sdf = (
                station_frame[
                    station_frame[
                        "Split"
                    ].eq(split)
                ]
                .copy()
            )


            X = (
                sdf[
                    ACTIVE_FEATURE_COLUMNS
                ]
                .to_numpy(
                    dtype=float
                )
                - x_mean
            ) / x_std


            y = (
                sdf[
                    target_col
                ]
                .to_numpy(
                    dtype=float
                )
            )


            TASK_CLIENT_DATASETS[
                task
            ][station][split] = {

                "X":
                    X,

                "y":
                    y,

                "Frame":
                    sdf.reset_index(
                        drop=True
                    ),

                "Date":
                    sdf[
                        "Date"
                    ].to_numpy(),

                "Target_Date":
                    sdf[
                        future_date_col
                    ].to_numpy(),
            }


            station_rows.append(
                {
                    "Task":
                        task,

                    "Target":
                        td["Target"],

                    "Horizon":
                        td["Horizon"],

                    "Station":
                        station,

                    "Split":
                        split,

                    "Rows":
                        len(sdf),

                    "Unified_Window":
                        FINAL_UTW,
                }
            )


            if len(sdf) > 0:

                date_record[
                    f"{split}_Start_Target_Date"
                ] = pd.to_datetime(
                    sdf[
                        future_date_col
                    ].min()
                )


                date_record[
                    f"{split}_End_Target_Date"
                ] = pd.to_datetime(
                    sdf[
                        future_date_col
                    ].max()
                )


        date_rows.append(
            date_record
        )


    # ========================================================
    # Scaling diagnostics
    # ========================================================

    X_train_scaled = (
        split_dict[
            "Train"
        ]["X"]
    )


    scaler_rows.append(
        {
            "Task":
                task,

            "Training_Rows":
                len(
                    split_dict[
                        "Train"
                    ]["y"]
                ),

            "Validation_Rows":
                len(
                    split_dict[
                        "Validation"
                    ]["y"]
                ),

            "Test_Rows":
                len(
                    split_dict[
                        "Test"
                    ]["y"]
                ),

            "Generated_Features":
                len(
                    FEATURE_COLUMNS
                ),

            "Active_Features":
                len(
                    ACTIVE_FEATURE_COLUMNS
                ),

            "Unified_Window":
                FINAL_UTW,

            "Training_Constant_Features":
                int(
                    (
                        np.std(
                            X_train_scaled,
                            axis=0,
                        )
                        < NUMERICAL_EPS
                    ).sum()
                ),

            "Maximum_Absolute_Scaled_Mean":
                float(
                    np.max(
                        np.abs(
                            np.mean(
                                X_train_scaled,
                                axis=0,
                            )
                        )
                    )
                ),

            "Maximum_Absolute_Std_Deviation_From_One":
                float(
                    np.max(
                        np.abs(
                            np.std(
                                X_train_scaled,
                                axis=0,
                            )
                            - 1.0
                        )
                    )
                ),
        }
    )


# ============================================================
# Summary DataFrames
# ============================================================

OVERALL_SPLIT_SUMMARY_DF = (
    pd.DataFrame(
        overall_rows
    )
)


STATION_SPLIT_SUMMARY_DF = (
    pd.DataFrame(
        station_rows
    )
)


TARGET_DATE_SPLIT_SUMMARY_DF = (
    pd.DataFrame(
        date_rows
    )
)


SCALER_DIAGNOSTICS_DF = (
    pd.DataFrame(
        scaler_rows
    )
)


# ============================================================
# Final frozen protocol
# ============================================================

UTW_FINAL_PROTOCOL_DF = (
    pd.DataFrame(
        [
            {
                "Selected_Unified_Window":
                    FINAL_UTW,

                "Train_Ratio":
                    FINAL_TRAIN_RATIO,

                "Validation_Ratio":
                    FINAL_VALIDATION_RATIO,

                "Test_Ratio":
                    FINAL_TEST_RATIO,

                "Window_Selected_From":
                    "Train + Validation only",

                "Test_Used_For_Window_Selection":
                    False,

                "Final_Generated_Features":
                    len(
                        FEATURE_COLUMNS
                    ),

                "Final_Active_Features":
                    len(
                        ACTIVE_FEATURE_COLUMNS
                    ),

                "Constant_Feature_Filter":
                    "Training rows only",

                "Scaling":
                    "Task training rows only",

                "Same_Window_All_Stations":
                    True,

                "Same_Window_All_Targets":
                    True,

                "Same_Window_All_Horizons":
                    True,

                "Same_Window_All_Models":
                    True,
            }
        ]
    )
)


# ============================================================
# Protocol checks
# ============================================================

if FINAL_UTW != int(
    SELECTED_UTW
):
    raise RuntimeError(
        "Frozen UTW does not match "
        "Cell 7 selection."
    )


if not np.isclose(
    FINAL_TRAIN_RATIO,
    BEST_TRAIN_RATIO,
):
    raise RuntimeError(
        "Training ratio mismatch "
        "with Cell 7."
    )


if not np.isclose(
    FINAL_VALIDATION_RATIO,
    BEST_VALIDATION_RATIO,
):
    raise RuntimeError(
        "Validation ratio mismatch "
        "with Cell 7."
    )


if len(
    ACTIVE_FEATURE_COLUMNS
) < 1:
    raise RuntimeError(
        "No active features remain "
        "after training-only filtering."
    )


# ============================================================
# Output
# ============================================================

print(
    "=" * 120
)

print(
    "SELECTED UTW FROZEN AND FINAL DATASETS CREATED"
)

print(
    "=" * 120
)


print(
    f"Selected unified window : "
    f"W={FINAL_UTW}"
)

print(
    f"Train ratio             : "
    f"{FINAL_TRAIN_RATIO:.2f}"
)

print(
    f"Validation ratio        : "
    f"{FINAL_VALIDATION_RATIO:.2f}"
)

print(
    f"Test ratio              : "
    f"{FINAL_TEST_RATIO:.2f}"
)

print(
    f"Generated features      : "
    f"{len(FEATURE_COLUMNS)}"
)

print(
    "Constant features removed "
    "from training only: "
    f"{len(REMOVED_CONSTANT_FEATURES)}"
)

print(
    f"Active shared features  : "
    f"{len(ACTIVE_FEATURE_COLUMNS)}"
)


print(
    "\nFeature groups:"
)

display(
    UTW_SELECTED_FEATURE_GROUPS_DF
)


print(
    "\nRemoved constant features:"
)

display(
    REMOVED_CONSTANT_FEATURES_DF
)


print(
    "\nOverall split summary:"
)

display(
    OVERALL_SPLIT_SUMMARY_DF
)


print(
    "\nTraining-only scaler diagnostics:"
)

display(
    SCALER_DIAGNOSTICS_DF
    .round(6)
)


print(
    "\nFinal UTW protocol:"
)

display(
    UTW_FINAL_PROTOCOL_DF
)


print(
    "\nPaper 3 UTW protocol check: PASSED"
)

print(
    f"\nCell 8 completed in "
    f"{time.time() - CELL_START:.2f} seconds"
)

In [ ]:
# ============================================================
# Cell 8A — Training-Only Correlation Analysis

import pandas as pd

variables = {
    "(°C)_raw": "Temp",
    "Turb(NTU)_raw": "Turb",
    "TDS_raw": "TDS",
    "pH_raw": "pH",
    "Alkalinity_raw": "Alkalinity",
    "Cond(us/cm)_raw": "Cond",
}

data = final_assign_split(MODEL_BASE.copy())
train = data.loc[data["Split"].eq("Train"), list(variables)]
correlation = train.rename(columns=variables).corr(method="pearson")

print(f"Training records: {len(train)}")
display(correlation.round(3))
correlation.to_csv("Paper3_Training_Pearson_Correlation.csv")

In [ ]:
# ============================================================
# Cell 9 — Common Algorithms and Fixed Training Protocol
# BEFORE Adaptive Tuning
# ============================================================

import time
import numpy as np
import pandas as pd

CELL_START = time.time()


# ============================================================
# Global fixed settings
# ============================================================

# Reproducibility — NOT a tuning parameter
GLOBAL_RANDOM_SEED = 42

# Centralized / Local training
# Tunable later
CENTRAL_LOCAL_EPOCHS = 120

# Gradient stability
# Tunable later, but keep fixed for baseline
GRADIENT_CLIP_NORM = 5.0

# Numerical stability — NOT a tuning parameter
EPS = 1e-12


# ============================================================
# Federated protocol
# Defined here only for protocol documentation.
# Actual FL implementation is handled later.
# ============================================================

# Tunable later
FEDERATED_ROUNDS_PROTOCOL = 60

# Tunable later
FEDERATED_LOCAL_EPOCHS_PROTOCOL = 2

# Keep all 4 stations participating for the main experiment
FEDERATED_CLIENT_FRACTION_PROTOCOL = 1.0


# ============================================================
# Baseline algorithm configurations
#
# IMPORTANT:
# These are the FIXED pre-tuning settings.
# No Validation/Test tuning is performed in this cell.
# ============================================================

ALGORITHM_CONFIGS = {

    # --------------------------------------------------------
    # Ridge
    # --------------------------------------------------------
    "Ridge": {
        "name": "Ridge",
        "loss": "Squared",
        "optimizer": "FullBatch",

        # Tunable later
        "learning_rate": 0.010,
        "learning_rate_decay": 0.995,

        "l1": 0.0,
        "l2": 0.0100,

        "huber_delta": None,
        "batch_size": None,
    },

    # --------------------------------------------------------
    # ElasticNet
    # --------------------------------------------------------
    "ElasticNet": {
        "name": "ElasticNet",
        "loss": "Squared",
        "optimizer": "FullBatchProximal",

        # Tunable later
        "learning_rate": 0.008,
        "learning_rate_decay": 0.995,

        "l1": 0.0005,
        "l2": 0.0015,

        "huber_delta": None,
        "batch_size": None,
    },

    # --------------------------------------------------------
    # SGD Linear
    # --------------------------------------------------------
    "SGD_Linear": {
        "name": "SGD Linear",
        "loss": "Squared",
        "optimizer": "MiniBatchSGD",

        # Tunable later
        "learning_rate": 0.010,
        "learning_rate_decay": 0.995,

        "l1": 0.0,
        "l2": 0.0001,

        "huber_delta": None,

        # Tunable later
        "batch_size": 64,
    },

    # --------------------------------------------------------
    # SGD Huber
    # --------------------------------------------------------
    "SGD_Huber": {
        "name": "SGD Huber",
        "loss": "Huber",
        "optimizer": "MiniBatchSGD",

        # Tunable later
        "learning_rate": 0.010,
        "learning_rate_decay": 0.995,

        "l1": 0.0,
        "l2": 0.0001,

        # Tunable later
        "huber_delta": 1.35,

        # Tunable later
        "batch_size": 64,
    },
}


# ============================================================
# Regression metrics
# ============================================================

def reg_metrics(y, p):

    y = np.asarray(
        y,
        dtype=float,
    ).reshape(-1)

    p = np.asarray(
        p,
        dtype=float,
    ).reshape(-1)

    valid = (
        np.isfinite(y)
        & np.isfinite(p)
    )

    y = y[valid]
    p = p[valid]

    if len(y) == 0:
        return {
            "N": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "R2": np.nan,
            "MBE": np.nan,
            "NRMSE_Std": np.nan,
            "NRMSE_Range": np.nan,
        }

    error = p - y

    mae = float(
        np.mean(
            np.abs(error)
        )
    )

    rmse = float(
        np.sqrt(
            np.mean(
                error ** 2
            )
        )
    )

    sst = float(
        np.sum(
            (
                y
                - y.mean()
            ) ** 2
        )
    )

    if sst > EPS:
        r2 = float(
            1.0
            - np.sum(
                error ** 2
            )
            / sst
        )
    else:
        r2 = np.nan

    std = float(
        np.std(
            y,
            ddof=0,
        )
    )

    value_range = float(
        np.max(y)
        - np.min(y)
    )

    return {
        "N": len(y),

        "MAE": mae,

        "RMSE": rmse,

        "R2": r2,

        "MBE": float(
            np.mean(error)
        ),

        "NRMSE_Std":
            rmse / std
            if std > EPS
            else np.nan,

        "NRMSE_Range":
            rmse / value_range
            if value_range > EPS
            else np.nan,
    }


# ============================================================
# Linear prediction
# ============================================================

def linear_predict(
    X,
    params,
):

    X = np.asarray(
        X,
        dtype=float,
    )

    params = np.asarray(
        params,
        dtype=float,
    ).reshape(-1)

    return (
        X @ params[:-1]
        + params[-1]
    )


# ============================================================
# Gradient clipping
# ============================================================

def clip_grad(
    gradient,
    max_norm=GRADIENT_CLIP_NORM,
):

    gradient = np.asarray(
        gradient,
        dtype=float,
    )

    norm = float(
        np.linalg.norm(
            gradient
        )
    )

    if (
        norm > max_norm
        and norm > 0
    ):
        gradient = (
            gradient
            * max_norm
            / norm
        )

    return gradient


# ============================================================
# Batch gradient
# ============================================================

def batch_gradient(
    X,
    y,
    params,
    cfg,
):

    prediction = linear_predict(
        X,
        params,
    )

    residual = (
        prediction
        - y
    )

    delta = cfg[
        "huber_delta"
    ]

    # Squared loss
    if delta is None:

        psi = residual

    # Huber loss
    else:

        psi = np.clip(
            residual,
            -delta,
            delta,
        )

    grad_weights = (
        X.T @ psi
        / max(
            len(y),
            1,
        )
    )

    # L2 regularization on weights only
    grad_weights = (
        grad_weights
        + cfg["l2"]
        * params[:-1]
    )

    grad_bias = np.array(
        [
            float(
                np.mean(
                    psi
                )
            )
        ]
    )

    gradient = np.concatenate(
        [
            grad_weights,
            grad_bias,
        ]
    )

    return clip_grad(
        gradient
    )


# ============================================================
# Common linear-model trainer
# ============================================================

def train_linear_model(
    X,
    y_raw,
    task,
    algorithm_key,
    epochs=CENTRAL_LOCAL_EPOCHS,
    seed=GLOBAL_RANDOM_SEED,
    initial=None,
):

    if algorithm_key not in ALGORITHM_CONFIGS:
        raise KeyError(
            f"Unknown algorithm: {algorithm_key}"
        )

    if task not in TASK_SCALERS:
        raise KeyError(
            f"No scaler found for task: {task}"
        )

    cfg = ALGORITHM_CONFIGS[
        algorithm_key
    ]

    scaler = TASK_SCALERS[
        task
    ]

    # --------------------------------------------------------
    # Standardize target using TRAIN statistics from Cell 8
    # --------------------------------------------------------

    y = (
        np.asarray(
            y_raw,
            dtype=float,
        )
        - scaler["y_mean"]
    ) / scaler["y_std"]

    X = np.asarray(
        X,
        dtype=float,
    )

    n_rows, n_features = X.shape

    # --------------------------------------------------------
    # Initial parameters
    # --------------------------------------------------------

    if initial is None:

        params = np.zeros(
            n_features + 1,
            dtype=float,
        )

    else:

        params = np.asarray(
            initial,
            dtype=float,
        ).copy()

        if len(params) != (
            n_features + 1
        ):
            raise ValueError(
                "Initial parameter size "
                "does not match feature count."
            )

    rng = np.random.default_rng(
        seed
    )

    history = []

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(
        int(epochs)
    ):

        learning_rate = (
            cfg[
                "learning_rate"
            ]
            * (
                cfg[
                    "learning_rate_decay"
                ]
                ** epoch
            )
        )

        if cfg[
            "batch_size"
        ] is None:

            batch_size = n_rows

        else:

            batch_size = min(
                int(
                    cfg[
                        "batch_size"
                    ]
                ),
                n_rows,
            )

        # Deterministic full batch
        if batch_size == n_rows:

            order = np.arange(
                n_rows
            )

        # Reproducible mini-batch shuffle
        else:

            order = rng.permutation(
                n_rows
            )

        for start in range(
            0,
            n_rows,
            batch_size,
        ):

            idx = order[
                start:
                start + batch_size
            ]

            gradient = (
                batch_gradient(
                    X[idx],
                    y[idx],
                    params,
                    cfg,
                )
            )

            params -= (
                learning_rate
                * gradient
            )

            # -----------------------------------------------
            # Proximal L1 update
            # -----------------------------------------------

            if cfg["l1"] > 0:

                weights = (
                    params[:-1]
                )

                params[:-1] = (
                    np.sign(
                        weights
                    )
                    * np.maximum(
                        np.abs(
                            weights
                        )
                        - learning_rate
                        * cfg["l1"],
                        0.0,
                    )
                )

        # ----------------------------------------------------
        # Lightweight training history
        # ----------------------------------------------------

        if (
            epoch == 0
            or epoch
            == int(epochs) - 1
            or (
                epoch + 1
            ) % 20 == 0
        ):

            history.append(
                {
                    "Epoch":
                        epoch + 1,

                    "Learning_Rate":
                        learning_rate,

                    "Parameter_Norm":
                        float(
                            np.linalg.norm(
                                params
                            )
                        ),
                }
            )

    return (
        params,
        history,
    )


# ============================================================
# Convert standardized prediction back to original dose scale
# ============================================================

def predict_raw(
    X,
    params,
    task,
):

    scaler = TASK_SCALERS[
        task
    ]

    standardized_prediction = (
        linear_predict(
            X,
            params,
        )
    )

    return (
        standardized_prediction
        * scaler["y_std"]
        + scaler["y_mean"]
    )


# ============================================================
# Algorithm configuration table
# ============================================================

ALGORITHM_CONFIGURATION_DF = (
    pd.DataFrame(
        [
            {
                "Algorithm_Key":
                    key,

                "Algorithm":
                    cfg["name"],

                "Loss":
                    cfg["loss"],

                "Optimizer":
                    cfg["optimizer"],

                "Learning_Rate":
                    cfg[
                        "learning_rate"
                    ],

                "Learning_Rate_Decay":
                    cfg[
                        "learning_rate_decay"
                    ],

                "L1_Penalty":
                    cfg["l1"],

                "L2_Penalty":
                    cfg["l2"],

                "Huber_Delta":
                    cfg[
                        "huber_delta"
                    ],

                "Batch_Size":
                    cfg[
                        "batch_size"
                    ],

                "Gradient_Clip_Norm":
                    GRADIENT_CLIP_NORM,

                "Epochs":
                    CENTRAL_LOCAL_EPOCHS,
            }

            for key, cfg
            in ALGORITHM_CONFIGS.items()
        ]
    )
)


# ============================================================
# Experiment protocol
# ============================================================

EXPERIMENT_PROTOCOL_DF = (
    pd.DataFrame(
        [
            {
                "Training_Paradigm":
                    "Centralized",

                "Training_Units":
                    "All stations pooled",

                "Passes_or_Rounds":
                    CENTRAL_LOCAL_EPOCHS,

                "Local_Epochs_Per_Round":
                    np.nan,

                "Client_Fraction":
                    np.nan,

                "Aggregation":
                    "None",
            },

            {
                "Training_Paradigm":
                    "Local",

                "Training_Units":
                    "One model per station",

                "Passes_or_Rounds":
                    CENTRAL_LOCAL_EPOCHS,

                "Local_Epochs_Per_Round":
                    np.nan,

                "Client_Fraction":
                    np.nan,

                "Aggregation":
                    "None",
            },

            {
                "Training_Paradigm":
                    "Federated",

                "Training_Units":
                    "Four station clients",

                "Passes_or_Rounds":
                    FEDERATED_ROUNDS_PROTOCOL,

                "Local_Epochs_Per_Round":
                    FEDERATED_LOCAL_EPOCHS_PROTOCOL,

                "Client_Fraction":
                    FEDERATED_CLIENT_FRACTION_PROTOCOL,

                "Aggregation":
                    "Sample-weighted",
            },
        ]
    )
)


# ============================================================
# Sanity checks with Cell 8
# ============================================================

if "TASK_SCALERS" not in globals():
    raise NameError(
        "TASK_SCALERS not found. "
        "Run Cell 8 first."
    )

if "ACTIVE_FEATURE_COLUMNS" not in globals():
    raise NameError(
        "ACTIVE_FEATURE_COLUMNS not found. "
        "Run Cell 8 first."
    )

if len(
    ACTIVE_FEATURE_COLUMNS
) == 0:
    raise RuntimeError(
        "No active features available."
    )


# ============================================================
# Output
# ============================================================

print(
    "=" * 110
)

print(
    "COMMON ALGORITHMS AND FIXED TRAINING PROTOCOL CREATED"
)

print(
    "=" * 110
)


print(
    "\nAlgorithm configurations:"
)

display(
    ALGORITHM_CONFIGURATION_DF
)


print(
    "\nExperiment protocol:"
)

display(
    EXPERIMENT_PROTOCOL_DF
)


print(
    "\nCell 8 compatibility:"
)

print(
    f"Active input features = "
    f"{len(ACTIVE_FEATURE_COLUMNS)}"
)

if "FINAL_UTW" in globals():

    print(
        f"Frozen UTW = "
        f"{FINAL_UTW}"
    )


print(
    "\nNo adaptive tuning performed in Cell 9."
)

print(
    f"\nCell 9 completed in "
    f"{time.time() - CELL_START:.2f} seconds"
)

In [ ]:
# ============================================================
# CELL 10 — FINAL CENTRALIZED MODEL TUNING AND SELECTION
# ============================================================
#
# Final Paper 3 protocol:
#
# - 4 centralized algorithms
# - 5 refined configurations per algorithm
# - 6 prediction tasks
# - 120 model fits
#
# Hyperparameter selection:
#   Validation ONLY
#
# Ranking:
#   1) Minimum Validation MAE
#   2) Minimum Validation RMSE
#   3) Maximum Validation R2
#
# IMPORTANT:
#   Test set is NOT evaluated or used in this cell.
#
# ============================================================

import time
import copy
import numpy as np
import pandas as pd

CELL_START = time.time()


# ============================================================
# 1. Required objects
# ============================================================

required_objects = [
    "TASK_DATASETS",
    "TASK_DEFINITIONS",
    "ALGORITHM_CONFIGS",
    "CENTRAL_LOCAL_EPOCHS",
    "GLOBAL_RANDOM_SEED",
    "EXPECTED_STATIONS",
    "train_linear_model",
    "predict_raw",
    "reg_metrics",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


# ============================================================
# 2. FINAL FIXED PROTOCOL
# ============================================================
#
# Not tuned here:
#
# W = 10
# Split = 55% Train / 15% Validation / 30% Test
# Epochs = 120
# Gradient clipping = 5.0
# Learning-rate decay = 0.995
# SGD batch size = 64
# Seed protocol = fixed/deterministic
#
# ============================================================

TUNING_EPOCHS = CENTRAL_LOCAL_EPOCHS
FIXED_DECAY = 0.995
FIXED_SGD_BATCH_SIZE = 64


# ============================================================
# 3. FINAL REFINED SEARCH GRID
#
# Exactly 5 configurations per algorithm
# ============================================================

CENTRAL_TUNING_GRID = {

    # --------------------------------------------------------
    # Ridge
    # Tuned:
    #   learning rate
    #   L2
    # --------------------------------------------------------

    "Ridge": [

        {
            "Config_ID": "RR1",
            "learning_rate": 0.015,
            "l2": 0.030,
        },

        {
            "Config_ID": "RR2",
            "learning_rate": 0.020,
            "l2": 0.050,
        },

        {
            "Config_ID": "RR3",
            "learning_rate": 0.025,
            "l2": 0.075,
        },

        {
            "Config_ID": "RR4",
            "learning_rate": 0.030,
            "l2": 0.100,
        },

        {
            "Config_ID": "RR5",
            "learning_rate": 0.040,
            "l2": 0.150,
        },
    ],


    # --------------------------------------------------------
    # ElasticNet
    # Tuned:
    #   learning rate
    #   L1
    #   L2
    # --------------------------------------------------------

    "ElasticNet": [

        {
            "Config_ID": "ER1",
            "learning_rate": 0.015,
            "l1": 0.0010,
            "l2": 0.0050,
        },

        {
            "Config_ID": "ER2",
            "learning_rate": 0.020,
            "l1": 0.0020,
            "l2": 0.0100,
        },

        {
            "Config_ID": "ER3",
            "learning_rate": 0.025,
            "l1": 0.0025,
            "l2": 0.0125,
        },

        {
            "Config_ID": "ER4",
            "learning_rate": 0.030,
            "l1": 0.0030,
            "l2": 0.0150,
        },

        {
            "Config_ID": "ER5",
            "learning_rate": 0.040,
            "l1": 0.0040,
            "l2": 0.0200,
        },
    ],


    # --------------------------------------------------------
    # SGD Linear
    # Tuned:
    #   learning rate
    #   L2
    #
    # Fixed:
    #   batch size = 64
    # --------------------------------------------------------

    "SGD_Linear": [

        {
            "Config_ID": "SR1",
            "learning_rate": 0.0020,
            "l2": 0.000010,
        },

        {
            "Config_ID": "SR2",
            "learning_rate": 0.0040,
            "l2": 0.000025,
        },

        {
            "Config_ID": "SR3",
            "learning_rate": 0.0075,
            "l2": 0.000075,
        },

        {
            "Config_ID": "SR4",
            "learning_rate": 0.0125,
            "l2": 0.000250,
        },

        {
            "Config_ID": "SR5",
            "learning_rate": 0.0175,
            "l2": 0.000750,
        },
    ],


    # --------------------------------------------------------
    # SGD Huber
    # Tuned:
    #   learning rate
    #   L2
    #   Huber delta
    #
    # Fixed:
    #   batch size = 64
    # --------------------------------------------------------

    "SGD_Huber": [

        {
            "Config_ID": "HR1",
            "learning_rate": 0.0010,
            "l2": 0.000010,
            "huber_delta": 0.50,
        },

        {
            "Config_ID": "HR2",
            "learning_rate": 0.0020,
            "l2": 0.000025,
            "huber_delta": 0.60,
        },

        {
            "Config_ID": "HR3",
            "learning_rate": 0.0030,
            "l2": 0.000050,
            "huber_delta": 0.75,
        },

        {
            "Config_ID": "HR4",
            "learning_rate": 0.0040,
            "l2": 0.000075,
            "huber_delta": 0.90,
        },

        {
            "Config_ID": "HR5",
            "learning_rate": 0.0050,
            "l2": 0.000100,
            "huber_delta": 1.00,
        },
    ],
}


# ============================================================
# 4. Validate tuning grid
# ============================================================

for algorithm_key in ALGORITHM_CONFIGS:

    if algorithm_key not in CENTRAL_TUNING_GRID:
        raise RuntimeError(
            f"Missing tuning grid for {algorithm_key}"
        )

    if len(CENTRAL_TUNING_GRID[algorithm_key]) != 5:
        raise RuntimeError(
            f"{algorithm_key} must have exactly 5 configurations."
        )


# ============================================================
# 5. Task and algorithm lists
# ============================================================

task_keys = [
    definition["Task"]
    for definition in TASK_DEFINITIONS
]

algorithm_keys = list(
    ALGORITHM_CONFIGS.keys()
)

TOTAL_TRAINING_RUNS = (
    len(task_keys)
    * len(algorithm_keys)
    * 5
)


# ============================================================
# 6. Readable grid table
# ============================================================

grid_rows = []

for algorithm_key, candidates in CENTRAL_TUNING_GRID.items():

    for candidate in candidates:

        cfg = copy.deepcopy(
            ALGORITHM_CONFIGS[algorithm_key]
        )

        for key, value in candidate.items():
            if key != "Config_ID":
                cfg[key] = value

        cfg["learning_rate_decay"] = FIXED_DECAY

        if algorithm_key in [
            "SGD_Linear",
            "SGD_Huber",
        ]:
            cfg["batch_size"] = FIXED_SGD_BATCH_SIZE

        grid_rows.append(
            {
                "Algorithm_Key":
                    algorithm_key,

                "Algorithm":
                    cfg["name"],

                "Config_ID":
                    candidate["Config_ID"],

                "Learning_Rate":
                    cfg["learning_rate"],

                "Learning_Rate_Decay":
                    cfg["learning_rate_decay"],

                "L1":
                    cfg["l1"],

                "L2":
                    cfg["l2"],

                "Huber_Delta":
                    cfg["huber_delta"],

                "Batch_Size":
                    cfg["batch_size"],

                "Epochs":
                    TUNING_EPOCHS,
            }
        )


CENTRAL_TUNING_GRID_DF = pd.DataFrame(
    grid_rows
)


# ============================================================
# 7. Containers
# ============================================================

CENTRALIZED_TUNING_MODELS = {}
CENTRALIZED_TUNING_HISTORIES = {}

CENTRALIZED_MODELS = {}
CENTRALIZED_HISTORIES = {}

tuning_rows = []

run_counter = 0


# ============================================================
# 8. Preserve original Cell 9 algorithm configurations
# ============================================================

ORIGINAL_ALGORITHM_CONFIGS = copy.deepcopy(
    ALGORITHM_CONFIGS
)


# ============================================================
# 9. FINAL TUNING
# ============================================================

try:

    for task_position, task_key in enumerate(
        task_keys,
        start=1,
    ):

        meta = TASK_DATASETS[
            task_key
        ]["Meta"]

        X_train = TASK_DATASETS[
            task_key
        ]["Train"]["X"]

        y_train = TASK_DATASETS[
            task_key
        ]["Train"]["y"]

        X_val = TASK_DATASETS[
            task_key
        ]["Validation"]["X"]

        y_val = TASK_DATASETS[
            task_key
        ]["Validation"]["y"]


        for algorithm_position, algorithm_key in enumerate(
            algorithm_keys,
            start=1,
        ):

            base_cfg = copy.deepcopy(
                ORIGINAL_ALGORITHM_CONFIGS[
                    algorithm_key
                ]
            )


            # ------------------------------------------------
            # Fixed deterministic seed
            #
            # All five hyperparameter configurations for the
            # same Task × Algorithm use the SAME seed.
            # ------------------------------------------------

            deterministic_seed = (
                GLOBAL_RANDOM_SEED
                + task_position * 100
                + algorithm_position
            )


            for candidate in CENTRAL_TUNING_GRID[
                algorithm_key
            ]:

                run_counter += 1

                config_id = candidate[
                    "Config_ID"
                ]


                candidate_cfg = copy.deepcopy(
                    base_cfg
                )


                for key, value in candidate.items():

                    if key != "Config_ID":
                        candidate_cfg[key] = value


                candidate_cfg[
                    "learning_rate_decay"
                ] = FIXED_DECAY


                if algorithm_key in [
                    "SGD_Linear",
                    "SGD_Huber",
                ]:

                    candidate_cfg[
                        "batch_size"
                    ] = FIXED_SGD_BATCH_SIZE


                # --------------------------------------------
                # Temporarily activate candidate configuration
                # --------------------------------------------

                ALGORITHM_CONFIGS[
                    algorithm_key
                ] = copy.deepcopy(
                    candidate_cfg
                )


                print(
                    f"Run "
                    f"{run_counter:3d}/"
                    f"{TOTAL_TRAINING_RUNS}: "
                    f"{task_key} | "
                    f"{algorithm_key} | "
                    f"{config_id}"
                )


                model_start = time.time()


                # ============================================
                # TRAIN
                # ============================================

                params, history = train_linear_model(
                    X_train,
                    y_train,
                    task_key,
                    algorithm_key,
                    epochs=TUNING_EPOCHS,
                    seed=deterministic_seed,
                )


                training_time = (
                    time.time()
                    - model_start
                )


                # ============================================
                # VALIDATION ONLY
                # ============================================

                val_prediction = predict_raw(
                    X_val,
                    params,
                    task_key,
                )


                metrics = reg_metrics(
                    y_val,
                    val_prediction,
                )


                # ============================================
                # Save model/configuration
                # ============================================

                model_key = (
                    task_key,
                    algorithm_key,
                    config_id,
                )


                CENTRALIZED_TUNING_MODELS[
                    model_key
                ] = {

                    "Parameters":
                        params.copy(),

                    "Algorithm_Key":
                        algorithm_key,

                    "Config_ID":
                        config_id,

                    "Config":
                        copy.deepcopy(
                            candidate_cfg
                        ),

                    "Seed":
                        deterministic_seed,

                    "Training_Time_Seconds":
                        training_time,
                }


                CENTRALIZED_TUNING_HISTORIES[
                    model_key
                ] = history


                tuning_rows.append(
                    {
                        "Task":
                            task_key,

                        "Target":
                            meta["Target"],

                        "Horizon":
                            meta["Horizon"],

                        "Algorithm_Key":
                            algorithm_key,

                        "Algorithm":
                            candidate_cfg["name"],

                        "Config_ID":
                            config_id,

                        "Deterministic_Seed":
                            deterministic_seed,

                        "Epochs":
                            TUNING_EPOCHS,

                        "Learning_Rate":
                            candidate_cfg[
                                "learning_rate"
                            ],

                        "Learning_Rate_Decay":
                            candidate_cfg[
                                "learning_rate_decay"
                            ],

                        "L1":
                            candidate_cfg["l1"],

                        "L2":
                            candidate_cfg["l2"],

                        "Huber_Delta":
                            candidate_cfg[
                                "huber_delta"
                            ],

                        "Batch_Size":
                            candidate_cfg[
                                "batch_size"
                            ],

                        "Validation_N":
                            metrics["N"],

                        "Validation_MAE":
                            metrics["MAE"],

                        "Validation_RMSE":
                            metrics["RMSE"],

                        "Validation_R2":
                            metrics["R2"],

                        "Validation_MBE":
                            metrics["MBE"],

                        "Validation_NRMSE_Std":
                            metrics[
                                "NRMSE_Std"
                            ],

                        "Validation_NRMSE_Range":
                            metrics[
                                "NRMSE_Range"
                            ],

                        "Training_Time_Seconds":
                            training_time,

                        "Test_Used":
                            False,
                    }
                )


finally:

    # ========================================================
    # Restore Cell 9 algorithm definitions
    # ========================================================

    ALGORITHM_CONFIGS.clear()

    ALGORITHM_CONFIGS.update(
        copy.deepcopy(
            ORIGINAL_ALGORITHM_CONFIGS
        )
    )


# ============================================================
# 10. Complete tuning results
# ============================================================

CENTRALIZED_TUNING_RESULTS_DF = pd.DataFrame(
    tuning_rows
)


# ============================================================
# 11. Rank configurations within each Task × Algorithm
#
# 1) MAE
# 2) RMSE
# 3) R2
# ============================================================

ranked_parts = []


for task_key in task_keys:

    for algorithm_key in algorithm_keys:

        part = (
            CENTRALIZED_TUNING_RESULTS_DF[
                (
                    CENTRALIZED_TUNING_RESULTS_DF[
                        "Task"
                    ].eq(task_key)
                )
                &
                (
                    CENTRALIZED_TUNING_RESULTS_DF[
                        "Algorithm_Key"
                    ].eq(algorithm_key)
                )
            ]
            .copy()
            .sort_values(
                [
                    "Validation_MAE",
                    "Validation_RMSE",
                    "Validation_R2",
                ],
                ascending=[
                    True,
                    True,
                    False,
                ],
            )
            .reset_index(drop=True)
        )


        part["Config_Rank"] = np.arange(
            1,
            len(part) + 1,
        )


        ranked_parts.append(
            part
        )


CENTRALIZED_TUNING_RESULTS_DF = pd.concat(
    ranked_parts,
    ignore_index=True,
)


# ============================================================
# 12. Best configuration for each Task × Algorithm
# ============================================================

CENTRALIZED_BEST_CONFIGS_DF = (
    CENTRALIZED_TUNING_RESULTS_DF[
        CENTRALIZED_TUNING_RESULTS_DF[
            "Config_Rank"
        ].eq(1)
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 13. Best configuration dictionary
# ============================================================

CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM = {}


for _, row in CENTRALIZED_BEST_CONFIGS_DF.iterrows():

    model_key = (
        row["Task"],
        row["Algorithm_Key"],
        row["Config_ID"],
    )

    selected_run = (
        CENTRALIZED_TUNING_MODELS[
            model_key
        ]
    )

    CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM[
        (
            row["Task"],
            row["Algorithm_Key"],
        )
    ] = copy.deepcopy(
        selected_run["Config"]
    )


# ============================================================
# 14. Downstream-compatible CENTRALIZED_MODELS
# ============================================================

for _, row in CENTRALIZED_BEST_CONFIGS_DF.iterrows():

    task_key = row[
        "Task"
    ]

    algorithm_key = row[
        "Algorithm_Key"
    ]

    config_id = row[
        "Config_ID"
    ]

    model_key = (
        task_key,
        algorithm_key,
        config_id,
    )

    selected_run = (
        CENTRALIZED_TUNING_MODELS[
            model_key
        ]
    )


    CENTRALIZED_MODELS[
        (
            task_key,
            algorithm_key,
        )
    ] = {

        "Parameters":
            selected_run[
                "Parameters"
            ].copy(),

        "Algorithm_Key":
            algorithm_key,

        "Config_ID":
            config_id,

        "Config":
            copy.deepcopy(
                selected_run[
                    "Config"
                ]
            ),

        "Seed":
            selected_run[
                "Seed"
            ],

        "Training_Time_Seconds":
            selected_run[
                "Training_Time_Seconds"
            ],
    }


    CENTRALIZED_HISTORIES[
        (
            task_key,
            algorithm_key,
        )
    ] = (
        CENTRALIZED_TUNING_HISTORIES[
            model_key
        ]
    )


# ============================================================
# 15. One tuned result per Task × Algorithm
# ============================================================

CENTRALIZED_RESULTS_DF = (
    CENTRALIZED_BEST_CONFIGS_DF
    .copy()
)


# ============================================================
# 16. Rank algorithms within each task
# ============================================================

algorithm_rank_parts = []


for task_key in task_keys:

    part = (
        CENTRALIZED_RESULTS_DF[
            CENTRALIZED_RESULTS_DF[
                "Task"
            ].eq(task_key)
        ]
        .copy()
        .sort_values(
            [
                "Validation_MAE",
                "Validation_RMSE",
                "Validation_R2",
            ],
            ascending=[
                True,
                True,
                False,
            ],
        )
        .reset_index(drop=True)
    )


    part["Validation_Rank"] = np.arange(
        1,
        len(part) + 1,
    )


    algorithm_rank_parts.append(
        part
    )


CENTRALIZED_RESULTS_DF = pd.concat(
    algorithm_rank_parts,
    ignore_index=True,
)


# ============================================================
# 17. Final centralized winner for each task
# ============================================================

CENTRALIZED_SELECTED_DF = (
    CENTRALIZED_RESULTS_DF[
        CENTRALIZED_RESULTS_DF[
            "Validation_Rank"
        ].eq(1)
    ]
    .copy()
    .sort_values(
        [
            "Target",
            "Horizon",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 18. Final selected algorithm/config dictionaries
# ============================================================

CENTRALIZED_SELECTED_ALGORITHM_BY_TASK = {}
CENTRALIZED_SELECTED_CONFIG_BY_TASK = {}


for _, row in CENTRALIZED_SELECTED_DF.iterrows():

    task_key = row[
        "Task"
    ]

    algorithm_key = row[
        "Algorithm_Key"
    ]


    CENTRALIZED_SELECTED_ALGORITHM_BY_TASK[
        task_key
    ] = algorithm_key


    CENTRALIZED_SELECTED_CONFIG_BY_TASK[
        task_key
    ] = copy.deepcopy(
        CENTRALIZED_MODELS[
            (
                task_key,
                algorithm_key,
            )
        ][
            "Config"
        ]
    )


# ============================================================
# 19. Validation predictions
#
# For the selected tuned configuration of every algorithm.
#
# Still NO TEST.
# ============================================================

station_rows = []
prediction_rows = []


for _, row in CENTRALIZED_RESULTS_DF.iterrows():

    task_key = row[
        "Task"
    ]

    algorithm_key = row[
        "Algorithm_Key"
    ]

    config_id = row[
        "Config_ID"
    ]


    meta = TASK_DATASETS[
        task_key
    ]["Meta"]


    dataset = TASK_DATASETS[
        task_key
    ]["Validation"]


    params = CENTRALIZED_MODELS[
        (
            task_key,
            algorithm_key,
        )
    ][
        "Parameters"
    ]


    prediction = predict_raw(
        dataset["X"],
        params,
        task_key,
    )


    frame = (
        dataset[
            "Frame"
        ]
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Row-level validation predictions
    # --------------------------------------------------------

    for i, (
        actual,
        predicted,
    ) in enumerate(
        zip(
            dataset["y"],
            prediction,
        )
    ):

        prediction_rows.append(
            {
                "Task":
                    task_key,

                "Target":
                    meta["Target"],

                "Horizon":
                    meta["Horizon"],

                "Algorithm_Key":
                    algorithm_key,

                "Algorithm":
                    row["Algorithm"],

                "Config_ID":
                    config_id,

                "Split":
                    "Validation",

                "Station":
                    frame.loc[
                        i,
                        "Station",
                    ],

                "Date":
                    frame.loc[
                        i,
                        "Date",
                    ],

                "Target_Date":
                    frame.loc[
                        i,
                        meta[
                            "Future_Date_Column"
                        ],
                    ],

                "Actual":
                    actual,

                "Prediction":
                    predicted,
            }
        )


    # --------------------------------------------------------
    # Station-specific Validation metrics
    # --------------------------------------------------------

    for station in EXPECTED_STATIONS:

        mask = (
            frame[
                "Station"
            ]
            .eq(station)
            .to_numpy()
        )


        if not np.any(mask):
            continue


        station_metrics = reg_metrics(
            dataset[
                "y"
            ][mask],
            prediction[
                mask
            ],
        )


        station_rows.append(
            {
                "Task":
                    task_key,

                "Target":
                    meta["Target"],

                "Horizon":
                    meta["Horizon"],

                "Algorithm_Key":
                    algorithm_key,

                "Algorithm":
                    row["Algorithm"],

                "Config_ID":
                    config_id,

                "Split":
                    "Validation",

                "Station":
                    station,

                **station_metrics,
            }
        )


CENTRALIZED_STATION_RESULTS_DF = pd.DataFrame(
    station_rows
)


CENTRALIZED_PREDICTIONS_DF = pd.DataFrame(
    prediction_rows
)


# ============================================================
# 20. Overall algorithm summary
# ============================================================

CENTRALIZED_ALGORITHM_SUMMARY_DF = (
    CENTRALIZED_RESULTS_DF
    .groupby(
        [
            "Algorithm_Key",
            "Algorithm",
        ],
        as_index=False,
    )
    .agg(

        Tasks=(
            "Task",
            "count",
        ),

        Mean_Validation_MAE=(
            "Validation_MAE",
            "mean",
        ),

        Mean_Validation_RMSE=(
            "Validation_RMSE",
            "mean",
        ),

        Mean_Validation_R2=(
            "Validation_R2",
            "mean",
        ),

        Mean_Validation_NRMSE_Std=(
            "Validation_NRMSE_Std",
            "mean",
        ),

        Mean_Validation_Rank=(
            "Validation_Rank",
            "mean",
        ),

        Task_Wins=(
            "Validation_Rank",
            lambda x:
                int(
                    np.sum(
                        np.asarray(x) == 1
                    )
                ),
        ),
    )
    .sort_values(
        [
            "Mean_Validation_Rank",
            "Mean_Validation_MAE",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)


# ============================================================
# 21. Boundary diagnostic
# ============================================================

boundary_rows = []


for _, row in CENTRALIZED_BEST_CONFIGS_DF.iterrows():

    algorithm_key = row[
        "Algorithm_Key"
    ]

    config_id = row[
        "Config_ID"
    ]


    config_ids = [
        candidate["Config_ID"]
        for candidate
        in CENTRAL_TUNING_GRID[
            algorithm_key
        ]
    ]


    boundary_rows.append(
        {
            "Task":
                row["Task"],

            "Algorithm_Key":
                algorithm_key,

            "Selected_Config":
                config_id,

            "Grid_First":
                config_ids[0],

            "Grid_Last":
                config_ids[-1],

            "Selected_On_Grid_Boundary":
                (
                    config_id == config_ids[0]
                    or
                    config_id == config_ids[-1]
                ),
        }
    )


CENTRALIZED_BOUNDARY_DIAGNOSTIC_DF = pd.DataFrame(
    boundary_rows
)


# ============================================================
# 22. Final protocol table
# ============================================================

CENTRALIZED_TUNING_PROTOCOL_DF = pd.DataFrame(
    [
        {
            "Item":
                "Algorithms",
            "Value":
                4,
        },

        {
            "Item":
                "Configurations per algorithm",
            "Value":
                5,
        },

        {
            "Item":
                "Configurations total",
            "Value":
                20,
        },

        {
            "Item":
                "Tasks",
            "Value":
                len(task_keys),
        },

        {
            "Item":
                "Total model fits",
            "Value":
                TOTAL_TRAINING_RUNS,
        },

        {
            "Item":
                "Selection data",
            "Value":
                "Validation only",
        },

        {
            "Item":
                "Primary criterion",
            "Value":
                "Validation MAE",
        },

        {
            "Item":
                "Tie-break 1",
            "Value":
                "Validation RMSE",
        },

        {
            "Item":
                "Tie-break 2",
            "Value":
                "Validation R2",
        },

        {
            "Item":
                "Epochs",
            "Value":
                TUNING_EPOCHS,
        },

        {
            "Item":
                "Epochs tuned",
            "Value":
                False,
        },

        {
            "Item":
                "Learning-rate decay",
            "Value":
                FIXED_DECAY,
        },

        {
            "Item":
                "Learning-rate decay tuned",
            "Value":
                False,
        },

        {
            "Item":
                "SGD batch size",
            "Value":
                FIXED_SGD_BATCH_SIZE,
        },

        {
            "Item":
                "Batch size tuned",
            "Value":
                False,
        },

        {
            "Item":
                "Test used",
            "Value":
                False,
        },
    ]
)


# ============================================================
# 23. FINAL SAFETY CHECKS
# ============================================================

if len(
    CENTRALIZED_TUNING_RESULTS_DF
) != TOTAL_TRAINING_RUNS:

    raise RuntimeError(
        "Unexpected number of tuning runs."
    )


if bool(
    CENTRALIZED_TUNING_RESULTS_DF[
        "Test_Used"
    ].any()
):

    raise RuntimeError(
        "Test data was used during hyperparameter tuning."
    )


expected_selected_rows = (
    len(task_keys)
    * len(algorithm_keys)
)


if len(
    CENTRALIZED_RESULTS_DF
) != expected_selected_rows:

    raise RuntimeError(
        "Unexpected number of "
        "Task × Algorithm selections."
    )


if len(
    CENTRALIZED_SELECTED_DF
) != len(task_keys):

    raise RuntimeError(
        "Exactly one final centralized winner "
        "must exist for every task."
    )


# ============================================================
# 24. OUTPUT
# ============================================================

print(
    "\n"
    + "=" * 125
)

print(
    "CELL 10 — FINAL CENTRALIZED TUNING COMPLETED"
)

print(
    "=" * 125
)


print(
    "\nFinal tuning protocol:"
)

display(
    CENTRALIZED_TUNING_PROTOCOL_DF
)


print(
    "\nFinal refined configurations:"
)

display(
    CENTRAL_TUNING_GRID_DF
    .round(6)
)


print(
    "\nBest configuration for every Task × Algorithm:"
)

display(
    CENTRALIZED_RESULTS_DF[
        [
            "Task",
            "Algorithm_Key",
            "Algorithm",
            "Config_ID",
            "Learning_Rate",
            "L1",
            "L2",
            "Huber_Delta",
            "Batch_Size",
            "Validation_MAE",
            "Validation_RMSE",
            "Validation_R2",
            "Validation_NRMSE_Std",
            "Validation_Rank",
        ]
    ]
    .sort_values(
        [
            "Task",
            "Validation_Rank",
        ]
    )
    .round(6)
)


print(
    "\nFINAL centralized winner for every task:"
)

display(
    CENTRALIZED_SELECTED_DF[
        [
            "Task",
            "Target",
            "Horizon",
            "Algorithm_Key",
            "Algorithm",
            "Config_ID",
            "Learning_Rate",
            "L1",
            "L2",
            "Huber_Delta",
            "Validation_MAE",
            "Validation_RMSE",
            "Validation_R2",
        ]
    ]
    .round(6)
)


print(
    "\nOverall centralized algorithm summary:"
)

display(
    CENTRALIZED_ALGORITHM_SUMMARY_DF
    .round(6)
)


print(
    "\nFinal boundary diagnostic:"
)

display(
    CENTRALIZED_BOUNDARY_DIAGNOSTIC_DF
)


print(
    "\nFINAL CELL 10 STATUS:"
)

print(
    "Hyperparameter tuning completed."
)

print(
    "Validation only was used for model/configuration selection."
)

print(
    "Test set was NOT evaluated or used."
)

print(
    "Centralized models/configurations are now frozen for downstream analysis."
)


print(
    f"\nCell 10 completed in "
    f"{time.time() - CELL_START:.2f} seconds"
)

In [ ]:
# ============================================================
# CELL 11 — FINAL LOCAL TRAINING
# ============================================================
#
# Purpose:
#   Train one independent model per station.
#
# Fairness rule:
#   For every Task × Algorithm, Local training uses EXACTLY
#   the validation-selected hyperparameter configuration
#   frozen in Cell 10.
#
# No additional hyperparameter tuning is performed here.
#
# Models:
#   6 Tasks × 4 Stations × 4 Algorithms = 96 local models
#
# Evaluation:
#   Validation ONLY
#
# Test is NOT evaluated or used in this cell.
# ============================================================

import time
import copy
import numpy as np
import pandas as pd

CELL_START = time.time()


# ============================================================
# 1. Safety checks
# ============================================================

required_objects = [
    "TASK_CLIENT_DATASETS",
    "TASK_DEFINITIONS",
    "EXPECTED_STATIONS",
    "ALGORITHM_CONFIGS",
    "CENTRAL_LOCAL_EPOCHS",
    "GLOBAL_RANDOM_SEED",
    "CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM",
    "CENTRALIZED_RESULTS_DF",
    "train_linear_model",
    "predict_raw",
    "reg_metrics",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


# ============================================================
# 2. Fixed Local protocol
# ============================================================

LOCAL_EPOCHS = CENTRAL_LOCAL_EPOCHS

task_keys = [
    definition["Task"]
    for definition in TASK_DEFINITIONS
]

algorithm_keys = list(
    ALGORITHM_CONFIGS.keys()
)


TOTAL_LOCAL_MODELS = (
    len(task_keys)
    * len(EXPECTED_STATIONS)
    * len(algorithm_keys)
)


# ============================================================
# 3. Containers
# ============================================================

LOCAL_MODELS = {}

LOCAL_HISTORIES = {}

station_rows = []
prediction_rows = []
pooled_rows = []

counter = 0


# ============================================================
# 4. Preserve Cell 9 algorithm configurations
# ============================================================

ORIGINAL_LOCAL_ALGORITHM_CONFIGS = copy.deepcopy(
    ALGORITHM_CONFIGS
)


# ============================================================
# 5. Train Local models
# ============================================================

try:

    for task_position, task_key in enumerate(
        task_keys,
        start=1,
    ):

        meta = TASK_CLIENT_DATASETS[
            task_key
        ][
            EXPECTED_STATIONS[0]
        ][
            "Train"
        ]


        for algorithm_position, algorithm_key in enumerate(
            algorithm_keys,
            start=1,
        ):

            # ------------------------------------------------
            # Use EXACTLY the Task × Algorithm configuration
            # selected by Cell 10.
            # ------------------------------------------------

            selected_config = copy.deepcopy(
                CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM[
                    (
                        task_key,
                        algorithm_key,
                    )
                ]
            )


            selected_row = (
                CENTRALIZED_RESULTS_DF[
                    (
                        CENTRALIZED_RESULTS_DF[
                            "Task"
                        ].eq(task_key)
                    )
                    &
                    (
                        CENTRALIZED_RESULTS_DF[
                            "Algorithm_Key"
                        ].eq(algorithm_key)
                    )
                ]
                .iloc[0]
            )


            selected_config_id = selected_row[
                "Config_ID"
            ]


            ALGORITHM_CONFIGS[
                algorithm_key
            ] = copy.deepcopy(
                selected_config
            )


            for station_position, station in enumerate(
                EXPECTED_STATIONS,
                start=1,
            ):

                counter += 1


                # --------------------------------------------
                # Deterministic reproducible seed
                # --------------------------------------------

                deterministic_seed = (
                    GLOBAL_RANDOM_SEED
                    + task_position * 10000
                    + station_position * 100
                    + algorithm_position
                )


                print(
                    f"Local model "
                    f"{counter:2d}/{TOTAL_LOCAL_MODELS}: "
                    f"{task_key} | "
                    f"{station} | "
                    f"{algorithm_key} | "
                    f"{selected_config_id}"
                )


                train_ds = (
                    TASK_CLIENT_DATASETS[
                        task_key
                    ][
                        station
                    ][
                        "Train"
                    ]
                )


                validation_ds = (
                    TASK_CLIENT_DATASETS[
                        task_key
                    ][
                        station
                    ][
                        "Validation"
                    ]
                )


                model_start = time.time()


                # ============================================
                # TRAIN
                # ============================================

                params, history = train_linear_model(
                    train_ds["X"],
                    train_ds["y"],
                    task_key,
                    algorithm_key,
                    epochs=LOCAL_EPOCHS,
                    seed=deterministic_seed,
                )


                training_time = (
                    time.time()
                    - model_start
                )


                # ============================================
                # VALIDATION ONLY
                # ============================================

                validation_prediction = predict_raw(
                    validation_ds["X"],
                    params,
                    task_key,
                )


                metrics = reg_metrics(
                    validation_ds["y"],
                    validation_prediction,
                )


                # ============================================
                # Save Local model
                # ============================================

                model_key = (
                    task_key,
                    station,
                    algorithm_key,
                )


                LOCAL_MODELS[
                    model_key
                ] = {

                    "Parameters":
                        params.copy(),

                    "Algorithm_Key":
                        algorithm_key,

                    "Config_ID":
                        selected_config_id,

                    "Config":
                        copy.deepcopy(
                            selected_config
                        ),

                    "Seed":
                        deterministic_seed,

                    "Training_Time_Seconds":
                        training_time,
                }


                LOCAL_HISTORIES[
                    model_key
                ] = history


                # ============================================
                # Station-level Validation result
                # ============================================

                station_rows.append(
                    {
                        "Task":
                            task_key,

                        "Target":
                            TASK_DATASETS[
                                task_key
                            ][
                                "Meta"
                            ][
                                "Target"
                            ],

                        "Horizon":
                            TASK_DATASETS[
                                task_key
                            ][
                                "Meta"
                            ][
                                "Horizon"
                            ],

                        "Station":
                            station,

                        "Algorithm_Key":
                            algorithm_key,

                        "Algorithm":
                            selected_config[
                                "name"
                            ],

                        "Config_ID":
                            selected_config_id,

                        "Deterministic_Seed":
                            deterministic_seed,

                        "Epochs":
                            LOCAL_EPOCHS,

                        "Learning_Rate":
                            selected_config[
                                "learning_rate"
                            ],

                        "Learning_Rate_Decay":
                            selected_config[
                                "learning_rate_decay"
                            ],

                        "L1":
                            selected_config[
                                "l1"
                            ],

                        "L2":
                            selected_config[
                                "l2"
                            ],

                        "Huber_Delta":
                            selected_config[
                                "huber_delta"
                            ],

                        "Batch_Size":
                            selected_config[
                                "batch_size"
                            ],

                        "Split":
                            "Validation",

                        "Training_Time_Seconds":
                            training_time,

                        **metrics,

                        "Test_Used":
                            False,
                    }
                )


                # ============================================
                # Row-level Validation predictions
                # ============================================

                frame = (
                    validation_ds[
                        "Frame"
                    ]
                    .reset_index(
                        drop=True
                    )
                )


                task_meta = TASK_DATASETS[
                    task_key
                ][
                    "Meta"
                ]


                for i, (
                    actual,
                    prediction,
                ) in enumerate(
                    zip(
                        validation_ds["y"],
                        validation_prediction,
                    )
                ):

                    prediction_rows.append(
                        {
                            "Task":
                                task_key,

                            "Target":
                                task_meta[
                                    "Target"
                                ],

                            "Horizon":
                                task_meta[
                                    "Horizon"
                                ],

                            "Station":
                                station,

                            "Algorithm_Key":
                                algorithm_key,

                            "Algorithm":
                                selected_config[
                                    "name"
                                ],

                            "Config_ID":
                                selected_config_id,

                            "Split":
                                "Validation",

                            "Date":
                                frame.loc[
                                    i,
                                    "Date",
                                ],

                            "Target_Date":
                                frame.loc[
                                    i,
                                    task_meta[
                                        "Future_Date_Column"
                                    ],
                                ],

                            "Actual":
                                actual,

                            "Prediction":
                                prediction,
                        }
                    )


finally:

    # ========================================================
    # Restore Cell 9 algorithm definitions
    # ========================================================

    ALGORITHM_CONFIGS.clear()

    ALGORITHM_CONFIGS.update(
        copy.deepcopy(
            ORIGINAL_LOCAL_ALGORITHM_CONFIGS
        )
    )


# ============================================================
# 6. DataFrames
# ============================================================

LOCAL_STATION_RESULTS_DF = pd.DataFrame(
    station_rows
)

LOCAL_PREDICTIONS_DF = pd.DataFrame(
    prediction_rows
)


# ============================================================
# 7. Pooled Validation performance
#
# Pool predictions from the four independent station models
# for each Task × Algorithm.
# ============================================================

for task_key in task_keys:

    task_meta = TASK_DATASETS[
        task_key
    ][
        "Meta"
    ]


    for algorithm_key in algorithm_keys:

        matching = (
            LOCAL_PREDICTIONS_DF[
                (
                    LOCAL_PREDICTIONS_DF[
                        "Task"
                    ].eq(task_key)
                )
                &
                (
                    LOCAL_PREDICTIONS_DF[
                        "Algorithm_Key"
                    ].eq(algorithm_key)
                )
            ]
        )


        if matching.empty:
            raise RuntimeError(
                f"No Local Validation predictions for "
                f"{task_key} / {algorithm_key}"
            )


        metrics = reg_metrics(
            matching[
                "Actual"
            ].to_numpy(),

            matching[
                "Prediction"
            ].to_numpy(),
        )


        selected_row = (
            CENTRALIZED_RESULTS_DF[
                (
                    CENTRALIZED_RESULTS_DF[
                        "Task"
                    ].eq(task_key)
                )
                &
                (
                    CENTRALIZED_RESULTS_DF[
                        "Algorithm_Key"
                    ].eq(algorithm_key)
                )
            ]
            .iloc[0]
        )


        pooled_rows.append(
            {
                "Task":
                    task_key,

                "Target":
                    task_meta[
                        "Target"
                    ],

                "Horizon":
                    task_meta[
                        "Horizon"
                    ],

                "Algorithm_Key":
                    algorithm_key,

                "Algorithm":
                    selected_row[
                        "Algorithm"
                    ],

                "Config_ID":
                    selected_row[
                        "Config_ID"
                    ],

                "Validation_N":
                    metrics[
                        "N"
                    ],

                "Validation_MAE":
                    metrics[
                        "MAE"
                    ],

                "Validation_RMSE":
                    metrics[
                        "RMSE"
                    ],

                "Validation_R2":
                    metrics[
                        "R2"
                    ],

                "Validation_MBE":
                    metrics[
                        "MBE"
                    ],

                "Validation_NRMSE_Std":
                    metrics[
                        "NRMSE_Std"
                    ],

                "Validation_NRMSE_Range":
                    metrics[
                        "NRMSE_Range"
                    ],

                "Test_Used":
                    False,
            }
        )


LOCAL_RESULTS_DF = pd.DataFrame(
    pooled_rows
)


# ============================================================
# 8. Rank algorithms within each task
#
# Same selection rule as Cell 10:
#
# 1) Minimum Validation MAE
# 2) Minimum Validation RMSE
# 3) Maximum Validation R2
# ============================================================

ranked_parts = []


for task_key in task_keys:

    part = (
        LOCAL_RESULTS_DF[
            LOCAL_RESULTS_DF[
                "Task"
            ].eq(task_key)
        ]
        .copy()
        .sort_values(
            [
                "Validation_MAE",
                "Validation_RMSE",
                "Validation_R2",
            ],
            ascending=[
                True,
                True,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


    part[
        "Validation_Rank"
    ] = np.arange(
        1,
        len(part) + 1,
    )


    ranked_parts.append(
        part
    )


LOCAL_RESULTS_DF = pd.concat(
    ranked_parts,
    ignore_index=True,
)


# ============================================================
# 9. Best Local algorithm per task
# ============================================================

LOCAL_SELECTED_DF = (
    LOCAL_RESULTS_DF[
        LOCAL_RESULTS_DF[
            "Validation_Rank"
        ].eq(1)
    ]
    .copy()
    .sort_values(
        [
            "Target",
            "Horizon",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 10. Overall Local algorithm summary
# ============================================================

LOCAL_ALGORITHM_SUMMARY_DF = (
    LOCAL_RESULTS_DF
    .groupby(
        [
            "Algorithm_Key",
            "Algorithm",
        ],
        as_index=False,
    )
    .agg(

        Tasks=(
            "Task",
            "count",
        ),

        Mean_Validation_MAE=(
            "Validation_MAE",
            "mean",
        ),

        Mean_Validation_RMSE=(
            "Validation_RMSE",
            "mean",
        ),

        Mean_Validation_R2=(
            "Validation_R2",
            "mean",
        ),

        Mean_Validation_NRMSE_Std=(
            "Validation_NRMSE_Std",
            "mean",
        ),

        Mean_Validation_Rank=(
            "Validation_Rank",
            "mean",
        ),

        Task_Wins=(
            "Validation_Rank",
            lambda x:
                int(
                    np.sum(
                        np.asarray(x)
                        == 1
                    )
                ),
        ),
    )
    .sort_values(
        [
            "Mean_Validation_Rank",
            "Mean_Validation_MAE",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 11. Protocol table
# ============================================================

LOCAL_PROTOCOL_DF = pd.DataFrame(
    [
        {
            "Item":
                "Tasks",

            "Value":
                len(task_keys),
        },

        {
            "Item":
                "Stations",

            "Value":
                len(EXPECTED_STATIONS),
        },

        {
            "Item":
                "Algorithms",

            "Value":
                len(algorithm_keys),
        },

        {
            "Item":
                "Total local models",

            "Value":
                TOTAL_LOCAL_MODELS,
        },

        {
            "Item":
                "Hyperparameters",

            "Value":
                "Frozen from Cell 10",
        },

        {
            "Item":
                "Additional tuning",

            "Value":
                False,
        },

        {
            "Item":
                "Epochs",

            "Value":
                LOCAL_EPOCHS,
        },

        {
            "Item":
                "Selection data",

            "Value":
                "Validation only",
        },

        {
            "Item":
                "Primary ranking criterion",

            "Value":
                "Validation MAE",
        },

        {
            "Item":
                "Tie-break 1",

            "Value":
                "Validation RMSE",
        },

        {
            "Item":
                "Tie-break 2",

            "Value":
                "Validation R2",
        },

        {
            "Item":
                "Test used",

            "Value":
                False,
        },
    ]
)


# ============================================================
# 12. Safety checks
# ============================================================

if len(
    LOCAL_MODELS
) != TOTAL_LOCAL_MODELS:

    raise RuntimeError(
        "Unexpected number of Local models."
    )


if bool(
    LOCAL_STATION_RESULTS_DF[
        "Test_Used"
    ].any()
):

    raise RuntimeError(
        "Test data was used in Cell 11."
    )


if bool(
    LOCAL_RESULTS_DF[
        "Test_Used"
    ].any()
):

    raise RuntimeError(
        "Test data was used in pooled Local evaluation."
    )


if len(
    LOCAL_SELECTED_DF
) != len(task_keys):

    raise RuntimeError(
        "Exactly one Local winner must exist per task."
    )


# ============================================================
# 13. Output
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "CELL 11 — FINAL LOCAL TRAINING COMPLETED"
)

print(
    "=" * 120
)


print(
    "\nLocal protocol:"
)

display(
    LOCAL_PROTOCOL_DF
)


print(
    "\nPooled Local Validation ranking:"
)

display(
    LOCAL_RESULTS_DF[
        [
            "Task",
            "Algorithm_Key",
            "Algorithm",
            "Config_ID",
            "Validation_MAE",
            "Validation_RMSE",
            "Validation_R2",
            "Validation_NRMSE_Std",
            "Validation_Rank",
        ]
    ]
    .sort_values(
        [
            "Task",
            "Validation_Rank",
        ]
    )
    .round(6)
)


print(
    "\nBest Local algorithm for each task:"
)

display(
    LOCAL_SELECTED_DF[
        [
            "Task",
            "Target",
            "Horizon",
            "Algorithm_Key",
            "Algorithm",
            "Config_ID",
            "Validation_MAE",
            "Validation_RMSE",
            "Validation_R2",
        ]
    ]
    .round(6)
)


print(
    "\nOverall Local algorithm summary:"
)

display(
    LOCAL_ALGORITHM_SUMMARY_DF
    .round(6)
)


print(
    "\nCELL 11 STATUS:"
)

print(
    "96 station-specific Local models trained."
)

print(
    "Hyperparameters were frozen from Cell 10."
)

print(
    "No additional Local hyperparameter tuning was performed."
)

print(
    "Validation only was evaluated."
)

print(
    "Test set was NOT evaluated or used."
)


print(
    f"\nCell 11 completed in "
    f"{time.time() - CELL_START:.2f} seconds"
)

In [ ]:
# ============================================================
# CELL 12 — FINAL FEDERATED PROTOCOL DEFINITION
# ============================================================
#
# Purpose:
#   Define the four federated optimization methods used in
#   Paper 3:
#
#       1. FedAvg
#       2. FedProx
#       3. SCAFFOLD
#       4. FedAdam
#
# IMPORTANT:
#
# - No model training is performed in this cell.
# - No Validation or Test data are evaluated here.
# - Task × Algorithm hyperparameters selected in Cell 10
#   remain frozen for later federated training.
#
# Fair training-budget rule:
#
#       60 rounds × 2 local epochs = 120 local passes
#
# which matches the 120-epoch centralized/local protocol.
#
# ============================================================

import time
import copy
import numpy as np
import pandas as pd

CELL_START = time.time()


# ============================================================
# 1. Safety checks
# ============================================================

required_objects = [
    "TASK_DEFINITIONS",
    "EXPECTED_STATIONS",
    "ALGORITHM_CONFIGS",
    "CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM",
    "CENTRALIZED_RESULTS_DF",
    "GLOBAL_RANDOM_SEED",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


# ============================================================
# 2. FINAL FEDERATED TRAINING BUDGET
# ============================================================
#
# FIXED — NOT tuned in this stage.
#
# 60 rounds × 2 local epochs = 120 local optimization passes.
#
# This preserves a comparable nominal optimization budget with:
#
#   Centralized = 120 epochs
#   Local       = 120 epochs
#   Federated   = 60 × 2 = 120 local passes
#
# ============================================================

FEDERATED_ROUNDS = 60

FEDERATED_LOCAL_EPOCHS = 2

FEDERATED_CLIENT_FRACTION = 1.0

FEDERATED_TOTAL_LOCAL_PASSES = (
    FEDERATED_ROUNDS
    * FEDERATED_LOCAL_EPOCHS
)


# ============================================================
# 3. METHOD-SPECIFIC PARAMETERS
# ============================================================
#
# These values are theoretically tunable.
#
# HOWEVER:
# They are intentionally FIXED in the initial federated
# comparison to avoid an unnecessarily large tuning search.
#
# If later validation evidence demonstrates that refinement
# is scientifically justified, tuning can be performed using
# Validation only.
#
# TEST must never be used for that purpose.
# ============================================================


# ------------------------------------------------------------
# FedProx
# ------------------------------------------------------------
#
# Potential tuning parameter:
#   mu
#
# Current fixed value:
#   0.01
# ------------------------------------------------------------

FEDPROX_MU = 0.01


# ------------------------------------------------------------
# FedAdam
# ------------------------------------------------------------
#
# Potential tuning parameters:
#   server learning rate
#   beta1
#   beta2
#   tau
#
# Current fixed protocol values:
# ------------------------------------------------------------

FEDADAM_SERVER_LR = 0.01

FEDADAM_BETA1 = 0.90

FEDADAM_BETA2 = 0.99

FEDADAM_TAU = 1e-3


# ============================================================
# 4. Federated methods
# ============================================================

FEDERATED_METHOD_NAMES = [
    "FedAvg",
    "FedProx",
    "SCAFFOLD",
    "FedAdam",
]


# ============================================================
# 5. Detailed method configurations
# ============================================================

FEDERATED_METHOD_CONFIGS = {

    # ========================================================
    # FedAvg
    # ========================================================

    "FedAvg": {

        "Local_Modification":
            "None",

        "Server_Update":
            "Sample-weighted averaging",

        "Proximal_Mu":
            0.0,

        "Server_Learning_Rate":
            np.nan,

        "Beta1":
            np.nan,

        "Beta2":
            np.nan,

        "Tau":
            np.nan,

        "Control_Variates":
            False,

        "Method_Specific_Tuning_Performed":
            False,
    },


    # ========================================================
    # FedProx
    # ========================================================

    "FedProx": {

        "Local_Modification":
            "Proximal penalty",

        "Server_Update":
            "Sample-weighted averaging",

        "Proximal_Mu":
            FEDPROX_MU,

        "Server_Learning_Rate":
            np.nan,

        "Beta1":
            np.nan,

        "Beta2":
            np.nan,

        "Tau":
            np.nan,

        "Control_Variates":
            False,

        "Method_Specific_Tuning_Performed":
            False,
    },


    # ========================================================
    # SCAFFOLD
    # ========================================================

    "SCAFFOLD": {

        "Local_Modification":
            "Control-variate correction",

        "Server_Update":
            "Sample-weighted averaging",

        "Proximal_Mu":
            0.0,

        "Server_Learning_Rate":
            np.nan,

        "Beta1":
            np.nan,

        "Beta2":
            np.nan,

        "Tau":
            np.nan,

        "Control_Variates":
            True,

        "Method_Specific_Tuning_Performed":
            False,
    },


    # ========================================================
    # FedAdam
    # ========================================================

    "FedAdam": {

        "Local_Modification":
            "None",

        "Server_Update":
            "Adam on aggregated client displacement",

        "Proximal_Mu":
            0.0,

        "Server_Learning_Rate":
            FEDADAM_SERVER_LR,

        "Beta1":
            FEDADAM_BETA1,

        "Beta2":
            FEDADAM_BETA2,

        "Tau":
            FEDADAM_TAU,

        "Control_Variates":
            False,

        "Method_Specific_Tuning_Performed":
            False,
    },
}


# ============================================================
# 6. Final method configuration table
# ============================================================

FEDERATED_METHOD_CONFIGURATION_DF = pd.DataFrame(
    [
        {
            "Federated_Method":
                method_name,

            **config,

            "Rounds":
                FEDERATED_ROUNDS,

            "Local_Epochs":
                FEDERATED_LOCAL_EPOCHS,

            "Total_Local_Passes":
                FEDERATED_TOTAL_LOCAL_PASSES,

            "Client_Fraction":
                FEDERATED_CLIENT_FRACTION,

            "Number_of_Clients":
                len(EXPECTED_STATIONS),

            "Algorithm_Hyperparameters":
                "Frozen from Cell 10",

            "Selection_Data":
                "None in Cell 12",

            "Test_Used":
                False,
        }

        for method_name, config
        in FEDERATED_METHOD_CONFIGS.items()
    ]
)


# ============================================================
# 7. Task × Algorithm frozen configuration helper
#
# Cell 13 MUST use this instead of the original untuned
# ALGORITHM_CONFIGS parameters from Cell 9.
# ============================================================

def get_federated_task_algorithm_config(
    task_key,
    algorithm_key,
):

    key = (
        task_key,
        algorithm_key,
    )

    if (
        key
        not in
        CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM
    ):
        raise KeyError(
            f"No frozen Cell 10 configuration "
            f"for {task_key} / {algorithm_key}"
        )

    return copy.deepcopy(
        CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM[
            key
        ]
    )


# ============================================================
# 8. Frozen Task × Algorithm configuration table
# ============================================================

frozen_rows = []


for task_definition in TASK_DEFINITIONS:

    task_key = task_definition[
        "Task"
    ]


    for algorithm_key in ALGORITHM_CONFIGS.keys():

        cfg = get_federated_task_algorithm_config(
            task_key,
            algorithm_key,
        )


        selected_row = (
            CENTRALIZED_RESULTS_DF[
                (
                    CENTRALIZED_RESULTS_DF[
                        "Task"
                    ].eq(task_key)
                )
                &
                (
                    CENTRALIZED_RESULTS_DF[
                        "Algorithm_Key"
                    ].eq(algorithm_key)
                )
            ]
            .iloc[0]
        )


        frozen_rows.append(
            {
                "Task":
                    task_key,

                "Algorithm_Key":
                    algorithm_key,

                "Algorithm":
                    cfg["name"],

                "Config_ID":
                    selected_row[
                        "Config_ID"
                    ],

                "Learning_Rate":
                    cfg[
                        "learning_rate"
                    ],

                "Learning_Rate_Decay":
                    cfg[
                        "learning_rate_decay"
                    ],

                "L1":
                    cfg["l1"],

                "L2":
                    cfg["l2"],

                "Huber_Delta":
                    cfg[
                        "huber_delta"
                    ],

                "Batch_Size":
                    cfg[
                        "batch_size"
                    ],

                "Source":
                    "Cell 10 Validation-selected",
            }
        )


FEDERATED_FROZEN_ALGORITHM_CONFIGS_DF = (
    pd.DataFrame(
        frozen_rows
    )
)


# ============================================================
# 9. Parameter classification
#
# Explicitly records what is tunable and what is fixed.
# ============================================================

FEDERATED_PARAMETER_POLICY_DF = pd.DataFrame(
    [
        {
            "Parameter":
                "Federated rounds",

            "Current_Value":
                FEDERATED_ROUNDS,

            "Potentially_Tunable":
                True,

            "Tuned_Now":
                False,

            "Reason":
                (
                    "Fixed at 60 so that "
                    "60 rounds × 2 local epochs "
                    "= 120 local passes."
                ),
        },

        {
            "Parameter":
                "Local epochs per round",

            "Current_Value":
                FEDERATED_LOCAL_EPOCHS,

            "Potentially_Tunable":
                True,

            "Tuned_Now":
                False,

            "Reason":
                (
                    "Fixed at 2 for the common "
                    "120-pass training budget."
                ),
        },

        {
            "Parameter":
                "Client fraction",

            "Current_Value":
                FEDERATED_CLIENT_FRACTION,

            "Potentially_Tunable":
                False,

            "Tuned_Now":
                False,

            "Reason":
                (
                    "All four WTP stations participate; "
                    "partial participation is unnecessary "
                    "for the primary experiment."
                ),
        },

        {
            "Parameter":
                "FedProx mu",

            "Current_Value":
                FEDPROX_MU,

            "Potentially_Tunable":
                True,

            "Tuned_Now":
                False,

            "Reason":
                (
                    "Method-specific parameter; "
                    "held fixed for initial comparison."
                ),
        },

        {
            "Parameter":
                "FedAdam server learning rate",

            "Current_Value":
                FEDADAM_SERVER_LR,

            "Potentially_Tunable":
                True,

            "Tuned_Now":
                False,

            "Reason":
                (
                    "Method-specific parameter; "
                    "held fixed for initial comparison."
                ),
        },

        {
            "Parameter":
                "FedAdam beta1",

            "Current_Value":
                FEDADAM_BETA1,

            "Potentially_Tunable":
                True,

            "Tuned_Now":
                False,

            "Reason":
                "Held fixed for initial comparison.",
        },

        {
            "Parameter":
                "FedAdam beta2",

            "Current_Value":
                FEDADAM_BETA2,

            "Potentially_Tunable":
                True,

            "Tuned_Now":
                False,

            "Reason":
                "Held fixed for initial comparison.",
        },

        {
            "Parameter":
                "FedAdam tau",

            "Current_Value":
                FEDADAM_TAU,

            "Potentially_Tunable":
                True,

            "Tuned_Now":
                False,

            "Reason":
                "Held fixed for initial comparison.",
        },

        {
            "Parameter":
                "Task × Algorithm hyperparameters",

            "Current_Value":
                "Cell 10 selected values",

            "Potentially_Tunable":
                False,

            "Tuned_Now":
                False,

            "Reason":
                (
                    "Already selected using Validation "
                    "in Cell 10 and now frozen."
                ),
        },
    ]
)


# ============================================================
# 10. Expected protocol checks
# ============================================================

EXPECTED_FEDERATED_METHODS = [
    "FedAvg",
    "FedProx",
    "SCAFFOLD",
    "FedAdam",
]


if (
    FEDERATED_METHOD_NAMES
    != EXPECTED_FEDERATED_METHODS
):
    raise RuntimeError(
        "Federated method list does not "
        "match the accepted Paper 3 protocol."
    )


if FEDERATED_ROUNDS != 60:
    raise RuntimeError(
        "Final protocol requires 60 federated rounds."
    )


if FEDERATED_LOCAL_EPOCHS != 2:
    raise RuntimeError(
        "Final protocol requires 2 local epochs per round."
    )


if (
    FEDERATED_TOTAL_LOCAL_PASSES
    != CENTRAL_LOCAL_EPOCHS
):
    raise RuntimeError(
        "Federated local-pass budget does not match "
        "the centralized/local 120-epoch protocol."
    )


if FEDERATED_CLIENT_FRACTION != 1.0:
    raise RuntimeError(
        "Primary Paper 3 experiment requires "
        "full participation of all four stations."
    )


if len(
    FEDERATED_FROZEN_ALGORITHM_CONFIGS_DF
) != (
    len(TASK_DEFINITIONS)
    * len(ALGORITHM_CONFIGS)
):
    raise RuntimeError(
        "Incomplete frozen Task × Algorithm configuration table."
    )


if bool(
    FEDERATED_METHOD_CONFIGURATION_DF[
        "Test_Used"
    ].any()
):
    raise RuntimeError(
        "Test cannot be used in Cell 12."
    )


# ============================================================
# 11. Planned experiment count
# ============================================================

TOTAL_FEDERATED_COMBINATIONS = (
    len(TASK_DEFINITIONS)
    * len(ALGORITHM_CONFIGS)
    * len(FEDERATED_METHOD_NAMES)
)


# ============================================================
# 12. Output
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "CELL 12 — FINAL FEDERATED PROTOCOL DEFINED"
)

print(
    "=" * 120
)


print(
    "\nFederated method configurations:"
)

display(
    FEDERATED_METHOD_CONFIGURATION_DF
)


print(
    "\nParameter tuning/fixation policy:"
)

display(
    FEDERATED_PARAMETER_POLICY_DF
)


print(
    "\nFrozen Task × Algorithm configurations from Cell 10:"
)

display(
    FEDERATED_FROZEN_ALGORITHM_CONFIGS_DF
    .round(6)
)


print(
    "\nPlanned federated experiments:"
)

print(
    f"- Prediction tasks       : "
    f"{len(TASK_DEFINITIONS)}"
)

print(
    f"- Algorithms             : "
    f"{len(ALGORITHM_CONFIGS)}"
)

print(
    f"- Federated methods      : "
    f"{len(FEDERATED_METHOD_NAMES)}"
)

print(
    f"- Total combinations     : "
    f"{TOTAL_FEDERATED_COMBINATIONS}"
)

print(
    f"- Rounds/combination     : "
    f"{FEDERATED_ROUNDS}"
)

print(
    f"- Local epochs/round     : "
    f"{FEDERATED_LOCAL_EPOCHS}"
)

print(
    f"- Total local passes     : "
    f"{FEDERATED_TOTAL_LOCAL_PASSES}"
)

print(
    f"- Participating stations : "
    f"{len(EXPECTED_STATIONS)} / "
    f"{len(EXPECTED_STATIONS)}"
)


print(
    "\nCELL 12 STATUS:"
)

print(
    "Four federated methods defined."
)

print(
    "Task × Algorithm hyperparameters are frozen from Cell 10."
)

print(
    "No federated method was selected in this cell."
)

print(
    "No additional hyperparameter tuning was performed."
)

print(
    "Validation and Test were NOT evaluated."
)


print(
    f"\nCell 12 completed in "
    f"{time.time() - CELL_START:.2f} seconds"
)

In [ ]:
# ============================================================
# CELL 13A — FINAL FEDERATED TRAINING AND VALIDATION COMPARISON
# ============================================================
#
# 6 Tasks × 4 Algorithms × 4 Federated Methods = 96 runs
#
# Federated methods:
#   1. FedAvg
#   2. FedProx
#   3. SCAFFOLD
#   4. FedAdam
#
# IMPORTANT:
#
# - Task × Algorithm hyperparameters are FROZEN from Cell 10.
# - Federated-method parameters are FROZEN from Cell 12.
# - Validation is used for comparison.
# - Test is NEVER evaluated in this cell.
# - No additional tuning is performed.
#
# ============================================================

import os
import time
import copy
import pickle
import numpy as np
import pandas as pd

CELL_START = time.time()


# ============================================================
# 1. Safety checks
# ============================================================

required_objects = [
    "TASK_DATASETS",
    "TASK_CLIENT_DATASETS",
    "TASK_DEFINITIONS",
    "TASK_SCALERS",
    "EXPECTED_STATIONS",
    "ACTIVE_FEATURE_COLUMNS",
    "CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM",
    "CENTRALIZED_RESULTS_DF",
    "FEDERATED_METHOD_NAMES",
    "FEDERATED_METHOD_CONFIGS",
    "FEDERATED_ROUNDS",
    "FEDERATED_LOCAL_EPOCHS",
    "FEDERATED_CLIENT_FRACTION",
    "FEDPROX_MU",
    "FEDADAM_SERVER_LR",
    "FEDADAM_BETA1",
    "FEDADAM_BETA2",
    "FEDADAM_TAU",
    "GLOBAL_RANDOM_SEED",
    "GRADIENT_CLIP_NORM",
    "EPS",
    "batch_gradient",
    "clip_grad",
    "predict_raw",
    "reg_metrics",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


# ============================================================
# 2. Protocol checks
# ============================================================

EXPECTED_FEDERATED_METHODS = [
    "FedAvg",
    "FedProx",
    "SCAFFOLD",
    "FedAdam",
]

if FEDERATED_METHOD_NAMES != EXPECTED_FEDERATED_METHODS:
    raise RuntimeError(
        "Unexpected federated-method list."
    )

if FEDERATED_ROUNDS != 60:
    raise RuntimeError(
        "Cell 13 requires 60 federated rounds."
    )

if FEDERATED_LOCAL_EPOCHS != 2:
    raise RuntimeError(
        "Cell 13 requires 2 local epochs per round."
    )

if FEDERATED_CLIENT_FRACTION != 1.0:
    raise RuntimeError(
        "Primary experiment requires all four stations."
    )


# ============================================================
# 3. Clean checkpoint
#
# Final protocol run starts from zero.
# ============================================================

CHECKPOINT_PATH = (
    "/content/"
    "paper3_final_federated_cell13_checkpoint.pkl"
    if os.path.isdir("/content")
    else
    "paper3_final_federated_cell13_checkpoint.pkl"
)

FORCE_CLEAN_FEDERATED_RUN = True

if (
    FORCE_CLEAN_FEDERATED_RUN
    and os.path.isfile(CHECKPOINT_PATH)
):
    os.remove(CHECKPOINT_PATH)


# ============================================================
# 4. Helper: frozen Cell 10 configuration
# ============================================================

def get_cell10_frozen_config(
    task_key,
    algorithm_key,
):

    key = (
        task_key,
        algorithm_key,
    )

    if (
        key
        not in
        CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM
    ):
        raise KeyError(
            f"Missing Cell 10 frozen configuration: "
            f"{task_key} / {algorithm_key}"
        )

    return copy.deepcopy(
        CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM[
            key
        ]
    )


# ============================================================
# 5. Federated client update
#
# IMPORTANT:
# The algorithm configuration is passed explicitly.
# Therefore this function cannot accidentally use old Cell 9
# hyperparameters.
# ============================================================

def federated_client_update(
    X,
    y_raw,
    task_key,
    method_key,
    algorithm_config,
    global_parameters,
    local_epochs,
    round_index,
    client_control=None,
    server_control=None,
    seed=GLOBAL_RANDOM_SEED,
):

    config = algorithm_config

    scaler = TASK_SCALERS[
        task_key
    ]

    X = np.asarray(
        X,
        dtype=float,
    )

    y_raw = np.asarray(
        y_raw,
        dtype=float,
    )

    y = (
        y_raw
        - scaler["y_mean"]
    ) / scaler["y_std"]


    local_parameters = np.asarray(
        global_parameters,
        dtype=float,
    ).copy()


    number_rows = len(y)

    if number_rows == 0:
        raise RuntimeError(
            f"Empty client training set for {task_key}"
        )


    batch_size = (
        number_rows
        if config["batch_size"] is None
        else min(
            int(config["batch_size"]),
            number_rows,
        )
    )


    rng = np.random.default_rng(
        seed
    )


    total_learning_rate = 0.0


    # ========================================================
    # Local optimization
    # ========================================================

    for local_epoch in range(
        local_epochs
    ):

        global_pass_index = (
            round_index
            * local_epochs
            + local_epoch
        )


        learning_rate = (
            config["learning_rate"]
            * (
                config[
                    "learning_rate_decay"
                ]
                ** global_pass_index
            )
        )


        order = (
            np.arange(number_rows)
            if batch_size == number_rows
            else rng.permutation(
                number_rows
            )
        )


        for start in range(
            0,
            number_rows,
            batch_size,
        ):

            batch_indices = order[
                start:
                start + batch_size
            ]


            gradient = batch_gradient(
                X[batch_indices],
                y[batch_indices],
                local_parameters,
                config,
            )


            # ------------------------------------------------
            # FedProx proximal regularization
            # ------------------------------------------------

            if method_key == "FedProx":

                gradient = (
                    gradient
                    + FEDPROX_MU
                    * (
                        local_parameters
                        - global_parameters
                    )
                )


            # ------------------------------------------------
            # SCAFFOLD control-variate correction
            # ------------------------------------------------

            if method_key == "SCAFFOLD":

                gradient = (
                    gradient
                    - client_control
                    + server_control
                )


            # ------------------------------------------------
            # Gradient clipping
            # ------------------------------------------------

            gradient = clip_grad(
                gradient,
                GRADIENT_CLIP_NORM,
            )


            # ------------------------------------------------
            # Gradient update
            # ------------------------------------------------

            local_parameters -= (
                learning_rate
                * gradient
            )


            # ------------------------------------------------
            # L1 proximal step, if applicable
            # ------------------------------------------------

            if config["l1"] > 0:

                weights = (
                    local_parameters[:-1]
                )

                local_parameters[:-1] = (
                    np.sign(weights)
                    * np.maximum(
                        np.abs(weights)
                        - learning_rate
                        * config["l1"],
                        0.0,
                    )
                )


            total_learning_rate += (
                learning_rate
            )


    # ========================================================
    # SCAFFOLD client-control update
    # ========================================================

    updated_client_control = (
        client_control
    )


    if method_key == "SCAFFOLD":

        updated_client_control = (
            client_control
            - server_control
            + (
                global_parameters
                - local_parameters
            )
            / max(
                total_learning_rate,
                EPS,
            )
        )


    return (
        local_parameters,
        updated_client_control,
    )


# ============================================================
# 6. Train one federated combination
# ============================================================

def train_federated_method(
    task_key,
    algorithm_key,
    method_key,
    task_position,
    algorithm_position,
    method_position,
    rounds=FEDERATED_ROUNDS,
    local_epochs=FEDERATED_LOCAL_EPOCHS,
):

    start_time = time.time()


    # --------------------------------------------------------
    # EXACT configuration selected in Cell 10
    # --------------------------------------------------------

    algorithm_config = (
        get_cell10_frozen_config(
            task_key,
            algorithm_key,
        )
    )


    parameter_count = (
        len(ACTIVE_FEATURE_COLUMNS)
        + 1
    )


    # --------------------------------------------------------
    # Initial global parameters
    # --------------------------------------------------------

    global_parameters = np.zeros(
        parameter_count,
        dtype=float,
    )


    # --------------------------------------------------------
    # FedAdam server states
    # --------------------------------------------------------

    fedadam_m = np.zeros(
        parameter_count,
        dtype=float,
    )

    fedadam_v = np.zeros(
        parameter_count,
        dtype=float,
    )


    # --------------------------------------------------------
    # SCAFFOLD controls
    # --------------------------------------------------------

    client_controls = {
        station:
            np.zeros(
                parameter_count,
                dtype=float,
            )
        for station
        in EXPECTED_STATIONS
    }


    server_control = np.zeros(
        parameter_count,
        dtype=float,
    )


    # --------------------------------------------------------
    # Sample-weighted aggregation sizes
    # --------------------------------------------------------

    client_sizes = {
        station:
            len(
                TASK_CLIENT_DATASETS[
                    task_key
                ][station]["Train"]["y"]
            )
        for station
        in EXPECTED_STATIONS
    }


    total_client_rows = sum(
        client_sizes.values()
    )


    if total_client_rows <= 0:
        raise RuntimeError(
            f"No federated training rows for {task_key}"
        )


    history = []


    # ========================================================
    # Federated rounds
    # ========================================================

    for round_index in range(
        rounds
    ):

        local_models = []

        updated_controls = {}


        # ----------------------------------------------------
        # Full client participation: all four stations
        # ----------------------------------------------------

        for station_position, station in enumerate(
            EXPECTED_STATIONS,
            start=1,
        ):

            dataset = (
                TASK_CLIENT_DATASETS[
                    task_key
                ][station]["Train"]
            )


            # ------------------------------------------------
            # Deterministic seed
            #
            # Reproducible but different across:
            # task / algorithm / method / round / station
            # ------------------------------------------------

            deterministic_seed = (
                GLOBAL_RANDOM_SEED
                + task_position
                * 1_000_000
                + algorithm_position
                * 10_000
                + method_position
                * 1_000
                + round_index
                * 10
                + station_position
            )


            (
                local_parameters,
                new_control,
            ) = federated_client_update(

                X=dataset["X"],

                y_raw=dataset["y"],

                task_key=task_key,

                method_key=method_key,

                algorithm_config=
                    algorithm_config,

                global_parameters=
                    global_parameters,

                local_epochs=
                    local_epochs,

                round_index=
                    round_index,

                client_control=
                    client_controls[
                        station
                    ],

                server_control=
                    server_control,

                seed=
                    deterministic_seed,
            )


            local_models.append(
                (
                    station,
                    local_parameters,
                )
            )


            updated_controls[
                station
            ] = new_control


        # ====================================================
        # Sample-weighted client aggregation
        # ====================================================

        averaged_parameters = sum(

            (
                client_sizes[station]
                / total_client_rows
            )
            * local_parameters

            for (
                station,
                local_parameters,
            )
            in local_models
        )


        # ====================================================
        # Server update
        # ====================================================

        if method_key == "FedAdam":

            displacement = (
                averaged_parameters
                - global_parameters
            )


            fedadam_m = (
                FEDADAM_BETA1
                * fedadam_m
                + (
                    1.0
                    - FEDADAM_BETA1
                )
                * displacement
            )


            fedadam_v = (
                FEDADAM_BETA2
                * fedadam_v
                + (
                    1.0
                    - FEDADAM_BETA2
                )
                * (
                    displacement
                    ** 2
                )
            )


            global_parameters = (
                global_parameters
                + FEDADAM_SERVER_LR
                * fedadam_m
                / (
                    np.sqrt(
                        fedadam_v
                    )
                    + FEDADAM_TAU
                )
            )


        else:

            # FedAvg / FedProx / SCAFFOLD
            global_parameters = (
                averaged_parameters
            )


        # ====================================================
        # SCAFFOLD server-control update
        # ====================================================

        if method_key == "SCAFFOLD":

            client_controls = (
                updated_controls
            )


            server_control = sum(

                (
                    client_sizes[
                        station
                    ]
                    / total_client_rows
                )
                * client_controls[
                    station
                ]

                for station
                in EXPECTED_STATIONS
            )


        # ====================================================
        # Validation history
        #
        # Diagnostic only.
        # No early stopping and no parameter selection here.
        # ====================================================

        if (
            round_index
            == rounds - 1
            or (
                round_index + 1
            ) % 10 == 0
        ):

            validation_ds = (
                TASK_DATASETS[
                    task_key
                ]["Validation"]
            )


            validation_prediction = (
                predict_raw(
                    validation_ds["X"],
                    global_parameters,
                    task_key,
                )
            )


            validation_metrics = (
                reg_metrics(
                    validation_ds["y"],
                    validation_prediction,
                )
            )


            history.append(
                {
                    "Round":
                        round_index + 1,

                    **validation_metrics,
                }
            )


    # ========================================================
    # 7. Final Validation evaluation
    # ========================================================

    validation_ds = (
        TASK_DATASETS[
            task_key
        ]["Validation"]
    )


    validation_prediction = (
        predict_raw(
            validation_ds["X"],
            global_parameters,
            task_key,
        )
    )


    validation_metrics = (
        reg_metrics(
            validation_ds["y"],
            validation_prediction,
        )
    )


    # ========================================================
    # 8. Validation performance by station
    # ========================================================

    validation_frame = (
        validation_ds["Frame"]
        .reset_index(
            drop=True
        )
    )


    station_metrics = {}

    station_mae = []


    for station in EXPECTED_STATIONS:

        mask = (
            validation_frame[
                "Station"
            ]
            .eq(station)
            .to_numpy()
        )


        if not np.any(mask):

            raise RuntimeError(
                f"No Validation rows for "
                f"{task_key} / {station}"
            )


        metrics_station = (
            reg_metrics(
                validation_ds["y"][
                    mask
                ],
                validation_prediction[
                    mask
                ],
            )
        )


        station_metrics[
            station
        ] = metrics_station


        station_mae.append(
            metrics_station["MAE"]
        )


    station_mae_mean = float(
        np.mean(station_mae)
    )


    station_mae_cv = (
        float(
            np.std(station_mae)
            / station_mae_mean
        )
        if station_mae_mean > EPS
        else np.nan
    )


    # ========================================================
    # 9. Communication estimate
    #
    # FedAvg/FedProx/FedAdam:
    #   upload + download = 2 parameter vectors/client/round
    #
    # SCAFFOLD:
    #   model + control communication = 4 vectors/client/round
    # ========================================================

    communication_multiplier = (
        4
        if method_key == "SCAFFOLD"
        else 2
    )


    transmitted_parameters = (
        rounds
        * len(EXPECTED_STATIONS)
        * parameter_count
        * communication_multiplier
    )


    # ========================================================
    # 10. Return
    # ========================================================

    return {

        "Parameters":
            global_parameters.copy(),

        "Task":
            task_key,

        "Algorithm_Key":
            algorithm_key,

        "Algorithm":
            algorithm_config[
                "name"
            ],

        "Algorithm_Config":
            copy.deepcopy(
                algorithm_config
            ),

        "Federated_Method":
            method_key,

        "Validation":
            validation_metrics,

        "Validation_By_Station":
            station_metrics,

        "Station_MAE_CV":
            station_mae_cv,

        "History":
            history,

        "Rounds":
            rounds,

        "Local_Epochs":
            local_epochs,

        "Training_Time_Seconds":
            time.time()
            - start_time,

        "Transmitted_Parameters":
            int(
                transmitted_parameters
            ),

        "Test_Used":
            False,
    }


# ============================================================
# 11. Build all 96 combinations
# ============================================================

task_keys = [
    definition["Task"]
    for definition
    in TASK_DEFINITIONS
]


algorithm_keys = list(
    {
        row["Algorithm_Key"]
        for _, row
        in CENTRALIZED_RESULTS_DF.iterrows()
    }
)

# Preserve the same logical ordering
algorithm_keys = [
    key
    for key in [
        "Ridge",
        "ElasticNet",
        "SGD_Linear",
        "SGD_Huber",
    ]
    if key in algorithm_keys
]


method_keys = list(
    FEDERATED_METHOD_NAMES
)


combinations = []


for task_position, task_key in enumerate(
    task_keys,
    start=1,
):

    for algorithm_position, algorithm_key in enumerate(
        algorithm_keys,
        start=1,
    ):

        for method_position, method_key in enumerate(
            method_keys,
            start=1,
        ):

            combinations.append(
                {
                    "Task":
                        task_key,

                    "Algorithm_Key":
                        algorithm_key,

                    "Method":
                        method_key,

                    "Task_Position":
                        task_position,

                    "Algorithm_Position":
                        algorithm_position,

                    "Method_Position":
                        method_position,
                }
            )


EXPECTED_TOTAL_RUNS = (
    len(task_keys)
    * len(algorithm_keys)
    * len(method_keys)
)


if EXPECTED_TOTAL_RUNS != 96:
    raise RuntimeError(
        f"Expected 96 combinations, "
        f"found {EXPECTED_TOTAL_RUNS}."
    )


# ============================================================
# 12. Train all federated combinations
# ============================================================

FEDERATED_RUNS = {}


print(
    "\n"
    + "=" * 120
)

print(
    "CELL 13A — FULL FEDERATED TRAINING"
)

print(
    "=" * 120
)

print(
    f"Tasks               : "
    f"{len(task_keys)}"
)

print(
    f"Algorithms          : "
    f"{len(algorithm_keys)}"
)

print(
    f"Federated methods   : "
    f"{len(method_keys)}"
)

print(
    f"Total combinations  : "
    f"{EXPECTED_TOTAL_RUNS}"
)

print(
    f"Rounds/run          : "
    f"{FEDERATED_ROUNDS}"
)

print(
    f"Local epochs/round  : "
    f"{FEDERATED_LOCAL_EPOCHS}"
)

print(
    "Test evaluation     : DISABLED"
)

print(
    "=" * 120
)


for run_number, combo in enumerate(
    combinations,
    start=1,
):

    task_key = combo[
        "Task"
    ]

    algorithm_key = combo[
        "Algorithm_Key"
    ]

    method_key = combo[
        "Method"
    ]


    config_row = (
        CENTRALIZED_RESULTS_DF[
            (
                CENTRALIZED_RESULTS_DF[
                    "Task"
                ].eq(task_key)
            )
            &
            (
                CENTRALIZED_RESULTS_DF[
                    "Algorithm_Key"
                ].eq(algorithm_key)
            )
        ]
        .iloc[0]
    )


    config_id = (
        config_row[
            "Config_ID"
        ]
    )


    print(
        f"\nFederated run "
        f"{run_number:2d}/"
        f"{EXPECTED_TOTAL_RUNS}: "
        f"{task_key} | "
        f"{algorithm_key} | "
        f"{config_id} | "
        f"{method_key}"
    )


    result = train_federated_method(

        task_key=
            task_key,

        algorithm_key=
            algorithm_key,

        method_key=
            method_key,

        task_position=
            combo[
                "Task_Position"
            ],

        algorithm_position=
            combo[
                "Algorithm_Position"
            ],

        method_position=
            combo[
                "Method_Position"
            ],
    )


    FEDERATED_RUNS[
        (
            task_key,
            algorithm_key,
            method_key,
        )
    ] = result


    print(
        f"Validation "
        f"MAE="
        f"{result['Validation']['MAE']:.6f} | "
        f"RMSE="
        f"{result['Validation']['RMSE']:.6f} | "
        f"R2="
        f"{result['Validation']['R2']:.6f} | "
        f"Time="
        f"{result['Training_Time_Seconds']:.2f}s"
    )


    # --------------------------------------------------------
    # Safe checkpoint after every completed experiment
    # --------------------------------------------------------

    try:

        temporary_path = (
            CHECKPOINT_PATH
            + ".tmp"
        )


        with open(
            temporary_path,
            "wb",
        ) as handle:

            pickle.dump(
                FEDERATED_RUNS,
                handle,
                protocol=
                    pickle.HIGHEST_PROTOCOL,
            )


        os.replace(
            temporary_path,
            CHECKPOINT_PATH,
        )


    except Exception as checkpoint_error:

        print(
            "Checkpoint warning:",
            checkpoint_error,
        )


# ============================================================
# 13. Build main Validation results
# ============================================================

federated_rows = []

station_rows = []

history_rows = []


for combo in combinations:

    task_key = combo[
        "Task"
    ]

    algorithm_key = combo[
        "Algorithm_Key"
    ]

    method_key = combo[
        "Method"
    ]


    result = FEDERATED_RUNS[
        (
            task_key,
            algorithm_key,
            method_key,
        )
    ]


    meta = TASK_DATASETS[
        task_key
    ]["Meta"]


    metrics = result[
        "Validation"
    ]


    selected_row = (
        CENTRALIZED_RESULTS_DF[
            (
                CENTRALIZED_RESULTS_DF[
                    "Task"
                ].eq(task_key)
            )
            &
            (
                CENTRALIZED_RESULTS_DF[
                    "Algorithm_Key"
                ].eq(algorithm_key)
            )
        ]
        .iloc[0]
    )


    federated_rows.append(
        {
            "Task":
                task_key,

            "Target":
                meta["Target"],

            "Horizon":
                meta["Horizon"],

            "Federated_Method":
                method_key,

            "Algorithm_Key":
                algorithm_key,

            "Algorithm":
                result[
                    "Algorithm"
                ],

            "Config_ID":
                selected_row[
                    "Config_ID"
                ],

            "Validation_N":
                metrics["N"],

            "Validation_MAE":
                metrics["MAE"],

            "Validation_RMSE":
                metrics["RMSE"],

            "Validation_R2":
                metrics["R2"],

            "Validation_MBE":
                metrics["MBE"],

            "Validation_NRMSE_Std":
                metrics[
                    "NRMSE_Std"
                ],

            "Validation_NRMSE_Range":
                metrics[
                    "NRMSE_Range"
                ],

            "Station_MAE_CV":
                result[
                    "Station_MAE_CV"
                ],

            "Training_Time_Seconds":
                result[
                    "Training_Time_Seconds"
                ],

            "Transmitted_Parameters":
                result[
                    "Transmitted_Parameters"
                ],

            "Rounds":
                result[
                    "Rounds"
                ],

            "Local_Epochs":
                result[
                    "Local_Epochs"
                ],

            "Test_Used":
                False,
        }
    )


    # --------------------------------------------------------
    # Station-level Validation results
    # --------------------------------------------------------

    for station in EXPECTED_STATIONS:

        station_metrics = (
            result[
                "Validation_By_Station"
            ][station]
        )


        station_rows.append(
            {
                "Task":
                    task_key,

                "Target":
                    meta["Target"],

                "Horizon":
                    meta["Horizon"],

                "Station":
                    station,

                "Federated_Method":
                    method_key,

                "Algorithm_Key":
                    algorithm_key,

                "Algorithm":
                    result[
                        "Algorithm"
                    ],

                "Config_ID":
                    selected_row[
                        "Config_ID"
                    ],

                "Split":
                    "Validation",

                **station_metrics,
            }
        )


    # --------------------------------------------------------
    # Round history
    # --------------------------------------------------------

    for history_item in result[
        "History"
    ]:

        history_rows.append(
            {
                "Task":
                    task_key,

                "Algorithm_Key":
                    algorithm_key,

                "Federated_Method":
                    method_key,

                **history_item,
            }
        )


FEDERATED_VALIDATION_RESULTS_DF = pd.DataFrame(
    federated_rows
)


FEDERATED_STATION_VALIDATION_DF = pd.DataFrame(
    station_rows
)


FEDERATED_HISTORY_DF = pd.DataFrame(
    history_rows
)


# ============================================================
# 14. Rank methods fairly within each Task × Algorithm
#
# Ranking rule:
#
#   1) Minimum Validation MAE
#   2) Minimum Validation RMSE
#   3) Maximum Validation R2
#
# ============================================================

ranked_method_parts = []


for task_key in task_keys:

    for algorithm_key in algorithm_keys:

        part = (
            FEDERATED_VALIDATION_RESULTS_DF[
                (
                    FEDERATED_VALIDATION_RESULTS_DF[
                        "Task"
                    ].eq(task_key)
                )
                &
                (
                    FEDERATED_VALIDATION_RESULTS_DF[
                        "Algorithm_Key"
                    ].eq(algorithm_key)
                )
            ]
            .copy()
            .sort_values(
                [
                    "Validation_MAE",
                    "Validation_RMSE",
                    "Validation_R2",
                ],
                ascending=[
                    True,
                    True,
                    False,
                ],
            )
            .reset_index(
                drop=True
            )
        )


        part[
            "Method_Rank_Within_Task_Algorithm"
        ] = np.arange(
            1,
            len(part) + 1,
        )


        part[
            "Method_Win"
        ] = (
            part[
                "Method_Rank_Within_Task_Algorithm"
            ]
            .eq(1)
        )


        ranked_method_parts.append(
            part
        )


FEDERATED_VALIDATION_RESULTS_DF = pd.concat(
    ranked_method_parts,
    ignore_index=True,
)


# ============================================================
# 15. Overall federated-method summary
#
# Each method has:
#   6 tasks × 4 algorithms = 24 paired comparisons
# ============================================================

FEDERATED_METHOD_VALIDATION_SUMMARY_DF = (
    FEDERATED_VALIDATION_RESULTS_DF
    .groupby(
        "Federated_Method",
        as_index=False,
    )
    .agg(

        Paired_Comparisons=(
            "Method_Rank_Within_Task_Algorithm",
            "count",
        ),

        Mean_Paired_Rank=(
            "Method_Rank_Within_Task_Algorithm",
            "mean",
        ),

        Median_Paired_Rank=(
            "Method_Rank_Within_Task_Algorithm",
            "median",
        ),

        Method_Wins=(
            "Method_Win",
            "sum",
        ),

        Mean_Validation_MAE=(
            "Validation_MAE",
            "mean",
        ),

        Mean_Validation_RMSE=(
            "Validation_RMSE",
            "mean",
        ),

        Mean_Validation_R2=(
            "Validation_R2",
            "mean",
        ),

        Mean_Validation_NRMSE_Std=(
            "Validation_NRMSE_Std",
            "mean",
        ),

        Median_Validation_NRMSE_Std=(
            "Validation_NRMSE_Std",
            "median",
        ),

        Mean_Station_MAE_CV=(
            "Station_MAE_CV",
            "mean",
        ),

        Total_Training_Time_Seconds=(
            "Training_Time_Seconds",
            "sum",
        ),

        Mean_Training_Time_Seconds=(
            "Training_Time_Seconds",
            "mean",
        ),

        Transmitted_Parameters_Per_Experiment=(
            "Transmitted_Parameters",
            "first",
        ),
    )
)


FEDERATED_METHOD_VALIDATION_SUMMARY_DF = (
    FEDERATED_METHOD_VALIDATION_SUMMARY_DF
    .sort_values(
        [
            "Mean_Paired_Rank",
            "Method_Wins",
            "Mean_Validation_MAE",
            "Mean_Validation_RMSE",
            "Mean_Validation_R2",
        ],
        ascending=[
            True,
            False,
            True,
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


FEDERATED_METHOD_VALIDATION_SUMMARY_DF[
    "Overall_Method_Rank"
] = np.arange(
    1,
    len(
        FEDERATED_METHOD_VALIDATION_SUMMARY_DF
    )
    + 1,
)


# ============================================================
# 16. Best federated method within every Task × Algorithm
# ============================================================

FEDERATED_BEST_METHOD_BY_TASK_ALGORITHM_DF = (
    FEDERATED_VALIDATION_RESULTS_DF[
        FEDERATED_VALIDATION_RESULTS_DF[
            "Method_Rank_Within_Task_Algorithm"
        ].eq(1)
    ]
    .copy()
    .sort_values(
        [
            "Task",
            "Algorithm_Key",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 17. Additional method × task summary
#
# This does NOT select the final method yet.
# It is diagnostic evidence for Cell 14.
# ============================================================

best_algorithm_inside_method_task = (
    FEDERATED_VALIDATION_RESULTS_DF
    .sort_values(
        [
            "Federated_Method",
            "Task",
            "Validation_MAE",
            "Validation_RMSE",
            "Validation_R2",
        ],
        ascending=[
            True,
            True,
            True,
            True,
            False,
        ],
    )
    .groupby(
        [
            "Federated_Method",
            "Task",
        ],
        as_index=False,
    )
    .first()
)


FEDERATED_METHOD_TASK_BEST_DF = (
    best_algorithm_inside_method_task
)


# ============================================================
# 18. Safety checks
# ============================================================

if len(
    FEDERATED_VALIDATION_RESULTS_DF
) != 96:

    raise RuntimeError(
        "Expected exactly 96 federated results."
    )


if len(
    FEDERATED_RUNS
) != 96:

    raise RuntimeError(
        "Expected exactly 96 trained federated models."
    )


if bool(
    FEDERATED_VALIDATION_RESULTS_DF[
        "Test_Used"
    ].any()
):

    raise RuntimeError(
        "Test data was used in Cell 13."
    )


expected_station_rows = (
    96
    * len(EXPECTED_STATIONS)
)


if len(
    FEDERATED_STATION_VALIDATION_DF
) != expected_station_rows:

    raise RuntimeError(
        "Unexpected number of station-level "
        "Validation results."
    )


if len(
    FEDERATED_BEST_METHOD_BY_TASK_ALGORITHM_DF
) != (
    len(task_keys)
    * len(algorithm_keys)
):

    raise RuntimeError(
        "Expected one winning federated method "
        "for every Task × Algorithm pair."
    )


# ============================================================
# 19. Protocol documentation table
# ============================================================

FEDERATED_CELL13_PROTOCOL_DF = pd.DataFrame(
    [
        {
            "Item":
                "Prediction tasks",

            "Value":
                len(task_keys),
        },

        {
            "Item":
                "Algorithms",

            "Value":
                len(algorithm_keys),
        },

        {
            "Item":
                "Federated methods",

            "Value":
                len(method_keys),
        },

        {
            "Item":
                "Total federated runs",

            "Value":
                len(
                    FEDERATED_VALIDATION_RESULTS_DF
                ),
        },

        {
            "Item":
                "Rounds",

            "Value":
                FEDERATED_ROUNDS,
        },

        {
            "Item":
                "Local epochs per round",

            "Value":
                FEDERATED_LOCAL_EPOCHS,
        },

        {
            "Item":
                "Total local passes",

            "Value":
                (
                    FEDERATED_ROUNDS
                    * FEDERATED_LOCAL_EPOCHS
                ),
        },

        {
            "Item":
                "Clients participating",

            "Value":
                (
                    f"{len(EXPECTED_STATIONS)}/"
                    f"{len(EXPECTED_STATIONS)}"
                ),
        },

        {
            "Item":
                "Algorithm hyperparameters",

            "Value":
                "Frozen from Cell 10",
        },

        {
            "Item":
                "Federated-method hyperparameters",

            "Value":
                "Frozen from Cell 12",
        },

        {
            "Item":
                "Additional tuning",

            "Value":
                False,
        },

        {
            "Item":
                "Evaluation split",

            "Value":
                "Validation only",
        },

        {
            "Item":
                "Primary ranking criterion",

            "Value":
                "Validation MAE",
        },

        {
            "Item":
                "Tie-break 1",

            "Value":
                "Validation RMSE",
        },

        {
            "Item":
                "Tie-break 2",

            "Value":
                "Validation R2",
        },

        {
            "Item":
                "Test used",

            "Value":
                False,
        },
    ]
)


# ============================================================
# 20. Output
# ============================================================

print(
    "\n"
    + "=" * 120
)

print(
    "CELL 13A — FEDERATED TRAINING AND VALIDATION COMPLETED"
)

print(
    "=" * 120
)


print(
    "\nCell 13 protocol:"
)

display(
    FEDERATED_CELL13_PROTOCOL_DF
)


print(
    "\nOverall federated-method Validation summary:"
)

display(
    FEDERATED_METHOD_VALIDATION_SUMMARY_DF
    .round(6)
)


print(
    "\nWinning federated method for every Task × Algorithm:"
)

display(
    FEDERATED_BEST_METHOD_BY_TASK_ALGORITHM_DF[
        [
            "Task",
            "Algorithm_Key",
            "Algorithm",
            "Config_ID",
            "Federated_Method",
            "Validation_MAE",
            "Validation_RMSE",
            "Validation_R2",
            "Validation_NRMSE_Std",
            "Station_MAE_CV",
            "Method_Rank_Within_Task_Algorithm",
        ]
    ]
    .round(6)
)


print(
    "\nBest algorithm inside each Method × Task:"
)

display(
    FEDERATED_METHOD_TASK_BEST_DF[
        [
            "Federated_Method",
            "Task",
            "Algorithm_Key",
            "Algorithm",
            "Config_ID",
            "Validation_MAE",
            "Validation_RMSE",
            "Validation_R2",
            "Validation_NRMSE_Std",
        ]
    ]
    .sort_values(
        [
            "Federated_Method",
            "Task",
        ]
    )
    .round(6)
)


print(
    "\nCELL 13 STATUS:"
)

print(
    "96 federated combinations completed."
)

print(
    "Algorithm hyperparameters were frozen from Cell 10."
)

print(
    "Federated-method hyperparameters were frozen from Cell 12."
)

print(
    "No additional tuning was performed."
)

print(
    "Comparison used Validation only."
)

print(
    "Test set was NOT evaluated or used."
)

print(
    "No final federated method has been forced or selected yet."
)


print(
    f"\nCheckpoint path: "
    f"{CHECKPOINT_PATH}"
)


print(
    f"\nCell 13 completed in "
    f"{time.time() - CELL_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 13B — EXTRACT REAL CELL-13 FEDERATED AGGREGATION WEIGHTS — FINAL
#
# PURPOSE
#   Reconstruct the exact sample-weighted aggregation coefficients used by Cell 13 directly from
#   TASK_CLIENT_DATASETS[task][station]["Train"]["y"].
#
# IMPORTANT
#   - Uses the same formula as Cell 13.
#   - No retraining.
#   - No equal-weight assumption.
#   - No Test use.
#   - No predictions or model parameters are modified.
#
# OUTPUTS
#   CELL13B_AGGREGATION_WEIGHTS_DF
#   CELL13B_WEIGHT_CHECK_DF
#   REAL_CELL13_CLIENT_WEIGHTS_BY_TASK
#   REAL_CELL13_CLIENT_WEIGHTS        (common dictionary when training counts are identical by task)
# ==================================================================================================

import numpy as np
import pandas as pd

print("=" * 180)
print("CELL 13B — EXTRACT REAL CELL-13 FEDERATED AGGREGATION WEIGHTS")
print("=" * 180)

# ==================================================================================================
# 1. REQUIRED STATE
# ==================================================================================================

required_13b = [
    "TASK_CLIENT_DATASETS",
    "EXPECTED_STATIONS",
    "FEDERATED_RUNS",
]

missing_13b = [
    name for name in required_13b
    if name not in globals()
]

if missing_13b:
    raise RuntimeError(
        "Cell 13B requires: "
        + ", ".join(missing_13b)
        + ". Run Cells 8 and 13 first."
    )

TASKS_13B = list(TASK_CLIENT_DATASETS.keys())

if len(TASKS_13B) == 0:
    raise RuntimeError("TASK_CLIENT_DATASETS is empty.")

# ==================================================================================================
# 2. EXACT SAMPLE COUNTS AND WEIGHTS USED BY CELL 13
# ==================================================================================================

rows_13b = []

for task_key in TASKS_13B:

    client_sizes_13b = {}

    for station in EXPECTED_STATIONS:

        try:
            y_train_13b = TASK_CLIENT_DATASETS[
                task_key
            ][station]["Train"]["y"]
        except Exception as exc:
            raise RuntimeError(
                f"Could not read Cell-13 training target for "
                f"{task_key} / {station}: {exc}"
            )

        n_rows_13b = int(len(y_train_13b))

        if n_rows_13b <= 0:
            raise RuntimeError(
                f"No training rows for {task_key} / {station}."
            )

        client_sizes_13b[station] = n_rows_13b

    total_rows_13b = int(sum(client_sizes_13b.values()))

    if total_rows_13b <= 0:
        raise RuntimeError(
            f"No federated training rows for {task_key}."
        )

    for station in EXPECTED_STATIONS:

        weight_13b = (
            client_sizes_13b[station]
            / total_rows_13b
        )

        rows_13b.append(
            {
                "Task": task_key,
                "Station": station,
                "Train_Rows": client_sizes_13b[station],
                "Total_Task_Train_Rows": total_rows_13b,
                "Aggregation_Weight": float(weight_13b),
            }
        )

CELL13B_AGGREGATION_WEIGHTS_DF = pd.DataFrame(rows_13b)

# ==================================================================================================
# 3. VALIDATE EACH TASK
# ==================================================================================================

CELL13B_WEIGHT_CHECK_DF = (
    CELL13B_AGGREGATION_WEIGHTS_DF
    .groupby("Task", as_index=False)
    .agg(
        Total_Train_Rows=("Train_Rows", "sum"),
        Weight_Sum=("Aggregation_Weight", "sum"),
    )
)

CELL13B_WEIGHT_CHECK_DF["Valid_Weight_Sum"] = np.isclose(
    CELL13B_WEIGHT_CHECK_DF["Weight_Sum"],
    1.0,
    atol=1e-12,
)

if not CELL13B_WEIGHT_CHECK_DF["Valid_Weight_Sum"].all():
    raise RuntimeError(
        "At least one task has aggregation weights that do not sum to 1."
    )

# ==================================================================================================
# 4. STORE EXACT PER-TASK WEIGHTS
# ==================================================================================================

REAL_CELL13_CLIENT_WEIGHTS_BY_TASK = {}

for task_key in TASKS_13B:

    task_rows_13b = CELL13B_AGGREGATION_WEIGHTS_DF[
        CELL13B_AGGREGATION_WEIGHTS_DF["Task"].eq(task_key)
    ]

    REAL_CELL13_CLIENT_WEIGHTS_BY_TASK[task_key] = {
        row.Station: float(row.Aggregation_Weight)
        for row in task_rows_13b.itertuples()
    }

# ==================================================================================================
# 5. CHECK WHETHER ONE COMMON WEIGHT DICTIONARY IS EXACT FOR ALL TASKS
# ==================================================================================================

reference_task_13b = TASKS_13B[0]
reference_weights_13b = REAL_CELL13_CLIENT_WEIGHTS_BY_TASK[
    reference_task_13b
]

COMMON_WEIGHTS_VALID_13B = all(
    all(
        np.isclose(
            REAL_CELL13_CLIENT_WEIGHTS_BY_TASK[task][station],
            reference_weights_13b[station],
            atol=1e-15,
            rtol=0.0,
        )
        for station in EXPECTED_STATIONS
    )
    for task in TASKS_13B
)

if COMMON_WEIGHTS_VALID_13B:
    REAL_CELL13_CLIENT_WEIGHTS = dict(reference_weights_13b)
else:
    # Keep the exact task-specific weights. This is the scientifically correct
    # representation when horizons/tasks have slightly different Train row counts.
    REAL_CELL13_CLIENT_WEIGHTS = None

# Compatibility aliases for downstream inspection/export.
CELL13_CLIENT_WEIGHTS = (
    dict(REAL_CELL13_CLIENT_WEIGHTS)
    if REAL_CELL13_CLIENT_WEIGHTS is not None
    else None
)

REAL_CELL13_CLIENT_SAMPLE_COUNTS_BY_TASK = {}

for task_key in TASKS_13B:
    task_df = CELL13B_AGGREGATION_WEIGHTS_DF[
        CELL13B_AGGREGATION_WEIGHTS_DF["Task"].eq(task_key)
    ]
    REAL_CELL13_CLIENT_SAMPLE_COUNTS_BY_TASK[task_key] = {
        row.Station: int(row.Train_Rows)
        for row in task_df.itertuples()
    }

CELL13_CLIENT_SAMPLE_COUNTS = (
    dict(
        REAL_CELL13_CLIENT_SAMPLE_COUNTS_BY_TASK[
            reference_task_13b
        ]
    )
    if COMMON_WEIGHTS_VALID_13B
    else None
)

CELL13_WEIGHT_SOURCE = (
    "Exact TASK_CLIENT_DATASETS Train sample counts used by Cell 13"
)
CELL13B_WEIGHT_TABLE = CELL13B_AGGREGATION_WEIGHTS_DF.copy()

# ==================================================================================================
# 6. DISPLAY
# ==================================================================================================

print("\n" + "=" * 180)
print("REAL CELL-13 SAMPLE COUNTS AND AGGREGATION WEIGHTS")
print("=" * 180)

display(
    CELL13B_AGGREGATION_WEIGHTS_DF.round(12)
)

print("\n" + "=" * 180)
print("WEIGHT-SUM CHECK")
print("=" * 180)

display(
    CELL13B_WEIGHT_CHECK_DF.round(12)
)

if REAL_CELL13_CLIENT_WEIGHTS is not None:

    print("\nREAL_CELL13_CLIENT_WEIGHTS:")

    for station in EXPECTED_STATIONS:
        print(
            f"  {station:<15} "
            f"{REAL_CELL13_CLIENT_WEIGHTS[station]:.12f}"
        )

    print(
        f"\nWeight sum = "
        f"{sum(REAL_CELL13_CLIENT_WEIGHTS.values()):.12f}"
    )

else:

    print(
        "\nAggregation weights are task-specific; "
        "REAL_CELL13_CLIENT_WEIGHTS_BY_TASK will be used downstream."
    )

print("\n" + "=" * 180)
print("CELL 13B COMPLETED SUCCESSFULLY")
print("=" * 180)

print("Weight source               : Actual Cell-13 Train sample counts")
print(
    f"Common across all tasks     : "
    f"{'YES' if COMMON_WEIGHTS_VALID_13B else 'NO — task-specific weights preserved'}"
)
print("Equal-weight assumption     : NO")
print("Retraining performed        : NO")
print("Test data used              : NO")
print("Predictions changed         : NO")


In [ ]:
# ============================================================
# CELL 14 — FINAL FEDERATED METHOD SELECTION
#           + FIRST TEST EVALUATION
# ============================================================
#
# Scientific selection sequence:
#
# STEP 1
#   For every Federated Method × Task:
#   select the best algorithm using Validation ONLY.
#
# STEP 2
#   Compare the four federated methods task-by-task using:
#
#       1) Validation MAE
#       2) Validation RMSE
#       3) Validation R2
#
# STEP 3
#   Select ONE global federated method using:
#
#       1) Mean Task Rank
#       2) Number of Task Wins
#       3) Mean Validation MAE
#       4) Mean Validation RMSE
#       5) Mean Validation R2
#
# STEP 4
#   Freeze:
#       - selected federated method
#       - selected algorithm for every task
#
# STEP 5
#   ONLY AFTER ALL SELECTION IS COMPLETE:
#   evaluate the frozen models on TEST.
#
# IMPORTANT:
#   TEST is never used for method or algorithm selection.
#
# ============================================================

import time
import numpy as np
import pandas as pd

CELL_START = time.time()


# ============================================================
# 1. Safety checks
# ============================================================

required_objects = [
    "FEDERATED_VALIDATION_RESULTS_DF",
    "FEDERATED_RUNS",
    "FEDERATED_METHOD_NAMES",
    "TASK_DATASETS",
    "TASK_DEFINITIONS",
    "EXPECTED_STATIONS",
    "predict_raw",
    "reg_metrics",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


# ============================================================
# 2. Confirm Cell 13 contains exactly 96 Validation runs
# ============================================================

EXPECTED_RUNS = (
    len(TASK_DEFINITIONS)
    * 4
    * len(FEDERATED_METHOD_NAMES)
)

if len(
    FEDERATED_VALIDATION_RESULTS_DF
) != EXPECTED_RUNS:

    raise RuntimeError(
        f"Expected {EXPECTED_RUNS} federated "
        f"Validation results."
    )


if "Test_Used" in FEDERATED_VALIDATION_RESULTS_DF.columns:

    if bool(
        FEDERATED_VALIDATION_RESULTS_DF[
            "Test_Used"
        ].any()
    ):

        raise RuntimeError(
            "Cell 13 used Test data. "
            "Final selection cannot continue."
        )


# ============================================================
# 3. STEP 1
#    Best algorithm inside each Federated Method × Task
#
# Selection:
#   MAE -> RMSE -> R2
# ============================================================

best_algorithm_parts = []


for method_key in FEDERATED_METHOD_NAMES:

    for task_definition in TASK_DEFINITIONS:

        task_key = task_definition[
            "Task"
        ]


        part = (
            FEDERATED_VALIDATION_RESULTS_DF[
                (
                    FEDERATED_VALIDATION_RESULTS_DF[
                        "Federated_Method"
                    ].eq(method_key)
                )
                &
                (
                    FEDERATED_VALIDATION_RESULTS_DF[
                        "Task"
                    ].eq(task_key)
                )
            ]
            .copy()
            .sort_values(
                [
                    "Validation_MAE",
                    "Validation_RMSE",
                    "Validation_R2",
                ],
                ascending=[
                    True,
                    True,
                    False,
                ],
            )
            .reset_index(
                drop=True
            )
        )


        if len(part) != 4:

            raise RuntimeError(
                f"Expected four algorithms for "
                f"{method_key} / {task_key}, "
                f"found {len(part)}."
            )


        part[
            "Algorithm_Rank_Within_Method_Task"
        ] = np.arange(
            1,
            len(part) + 1,
        )


        best_algorithm_parts.append(
            part.iloc[[0]].copy()
        )


FED14_BEST_ALGORITHM_WITHIN_METHOD_TASK_DF = (
    pd.concat(
        best_algorithm_parts,
        ignore_index=True,
    )
)


# ============================================================
# 4. STEP 2
#    Rank federated methods within each Task
#
# Each method is represented by its Validation-best algorithm
# for that task.
# ============================================================

task_method_rank_parts = []


for task_definition in TASK_DEFINITIONS:

    task_key = task_definition[
        "Task"
    ]


    part = (
        FED14_BEST_ALGORITHM_WITHIN_METHOD_TASK_DF[
            FED14_BEST_ALGORITHM_WITHIN_METHOD_TASK_DF[
                "Task"
            ].eq(task_key)
        ]
        .copy()
        .sort_values(
            [
                "Validation_MAE",
                "Validation_RMSE",
                "Validation_R2",
            ],
            ascending=[
                True,
                True,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


    if len(part) != len(
        FEDERATED_METHOD_NAMES
    ):

        raise RuntimeError(
            f"Expected four federated methods "
            f"for task {task_key}."
        )


    part[
        "Task_Rank"
    ] = np.arange(
        1,
        len(part) + 1,
    )


    part[
        "Task_Win"
    ] = (
        part[
            "Task_Rank"
        ].eq(1)
    )


    task_method_rank_parts.append(
        part
    )


FED14_METHOD_TASK_RESULTS_DF = pd.concat(
    task_method_rank_parts,
    ignore_index=True,
)


# ============================================================
# 5. STEP 3
#    Global federated-method summary
#
# Final selection is based ONLY on Validation.
# ============================================================

FED14_METHOD_SUMMARY_DF = (
    FED14_METHOD_TASK_RESULTS_DF
    .groupby(
        "Federated_Method",
        as_index=False,
    )
    .agg(

        Tasks=(
            "Task",
            "count",
        ),

        Mean_Task_Rank=(
            "Task_Rank",
            "mean",
        ),

        Median_Task_Rank=(
            "Task_Rank",
            "median",
        ),

        Task_Wins=(
            "Task_Win",
            "sum",
        ),

        Mean_Validation_MAE=(
            "Validation_MAE",
            "mean",
        ),

        Mean_Validation_RMSE=(
            "Validation_RMSE",
            "mean",
        ),

        Mean_Validation_R2=(
            "Validation_R2",
            "mean",
        ),

        Mean_Validation_NRMSE_Std=(
            "Validation_NRMSE_Std",
            "mean",
        ),

        Median_Validation_NRMSE_Std=(
            "Validation_NRMSE_Std",
            "median",
        ),

        Mean_Station_MAE_CV=(
            "Station_MAE_CV",
            "mean",
        ),

        Total_Selected_Training_Time_Seconds=(
            "Training_Time_Seconds",
            "sum",
        ),

        Communication_Parameters_Per_Task=(
            "Transmitted_Parameters",
            "first",
        ),
    )
)


# ============================================================
# 6. Explicit final Validation ranking rule
#
# 1. Mean task rank
# 2. Task wins
# 3. Mean MAE
# 4. Mean RMSE
# 5. Mean R2
# ============================================================

FED14_METHOD_SUMMARY_DF = (
    FED14_METHOD_SUMMARY_DF
    .sort_values(
        [
            "Mean_Task_Rank",
            "Task_Wins",
            "Mean_Validation_MAE",
            "Mean_Validation_RMSE",
            "Mean_Validation_R2",
        ],
        ascending=[
            True,
            False,
            True,
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


FED14_METHOD_SUMMARY_DF[
    "Final_Validation_Method_Rank"
] = np.arange(
    1,
    len(
        FED14_METHOD_SUMMARY_DF
    )
    + 1,
)


# ============================================================
# 7. Select final federated method
#
# NO hard-coded FedAdam/SCAFFOLD override.
# ============================================================

SELECTED_FEDERATED_METHOD = str(
    FED14_METHOD_SUMMARY_DF
    .iloc[0][
        "Federated_Method"
    ]
)


# ============================================================
# 8. Freeze selected algorithm for every task
#
# Use ONLY rows belonging to the selected federated method.
# ============================================================

FEDERATED_SELECTED_ALGORITHM_DF = (
    FED14_METHOD_TASK_RESULTS_DF[
        FED14_METHOD_TASK_RESULTS_DF[
            "Federated_Method"
        ].eq(
            SELECTED_FEDERATED_METHOD
        )
    ]
    .copy()
    .sort_values(
        [
            "Target",
            "Horizon",
        ]
    )
    .reset_index(
        drop=True
    )
)


if len(
    FEDERATED_SELECTED_ALGORITHM_DF
) != len(
    TASK_DEFINITIONS
):

    raise RuntimeError(
        "Expected exactly one selected algorithm "
        "per task."
    )


FEDERATED_SELECTED_ALGORITHM_DF[
    "Selection_Status"
] = (
    "Federated method and algorithm "
    "selected using Validation only"
)


# ============================================================
# 9. Record frozen final selection BEFORE opening Test
# ============================================================

FED14_FINAL_SELECTION_DF = (
    FEDERATED_SELECTED_ALGORITHM_DF[
        [
            "Task",
            "Target",
            "Horizon",
            "Federated_Method",
            "Algorithm_Key",
            "Algorithm",
            "Config_ID",
            "Validation_MAE",
            "Validation_RMSE",
            "Validation_R2",
            "Validation_NRMSE_Std",
            "Task_Rank",
            "Selection_Status",
        ]
    ]
    .copy()
)


# ============================================================
# 10. Selection protocol documentation
# ============================================================

FED14_SELECTION_PROTOCOL_DF = pd.DataFrame(
    [
        {
            "Stage":
                "Algorithm selection",

            "Data":
                "Validation",

            "Rule":
                (
                    "Best algorithm within each "
                    "Federated Method × Task"
                ),

            "Primary_Criterion":
                "MAE",

            "Tie_Break_1":
                "RMSE",

            "Tie_Break_2":
                "R2",

            "Test_Used":
                False,
        },

        {
            "Stage":
                "Federated method selection",

            "Data":
                "Validation",

            "Rule":
                (
                    "Mean Task Rank -> Task Wins -> "
                    "Mean MAE -> Mean RMSE -> Mean R2"
                ),

            "Primary_Criterion":
                "Mean Task Rank",

            "Tie_Break_1":
                "Task Wins",

            "Tie_Break_2":
                "Validation metrics",

            "Test_Used":
                False,
        },

        {
            "Stage":
                "Final evaluation",

            "Data":
                "Test",

            "Rule":
                "Frozen selected method and algorithms only",

            "Primary_Criterion":
                "Evaluation only",

            "Tie_Break_1":
                "None",

            "Tie_Break_2":
                "None",

            "Test_Used":
                True,
        },
    ]
)


# ============================================================
# 11. IMPORTANT FREEZE POINT
#
# From this point onward:
#
# - Federated method is fixed.
# - Algorithm for every task is fixed.
#
# Test cannot change either decision.
# ============================================================

FINAL_SELECTION_FROZEN = True


# ============================================================
# 12. STEP 5 — First TEST evaluation
# ============================================================

test_task_rows = []

test_prediction_rows = []

test_station_rows = []


for _, selected in (
    FEDERATED_SELECTED_ALGORITHM_DF
    .iterrows()
):

    task_key = selected[
        "Task"
    ]

    algorithm_key = selected[
        "Algorithm_Key"
    ]


    meta = TASK_DATASETS[
        task_key
    ]["Meta"]


    # --------------------------------------------------------
    # Frozen federated model from Cell 13
    # --------------------------------------------------------

    run_key = (
        task_key,
        algorithm_key,
        SELECTED_FEDERATED_METHOD,
    )


    if run_key not in FEDERATED_RUNS:

        raise KeyError(
            f"Missing federated model: "
            f"{run_key}"
        )


    params = FEDERATED_RUNS[
        run_key
    ][
        "Parameters"
    ]


    # --------------------------------------------------------
    # Test data
    # --------------------------------------------------------

    test_ds = TASK_DATASETS[
        task_key
    ][
        "Test"
    ]


    prediction = predict_raw(
        test_ds[
            "X"
        ],
        params,
        task_key,
    )


    metrics = reg_metrics(
        test_ds[
            "y"
        ],
        prediction,
    )


    frame = (
        test_ds[
            "Frame"
        ]
        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # Task-level Test result
    # ========================================================

    test_task_rows.append(
        {
            "Task":
                task_key,

            "Target":
                meta[
                    "Target"
                ],

            "Horizon":
                meta[
                    "Horizon"
                ],

            "Federated_Method":
                SELECTED_FEDERATED_METHOD,

            "Algorithm_Key":
                algorithm_key,

            "Algorithm":
                selected[
                    "Algorithm"
                ],

            "Config_ID":
                selected[
                    "Config_ID"
                ],

            **metrics,

            "Selection_Data":
                "Validation only",

            "Test_Role":
                "Final evaluation only",
        }
    )


    # ========================================================
    # Station-level Test metrics
    # ========================================================

    for station in EXPECTED_STATIONS:

        station_mask = (
            frame[
                "Station"
            ]
            .eq(station)
            .to_numpy()
        )


        if not np.any(
            station_mask
        ):

            continue


        station_metrics = reg_metrics(
            test_ds[
                "y"
            ][
                station_mask
            ],
            prediction[
                station_mask
            ],
        )


        test_station_rows.append(
            {
                "Task":
                    task_key,

                "Target":
                    meta[
                        "Target"
                    ],

                "Horizon":
                    meta[
                        "Horizon"
                    ],

                "Station":
                    station,

                "Federated_Method":
                    SELECTED_FEDERATED_METHOD,

                "Algorithm_Key":
                    algorithm_key,

                "Algorithm":
                    selected[
                        "Algorithm"
                    ],

                "Config_ID":
                    selected[
                        "Config_ID"
                    ],

                **station_metrics,
            }
        )


    # ========================================================
    # Row-level Test predictions
    # ========================================================

    for i, (
        actual,
        pred,
    ) in enumerate(
        zip(
            test_ds[
                "y"
            ],
            prediction,
        )
    ):

        test_prediction_rows.append(
            {
                "Task":
                    task_key,

                "Target":
                    meta[
                        "Target"
                    ],

                "Horizon":
                    meta[
                        "Horizon"
                    ],

                "Station":
                    frame.loc[
                        i,
                        "Station",
                    ],

                "Date":
                    frame.loc[
                        i,
                        "Date",
                    ],

                "Target_Date":
                    frame.loc[
                        i,
                        meta[
                            "Future_Date_Column"
                        ],
                    ],

                "Federated_Method":
                    SELECTED_FEDERATED_METHOD,

                "Algorithm_Key":
                    algorithm_key,

                "Config_ID":
                    selected[
                        "Config_ID"
                    ],

                "Actual":
                    actual,

                "Prediction":
                    pred,
            }
        )


# ============================================================
# 13. Final Test DataFrames
# ============================================================

FED14_TEST_RESULTS_DF = pd.DataFrame(
    test_task_rows
)


FED14_TEST_STATION_RESULTS_DF = pd.DataFrame(
    test_station_rows
)


FED14_TEST_PREDICTIONS_DF = pd.DataFrame(
    test_prediction_rows
)


# ============================================================
# 14. Overall final Test summary
#
# IMPORTANT:
# This summary is descriptive ONLY.
# It does not affect selection.
# ============================================================

FED14_TEST_OVERALL_DF = pd.DataFrame(
    [
        {
            "Federated_Method":
                SELECTED_FEDERATED_METHOD,

            "Tasks":
                len(
                    FED14_TEST_RESULTS_DF
                ),

            "Mean_Test_MAE":
                FED14_TEST_RESULTS_DF[
                    "MAE"
                ].mean(),

            "Mean_Test_RMSE":
                FED14_TEST_RESULTS_DF[
                    "RMSE"
                ].mean(),

            "Mean_Test_R2":
                FED14_TEST_RESULTS_DF[
                    "R2"
                ].mean(),

            "Mean_Test_NRMSE_Std":
                FED14_TEST_RESULTS_DF[
                    "NRMSE_Std"
                ].mean(),

            "Mean_Absolute_Test_MBE":
                FED14_TEST_RESULTS_DF[
                    "MBE"
                ].abs().mean(),

            "Selection_Data":
                "Validation only",

            "Test_Used_For_Selection":
                False,
        }
    ]
)


# ============================================================
# 15. Safety checks
# ============================================================

if not FINAL_SELECTION_FROZEN:

    raise RuntimeError(
        "Final selection must be frozen "
        "before Test evaluation."
    )


if len(
    FED14_TEST_RESULTS_DF
) != len(
    TASK_DEFINITIONS
):

    raise RuntimeError(
        "Expected one final Test result per task."
    )


if (
    FED14_TEST_RESULTS_DF[
        "Selection_Data"
    ]
    .ne(
        "Validation only"
    )
    .any()
):

    raise RuntimeError(
        "Unexpected selection source."
    )


# ============================================================
# 16. Output
# ============================================================

print(
    "\n"
    + "=" * 125
)

print(
    "CELL 14 — FINAL FEDERATED SELECTION + TEST EVALUATION"
)

print(
    "=" * 125
)


print(
    "\nSelection protocol:"
)

display(
    FED14_SELECTION_PROTOCOL_DF
)


print(
    "\nBest algorithm inside every Federated Method × Task:"
)

display(
    FED14_BEST_ALGORITHM_WITHIN_METHOD_TASK_DF[
        [
            "Federated_Method",
            "Task",
            "Algorithm_Key",
            "Algorithm",
            "Config_ID",
            "Validation_MAE",
            "Validation_RMSE",
            "Validation_R2",
        ]
    ]
    .sort_values(
        [
            "Task",
            "Validation_MAE",
        ]
    )
    .round(6)
)


print(
    "\nFederated method ranking based ONLY on Validation:"
)

display(
    FED14_METHOD_SUMMARY_DF
    .round(6)
)


print(
    "\nSELECTED FEDERATED METHOD:"
)

print(
    SELECTED_FEDERATED_METHOD
)


print(
    "\nFrozen selected algorithm for every task:"
)

display(
    FED14_FINAL_SELECTION_DF
    .round(6)
)


print(
    "\n"
    + "-" * 125
)

print(
    "SELECTION IS NOW FROZEN."
)

print(
    "The following Test results cannot alter "
    "the selected method or algorithm."
)

print(
    "-" * 125
)


print(
    "\nFinal federated TEST results by task:"
)

display(
    FED14_TEST_RESULTS_DF
    .round(6)
)


print(
    "\nOverall final federated TEST summary:"
)

display(
    FED14_TEST_OVERALL_DF
    .round(6)
)


print(
    "\nCELL 14 STATUS:"
)

print(
    "Federated method selected using Validation only."
)

print(
    "Task-specific algorithms selected using Validation only."
)

print(
    "Final selection was frozen before Test evaluation."
)

print(
    "Test was used ONLY for final evaluation."
)


print(
    f"\nCell 14 completed in "
    f"{time.time() - CELL_START:.2f} seconds"
)

In [ ]:
# ============================================================
# CELL 15 — FINAL CENTRALIZED vs LOCAL vs FEDERATED COMPARISON
# ============================================================
#
# Purpose:
#   Compare the three training paradigms on the FINAL TEST set:
#
#       1. Centralized
#       2. Local
#       3. Federated (selected in Cell 14; currently FedAdam)
#
# FAIRNESS RULE:
#
#   For each task, ALL THREE paradigms use the SAME
#   task-specific algorithm selected in Cell 14.
#
# This isolates the effect of TRAINING PARADIGM rather than
# confounding it with a different algorithm choice.
#
# IMPORTANT:
#   - No training occurs in this cell.
#   - No hyperparameter tuning occurs in this cell.
#   - Test is used for FINAL evaluation only.
#   - Test cannot alter any previous model-selection decision.
#
# ============================================================

import time
import numpy as np
import pandas as pd

CELL_START = time.time()


# ============================================================
# 1. Safety checks
# ============================================================

required_objects = [
    "TASK_DATASETS",
    "TASK_DEFINITIONS",
    "EXPECTED_STATIONS",
    "CENTRALIZED_MODELS",
    "LOCAL_MODELS",
    "FEDERATED_RUNS",
    "FEDERATED_SELECTED_ALGORITHM_DF",
    "SELECTED_FEDERATED_METHOD",
    "predict_raw",
    "reg_metrics",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects: "
        + ", ".join(missing_objects)
    )


# ============================================================
# 2. Confirm final federated selection exists
# ============================================================

if len(
    FEDERATED_SELECTED_ALGORITHM_DF
) != len(
    TASK_DEFINITIONS
):

    raise RuntimeError(
        "Expected exactly one selected algorithm per task."
    )


# ============================================================
# 3. Comparison protocol
# ============================================================

PARADIGM_COMPARISON_PROTOCOL_DF = pd.DataFrame(
    [
        {
            "Comparison":
                "Strict Same Algorithm",

            "Purpose":
                "Isolate the effect of training paradigm",

            "Paradigms":
                "Centralized / Local / Federated",

            "Algorithm_Rule":
                (
                    "Same Cell 14 selected "
                    "task-specific algorithm "
                    "for all three paradigms"
                ),

            "Federated_Method":
                SELECTED_FEDERATED_METHOD,

            "Selection_Data":
                "Validation only",

            "Evaluation_Data":
                "Test only",

            "Additional_Training":
                False,

            "Additional_Tuning":
                False,

            "Test_Used_For_Selection":
                False,
        }
    ]
)


# ============================================================
# 4. Containers
# ============================================================

task_rows = []

prediction_rows = []

station_rows = []


# ============================================================
# 5. Evaluate every task
# ============================================================

for _, selected in (
    FEDERATED_SELECTED_ALGORITHM_DF
    .iterrows()
):

    task_key = selected[
        "Task"
    ]

    algorithm_key = selected[
        "Algorithm_Key"
    ]

    algorithm_name = selected[
        "Algorithm"
    ]

    config_id = selected[
        "Config_ID"
    ]


    meta = TASK_DATASETS[
        task_key
    ][
        "Meta"
    ]


    test_ds = TASK_DATASETS[
        task_key
    ][
        "Test"
    ]


    X_test = test_ds[
        "X"
    ]

    y_test = test_ds[
        "y"
    ]


    frame = (
        test_ds[
            "Frame"
        ]
        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # 5A. CENTRALIZED
    # ========================================================

    central_key = (
        task_key,
        algorithm_key,
    )


    if central_key not in CENTRALIZED_MODELS:

        raise KeyError(
            f"Missing centralized model: "
            f"{central_key}"
        )


    centralized_parameters = (
        CENTRALIZED_MODELS[
            central_key
        ][
            "Parameters"
        ]
    )


    centralized_prediction = predict_raw(
        X_test,
        centralized_parameters,
        task_key,
    )


    centralized_metrics = reg_metrics(
        y_test,
        centralized_prediction,
    )


    task_rows.append(
        {
            "Task":
                task_key,

            "Target":
                meta[
                    "Target"
                ],

            "Horizon":
                meta[
                    "Horizon"
                ],

            "Paradigm":
                "Centralized",

            "Method":
                "Centralized pooled",

            "Algorithm_Key":
                algorithm_key,

            "Algorithm":
                algorithm_name,

            "Config_ID":
                config_id,

            **centralized_metrics,

            "Selection_Source":
                "Validation-selected in earlier cells",

            "Test_Role":
                "Final evaluation only",
        }
    )


    # ========================================================
    # 5B. FEDERATED
    # ========================================================

    federated_key = (
        task_key,
        algorithm_key,
        SELECTED_FEDERATED_METHOD,
    )


    if federated_key not in FEDERATED_RUNS:

        raise KeyError(
            f"Missing federated model: "
            f"{federated_key}"
        )


    federated_parameters = (
        FEDERATED_RUNS[
            federated_key
        ][
            "Parameters"
        ]
    )


    federated_prediction = predict_raw(
        X_test,
        federated_parameters,
        task_key,
    )


    federated_metrics = reg_metrics(
        y_test,
        federated_prediction,
    )


    task_rows.append(
        {
            "Task":
                task_key,

            "Target":
                meta[
                    "Target"
                ],

            "Horizon":
                meta[
                    "Horizon"
                ],

            "Paradigm":
                "Federated",

            "Method":
                SELECTED_FEDERATED_METHOD,

            "Algorithm_Key":
                algorithm_key,

            "Algorithm":
                algorithm_name,

            "Config_ID":
                config_id,

            **federated_metrics,

            "Selection_Source":
                "Validation-selected in Cell 14",

            "Test_Role":
                "Final evaluation only",
        }
    )


    # ========================================================
    # 5C. LOCAL
    #
    # Each test row is predicted by the Local model belonging
    # to that row's own station.
    # ========================================================

    local_prediction = np.empty(
        len(
            frame
        ),
        dtype=float,
    )


    for station in EXPECTED_STATIONS:

        station_mask = (
            frame[
                "Station"
            ]
            .eq(
                station
            )
            .to_numpy()
        )


        if not np.any(
            station_mask
        ):
            continue


        local_key = (
            task_key,
            station,
            algorithm_key,
        )


        if local_key not in LOCAL_MODELS:

            raise KeyError(
                f"Missing local model: "
                f"{local_key}"
            )


        local_parameters = (
            LOCAL_MODELS[
                local_key
            ][
                "Parameters"
            ]
        )


        local_prediction[
            station_mask
        ] = predict_raw(

            X_test[
                station_mask
            ],

            local_parameters,

            task_key,
        )


    local_metrics = reg_metrics(
        y_test,
        local_prediction,
    )


    task_rows.append(
        {
            "Task":
                task_key,

            "Target":
                meta[
                    "Target"
                ],

            "Horizon":
                meta[
                    "Horizon"
                ],

            "Paradigm":
                "Local",

            "Method":
                "Station-specific models",

            "Algorithm_Key":
                algorithm_key,

            "Algorithm":
                algorithm_name,

            "Config_ID":
                config_id,

            **local_metrics,

            "Selection_Source":
                "Same algorithm frozen from Cell 14",

            "Test_Role":
                "Final evaluation only",
        }
    )


    # ========================================================
    # 6. Row-level predictions
    # ========================================================

    paradigm_predictions = [
        (
            "Centralized",
            "Centralized pooled",
            centralized_prediction,
        ),

        (
            "Federated",
            SELECTED_FEDERATED_METHOD,
            federated_prediction,
        ),

        (
            "Local",
            "Station-specific models",
            local_prediction,
        ),
    ]


    for (
        paradigm,
        method,
        prediction,
    ) in paradigm_predictions:

        for i, (
            actual,
            pred,
        ) in enumerate(
            zip(
                y_test,
                prediction,
            )
        ):

            prediction_rows.append(
                {
                    "Task":
                        task_key,

                    "Target":
                        meta[
                            "Target"
                        ],

                    "Horizon":
                        meta[
                            "Horizon"
                        ],

                    "Station":
                        frame.loc[
                            i,
                            "Station",
                        ],

                    "Date":
                        frame.loc[
                            i,
                            "Date",
                        ],

                    "Target_Date":
                        frame.loc[
                            i,
                            meta[
                                "Future_Date_Column"
                            ],
                        ],

                    "Paradigm":
                        paradigm,

                    "Method":
                        method,

                    "Algorithm_Key":
                        algorithm_key,

                    "Algorithm":
                        algorithm_name,

                    "Config_ID":
                        config_id,

                    "Actual":
                        actual,

                    "Prediction":
                        pred,
                }
            )


    # ========================================================
    # 7. Station-level Test results
    # ========================================================

    for (
        paradigm,
        method,
        prediction,
    ) in paradigm_predictions:

        for station in EXPECTED_STATIONS:

            station_mask = (
                frame[
                    "Station"
                ]
                .eq(
                    station
                )
                .to_numpy()
            )


            if not np.any(
                station_mask
            ):
                continue


            metrics_station = reg_metrics(

                y_test[
                    station_mask
                ],

                prediction[
                    station_mask
                ],
            )


            station_rows.append(
                {
                    "Task":
                        task_key,

                    "Target":
                        meta[
                            "Target"
                        ],

                    "Horizon":
                        meta[
                            "Horizon"
                        ],

                    "Station":
                        station,

                    "Paradigm":
                        paradigm,

                    "Method":
                        method,

                    "Algorithm_Key":
                        algorithm_key,

                    "Algorithm":
                        algorithm_name,

                    "Config_ID":
                        config_id,

                    **metrics_station,
                }
            )


# ============================================================
# 8. Main DataFrames
# ============================================================

PARADIGM15_TASK_RESULTS_DF = pd.DataFrame(
    task_rows
)


PARADIGM15_PREDICTIONS_DF = pd.DataFrame(
    prediction_rows
)


PARADIGM15_STATION_RESULTS_DF = pd.DataFrame(
    station_rows
)


# ============================================================
# 9. Rank paradigms within each task
#
# Same scientific ranking rule:
#
# 1) Minimum MAE
# 2) Minimum RMSE
# 3) Maximum R2
#
# ============================================================

ranked_parts = []


for task_definition in TASK_DEFINITIONS:

    task_key = task_definition[
        "Task"
    ]


    part = (
        PARADIGM15_TASK_RESULTS_DF[
            PARADIGM15_TASK_RESULTS_DF[
                "Task"
            ].eq(
                task_key
            )
        ]
        .copy()
        .sort_values(
            [
                "MAE",
                "RMSE",
                "R2",
            ],
            ascending=[
                True,
                True,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


    if len(part) != 3:

        raise RuntimeError(
            f"Expected three paradigms "
            f"for {task_key}."
        )


    part[
        "Task_Rank"
    ] = np.arange(
        1,
        len(part) + 1,
    )


    part[
        "Task_Win"
    ] = (
        part[
            "Task_Rank"
        ]
        .eq(1)
    )


    ranked_parts.append(
        part
    )


PARADIGM15_TASK_RESULTS_DF = pd.concat(
    ranked_parts,
    ignore_index=True,
)


# ============================================================
# 10. Overall paradigm summary
# ============================================================

PARADIGM15_OVERALL_DF = (
    PARADIGM15_TASK_RESULTS_DF
    .groupby(
        "Paradigm",
        as_index=False,
    )
    .agg(

        Tasks=(
            "Task",
            "count",
        ),

        Mean_MAE=(
            "MAE",
            "mean",
        ),

        Mean_RMSE=(
            "RMSE",
            "mean",
        ),

        Mean_R2=(
            "R2",
            "mean",
        ),

        Mean_NRMSE_Std=(
            "NRMSE_Std",
            "mean",
        ),

        Mean_Absolute_MBE=(
            "MBE",
            lambda x:
                float(
                    np.mean(
                        np.abs(
                            np.asarray(
                                x,
                                dtype=float,
                            )
                        )
                    )
                ),
        ),

        Mean_Task_Rank=(
            "Task_Rank",
            "mean",
        ),

        Task_Wins=(
            "Task_Win",
            "sum",
        ),
    )
)


# ============================================================
# 11. Overall descriptive ranking
#
# Test ranking is REPORTING ONLY.
# It does NOT change any earlier selection.
# ============================================================

PARADIGM15_OVERALL_DF = (
    PARADIGM15_OVERALL_DF
    .sort_values(
        [
            "Mean_Task_Rank",
            "Task_Wins",
            "Mean_MAE",
            "Mean_RMSE",
            "Mean_R2",
        ],
        ascending=[
            True,
            False,
            True,
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


PARADIGM15_OVERALL_DF[
    "Overall_Test_Rank"
] = np.arange(
    1,
    len(
        PARADIGM15_OVERALL_DF
    )
    + 1,
)


# ============================================================
# 12. Win matrix / task winners
# ============================================================

PARADIGM15_TASK_WINNERS_DF = (
    PARADIGM15_TASK_RESULTS_DF[
        PARADIGM15_TASK_RESULTS_DF[
            "Task_Rank"
        ].eq(1)
    ]
    .copy()
    .sort_values(
        [
            "Target",
            "Horizon",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 13. Pairwise improvement vs Local
#
# Descriptive only.
# Positive percentages mean MAE improvement over Local.
# ============================================================

improvement_rows = []


for task_definition in TASK_DEFINITIONS:

    task_key = task_definition[
        "Task"
    ]


    task_part = (
        PARADIGM15_TASK_RESULTS_DF[
            PARADIGM15_TASK_RESULTS_DF[
                "Task"
            ].eq(
                task_key
            )
        ]
    )


    local_mae = float(
        task_part.loc[
            task_part[
                "Paradigm"
            ].eq(
                "Local"
            ),
            "MAE",
        ]
        .iloc[0]
    )


    for paradigm in [
        "Centralized",
        "Federated",
    ]:

        paradigm_mae = float(
            task_part.loc[
                task_part[
                    "Paradigm"
                ].eq(
                    paradigm
                ),
                "MAE",
            ]
            .iloc[0]
        )


        improvement = (
            100.0
            * (
                local_mae
                - paradigm_mae
            )
            / local_mae
        )


        improvement_rows.append(
            {
                "Task":
                    task_key,

                "Compared_Paradigm":
                    paradigm,

                "Reference":
                    "Local",

                "Local_MAE":
                    local_mae,

                "Compared_MAE":
                    paradigm_mae,

                "MAE_Improvement_vs_Local_Percent":
                    improvement,
            }
        )


PARADIGM15_IMPROVEMENT_VS_LOCAL_DF = (
    pd.DataFrame(
        improvement_rows
    )
)


# ============================================================
# 14. Safety checks
# ============================================================

expected_task_rows = (
    len(
        TASK_DEFINITIONS
    )
    * 3
)


if len(
    PARADIGM15_TASK_RESULTS_DF
) != expected_task_rows:

    raise RuntimeError(
        "Unexpected number of paradigm task results."
    )


expected_station_rows = (
    len(
        TASK_DEFINITIONS
    )
    * len(
        EXPECTED_STATIONS
    )
    * 3
)


if len(
    PARADIGM15_STATION_RESULTS_DF
) != expected_station_rows:

    raise RuntimeError(
        "Unexpected number of station-level results."
    )


if len(
    PARADIGM15_TASK_WINNERS_DF
) != len(
    TASK_DEFINITIONS
):

    raise RuntimeError(
        "Expected one Test winner per task."
    )


# ============================================================
# 15. Output
# ============================================================

print(
    "\n"
    + "=" * 125
)

print(
    "CELL 15 — CENTRALIZED vs LOCAL vs FEDERATED TEST COMPARISON"
)

print(
    "=" * 125
)


print(
    "\nComparison protocol:"
)

display(
    PARADIGM_COMPARISON_PROTOCOL_DF
)


print(
    "\nTask-level TEST comparison:"
)

display(
    PARADIGM15_TASK_RESULTS_DF[
        [
            "Task",
            "Paradigm",
            "Method",
            "Algorithm_Key",
            "Algorithm",
            "Config_ID",
            "N",
            "MAE",
            "RMSE",
            "R2",
            "MBE",
            "NRMSE_Std",
            "Task_Rank",
        ]
    ]
    .sort_values(
        [
            "Task",
            "Task_Rank",
        ]
    )
    .round(6)
)


print(
    "\nTEST winner for every task:"
)

display(
    PARADIGM15_TASK_WINNERS_DF[
        [
            "Task",
            "Target",
            "Horizon",
            "Paradigm",
            "Method",
            "Algorithm",
            "Config_ID",
            "MAE",
            "RMSE",
            "R2",
        ]
    ]
    .round(6)
)


print(
    "\nOverall TEST paradigm summary:"
)

display(
    PARADIGM15_OVERALL_DF
    .round(6)
)


print(
    "\nMAE improvement relative to Local models:"
)

display(
    PARADIGM15_IMPROVEMENT_VS_LOCAL_DF
    .round(6)
)


print(
    "\nCELL 15 STATUS:"
)

print(
    "No new models were trained."
)

print(
    "No hyperparameter tuning was performed."
)

print(
    f"Federated method remained frozen as: "
    f"{SELECTED_FEDERATED_METHOD}"
)

print(
    "The same task-specific algorithm was used "
    "for Centralized, Local, and Federated."
)

print(
    "Test was used only for final comparative evaluation."
)

print(
    "Test rankings are descriptive and do not alter "
    "any previous selection."
)


print(
    f"\nCell 15 completed in "
    f"{time.time() - CELL_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 16 — UNIFIED FINAL UTW-RG-AdaFedResidual PIPELINE
# FINAL CORRECTED VERSION
# Peer-only Relation Guidance + Adaptive Gate + Delayed Feedback
# Depends ONLY on Cells 1–15
# ==================================================================================================

import time
import numpy as np
import pandas as pd

CELL16_START = time.time()

print("=" * 130)
print("UNIFIED CELL 16 — FINAL EXPERIMENTAL PIPELINE")
print("UTW-RG-AdaFedResidual | PEER-ONLY RELATION + DELAYED FEEDBACK")
print("=" * 130)


# ==================================================================================================
# 1. REQUIRED OBJECTS — ONLY FROM CELLS 1–15
# ==================================================================================================

required_objects = [
    "TASK_DEFINITIONS",
    "TASK_DATASETS",
    "TASK_CLIENT_DATASETS",
    "EXPECTED_STATIONS",
    "ACTIVE_FEATURE_COLUMNS",
    "FEDERATED_RUNS",
    "LOCAL_MODELS",
    "CENTRALIZED_MODELS",
    "predict_raw",
    "reg_metrics",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise NameError(
        "Missing required objects from Cells 1–15:\n- "
        + "\n- ".join(missing)
    )

print("✓ Required Cells 1–15 objects are available.")


# ==================================================================================================
# 2. RESOLVE FINAL FEDERATED METHOD
# ==================================================================================================

if "SELECTED_FEDERATED_METHOD" in globals():
    U1619_FED_METHOD = SELECTED_FEDERATED_METHOD
else:
    U1619_FED_METHOD = "FedAdam"

if U1619_FED_METHOD != "FedAdam":
    print(
        f"Warning: selected federated method is {U1619_FED_METHOD}; "
        "final accepted protocol expects FedAdam."
    )

print(f"✓ Federated backbone: {U1619_FED_METHOD}")


# ==================================================================================================
# 3. RESOLVE FROZEN SELECTED ALGORITHM FOR EACH TASK
# ==================================================================================================

def u1619_selected_algorithm(task_key):

    candidate_tables = [
        "FEDERATED_SELECTED_ALGORITHM_DF",
        "FINAL_FEDERATED_SELECTION_DF",
        "FEDERATED_FINAL_SELECTION_DF",
    ]

    for table_name in candidate_tables:

        if table_name not in globals():
            continue

        df = globals()[table_name]

        if not isinstance(df, pd.DataFrame):
            continue

        if "Task" not in df.columns:
            continue

        q = df[df["Task"].eq(task_key)]

        if len(q) == 0:
            continue

        for col in [
            "Algorithm_Key",
            "Selected_Algorithm_Key",
            "Selected_Algorithm",
        ]:
            if col in q.columns:
                value = q.iloc[0][col]

                if value in {
                    "Ridge",
                    "ElasticNet",
                    "SGD_Linear",
                    "SGD_Huber",
                }:
                    return value

    # Robust fallback from frozen Cell-14 accepted choices
    frozen = {
        "Alum_H1": "SGD_Huber",
        "Alum_H2": "SGD_Huber",
        "Alum_H3": "SGD_Huber",
        "Chlorine_H1": "SGD_Linear",
        "Chlorine_H2": "ElasticNet",
        "Chlorine_H3": "SGD_Linear",
    }

    if task_key not in frozen:
        raise KeyError(
            f"No frozen selected algorithm found for {task_key}"
        )

    return frozen[task_key]


# ==================================================================================================
# 4. RELATION FEATURE SUBSET
# ==================================================================================================

U1619_RELATION_FEATURE_IDX = [
    i
    for i, col in enumerate(ACTIVE_FEATURE_COLUMNS)
    if not str(col).startswith("Station_")
]

if len(U1619_RELATION_FEATURE_IDX) == 0:
    raise RuntimeError(
        "No non-station features available for relation guidance."
    )


# ==================================================================================================
# 5. BASIC HELPERS
# ==================================================================================================

def u1619_ewm(values, decay):

    arr = np.asarray(values, dtype=float)

    if len(arr) == 0:
        return np.nan

    weights = (
        float(decay)
        ** np.arange(
            len(arr) - 1,
            -1,
            -1,
            dtype=float,
        )
    )

    return float(
        np.sum(weights * arr)
        / np.sum(weights)
    )


def u1619_softmax_negative_distance(
    distances,
    temperature=1.0,
):

    d = np.asarray(
        distances,
        dtype=float,
    )

    z = (
        -d
        / max(
            float(temperature),
            1e-12,
        )
    )

    z -= np.max(z)

    exp_z = np.exp(z)

    return (
        exp_z
        / np.sum(exp_z)
    )


# ==================================================================================================
# 6. PEER-ONLY RELATION WEIGHTS
# IMPORTANT:
#   target station is excluded completely from Relation Guidance.
# ==================================================================================================

def u1619_peer_weights(
    task_key,
    target_station,
):

    source_stations = [
        station
        for station in EXPECTED_STATIONS
        if station != target_station
    ]

    if len(source_stations) == 0:
        raise RuntimeError(
            "Peer relation requires at least one source station."
        )

    centroids = {}

    for station in EXPECTED_STATIONS:

        X = np.asarray(
            TASK_CLIENT_DATASETS[
                task_key
            ][station]["Train"]["X"],
            dtype=float,
        )

        X = X[
            :,
            U1619_RELATION_FEATURE_IDX
        ]

        centroids[station] = np.mean(
            X,
            axis=0,
        )

    target_centroid = centroids[
        target_station
    ]

    distances = []

    for station in source_stations:

        distance = np.sqrt(
            np.mean(
                (
                    target_centroid
                    - centroids[station]
                )
                ** 2
            )
        )

        distances.append(
            float(distance)
        )

    weights = (
        u1619_softmax_negative_distance(
            distances,
            temperature=1.0,
        )
    )

    return {
        station: float(weight)
        for station, weight
        in zip(
            source_stations,
            weights,
        )
    }


# ==================================================================================================
# 7. STATIC EXPERT PREDICTIONS
# ==================================================================================================

def u1619_static_predictions(
    task_key,
    station,
    split_name,
):

    algorithm_key = (
        u1619_selected_algorithm(
            task_key
        )
    )

    dataset = (
        TASK_CLIENT_DATASETS[
            task_key
        ][station][split_name]
    )

    X = np.asarray(
        dataset["X"],
        dtype=float,
    )

    # ----------------------------------------------------------------------------------------------
    # Global frozen FedAdam expert
    # ----------------------------------------------------------------------------------------------

    fed_key = (
        task_key,
        algorithm_key,
        U1619_FED_METHOD,
    )

    if fed_key not in FEDERATED_RUNS:
        raise KeyError(
            f"Missing federated model: {fed_key}"
        )

    global_parameters = (
        FEDERATED_RUNS[
            fed_key
        ]["Parameters"]
    )

    global_prediction = predict_raw(
        X,
        global_parameters,
        task_key,
    )

    # ----------------------------------------------------------------------------------------------
    # Self-local frozen expert
    # ----------------------------------------------------------------------------------------------

    self_key = (
        task_key,
        station,
        algorithm_key,
    )

    if self_key not in LOCAL_MODELS:
        raise KeyError(
            f"Missing local model: {self_key}"
        )

    self_parameters = (
        LOCAL_MODELS[
            self_key
        ]["Parameters"]
    )

    self_prediction = predict_raw(
        X,
        self_parameters,
        task_key,
    )

    # ----------------------------------------------------------------------------------------------
    # Peer-only Relation Guidance
    # ----------------------------------------------------------------------------------------------

    peer_weights = (
        u1619_peer_weights(
            task_key,
            station,
        )
    )

    peer_prediction = np.zeros(
        len(X),
        dtype=float,
    )

    for peer_station, weight in (
        peer_weights.items()
    ):

        peer_key = (
            task_key,
            peer_station,
            algorithm_key,
        )

        peer_parameters = (
            LOCAL_MODELS[
                peer_key
            ]["Parameters"]
        )

        peer_pred = predict_raw(
            X,
            peer_parameters,
            task_key,
        )

        peer_prediction += (
            float(weight)
            * np.asarray(
                peer_pred,
                dtype=float,
            )
        )

    return {
        "Global": np.asarray(
            global_prediction,
            dtype=float,
        ),
        "Self": np.asarray(
            self_prediction,
            dtype=float,
        ),
        "Relation": np.asarray(
            peer_prediction,
            dtype=float,
        ),
        "Peer_Weights":
            peer_weights,
    }


# ==================================================================================================
# 8. FINAL ONLINE ENGINE
#
# Delayed feedback:
# An observation becomes usable only when:
#
#       Target_Date <= current Date
#
# Relation gate:
#
#       Relation active if historical peer-relation error
#       is lower than historical self-local error.
#
# No new gate threshold is tuned.
# ==================================================================================================

def u1619_run_online(
    dates,
    target_dates,
    y,
    global_pred,
    self_pred,
    relation_pred,
    prior_global,
    expert_power,
    lookback,
    decay,
    alpha,
    local_relation_mix,
    use_relation=True,
    use_adaptive=True,
    use_residual=True,
):

    dates = pd.to_datetime(
        pd.Series(dates)
    ).reset_index(drop=True)

    target_dates = pd.to_datetime(
        pd.Series(target_dates)
    ).reset_index(drop=True)

    y = np.asarray(
        y,
        dtype=float,
    )

    global_pred = np.asarray(
        global_pred,
        dtype=float,
    )

    self_pred = np.asarray(
        self_pred,
        dtype=float,
    )

    relation_pred = np.asarray(
        relation_pred,
        dtype=float,
    )

    n = len(y)

    output = np.empty(
        n,
        dtype=float,
    )

    global_weights = np.empty(
        n,
        dtype=float,
    )

    relation_active = np.zeros(
        n,
        dtype=bool,
    )

    # Matured histories only
    global_errors = []
    local_errors = []
    self_errors = []
    relation_errors = []
    residual_history = []

    # Predictions awaiting target release
    pending = []

    for i in range(n):

        current_date = dates.iloc[i]

        # ------------------------------------------------------------------------------------------
        # Release only observations whose target is now known
        # ------------------------------------------------------------------------------------------

        still_pending = []

        for item in pending:

            if item["Target_Date"] <= current_date:

                global_errors.append(
                    abs(
                        item["Actual"]
                        - item["Global"]
                    )
                )

                local_errors.append(
                    abs(
                        item["Actual"]
                        - item["Local"]
                    )
                )

                self_errors.append(
                    abs(
                        item["Actual"]
                        - item["Self"]
                    )
                )

                relation_errors.append(
                    abs(
                        item["Actual"]
                        - item["Relation"]
                    )
                )

                residual_history.append(
                    item["Actual"]
                    - item["Base"]
                )

            else:
                still_pending.append(
                    item
                )

        pending = still_pending

        # ------------------------------------------------------------------------------------------
        # Adaptive Relation Gate
        # ------------------------------------------------------------------------------------------

        gate = False

        if (
            use_relation
            and len(self_errors) > 0
            and len(relation_errors) > 0
        ):

            self_error = u1619_ewm(
                self_errors[
                    -int(lookback):
                ],
                decay,
            )

            peer_error = u1619_ewm(
                relation_errors[
                    -int(lookback):
                ],
                decay,
            )

            # No tuned threshold:
            # activate peer relation only when historically better than self-local.
            gate = (
                peer_error
                < self_error
            )

        relation_active[i] = gate

        # ------------------------------------------------------------------------------------------
        # Self-local remains a separate expert.
        # Relation contribution is activated only when gate=True.
        # ------------------------------------------------------------------------------------------

        if gate:

            local_expert = (
                float(local_relation_mix)
                * self_pred[i]
                + (
                    1.0
                    - float(
                        local_relation_mix
                    )
                )
                * relation_pred[i]
            )

        else:

            local_expert = (
                self_pred[i]
            )

        # ------------------------------------------------------------------------------------------
        # Adaptive Global / Local weighting
        # ------------------------------------------------------------------------------------------

        if (
            use_adaptive
            and len(global_errors) > 0
            and len(local_errors) > 0
        ):

            eg = u1619_ewm(
                global_errors[
                    -int(lookback):
                ],
                decay,
            )

            el = u1619_ewm(
                local_errors[
                    -int(lookback):
                ],
                decay,
            )

            inv_g = (
                1.0
                / (
                    max(
                        eg,
                        1e-12,
                    )
                    ** float(
                        expert_power
                    )
                )
            )

            inv_l = (
                1.0
                / (
                    max(
                        el,
                        1e-12,
                    )
                    ** float(
                        expert_power
                    )
                )
            )

            data_weight_global = (
                inv_g
                / (
                    inv_g
                    + inv_l
                )
            )

            global_weight = (
                0.5
                * float(
                    prior_global
                )
                + 0.5
                * data_weight_global
            )

        else:

            global_weight = float(
                prior_global
            )

        global_weights[i] = (
            global_weight
        )

        base_prediction = (
            global_weight
            * global_pred[i]
            + (
                1.0
                - global_weight
            )
            * local_expert
        )

        # ------------------------------------------------------------------------------------------
        # Residual correction — matured feedback only
        # ------------------------------------------------------------------------------------------

        if (
            use_residual
            and len(
                residual_history
            ) > 0
        ):

            correction = (
                u1619_ewm(
                    residual_history[
                        -int(lookback):
                    ],
                    decay,
                )
            )

        else:

            correction = 0.0

        if use_residual:

            prediction = (
                base_prediction
                + float(alpha)
                * correction
            )

        else:

            prediction = (
                base_prediction
            )

        output[i] = prediction

        # ------------------------------------------------------------------------------------------
        # Add prediction to pending queue.
        # Its true outcome CANNOT influence the current prediction.
        # ------------------------------------------------------------------------------------------

        pending.append({
            "Target_Date":
                target_dates.iloc[i],

            "Actual":
                float(y[i]),

            "Global":
                float(
                    global_pred[i]
                ),

            "Local":
                float(
                    local_expert
                ),

            "Self":
                float(
                    self_pred[i]
                ),

            "Relation":
                float(
                    relation_pred[i]
                ),

            "Base":
                float(
                    base_prediction
                ),
        })

    state = {
        "Global_Weights":
            global_weights,

        "Relation_Active":
            relation_active,

        "Global_Errors":
            global_errors,

        "Local_Errors":
            local_errors,

        "Self_Errors":
            self_errors,

        "Relation_Errors":
            relation_errors,

        "Residual_History":
            residual_history,
    }

    return output, state


# ==================================================================================================
# 9. EXACTLY FIVE LOGICAL NOVEL CONFIGURATIONS
# Validation only
# ==================================================================================================

U1619_CONFIGS = {
    "RG1": {
        "prior_global": 0.60,
        "expert_power": 0.50,
        "lookback": 20,
        "decay": 0.90,
        "alpha": 0.50,
        "local_relation_mix": 0.50,
    },

    "RG2": {
        "prior_global": 0.70,
        "expert_power": 0.50,
        "lookback": 40,
        "decay": 0.95,
        "alpha": 0.50,
        "local_relation_mix": 0.50,
    },

    "RG3": {
        "prior_global": 0.80,
        "expert_power": 0.75,
        "lookback": 60,
        "decay": 0.95,
        "alpha": 0.50,
        "local_relation_mix": 0.75,
    },

    "RG4": {
        "prior_global": 0.70,
        "expert_power": 1.00,
        "lookback": 40,
        "decay": 0.90,
        "alpha": 1.00,
        "local_relation_mix": 0.75,
    },

    "RG5": {
        "prior_global": 0.90,
        "expert_power": 1.00,
        "lookback": 60,
        "decay": 0.95,
        "alpha": 1.00,
        "local_relation_mix": 0.50,
    },
}

U1619_CANDIDATE_GRID_DF = pd.DataFrame([
    {
        "Config":
            name,
        **cfg,
    }
    for name, cfg
    in U1619_CONFIGS.items()
])


# ==================================================================================================
# 10. VALIDATION-ONLY CONFIGURATION SELECTION
# ==================================================================================================

selection_rows = []

task_keys = [
    definition["Task"]
    for definition in TASK_DEFINITIONS
]

for task_key in task_keys:

    meta = (
        TASK_DATASETS[
            task_key
        ]["Meta"]
    )

    for station in EXPECTED_STATIONS:

        validation_ds = (
            TASK_CLIENT_DATASETS[
                task_key
            ][station]["Validation"]
        )

        validation_frame = (
            validation_ds["Frame"]
            .reset_index(drop=True)
        )

        experts = (
            u1619_static_predictions(
                task_key,
                station,
                "Validation",
            )
        )

        best = None

        for config_name, cfg in (
            U1619_CONFIGS.items()
        ):

            pred, state = (
                u1619_run_online(
                    dates=
                        validation_frame[
                            "Date"
                        ],

                    target_dates=
                        validation_frame[
                            meta[
                                "Future_Date_Column"
                            ]
                        ],

                    y=
                        validation_ds[
                            "y"
                        ],

                    global_pred=
                        experts[
                            "Global"
                        ],

                    self_pred=
                        experts[
                            "Self"
                        ],

                    relation_pred=
                        experts[
                            "Relation"
                        ],

                    **cfg,
                )
            )

            metrics = reg_metrics(
                validation_ds["y"],
                pred,
            )

            score = (
                metrics["MAE"],
                metrics["RMSE"],
                -metrics["R2"],
            )

            if (
                best is None
                or score < best["Score"]
            ):

                best = {
                    "Score":
                        score,

                    "Config":
                        config_name,

                    "Config_Data":
                        cfg,

                    "Metrics":
                        metrics,

                    "Relation_Activation":
                        100.0
                        * float(
                            np.mean(
                                state[
                                    "Relation_Active"
                                ]
                            )
                        ),
                }

        cfg = best[
            "Config_Data"
        ]

        selection_rows.append({
            "Task":
                task_key,

            "Target":
                meta["Target"],

            "Horizon":
                meta["Horizon"],

            "Station":
                station,

            "Selected_Config":
                best["Config"],

            **cfg,

            "Validation_MAE":
                best[
                    "Metrics"
                ]["MAE"],

            "Validation_RMSE":
                best[
                    "Metrics"
                ]["RMSE"],

            "Validation_R2":
                best[
                    "Metrics"
                ]["R2"],

            "Validation_Relation_Activation_Percent":
                best[
                    "Relation_Activation"
                ],

            "Selection_Source":
                "Validation only",
        })


U1619_SELECTION_DF = pd.DataFrame(
    selection_rows
)

# Compatibility alias
RGADA16_SELECTION_DF = (
    U1619_SELECTION_DF
)


# ==================================================================================================
# 11. FINAL CONTINUOUS VALIDATION → TEST REPLAY
#
# CRITICAL:
# Validation and Test are concatenated FIRST.
# The online engine is called ONCE.
# Then Test predictions are sliced.
# ==================================================================================================

prediction_rows = []

for task_key in task_keys:

    meta = (
        TASK_DATASETS[
            task_key
        ]["Meta"]
    )

    for station in EXPECTED_STATIONS:

        selected = (
            U1619_SELECTION_DF[
                U1619_SELECTION_DF[
                    "Task"
                ].eq(task_key)
                &
                U1619_SELECTION_DF[
                    "Station"
                ].eq(station)
            ]
            .iloc[0]
        )

        cfg = {
            "prior_global":
                float(
                    selected[
                        "prior_global"
                    ]
                ),

            "expert_power":
                float(
                    selected[
                        "expert_power"
                    ]
                ),

            "lookback":
                int(
                    selected[
                        "lookback"
                    ]
                ),

            "decay":
                float(
                    selected[
                        "decay"
                    ]
                ),

            "alpha":
                float(
                    selected[
                        "alpha"
                    ]
                ),

            "local_relation_mix":
                float(
                    selected[
                        "local_relation_mix"
                    ]
                ),
        }

        validation_ds = (
            TASK_CLIENT_DATASETS[
                task_key
            ][station]["Validation"]
        )

        test_ds = (
            TASK_CLIENT_DATASETS[
                task_key
            ][station]["Test"]
        )

        validation_frame = (
            validation_ds[
                "Frame"
            ].reset_index(
                drop=True
            )
        )

        test_frame = (
            test_ds[
                "Frame"
            ].reset_index(
                drop=True
            )
        )

        validation_experts = (
            u1619_static_predictions(
                task_key,
                station,
                "Validation",
            )
        )

        test_experts = (
            u1619_static_predictions(
                task_key,
                station,
                "Test",
            )
        )

        all_dates = pd.concat(
            [
                validation_frame[
                    "Date"
                ],
                test_frame[
                    "Date"
                ],
            ],
            ignore_index=True,
        )

        all_target_dates = pd.concat(
            [
                validation_frame[
                    meta[
                        "Future_Date_Column"
                    ]
                ],
                test_frame[
                    meta[
                        "Future_Date_Column"
                    ]
                ],
            ],
            ignore_index=True,
        )

        all_y = np.concatenate(
            [
                np.asarray(
                    validation_ds["y"],
                    dtype=float,
                ),
                np.asarray(
                    test_ds["y"],
                    dtype=float,
                ),
            ]
        )

        all_global = np.concatenate(
            [
                validation_experts[
                    "Global"
                ],
                test_experts[
                    "Global"
                ],
            ]
        )

        all_self = np.concatenate(
            [
                validation_experts[
                    "Self"
                ],
                test_experts[
                    "Self"
                ],
            ]
        )

        all_relation = np.concatenate(
            [
                validation_experts[
                    "Relation"
                ],
                test_experts[
                    "Relation"
                ],
            ]
        )

        all_pred, all_state = (
            u1619_run_online(
                dates=
                    all_dates,

                target_dates=
                    all_target_dates,

                y=
                    all_y,

                global_pred=
                    all_global,

                self_pred=
                    all_self,

                relation_pred=
                    all_relation,

                **cfg,
            )
        )

        n_validation = len(
            validation_ds["y"]
        )

        test_prediction = (
            all_pred[
                n_validation:
            ]
        )

        test_global_weight = (
            all_state[
                "Global_Weights"
            ][
                n_validation:
            ]
        )

        test_relation_active = (
            all_state[
                "Relation_Active"
            ][
                n_validation:
            ]
        )

        for i in range(
            len(test_prediction)
        ):

            prediction_rows.append({
                "Task":
                    task_key,

                "Target":
                    meta["Target"],

                "Horizon":
                    meta["Horizon"],

                "Station":
                    station,

                "Date":
                    test_frame.loc[
                        i,
                        "Date",
                    ],

                "Target_Date":
                    test_frame.loc[
                        i,
                        meta[
                            "Future_Date_Column"
                        ],
                    ],

                "Actual":
                    float(
                        test_ds[
                            "y"
                        ][i]
                    ),

                "RGAdaFedResidual_Pred":
                    float(
                        test_prediction[i]
                    ),

                "FedAdam_Pred":
                    float(
                        test_experts[
                            "Global"
                        ][i]
                    ),

                "SelfLocal_Pred":
                    float(
                        test_experts[
                            "Self"
                        ][i]
                    ),

                "RelationLocal_Pred":
                    float(
                        test_experts[
                            "Relation"
                        ][i]
                    ),

                "Adaptive_Global_Weight":
                    float(
                        test_global_weight[
                            i
                        ]
                    ),

                "Relation_Active":
                    bool(
                        test_relation_active[
                            i
                        ]
                    ),

                "Selected_Config":
                    selected[
                        "Selected_Config"
                    ],
            })


U1619_TEST_PREDICTIONS_DF = (
    pd.DataFrame(
        prediction_rows
    )
)

# Compatibility variables required by later final cells
RGADA16_TEST_PREDICTIONS_DF = (
    U1619_TEST_PREDICTIONS_DF
)

RGADA18_TEST_PREDICTIONS_DF = (
    U1619_TEST_PREDICTIONS_DF.copy()
)


# ==================================================================================================
# 12. FINAL NOVEL TEST RESULTS — PER TASK
# ==================================================================================================

novel_task_rows = []

for task_key in task_keys:

    q = (
        U1619_TEST_PREDICTIONS_DF[
            U1619_TEST_PREDICTIONS_DF[
                "Task"
            ].eq(task_key)
        ]
    )

    metrics = reg_metrics(
        q["Actual"],
        q[
            "RGAdaFedResidual_Pred"
        ],
    )

    novel_task_rows.append({
        "Task":
            task_key,

        "Target":
            q["Target"].iloc[0],

        "Horizon":
            int(
                q[
                    "Horizon"
                ].iloc[0]
            ),

        **metrics,

        "Relation_Activation_Percent":
            100.0
            * float(
                q[
                    "Relation_Active"
                ].mean()
            ),
    })


U1619_NOVEL_TEST_RESULTS_DF = (
    pd.DataFrame(
        novel_task_rows
    )
)

RGADA16_TEST_RESULTS_DF = (
    U1619_NOVEL_TEST_RESULTS_DF
)


# ==================================================================================================
# 13. STATIC BASELINE TEST PREDICTIONS
# Centralized / Local / FedAdam
# No retuning.
# ==================================================================================================

baseline_prediction_rows = []

for task_key in task_keys:

    meta = (
        TASK_DATASETS[
            task_key
        ]["Meta"]
    )

    algorithm_key = (
        u1619_selected_algorithm(
            task_key
        )
    )

    for station in EXPECTED_STATIONS:

        test_ds = (
            TASK_CLIENT_DATASETS[
                task_key
            ][station]["Test"]
        )

        frame = (
            test_ds[
                "Frame"
            ].reset_index(
                drop=True
            )
        )

        X = np.asarray(
            test_ds["X"],
            dtype=float,
        )

        # Centralized
        central_params = (
            CENTRALIZED_MODELS[
                (
                    task_key,
                    algorithm_key,
                )
            ][
                "Parameters"
            ]
        )

        central_pred = predict_raw(
            X,
            central_params,
            task_key,
        )

        # Local
        local_params = (
            LOCAL_MODELS[
                (
                    task_key,
                    station,
                    algorithm_key,
                )
            ][
                "Parameters"
            ]
        )

        local_pred = predict_raw(
            X,
            local_params,
            task_key,
        )

        # FedAdam
        fed_params = (
            FEDERATED_RUNS[
                (
                    task_key,
                    algorithm_key,
                    U1619_FED_METHOD,
                )
            ][
                "Parameters"
            ]
        )

        fed_pred = predict_raw(
            X,
            fed_params,
            task_key,
        )

        novel_q = (
            U1619_TEST_PREDICTIONS_DF[
                U1619_TEST_PREDICTIONS_DF[
                    "Task"
                ].eq(task_key)
                &
                U1619_TEST_PREDICTIONS_DF[
                    "Station"
                ].eq(station)
            ]
            .reset_index(
                drop=True
            )
        )

        models = {
            "Centralized":
                central_pred,

            "Local":
                local_pred,

            "FedAdam":
                fed_pred,

            "UTW-RG-AdaFedResidual":
                novel_q[
                    "RGAdaFedResidual_Pred"
                ].to_numpy(),
        }

        for model_name, prediction in (
            models.items()
        ):

            for i, pred in enumerate(
                prediction
            ):

                baseline_prediction_rows.append({
                    "Task":
                        task_key,

                    "Target":
                        meta["Target"],

                    "Horizon":
                        meta["Horizon"],

                    "Station":
                        station,

                    "Date":
                        frame.loc[
                            i,
                            "Date",
                        ],

                    "Target_Date":
                        frame.loc[
                            i,
                            meta[
                                "Future_Date_Column"
                            ],
                        ],

                    "Actual":
                        float(
                            test_ds[
                                "y"
                            ][i]
                        ),

                    "Model":
                        model_name,

                    "Prediction":
                        float(pred),
                })


U1619_ALL_MODEL_TEST_PREDICTIONS_DF = (
    pd.DataFrame(
        baseline_prediction_rows
    )
)


# ==================================================================================================
# 14. TASK-LEVEL MODEL COMPARISON
# ==================================================================================================

comparison_rows = []

for (
    task_key,
    model_name,
), group in (
    U1619_ALL_MODEL_TEST_PREDICTIONS_DF
    .groupby(
        [
            "Task",
            "Model",
        ]
    )
):

    metrics = reg_metrics(
        group["Actual"],
        group["Prediction"],
    )

    comparison_rows.append({
        "Task":
            task_key,

        "Target":
            group[
                "Target"
            ].iloc[0],

        "Horizon":
            int(
                group[
                    "Horizon"
                ].iloc[0]
            ),

        "Model":
            model_name,

        **metrics,
    })


U1619_TASK_COMPARISON_DF = (
    pd.DataFrame(
        comparison_rows
    )
)

for metric, ascending in [
    ("MAE", True),
    ("RMSE", True),
    ("R2", False),
]:

    U1619_TASK_COMPARISON_DF[
        f"{metric}_Rank"
    ] = (
        U1619_TASK_COMPARISON_DF
        .groupby(
            "Task"
        )[metric]
        .rank(
            method="average",
            ascending=ascending,
        )
    )

U1619_TASK_COMPARISON_DF[
    "Mean_Metric_Rank"
] = (
    U1619_TASK_COMPARISON_DF[
        [
            "MAE_Rank",
            "RMSE_Rank",
            "R2_Rank",
        ]
    ]
    .mean(
        axis=1
    )
)

U1619_TASK_COMPARISON_DF[
    "Task_Winner"
] = (
    U1619_TASK_COMPARISON_DF
    .groupby(
        "Task"
    )[
        "Mean_Metric_Rank"
    ]
    .transform(
        "min"
    )
    .eq(
        U1619_TASK_COMPARISON_DF[
            "Mean_Metric_Rank"
        ]
    )
)


U1619_OVERALL_COMPARISON_DF = (
    U1619_TASK_COMPARISON_DF
    .groupby(
        "Model",
        as_index=False,
    )
    .agg(
        Tasks=(
            "Task",
            "count",
        ),

        Mean_MAE=(
            "MAE",
            "mean",
        ),

        Mean_RMSE=(
            "RMSE",
            "mean",
        ),

        Mean_R2=(
            "R2",
            "mean",
        ),

        Mean_NRMSE_Std=(
            "NRMSE_Std",
            "mean",
        ),

        Mean_Abs_MBE=(
            "MBE",
            lambda x:
                np.mean(
                    np.abs(x)
                ),
        ),

        Mean_Metric_Rank=(
            "Mean_Metric_Rank",
            "mean",
        ),

        Task_Wins=(
            "Task_Winner",
            "sum",
        ),
    )
)

U1619_OVERALL_COMPARISON_DF = (
    U1619_OVERALL_COMPARISON_DF
    .sort_values(
        [
            "Mean_Metric_Rank",
            "Mean_MAE",
        ]
    )
    .reset_index(
        drop=True
    )
)

U1619_OVERALL_COMPARISON_DF[
    "Overall_Rank"
] = np.arange(
    1,
    len(
        U1619_OVERALL_COMPARISON_DF
    )
    + 1,
)


# ==================================================================================================
# 15. ABLATION — SAME SELECTED CONFIGURATION, NO RETUNING
# ==================================================================================================

U1619_ABLATION_VARIANTS = {
    "Full UTW-RG-AdaFedResidual":
        {
            "use_relation": True,
            "use_adaptive": True,
            "use_residual": True,
        },

    "Without Relation Guidance":
        {
            "use_relation": False,
            "use_adaptive": True,
            "use_residual": True,
        },

    "Without Adaptive Expert Weighting":
        {
            "use_relation": True,
            "use_adaptive": False,
            "use_residual": True,
        },

    "Without Residual Correction":
        {
            "use_relation": True,
            "use_adaptive": True,
            "use_residual": False,
        },
}

ablation_rows = []

for task_key in task_keys:

    meta = (
        TASK_DATASETS[
            task_key
        ]["Meta"]
    )

    for station in EXPECTED_STATIONS:

        selected = (
            U1619_SELECTION_DF[
                U1619_SELECTION_DF[
                    "Task"
                ].eq(task_key)
                &
                U1619_SELECTION_DF[
                    "Station"
                ].eq(station)
            ]
            .iloc[0]
        )

        cfg = {
            "prior_global":
                float(
                    selected[
                        "prior_global"
                    ]
                ),

            "expert_power":
                float(
                    selected[
                        "expert_power"
                    ]
                ),

            "lookback":
                int(
                    selected[
                        "lookback"
                    ]
                ),

            "decay":
                float(
                    selected[
                        "decay"
                    ]
                ),

            "alpha":
                float(
                    selected[
                        "alpha"
                    ]
                ),

            "local_relation_mix":
                float(
                    selected[
                        "local_relation_mix"
                    ]
                ),
        }

        validation_ds = (
            TASK_CLIENT_DATASETS[
                task_key
            ][station][
                "Validation"
            ]
        )

        test_ds = (
            TASK_CLIENT_DATASETS[
                task_key
            ][station][
                "Test"
            ]
        )

        validation_frame = (
            validation_ds[
                "Frame"
            ].reset_index(
                drop=True
            )
        )

        test_frame = (
            test_ds[
                "Frame"
            ].reset_index(
                drop=True
            )
        )

        ve = (
            u1619_static_predictions(
                task_key,
                station,
                "Validation",
            )
        )

        te = (
            u1619_static_predictions(
                task_key,
                station,
                "Test",
            )
        )

        all_dates = pd.concat(
            [
                validation_frame[
                    "Date"
                ],
                test_frame[
                    "Date"
                ],
            ],
            ignore_index=True,
        )

        all_target_dates = pd.concat(
            [
                validation_frame[
                    meta[
                        "Future_Date_Column"
                    ]
                ],
                test_frame[
                    meta[
                        "Future_Date_Column"
                    ]
                ],
            ],
            ignore_index=True,
        )

        all_y = np.concatenate(
            [
                validation_ds[
                    "y"
                ],
                test_ds[
                    "y"
                ],
            ]
        )

        all_g = np.concatenate(
            [
                ve["Global"],
                te["Global"],
            ]
        )

        all_s = np.concatenate(
            [
                ve["Self"],
                te["Self"],
            ]
        )

        all_r = np.concatenate(
            [
                ve["Relation"],
                te["Relation"],
            ]
        )

        n_validation = len(
            validation_ds["y"]
        )

        for (
            variant_name,
            flags,
        ) in (
            U1619_ABLATION_VARIANTS
            .items()
        ):

            pred, _ = (
                u1619_run_online(
                    dates=
                        all_dates,

                    target_dates=
                        all_target_dates,

                    y=
                        all_y,

                    global_pred=
                        all_g,

                    self_pred=
                        all_s,

                    relation_pred=
                        all_r,

                    **cfg,
                    **flags,
                )
            )

            test_pred = (
                pred[
                    n_validation:
                ]
            )

            for i, pp in enumerate(
                test_pred
            ):

                ablation_rows.append({
                    "Task":
                        task_key,

                    "Target":
                        meta["Target"],

                    "Horizon":
                        meta["Horizon"],

                    "Station":
                        station,

                    "Variant":
                        variant_name,

                    "Actual":
                        float(
                            test_ds[
                                "y"
                            ][i]
                        ),

                    "Prediction":
                        float(pp),
                })


U1619_ABLATION_PREDICTIONS_DF = (
    pd.DataFrame(
        ablation_rows
    )
)

ablation_task_rows = []

for (
    task_key,
    variant,
), group in (
    U1619_ABLATION_PREDICTIONS_DF
    .groupby(
        [
            "Task",
            "Variant",
        ]
    )
):

    metrics = reg_metrics(
        group["Actual"],
        group["Prediction"],
    )

    ablation_task_rows.append({
        "Task":
            task_key,

        "Variant":
            variant,

        **metrics,
    })


U1619_ABLATION_TASK_DF = (
    pd.DataFrame(
        ablation_task_rows
    )
)

U1619_ABLATION_SUMMARY_DF = (
    U1619_ABLATION_TASK_DF
    .groupby(
        "Variant",
        as_index=False,
    )
    .agg(
        Tasks=(
            "Task",
            "count",
        ),

        Mean_MAE=(
            "MAE",
            "mean",
        ),

        Mean_RMSE=(
            "RMSE",
            "mean",
        ),

        Mean_R2=(
            "R2",
            "mean",
        ),

        Mean_NRMSE_Std=(
            "NRMSE_Std",
            "mean",
        ),
    )
)


# ==================================================================================================
# 16. COMPONENT EFFECT
# ==================================================================================================

full_row = (
    U1619_ABLATION_SUMMARY_DF[
        U1619_ABLATION_SUMMARY_DF[
            "Variant"
        ].eq(
            "Full UTW-RG-AdaFedResidual"
        )
    ]
    .iloc[0]
)

component_rows = []

for _, row in (
    U1619_ABLATION_SUMMARY_DF
    .iterrows()
):

    if (
        row["Variant"]
        == "Full UTW-RG-AdaFedResidual"
    ):
        continue

    component_rows.append({
        "Removed_Component":
            row[
                "Variant"
            ].replace(
                "Without ",
                "",
            ),

        "MAE_Degradation_Percent":
            100.0
            * (
                row[
                    "Mean_MAE"
                ]
                - full_row[
                    "Mean_MAE"
                ]
            )
            / full_row[
                "Mean_MAE"
            ],

        "RMSE_Degradation_Percent":
            100.0
            * (
                row[
                    "Mean_RMSE"
                ]
                - full_row[
                    "Mean_RMSE"
                ]
            )
            / full_row[
                "Mean_RMSE"
            ],

        "R2_Loss":
            (
                full_row[
                    "Mean_R2"
                ]
                - row[
                    "Mean_R2"
                ]
            ),
    })


U1619_COMPONENT_EFFECT_DF = (
    pd.DataFrame(
        component_rows
    )
)


# ==================================================================================================
# 17. PROTOCOL
# ==================================================================================================

U1619_PROTOCOL_DF = pd.DataFrame([
    {
        "Item":
            "Method",
        "Value":
            "UTW-RG-AdaFedResidual",
    },
    {
        "Item":
            "UTW",
        "Value":
            "W=10 frozen",
    },
    {
        "Item":
            "Novel configurations",
        "Value":
            5,
    },
    {
        "Item":
            "Hyperparameter selection",
        "Value":
            "Validation only",
    },
    {
        "Item":
            "Primary selection criterion",
        "Value":
            "MAE -> RMSE -> R2",
    },
    {
        "Item":
            "Federated backbone",
        "Value":
            U1619_FED_METHOD,
    },
    {
        "Item":
            "Relation topology",
        "Value":
            "Peer-only",
    },
    {
        "Item":
            "Target station inside relation",
        "Value":
            False,
    },
    {
        "Item":
            "Relation mechanism",
        "Value":
            "Adaptive historical-error gate",
    },
    {
        "Item":
            "New gate tuning parameter",
        "Value":
            False,
    },
    {
        "Item":
            "Feedback timing",
        "Value":
            "Delayed until Target_Date <= current Date",
    },
    {
        "Item":
            "Validation/Test chronology",
        "Value":
            "Continuous",
    },
    {
        "Item":
            "Test used for configuration selection",
        "Value":
            False,
    },
    {
        "Item":
            "Ablation retuning",
        "Value":
            False,
    },
])


# ==================================================================================================
# 18. OUTPUT
# ==================================================================================================

print("\n" + "=" * 130)
print("A — VALIDATION-SELECTED CONFIGURATIONS")
print("=" * 130)

display(
    U1619_SELECTION_DF
    .round(6)
)


print("\n" + "=" * 130)
print("B — FINAL NOVEL TEST RESULTS")
print("=" * 130)

display(
    U1619_NOVEL_TEST_RESULTS_DF
    .round(6)
)


print("\n" + "=" * 130)
print("C — FINAL MODEL COMPARISON")
print("=" * 130)

display(
    U1619_TASK_COMPARISON_DF
    .sort_values(
        [
            "Task",
            "Mean_Metric_Rank",
        ]
    )
    .round(6)
)


print("\nOverall comparison:")

display(
    U1619_OVERALL_COMPARISON_DF
    .round(6)
)


print("\n" + "=" * 130)
print("D — ABLATION")
print("=" * 130)

display(
    U1619_ABLATION_SUMMARY_DF
    .round(6)
)

print("\nComponent effects:")

display(
    U1619_COMPONENT_EFFECT_DF
    .round(6)
)


print("\n" + "=" * 130)
print("E — PROTOCOL")
print("=" * 130)

display(
    U1619_PROTOCOL_DF
)


# ==================================================================================================
# 19. FINAL COMPATIBILITY ALIASES
# ==================================================================================================

# Final accepted prediction source for later robustness / skill / seed cells
RGADA18_TEST_PREDICTIONS_DF = (
    U1619_TEST_PREDICTIONS_DF.copy()
)

# Explicit accepted final result
U1619_FINAL_TEST_RESULTS_DF = (
    U1619_NOVEL_TEST_RESULTS_DF.copy()
)


print("\n" + "=" * 130)
print("UNIFIED CELL 16 COMPLETED SUCCESSFULLY")
print("=" * 130)

print(
    f"Execution time: "
    f"{time.time() - CELL16_START:.2f} seconds"
)

In [ ]:
!lscpu | grep -E 'Model name|CPU\(s\)'
!free -h
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!python --version

In [ ]:

# ================================================================
# CELL 16B — UNIFIED REALISTIC SENSOR FAULT ROBUSTNESS
# PAPER 3: UTW-RG-AdaFedResidual
#
# Run directly AFTER ORIGINAL Cell 16
# No dependency on experimental Cells 16A, 16C, or 16D
# NO RETRAINING / NO ASFDR
# ================================================================

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

# ================================================================
# 1. VERIFY ORIGINAL NOTEBOOK OBJECTS
# ================================================================

B16_REQUIRED = [
    'MODEL_BASE', 'RAW_FEATURE_COLUMNS',
    'ACTIVE_FEATURE_COLUMNS', 'EXPECTED_STATIONS',
    'TASK_DEFINITIONS', 'TASK_DATASETS',
    'TASK_CLIENT_DATASETS', 'TASK_SCALERS',
    'FINAL_UTW', 'CENTRALIZED_MODELS',
    'LOCAL_MODELS', 'FEDERATED_RUNS',
    'U1619_FED_METHOD', 'U1619_TEST_PREDICTIONS_DF',
    'U1619_SELECTION_DF',
    'u1619_selected_algorithm',
    'u1619_peer_weights',
    'u1619_static_predictions',
    'u1619_run_online',
    'make_descriptors', 'predict_raw'
]

missing = [
    x for x in B16_REQUIRED
    if x not in globals()
]

if missing:
    raise RuntimeError(
        'Run original Cells 1–16 first. Missing: '
        + ', '.join(missing)
    )

# ================================================================
# 2. EXPERIMENTAL CONFIGURATION
# ================================================================

B16_SENSORS = list(RAW_FEATURE_COLUMNS)

B16_TASKS = [
    d['Task'] for d in TASK_DEFINITIONS
]

B16_W = int(FINAL_UTW)

B16_FAULTS = (
    'Offset',
    'Gain',
    'StuckAt',
    'Spike'
)

B16_LEVELS = (
    'moderate',
    'severe'
)

B16_SEEDS = (
    1701,
    1702,
    1703
)

B16_NOVEL = 'UTW-RG-AdaFedResidual'

B16_MODELS = (
    B16_NOVEL,
    'FedAdam',
    'Local',
    'Centralized'
)

B16_IDX = {
    name: i
    for i, name in enumerate(ACTIVE_FEATURE_COLUMNS)
}

B16_DESC = (
    'mean', 'std', 'min', 'max',
    'slope', 'recent_change'
)

B16_COLS = {
    sensor: [
        name for name in ACTIVE_FEATURE_COLUMNS
        if name.startswith(sensor + '__UTW_')
        and (
            '__UTW_tminus_' in name
            or name.endswith(
                tuple('__UTW_' + x for x in B16_DESC)
            )
        )
    ]
    for sensor in B16_SENSORS
}

B16_ACTIVE = [
    x for x in B16_SENSORS
    if len(B16_COLS[x]) > 0
]

B16_INACTIVE = [
    x for x in B16_SENSORS
    if len(B16_COLS[x]) == 0
]

assert len(B16_SENSORS) == 7
assert len(B16_TASKS) == 6

# ================================================================
# 3. METRICS
# ================================================================

def b16_metrics(y, p):

    y = np.asarray(y, float)
    p = np.asarray(p, float)

    if not len(y):
        return dict(
            MAE=np.nan,
            RMSE=np.nan,
            R2=np.nan
        )

    if not (
        np.isfinite(y).all()
        and np.isfinite(p).all()
    ):
        raise ValueError(
            'Nonfinite targets or predictions.'
        )

    e = y - p
    var = np.sum((y - y.mean()) ** 2)

    return dict(
        MAE=float(np.mean(abs(e))),
        RMSE=float(np.sqrt(np.mean(e * e))),
        R2=(
            float(1 - np.sum(e * e) / var)
            if var > 1e-12
            else np.nan
        )
    )


# ================================================================
# 4. ORIGINAL MODEL CONFIGURATION
# ================================================================

def b16_configuration(task, station):

    d = U1619_SELECTION_DF

    row = d.loc[
        d['Task'].eq(task)
        & d['Station'].eq(station)
    ]

    if len(row) != 1:
        raise RuntimeError(
            f'Missing selected configuration: '
            f'{task}/{station}'
        )

    row = row.iloc[0]

    names = (
        'prior_global',
        'expert_power',
        'lookback',
        'decay',
        'alpha',
        'local_relation_mix'
    )

    return {
        k: (
            int(row[k])
            if k == 'lookback'
            else float(row[k])
        )
        for k in names
    }


def b16_experts(X, obj):

    task = obj['task']

    return dict(
        Global=predict_raw(
            X, obj['fed'], task
        ),

        Self=predict_raw(
            X, obj['local'], task
        ),

        Relation=sum(
            w * predict_raw(X, par, task)
            for w, par in obj['peers']
        ),

        Centralized=predict_raw(
            X, obj['central'], task
        )
    )


def b16_novel(obj, experts):

    nval = len(obj['yv'])

    pred, _ = u1619_run_online(
        dates=obj['dates'],
        target_dates=obj['future_dates'],
        y=obj['all_y'],

        global_pred=np.r_[
            obj['val_experts']['Global'],
            experts['Global']
        ],

        self_pred=np.r_[
            obj['val_experts']['Self'],
            experts['Self']
        ],

        relation_pred=np.r_[
            obj['val_experts']['Relation'],
            experts['Relation']
        ],

        **obj['cfg']
    )

    return np.asarray(
        pred[nval:], float
    )


def b16_forecasts(obj, X):

    e = b16_experts(X, obj)

    return {
        B16_NOVEL: b16_novel(obj, e),
        'FedAdam': e['Global'],
        'Local': e['Self'],
        'Centralized': e['Centralized']
    }


# ================================================================
# 5. PREPARE CHRONOLOGICAL SENSOR HISTORIES
# ================================================================

B16_STATIONS = {}

for station in EXPECTED_STATIONS:

    raw = (
        MODEL_BASE.loc[
            MODEL_BASE['Station'].eq(station)
        ]
        .sort_values('Date')
        .reset_index(drop=True)
    )

    dates = pd.DatetimeIndex(
        pd.to_datetime(raw['Date'])
    )

    if dates.has_duplicates or dates.hasnans:
        raise RuntimeError(
            f'Invalid chronological dates: {station}'
        )

    common = None

    for task in B16_TASKS:

        d = set(
            pd.to_datetime(
                TASK_CLIENT_DATASETS[task]
                [station]['Test']['Frame']['Date']
            )
        )

        common = (
            d if common is None
            else common.intersection(d)
        )

    test_idx = dates.get_indexer(
        pd.DatetimeIndex(sorted(common))
    )

    if (
        len(test_idx) < 30
        or np.any(test_idx < 0)
    ):
        raise RuntimeError(
            f'Not enough common test dates: {station}'
        )

    train_end = min(
        pd.to_datetime(
            TASK_CLIENT_DATASETS[t]
            [station]['Train']['Frame']['Date']
        ).max()
        for t in B16_TASKS
    )

    B16_STATIONS[station] = dict(
        raw=raw,
        dates=dates,
        test_idx=test_idx,
        train_mask=dates <= train_end
    )


# ================================================================
# 6. CACHE FROZEN ORIGINAL MODELS
# ================================================================

B16_CACHE = {}

for task in B16_TASKS:

    meta = TASK_DATASETS[task]['Meta']
    algo = u1619_selected_algorithm(task)

    for station in EXPECTED_STATIONS:

        client = TASK_CLIENT_DATASETS[task][station]

        va = client['Validation']
        te = client['Test']

        vf = va['Frame'].reset_index(drop=True)
        tf = te['Frame'].reset_index(drop=True)

        idx = B16_STATIONS[station][
            'dates'
        ].get_indexer(
            pd.to_datetime(tf['Date'])
        )

        if (idx < 0).any():
            raise RuntimeError(
                f'Test-date mismatch: {task}/{station}'
            )

        weights = u1619_peer_weights(
            task, station
        )

        obj = dict(
            task=task,
            station=station,
            target=task.split('_')[0],
            horizon=int(meta['Horizon']),

            cfg=b16_configuration(
                task, station
            ),

            test_idx=idx,

            X=np.asarray(te['X'], float),
            yt=np.asarray(te['y'], float),
            yv=np.asarray(va['y'], float),

            all_y=np.r_[
                np.asarray(va['y'], float),
                np.asarray(te['y'], float)
            ],

            dates=pd.concat(
                [vf['Date'], tf['Date']],
                ignore_index=True
            ),

            future_dates=pd.concat(
                [
                    vf[meta['Future_Date_Column']],
                    tf[meta['Future_Date_Column']]
                ],
                ignore_index=True
            ),

            fed=FEDERATED_RUNS[
                (task, algo, U1619_FED_METHOD)
            ]['Parameters'],

            local=LOCAL_MODELS[
                (task, station, algo)
            ]['Parameters'],

            central=CENTRALIZED_MODELS[
                (task, algo)
            ]['Parameters'],

            peers=[
                (
                    w,
                    LOCAL_MODELS[
                        (task, peer, algo)
                    ]['Parameters']
                )
                for peer, w in weights.items()
            ],

            val_experts=u1619_static_predictions(
                task, station, 'Validation'
            )
        )

        obj['original'] = b16_forecasts(
            obj, obj['X']
        )

        original = (
            U1619_TEST_PREDICTIONS_DF.loc[
                U1619_TEST_PREDICTIONS_DF[
                    'Task'
                ].eq(task)
                &
                U1619_TEST_PREDICTIONS_DF[
                    'Station'
                ].eq(station)
            ]
            .sort_values('Date')
            .reset_index(drop=True)
        )

        same_dates = (
            pd.DatetimeIndex(
                pd.to_datetime(original['Date'])
            ).equals(
                pd.DatetimeIndex(
                    pd.to_datetime(tf['Date'])
                )
            )
        )

        same_predictions = np.allclose(
            obj['original'][B16_NOVEL],

            original[
                'RGAdaFedResidual_Pred'
            ].to_numpy(float),

            rtol=1e-8,
            atol=1e-7
        )

        if not (same_dates and same_predictions):
            raise RuntimeError(
                f'Original Cell 16 replay FAILED: '
                f'{task}/{station}'
            )

        B16_CACHE[(task, station)] = obj

print(
    'PASS: Original Cell 16 predictions reproduced.'
)
print(
    'All original forecasting models remain frozen.'
)


# ================================================================
# 7. REBUILD EXACT ORIGINAL UTW FEATURES
# ================================================================

def b16_features(signal, sensor):

    if not B16_COLS[sensor]:
        return {}

    s = pd.Series(
        signal, dtype=float
    )

    lags = np.column_stack([
        s.shift(j).to_numpy(float)
        for j in range(B16_W)
    ])

    desc = make_descriptors(
        lags[:, ::-1]
    )

    out = {
        f'{sensor}__UTW_tminus_{j}':
            lags[:, j]
        for j in range(B16_W)
    }

    out.update({
        f'{sensor}__UTW_{k}':
            desc[k]
        for k in B16_DESC
    })

    return {
        k: v
        for k, v in out.items()
        if k in B16_IDX
    }


def b16_override(obj, features):

    X = obj['X'].copy()

    sc = TASK_SCALERS[obj['task']]

    for name, array in features.items():

        j = B16_IDX[name]

        X[:, j] = (
            array[obj['test_idx']]
            - sc['x_mean'][j]
        ) / sc['x_std'][j]

    if not np.isfinite(X).all():
        raise RuntimeError(
            f'Invalid rebuilt UTW features: '
            f'{obj["task"]}'
        )

    return X


# Verify exact clean reconstruction.

for station in EXPECTED_STATIONS:

    for sensor in B16_ACTIVE:

        rebuilt = b16_features(
            B16_STATIONS[station]['raw']
            [sensor].to_numpy(float),
            sensor
        )

        for task in B16_TASKS:

            obj = B16_CACHE[
                (task, station)
            ]

            if not np.allclose(
                b16_override(obj, rebuilt),
                obj['X'],
                atol=1e-7,
                rtol=1e-8
            ):
                raise RuntimeError(
                    f'Clean UTW mismatch: '
                    f'{task}/{station}/{sensor}'
                )

print(
    'PASS: Original UTW features reconstructed.'
)


# ================================================================
# 8. TRAIN-ONLY OUTLIER GUARD
# ================================================================

B16_GUARD_PARAMS = {}

for station in EXPECTED_STATIONS:

    data = B16_STATIONS[station]

    for sensor in B16_ACTIVE:

        a = data['raw'].loc[
            data['train_mask'],
            sensor
        ].to_numpy(float)

        a = a[np.isfinite(a)]

        if len(a) < 10:
            raise RuntimeError(
                f'Insufficient TRAIN values: '
                f'{station}/{sensor}'
            )

        median = float(np.median(a))

        iqr = float(
            np.quantile(a, .75)
            - np.quantile(a, .25)
        )

        scale = max(
            float(
                1.4826 *
                np.median(abs(a - median))
            ),
            float(np.std(a)) * .15,
            1e-6
        )

        q1, q3 = np.quantile(
            a, [.25, .75]
        )

        lo = float(
            q1 - 3 * max(iqr, scale)
        )

        hi = float(
            q3 + 3 * max(iqr, scale)
        )

        if sensor == 'pH_raw':
            lo = max(0., lo)
            hi = min(14., hi)

        elif sensor == '(°C)_raw':
            lo = max(-273.15, lo)

        else:
            lo = max(0., lo)

        if lo >= hi:
            hi = lo + max(scale, 1e-4)

        B16_GUARD_PARAMS[
            (station, sensor)
        ] = dict(
            median=median,
            scale=scale,
            lo=lo,
            hi=hi
        )


def b16_guard(signal, station, sensor):

    p = B16_GUARD_PARAMS[
        (station, sensor)
    ]

    out = np.asarray(
        signal, float
    ).copy()

    rejected = np.zeros(
        len(out), dtype=bool
    )

    last = p['median']

    for k, x in enumerate(out):

        if (
            not np.isfinite(x)
            or x < p['lo']
            or x > p['hi']
        ):
            out[k] = last
            rejected[k] = True

        else:
            last = x

    return out, rejected


def b16_affected(test_indices, altered):

    affected = np.zeros(
        len(test_indices), bool
    )

    for lag in range(B16_W):

        ix = test_indices - lag

        ok = ix >= 0

        affected[ok] |= altered[
            ix[ok]
        ]

    return affected


# ================================================================
# 9. HEALTHY-DATA CONTROL
# ================================================================

B16_HEALTHY = []
B16_CONTROL = []
B16_CLEAN = {}

for station in EXPECTED_STATIONS:

    data = B16_STATIONS[station]

    for sensor in B16_ACTIVE:

        original = (
            data['raw'][sensor]
            .to_numpy(float)
        )

        guarded, rejected = b16_guard(
            original, station, sensor
        )

        B16_HEALTHY.append(dict(
            Station=station,
            Sensor=sensor,
            Healthy_History_Replacements=int(
                rejected.sum()
            ),
            Healthy_Test_Replacements=int(
                rejected[data['test_idx']].sum()
            )
        ))

        new_features = b16_features(
            guarded, sensor
        )

        for task in B16_TASKS:

            obj = B16_CACHE[
                (task, station)
            ]

            pred = b16_forecasts(
                obj,
                b16_override(
                    obj, new_features
                )
            )

            B16_CLEAN[
                (task, station, sensor)
            ] = pred

            for model in B16_MODELS:

                original_mae = b16_metrics(
                    obj['yt'],
                    obj['original'][model]
                )['MAE']

                guarded_mae = b16_metrics(
                    obj['yt'],
                    pred[model]
                )['MAE']

                B16_CONTROL.append(dict(
                    Target=obj['target'],
                    Horizon=obj['horizon'],
                    Station=station,
                    Sensor=sensor,
                    Model=model,
                    Original_Healthy_MAE=original_mae,
                    Guarded_Healthy_MAE=guarded_mae,
                    Healthy_Guard_Delta=(
                        guarded_mae - original_mae
                    )
                ))

B16_HEALTHY_DF = pd.DataFrame(
    B16_HEALTHY
)

B16_CONTROL_DF = pd.DataFrame(
    B16_CONTROL
)


# ================================================================
# 10. REALISTIC SENSOR FAULT INJECTION
# ================================================================

def b16_inject(
    station,
    sensor,
    fault,
    severity,
    seed
):

    data = B16_STATIONS[station]

    clean = (
        data['raw'][sensor]
        .to_numpy(float)
    )

    corrupt = clean.copy()

    test_idx = data['test_idx']
    n = len(test_idx)

    p = B16_GUARD_PARAMS[
        (station, sensor)
    ]

    rng = np.random.default_rng(
        int(seed)
        + sum(
            map(
                ord,
                station + sensor + fault
            )
        )
    )

    chosen = np.zeros(
        len(clean), bool
    )

    length = max(
        3,
        int(
            np.ceil(
                n * (
                    .035
                    if severity == 'moderate'
                    else .07
                )
            )
        )
    )

    for fraction in (.25, .65):

        start = (
            int(n * fraction)
            + int(rng.integers(-2, 3))
        )

        start = max(
            0,
            min(n - length, start)
        )

        episode = test_idx[
            start:start + length
        ]

        if not len(episode):
            continue

        sign = float(
            rng.choice([-1, 1])
        )

        if fault == 'Offset':

            corrupt[episode] = (
                clean[episode]
                + sign * p['scale']
                * (
                    .30
                    if severity == 'moderate'
                    else .75
                )
            )

        elif fault == 'Gain':

            factor = (
                1.04
                if severity == 'moderate'
                else 1.10
            )

            if sign < 0:
                factor = 1.0 / factor

            corrupt[episode] = (
                clean[episode] * factor
            )

        elif fault == 'StuckAt':

            corrupt[episode] = clean[
                max(
                    int(episode[0]) - 1,
                    0
                )
            ]

        elif fault == 'Spike':

            point = int(
                episode[
                    len(episode) // 2
                ]
            )

            if sign > 0:
                value = (
                    p['hi']
                    + p['scale']
                    * (
                        .20
                        if severity == 'moderate'
                        else .60
                    )
                )

            else:
                value = (
                    p['lo']
                    - p['scale']
                    * (
                        .20
                        if severity == 'moderate'
                        else .60
                    )
                )

            corrupt[point] = value

            episode = np.array(
                [point], dtype=int
            )

        else:
            raise ValueError(
                f'Unknown fault: {fault}'
            )

        chosen[episode] = True

    effective = (
        chosen
        &
        (
            np.abs(
                corrupt - clean
            ) > 1e-10
        )
    )

    return corrupt, effective


# ================================================================
# 11. RUN ALL FAULT EXPERIMENTS
# ================================================================

B16_ROWS = []

B16_N = (
    len(EXPECTED_STATIONS)
    * len(B16_ACTIVE)
    * len(B16_FAULTS)
    * len(B16_LEVELS)
    * len(B16_SEEDS)
    * len(B16_TASKS)
)

bar = tqdm(
    total=B16_N,
    desc='Cell 16B realistic fault robustness',
    unit='case'
)

for station in EXPECTED_STATIONS:

    for sensor in B16_ACTIVE:

        for fault in B16_FAULTS:

            for severity in B16_LEVELS:

                for seed in B16_SEEDS:

                    corrupted, effective = b16_inject(
                        station,
                        sensor,
                        fault,
                        severity,
                        seed
                    )

                    guarded, rejected = b16_guard(
                        corrupted,
                        station,
                        sensor
                    )

                    features = b16_features(
                        guarded, sensor
                    )

                    caught = int(
                        (
                            rejected & effective
                        ).sum()
                    )

                    for task in B16_TASKS:

                        obj = B16_CACHE[
                            (task, station)
                        ]

                        hit = b16_affected(
                            obj['test_idx'],
                            effective
                        )

                        clean_pred = B16_CLEAN[
                            (task, station, sensor)
                        ]

                        if hit.any():

                            pred = b16_forecasts(
                                obj,
                                b16_override(
                                    obj, features
                                )
                            )

                        else:
                            pred = clean_pred

                        for model in B16_MODELS:

                            y = obj['yt']

                            cp = np.asarray(
                                clean_pred[model]
                            )

                            fp = np.asarray(
                                pred[model]
                            )

                            cm = b16_metrics(
                                y[hit], cp[hit]
                            )

                            fm = b16_metrics(
                                y[hit], fp[hit]
                            )

                            cf = b16_metrics(
                                y, cp
                            )

                            ff = b16_metrics(
                                y, fp
                            )

                            B16_ROWS.append(dict(
                                Target=obj['target'],
                                Horizon=obj['horizon'],
                                Task=task,
                                Station=station,
                                Sensor=sensor,
                                Fault=fault,
                                Severity=severity,
                                Seed=seed,
                                Model=model,

                                Effective_Points=int(
                                    effective.sum()
                                ),

                                Guard_Caught=caught,

                                Affected_Windows=int(
                                    hit.sum()
                                ),

                                Clean_MAE=cm['MAE'],
                                Fault_MAE=fm['MAE'],

                                Clean_RMSE=cm['RMSE'],
                                Fault_RMSE=fm['RMSE'],

                                Clean_R2=cm['R2'],
                                Fault_R2=fm['R2'],

                                Full_Clean_MAE=cf['MAE'],
                                Full_Fault_MAE=ff['MAE'],

                                MAE_Delta=(
                                    fm['MAE']
                                    - cm['MAE']
                                ),

                                MAE_Change_Pct=(
                                    100 * (
                                        fm['MAE']
                                        - cm['MAE']
                                    ) / cm['MAE']
                                    if cm['MAE'] > 1e-10
                                    else np.nan
                                ),

                                Full_MAE_Change_Pct=(
                                    100 * (
                                        ff['MAE']
                                        - cf['MAE']
                                    ) / cf['MAE']
                                    if cf['MAE'] > 1e-10
                                    else np.nan
                                )
                            ))

                        bar.update(1)

bar.close()

B16_RESULTS_DF = pd.DataFrame(
    B16_ROWS
)

assert len(B16_RESULTS_DF) == (
    4 * B16_N
)

B16_VALID_DF = B16_RESULTS_DF.loc[
    (
        B16_RESULTS_DF[
            'Effective_Points'
        ] > 0
    )
    &
    (
        B16_RESULTS_DF[
            'Affected_Windows'
        ] > 0
    )
].copy()


# ================================================================
# 12. PAIRED NOVEL VS BASELINE COMPARISON
# ================================================================

keys = [
    'Target', 'Horizon',
    'Task', 'Station',
    'Sensor', 'Fault',
    'Severity', 'Seed'
]

wide = B16_VALID_DF.pivot(
    index=keys,
    columns='Model',
    values=[
        'Fault_MAE',
        'MAE_Change_Pct'
    ]
)

rows = []

for baseline in (
    'FedAdam',
    'Local',
    'Centralized'
):

    for key, r in wide.iterrows():

        novel_mae = r[
            ('Fault_MAE', B16_NOVEL)
        ]

        base_mae = r[
            ('Fault_MAE', baseline)
        ]

        novel_change = r[
            ('MAE_Change_Pct', B16_NOVEL)
        ]

        base_change = r[
            ('MAE_Change_Pct', baseline)
        ]

        if not (
            np.isfinite(novel_mae)
            and np.isfinite(base_mae)
        ):
            continue

        if np.isclose(
            novel_mae,
            base_mae,
            atol=1e-9,
            rtol=0
        ):
            accuracy_win = .5

        else:
            accuracy_win = float(
                novel_mae < base_mae
            )

        if not (
            np.isfinite(novel_change)
            and np.isfinite(base_change)
        ):
            sensitivity_win = np.nan

        elif np.isclose(
            novel_change,
            base_change,
            atol=1e-9,
            rtol=0
        ):
            sensitivity_win = .5

        else:
            sensitivity_win = float(
                novel_change < base_change
            )

        rows.append(
            dict(zip(keys, key))
            |
            dict(
                Comparator=baseline,

                Lower_Fault_MAE=accuracy_win,

                Less_Degradation=sensitivity_win,

                Fault_MAE_Advantage=(
                    base_mae - novel_mae
                )
            )
        )

B16_PAIRED_DF = pd.DataFrame(rows)


# ================================================================
# 13. HORIZON-LEVEL COMPARISON
# ================================================================

B16_COMPARE_DF = (
    B16_PAIRED_DF.groupby(
        [
            'Target',
            'Horizon',
            'Sensor',
            'Fault',
            'Severity',
            'Comparator'
        ],
        as_index=False
    )
    .agg(
        Paired_Cases=(
            'Lower_Fault_MAE',
            'size'
        ),

        Novel_Lower_MAE_Rate=(
            'Lower_Fault_MAE',
            'mean'
        ),

        Novel_Less_Degradation_Rate=(
            'Less_Degradation',
            'mean'
        ),

        Mean_Fault_MAE_Advantage=(
            'Fault_MAE_Advantage',
            'mean'
        )
    )
)


# ================================================================
# 14. SENSOR-LEVEL COMPARISON
# ================================================================

B16_SENSOR_DF = (
    pd.DataFrame({
        'Sensor': B16_SENSORS
    })
    .merge(
        B16_PAIRED_DF.groupby(
            [
                'Sensor',
                'Comparator'
            ],
            as_index=False
        )
        .agg(
            Paired_Cases=(
                'Lower_Fault_MAE',
                'size'
            ),

            Novel_Lower_MAE_Rate=(
                'Lower_Fault_MAE',
                'mean'
            ),

            Novel_Less_Degradation_Rate=(
                'Less_Degradation',
                'mean'
            )
        ),

        on='Sensor',
        how='left'
    )
)


# ================================================================
# 15. DETAILED TASK RESULTS
# ================================================================

B16_TASK_DF = (
    B16_VALID_DF.groupby(
        [
            'Target',
            'Horizon',
            'Sensor',
            'Fault',
            'Severity',
            'Model'
        ],
        as_index=False
    )
    .agg(
        Cases=('Seed', 'size'),

        Clean_MAE=(
            'Clean_MAE', 'mean'
        ),

        Fault_MAE=(
            'Fault_MAE', 'mean'
        ),

        Mean_MAE_Change_Pct=(
            'MAE_Change_Pct', 'mean'
        ),

        Median_MAE_Change_Pct=(
            'MAE_Change_Pct', 'median'
        ),

        Full_MAE_Change_Pct=(
            'Full_MAE_Change_Pct', 'mean'
        )
    )
)


# ================================================================
# 16. SENSOR STATUS
# ================================================================

B16_STATUS_DF = pd.DataFrame({
    'Sensor': B16_SENSORS,

    'Active_UTW_Features': [
        len(B16_COLS[s])
        for s in B16_SENSORS
    ],

    'Status': [
        'TESTED'
        if s in B16_ACTIVE
        else 'INACTIVE—N/A'
        for s in B16_SENSORS
    ]
})


# ================================================================
# 17. FINAL REPORT
# ================================================================

print('\n' + '=' * 65)
print('CELL 16B — REALISTIC SENSOR FAULT ROBUSTNESS')
print('=' * 65)

print(
    'Original models: FROZEN'
)

print(
    'No retraining. No ASFDR.'
)

print(
    f'Active sensors: {len(B16_ACTIVE)} / 7'
)

print(
    f'Inactive sensors: {B16_INACTIVE}'
)

print(
    f'Scenarios: {B16_N:,}'
)

print(
    f'Model evaluations: {len(B16_RESULTS_DF):,}'
)

print('\n1. SENSOR STATUS')
display(B16_STATUS_DF)

print('\n2. HEALTHY-DATA CONTROL')
display(B16_HEALTHY_DF)

print(
    '\n3. CLEAN GUARD CONTROL — NOVEL MODEL'
)

display(
    B16_CONTROL_DF.loc[
        B16_CONTROL_DF[
            'Model'
        ].eq(B16_NOVEL)
    ].round(5)
)

print(
    '\n4. NOVEL VS BASELINES — SENSOR LEVEL'
)

display(
    B16_SENSOR_DF.round(3)
)

print(
    '\n5. NOVEL VS BASELINES — H1, H2, H3'
)

display(
    B16_COMPARE_DF.round(3)
)

print(
    '\n6. DETAILED FAULT PERFORMANCE'
)

display(
    B16_TASK_DF.round(4)
)

print('\nRESULT DATAFRAMES:')

print('B16_RESULTS_DF')
print('B16_VALID_DF')
print('B16_PAIRED_DF')
print('B16_COMPARE_DF')
print('B16_SENSOR_DF')
print('B16_TASK_DF')
print('B16_HEALTHY_DF')
print('B16_CONTROL_DF')
print('B16_STATUS_DF')

print(
    '\nInterpretation:'
    '\nNovel_Lower_MAE_Rate: larger is better.'
    '\nNovel_Less_Degradation_Rate: larger is better.'
    '\n1 = win; 0.5 = tie; 0 = loss.'
    '\nPositive MAE change = degradation.'
    '\nInactive sensors are not evidence of robustness.'
)


In [ ]:

# ================================================================
# CELL 16B-1 — FINAL REPLACEMENT
# SUPPLEMENTARY TABLE S30
#
# Full H1–H3 sensor-fault results:
# Scenario | Task | Runs | MAE | RMSE | R² | R² SD
#
# Corrected physical-validity guard
# Four frozen models; no retraining; no ASFDR
# Independent of any previous Cell 16B-1
# ================================================================

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

# ------------------------------------------------
# 1. Verify original Cell 16B
# ------------------------------------------------

required = [
    "B16_CACHE",
    "B16_STATIONS",
    "B16_ACTIVE",
    "B16_TASKS",
    "B16_FAULTS",
    "B16_LEVELS",
    "B16_SEEDS",
    "B16_MODELS",
    "B16_NOVEL",
    "B16_GUARD_PARAMS",
    "EXPECTED_STATIONS",
    "b16_inject",
    "b16_features",
    "b16_override",
    "b16_forecasts",
    "b16_metrics",
    "U1619_NOVEL_TEST_RESULTS_DF",
]

missing = [v for v in required if v not in globals()]

if missing:
    raise RuntimeError(
        "Run original Cells 1–16 and unified Cell 16B first. "
        "Missing: " + ", ".join(missing)
)

S30_NOVEL = B16_NOVEL
S30_MODELS = list(B16_MODELS)
S30_STATIONS = list(EXPECTED_STATIONS)
S30_SENSORS = list(B16_ACTIVE)

S30_TASKS = sorted(
    B16_TASKS,
    key=lambda t: (
        0 if B16_CACHE[(t, S30_STATIONS[0])]["target"]
        == "Alum" else 1,
        int(B16_CACHE[(t, S30_STATIONS[0])]["horizon"])
    )
)

S30_FAULT_LABELS = {
    "Offset": "Offset",
    "Gain": "Gain",
    "StuckAt": "Stuck-at",
    "Spike": "Spike",
}

S30_SCENARIOS = [
    (
        fault,
        severity,
        f"{S30_FAULT_LABELS[fault]} – {severity.title()}"
    )
    for fault in B16_FAULTS
    for severity in B16_LEVELS
]

assert len(S30_TASKS) == 6
assert len(S30_STATIONS) == 4
assert len(S30_SENSORS) == 6
assert len(S30_SCENARIOS) == 8


# ------------------------------------------------
# 2. Conservative physical-validity guard
#
# Exactly the corrected guard used for Table 15.
# Valid but unusual readings are NOT clipped.
# ------------------------------------------------

def s30_physical_guard(series, station, sensor):

    a = np.asarray(series, dtype=float).copy()

    rejected = np.zeros(len(a), dtype=bool)

    last = float(
        B16_GUARD_PARAMS[
            (station, sensor)
        ]["median"]
    )

    for i, value in enumerate(a):

        invalid = not np.isfinite(value)

        if not invalid:

            if sensor == "pH_raw":
                invalid = (
                    value < 0.0 or value > 14.0
                )

            elif sensor == "(°C)_raw":
                invalid = value < -273.15

            else:
                invalid = value < 0.0

        if invalid:
            a[i] = last
            rejected[i] = True
        else:
            last = value

    return a, rejected


# ------------------------------------------------
# 3. Validate healthy sensor histories
#
# Physical guard must leave original clean
# histories unchanged.
# ------------------------------------------------

S30_CLEAN_HISTORIES = {}
healthy_control = []

for station in S30_STATIONS:

    for sensor in S30_SENSORS:

        original = (
            B16_STATIONS[station]["raw"][sensor]
            .to_numpy(float)
        )

        guarded, rejected = s30_physical_guard(
            original, station, sensor
        )

        replacements = int(rejected.sum())

        healthy_control.append({
            "Station": station,
            "Sensor": sensor,
            "Healthy_Replacements": replacements
        })

        if replacements != 0 or not np.array_equal(
            original, guarded
        ):
            raise RuntimeError(
                "Healthy sensor measurements were modified: "
                f"{station}/{sensor}. "
                "Stop and inspect the clean data."
            )

        S30_CLEAN_HISTORIES[
            (station, sensor)
        ] = guarded

S30_HEALTHY_CONTROL_DF = pd.DataFrame(
    healthy_control
)

print("PASS: No healthy sensor readings modified.")


# ------------------------------------------------
# 4. Pool actual targets and original predictions
# ------------------------------------------------

S30_Y = {}
S30_CLEAN_METRICS = {}

for task in S30_TASKS:

    y = np.concatenate([
        np.asarray(
            B16_CACHE[(task, st)]["yt"],
            dtype=float
        )
        for st in S30_STATIONS
    ])

    S30_Y[task] = y

    for model in S30_MODELS:

        p = np.concatenate([
            np.asarray(
                B16_CACHE[(task, st)]["original"][model],
                dtype=float
            )
            for st in S30_STATIONS
        ])

        S30_CLEAN_METRICS[
            (task, model)
        ] = b16_metrics(y, p)


# ------------------------------------------------
# 5. Verify original clean H1 R²
# ------------------------------------------------

for task in S30_TASKS:

    obj = B16_CACHE[
        (task, S30_STATIONS[0])
    ]

    if int(obj["horizon"]) != 1:
        continue

    published = (
        U1619_NOVEL_TEST_RESULTS_DF.loc[
            U1619_NOVEL_TEST_RESULTS_DF[
                "Task"
            ].eq(task),
            "R2"
        ]
    )

    if len(published) != 1:
        raise RuntimeError(
            f"Missing clean reference: {task}"
        )

    actual = S30_CLEAN_METRICS[
        (task, S30_NOVEL)
    ]["R2"]

    if not np.isclose(
        actual,
        float(published.iloc[0]),
        rtol=0,
        atol=1e-9
    ):
        raise RuntimeError(
            f"Original H1 R² mismatch: {task}"
        )

print("PASS: Original clean H1 references verified.")


# ------------------------------------------------
# 6. Re-evaluate frozen models under faults
#
# One scenario run = one sensor × one seed.
# Four stations are pooled within each run.
#
# Runs per scenario/task = 6 sensors × 3 seeds = 18.
# ------------------------------------------------

S30_TRIAL_ROWS = []

total = (
    len(S30_SENSORS)
    * len(S30_SCENARIOS)
    * len(B16_SEEDS)
    * len(S30_TASKS)
)

with tqdm(
    total=total,
    desc="Cell 16B-1: Full S30 results",
    unit="task-scenario"
) as progress:

    for sensor in S30_SENSORS:

        for fault, severity, scenario in S30_SCENARIOS:

            for seed in B16_SEEDS:

                # Inject once per station, reusing the
                # same corrupted histories for all tasks.

                prepared_features = {}
                fault_count = 0
                rejected_count = 0

                for station in S30_STATIONS:

                    corrupted, effective = b16_inject(
                        station,
                        sensor,
                        fault,
                        severity,
                        seed
                    )

                    guarded, rejected = s30_physical_guard(
                        corrupted,
                        station,
                        sensor
                    )

                    fault_count += int(
                        effective.sum()
                    )

                    rejected_count += int(
                        (effective & rejected).sum()
                    )

                    clean = S30_CLEAN_HISTORIES[
                        (station, sensor)
                    ]

                    if np.array_equal(guarded, clean):
                        prepared_features[station] = None
                    else:
                        prepared_features[station] = (
                            b16_features(
                                guarded, sensor
                            )
                        )

                # Evaluate six forecasting tasks.
                for task in S30_TASKS:

                    obj0 = B16_CACHE[
                        (task, S30_STATIONS[0])
                    ]

                    preds = {
                        model: []
                        for model in S30_MODELS
                    }

                    for station in S30_STATIONS:

                        obj = B16_CACHE[
                            (task, station)
                        ]

                        features = prepared_features[
                            station
                        ]

                        if features is None:

                            model_preds = obj["original"]

                        else:

                            X = b16_override(
                                obj, features
                            )

                            model_preds = b16_forecasts(
                                obj, X
                            )

                        for model in S30_MODELS:

                            preds[model].append(
                                np.asarray(
                                    model_preds[model],
                                    dtype=float
                                )
                            )

                    # Four models evaluated on identical
                    # observations and fault conditions.
                    for model in S30_MODELS:

                        pooled_pred = np.concatenate(
                            preds[model]
                        )

                        m = b16_metrics(
                            S30_Y[task],
                            pooled_pred
                        )

                        clean_m = S30_CLEAN_METRICS[
                            (task, model)
                        ]

                        S30_TRIAL_ROWS.append({
                            "Scenario": scenario,
                            "Fault": fault,
                            "Severity": severity,
                            "Sensor": sensor,
                            "Seed": seed,
                            "Task": task,
                            "Model": model,

                            "Injected Points": fault_count,
                            "Rejected Fault Points":
                                rejected_count,

                            "MAE": m["MAE"],
                            "RMSE": m["RMSE"],
                            "R²": m["R2"],

                            "Clean MAE": clean_m["MAE"],
                            "Clean RMSE": clean_m["RMSE"],
                            "Clean R²": clean_m["R2"],

                            "ΔMAE":
                                m["MAE"] - clean_m["MAE"],

                            "ΔRMSE":
                                m["RMSE"] - clean_m["RMSE"],

                            "ΔR²":
                                m["R2"] - clean_m["R2"],
                        })

                    progress.update(1)


S30_TRIALS_DF = pd.DataFrame(
    S30_TRIAL_ROWS
)

expected_trials = total * len(S30_MODELS)

if len(S30_TRIALS_DF) != expected_trials:
    raise RuntimeError(
        "Incomplete S30 experiment results."
    )

print(
    f"PASS: {len(S30_TRIALS_DF):,} "
    "model-level scenario evaluations."
)


# ------------------------------------------------
# 7. Summarize 18 runs per scenario/task/model
#
# R² SD uses sample standard deviation (ddof=1)
# across six sensor scenarios × three seeds.
# ------------------------------------------------

S30_SUMMARY_DF = (
    S30_TRIALS_DF.groupby(
        ["Scenario", "Task", "Model"],
        as_index=False,
        sort=False
    )
    .agg(
        Runs=("R²", "size"),
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        R2=("R²", "mean"),
        R2_SD=("R²", "std"),
        Delta_R2=("ΔR²", "mean")
    )
)

if not (
    S30_SUMMARY_DF["Runs"]
    == len(S30_SENSORS) * len(B16_SEEDS)
).all():
    raise RuntimeError(
        "Incorrect number of experimental runs."
    )


# ------------------------------------------------
# 8. Build publication-ready Table S30
#
# Columns match Supplementary Table S29.
# Model: UTW-RG-AdaFedResidual
# ------------------------------------------------

table_rows = []

# Clean reference: a single deterministic
# evaluation of each frozen task model.
for task in S30_TASKS:

    m = S30_CLEAN_METRICS[
        (task, S30_NOVEL)
    ]

    table_rows.append({
        "Scenario": "Clean",
        "Task": task,
        "Runs": 1,
        "MAE": m["MAE"],
        "RMSE": m["RMSE"],
        "R²": m["R2"],
        "R² SD": np.nan
    })


# Fault conditions, each with 18 evaluations.
for _, _, scenario in S30_SCENARIOS:

    for task in S30_TASKS:

        selected = S30_SUMMARY_DF.loc[
            S30_SUMMARY_DF["Scenario"].eq(scenario)
            &
            S30_SUMMARY_DF["Task"].eq(task)
            &
            S30_SUMMARY_DF["Model"].eq(S30_NOVEL)
        ]

        if len(selected) != 1:
            raise RuntimeError(
                f"Missing S30 row: {scenario}/{task}"
            )

        r = selected.iloc[0]

        table_rows.append({
            "Scenario": scenario,
            "Task": task,
            "Runs": int(r["Runs"]),
            "MAE": float(r["MAE"]),
            "RMSE": float(r["RMSE"]),
            "R²": float(r["R2"]),
            "R² SD": float(r["R2_SD"])
        })


S30_TABLE_DF = pd.DataFrame(
    table_rows,
    columns=[
        "Scenario",
        "Task",
        "Runs",
        "MAE",
        "RMSE",
        "R²",
        "R² SD"
    ]
)

assert len(S30_TABLE_DF) == 54


# ------------------------------------------------
# 9. Additional comparison table:
#    Novel vs FedAdam, Local, Centralized
#
# Same physical guard and identical fault cases.
# Saved for Supplementary Materials.
# ------------------------------------------------

comparison_rows = []

for model in S30_MODELS:

    for task in S30_TASKS:

        m = S30_CLEAN_METRICS[
            (task, model)
        ]

        comparison_rows.append({
            "Scenario": "Clean",
            "Task": task,
            "Model": model,
            "Runs": 1,
            "MAE": m["MAE"],
            "RMSE": m["RMSE"],
            "R²": m["R2"],
            "R² SD": np.nan,
            "ΔR²": 0.0
        })

for _, _, scenario in S30_SCENARIOS:

    for task in S30_TASKS:

        for model in S30_MODELS:

            selected = S30_SUMMARY_DF.loc[
                S30_SUMMARY_DF[
                    "Scenario"
                ].eq(scenario)
                &
                S30_SUMMARY_DF[
                    "Task"
                ].eq(task)
                &
                S30_SUMMARY_DF[
                    "Model"
                ].eq(model)
            ]

            if len(selected) != 1:
                raise RuntimeError(
                    f"Missing comparison: "
                    f"{scenario}/{task}/{model}"
                )

            r = selected.iloc[0]

            comparison_rows.append({
                "Scenario": scenario,
                "Task": task,
                "Model": model,
                "Runs": int(r["Runs"]),
                "MAE": float(r["MAE"]),
                "RMSE": float(r["RMSE"]),
                "R²": float(r["R2"]),
                "R² SD": float(r["R2_SD"]),
                "ΔR²": float(r["Delta_R2"])
            })


S30_MODEL_COMPARISON_DF = pd.DataFrame(
    comparison_rows
)


# ------------------------------------------------
# 10. Sensor-specific results
#     Three random seeds per sensor/fault/task/model
# ------------------------------------------------

S30_SENSOR_DETAIL_DF = (
    S30_TRIALS_DF.groupby(
        [
            "Sensor",
            "Scenario",
            "Task",
            "Model"
        ],
        as_index=False
    )
    .agg(
        Runs=("R²", "size"),
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        R2=("R²", "mean"),
        R2_SD=("R²", "std"),
        Delta_R2=("ΔR²", "mean")
    )
)


# ------------------------------------------------
# 11. Recreate exactly eight H1 rows for Table 15
# ------------------------------------------------

h1_rows = []

for _, _, scenario in S30_SCENARIOS:

    row = {
        "Disturbance": (
            "Sensor Fault Injection"
            if len(h1_rows) == 0 else ""
        ),
        "Condition": scenario
    }

    for target in ["Alum", "Chlorine"]:

        task = next(
            t for t in S30_TASKS
            if (
                B16_CACHE[
                    (t, S30_STATIONS[0])
                ]["target"] == target
                and
                int(
                    B16_CACHE[
                        (t, S30_STATIONS[0])
                    ]["horizon"]
                ) == 1
            )
        )

        selected = S30_TABLE_DF.loc[
            S30_TABLE_DF["Scenario"].eq(scenario)
            &
            S30_TABLE_DF["Task"].eq(task),
            "R²"
        ]

        if len(selected) != 1:
            raise RuntimeError(
                "Missing Table 15 H1 result."
            )

        row[f"{target} R²"] = float(
            selected.iloc[0]
        )

    h1_rows.append(row)


B16_TABLE15_FAULT_ROWS_DF = pd.DataFrame(
    h1_rows
)


# ------------------------------------------------
# 12. Check consistency with corrected Table 15
# ------------------------------------------------

expected_h1 = {
    "Offset – Moderate": (0.692806, 0.810669),
    "Offset – Severe":   (0.692516, 0.809871),
    "Gain – Moderate":   (0.692731, 0.810294),
    "Gain – Severe":     (0.692296, 0.804933),
    "Stuck-at – Moderate": (0.692818, 0.810366),
    "Stuck-at – Severe":   (0.692708, 0.810497),
    "Spike – Moderate": (0.691378, 0.804517),
    "Spike – Severe":   (0.691400, 0.802392),
}

mismatches = []

for _, r in B16_TABLE15_FAULT_ROWS_DF.iterrows():

    target_values = expected_h1[
        r["Condition"]
    ]

    actual_values = (
        float(r["Alum R²"]),
        float(r["Chlorine R²"])
    )

    if not np.allclose(
        actual_values,
        target_values,
        rtol=0,
        atol=0.000005
    ):
        mismatches.append(r["Condition"])

if mismatches:

    print(
        "\nWARNING: H1 values differ from "
        "the previously verified Table 15."
    )
    print("Conditions:", mismatches)

else:

    print(
        "\nPASS: All eight H1 conditions "
        "match corrected Table 15."
    )


# ------------------------------------------------
# 13. Final outputs
# ------------------------------------------------

print("\n" + "=" * 72)
print("SUPPLEMENTARY TABLE S30")
print("FULL SENSOR-FAULT FORECASTING RESULTS — H1 TO H3")
print("=" * 72)

display(
    S30_TABLE_DF.round(6)
)

print("\n" + "=" * 72)
print("TABLE 15 — EIGHT VERIFIED H1 ROWS")
print("=" * 72)

display(
    B16_TABLE15_FAULT_ROWS_DF.round(6)
)

print("\n" + "=" * 72)
print("CORRECTED COMPARISON WITH ALL BASELINES")
print("=" * 72)

display(
    S30_MODEL_COMPARISON_DF.round(6)
)

print("\n" + "=" * 72)
print("HEALTHY-DATA CONTROL")
print("=" * 72)

display(
    S30_HEALTHY_CONTROL_DF
)

print("\nSaved DataFrames:")
print("S30_TABLE_DF")
print("S30_TRIALS_DF")
print("S30_SUMMARY_DF")
print("S30_MODEL_COMPARISON_DF")
print("S30_SENSOR_DETAIL_DF")
print("S30_HEALTHY_CONTROL_DF")
print("B16_TABLE15_FAULT_ROWS_DF")

print(
    "\nNotes:"
    "\n- S30 reports six chemical-horizon tasks."
    "\n- Fault Runs = 18 (6 sensors x 3 seeds)."
    "\n- Each run pools four stations."
    "\n- R² SD is across those 18 sensor-seed trials."
    "\n- Clean Runs = 1, so clean R² SD is undefined."
    "\n- Physical-validity guard only."
    "\n- Original forecasting models remain frozen."
    "\n- No retraining and no ASFDR."
)


In [ ]:
# ==================================================================================================
# CELL 17A — SENSOR / INPUT NOISE ROBUSTNESS
# FINAL CORRECTED VERSION — compatible with corrected Cell 16
# ==================================================================================================
# Protocol:
#   - Frozen final Cell-16 UTW-RG-AdaFedResidual
#   - Test-input Gaussian noise only
#   - Levels: 0%, 1%, 2%, 5%, 10%
#   - 5 repetitions: seeds 1701..1705
#   - Station one-hot features are NOT perturbed
#   - Validation remains clean
#   - No retraining
#   - No tuning
#   - No model re-selection
#   - Continuous Validation -> Test chronology
#   - Delayed target feedback preserved
# ==================================================================================================

import time
import numpy as np
import pandas as pd

CELL17A_START = time.time()

print("=" * 125)
print("CELL 17A — SENSOR / INPUT NOISE ROBUSTNESS")
print("=" * 125)


# ==================================================================================================
# 1. REQUIRED OBJECTS FROM FINAL CELL 16
# ==================================================================================================

required = [
    "TASK_DEFINITIONS",
    "TASK_DATASETS",
    "TASK_CLIENT_DATASETS",
    "EXPECTED_STATIONS",
    "ACTIVE_FEATURE_COLUMNS",
    "FEDERATED_RUNS",
    "LOCAL_MODELS",
    "predict_raw",
    "reg_metrics",
    "U1619_SELECTION_DF",
    "U1619_NOVEL_TEST_RESULTS_DF",
    "U1619_TEST_PREDICTIONS_DF",
    "u1619_selected_algorithm",
    "u1619_peer_weights",
    "u1619_run_online",
    "u1619_static_predictions",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise NameError(
        "Missing required Cell 16 objects:\n- "
        + "\n- ".join(missing)
    )

print("✓ Corrected Cell 16 objects found.")


# ==================================================================================================
# 2. FIXED ROBUSTNESS PROTOCOL
# ==================================================================================================

NOISE_LEVELS_17A = [
    0.00,
    0.01,
    0.02,
    0.05,
    0.10,
]

NOISE_SEEDS_17A = [
    1701,
    1702,
    1703,
    1704,
    1705,
]

NON_STATION_FEATURE_IDX_17A = [
    i
    for i, col in enumerate(ACTIVE_FEATURE_COLUMNS)
    if not str(col).startswith("Station_")
]

STATION_FEATURE_IDX_17A = [
    i
    for i, col in enumerate(ACTIVE_FEATURE_COLUMNS)
    if str(col).startswith("Station_")
]


# ==================================================================================================
# 3. STATIC PREDICTIONS USING AN OVERRIDE X
# ==================================================================================================

def noise17a_static_predictions(
    task_key,
    station,
    X_override,
):

    X = np.asarray(
        X_override,
        dtype=float,
    )

    algorithm_key = (
        u1619_selected_algorithm(
            task_key
        )
    )

    # Frozen FedAdam
    global_parameters = (
        FEDERATED_RUNS[
            (
                task_key,
                algorithm_key,
                "FedAdam",
            )
        ]["Parameters"]
    )

    global_prediction = predict_raw(
        X,
        global_parameters,
        task_key,
    )

    # Frozen self-local
    self_parameters = (
        LOCAL_MODELS[
            (
                task_key,
                station,
                algorithm_key,
            )
        ]["Parameters"]
    )

    self_prediction = predict_raw(
        X,
        self_parameters,
        task_key,
    )

    # Frozen peer-only relation topology
    peer_weights = (
        u1619_peer_weights(
            task_key,
            station,
        )
    )

    relation_prediction = np.zeros(
        len(X),
        dtype=float,
    )

    for peer_station, weight in (
        peer_weights.items()
    ):

        peer_parameters = (
            LOCAL_MODELS[
                (
                    task_key,
                    peer_station,
                    algorithm_key,
                )
            ]["Parameters"]
        )

        peer_prediction = predict_raw(
            X,
            peer_parameters,
            task_key,
        )

        relation_prediction += (
            float(weight)
            * np.asarray(
                peer_prediction,
                dtype=float,
            )
        )

    return {
        "Global":
            np.asarray(
                global_prediction,
                dtype=float,
            ),

        "Self":
            np.asarray(
                self_prediction,
                dtype=float,
            ),

        "Relation":
            np.asarray(
                relation_prediction,
                dtype=float,
            ),
    }


# ==================================================================================================
# 4. FROZEN SELECTED CELL-16 CONFIG
# ==================================================================================================

def noise17a_selected_config(
    task_key,
    station,
):

    row = (
        U1619_SELECTION_DF[
            U1619_SELECTION_DF[
                "Task"
            ].eq(task_key)
            &
            U1619_SELECTION_DF[
                "Station"
            ].eq(station)
        ]
        .iloc[0]
    )

    return {
        "prior_global":
            float(
                row[
                    "prior_global"
                ]
            ),

        "expert_power":
            float(
                row[
                    "expert_power"
                ]
            ),

        "lookback":
            int(
                row[
                    "lookback"
                ]
            ),

        "decay":
            float(
                row[
                    "decay"
                ]
            ),

        "alpha":
            float(
                row[
                    "alpha"
                ]
            ),

        "local_relation_mix":
            float(
                row[
                    "local_relation_mix"
                ]
            ),
    }


# ==================================================================================================
# 5. RUN ONE TASK × STATION × NOISE LEVEL × SEED
# ==================================================================================================

def noise17a_run_one(
    task_key,
    station,
    noise_level,
    seed,
):

    meta = (
        TASK_DATASETS[
            task_key
        ]["Meta"]
    )

    validation_ds = (
        TASK_CLIENT_DATASETS[
            task_key
        ][station][
            "Validation"
        ]
    )

    test_ds = (
        TASK_CLIENT_DATASETS[
            task_key
        ][station][
            "Test"
        ]
    )

    validation_frame = (
        validation_ds[
            "Frame"
        ]
        .reset_index(
            drop=True
        )
    )

    test_frame = (
        test_ds[
            "Frame"
        ]
        .reset_index(
            drop=True
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Clean validation — exact frozen Cell 16 behavior
    # ----------------------------------------------------------------------------------------------

    validation_experts = (
        u1619_static_predictions(
            task_key,
            station,
            "Validation",
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Perturb Test X only
    #
    # X is already standardized from TRAIN-only scaling.
    # Therefore sigma=noise_level means 1%, 2%, 5%, 10% of one standardized unit.
    # ----------------------------------------------------------------------------------------------

    X_test_clean = np.asarray(
        test_ds["X"],
        dtype=float,
    )

    X_test_noisy = (
        X_test_clean.copy()
    )

    if noise_level > 0.0:

        rng = np.random.default_rng(
            int(seed)
        )

        noise = rng.normal(
            loc=0.0,
            scale=float(
                noise_level
            ),
            size=(
                len(
                    X_test_noisy
                ),
                len(
                    NON_STATION_FEATURE_IDX_17A
                ),
            ),
        )

        X_test_noisy[
            :,
            NON_STATION_FEATURE_IDX_17A
        ] += noise

    # Station one-hot columns unchanged automatically.

    test_experts = (
        noise17a_static_predictions(
            task_key,
            station,
            X_test_noisy,
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Continuous Validation -> Test chronology
    # ----------------------------------------------------------------------------------------------

    all_dates = pd.concat(
        [
            validation_frame[
                "Date"
            ],
            test_frame[
                "Date"
            ],
        ],
        ignore_index=True,
    )

    all_target_dates = pd.concat(
        [
            validation_frame[
                meta[
                    "Future_Date_Column"
                ]
            ],
            test_frame[
                meta[
                    "Future_Date_Column"
                ]
            ],
        ],
        ignore_index=True,
    )

    all_y = np.concatenate(
        [
            np.asarray(
                validation_ds[
                    "y"
                ],
                dtype=float,
            ),
            np.asarray(
                test_ds[
                    "y"
                ],
                dtype=float,
            ),
        ]
    )

    all_global = np.concatenate(
        [
            validation_experts[
                "Global"
            ],
            test_experts[
                "Global"
            ],
        ]
    )

    all_self = np.concatenate(
        [
            validation_experts[
                "Self"
            ],
            test_experts[
                "Self"
            ],
        ]
    )

    all_relation = np.concatenate(
        [
            validation_experts[
                "Relation"
            ],
            test_experts[
                "Relation"
            ],
        ]
    )

    cfg = (
        noise17a_selected_config(
            task_key,
            station,
        )
    )

    all_prediction, state = (
        u1619_run_online(
            dates=
                all_dates,

            target_dates=
                all_target_dates,

            y=
                all_y,

            global_pred=
                all_global,

            self_pred=
                all_self,

            relation_pred=
                all_relation,

            **cfg,
        )
    )

    n_validation = len(
        validation_ds["y"]
    )

    test_prediction = (
        all_prediction[
            n_validation:
        ]
    )

    return {
        "Actual":
            np.asarray(
                test_ds[
                    "y"
                ],
                dtype=float,
            ),

        "Prediction":
            np.asarray(
                test_prediction,
                dtype=float,
            ),

        "Frame":
            test_frame,
    }


# ==================================================================================================
# 6. FULL SENSOR-NOISE EXPERIMENT
# ==================================================================================================

task_keys = [
    d["Task"]
    for d in TASK_DEFINITIONS
]

prediction_rows = []

total_runs = (
    len(task_keys)
    * len(EXPECTED_STATIONS)
    * len(NOISE_LEVELS_17A)
    * len(NOISE_SEEDS_17A)
)

run_counter = 0

for noise_level in NOISE_LEVELS_17A:

    print(
        f"\nNoise level: "
        f"{100.0 * noise_level:.0f}%"
    )

    for seed in NOISE_SEEDS_17A:

        for task_key in task_keys:

            meta = (
                TASK_DATASETS[
                    task_key
                ]["Meta"]
            )

            for station in EXPECTED_STATIONS:

                run_counter += 1

                result = (
                    noise17a_run_one(
                        task_key=
                            task_key,

                        station=
                            station,

                        noise_level=
                            noise_level,

                        seed=
                            seed,
                    )
                )

                frame = (
                    result[
                        "Frame"
                    ]
                )

                for i, (
                    actual,
                    prediction,
                ) in enumerate(
                    zip(
                        result[
                            "Actual"
                        ],
                        result[
                            "Prediction"
                        ],
                    )
                ):

                    prediction_rows.append({
                        "Noise_Level":
                            float(
                                noise_level
                            ),

                        "Noise_Percent":
                            100.0
                            * float(
                                noise_level
                            ),

                        "Seed":
                            int(seed),

                        "Task":
                            task_key,

                        "Target":
                            meta[
                                "Target"
                            ],

                        "Horizon":
                            meta[
                                "Horizon"
                            ],

                        "Station":
                            station,

                        "Date":
                            frame.loc[
                                i,
                                "Date",
                            ],

                        "Target_Date":
                            frame.loc[
                                i,
                                meta[
                                    "Future_Date_Column"
                                ],
                            ],

                        "Actual":
                            float(
                                actual
                            ),

                        "Prediction":
                            float(
                                prediction
                            ),
                    })


SENSOR17A_PREDICTIONS_DF = (
    pd.DataFrame(
        prediction_rows
    )
)


# ==================================================================================================
# 7. TASK-LEVEL RESULTS BY SEED
# ==================================================================================================

task_seed_rows = []

for (
    noise_level,
    seed,
    task_key,
), group in (
    SENSOR17A_PREDICTIONS_DF
    .groupby(
        [
            "Noise_Level",
            "Seed",
            "Task",
        ]
    )
):

    metrics = reg_metrics(
        group[
            "Actual"
        ],
        group[
            "Prediction"
        ],
    )

    task_seed_rows.append({
        "Noise_Level":
            noise_level,

        "Noise_Percent":
            100.0
            * noise_level,

        "Seed":
            seed,

        "Task":
            task_key,

        **metrics,
    })


SENSOR17A_TASK_SEED_RESULTS_DF = (
    pd.DataFrame(
        task_seed_rows
    )
)


# ==================================================================================================
# 8. MEAN TASK RESULTS ACROSS FIVE SEEDS
# ==================================================================================================

SENSOR17A_TASK_RESULTS_DF = (
    SENSOR17A_TASK_SEED_RESULTS_DF
    .groupby(
        [
            "Noise_Level",
            "Noise_Percent",
            "Task",
        ],
        as_index=False,
    )
    .agg(
        Repetitions=(
            "Seed",
            "nunique",
        ),

        Mean_MAE=(
            "MAE",
            "mean",
        ),

        SD_MAE=(
            "MAE",
            "std",
        ),

        Mean_RMSE=(
            "RMSE",
            "mean",
        ),

        SD_RMSE=(
            "RMSE",
            "std",
        ),

        Mean_R2=(
            "R2",
            "mean",
        ),

        SD_R2=(
            "R2",
            "std",
        ),

        Mean_NRMSE_Std=(
            "NRMSE_Std",
            "mean",
        ),
    )
)


# ==================================================================================================
# 9. OVERALL RESULTS
# ==================================================================================================

overall_rows = []

for (
    noise_level,
    seed,
), group in (
    SENSOR17A_TASK_SEED_RESULTS_DF
    .groupby(
        [
            "Noise_Level",
            "Seed",
        ]
    )
):

    overall_rows.append({
        "Noise_Level":
            noise_level,

        "Noise_Percent":
            100.0
            * noise_level,

        "Seed":
            seed,

        "Mean_MAE":
            float(
                group[
                    "MAE"
                ].mean()
            ),

        "Mean_RMSE":
            float(
                group[
                    "RMSE"
                ].mean()
            ),

        "Mean_R2":
            float(
                group[
                    "R2"
                ].mean()
            ),

        "Mean_NRMSE_Std":
            float(
                group[
                    "NRMSE_Std"
                ].mean()
            ),
    })


SENSOR17A_OVERALL_SEED_DF = (
    pd.DataFrame(
        overall_rows
    )
)


SENSOR17A_OVERALL_DF = (
    SENSOR17A_OVERALL_SEED_DF
    .groupby(
        [
            "Noise_Level",
            "Noise_Percent",
        ],
        as_index=False,
    )
    .agg(
        Repetitions=(
            "Seed",
            "nunique",
        ),

        Mean_MAE=(
            "Mean_MAE",
            "mean",
        ),

        SD_MAE=(
            "Mean_MAE",
            "std",
        ),

        Mean_RMSE=(
            "Mean_RMSE",
            "mean",
        ),

        SD_RMSE=(
            "Mean_RMSE",
            "std",
        ),

        Mean_R2=(
            "Mean_R2",
            "mean",
        ),

        SD_R2=(
            "Mean_R2",
            "std",
        ),

        Mean_NRMSE_Std=(
            "Mean_NRMSE_Std",
            "mean",
        ),
    )
)


# ==================================================================================================
# 10. CLEAN REFERENCE AND DEGRADATION
# ==================================================================================================

clean_reference = (
    SENSOR17A_OVERALL_DF[
        SENSOR17A_OVERALL_DF[
            "Noise_Level"
        ].eq(0.0)
    ]
    .iloc[0]
)

clean_mae = float(
    clean_reference[
        "Mean_MAE"
    ]
)

clean_rmse = float(
    clean_reference[
        "Mean_RMSE"
    ]
)

clean_r2 = float(
    clean_reference[
        "Mean_R2"
    ]
)


SENSOR17A_OVERALL_DF[
    "MAE_Degradation_Percent"
] = (
    100.0
    * (
        SENSOR17A_OVERALL_DF[
            "Mean_MAE"
        ]
        - clean_mae
    )
    / clean_mae
)


SENSOR17A_OVERALL_DF[
    "RMSE_Degradation_Percent"
] = (
    100.0
    * (
        SENSOR17A_OVERALL_DF[
            "Mean_RMSE"
        ]
        - clean_rmse
    )
    / clean_rmse
)


SENSOR17A_OVERALL_DF[
    "R2_Loss"
] = (
    clean_r2
    - SENSOR17A_OVERALL_DF[
        "Mean_R2"
    ]
)


# ==================================================================================================
# 11. TASK DEGRADATION RELATIVE TO CLEAN
# ==================================================================================================

clean_task = (
    SENSOR17A_TASK_RESULTS_DF[
        SENSOR17A_TASK_RESULTS_DF[
            "Noise_Level"
        ].eq(0.0)
    ][
        [
            "Task",
            "Mean_MAE",
            "Mean_RMSE",
            "Mean_R2",
        ]
    ]
    .rename(
        columns={
            "Mean_MAE":
                "Clean_MAE",

            "Mean_RMSE":
                "Clean_RMSE",

            "Mean_R2":
                "Clean_R2",
        }
    )
)


SENSOR17A_TASK_RESULTS_DF = (
    SENSOR17A_TASK_RESULTS_DF
    .merge(
        clean_task,
        on="Task",
        how="left",
    )
)


SENSOR17A_TASK_RESULTS_DF[
    "MAE_Degradation_Percent"
] = (
    100.0
    * (
        SENSOR17A_TASK_RESULTS_DF[
            "Mean_MAE"
        ]
        - SENSOR17A_TASK_RESULTS_DF[
            "Clean_MAE"
        ]
    )
    / SENSOR17A_TASK_RESULTS_DF[
        "Clean_MAE"
    ]
)


SENSOR17A_TASK_RESULTS_DF[
    "RMSE_Degradation_Percent"
] = (
    100.0
    * (
        SENSOR17A_TASK_RESULTS_DF[
            "Mean_RMSE"
        ]
        - SENSOR17A_TASK_RESULTS_DF[
            "Clean_RMSE"
        ]
    )
    / SENSOR17A_TASK_RESULTS_DF[
        "Clean_RMSE"
    ]
)


SENSOR17A_TASK_RESULTS_DF[
    "R2_Loss"
] = (
    SENSOR17A_TASK_RESULTS_DF[
        "Clean_R2"
    ]
    - SENSOR17A_TASK_RESULTS_DF[
        "Mean_R2"
    ]
)


# ==================================================================================================
# 12. ROBUSTNESS LABEL — DESCRIPTIVE ONLY
# ==================================================================================================

def sensor17a_label(
    mae_degradation_percent,
):

    value = abs(
        float(
            mae_degradation_percent
        )
    )

    if value < 2.0:
        return "Very robust"

    if value < 5.0:
        return "Robust"

    if value < 10.0:
        return "Moderately robust"

    return "Sensitive"


SENSOR17A_OVERALL_DF[
    "Robustness"
] = (
    SENSOR17A_OVERALL_DF[
        "MAE_Degradation_Percent"
    ]
    .apply(
        sensor17a_label
    )
)


# ==================================================================================================
# 13. MANDATORY CLEAN REPRODUCTION CHECK AGAINST CELL 16
# ==================================================================================================

reference = (
    U1619_NOVEL_TEST_RESULTS_DF[
        [
            "Task",
            "MAE",
            "RMSE",
            "R2",
        ]
    ]
    .copy()
)

observed_clean = (
    SENSOR17A_TASK_RESULTS_DF[
        SENSOR17A_TASK_RESULTS_DF[
            "Noise_Level"
        ].eq(0.0)
    ][
        [
            "Task",
            "Mean_MAE",
            "Mean_RMSE",
            "Mean_R2",
        ]
    ]
    .copy()
)


SENSOR17A_CLEAN_REPRODUCTION_DF = (
    observed_clean
    .merge(
        reference,
        on="Task",
        how="left",
    )
)


SENSOR17A_CLEAN_REPRODUCTION_DF[
    "MAE_Difference"
] = (
    SENSOR17A_CLEAN_REPRODUCTION_DF[
        "Mean_MAE"
    ]
    - SENSOR17A_CLEAN_REPRODUCTION_DF[
        "MAE"
    ]
).abs()


SENSOR17A_CLEAN_REPRODUCTION_DF[
    "RMSE_Difference"
] = (
    SENSOR17A_CLEAN_REPRODUCTION_DF[
        "Mean_RMSE"
    ]
    - SENSOR17A_CLEAN_REPRODUCTION_DF[
        "RMSE"
    ]
).abs()


SENSOR17A_CLEAN_REPRODUCTION_DF[
    "R2_Difference"
] = (
    SENSOR17A_CLEAN_REPRODUCTION_DF[
        "Mean_R2"
    ]
    - SENSOR17A_CLEAN_REPRODUCTION_DF[
        "R2"
    ]
).abs()


SENSOR17A_CLEAN_REPRODUCTION_DF[
    "Reproduced"
] = (
    SENSOR17A_CLEAN_REPRODUCTION_DF[
        [
            "MAE_Difference",
            "RMSE_Difference",
            "R2_Difference",
        ]
    ]
    .max(
        axis=1
    )
    < 1e-10
)


if not (
    SENSOR17A_CLEAN_REPRODUCTION_DF[
        "Reproduced"
    ].all()
):
    raise RuntimeError(
        "Cell 17A clean 0% condition does not exactly reproduce final Cell 16."
    )


# ==================================================================================================
# 14. PROTOCOL
# ==================================================================================================

SENSOR17A_PROTOCOL_DF = pd.DataFrame([
    {
        "Item":
            "Test",
        "Value":
            "Sensor / input Gaussian noise robustness",
    },
    {
        "Item":
            "Noise levels",
        "Value":
            "0%, 1%, 2%, 5%, 10%",
    },
    {
        "Item":
            "Repetitions per level",
        "Value":
            5,
    },
    {
        "Item":
            "Seeds",
        "Value":
            "1701–1705",
    },
    {
        "Item":
            "Noise space",
        "Value":
            "Standardized Test features",
    },
    {
        "Item":
            "Station one-hot perturbed",
        "Value":
            False,
    },
    {
        "Item":
            "Validation perturbed",
        "Value":
            False,
    },
    {
        "Item":
            "Retraining",
        "Value":
            False,
    },
    {
        "Item":
            "Hyperparameter tuning",
        "Value":
            False,
    },
    {
        "Item":
            "Model re-selection",
        "Value":
            False,
    },
    {
        "Item":
            "Delayed feedback preserved",
        "Value":
            True,
    },
    {
        "Item":
            "Validation/Test chronology",
        "Value":
            "Continuous",
    },
])


# ==================================================================================================
# 15. OUTPUT
# ==================================================================================================

print(
    "\nMandatory 0% clean reproduction:"
)

display(
    SENSOR17A_CLEAN_REPRODUCTION_DF
    .round(12)
)


print(
    "\nOverall sensor/input-noise robustness:"
)

display(
    SENSOR17A_OVERALL_DF
    .round(6)
)


print(
    "\nTask-level robustness:"
)

display(
    SENSOR17A_TASK_RESULTS_DF[
        [
            "Noise_Percent",
            "Task",
            "Repetitions",
            "Mean_MAE",
            "Mean_RMSE",
            "Mean_R2",
            "MAE_Degradation_Percent",
            "RMSE_Degradation_Percent",
            "R2_Loss",
        ]
    ]
    .round(6)
)


print(
    "\nProtocol:"
)

display(
    SENSOR17A_PROTOCOL_DF
)


print("\n" + "=" * 125)
print("CELL 17A COMPLETED SUCCESSFULLY")
print("=" * 125)

print(
    f"Execution time: "
    f"{time.time() - CELL17A_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 17B — INFERENCE COMMUNICATION / NETWORK ROBUSTNESS
# FINAL CORRECTED VERSION — compatible with corrected Cell 16
# ==================================================================================================
# Protocol:
#   - Frozen final Cell-16 model
#   - Peer-message dropout at TEST inference only
#   - Loss levels: 0%, 5%, 10%, 20%
#   - 5 repetitions: seeds 2711..2715
#   - Available peer weights are renormalized
#   - If all peers are unavailable -> self-local fallback
#   - Validation remains clean
#   - No retraining
#   - No tuning
#   - No model re-selection
#   - Delayed target feedback preserved
# ==================================================================================================

import time
import numpy as np
import pandas as pd

CELL17B_START = time.time()

print("=" * 125)
print("CELL 17B — INFERENCE COMMUNICATION / NETWORK ROBUSTNESS")
print("=" * 125)


# ==================================================================================================
# 1. REQUIRED OBJECTS
# ==================================================================================================

required = [
    "TASK_DEFINITIONS",
    "TASK_DATASETS",
    "TASK_CLIENT_DATASETS",
    "EXPECTED_STATIONS",
    "FEDERATED_RUNS",
    "LOCAL_MODELS",
    "predict_raw",
    "reg_metrics",
    "U1619_SELECTION_DF",
    "U1619_NOVEL_TEST_RESULTS_DF",
    "u1619_selected_algorithm",
    "u1619_peer_weights",
    "u1619_run_online",
    "u1619_static_predictions",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise NameError(
        "Missing required Cell 16 objects:\n- "
        + "\n- ".join(missing)
    )

print("✓ Corrected Cell 16 objects found.")


# ==================================================================================================
# 2. FIXED ROBUSTNESS SETTINGS
# ==================================================================================================

NETWORK_LOSS_LEVELS_17B = [
    0.00,
    0.05,
    0.10,
    0.20,
]

NETWORK_SEEDS_17B = [
    2711,
    2712,
    2713,
    2714,
    2715,
]


# ==================================================================================================
# 3. GET FROZEN CONFIG
# ==================================================================================================

def net17b_selected_config(
    task_key,
    station,
):

    row = (
        U1619_SELECTION_DF[
            U1619_SELECTION_DF["Task"].eq(task_key)
            &
            U1619_SELECTION_DF["Station"].eq(station)
        ]
        .iloc[0]
    )

    return {
        "prior_global":
            float(row["prior_global"]),

        "expert_power":
            float(row["expert_power"]),

        "lookback":
            int(row["lookback"]),

        "decay":
            float(row["decay"]),

        "alpha":
            float(row["alpha"]),

        "local_relation_mix":
            float(row["local_relation_mix"]),
    }


# ==================================================================================================
# 4. PEER-DROPOUT RELATION PREDICTION
# ==================================================================================================

def net17b_relation_with_dropout(
    task_key,
    target_station,
    split_name,
    loss_probability,
    rng,
):

    dataset = (
        TASK_CLIENT_DATASETS[
            task_key
        ][target_station][split_name]
    )

    X = np.asarray(
        dataset["X"],
        dtype=float,
    )

    algorithm_key = (
        u1619_selected_algorithm(
            task_key
        )
    )

    base_weights = (
        u1619_peer_weights(
            task_key,
            target_station,
        )
    )

    peers = list(
        base_weights.keys()
    )

    n = len(X)

    relation_prediction = np.zeros(
        n,
        dtype=float,
    )

    active_peer_count = np.zeros(
        n,
        dtype=float,
    )

    complete_failure = np.zeros(
        n,
        dtype=bool,
    )

    peer_predictions = {}

    for peer in peers:

        parameters = (
            LOCAL_MODELS[
                (
                    task_key,
                    peer,
                    algorithm_key,
                )
            ]["Parameters"]
        )

        peer_predictions[peer] = (
            np.asarray(
                predict_raw(
                    X,
                    parameters,
                    task_key,
                ),
                dtype=float,
            )
        )

    self_parameters = (
        LOCAL_MODELS[
            (
                task_key,
                target_station,
                algorithm_key,
            )
        ]["Parameters"]
    )

    self_prediction = np.asarray(
        predict_raw(
            X,
            self_parameters,
            task_key,
        ),
        dtype=float,
    )

    for i in range(n):

        available = []

        for peer in peers:

            delivered = (
                rng.random()
                >= float(
                    loss_probability
                )
            )

            if delivered:
                available.append(
                    peer
                )

        active_peer_count[i] = (
            len(available)
        )

        if len(available) == 0:

            relation_prediction[i] = (
                self_prediction[i]
            )

            complete_failure[i] = True

        else:

            weight_sum = sum(
                base_weights[p]
                for p in available
            )

            relation_prediction[i] = sum(
                (
                    base_weights[p]
                    / weight_sum
                )
                * peer_predictions[p][i]
                for p in available
            )

    return {
        "Relation":
            relation_prediction,

        "Self":
            self_prediction,

        "Active_Peers":
            active_peer_count,

        "Complete_Failure":
            complete_failure,

        "Total_Peers":
            len(peers),
    }


# ==================================================================================================
# 5. ONE RUN
# ==================================================================================================

def net17b_run_one(
    task_key,
    station,
    loss_probability,
    seed,
):

    meta = (
        TASK_DATASETS[
            task_key
        ]["Meta"]
    )

    validation_ds = (
        TASK_CLIENT_DATASETS[
            task_key
        ][station]["Validation"]
    )

    test_ds = (
        TASK_CLIENT_DATASETS[
            task_key
        ][station]["Test"]
    )

    validation_frame = (
        validation_ds["Frame"]
        .reset_index(drop=True)
    )

    test_frame = (
        test_ds["Frame"]
        .reset_index(drop=True)
    )

    # Clean validation
    validation_experts = (
        u1619_static_predictions(
            task_key,
            station,
            "Validation",
        )
    )

    # Clean global Test expert
    test_clean_experts = (
        u1619_static_predictions(
            task_key,
            station,
            "Test",
        )
    )

    rng = np.random.default_rng(
        int(seed)
    )

    dropout_test = (
        net17b_relation_with_dropout(
            task_key=
                task_key,

            target_station=
                station,

            split_name=
                "Test",

            loss_probability=
                loss_probability,

            rng=
                rng,
        )
    )

    all_dates = pd.concat(
        [
            validation_frame["Date"],
            test_frame["Date"],
        ],
        ignore_index=True,
    )

    all_target_dates = pd.concat(
        [
            validation_frame[
                meta[
                    "Future_Date_Column"
                ]
            ],
            test_frame[
                meta[
                    "Future_Date_Column"
                ]
            ],
        ],
        ignore_index=True,
    )

    all_y = np.concatenate(
        [
            np.asarray(
                validation_ds["y"],
                dtype=float,
            ),
            np.asarray(
                test_ds["y"],
                dtype=float,
            ),
        ]
    )

    # FIX:
    # Extract arrays explicitly from dictionaries.
    combined_global = np.concatenate(
        [
            np.asarray(
                validation_experts["Global"],
                dtype=float,
            ),
            np.asarray(
                test_clean_experts["Global"],
                dtype=float,
            ),
        ]
    )

    combined_self = np.concatenate(
        [
            np.asarray(
                validation_experts["Self"],
                dtype=float,
            ),
            np.asarray(
                dropout_test["Self"],
                dtype=float,
            ),
        ]
    )

    combined_relation = np.concatenate(
        [
            np.asarray(
                validation_experts["Relation"],
                dtype=float,
            ),
            np.asarray(
                dropout_test["Relation"],
                dtype=float,
            ),
        ]
    )

    cfg = (
        net17b_selected_config(
            task_key,
            station,
        )
    )

    all_prediction, state = (
        u1619_run_online(
            dates=
                all_dates,

            target_dates=
                all_target_dates,

            y=
                all_y,

            global_pred=
                combined_global,

            self_pred=
                combined_self,

            relation_pred=
                combined_relation,

            **cfg,
        )
    )

    n_validation = len(
        validation_ds["y"]
    )

    return {
        "Actual":
            np.asarray(
                test_ds["y"],
                dtype=float,
            ),

        "Prediction":
            np.asarray(
                all_prediction[
                    n_validation:
                ],
                dtype=float,
            ),

        "Frame":
            test_frame,

        "Active_Peers":
            dropout_test[
                "Active_Peers"
            ],

        "Complete_Failure":
            dropout_test[
                "Complete_Failure"
            ],

        "Total_Peers":
            dropout_test[
                "Total_Peers"
            ],
    }


# ==================================================================================================
# 6. FULL EXPERIMENT
# ==================================================================================================

task_keys = [
    d["Task"]
    for d in TASK_DEFINITIONS
]

prediction_rows = []

for loss_probability in (
    NETWORK_LOSS_LEVELS_17B
):

    print(
        f"\nPeer-message loss: "
        f"{100 * loss_probability:.0f}%"
    )

    for seed in (
        NETWORK_SEEDS_17B
    ):

        for task_key in task_keys:

            meta = (
                TASK_DATASETS[
                    task_key
                ]["Meta"]
            )

            for station in (
                EXPECTED_STATIONS
            ):

                result = (
                    net17b_run_one(
                        task_key=
                            task_key,

                        station=
                            station,

                        loss_probability=
                            loss_probability,

                        seed=
                            seed,
                    )
                )

                frame = (
                    result[
                        "Frame"
                    ]
                )

                for i, (
                    actual,
                    prediction,
                ) in enumerate(
                    zip(
                        result[
                            "Actual"
                        ],
                        result[
                            "Prediction"
                        ],
                    )
                ):

                    prediction_rows.append({
                        "Loss_Probability":
                            float(
                                loss_probability
                            ),

                        "Loss_Percent":
                            100.0
                            * float(
                                loss_probability
                            ),

                        "Seed":
                            int(seed),

                        "Task":
                            task_key,

                        "Target":
                            meta[
                                "Target"
                            ],

                        "Horizon":
                            meta[
                                "Horizon"
                            ],

                        "Station":
                            station,

                        "Date":
                            frame.loc[
                                i,
                                "Date",
                            ],

                        "Target_Date":
                            frame.loc[
                                i,
                                meta[
                                    "Future_Date_Column"
                                ],
                            ],

                        "Actual":
                            float(
                                actual
                            ),

                        "Prediction":
                            float(
                                prediction
                            ),

                        "Active_Peers":
                            float(
                                result[
                                    "Active_Peers"
                                ][i]
                            ),

                        "Complete_Peer_Failure":
                            bool(
                                result[
                                    "Complete_Failure"
                                ][i]
                            ),

                        "Total_Peers":
                            int(
                                result[
                                    "Total_Peers"
                                ]
                            ),
                    })


NET17B_PREDICTIONS_DF = (
    pd.DataFrame(
        prediction_rows
    )
)


# ==================================================================================================
# 7. TASK × SEED RESULTS
# ==================================================================================================

task_seed_rows = []

for (
    loss,
    seed,
    task,
), group in (
    NET17B_PREDICTIONS_DF
    .groupby(
        [
            "Loss_Probability",
            "Seed",
            "Task",
        ]
    )
):

    metrics = reg_metrics(
        group["Actual"],
        group["Prediction"],
    )

    task_seed_rows.append({
        "Loss_Probability":
            loss,

        "Loss_Percent":
            100.0 * loss,

        "Seed":
            seed,

        "Task":
            task,

        **metrics,

        "Mean_Active_Peers":
            float(
                group[
                    "Active_Peers"
                ].mean()
            ),

        "Complete_Failure_Percent":
            100.0
            * float(
                group[
                    "Complete_Peer_Failure"
                ].mean()
            ),
    })


NET17B_TASK_SEED_RESULTS_DF = (
    pd.DataFrame(
        task_seed_rows
    )
)


# ==================================================================================================
# 8. TASK-LEVEL SUMMARY
# ==================================================================================================

NET17B_TASK_RESULTS_DF = (
    NET17B_TASK_SEED_RESULTS_DF
    .groupby(
        [
            "Loss_Probability",
            "Loss_Percent",
            "Task",
        ],
        as_index=False,
    )
    .agg(
        Repetitions=(
            "Seed",
            "nunique",
        ),

        Mean_MAE=(
            "MAE",
            "mean",
        ),

        SD_MAE=(
            "MAE",
            "std",
        ),

        Mean_RMSE=(
            "RMSE",
            "mean",
        ),

        Mean_R2=(
            "R2",
            "mean",
        ),

        Mean_Active_Peers=(
            "Mean_Active_Peers",
            "mean",
        ),

        Complete_Failure_Percent=(
            "Complete_Failure_Percent",
            "mean",
        ),
    )
)


# ==================================================================================================
# 9. OVERALL SUMMARY
# ==================================================================================================

seed_overall_rows = []

for (
    loss,
    seed,
), group in (
    NET17B_TASK_SEED_RESULTS_DF
    .groupby(
        [
            "Loss_Probability",
            "Seed",
        ]
    )
):

    seed_overall_rows.append({
        "Loss_Probability":
            loss,

        "Loss_Percent":
            100.0 * loss,

        "Seed":
            seed,

        "Mean_MAE":
            float(
                group["MAE"].mean()
            ),

        "Mean_RMSE":
            float(
                group["RMSE"].mean()
            ),

        "Mean_R2":
            float(
                group["R2"].mean()
            ),

        "Mean_Active_Peers":
            float(
                group[
                    "Mean_Active_Peers"
                ].mean()
            ),

        "Complete_Failure_Percent":
            float(
                group[
                    "Complete_Failure_Percent"
                ].mean()
            ),
    })


NET17B_OVERALL_SEED_DF = (
    pd.DataFrame(
        seed_overall_rows
    )
)


NET17B_OVERALL_DF = (
    NET17B_OVERALL_SEED_DF
    .groupby(
        [
            "Loss_Probability",
            "Loss_Percent",
        ],
        as_index=False,
    )
    .agg(
        Repetitions=(
            "Seed",
            "nunique",
        ),

        Mean_MAE=(
            "Mean_MAE",
            "mean",
        ),

        SD_MAE=(
            "Mean_MAE",
            "std",
        ),

        Mean_RMSE=(
            "Mean_RMSE",
            "mean",
        ),

        SD_RMSE=(
            "Mean_RMSE",
            "std",
        ),

        Mean_R2=(
            "Mean_R2",
            "mean",
        ),

        SD_R2=(
            "Mean_R2",
            "std",
        ),

        Mean_Active_Peers=(
            "Mean_Active_Peers",
            "mean",
        ),

        Complete_Failure_Percent=(
            "Complete_Failure_Percent",
            "mean",
        ),
    )
)


# ==================================================================================================
# 10. DEGRADATION VS CLEAN
# ==================================================================================================

clean = (
    NET17B_OVERALL_DF[
        NET17B_OVERALL_DF[
            "Loss_Probability"
        ].eq(0.0)
    ]
    .iloc[0]
)

clean_mae = float(
    clean["Mean_MAE"]
)

clean_rmse = float(
    clean["Mean_RMSE"]
)

clean_r2 = float(
    clean["Mean_R2"]
)


NET17B_OVERALL_DF[
    "MAE_Degradation_Percent"
] = (
    100.0
    * (
        NET17B_OVERALL_DF[
            "Mean_MAE"
        ]
        - clean_mae
    )
    / clean_mae
)


NET17B_OVERALL_DF[
    "RMSE_Degradation_Percent"
] = (
    100.0
    * (
        NET17B_OVERALL_DF[
            "Mean_RMSE"
        ]
        - clean_rmse
    )
    / clean_rmse
)


NET17B_OVERALL_DF[
    "R2_Loss"
] = (
    clean_r2
    - NET17B_OVERALL_DF[
        "Mean_R2"
    ]
)


# ==================================================================================================
# 11. CLEAN REPRODUCTION CHECK
# ==================================================================================================

reference = (
    U1619_NOVEL_TEST_RESULTS_DF[
        [
            "Task",
            "MAE",
            "RMSE",
            "R2",
        ]
    ]
    .copy()
)

clean_task = (
    NET17B_TASK_RESULTS_DF[
        NET17B_TASK_RESULTS_DF[
            "Loss_Probability"
        ].eq(0.0)
    ][
        [
            "Task",
            "Mean_MAE",
            "Mean_RMSE",
            "Mean_R2",
        ]
    ]
    .copy()
)


NET17B_CLEAN_REPRODUCTION_DF = (
    clean_task.merge(
        reference,
        on="Task",
        how="left",
    )
)


NET17B_CLEAN_REPRODUCTION_DF[
    "MAE_Difference"
] = (
    NET17B_CLEAN_REPRODUCTION_DF[
        "Mean_MAE"
    ]
    - NET17B_CLEAN_REPRODUCTION_DF[
        "MAE"
    ]
).abs()


NET17B_CLEAN_REPRODUCTION_DF[
    "RMSE_Difference"
] = (
    NET17B_CLEAN_REPRODUCTION_DF[
        "Mean_RMSE"
    ]
    - NET17B_CLEAN_REPRODUCTION_DF[
        "RMSE"
    ]
).abs()


NET17B_CLEAN_REPRODUCTION_DF[
    "R2_Difference"
] = (
    NET17B_CLEAN_REPRODUCTION_DF[
        "Mean_R2"
    ]
    - NET17B_CLEAN_REPRODUCTION_DF[
        "R2"
    ]
).abs()


NET17B_CLEAN_REPRODUCTION_DF[
    "Reproduced"
] = (
    NET17B_CLEAN_REPRODUCTION_DF[
        [
            "MAE_Difference",
            "RMSE_Difference",
            "R2_Difference",
        ]
    ]
    .max(axis=1)
    < 1e-10
)


if not (
    NET17B_CLEAN_REPRODUCTION_DF[
        "Reproduced"
    ].all()
):
    raise RuntimeError(
        "Cell 17B clean 0% condition does not exactly reproduce Cell 16."
    )


# ==================================================================================================
# 12. PROTOCOL
# ==================================================================================================

NET17B_PROTOCOL_DF = pd.DataFrame([
    {
        "Item": "Model",
        "Value":
            "UTW-RG-AdaFedResidual",
    },
    {
        "Item": "Configuration",
        "Value":
            "Frozen from Cell 16",
    },
    {
        "Item": "Network loss levels",
        "Value":
            "0%, 5%, 10%, 20%",
    },
    {
        "Item": "Perturbation type",
        "Value":
            "Independent peer-message dropout",
    },
    {
        "Item": "Perturbation stage",
        "Value":
            "Test inference only",
    },
    {
        "Item": "Available peer weights",
        "Value":
            "Renormalized",
    },
    {
        "Item": "No-peer fallback",
        "Value":
            "Self-local prediction",
    },
    {
        "Item": "Repetitions",
        "Value":
            5,
    },
    {
        "Item": "Seeds",
        "Value":
            "2711–2715",
    },
    {
        "Item": "Retraining",
        "Value":
            False,
    },
    {
        "Item": "Hyperparameter tuning",
        "Value":
            False,
    },
    {
        "Item": "Test used for selection",
        "Value":
            False,
    },
    {
        "Item": "Delayed feedback preserved",
        "Value":
            True,
    },
])


# ==================================================================================================
# 13. OUTPUT
# ==================================================================================================

print(
    "\nMandatory 0% communication-loss reproduction:"
)

display(
    NET17B_CLEAN_REPRODUCTION_DF
    .round(12)
)


print(
    "\nOverall inference communication robustness:"
)

display(
    NET17B_OVERALL_DF
    .round(6)
)


print(
    "\nTask-level results:"
)

display(
    NET17B_TASK_RESULTS_DF
    .round(6)
)


print(
    "\nProtocol:"
)

display(
    NET17B_PROTOCOL_DF
)


print("\n" + "=" * 125)
print("CELL 17B COMPLETED SUCCESSFULLY")
print("=" * 125)

print(
    f"Execution time: "
    f"{time.time() - CELL17B_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 17C — FEDERATED TRAINING COMMUNICATION ROBUSTNESS
# FINAL SELF-CONTAINED CORRECTED VERSION
# Compatible with corrected Cell 16
# ==================================================================================================

import time
import numpy as np
import pandas as pd

CELL17C_START = time.time()

print("=" * 140)
print("CELL 17C — FEDERATED TRAINING COMMUNICATION ROBUSTNESS")
print("=" * 140)


# ==================================================================================================
# 1. REQUIRED OBJECTS
# ==================================================================================================

required_objects = [
    "TASK_DEFINITIONS",
    "TASK_DATASETS",
    "TASK_CLIENT_DATASETS",
    "TASK_SCALERS",
    "EXPECTED_STATIONS",
    "ALGORITHM_CONFIGS",
    "CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM",
    "FEDERATED_RUNS",
    "FEDERATED_METHOD_NAMES",
    "batch_gradient",
    "clip_grad",
    "predict_raw",
    "reg_metrics",
    "GLOBAL_RANDOM_SEED",
    "GRADIENT_CLIP_NORM",
    "FEDERATED_ROUNDS",
    "FEDERATED_LOCAL_EPOCHS",
    "FEDADAM_SERVER_LR",
    "FEDADAM_BETA1",
    "FEDADAM_BETA2",
    "FEDADAM_TAU",
    "U1619_SELECTION_DF",
    "U1619_NOVEL_TEST_RESULTS_DF",
    "u1619_selected_algorithm",
    "u1619_static_predictions",
    "u1619_run_online",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise NameError(
        "Missing required objects:\n- "
        + "\n- ".join(missing)
    )

print("✓ Required Cell 13 / Cell 16 objects are available.")


# ==================================================================================================
# 2. FIXED PROTOCOL
# ==================================================================================================

FED17C_METHOD = "FedAdam"

FED17C_ROUNDS = int(
    FEDERATED_ROUNDS
)

FED17C_LOCAL_EPOCHS = int(
    FEDERATED_LOCAL_EPOCHS
)

FED17C_REPETITION_SEEDS = list(
    range(4701, 4711)
)

FED17C_TASKS = [
    d["Task"]
    for d in TASK_DEFINITIONS
]

FED17C_ALGORITHM_ORDER = list(
    ALGORITHM_CONFIGS.keys()
)

FED17C_METHOD_ORDER = list(
    FEDERATED_METHOD_NAMES
)

FED17C_METHOD_POSITION = (
    FED17C_METHOD_ORDER.index(
        FED17C_METHOD
    )
    + 1
)


# ==================================================================================================
# 3. COMMUNICATION STRESS CONDITIONS
# Fixed robustness conditions — NOT tuning parameters
# ==================================================================================================

FED17C_SCENARIOS = {

    "Clean": {
        "packet_loss": 0.00,
        "base_latency": 0.30,
        "latency_jitter": 0.00,
        "deadline": 1.00,
        "burst_start": 0.00,
        "burst_continue": 0.00,
        "unavailable": 0.00,
        "straggler": 0.00,
        "straggler_delay": 0.00,
    },

    "PacketLoss20": {
        "packet_loss": 0.20,
        "base_latency": 0.30,
        "latency_jitter": 0.00,
        "deadline": 1.00,
        "burst_start": 0.00,
        "burst_continue": 0.00,
        "unavailable": 0.00,
        "straggler": 0.00,
        "straggler_delay": 0.00,
    },

    "LatencyJitter": {
        "packet_loss": 0.00,
        "base_latency": 0.55,
        "latency_jitter": 0.30,
        "deadline": 1.00,
        "burst_start": 0.00,
        "burst_continue": 0.00,
        "unavailable": 0.00,
        "straggler": 0.00,
        "straggler_delay": 0.00,
    },

    "BurstOutage": {
        "packet_loss": 0.00,
        "base_latency": 0.30,
        "latency_jitter": 0.00,
        "deadline": 1.00,
        "burst_start": 0.08,
        "burst_continue": 0.70,
        "unavailable": 0.00,
        "straggler": 0.00,
        "straggler_delay": 0.00,
    },

    "Stragglers": {
        "packet_loss": 0.00,
        "base_latency": 0.30,
        "latency_jitter": 0.10,
        "deadline": 1.00,
        "burst_start": 0.00,
        "burst_continue": 0.00,
        "unavailable": 0.10,
        "straggler": 0.20,
        "straggler_delay": 0.80,
    },

    "CombinedStress": {
        "packet_loss": 0.10,
        "base_latency": 0.45,
        "latency_jitter": 0.20,
        "deadline": 1.00,
        "burst_start": 0.05,
        "burst_continue": 0.60,
        "unavailable": 0.05,
        "straggler": 0.15,
        "straggler_delay": 0.60,
    },
}


# ==================================================================================================
# 4. FROZEN CELL-10 CONFIG RESOLUTION
# ==================================================================================================

def fed17c_get_frozen_config(
    task,
    algorithm,
):

    record = (
        CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM[
            (
                task,
                algorithm,
            )
        ]
    )

    # Direct configuration dictionary
    if isinstance(record, dict):

        # Some notebook versions store config under a nested field.
        for nested_name in [
            "Config",
            "config",
            "Best_Config",
            "Selected_Config",
        ]:
            if (
                nested_name in record
                and isinstance(
                    record[nested_name],
                    dict,
                )
            ):
                record = record[
                    nested_name
                ]
                break

    if not isinstance(record, dict):
        raise TypeError(
            f"Frozen configuration for {(task, algorithm)} "
            "is not a dictionary."
        )

    cfg = dict(record)

    # Normalize possible column-style names
    aliases = {
        "Learning_Rate":
            "learning_rate",

        "Learning_Rate_Decay":
            "learning_rate_decay",

        "L1":
            "l1",

        "L2":
            "l2",

        "Huber_Delta":
            "huber_delta",

        "Batch_Size":
            "batch_size",

        "Epochs":
            "epochs",

        "Loss":
            "loss",
    }

    for old, new in aliases.items():
        if (
            new not in cfg
            and old in cfg
        ):
            cfg[new] = cfg[old]

    # Preserve algorithm baseline fields if the selected record
    # contains only the tuned values.
    base = dict(
        ALGORITHM_CONFIGS[
            algorithm
        ]
    )

    base.update(cfg)

    cfg = base

    # NaN batch size means full batch.
    if (
        "batch_size" in cfg
        and pd.isna(
            cfg["batch_size"]
        )
    ):
        cfg["batch_size"] = None

    if (
        "huber_delta" in cfg
        and pd.isna(
            cfg["huber_delta"]
        )
    ):
        cfg["huber_delta"] = None

    return cfg


# ==================================================================================================
# 5. EXACT LOCAL CLIENT UPDATE
#
# Self-contained to avoid any later redefinition of federated_client_update().
# For FedAdam the client optimization itself is ordinary local optimization;
# FedAdam acts at the SERVER aggregation/update step.
# ==================================================================================================

def fed17c_client_update_exact(
    X,
    y_raw,
    task,
    algorithm,
    global_parameters,
    local_epochs,
    round_index,
    seed,
):

    config = (
        fed17c_get_frozen_config(
            task,
            algorithm,
        )
    )

    scaler = TASK_SCALERS[
        task
    ]

    X = np.asarray(
        X,
        dtype=float,
    )

    y_raw = np.asarray(
        y_raw,
        dtype=float,
    )

    y = (
        y_raw
        - float(
            scaler["y_mean"]
        )
    ) / float(
        scaler["y_std"]
    )

    parameters = np.asarray(
        global_parameters,
        dtype=float,
    ).copy()

    n = len(y)

    batch_value = config.get(
        "batch_size",
        None,
    )

    if batch_value is None:
        batch_size = n
    else:
        batch_size = min(
            int(batch_value),
            n,
        )

    rng = np.random.default_rng(
        int(seed)
    )

    learning_rate = float(
        config[
            "learning_rate"
        ]
    )

    learning_rate_decay = float(
        config.get(
            "learning_rate_decay",
            1.0,
        )
    )

    l1_value = float(
        config.get(
            "l1",
            0.0,
        )
    )

    for local_epoch in range(
        int(local_epochs)
    ):

        global_pass_index = (
            int(round_index)
            * int(local_epochs)
            + local_epoch
        )

        current_lr = (
            learning_rate
            * (
                learning_rate_decay
                ** global_pass_index
            )
        )

        if batch_size == n:

            order = np.arange(
                n
            )

        else:

            order = rng.permutation(
                n
            )

        for start in range(
            0,
            n,
            batch_size,
        ):

            indices = order[
                start:
                start + batch_size
            ]

            gradient = (
                batch_gradient(
                    X[indices],
                    y[indices],
                    parameters,
                    config,
                )
            )

            gradient = clip_grad(
                gradient,
                GRADIENT_CLIP_NORM,
            )

            parameters -= (
                current_lr
                * gradient
            )

            # Proximal L1 step
            if l1_value > 0.0:

                weights = (
                    parameters[:-1]
                )

                parameters[:-1] = (
                    np.sign(weights)
                    * np.maximum(
                        np.abs(
                            weights
                        )
                        - current_lr
                        * l1_value,
                        0.0,
                    )
                )

    return parameters


# ==================================================================================================
# 6. CELL-16 FROZEN ONLINE CONFIG
# ==================================================================================================

def fed17c_rg_config(
    task,
    station,
):

    row = (
        U1619_SELECTION_DF[
            U1619_SELECTION_DF[
                "Task"
            ].eq(task)
            &
            U1619_SELECTION_DF[
                "Station"
            ].eq(station)
        ]
        .iloc[0]
    )

    return {
        "prior_global":
            float(
                row[
                    "prior_global"
                ]
            ),

        "expert_power":
            float(
                row[
                    "expert_power"
                ]
            ),

        "lookback":
            int(
                row[
                    "lookback"
                ]
            ),

        "decay":
            float(
                row[
                    "decay"
                ]
            ),

        "alpha":
            float(
                row[
                    "alpha"
                ]
            ),

        "local_relation_mix":
            float(
                row[
                    "local_relation_mix"
                ]
            ),
    }


# ==================================================================================================
# 7. COMMUNICATION DELIVERY SIMULATION
# ==================================================================================================

def fed17c_delivery(
    rng,
    scenario_name,
    station,
    burst_state,
):

    c = FED17C_SCENARIOS[
        scenario_name
    ]

    # Burst outage
    in_burst = burst_state[
        station
    ]

    if in_burst:

        in_burst = (
            rng.random()
            < c[
                "burst_continue"
            ]
        )

    else:

        in_burst = (
            rng.random()
            < c[
                "burst_start"
            ]
        )

    burst_state[
        station
    ] = in_burst

    if in_burst:
        return False, "Burst"

    # Client unavailable
    if (
        rng.random()
        < c[
            "unavailable"
        ]
    ):
        return (
            False,
            "Unavailable",
        )

    # Packet loss
    if (
        rng.random()
        < c[
            "packet_loss"
        ]
    ):
        return (
            False,
            "PacketLoss",
        )

    latency = float(
        c[
            "base_latency"
        ]
    )

    if (
        c[
            "latency_jitter"
        ]
        > 0
    ):

        latency += float(
            rng.normal(
                0.0,
                c[
                    "latency_jitter"
                ],
            )
        )

    latency = max(
        0.0,
        latency,
    )

    if (
        rng.random()
        < c[
            "straggler"
        ]
    ):

        latency += c[
            "straggler_delay"
        ]

    if (
        latency
        > c[
            "deadline"
        ]
    ):

        return (
            False,
            "DeadlineMiss",
        )

    return (
        True,
        "Delivered",
    )


# ==================================================================================================
# 8. FEDADAM TRAINING UNDER COMMUNICATION STRESS
# ==================================================================================================

def fed17c_train_fedadam(
    task,
    scenario,
    repetition_seed,
):

    task_position = (
        FED17C_TASKS.index(
            task
        )
        + 1
    )

    algorithm = (
        u1619_selected_algorithm(
            task
        )
    )

    algorithm_position = (
        FED17C_ALGORITHM_ORDER.index(
            algorithm
        )
        + 1
    )

    p = (
        TASK_CLIENT_DATASETS[
            task
        ][
            EXPECTED_STATIONS[0]
        ][
            "Train"
        ][
            "X"
        ].shape[1]
        + 1
    )

    global_parameters = np.zeros(
        p,
        dtype=float,
    )

    m = np.zeros(
        p,
        dtype=float,
    )

    v = np.zeros(
        p,
        dtype=float,
    )

    client_sizes = {
        station:
            len(
                TASK_CLIENT_DATASETS[
                    task
                ][station][
                    "Train"
                ][
                    "y"
                ]
            )
        for station
        in EXPECTED_STATIONS
    }

    scenario_position = (
        list(
            FED17C_SCENARIOS
            .keys()
        )
        .index(
            scenario
        )
        + 1
    )

    communication_rng = (
        np.random.default_rng(
            np.random.SeedSequence(
                [
                    int(
                        repetition_seed
                    ),
                    int(
                        scenario_position
                    ),
                    int(
                        task_position
                    ),
                ]
            )
        )
    )

    burst_state = {
        station: False
        for station
        in EXPECTED_STATIONS
    }

    attempted = 0
    delivered = 0
    empty_rounds = 0

    reasons = {
        "PacketLoss": 0,
        "Burst": 0,
        "Unavailable": 0,
        "DeadlineMiss": 0,
        "Delivered": 0,
    }

    for round_index in range(
        FED17C_ROUNDS
    ):

        received_updates = []

        for (
            station_position,
            station,
        ) in enumerate(
            EXPECTED_STATIONS,
            start=1,
        ):

            attempted += 1

            ds = (
                TASK_CLIENT_DATASETS[
                    task
                ][station][
                    "Train"
                ]
            )

            # EXACT Cell-13 deterministic optimizer seed
            optimizer_seed = (
                GLOBAL_RANDOM_SEED
                + task_position
                * 1_000_000
                + algorithm_position
                * 10_000
                + FED17C_METHOD_POSITION
                * 1_000
                + round_index
                * 10
                + station_position
            )

            local_parameters = (
                fed17c_client_update_exact(
                    X=
                        ds[
                            "X"
                        ],

                    y_raw=
                        ds[
                            "y"
                        ],

                    task=
                        task,

                    algorithm=
                        algorithm,

                    global_parameters=
                        global_parameters,

                    local_epochs=
                        FED17C_LOCAL_EPOCHS,

                    round_index=
                        round_index,

                    seed=
                        optimizer_seed,
                )
            )

            is_delivered, reason = (
                fed17c_delivery(
                    rng=
                        communication_rng,

                    scenario_name=
                        scenario,

                    station=
                        station,

                    burst_state=
                        burst_state,
                )
            )

            reasons[
                reason
            ] += 1

            if is_delivered:

                delivered += 1

                received_updates.append(
                    (
                        station,
                        local_parameters,
                    )
                )

        if len(
            received_updates
        ) == 0:

            empty_rounds += 1
            continue

        total_samples = sum(
            client_sizes[
                station
            ]
            for (
                station,
                _
            )
            in received_updates
        )

        averaged_parameters = sum(
            (
                client_sizes[
                    station
                ]
                / total_samples
            )
            * parameters
            for (
                station,
                parameters
            )
            in received_updates
        )

        displacement = (
            averaged_parameters
            - global_parameters
        )

        m = (
            FEDADAM_BETA1
            * m
            + (
                1.0
                - FEDADAM_BETA1
            )
            * displacement
        )

        v = (
            FEDADAM_BETA2
            * v
            + (
                1.0
                - FEDADAM_BETA2
            )
            * (
                displacement
                ** 2
            )
        )

        # Exact accepted Cell-13 server rule:
        # no bias correction
        global_parameters = (
            global_parameters
            + FEDADAM_SERVER_LR
            * m
            / (
                np.sqrt(
                    v
                )
                + FEDADAM_TAU
            )
        )

    effective_loss = (
        100.0
        * (
            attempted
            - delivered
        )
        / attempted
    )

    return {
        "Algorithm":
            algorithm,

        "Parameters":
            global_parameters,

        "Attempted":
            attempted,

        "Delivered":
            delivered,

        "Effective_Update_Loss_Percent":
            effective_loss,

        "Empty_Rounds":
            empty_rounds,

        "Reasons":
            reasons,
    }


# ==================================================================================================
# 9. FINAL CELL-16 EVALUATION USING NEW FEDADAM PARAMETERS
# ==================================================================================================

def fed17c_evaluate_final_model(
    task,
    station,
    global_parameters,
):

    meta = TASK_DATASETS[
        task
    ][
        "Meta"
    ]

    val_ds = (
        TASK_CLIENT_DATASETS[
            task
        ][station][
            "Validation"
        ]
    )

    test_ds = (
        TASK_CLIENT_DATASETS[
            task
        ][station][
            "Test"
        ]
    )

    val_frame = (
        val_ds[
            "Frame"
        ]
        .reset_index(
            drop=True
        )
    )

    test_frame = (
        test_ds[
            "Frame"
        ]
        .reset_index(
            drop=True
        )
    )

    # Corrected Cell-16 dictionary interface
    val_experts = (
        u1619_static_predictions(
            task,
            station,
            "Validation",
        )
    )

    test_experts = (
        u1619_static_predictions(
            task,
            station,
            "Test",
        )
    )

    val_self = np.asarray(
        val_experts[
            "Self"
        ],
        dtype=float,
    )

    val_relation = np.asarray(
        val_experts[
            "Relation"
        ],
        dtype=float,
    )

    test_self = np.asarray(
        test_experts[
            "Self"
        ],
        dtype=float,
    )

    test_relation = np.asarray(
        test_experts[
            "Relation"
        ],
        dtype=float,
    )

    # Global expert must use the communication-stressed FedAdam model
    val_global = np.asarray(
        predict_raw(
            val_ds[
                "X"
            ],
            global_parameters,
            task,
        ),
        dtype=float,
    )

    test_global = np.asarray(
        predict_raw(
            test_ds[
                "X"
            ],
            global_parameters,
            task,
        ),
        dtype=float,
    )

    combined_dates = (
        pd.concat(
            [
                val_frame[
                    "Date"
                ],
                test_frame[
                    "Date"
                ],
            ],
            ignore_index=True,
        )
    )

    combined_target_dates = (
        pd.concat(
            [
                val_frame[
                    meta[
                        "Future_Date_Column"
                    ]
                ],
                test_frame[
                    meta[
                        "Future_Date_Column"
                    ]
                ],
            ],
            ignore_index=True,
        )
    )

    combined_y = np.concatenate(
        [
            np.asarray(
                val_ds[
                    "y"
                ],
                dtype=float,
            ),
            np.asarray(
                test_ds[
                    "y"
                ],
                dtype=float,
            ),
        ]
    )

    combined_global = np.concatenate(
        [
            val_global,
            test_global,
        ]
    )

    combined_self = np.concatenate(
        [
            val_self,
            test_self,
        ]
    )

    combined_relation = np.concatenate(
        [
            val_relation,
            test_relation,
        ]
    )

    config = (
        fed17c_rg_config(
            task,
            station,
        )
    )

    combined_prediction, _ = (
        u1619_run_online(
            dates=
                combined_dates,

            target_dates=
                combined_target_dates,

            y=
                combined_y,

            global_pred=
                combined_global,

            self_pred=
                combined_self,

            relation_pred=
                combined_relation,

            **config,
        )
    )

    n_val = len(
        val_ds[
            "y"
        ]
    )

    return (
        np.asarray(
            test_ds[
                "y"
            ],
            dtype=float,
        ),
        np.asarray(
            combined_prediction[
                n_val:
            ],
            dtype=float,
        ),
    )


# ==================================================================================================
# 10. MANDATORY CONTROL A — EXACT CELL-13 FEDADAM REPRODUCTION
# ==================================================================================================

cell13_rows = []
clean_parameters = {}

for task in FED17C_TASKS:

    algorithm = (
        u1619_selected_algorithm(
            task
        )
    )

    result = (
        fed17c_train_fedadam(
            task=
                task,

            scenario=
                "Clean",

            repetition_seed=
                4701,
        )
    )

    reproduced_parameters = (
        np.asarray(
            result[
                "Parameters"
            ],
            dtype=float,
        )
    )

    reference_parameters = (
        np.asarray(
            FEDERATED_RUNS[
                (
                    task,
                    algorithm,
                    "FedAdam",
                )
            ][
                "Parameters"
            ],
            dtype=float,
        )
    )

    difference = float(
        np.max(
            np.abs(
                reproduced_parameters
                - reference_parameters
            )
        )
    )

    passed = (
        difference
        < 1e-12
    )

    clean_parameters[
        task
    ] = (
        reproduced_parameters
    )

    cell13_rows.append({
        "Task":
            task,

        "Algorithm":
            algorithm,

        "Max_Parameter_Difference":
            difference,

        "Cell13_Reproduced":
            passed,
    })


FED17C_CELL13_REPRODUCTION_DF = (
    pd.DataFrame(
        cell13_rows
    )
)

display(
    FED17C_CELL13_REPRODUCTION_DF
    .round(12)
)

if not (
    FED17C_CELL13_REPRODUCTION_DF[
        "Cell13_Reproduced"
    ].all()
):

    raise RuntimeError(
        "MANDATORY CONTROL A FAILED: "
        "Clean FedAdam does not reproduce Cell 13 exactly."
    )

print(
    "✓ CURRENT Cell 13 FedAdam reproduction PASSED."
)


# ==================================================================================================
# 11. MANDATORY CONTROL B — EXACT FINAL CELL-16 REPRODUCTION
# ==================================================================================================

cell16_rows = []

for task in FED17C_TASKS:

    actual_parts = []
    prediction_parts = []

    for station in EXPECTED_STATIONS:

        actual, prediction = (
            fed17c_evaluate_final_model(
                task,
                station,
                clean_parameters[
                    task
                ],
            )
        )

        actual_parts.append(
            actual
        )

        prediction_parts.append(
            prediction
        )

    actual_all = np.concatenate(
        actual_parts
    )

    pred_all = np.concatenate(
        prediction_parts
    )

    metrics = reg_metrics(
        actual_all,
        pred_all,
    )

    reference = (
        U1619_NOVEL_TEST_RESULTS_DF[
            U1619_NOVEL_TEST_RESULTS_DF[
                "Task"
            ].eq(task)
        ]
        .iloc[0]
    )

    difference = max(
        abs(
            metrics["MAE"]
            - reference["MAE"]
        ),
        abs(
            metrics["RMSE"]
            - reference["RMSE"]
        ),
        abs(
            metrics["R2"]
            - reference["R2"]
        ),
    )

    cell16_rows.append({
        "Task":
            task,

        "MAE":
            metrics[
                "MAE"
            ],

        "Reference_MAE":
            reference[
                "MAE"
            ],

        "RMSE":
            metrics[
                "RMSE"
            ],

        "Reference_RMSE":
            reference[
                "RMSE"
            ],

        "R2":
            metrics[
                "R2"
            ],

        "Reference_R2":
            reference[
                "R2"
            ],

        "Max_Metric_Difference":
            difference,

        "Cell16_Reproduced":
            difference
            < 1e-10,
    })


FED17C_CELL16_REPRODUCTION_DF = (
    pd.DataFrame(
        cell16_rows
    )
)

display(
    FED17C_CELL16_REPRODUCTION_DF
    .round(12)
)

if not (
    FED17C_CELL16_REPRODUCTION_DF[
        "Cell16_Reproduced"
    ].all()
):

    raise RuntimeError(
        "MANDATORY CONTROL B FAILED: "
        "Clean condition does not reproduce Cell 16 exactly."
    )

print(
    "✓ FINAL Cell 16 reproduction PASSED."
)


# ==================================================================================================
# 12. FULL ROBUSTNESS EXPERIMENT
# ==================================================================================================

task_rep_rows = []
communication_rows = []

for scenario in (
    FED17C_SCENARIOS.keys()
):

    print(
        "\n"
        + "=" * 110
    )

    print(
        f"Scenario: {scenario}"
    )

    print(
        "=" * 110
    )

    for rep_number, rep_seed in enumerate(
        FED17C_REPETITION_SEEDS,
        start=1,
    ):

        print(
            f"  Repetition "
            f"{rep_number:02d}/10"
        )

        task_parameters = {}

        for task in FED17C_TASKS:

            trained = (
                fed17c_train_fedadam(
                    task=
                        task,

                    scenario=
                        scenario,

                    repetition_seed=
                        rep_seed,
                )
            )

            task_parameters[
                task
            ] = trained[
                "Parameters"
            ]

            communication_rows.append({
                "Scenario":
                    scenario,

                "Repetition":
                    rep_number,

                "Seed":
                    rep_seed,

                "Task":
                    task,

                "Attempted_Updates":
                    trained[
                        "Attempted"
                    ],

                "Delivered_Updates":
                    trained[
                        "Delivered"
                    ],

                "Effective_Update_Loss_Percent":
                    trained[
                        "Effective_Update_Loss_Percent"
                    ],

                "Rounds_With_No_Update":
                    trained[
                        "Empty_Rounds"
                    ],

                "PacketLoss_Count":
                    trained[
                        "Reasons"
                    ][
                        "PacketLoss"
                    ],

                "Burst_Count":
                    trained[
                        "Reasons"
                    ][
                        "Burst"
                    ],

                "Unavailable_Count":
                    trained[
                        "Reasons"
                    ][
                        "Unavailable"
                    ],

                "DeadlineMiss_Count":
                    trained[
                        "Reasons"
                    ][
                        "DeadlineMiss"
                    ],
            })

        for task in FED17C_TASKS:

            actual_parts = []
            pred_parts = []

            for station in (
                EXPECTED_STATIONS
            ):

                actual, prediction = (
                    fed17c_evaluate_final_model(
                        task=
                            task,

                        station=
                            station,

                        global_parameters=
                            task_parameters[
                                task
                            ],
                    )
                )

                actual_parts.append(
                    actual
                )

                pred_parts.append(
                    prediction
                )

            actual_all = np.concatenate(
                actual_parts
            )

            pred_all = np.concatenate(
                pred_parts
            )

            metrics = reg_metrics(
                actual_all,
                pred_all,
            )

            task_rep_rows.append({
                "Scenario":
                    scenario,

                "Repetition":
                    rep_number,

                "Seed":
                    rep_seed,

                "Task":
                    task,

                **metrics,
            })


FED17C_TASK_REPETITION_DF = (
    pd.DataFrame(
        task_rep_rows
    )
)

FED17C_COMMUNICATION_DF = (
    pd.DataFrame(
        communication_rows
    )
)


# ==================================================================================================
# 13. TASK SUMMARY
# ==================================================================================================

FED17C_TASK_RESULTS_DF = (
    FED17C_TASK_REPETITION_DF
    .groupby(
        [
            "Scenario",
            "Task",
        ],
        as_index=False,
    )
    .agg(
        Repetitions=(
            "Repetition",
            "nunique",
        ),

        MAE=(
            "MAE",
            "mean",
        ),

        MAE_SD=(
            "MAE",
            "std",
        ),

        RMSE=(
            "RMSE",
            "mean",
        ),

        RMSE_SD=(
            "RMSE",
            "std",
        ),

        R2=(
            "R2",
            "mean",
        ),

        R2_SD=(
            "R2",
            "std",
        ),
    )
)


clean_task = (
    FED17C_TASK_RESULTS_DF[
        FED17C_TASK_RESULTS_DF[
            "Scenario"
        ].eq("Clean")
    ][
        [
            "Task",
            "MAE",
            "RMSE",
            "R2",
        ]
    ]
    .rename(
        columns={
            "MAE":
                "Clean_MAE",

            "RMSE":
                "Clean_RMSE",

            "R2":
                "Clean_R2",
        }
    )
)


FED17C_TASK_RESULTS_DF = (
    FED17C_TASK_RESULTS_DF
    .merge(
        clean_task,
        on="Task",
        how="left",
    )
)


FED17C_TASK_RESULTS_DF[
    "MAE_Degradation_Percent"
] = (
    100.0
    * (
        FED17C_TASK_RESULTS_DF[
            "MAE"
        ]
        - FED17C_TASK_RESULTS_DF[
            "Clean_MAE"
        ]
    )
    / FED17C_TASK_RESULTS_DF[
        "Clean_MAE"
    ]
)


FED17C_TASK_RESULTS_DF[
    "RMSE_Degradation_Percent"
] = (
    100.0
    * (
        FED17C_TASK_RESULTS_DF[
            "RMSE"
        ]
        - FED17C_TASK_RESULTS_DF[
            "Clean_RMSE"
        ]
    )
    / FED17C_TASK_RESULTS_DF[
        "Clean_RMSE"
    ]
)


FED17C_TASK_RESULTS_DF[
    "R2_Loss"
] = (
    FED17C_TASK_RESULTS_DF[
        "Clean_R2"
    ]
    - FED17C_TASK_RESULTS_DF[
        "R2"
    ]
)


# ==================================================================================================
# 14. OVERALL REPETITION SUMMARY
# ==================================================================================================

overall_rep_rows = []

for (
    scenario,
    repetition,
), group in (
    FED17C_TASK_REPETITION_DF
    .groupby(
        [
            "Scenario",
            "Repetition",
        ]
    )
):

    overall_rep_rows.append({
        "Scenario":
            scenario,

        "Repetition":
            repetition,

        "Mean_MAE":
            float(
                group[
                    "MAE"
                ].mean()
            ),

        "Mean_RMSE":
            float(
                group[
                    "RMSE"
                ].mean()
            ),

        "Mean_R2":
            float(
                group[
                    "R2"
                ].mean()
            ),
    })


FED17C_OVERALL_REPETITION_DF = (
    pd.DataFrame(
        overall_rep_rows
    )
)


# ==================================================================================================
# 15. OVERALL SCENARIO SUMMARY
# ==================================================================================================

FED17C_OVERALL_DF = (
    FED17C_OVERALL_REPETITION_DF
    .groupby(
        "Scenario",
        as_index=False,
    )
    .agg(
        Repetitions=(
            "Repetition",
            "nunique",
        ),

        Mean_MAE=(
            "Mean_MAE",
            "mean",
        ),

        SD_MAE=(
            "Mean_MAE",
            "std",
        ),

        Mean_RMSE=(
            "Mean_RMSE",
            "mean",
        ),

        SD_RMSE=(
            "Mean_RMSE",
            "std",
        ),

        Mean_R2=(
            "Mean_R2",
            "mean",
        ),

        SD_R2=(
            "Mean_R2",
            "std",
        ),
    )
)


FED17C_COMMUNICATION_SUMMARY_DF = (
    FED17C_COMMUNICATION_DF
    .groupby(
        "Scenario",
        as_index=False,
    )
    .agg(
        Effective_Update_Loss_Percent=(
            "Effective_Update_Loss_Percent",
            "mean",
        ),

        Mean_Rounds_With_No_Update=(
            "Rounds_With_No_Update",
            "mean",
        ),
    )
)


FED17C_OVERALL_DF = (
    FED17C_OVERALL_DF
    .merge(
        FED17C_COMMUNICATION_SUMMARY_DF,
        on="Scenario",
        how="left",
    )
)


clean_overall = (
    FED17C_OVERALL_DF[
        FED17C_OVERALL_DF[
            "Scenario"
        ].eq("Clean")
    ]
    .iloc[0]
)


FED17C_OVERALL_DF[
    "MAE_Degradation_Percent"
] = (
    100.0
    * (
        FED17C_OVERALL_DF[
            "Mean_MAE"
        ]
        - clean_overall[
            "Mean_MAE"
        ]
    )
    / clean_overall[
        "Mean_MAE"
    ]
)


FED17C_OVERALL_DF[
    "RMSE_Degradation_Percent"
] = (
    100.0
    * (
        FED17C_OVERALL_DF[
            "Mean_RMSE"
        ]
        - clean_overall[
            "Mean_RMSE"
        ]
    )
    / clean_overall[
        "Mean_RMSE"
    ]
)


FED17C_OVERALL_DF[
    "R2_Loss"
] = (
    clean_overall[
        "Mean_R2"
    ]
    - FED17C_OVERALL_DF[
        "Mean_R2"
    ]
)


# ==================================================================================================
# 16. WORST TASK / WORST REPETITION
# ==================================================================================================

worst_task = (
    FED17C_TASK_RESULTS_DF
    .groupby(
        "Scenario",
        as_index=False,
    )
    .agg(
        Worst_Task_MAE_Degradation_Percent=(
            "MAE_Degradation_Percent",
            "max",
        )
    )
)


FED17C_OVERALL_DF = (
    FED17C_OVERALL_DF
    .merge(
        worst_task,
        on="Scenario",
        how="left",
    )
)


clean_rep_mae = float(
    FED17C_OVERALL_REPETITION_DF[
        FED17C_OVERALL_REPETITION_DF[
            "Scenario"
        ].eq(
            "Clean"
        )
    ][
        "Mean_MAE"
    ].mean()
)


FED17C_OVERALL_REPETITION_DF[
    "MAE_Degradation_Percent"
] = (
    100.0
    * (
        FED17C_OVERALL_REPETITION_DF[
            "Mean_MAE"
        ]
        - clean_rep_mae
    )
    / clean_rep_mae
)


worst_rep = (
    FED17C_OVERALL_REPETITION_DF
    .groupby(
        "Scenario",
        as_index=False,
    )
    .agg(
        Worst_Repetition_MAE_Degradation_Percent=(
            "MAE_Degradation_Percent",
            "max",
        )
    )
)


FED17C_OVERALL_DF = (
    FED17C_OVERALL_DF
    .merge(
        worst_rep,
        on="Scenario",
        how="left",
    )
)


# ==================================================================================================
# 17. DESCRIPTIVE ROBUSTNESS LABEL
# ==================================================================================================

def fed17c_label(x):

    x = abs(
        float(x)
    )

    if x < 1.0:
        return "Very robust"

    if x < 2.5:
        return "Robust"

    if x < 5.0:
        return "Moderately robust"

    return "Sensitive"


FED17C_OVERALL_DF[
    "Robustness"
] = (
    FED17C_OVERALL_DF[
        "MAE_Degradation_Percent"
    ]
    .apply(
        fed17c_label
    )
)


# ==================================================================================================
# 18. FINAL CONTROL SUMMARY
# ==================================================================================================

FED17C_MANDATORY_CONTROL_DF = (
    pd.DataFrame([
        {
            "Control":
                "Cell 13 FedAdam parameters",

            "Passed":
                bool(
                    FED17C_CELL13_REPRODUCTION_DF[
                        "Cell13_Reproduced"
                    ].all()
                ),
        },

        {
            "Control":
                "Final unified Cell 16 results",

            "Passed":
                bool(
                    FED17C_CELL16_REPRODUCTION_DF[
                        "Cell16_Reproduced"
                    ].all()
                ),
        },
    ])
)


# ==================================================================================================
# 19. PROTOCOL TABLE
# ==================================================================================================

FED17C_PROTOCOL_DF = pd.DataFrame([
    {
        "Item":
            "Test",

        "Value":
            "Federated training communication robustness",
    },

    {
        "Item":
            "Federated backbone",

        "Value":
            "FedAdam",
    },

    {
        "Item":
            "Rounds",

        "Value":
            FED17C_ROUNDS,
    },

    {
        "Item":
            "Local epochs",

        "Value":
            FED17C_LOCAL_EPOCHS,
    },

    {
        "Item":
            "Clients",

        "Value":
            len(
                EXPECTED_STATIONS
            ),
    },

    {
        "Item":
            "Communication repetitions",

        "Value":
            10,
    },

    {
        "Item":
            "Communication seeds",

        "Value":
            "4701–4710",
    },

    {
        "Item":
            "Optimizer seeds",

        "Value":
            "Exact Cell-13 deterministic schedule",
    },

    {
        "Item":
            "Task×Algorithm config",

        "Value":
            "Frozen Cell-10 configuration",
    },

    {
        "Item":
            "Hyperparameter tuning",

        "Value":
            False,
    },

    {
        "Item":
            "Model re-selection",

        "Value":
            False,
    },

    {
        "Item":
            "Test-based selection",

        "Value":
            False,
    },

    {
        "Item":
            "Final online evaluator",

        "Value":
            "Corrected u1619_run_online",
    },

    {
        "Item":
            "Relation topology",

        "Value":
            "Peer-only",
    },

    {
        "Item":
            "Delayed feedback",

        "Value":
            True,
    },
])


# ==================================================================================================
# 20. OUTPUT
# ==================================================================================================

print("\nCell 13 parameter reproduction:")
display(
    FED17C_CELL13_REPRODUCTION_DF
    .round(12)
)

print("\nCell 16 final-result reproduction:")
display(
    FED17C_CELL16_REPRODUCTION_DF
    .round(12)
)

print("\nMandatory controls:")
display(
    FED17C_MANDATORY_CONTROL_DF
)

print("\nTask-level robustness:")
display(
    FED17C_TASK_RESULTS_DF[
        [
            "Scenario",
            "Task",
            "Repetitions",
            "MAE",
            "MAE_SD",
            "RMSE",
            "RMSE_SD",
            "R2",
            "R2_SD",
            "MAE_Degradation_Percent",
            "RMSE_Degradation_Percent",
            "R2_Loss",
        ]
    ]
    .sort_values(
        [
            "Scenario",
            "Task",
        ]
    )
    .round(6)
)

print("\nOverall robustness:")
display(
    FED17C_OVERALL_DF
    .round(6)
)

print("\nCommunication summary:")
display(
    FED17C_COMMUNICATION_SUMMARY_DF
    .round(6)
)

print("\nProtocol:")
display(
    FED17C_PROTOCOL_DF
)

print(
    "\nNegative degradation values indicate stochastic variation, "
    "not communication-induced improvement."
)

print("\n" + "=" * 140)
print("CELL 17C COMPLETED SUCCESSFULLY")
print("=" * 140)

print(
    f"Execution time: "
    f"{time.time() - CELL17C_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 18 — PERSISTENCE AUDIT & CHANGE-EVENT SKILL ANALYSIS
# FINAL CORRECTED VERSION
# ==================================================================================================
# Scientific protocol:
#   Persistence prediction = current observed target y(t)
#   Actual                  = future target y(t+h)
#
#   Current target:
#       Alum_Dose_Target
#       Chlorine_Dose_Target
#
#   Future target:
#       already stored in TASK_CLIENT_DATASETS[task][station][split]["y"]
#
#   Meaningful-change threshold:
#       Train-only Q75 of |y(t+h) - y(t)|
#       if Q75 <= 0 -> median of positive Train changes
#
# No retraining.
# No tuning.
# No model re-selection.
# Test never defines the threshold.
# ==================================================================================================

import time
import numpy as np
import pandas as pd

CELL18_START = time.time()

print("=" * 130)
print("CELL 18 — PERSISTENCE AUDIT & CHANGE-EVENT SKILL ANALYSIS")
print("=" * 130)


# ==================================================================================================
# 1. REQUIRED OBJECTS
# ==================================================================================================

required_objects = [
    "TASK_DEFINITIONS",
    "TASK_DATASETS",
    "TASK_CLIENT_DATASETS",
    "EXPECTED_STATIONS",
    "RGADA18_TEST_PREDICTIONS_DF",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise NameError(
        "Missing required objects:\n- "
        + "\n- ".join(missing)
    )

print("✓ Required final prediction objects are available.")


# ==================================================================================================
# 2. TASKS
# ==================================================================================================

CELL18_TASKS = [
    d["Task"]
    for d in TASK_DEFINITIONS
]


# ==================================================================================================
# 3. CORRECT CURRENT y(t) COLUMN RESOLVER
#
# CRITICAL:
#   meta["Target_Column"] may represent y(t+h).
#   We therefore NEVER use a *_t_plus_* column as persistence y(t).
# ==================================================================================================

def skill18_current_target_column(
    task_key,
    frame,
):

    meta = (
        TASK_DATASETS[
            task_key
        ]["Meta"]
    )

    # ----------------------------------------------------------------------------------------------
    # FIRST CHOICE:
    # meta["Target"] is the base/current target name in this project:
    # Alum_Dose_Target / Chlorine_Dose_Target
    # ----------------------------------------------------------------------------------------------

    target_name = str(
        meta["Target"]
    )

    if (
        target_name in frame.columns
        and "_t_plus_" not in target_name.lower()
    ):
        return target_name

    # ----------------------------------------------------------------------------------------------
    # Explicit fallbacks
    # ----------------------------------------------------------------------------------------------

    lower_task = (
        task_key.lower()
    )

    if "alum" in lower_task:

        candidates = [
            "Alum_Dose_Target",
            "Alum_Dose",
            "Alum",
        ]

    elif "chlorine" in lower_task:

        candidates = [
            "Chlorine_Dose_Target",
            "Chlorine_Dose",
            "Chlorine",
        ]

    else:

        candidates = []

    for col in candidates:

        if (
            col in frame.columns
            and "_t_plus_" not in col.lower()
        ):
            return col

    # ----------------------------------------------------------------------------------------------
    # Last conservative search:
    # base target only; future-target columns explicitly rejected.
    # ----------------------------------------------------------------------------------------------

    target_keyword = (
        "alum"
        if "alum" in lower_task
        else
        "chlorine"
    )

    possible = [
        col
        for col in frame.columns
        if target_keyword
        in str(col).lower()
        and "_t_plus_" not in str(col).lower()
        and "future" not in str(col).lower()
    ]

    if len(possible) == 1:
        return possible[0]

    raise KeyError(
        f"Could not uniquely resolve current y(t) column for {task_key}.\n"
        f"Candidates found: {possible}"
    )


# ==================================================================================================
# 4. TRAIN-ONLY MEANINGFUL-CHANGE THRESHOLDS
# ==================================================================================================

threshold_rows = []

for task_key in CELL18_TASKS:

    meta = (
        TASK_DATASETS[
            task_key
        ]["Meta"]
    )

    for station in EXPECTED_STATIONS:

        train_ds = (
            TASK_CLIENT_DATASETS[
                task_key
            ][station]["Train"]
        )

        train_frame = (
            train_ds["Frame"]
            .reset_index(drop=True)
        )

        current_col = (
            skill18_current_target_column(
                task_key,
                train_frame,
            )
        )

        current_y = pd.to_numeric(
            train_frame[current_col],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        future_y = np.asarray(
            train_ds["y"],
            dtype=float,
        )

        if len(current_y) != len(future_y):
            raise RuntimeError(
                f"Train y(t)/y(t+h) length mismatch: "
                f"{task_key} / {station}"
            )

        valid = (
            np.isfinite(current_y)
            &
            np.isfinite(future_y)
        )

        changes = np.abs(
            future_y[valid]
            - current_y[valid]
        )

        if len(changes) == 0:
            raise RuntimeError(
                f"No valid Train changes for "
                f"{task_key} / {station}"
            )

        q75 = float(
            np.quantile(
                changes,
                0.75,
            )
        )

        fallback_used = False

        if (
            not np.isfinite(q75)
            or q75 <= 0.0
        ):

            positive_changes = (
                changes[
                    changes > 0.0
                ]
            )

            if len(positive_changes) == 0:
                raise RuntimeError(
                    f"No positive Train target changes for "
                    f"{task_key} / {station}"
                )

            threshold = float(
                np.median(
                    positive_changes
                )
            )

            fallback_used = True

        else:

            threshold = q75

        threshold_rows.append({
            "Task":
                task_key,

            "Target":
                meta["Target"],

            "Horizon":
                meta["Horizon"],

            "Station":
                station,

            "Current_Target_Column":
                current_col,

            "Train_Change_Threshold":
                threshold,

            "Threshold_Rule":
                (
                    "Train Q75 |y(t+h)-y(t)|"
                    if not fallback_used
                    else
                    "Train positive-change median fallback"
                ),

            "Train_N":
                int(
                    len(changes)
                ),

            "Train_Exact_No_Change_Percent":
                100.0
                * float(
                    np.mean(
                        np.isclose(
                            changes,
                            0.0,
                            atol=1e-12,
                            rtol=0.0,
                        )
                    )
                ),
        })


SKILL18_THRESHOLDS_DF = (
    pd.DataFrame(
        threshold_rows
    )
)


# ==================================================================================================
# 5. BUILD CORRECT TEST PAIRED DATA
# ==================================================================================================

paired_rows = []

for task_key in CELL18_TASKS:

    meta = (
        TASK_DATASETS[
            task_key
        ]["Meta"]
    )

    for station in EXPECTED_STATIONS:

        test_ds = (
            TASK_CLIENT_DATASETS[
                task_key
            ][station]["Test"]
        )

        test_frame = (
            test_ds["Frame"]
            .reset_index(drop=True)
        )

        current_col = (
            skill18_current_target_column(
                task_key,
                test_frame,
            )
        )

        current_y = pd.to_numeric(
            test_frame[current_col],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        future_y = np.asarray(
            test_ds["y"],
            dtype=float,
        )

        prediction_df = (
            RGADA18_TEST_PREDICTIONS_DF[
                RGADA18_TEST_PREDICTIONS_DF[
                    "Task"
                ].eq(task_key)
                &
                RGADA18_TEST_PREDICTIONS_DF[
                    "Station"
                ].eq(station)
            ]
            .sort_values(
                [
                    "Date",
                    "Target_Date",
                ]
            )
            .reset_index(drop=True)
        )

        test_order = (
            pd.DataFrame({
                "_index":
                    np.arange(
                        len(test_frame)
                    ),

                "Date":
                    pd.to_datetime(
                        test_frame["Date"]
                    ),

                "Target_Date":
                    pd.to_datetime(
                        test_frame[
                            meta[
                                "Future_Date_Column"
                            ]
                        ]
                    ),
            })
            .sort_values(
                [
                    "Date",
                    "Target_Date",
                ]
            )
            .reset_index(drop=True)
        )

        if (
            len(prediction_df)
            != len(test_order)
        ):
            raise RuntimeError(
                f"Prediction/Test size mismatch: "
                f"{task_key} / {station}"
            )

        # Verify chronology alignment before using predictions
        prediction_dates = pd.to_datetime(
            prediction_df["Date"]
        ).reset_index(drop=True)

        prediction_target_dates = pd.to_datetime(
            prediction_df["Target_Date"]
        ).reset_index(drop=True)

        if not (
            prediction_dates.equals(
                test_order["Date"]
            )
            and
            prediction_target_dates.equals(
                test_order["Target_Date"]
            )
        ):
            raise RuntimeError(
                f"Prediction/Test chronology mismatch: "
                f"{task_key} / {station}"
            )

        idx = (
            test_order["_index"]
            .to_numpy(
                dtype=int
            )
        )

        current_y = (
            current_y[idx]
        )

        future_y = (
            future_y[idx]
        )

        novel_pred = (
            prediction_df[
                "RGAdaFedResidual_Pred"
            ]
            .to_numpy(
                dtype=float
            )
        )

        # Strong consistency control:
        # Cell-16 stored Actual must equal Test future y.
        if not np.allclose(
            prediction_df[
                "Actual"
            ].to_numpy(
                dtype=float
            ),
            future_y,
            atol=1e-12,
            rtol=0.0,
            equal_nan=False,
        ):

            raise RuntimeError(
                f"Final prediction Actual does not match "
                f"Test y(t+h): {task_key} / {station}"
            )

        threshold = float(
            SKILL18_THRESHOLDS_DF[
                SKILL18_THRESHOLDS_DF[
                    "Task"
                ].eq(task_key)
                &
                SKILL18_THRESHOLDS_DF[
                    "Station"
                ].eq(station)
            ][
                "Train_Change_Threshold"
            ]
            .iloc[0]
        )

        dates = (
            test_order["Date"]
        )

        target_dates = (
            test_order[
                "Target_Date"
            ]
        )

        for i in range(
            len(future_y)
        ):

            y_t = float(
                current_y[i]
            )

            y_future = float(
                future_y[i]
            )

            change = abs(
                y_future
                - y_t
            )

            # Strict > avoids classifying exact threshold ties as changes.
            meaningful_change = (
                change
                > threshold
            )

            exact_no_change = (
                np.isclose(
                    change,
                    0.0,
                    atol=1e-12,
                    rtol=0.0,
                )
            )

            paired_rows.append({
                "Task":
                    task_key,

                "Target":
                    meta["Target"],

                "Horizon":
                    meta["Horizon"],

                "Station":
                    station,

                "Date":
                    dates.iloc[i],

                "Target_Date":
                    target_dates.iloc[i],

                "Current_Target_Column":
                    current_col,

                "Current_Observed_Target":
                    y_t,

                "Actual_Future_Target":
                    y_future,

                "Persistence_Prediction":
                    y_t,

                "Novel_Prediction":
                    float(
                        novel_pred[i]
                    ),

                "Absolute_Target_Change":
                    change,

                "Train_Change_Threshold":
                    threshold,

                "Meaningful_Change":
                    bool(
                        meaningful_change
                    ),

                "Exact_No_Change":
                    bool(
                        exact_no_change
                    ),

                "Target_Date_After_Date":
                    bool(
                        target_dates.iloc[i]
                        > dates.iloc[i]
                    ),
            })


SKILL18_PAIRED_DF = (
    pd.DataFrame(
        paired_rows
    )
)


# ==================================================================================================
# 6. PERSISTENCE AUDIT
# ==================================================================================================

audit_rows = []

for (
    task,
    station,
), group in (
    SKILL18_PAIRED_DF
    .groupby(
        [
            "Task",
            "Station",
        ]
    )
):

    persistence_correct = bool(
        np.allclose(
            group[
                "Persistence_Prediction"
            ],
            group[
                "Current_Observed_Target"
            ],
            atol=0.0,
            rtol=0.0,
        )
    )

    future_date_correct = bool(
        group[
            "Target_Date_After_Date"
        ].all()
    )

    has_nonzero_changes = bool(
        (
            group[
                "Absolute_Target_Change"
            ]
            > 0
        ).any()
    )

    audit_rows.append({
        "Task":
            task,

        "Station":
            station,

        "N":
            int(
                len(group)
            ),

        "Persistence_Equals_Current_y_t":
            persistence_correct,

        "All_Target_Date_After_Date":
            future_date_correct,

        "Contains_Real_Target_Changes":
            has_nonzero_changes,

        "Audit_Passed":
            bool(
                persistence_correct
                and
                future_date_correct
                and
                has_nonzero_changes
            ),
    })


SKILL18_PERSISTENCE_AUDIT_DF = (
    pd.DataFrame(
        audit_rows
    )
)


if not (
    SKILL18_PERSISTENCE_AUDIT_DF[
        "Audit_Passed"
    ].all()
):

    display(
        SKILL18_PERSISTENCE_AUDIT_DF
    )

    raise RuntimeError(
        "Persistence audit failed. "
        "Check y(t) versus y(t+h) mapping."
    )

print(
    "✓ Persistence audit PASSED for all Task × Station groups."
)


# ==================================================================================================
# 7. PERSISTENCE STRENGTH / CHANGE PREVALENCE
# ==================================================================================================

strength_rows = []

for task_key in CELL18_TASKS:

    group = (
        SKILL18_PAIRED_DF[
            SKILL18_PAIRED_DF[
                "Task"
            ].eq(task_key)
        ]
    )

    strength_rows.append({
        "Task":
            task_key,

        "Test_N":
            int(
                len(group)
            ),

        "Exact_No_Change_Percent":
            100.0
            * float(
                group[
                    "Exact_No_Change"
                ].mean()
            ),

        "Meaningful_Change_Percent":
            100.0
            * float(
                group[
                    "Meaningful_Change"
                ].mean()
            ),

        "Median_Absolute_Change":
            float(
                group[
                    "Absolute_Target_Change"
                ].median()
            ),
    })


SKILL18_PERSISTENCE_STRENGTH_DF = (
    pd.DataFrame(
        strength_rows
    )
)


# ==================================================================================================
# 8. MEANINGFUL-CHANGE POPULATION
# ==================================================================================================

SKILL18B_PAIRED_DF = (
    SKILL18_PAIRED_DF[
        SKILL18_PAIRED_DF[
            "Meaningful_Change"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

SKILL18B_PAIRED_DF[
    "Change_Regime"
] = "Meaningful Change"


if len(
    SKILL18B_PAIRED_DF
) == 0:

    raise RuntimeError(
        "No meaningful-change observations detected."
    )


# ==================================================================================================
# 9. SKILL FUNCTIONS
# ==================================================================================================

def skill18_metrics(
    actual,
    novel,
    persistence,
):

    actual = np.asarray(
        actual,
        dtype=float,
    )

    novel = np.asarray(
        novel,
        dtype=float,
    )

    persistence = np.asarray(
        persistence,
        dtype=float,
    )

    novel_abs = np.abs(
        actual
        - novel
    )

    persistence_abs = np.abs(
        actual
        - persistence
    )

    novel_sq = (
        actual
        - novel
    ) ** 2

    persistence_sq = (
        actual
        - persistence
    ) ** 2

    novel_mae = float(
        np.mean(
            novel_abs
        )
    )

    persistence_mae = float(
        np.mean(
            persistence_abs
        )
    )

    novel_rmse = float(
        np.sqrt(
            np.mean(
                novel_sq
            )
        )
    )

    persistence_rmse = float(
        np.sqrt(
            np.mean(
                persistence_sq
            )
        )
    )

    mae_skill = (
        np.nan
        if persistence_mae <= 0
        else
        1.0
        - novel_mae
        / persistence_mae
    )

    rmse_skill = (
        np.nan
        if persistence_rmse <= 0
        else
        1.0
        - novel_rmse
        / persistence_rmse
    )

    return {
        "Novel_MAE":
            novel_mae,

        "Persistence_MAE":
            persistence_mae,

        "MAE_Skill":
            float(
                mae_skill
            ),

        "MAE_Improvement_Percent":
            100.0
            * float(
                mae_skill
            ),

        "Novel_RMSE":
            novel_rmse,

        "Persistence_RMSE":
            persistence_rmse,

        "RMSE_Skill":
            float(
                rmse_skill
            ),

        "RMSE_Improvement_Percent":
            100.0
            * float(
                rmse_skill
            ),
    }


# ==================================================================================================
# 10. TASK-LEVEL CHANGE-EVENT SKILL
# ==================================================================================================

task_rows = []

for task_key in CELL18_TASKS:

    group = (
        SKILL18B_PAIRED_DF[
            SKILL18B_PAIRED_DF[
                "Task"
            ].eq(task_key)
        ]
    )

    if len(group) == 0:
        raise RuntimeError(
            f"No meaningful-change observations for {task_key}"
        )

    result = skill18_metrics(
        actual=
            group[
                "Actual_Future_Target"
            ],

        novel=
            group[
                "Novel_Prediction"
            ],

        persistence=
            group[
                "Persistence_Prediction"
            ],
    )

    task_rows.append({
        "Task":
            task_key,

        "N":
            int(
                len(group)
            ),

        **result,

        "Novel_Beats_Persistence_MAE":
            bool(
                result[
                    "Novel_MAE"
                ]
                <
                result[
                    "Persistence_MAE"
                ]
            ),

        "Novel_Beats_Persistence_RMSE":
            bool(
                result[
                    "Novel_RMSE"
                ]
                <
                result[
                    "Persistence_RMSE"
                ]
            ),
    })


SKILL18_CHANGE_EVENT_TASK_DF = (
    pd.DataFrame(
        task_rows
    )
)


# ==================================================================================================
# 11. STATION × TASK CHANGE-EVENT SKILL
# ==================================================================================================

station_rows = []

for (
    task,
    station,
), group in (
    SKILL18B_PAIRED_DF
    .groupby(
        [
            "Task",
            "Station",
        ]
    )
):

    result = skill18_metrics(
        actual=
            group[
                "Actual_Future_Target"
            ],

        novel=
            group[
                "Novel_Prediction"
            ],

        persistence=
            group[
                "Persistence_Prediction"
            ],
    )

    station_rows.append({
        "Task":
            task,

        "Station":
            station,

        "N":
            int(
                len(group)
            ),

        **result,

        "Novel_Beats_Persistence_MAE":
            bool(
                result[
                    "Novel_MAE"
                ]
                <
                result[
                    "Persistence_MAE"
                ]
            ),

        "Novel_Beats_Persistence_RMSE":
            bool(
                result[
                    "Novel_RMSE"
                ]
                <
                result[
                    "Persistence_RMSE"
                ]
            ),
    })


SKILL18_CHANGE_EVENT_STATION_DF = (
    pd.DataFrame(
        station_rows
    )
)


# ==================================================================================================
# 12. OVERALL MACRO SUMMARY
# ==================================================================================================

SKILL18_CHANGE_EVENT_OVERALL_DF = (
    pd.DataFrame([
        {
            "Tasks":
                int(
                    len(
                        SKILL18_CHANGE_EVENT_TASK_DF
                    )
                ),

            "Mean_Novel_MAE":
                float(
                    SKILL18_CHANGE_EVENT_TASK_DF[
                        "Novel_MAE"
                    ].mean()
                ),

            "Mean_Persistence_MAE":
                float(
                    SKILL18_CHANGE_EVENT_TASK_DF[
                        "Persistence_MAE"
                    ].mean()
                ),

            "Mean_MAE_Skill":
                float(
                    SKILL18_CHANGE_EVENT_TASK_DF[
                        "MAE_Skill"
                    ].mean()
                ),

            "Mean_MAE_Improvement_Percent":
                100.0
                * float(
                    SKILL18_CHANGE_EVENT_TASK_DF[
                        "MAE_Skill"
                    ].mean()
                ),

            "Mean_Novel_RMSE":
                float(
                    SKILL18_CHANGE_EVENT_TASK_DF[
                        "Novel_RMSE"
                    ].mean()
                ),

            "Mean_Persistence_RMSE":
                float(
                    SKILL18_CHANGE_EVENT_TASK_DF[
                        "Persistence_RMSE"
                    ].mean()
                ),

            "Mean_RMSE_Skill":
                float(
                    SKILL18_CHANGE_EVENT_TASK_DF[
                        "RMSE_Skill"
                    ].mean()
                ),

            "Tasks_Beating_Persistence_MAE":
                int(
                    SKILL18_CHANGE_EVENT_TASK_DF[
                        "Novel_Beats_Persistence_MAE"
                    ].sum()
                ),

            "Tasks_Beating_Persistence_RMSE":
                int(
                    SKILL18_CHANGE_EVENT_TASK_DF[
                        "Novel_Beats_Persistence_RMSE"
                    ].sum()
                ),

            "StationTask_MAE_Wins":
                int(
                    SKILL18_CHANGE_EVENT_STATION_DF[
                        "Novel_Beats_Persistence_MAE"
                    ].sum()
                ),

            "StationTask_RMSE_Wins":
                int(
                    SKILL18_CHANGE_EVENT_STATION_DF[
                        "Novel_Beats_Persistence_RMSE"
                    ].sum()
                ),

            "StationTask_Total":
                int(
                    len(
                        SKILL18_CHANGE_EVENT_STATION_DF
                    )
                ),
        }
    ])
)


# ==================================================================================================
# 13. WORST PERSISTENCE-ERROR TAIL DIAGNOSTIC
# ==================================================================================================

tail_rows = []

for task_key in CELL18_TASKS:

    group = (
        SKILL18_PAIRED_DF[
            SKILL18_PAIRED_DF[
                "Task"
            ].eq(task_key)
        ]
        .copy()
    )

    persistence_error = np.abs(
        group[
            "Actual_Future_Target"
        ].to_numpy(
            dtype=float
        )
        -
        group[
            "Persistence_Prediction"
        ].to_numpy(
            dtype=float
        )
    )

    tail_threshold = float(
        np.quantile(
            persistence_error,
            0.75,
        )
    )

    # Strict greater where possible.
    mask = (
        persistence_error
        > tail_threshold
    )

    if mask.sum() == 0:
        mask = (
            persistence_error
            >= tail_threshold
        )

    tail = (
        group.loc[
            mask
        ]
        .copy()
    )

    result = skill18_metrics(
        actual=
            tail[
                "Actual_Future_Target"
            ],

        novel=
            tail[
                "Novel_Prediction"
            ],

        persistence=
            tail[
                "Persistence_Prediction"
            ],
    )

    tail_rows.append({
        "Task":
            task_key,

        "Tail_N":
            int(
                len(tail)
            ),

        **result,
    })


SKILL18_PERSISTENCE_ERROR_TAIL_DF = (
    pd.DataFrame(
        tail_rows
    )
)


# ==================================================================================================
# 14. SANITY CHECKS
# ==================================================================================================

if (
    SKILL18_PERSISTENCE_STRENGTH_DF[
        "Exact_No_Change_Percent"
    ].eq(100.0).all()
):

    raise RuntimeError(
        "Invalid persistence construction: all Test targets appear unchanged."
    )


if (
    SKILL18_CHANGE_EVENT_TASK_DF[
        "Persistence_MAE"
    ].le(0.0).any()
):

    raise RuntimeError(
        "Invalid persistence baseline: Persistence MAE <= 0 "
        "inside meaningful-change population."
    )


if (
    SKILL18_CHANGE_EVENT_TASK_DF[
        "MAE_Skill"
    ].isna().any()
):

    raise RuntimeError(
        "Invalid MAE Skill: NaN detected."
    )


# ==================================================================================================
# 15. PROTOCOL TABLE
# ==================================================================================================

SKILL18_PROTOCOL_DF = pd.DataFrame([
    {
        "Item":
            "Evaluation",

        "Value":
            "Persistence audit + change-event skill",
    },

    {
        "Item":
            "Final model",

        "Value":
            "Frozen UTW-RG-AdaFedResidual",
    },

    {
        "Item":
            "Persistence prediction",

        "Value":
            "Current observed target y(t)",
    },

    {
        "Item":
            "Current target source",

        "Value":
            "Base Alum_Dose_Target / Chlorine_Dose_Target",
    },

    {
        "Item":
            "Future actual",

        "Value":
            "Observed y(t+h) from task dataset",
    },

    {
        "Item":
            "Meaningful-change threshold",

        "Value":
            "Train-only Q75 of |y(t+h)-y(t)|",
    },

    {
        "Item":
            "Fallback",

        "Value":
            "Median positive Train change if Q75 <= 0",
    },

    {
        "Item":
            "Test used to define threshold",

        "Value":
            False,
    },

    {
        "Item":
            "Retraining",

        "Value":
            False,
    },

    {
        "Item":
            "Hyperparameter tuning",

        "Value":
            False,
    },

    {
        "Item":
            "Model re-selection",

        "Value":
            False,
    },

    {
        "Item":
            "Primary Skill",

        "Value":
            "Change-event MAE Skill Score",
    },

    {
        "Item":
            "Secondary Skill",

        "Value":
            "Change-event RMSE Skill Score",
    },
])


# ==================================================================================================
# 16. OUTPUT
# ==================================================================================================

print("\nPersistence audit:")
display(
    SKILL18_PERSISTENCE_AUDIT_DF
)


print("\nTrain-only meaningful-change thresholds:")
display(
    SKILL18_THRESHOLDS_DF
    .round(6)
)


print("\nPersistence strength / change prevalence:")
display(
    SKILL18_PERSISTENCE_STRENGTH_DF
    .round(6)
)


print("\nChange-event Skill Score — per task:")
display(
    SKILL18_CHANGE_EVENT_TASK_DF
    .round(6)
)


print("\nChange-event Skill Score — Station × Task:")
display(
    SKILL18_CHANGE_EVENT_STATION_DF
    .round(6)
)


print("\nOverall meaningful-change summary:")
display(
    SKILL18_CHANGE_EVENT_OVERALL_DF
    .round(6)
)


print("\nWorst persistence-error tail diagnostic:")
display(
    SKILL18_PERSISTENCE_ERROR_TAIL_DF
    .round(6)
)


print("\nProtocol:")
display(
    SKILL18_PROTOCOL_DF
)


print("\n" + "=" * 130)
print("CELL 18 COMPLETED SUCCESSFULLY")
print("=" * 130)

print(
    f"Execution time: "
    f"{time.time() - CELL18_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 19 — STATISTICAL SIGNIFICANCE OF CHANGE-EVENT SKILL
# FINAL CORRECTED VERSION — compatible with corrected Cell 18
# ==================================================================================================
# Purpose:
#   Statistical significance analysis of the positive Change-Event Skill
#   established in Cell 18.
#
# Protocol:
#   - Frozen predictions from Cell 18
#   - Meaningful-change population already defined using TRAIN-only thresholds
#   - Paired station-stratified Moving-Block Bootstrap
#   - 5000 bootstrap repetitions
#   - Reproducibility seed = 19001
#   - 95% confidence intervals
#   - Primary endpoint: MAE Skill
#   - Secondary endpoint: RMSE Skill / squared-error improvement
#   - Holm correction across the six tasks
#   - Global macro skill also tested
#
# No retraining.
# No tuning.
# No model selection.
# No prediction modification.
# Test is used only for FINAL statistical inference.
# ==================================================================================================

import time
import hashlib
import numpy as np
import pandas as pd

CELL19_START = time.time()

print("=" * 135)
print("CELL 19 — STATISTICAL SIGNIFICANCE OF CHANGE-EVENT SKILL")
print("=" * 135)


# ==================================================================================================
# 1. REQUIRED OBJECTS
# ==================================================================================================

required_objects = [
    "SKILL18_PAIRED_DF",
    "SKILL18B_PAIRED_DF",
    "SKILL18_CHANGE_EVENT_TASK_DF",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise NameError(
        "Missing required Cell 18 objects:\n- "
        + "\n- ".join(missing)
    )

print("✓ Corrected Cell 18 objects found.")


# ==================================================================================================
# 2. FROZEN INPUT
# ==================================================================================================

CELL19_INPUT_DF = (
    SKILL18B_PAIRED_DF
    .copy(deep=True)
    .reset_index(drop=True)
)


required_columns = [
    "Task",
    "Station",
    "Date",
    "Target_Date",
    "Actual_Future_Target",
    "Novel_Prediction",
    "Persistence_Prediction",
    "Meaningful_Change",
]

missing_columns = [
    c
    for c in required_columns
    if c not in CELL19_INPUT_DF.columns
]

if missing_columns:
    raise KeyError(
        "Cell 19 input source is missing columns: "
        + ", ".join(missing_columns)
    )


# ==================================================================================================
# 3. TECHNICAL FINGERPRINT HELPER
# ==================================================================================================

def cell19_dataframe_sha256(df):

    temp = df.copy(deep=True)

    hashed_values = pd.util.hash_pandas_object(
        temp,
        index=True,
    ).values

    metadata = (
        "|".join(
            [
                str(temp.shape),
                "|".join(
                    map(
                        str,
                        temp.columns,
                    )
                ),
                "|".join(
                    map(
                        str,
                        temp.dtypes.astype(str),
                    )
                ),
            ]
        )
        .encode("utf-8")
    )

    h = hashlib.sha256()

    h.update(
        hashed_values.tobytes()
    )

    h.update(
        metadata
    )

    return h.hexdigest()


# ==================================================================================================
# 4. PRE-ANALYSIS FROZEN-INPUT AUDIT
# ==================================================================================================

CELL19_PRE_INPUT_COPY = (
    CELL19_INPUT_DF
    .copy(deep=True)
)

CELL19_PRE_INPUT_HASH = (
    cell19_dataframe_sha256(
        CELL19_INPUT_DF
    )
)

CELL19_PRE_FULL_PREDICTIONS_HASH = None

if (
    "RGADA18_TEST_PREDICTIONS_DF"
    in globals()
):

    CELL19_PRE_FULL_PREDICTIONS_COPY = (
        RGADA18_TEST_PREDICTIONS_DF
        .copy(deep=True)
    )

    CELL19_PRE_FULL_PREDICTIONS_HASH = (
        cell19_dataframe_sha256(
            RGADA18_TEST_PREDICTIONS_DF
        )
    )


CELL19_PRE_MEANINGFUL_COPY = (
    SKILL18_PAIRED_DF[
        SKILL18_PAIRED_DF[
            "Meaningful_Change"
        ]
    ]
    .copy(deep=True)
    .reset_index(drop=True)
)

CELL19_PRE_MEANINGFUL_HASH = (
    cell19_dataframe_sha256(
        CELL19_PRE_MEANINGFUL_COPY
    )
)


CELL19_PROTOCOL_DECLARATIONS_DF = pd.DataFrame([
    {
        "Declaration":
            "Model training completed before Cell 19",
        "Confirmed":
            True,
    },
    {
        "Declaration":
            "Model selection completed before Cell 19",
        "Confirmed":
            True,
    },
    {
        "Declaration":
            "Predictions frozen before Cell 19",
        "Confirmed":
            True,
    },
    {
        "Declaration":
            "Cell 18 statistical input frozen before Cell 19",
        "Confirmed":
            True,
    },
    {
        "Declaration":
            "Meaningful-change population frozen before Cell 19",
        "Confirmed":
            True,
    },
    {
        "Declaration":
            "Cell 19 performs statistical analysis only",
        "Confirmed":
            True,
    },
    {
        "Declaration":
            "No retraining allowed in Cell 19",
        "Confirmed":
            True,
    },
    {
        "Declaration":
            "No hyperparameter tuning allowed in Cell 19",
        "Confirmed":
            True,
    },
    {
        "Declaration":
            "No Test-based model selection allowed in Cell 19",
        "Confirmed":
            True,
    },
    {
        "Declaration":
            "No prediction modification allowed in Cell 19",
        "Confirmed":
            True,
    },
    {
        "Declaration":
            "No Cell 18 input modification allowed in Cell 19",
        "Confirmed":
            True,
    },
])


# ==================================================================================================
# 5. SETTINGS
# ==================================================================================================

CELL19_BOOTSTRAP_REPS = 5000
CELL19_SEED = 19001
CELL19_ALPHA = 0.05

CELL19_TASKS = (
    list(
        CELL19_INPUT_DF[
            "Task"
        ].drop_duplicates()
    )
)

if len(CELL19_TASKS) != 6:
    raise RuntimeError(
        f"Expected 6 tasks, found {len(CELL19_TASKS)}."
    )

rng_master = np.random.default_rng(
    CELL19_SEED
)


# ==================================================================================================
# 6. OBSERVED SKILL FUNCTIONS
# ==================================================================================================

def cell19_skill_metrics(
    actual,
    novel,
    persistence,
):

    actual = np.asarray(
        actual,
        dtype=float,
    )

    novel = np.asarray(
        novel,
        dtype=float,
    )

    persistence = np.asarray(
        persistence,
        dtype=float,
    )

    novel_abs = np.abs(
        actual - novel
    )

    persistence_abs = np.abs(
        actual - persistence
    )

    novel_sq = (
        actual - novel
    ) ** 2

    persistence_sq = (
        actual - persistence
    ) ** 2

    novel_mae = float(
        np.mean(
            novel_abs
        )
    )

    persistence_mae = float(
        np.mean(
            persistence_abs
        )
    )

    novel_rmse = float(
        np.sqrt(
            np.mean(
                novel_sq
            )
        )
    )

    persistence_rmse = float(
        np.sqrt(
            np.mean(
                persistence_sq
            )
        )
    )

    mae_skill = (
        1.0
        - novel_mae
        / persistence_mae
    )

    rmse_skill = (
        1.0
        - novel_rmse
        / persistence_rmse
    )

    mean_abs_improvement = float(
        np.mean(
            persistence_abs
            - novel_abs
        )
    )

    mean_sq_improvement = float(
        np.mean(
            persistence_sq
            - novel_sq
        )
    )

    return {
        "MAE_Skill":
            float(
                mae_skill
            ),

        "RMSE_Skill":
            float(
                rmse_skill
            ),

        "Mean_Absolute_Error_Improvement":
            mean_abs_improvement,

        "Mean_Squared_Error_Improvement":
            mean_sq_improvement,
    }


# ==================================================================================================
# 7. MOVING-BLOCK RESAMPLING
# ==================================================================================================

def cell19_block_length(
    n,
):

    return max(
        1,
        int(
            round(
                n ** (1.0 / 3.0)
            )
        ),
    )


def cell19_moving_block_indices(
    n,
    block_length,
    rng,
):

    if n <= block_length:
        return np.arange(
            n,
            dtype=int,
        )

    max_start = (
        n
        - block_length
    )

    sampled = []

    while len(sampled) < n:

        start = int(
            rng.integers(
                0,
                max_start + 1,
            )
        )

        block = list(
            range(
                start,
                start
                + block_length,
            )
        )

        sampled.extend(
            block
        )

    return np.asarray(
        sampled[:n],
        dtype=int,
    )


# ==================================================================================================
# 8. TASK-LEVEL STRATIFIED BLOCK BOOTSTRAP
# ==================================================================================================

def cell19_bootstrap_task(
    task_df,
    seed,
):

    task_df = (
        task_df
        .sort_values(
            [
                "Station",
                "Date",
            ]
        )
        .reset_index(drop=True)
    )

    observed = (
        cell19_skill_metrics(
            task_df[
                "Actual_Future_Target"
            ],

            task_df[
                "Novel_Prediction"
            ],

            task_df[
                "Persistence_Prediction"
            ],
        )
    )

    station_groups = {}

    for station, g in (
        task_df
        .groupby(
            "Station",
            sort=False,
        )
    ):

        g = (
            g
            .sort_values(
                "Date"
            )
            .reset_index(
                drop=True
            )
        )

        station_groups[
            station
        ] = g

    rng = np.random.default_rng(
        int(seed)
    )

    bootstrap_mae_skill = np.empty(
        CELL19_BOOTSTRAP_REPS,
        dtype=float,
    )

    bootstrap_rmse_skill = np.empty(
        CELL19_BOOTSTRAP_REPS,
        dtype=float,
    )

    bootstrap_abs_improvement = np.empty(
        CELL19_BOOTSTRAP_REPS,
        dtype=float,
    )

    bootstrap_sq_improvement = np.empty(
        CELL19_BOOTSTRAP_REPS,
        dtype=float,
    )

    block_lengths = {}

    for station, g in (
        station_groups.items()
    ):

        block_lengths[
            station
        ] = (
            cell19_block_length(
                len(g)
            )
        )

    for b in range(
        CELL19_BOOTSTRAP_REPS
    ):

        sampled_groups = []

        for station, g in (
            station_groups.items()
        ):

            n = len(g)

            block_length = (
                block_lengths[
                    station
                ]
            )

            idx = (
                cell19_moving_block_indices(
                    n,
                    block_length,
                    rng,
                )
            )

            sampled_groups.append(
                g.iloc[
                    idx
                ]
            )

        sample = pd.concat(
            sampled_groups,
            ignore_index=True,
        )

        metrics = (
            cell19_skill_metrics(
                sample[
                    "Actual_Future_Target"
                ],

                sample[
                    "Novel_Prediction"
                ],

                sample[
                    "Persistence_Prediction"
                ],
            )
        )

        bootstrap_mae_skill[b] = (
            metrics[
                "MAE_Skill"
            ]
        )

        bootstrap_rmse_skill[b] = (
            metrics[
                "RMSE_Skill"
            ]
        )

        bootstrap_abs_improvement[b] = (
            metrics[
                "Mean_Absolute_Error_Improvement"
            ]
        )

        bootstrap_sq_improvement[b] = (
            metrics[
                "Mean_Squared_Error_Improvement"
            ]
        )

    return {
        "Observed":
            observed,

        "MAE_Skill_Bootstrap":
            bootstrap_mae_skill,

        "RMSE_Skill_Bootstrap":
            bootstrap_rmse_skill,

        "Abs_Improvement_Bootstrap":
            bootstrap_abs_improvement,

        "Sq_Improvement_Bootstrap":
            bootstrap_sq_improvement,

        "Block_Lengths":
            block_lengths,
    }


# ==================================================================================================
# 9. HOLM CORRECTION
# ==================================================================================================

def cell19_holm_adjust(
    p_values,
):

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    m = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    adjusted = np.empty(
        m,
        dtype=float,
    )

    running_max = 0.0

    for rank, idx in enumerate(
        order
    ):

        multiplier = (
            m
            - rank
        )

        value = min(
            1.0,
            multiplier
            * p_values[
                idx
            ],
        )

        running_max = max(
            running_max,
            value,
        )

        adjusted[
            idx
        ] = (
            running_max
        )

    return adjusted


# ==================================================================================================
# 10. RUN TASK-LEVEL ANALYSIS
# ==================================================================================================

task_rows = []
task_bootstrap_store = {}

for task_index, task in enumerate(
    CELL19_TASKS
):

    group = (
        CELL19_INPUT_DF[
            CELL19_INPUT_DF[
                "Task"
            ].eq(task)
        ]
        .copy()
    )

    if len(group) == 0:
        raise RuntimeError(
            f"No Cell 19 observations for {task}"
        )

    result = (
        cell19_bootstrap_task(
            group,
            seed=
                CELL19_SEED
                + task_index
                * 1000,
        )
    )

    task_bootstrap_store[
        task
    ] = result

    observed = (
        result[
            "Observed"
        ]
    )

    mae_boot = (
        result[
            "MAE_Skill_Bootstrap"
        ]
    )

    rmse_boot = (
        result[
            "RMSE_Skill_Bootstrap"
        ]
    )

    abs_imp_boot = (
        result[
            "Abs_Improvement_Bootstrap"
        ]
    )

    sq_imp_boot = (
        result[
            "Sq_Improvement_Bootstrap"
        ]
    )

    mae_ci_low, mae_ci_high = (
        np.quantile(
            mae_boot,
            [
                CELL19_ALPHA / 2.0,
                1.0
                - CELL19_ALPHA / 2.0,
            ]
        )
    )

    rmse_ci_low, rmse_ci_high = (
        np.quantile(
            rmse_boot,
            [
                CELL19_ALPHA / 2.0,
                1.0
                - CELL19_ALPHA / 2.0,
            ]
        )
    )

    # One-sided p-value:
    # H0: no positive improvement
    # H1: Novel model improves over persistence
    mae_p_one = (
        np.sum(
            abs_imp_boot <= 0.0
        )
        + 1
    ) / (
        CELL19_BOOTSTRAP_REPS
        + 1
    )

    sq_p_one = (
        np.sum(
            sq_imp_boot <= 0.0
        )
        + 1
    ) / (
        CELL19_BOOTSTRAP_REPS
        + 1
    )

    # Two-sided descriptive bootstrap p-value
    mae_negative_fraction = (
        np.sum(
            abs_imp_boot <= 0.0
        )
        + 1
    ) / (
        CELL19_BOOTSTRAP_REPS
        + 1
    )

    mae_positive_fraction = (
        np.sum(
            abs_imp_boot >= 0.0
        )
        + 1
    ) / (
        CELL19_BOOTSTRAP_REPS
        + 1
    )

    mae_p_two = min(
        1.0,
        2.0
        * min(
            mae_negative_fraction,
            mae_positive_fraction,
        ),
    )

    block_description = "; ".join(
        [
            f"{station}:{block}"
            for station, block
            in result[
                "Block_Lengths"
            ].items()
        ]
    )

    task_rows.append({
        "Task":
            task,

        "N":
            int(
                len(group)
            ),

        "Observed_MAE_Skill":
            observed[
                "MAE_Skill"
            ],

        "MAE_Skill_CI_Lower":
            float(
                mae_ci_low
            ),

        "MAE_Skill_CI_Upper":
            float(
                mae_ci_high
            ),

        "MAE_One_Sided_P":
            float(
                mae_p_one
            ),

        "MAE_Two_Sided_P":
            float(
                mae_p_two
            ),

        "Observed_RMSE_Skill":
            observed[
                "RMSE_Skill"
            ],

        "RMSE_Skill_CI_Lower":
            float(
                rmse_ci_low
            ),

        "RMSE_Skill_CI_Upper":
            float(
                rmse_ci_high
            ),

        "MSE_Improvement_One_Sided_P":
            float(
                sq_p_one
            ),

        "Block_Lengths":
            block_description,
    })


CELL19_TASK_SIGNIFICANCE_DF = (
    pd.DataFrame(
        task_rows
    )
)


# ==================================================================================================
# 11. HOLM CORRECTION ACROSS SIX TASKS
# ==================================================================================================

CELL19_TASK_SIGNIFICANCE_DF[
    "MAE_Holm_P"
] = (
    cell19_holm_adjust(
        CELL19_TASK_SIGNIFICANCE_DF[
            "MAE_One_Sided_P"
        ].to_numpy(
            dtype=float
        )
    )
)

CELL19_TASK_SIGNIFICANCE_DF[
    "MSE_Holm_P"
] = (
    cell19_holm_adjust(
        CELL19_TASK_SIGNIFICANCE_DF[
            "MSE_Improvement_One_Sided_P"
        ].to_numpy(
            dtype=float
        )
    )
)


CELL19_TASK_SIGNIFICANCE_DF[
    "MAE_CI_Entirely_Positive"
] = (
    CELL19_TASK_SIGNIFICANCE_DF[
        "MAE_Skill_CI_Lower"
    ]
    > 0.0
)


CELL19_TASK_SIGNIFICANCE_DF[
    "MAE_Holm_Significant"
] = (
    CELL19_TASK_SIGNIFICANCE_DF[
        "MAE_Holm_P"
    ]
    < CELL19_ALPHA
)


CELL19_TASK_SIGNIFICANCE_DF[
    "RMSE_CI_Entirely_Positive"
] = (
    CELL19_TASK_SIGNIFICANCE_DF[
        "RMSE_Skill_CI_Lower"
    ]
    > 0.0
)


CELL19_TASK_SIGNIFICANCE_DF[
    "MSE_Holm_Significant"
] = (
    CELL19_TASK_SIGNIFICANCE_DF[
        "MSE_Holm_P"
    ]
    < CELL19_ALPHA
)


# ==================================================================================================
# 12. GLOBAL MACRO BOOTSTRAP
# ==================================================================================================

observed_macro_mae = float(
    CELL19_TASK_SIGNIFICANCE_DF[
        "Observed_MAE_Skill"
    ].mean()
)

observed_macro_rmse = float(
    CELL19_TASK_SIGNIFICANCE_DF[
        "Observed_RMSE_Skill"
    ].mean()
)


global_mae_bootstrap = np.empty(
    CELL19_BOOTSTRAP_REPS,
    dtype=float,
)

global_rmse_bootstrap = np.empty(
    CELL19_BOOTSTRAP_REPS,
    dtype=float,
)


for b in range(
    CELL19_BOOTSTRAP_REPS
):

    global_mae_bootstrap[
        b
    ] = np.mean(
        [
            task_bootstrap_store[
                task
            ][
                "MAE_Skill_Bootstrap"
            ][b]
            for task
            in CELL19_TASKS
        ]
    )

    global_rmse_bootstrap[
        b
    ] = np.mean(
        [
            task_bootstrap_store[
                task
            ][
                "RMSE_Skill_Bootstrap"
            ][b]
            for task
            in CELL19_TASKS
        ]
    )


global_mae_ci = np.quantile(
    global_mae_bootstrap,
    [
        0.025,
        0.975,
    ],
)

global_rmse_ci = np.quantile(
    global_rmse_bootstrap,
    [
        0.025,
        0.975,
    ],
)


global_mae_p = (
    np.sum(
        global_mae_bootstrap
        <= 0.0
    )
    + 1
) / (
    CELL19_BOOTSTRAP_REPS
    + 1
)


global_rmse_p = (
    np.sum(
        global_rmse_bootstrap
        <= 0.0
    )
    + 1
) / (
    CELL19_BOOTSTRAP_REPS
    + 1
)


CELL19_GLOBAL_SIGNIFICANCE_DF = pd.DataFrame([
    {
        "Metric":
            "Macro MAE Skill",

        "Observed_Skill":
            observed_macro_mae,

        "CI_Lower":
            float(
                global_mae_ci[0]
            ),

        "CI_Upper":
            float(
                global_mae_ci[1]
            ),

        "One_Sided_P":
            float(
                global_mae_p
            ),

        "CI_Entirely_Positive":
            bool(
                global_mae_ci[0]
                > 0.0
            ),
    },

    {
        "Metric":
            "Macro RMSE Skill",

        "Observed_Skill":
            observed_macro_rmse,

        "CI_Lower":
            float(
                global_rmse_ci[0]
            ),

        "CI_Upper":
            float(
                global_rmse_ci[1]
            ),

        "One_Sided_P":
            float(
                global_rmse_p
            ),

        "CI_Entirely_Positive":
            bool(
                global_rmse_ci[0]
                > 0.0
            ),
    },
])


# ==================================================================================================
# 13. SUMMARY
# ==================================================================================================

CELL19_SUMMARY_DF = pd.DataFrame([
    {
        "Measure":
            "MAE positive tasks",

        "Value":
            int(
                (
                    CELL19_TASK_SIGNIFICANCE_DF[
                        "Observed_MAE_Skill"
                    ]
                    > 0
                ).sum()
            ),

        "Out_of":
            6,
    },

    {
        "Measure":
            "MAE CI entirely positive",

        "Value":
            int(
                CELL19_TASK_SIGNIFICANCE_DF[
                    "MAE_CI_Entirely_Positive"
                ].sum()
            ),

        "Out_of":
            6,
    },

    {
        "Measure":
            "MAE Holm significant",

        "Value":
            int(
                CELL19_TASK_SIGNIFICANCE_DF[
                    "MAE_Holm_Significant"
                ].sum()
            ),

        "Out_of":
            6,
    },

    {
        "Measure":
            "RMSE positive tasks",

        "Value":
            int(
                (
                    CELL19_TASK_SIGNIFICANCE_DF[
                        "Observed_RMSE_Skill"
                    ]
                    > 0
                ).sum()
            ),

        "Out_of":
            6,
    },

    {
        "Measure":
            "RMSE CI entirely positive",

        "Value":
            int(
                CELL19_TASK_SIGNIFICANCE_DF[
                    "RMSE_CI_Entirely_Positive"
                ].sum()
            ),

        "Out_of":
            6,
    },

    {
        "Measure":
            "Mean MAE Skill",

        "Value":
            observed_macro_mae,

        "Out_of":
            np.nan,
    },

    {
        "Measure":
            "Mean RMSE Skill",

        "Value":
            observed_macro_rmse,

        "Out_of":
            np.nan,
    },
])


# ==================================================================================================
# 14. POST-ANALYSIS FROZEN-INPUT AUDIT
# ==================================================================================================

CELL19_POST_INPUT_HASH = (
    cell19_dataframe_sha256(
        CELL19_INPUT_DF
    )
)

CELL19_POST_MEANINGFUL_COPY = (
    SKILL18_PAIRED_DF[
        SKILL18_PAIRED_DF[
            "Meaningful_Change"
        ]
    ]
    .copy(deep=True)
    .reset_index(drop=True)
)

CELL19_POST_MEANINGFUL_HASH = (
    cell19_dataframe_sha256(
        CELL19_POST_MEANINGFUL_COPY
    )
)


if (
    CELL19_PRE_FULL_PREDICTIONS_HASH
    is not None
):

    CELL19_POST_FULL_PREDICTIONS_HASH = (
        cell19_dataframe_sha256(
            RGADA18_TEST_PREDICTIONS_DF
        )
    )

else:

    CELL19_POST_FULL_PREDICTIONS_HASH = None


audit_rows = [
    {
        "Check":
            "Cell19 input SHA256 unchanged",

        "Passed":
            (
                CELL19_PRE_INPUT_HASH
                ==
                CELL19_POST_INPUT_HASH
            ),
    },

    {
        "Check":
            "Cell19 input exact dataframe equality",

        "Passed":
            CELL19_PRE_INPUT_COPY.equals(
                CELL19_INPUT_DF
            ),
    },

    {
        "Check":
            "Cell19 input row count unchanged",

        "Passed":
            (
                len(
                    CELL19_PRE_INPUT_COPY
                )
                ==
                len(
                    CELL19_INPUT_DF
                )
            ),
    },

    {
        "Check":
            "Cell19 input column count unchanged",

        "Passed":
            (
                len(
                    CELL19_PRE_INPUT_COPY.columns
                )
                ==
                len(
                    CELL19_INPUT_DF.columns
                )
            ),
    },

    {
        "Check":
            "Meaningful-change SHA256 unchanged",

        "Passed":
            (
                CELL19_PRE_MEANINGFUL_HASH
                ==
                CELL19_POST_MEANINGFUL_HASH
            ),
    },

    {
        "Check":
            "Meaningful-change exact dataframe equality",

        "Passed":
            CELL19_PRE_MEANINGFUL_COPY.equals(
                CELL19_POST_MEANINGFUL_COPY
            ),
    },

    {
        "Check":
            "Meaningful-change row count unchanged",

        "Passed":
            (
                len(
                    CELL19_PRE_MEANINGFUL_COPY
                )
                ==
                len(
                    CELL19_POST_MEANINGFUL_COPY
                )
            ),
    },

    {
        "Check":
            "Meaningful-change column count unchanged",

        "Passed":
            (
                len(
                    CELL19_PRE_MEANINGFUL_COPY.columns
                )
                ==
                len(
                    CELL19_POST_MEANINGFUL_COPY.columns
                )
            ),
    },
]


if (
    CELL19_PRE_FULL_PREDICTIONS_HASH
    is not None
):

    audit_rows.extend([
        {
            "Check":
                "Final predictions SHA256 unchanged",

            "Passed":
                (
                    CELL19_PRE_FULL_PREDICTIONS_HASH
                    ==
                    CELL19_POST_FULL_PREDICTIONS_HASH
                ),
        },

        {
            "Check":
                "Final predictions exact dataframe equality",

            "Passed":
                CELL19_PRE_FULL_PREDICTIONS_COPY.equals(
                    RGADA18_TEST_PREDICTIONS_DF
                ),
        },

        {
            "Check":
                "Final predictions row count unchanged",

            "Passed":
                (
                    len(
                        CELL19_PRE_FULL_PREDICTIONS_COPY
                    )
                    ==
                    len(
                        RGADA18_TEST_PREDICTIONS_DF
                    )
                ),
        },

        {
            "Check":
                "Final predictions column count unchanged",

            "Passed":
                (
                    len(
                        CELL19_PRE_FULL_PREDICTIONS_COPY.columns
                    )
                    ==
                    len(
                        RGADA18_TEST_PREDICTIONS_DF.columns
                    )
                ),
        },
    ])


CELL19_TECHNICAL_AUDIT_DF = pd.DataFrame(
    audit_rows
)


if not (
    CELL19_TECHNICAL_AUDIT_DF[
        "Passed"
    ].all()
):

    raise RuntimeError(
        "Cell 19 frozen-input technical audit FAILED."
    )


# ==================================================================================================
# 15. PROTOCOL
# ==================================================================================================

CELL19_PROTOCOL_DF = pd.DataFrame([
    {
        "Item":
            "Analysis",

        "Value":
            "Statistical significance of Change-Event Skill",
    },

    {
        "Item":
            "Primary endpoint",

        "Value":
            "MAE Skill Score",
    },

    {
        "Item":
            "Secondary endpoint",

        "Value":
            "RMSE / squared-error improvement",
    },

    {
        "Item":
            "Bootstrap",

        "Value":
            "Paired station-stratified moving-block bootstrap",
    },

    {
        "Item":
            "Bootstrap repetitions",

        "Value":
            CELL19_BOOTSTRAP_REPS,
    },

    {
        "Item":
            "Seed",

        "Value":
            CELL19_SEED,
    },

    {
        "Item":
            "Seed purpose",

        "Value":
            "Reproducibility only",
    },

    {
        "Item":
            "Confidence level",

        "Value":
            "95%",
    },

    {
        "Item":
            "Block length",

        "Value":
            "round(n^(1/3)) per Task × Station",
    },

    {
        "Item":
            "Multiple-comparison correction",

        "Value":
            "Holm across six tasks",
    },

    {
        "Item":
            "Retraining",

        "Value":
            False,
    },

    {
        "Item":
            "Hyperparameter tuning",

        "Value":
            False,
    },

    {
        "Item":
            "Test-based model selection",

        "Value":
            False,
    },

    {
        "Item":
            "Prediction modification",

        "Value":
            False,
    },
])


# ==================================================================================================
# 16. OUTPUT
# ==================================================================================================

print("\nTask-level statistical significance:")
display(
    CELL19_TASK_SIGNIFICANCE_DF
    .round(6)
)


print("\nGlobal macro significance:")
display(
    CELL19_GLOBAL_SIGNIFICANCE_DF
    .round(6)
)


print("\nSummary:")
display(
    CELL19_SUMMARY_DF
    .round(6)
)


print("\nFrozen-input technical audit:")
display(
    CELL19_TECHNICAL_AUDIT_DF
)


print("\nProtocol declarations:")
display(
    CELL19_PROTOCOL_DECLARATIONS_DF
)


print("\nProtocol:")
display(
    CELL19_PROTOCOL_DF
)


print("\n" + "=" * 135)
print("CELL 19 COMPLETED SUCCESSFULLY")
print("=" * 135)

print(
    f"Execution time: "
    f"{time.time() - CELL19_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 20 — EXACT-REPLAY-CONTROLLED SEED STABILITY / REPRODUCIBILITY
# FINAL CORRECTED VERSION — compatible with corrected Cell 16
# ==================================================================================================

import time
import numpy as np
import pandas as pd

CELL20_START = time.time()

print("=" * 145)
print("CELL 20 — EXACT-REPLAY-CONTROLLED SEED STABILITY")
print("=" * 145)


# ==================================================================================================
# 1. REQUIRED OBJECTS
# ==================================================================================================

required_objects = [
    "TASK_DEFINITIONS",
    "TASK_DATASETS",
    "TASK_CLIENT_DATASETS",
    "TASK_SCALERS",
    "EXPECTED_STATIONS",
    "ALGORITHM_CONFIGS",
    "CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM",
    "LOCAL_MODELS",
    "FEDERATED_RUNS",
    "FEDERATED_METHOD_NAMES",
    "GLOBAL_RANDOM_SEED",
    "CENTRAL_LOCAL_EPOCHS",
    "FEDERATED_ROUNDS",
    "FEDERATED_LOCAL_EPOCHS",
    "FEDADAM_SERVER_LR",
    "FEDADAM_BETA1",
    "FEDADAM_BETA2",
    "FEDADAM_TAU",
    "GRADIENT_CLIP_NORM",
    "batch_gradient",
    "clip_grad",
    "predict_raw",
    "reg_metrics",
    "U1619_SELECTION_DF",
    "U1619_NOVEL_TEST_RESULTS_DF",
    "u1619_selected_algorithm",
    "u1619_peer_weights",
    "u1619_run_online",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise NameError(
        "Missing required objects:\n- "
        + "\n- ".join(missing)
    )

print("✓ Required Cells 10–16 objects are available.")


# ==================================================================================================
# 2. FIXED SEED PROTOCOL
# ==================================================================================================

ORIGINAL_SEED_20 = int(
    GLOBAL_RANDOM_SEED
)

SEEDS_20 = list(
    range(
        ORIGINAL_SEED_20,
        ORIGINAL_SEED_20 + 10,
    )
)

SEED_GROUPS_20 = {
    "3 Seeds":
        SEEDS_20[:3],

    "5 Seeds":
        SEEDS_20[:5],

    "10 Seeds":
        SEEDS_20[:10],
}


TASKS_20 = [
    d["Task"]
    for d in TASK_DEFINITIONS
]

ALGORITHM_ORDER_20 = list(
    ALGORITHM_CONFIGS.keys()
)

METHOD_ORDER_20 = list(
    FEDERATED_METHOD_NAMES
)

FED_METHOD_20 = "FedAdam"

METHOD_POSITION_20 = (
    METHOD_ORDER_20.index(
        FED_METHOD_20
    )
    + 1
)


# ==================================================================================================
# 3. FROZEN TASK × ALGORITHM CONFIG
# ==================================================================================================

def get_frozen_config20(
    task,
    algorithm,
):

    record = (
        CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM[
            (
                task,
                algorithm,
            )
        ]
    )

    if not isinstance(
        record,
        dict,
    ):
        raise TypeError(
            f"Invalid frozen configuration for "
            f"{task} / {algorithm}"
        )

    record = dict(
        record
    )

    for nested_key in [
        "Config",
        "config",
        "Best_Config",
        "Selected_Config",
    ]:

        if (
            nested_key in record
            and isinstance(
                record[nested_key],
                dict,
            )
        ):

            nested = dict(
                record[nested_key]
            )

            record.update(
                nested
            )

    base = dict(
        ALGORITHM_CONFIGS[
            algorithm
        ]
    )

    aliases = {
        "Learning_Rate":
            "learning_rate",

        "Learning_Rate_Decay":
            "learning_rate_decay",

        "LR_Decay":
            "learning_rate_decay",

        "L1":
            "l1",

        "L2":
            "l2",

        "Huber_Delta":
            "huber_delta",

        "Batch_Size":
            "batch_size",

        "Loss":
            "loss",
    }

    for old, new in aliases.items():

        if (
            new not in record
            and old in record
        ):
            record[new] = (
                record[
                    old
                ]
            )

    base.update(
        record
    )

    if (
        "batch_size" in base
        and pd.isna(
            base[
                "batch_size"
            ]
        )
    ):
        base[
            "batch_size"
        ] = None

    if (
        "huber_delta" in base
        and pd.isna(
            base[
                "huber_delta"
            ]
        )
    ):
        base[
            "huber_delta"
        ] = None

    return base


# ==================================================================================================
# 4. EXACT LOCAL OPTIMIZATION
# ==================================================================================================

def train_exact20(
    X,
    y_raw,
    task,
    algorithm,
    initial_parameters,
    epochs,
    seed,
    global_epoch_offset=0,
):

    cfg = (
        get_frozen_config20(
            task,
            algorithm,
        )
    )

    scaler = (
        TASK_SCALERS[
            task
        ]
    )

    X = np.asarray(
        X,
        dtype=float,
    )

    y_raw = np.asarray(
        y_raw,
        dtype=float,
    )

    y = (
        y_raw
        - float(
            scaler[
                "y_mean"
            ]
        )
    ) / float(
        scaler[
            "y_std"
        ]
    )

    parameters = (
        np.asarray(
            initial_parameters,
            dtype=float,
        )
        .copy()
    )

    n = len(
        y
    )

    batch_setting = (
        cfg.get(
            "batch_size",
            None,
        )
    )

    if batch_setting is None:

        batch_size = n

    else:

        batch_size = min(
            int(
                batch_setting
            ),
            n,
        )

    rng = np.random.default_rng(
        int(
            seed
        )
    )

    learning_rate = float(
        cfg[
            "learning_rate"
        ]
    )

    decay = float(
        cfg.get(
            "learning_rate_decay",
            1.0,
        )
    )

    l1_value = float(
        cfg.get(
            "l1",
            0.0,
        )
    )

    for epoch in range(
        int(
            epochs
        )
    ):

        global_epoch = (
            int(
                global_epoch_offset
            )
            + epoch
        )

        current_lr = (
            learning_rate
            * (
                decay
                ** global_epoch
            )
        )

        if batch_size >= n:

            order = np.arange(
                n
            )

        else:

            order = (
                rng.permutation(
                    n
                )
            )

        for start in range(
            0,
            n,
            batch_size,
        ):

            idx = order[
                start:
                start
                + batch_size
            ]

            gradient = (
                batch_gradient(
                    X[
                        idx
                    ],
                    y[
                        idx
                    ],
                    parameters,
                    cfg,
                )
            )

            gradient = (
                clip_grad(
                    gradient,
                    GRADIENT_CLIP_NORM,
                )
            )

            parameters -= (
                current_lr
                * gradient
            )

            if l1_value > 0.0:

                weights = (
                    parameters[
                        :-1
                    ]
                )

                parameters[
                    :-1
                ] = (
                    np.sign(
                        weights
                    )
                    * np.maximum(
                        np.abs(
                            weights
                        )
                        - current_lr
                        * l1_value,
                        0.0,
                    )
                )

    return parameters


# ==================================================================================================
# 5. EXACT LOCAL MODEL REPLAY FOR ONE BASE SEED
# ==================================================================================================

def train_local_models20(
    base_seed,
):

    models = {}

    for (
        task_position,
        task,
    ) in enumerate(
        TASKS_20,
        start=1,
    ):

        algorithm = (
            u1619_selected_algorithm(
                task
            )
        )

        algorithm_position = (
            ALGORITHM_ORDER_20.index(
                algorithm
            )
            + 1
        )

        for (
            station_position,
            station,
        ) in enumerate(
            EXPECTED_STATIONS,
            start=1,
        ):

            ds = (
                TASK_CLIENT_DATASETS[
                    task
                ][station][
                    "Train"
                ]
            )

            parameter_count = (
                ds[
                    "X"
                ].shape[1]
                + 1
            )

            seed = (
                int(
                    base_seed
                )
                + task_position
                * 10000
                + station_position
                * 100
                + algorithm_position
            )

            parameters = (
                train_exact20(
                    X=
                        ds[
                            "X"
                        ],

                    y_raw=
                        ds[
                            "y"
                        ],

                    task=
                        task,

                    algorithm=
                        algorithm,

                    initial_parameters=
                        np.zeros(
                            parameter_count,
                            dtype=float,
                        ),

                    epochs=
                        CENTRAL_LOCAL_EPOCHS,

                    seed=
                        seed,

                    global_epoch_offset=
                        0,
                )
            )

            models[
                (
                    task,
                    station,
                    algorithm,
                )
            ] = (
                parameters
            )

    return models


# ==================================================================================================
# 6. EXACT FEDADAM REPLAY FOR ONE BASE SEED
# ==================================================================================================

def train_fedadam_models20(
    base_seed,
):

    models = {}

    for (
        task_position,
        task,
    ) in enumerate(
        TASKS_20,
        start=1,
    ):

        algorithm = (
            u1619_selected_algorithm(
                task
            )
        )

        algorithm_position = (
            ALGORITHM_ORDER_20.index(
                algorithm
            )
            + 1
        )

        sample_dataset = (
            TASK_CLIENT_DATASETS[
                task
            ][
                EXPECTED_STATIONS[0]
            ][
                "Train"
            ]
        )

        parameter_count = (
            sample_dataset[
                "X"
            ].shape[1]
            + 1
        )

        global_parameters = (
            np.zeros(
                parameter_count,
                dtype=float,
            )
        )

        m = np.zeros(
            parameter_count,
            dtype=float,
        )

        v = np.zeros(
            parameter_count,
            dtype=float,
        )

        client_sizes = {
            station:
                len(
                    TASK_CLIENT_DATASETS[
                        task
                    ][station][
                        "Train"
                    ][
                        "y"
                    ]
                )
            for station
            in EXPECTED_STATIONS
        }

        for round_index in range(
            int(
                FEDERATED_ROUNDS
            )
        ):

            local_parameters = {}

            for (
                station_position,
                station,
            ) in enumerate(
                EXPECTED_STATIONS,
                start=1,
            ):

                ds = (
                    TASK_CLIENT_DATASETS[
                        task
                    ][station][
                        "Train"
                    ]
                )

                optimizer_seed = (
                    int(
                        base_seed
                    )
                    + task_position
                    * 1_000_000
                    + algorithm_position
                    * 10_000
                    + METHOD_POSITION_20
                    * 1_000
                    + round_index
                    * 10
                    + station_position
                )

                local_parameters[
                    station
                ] = (
                    train_exact20(
                        X=
                            ds[
                                "X"
                            ],

                        y_raw=
                            ds[
                                "y"
                            ],

                        task=
                            task,

                        algorithm=
                            algorithm,

                        initial_parameters=
                            global_parameters,

                        epochs=
                            FEDERATED_LOCAL_EPOCHS,

                        seed=
                            optimizer_seed,

                        global_epoch_offset=
                            round_index
                            * int(
                                FEDERATED_LOCAL_EPOCHS
                            ),
                    )
                )

            total_samples = sum(
                client_sizes.values()
            )

            averaged_parameters = sum(
                (
                    client_sizes[
                        station
                    ]
                    / total_samples
                )
                * local_parameters[
                    station
                ]
                for station
                in EXPECTED_STATIONS
            )

            displacement = (
                averaged_parameters
                - global_parameters
            )

            m = (
                FEDADAM_BETA1
                * m
                + (
                    1.0
                    - FEDADAM_BETA1
                )
                * displacement
            )

            v = (
                FEDADAM_BETA2
                * v
                + (
                    1.0
                    - FEDADAM_BETA2
                )
                * (
                    displacement
                    ** 2
                )
            )

            global_parameters = (
                global_parameters
                + FEDADAM_SERVER_LR
                * m
                / (
                    np.sqrt(
                        v
                    )
                    + FEDADAM_TAU
                )
            )

        models[
            (
                task,
                algorithm,
            )
        ] = (
            global_parameters
        )

    return models


# ==================================================================================================
# 7. FROZEN CELL-16 RG CONFIG
# ==================================================================================================

def config20(
    task,
    station,
):

    row = (
        U1619_SELECTION_DF[
            U1619_SELECTION_DF[
                "Task"
            ].eq(
                task
            )
            &
            U1619_SELECTION_DF[
                "Station"
            ].eq(
                station
            )
        ]
        .iloc[0]
    )

    return {
        "prior_global":
            float(
                row[
                    "prior_global"
                ]
            ),

        "expert_power":
            float(
                row[
                    "expert_power"
                ]
            ),

        "lookback":
            int(
                row[
                    "lookback"
                ]
            ),

        "decay":
            float(
                row[
                    "decay"
                ]
            ),

        "alpha":
            float(
                row[
                    "alpha"
                ]
            ),

        "local_relation_mix":
            float(
                row[
                    "local_relation_mix"
                ]
            ),
    }


# ==================================================================================================
# 8. STATIC EXPERT PREDICTIONS USING SEEDED MODELS
# ==================================================================================================

def experts20(
    task,
    station,
    split_name,
    local_models,
    federated_models,
):

    algorithm = (
        u1619_selected_algorithm(
            task
        )
    )

    ds = (
        TASK_CLIENT_DATASETS[
            task
        ][station][
            split_name
        ]
    )

    X = (
        ds[
            "X"
        ]
    )

    global_parameters = (
        federated_models[
            (
                task,
                algorithm,
            )
        ]
    )

    self_parameters = (
        local_models[
            (
                task,
                station,
                algorithm,
            )
        ]
    )

    global_prediction = (
        np.asarray(
            predict_raw(
                X,
                global_parameters,
                task,
            ),
            dtype=float,
        )
    )

    self_prediction = (
        np.asarray(
            predict_raw(
                X,
                self_parameters,
                task,
            ),
            dtype=float,
        )
    )

    peer_weights = (
        u1619_peer_weights(
            task,
            station,
        )
    )

    relation_prediction = np.zeros(
        len(
            self_prediction
        ),
        dtype=float,
    )

    weight_sum = 0.0

    for peer_station, weight in (
        peer_weights.items()
    ):

        if (
            peer_station
            == station
        ):
            continue

        peer_parameters = (
            local_models[
                (
                    task,
                    peer_station,
                    algorithm,
                )
            ]
        )

        peer_prediction = (
            np.asarray(
                predict_raw(
                    X,
                    peer_parameters,
                    task,
                ),
                dtype=float,
            )
        )

        relation_prediction += (
            float(
                weight
            )
            * peer_prediction
        )

        weight_sum += float(
            weight
        )

    if weight_sum > 0.0:

        relation_prediction /= (
            weight_sum
        )

    else:

        relation_prediction = (
            self_prediction.copy()
        )

    return {
        "Global":
            global_prediction,

        "Self":
            self_prediction,

        "Relation":
            relation_prediction,
    }


# ==================================================================================================
# 9. EXACT FINAL CELL-16 EVALUATION
# ==================================================================================================

def evaluate_final20(
    local_models,
    federated_models,
):

    task_rows = []

    for task in TASKS_20:

        meta = (
            TASK_DATASETS[
                task
            ][
                "Meta"
            ]
        )

        actual_parts = []
        prediction_parts = []

        for station in (
            EXPECTED_STATIONS
        ):

            val_ds = (
                TASK_CLIENT_DATASETS[
                    task
                ][station][
                    "Validation"
                ]
            )

            test_ds = (
                TASK_CLIENT_DATASETS[
                    task
                ][station][
                    "Test"
                ]
            )

            val_frame = (
                val_ds[
                    "Frame"
                ]
                .reset_index(
                    drop=True
                )
            )

            test_frame = (
                test_ds[
                    "Frame"
                ]
                .reset_index(
                    drop=True
                )
            )

            val_experts = (
                experts20(
                    task,
                    station,
                    "Validation",
                    local_models,
                    federated_models,
                )
            )

            test_experts = (
                experts20(
                    task,
                    station,
                    "Test",
                    local_models,
                    federated_models,
                )
            )

            combined_dates = (
                pd.concat(
                    [
                        val_frame[
                            "Date"
                        ],
                        test_frame[
                            "Date"
                        ],
                    ],
                    ignore_index=True,
                )
            )

            combined_target_dates = (
                pd.concat(
                    [
                        val_frame[
                            meta[
                                "Future_Date_Column"
                            ]
                        ],
                        test_frame[
                            meta[
                                "Future_Date_Column"
                            ]
                        ],
                    ],
                    ignore_index=True,
                )
            )

            combined_y = (
                np.concatenate(
                    [
                        np.asarray(
                            val_ds[
                                "y"
                            ],
                            dtype=float,
                        ),

                        np.asarray(
                            test_ds[
                                "y"
                            ],
                            dtype=float,
                        ),
                    ]
                )
            )

            combined_global = (
                np.concatenate(
                    [
                        val_experts[
                            "Global"
                        ],
                        test_experts[
                            "Global"
                        ],
                    ]
                )
            )

            combined_self = (
                np.concatenate(
                    [
                        val_experts[
                            "Self"
                        ],
                        test_experts[
                            "Self"
                        ],
                    ]
                )
            )

            combined_relation = (
                np.concatenate(
                    [
                        val_experts[
                            "Relation"
                        ],
                        test_experts[
                            "Relation"
                        ],
                    ]
                )
            )

            cfg = (
                config20(
                    task,
                    station,
                )
            )

            # --------------------------------------------------------------------------------------
            # CORRECTED CURRENT Cell-16 INTERFACE:
            # config fields are passed explicitly.
            # --------------------------------------------------------------------------------------

            combined_prediction, _ = (
                u1619_run_online(
                    dates=
                        combined_dates,

                    target_dates=
                        combined_target_dates,

                    y=
                        combined_y,

                    global_pred=
                        combined_global,

                    self_pred=
                        combined_self,

                    relation_pred=
                        combined_relation,

                    prior_global=
                        cfg[
                            "prior_global"
                        ],

                    expert_power=
                        cfg[
                            "expert_power"
                        ],

                    lookback=
                        cfg[
                            "lookback"
                        ],

                    decay=
                        cfg[
                            "decay"
                        ],

                    alpha=
                        cfg[
                            "alpha"
                        ],

                    local_relation_mix=
                        cfg[
                            "local_relation_mix"
                        ],
                )
            )

            n_val = len(
                val_ds[
                    "y"
                ]
            )

            test_prediction = (
                np.asarray(
                    combined_prediction[
                        n_val:
                    ],
                    dtype=float,
                )
            )

            actual_parts.append(
                np.asarray(
                    test_ds[
                        "y"
                    ],
                    dtype=float,
                )
            )

            prediction_parts.append(
                test_prediction
            )

        actual = (
            np.concatenate(
                actual_parts
            )
        )

        prediction = (
            np.concatenate(
                prediction_parts
            )
        )

        metrics = (
            reg_metrics(
                actual,
                prediction,
            )
        )

        task_rows.append({
            "Task":
                task,

            "MAE":
                float(
                    metrics[
                        "MAE"
                    ]
                ),

            "RMSE":
                float(
                    metrics[
                        "RMSE"
                    ]
                ),

            "R2":
                float(
                    metrics[
                        "R2"
                    ]
                ),
        })

    return pd.DataFrame(
        task_rows
    )


# ==================================================================================================
# 10. MANDATORY CONTROL A — CELL 11 LOCAL PARAMETER REPRODUCTION
# ==================================================================================================

print(
    "\nRunning mandatory original-seed reproduction control ..."
)

CONTROL20_LOCAL_MODELS = (
    train_local_models20(
        ORIGINAL_SEED_20
    )
)

control_local_rows = []

for task in TASKS_20:

    algorithm = (
        u1619_selected_algorithm(
            task
        )
    )

    for station in (
        EXPECTED_STATIONS
    ):

        reproduced = (
            CONTROL20_LOCAL_MODELS[
                (
                    task,
                    station,
                    algorithm,
                )
            ]
        )

        reference = np.asarray(
            LOCAL_MODELS[
                (
                    task,
                    station,
                    algorithm,
                )
            ][
                "Parameters"
            ],
            dtype=float,
        )

        difference = float(
            np.max(
                np.abs(
                    reproduced
                    - reference
                )
            )
        )

        control_local_rows.append({
            "Task":
                task,

            "Station":
                station,

            "Algorithm":
                algorithm,

            "Max_Parameter_Difference":
                difference,

            "Reproduced":
                bool(
                    difference
                    < 1e-12
                ),
        })


CELL20_CONTROL_LOCAL_DF = (
    pd.DataFrame(
        control_local_rows
    )
)

print(
    "\nCONTROL A — Cell 11 local parameters:"
)

display(
    CELL20_CONTROL_LOCAL_DF
    .round(12)
)

if not (
    CELL20_CONTROL_LOCAL_DF[
        "Reproduced"
    ].all()
):

    raise RuntimeError(
        "CONTROL A FAILED: "
        "Cell 11 local parameters were not exactly reproduced."
    )


# ==================================================================================================
# 11. MANDATORY CONTROL B — CELL 13 FEDADAM PARAMETER REPRODUCTION
# ==================================================================================================

CONTROL20_FED_MODELS = (
    train_fedadam_models20(
        ORIGINAL_SEED_20
    )
)

control_fed_rows = []

for task in TASKS_20:

    algorithm = (
        u1619_selected_algorithm(
            task
        )
    )

    reproduced = (
        CONTROL20_FED_MODELS[
            (
                task,
                algorithm,
            )
        ]
    )

    reference = (
        np.asarray(
            FEDERATED_RUNS[
                (
                    task,
                    algorithm,
                    "FedAdam",
                )
            ][
                "Parameters"
            ],
            dtype=float,
        )
    )

    difference = float(
        np.max(
            np.abs(
                reproduced
                - reference
            )
        )
    )

    control_fed_rows.append({
        "Task":
            task,

        "Algorithm":
            algorithm,

        "Max_Parameter_Difference":
            difference,

        "Reproduced":
            bool(
                difference
                < 1e-12
            ),
    })


CELL20_CONTROL_FED_DF = (
    pd.DataFrame(
        control_fed_rows
    )
)

print(
    "\nCONTROL B — Cell 13 FedAdam parameters:"
)

display(
    CELL20_CONTROL_FED_DF
    .round(12)
)

if not (
    CELL20_CONTROL_FED_DF[
        "Reproduced"
    ].all()
):

    raise RuntimeError(
        "CONTROL B FAILED: "
        "Cell 13 FedAdam parameters were not exactly reproduced."
    )


# ==================================================================================================
# 12. MANDATORY CONTROL C — CELL 16 FINAL RESULT REPRODUCTION
# ==================================================================================================

CONTROL20_FINAL_DF = (
    evaluate_final20(
        CONTROL20_LOCAL_MODELS,
        CONTROL20_FED_MODELS,
    )
)

control_final_rows = []

for _, row in (
    CONTROL20_FINAL_DF.iterrows()
):

    task = (
        row[
            "Task"
        ]
    )

    reference = (
        U1619_NOVEL_TEST_RESULTS_DF[
            U1619_NOVEL_TEST_RESULTS_DF[
                "Task"
            ].eq(
                task
            )
        ]
        .iloc[0]
    )

    mae_diff = abs(
        float(
            row[
                "MAE"
            ]
        )
        - float(
            reference[
                "MAE"
            ]
        )
    )

    rmse_diff = abs(
        float(
            row[
                "RMSE"
            ]
        )
        - float(
            reference[
                "RMSE"
            ]
        )
    )

    r2_diff = abs(
        float(
            row[
                "R2"
            ]
        )
        - float(
            reference[
                "R2"
            ]
        )
    )

    maximum_difference = max(
        mae_diff,
        rmse_diff,
        r2_diff,
    )

    control_final_rows.append({
        "Task":
            task,

        "MAE":
            row[
                "MAE"
            ],

        "Reference_MAE":
            reference[
                "MAE"
            ],

        "RMSE":
            row[
                "RMSE"
            ],

        "Reference_RMSE":
            reference[
                "RMSE"
            ],

        "R2":
            row[
                "R2"
            ],

        "Reference_R2":
            reference[
                "R2"
            ],

        "Max_Metric_Difference":
            maximum_difference,

        "Reproduced":
            bool(
                maximum_difference
                < 1e-10
            ),
    })


CELL20_CONTROL_FINAL_DF = (
    pd.DataFrame(
        control_final_rows
    )
)

print(
    "\nCONTROL C — Cell 16 final results:"
)

display(
    CELL20_CONTROL_FINAL_DF
    .round(12)
)

if not (
    CELL20_CONTROL_FINAL_DF[
        "Reproduced"
    ].all()
):

    raise RuntimeError(
        "CONTROL C FAILED: "
        "Cell 16 final results were not exactly reproduced."
    )


print(
    "\n✓ ORIGINAL SEED EXACT REPLAY PASSED."
)


# ==================================================================================================
# 13. RUN 10 SEEDS
# ==================================================================================================

seed_task_rows = []

for seed_index, seed in enumerate(
    SEEDS_20,
    start=1,
):

    print(
        f"\nSeed {seed_index:02d}/10: "
        f"{seed}"
    )

    if seed == ORIGINAL_SEED_20:

        local_models_seed = (
            CONTROL20_LOCAL_MODELS
        )

        federated_models_seed = (
            CONTROL20_FED_MODELS
        )

        task_results = (
            CONTROL20_FINAL_DF[
                [
                    "Task",
                    "MAE",
                    "RMSE",
                    "R2",
                ]
            ]
            .copy()
        )

    else:

        local_models_seed = (
            train_local_models20(
                seed
            )
        )

        federated_models_seed = (
            train_fedadam_models20(
                seed
            )
        )

        task_results = (
            evaluate_final20(
                local_models_seed,
                federated_models_seed,
            )
        )

    for _, result in (
        task_results.iterrows()
    ):

        seed_task_rows.append({
            "Seed":
                int(
                    seed
                ),

            "Task":
                result[
                    "Task"
                ],

            "MAE":
                float(
                    result[
                        "MAE"
                    ]
                ),

            "RMSE":
                float(
                    result[
                        "RMSE"
                    ]
                ),

            "R2":
                float(
                    result[
                        "R2"
                    ]
                ),
        })


CELL20_SEED_TASK_RESULTS_DF = (
    pd.DataFrame(
        seed_task_rows
    )
)


# ==================================================================================================
# 14. TASK-LEVEL SUMMARY FOR 3 / 5 / 10 SEEDS
# ==================================================================================================

group_task_rows = []

for group_name, group_seeds in (
    SEED_GROUPS_20.items()
):

    subset = (
        CELL20_SEED_TASK_RESULTS_DF[
            CELL20_SEED_TASK_RESULTS_DF[
                "Seed"
            ].isin(
                group_seeds
            )
        ]
    )

    for task in TASKS_20:

        g = (
            subset[
                subset[
                    "Task"
                ].eq(
                    task
                )
            ]
        )

        mean_mae = float(
            g[
                "MAE"
            ].mean()
        )

        sd_mae = float(
            g[
                "MAE"
            ].std(
                ddof=1
            )
        ) if len(g) > 1 else 0.0

        mean_rmse = float(
            g[
                "RMSE"
            ].mean()
        )

        sd_rmse = float(
            g[
                "RMSE"
            ].std(
                ddof=1
            )
        ) if len(g) > 1 else 0.0

        mean_r2 = float(
            g[
                "R2"
            ].mean()
        )

        sd_r2 = float(
            g[
                "R2"
            ].std(
                ddof=1
            )
        ) if len(g) > 1 else 0.0

        cv_mae = (
            100.0
            * sd_mae
            / mean_mae
            if mean_mae != 0.0
            else np.nan
        )

        cv_rmse = (
            100.0
            * sd_rmse
            / mean_rmse
            if mean_rmse != 0.0
            else np.nan
        )

        group_task_rows.append({
            "Seed_Group":
                group_name,

            "Task":
                task,

            "N_Seeds":
                len(
                    group_seeds
                ),

            "Mean_MAE":
                mean_mae,

            "SD_MAE":
                sd_mae,

            "CV_MAE_Percent":
                cv_mae,

            "Mean_RMSE":
                mean_rmse,

            "SD_RMSE":
                sd_rmse,

            "CV_RMSE_Percent":
                cv_rmse,

            "Mean_R2":
                mean_r2,

            "SD_R2":
                sd_r2,

            "Min_R2":
                float(
                    g[
                        "R2"
                    ].min()
                ),

            "Max_R2":
                float(
                    g[
                        "R2"
                    ].max()
                ),
        })


CELL20_TASK_STABILITY_DF = (
    pd.DataFrame(
        group_task_rows
    )
)


# ==================================================================================================
# 15. STABILITY LABEL
# ==================================================================================================

def stability_label20(
    cv_mae,
):

    if not np.isfinite(
        cv_mae
    ):
        return "Undefined"

    if cv_mae < 1.0:
        return "Very stable"

    if cv_mae < 2.5:
        return "Stable"

    if cv_mae < 5.0:
        return "Moderately stable"

    return "Sensitive"


CELL20_TASK_STABILITY_DF[
    "Stability"
] = (
    CELL20_TASK_STABILITY_DF[
        "CV_MAE_Percent"
    ]
    .apply(
        stability_label20
    )
)


# ==================================================================================================
# 16. OVERALL RESULT PER SEED
# ==================================================================================================

overall_seed_rows = []

for seed in SEEDS_20:

    g = (
        CELL20_SEED_TASK_RESULTS_DF[
            CELL20_SEED_TASK_RESULTS_DF[
                "Seed"
            ].eq(
                seed
            )
        ]
    )

    overall_seed_rows.append({
        "Seed":
            seed,

        "Mean_MAE":
            float(
                g[
                    "MAE"
                ].mean()
            ),

        "Mean_RMSE":
            float(
                g[
                    "RMSE"
                ].mean()
            ),

        "Mean_R2":
            float(
                g[
                    "R2"
                ].mean()
            ),
    })


CELL20_OVERALL_BY_SEED_DF = (
    pd.DataFrame(
        overall_seed_rows
    )
)


# ==================================================================================================
# 17. OVERALL 3 / 5 / 10-SEED STABILITY
# ==================================================================================================

overall_group_rows = []

for group_name, group_seeds in (
    SEED_GROUPS_20.items()
):

    g = (
        CELL20_OVERALL_BY_SEED_DF[
            CELL20_OVERALL_BY_SEED_DF[
                "Seed"
            ].isin(
                group_seeds
            )
        ]
    )

    mean_mae = float(
        g[
            "Mean_MAE"
        ].mean()
    )

    sd_mae = float(
        g[
            "Mean_MAE"
        ].std(
            ddof=1
        )
    ) if len(g) > 1 else 0.0

    mean_rmse = float(
        g[
            "Mean_RMSE"
        ].mean()
    )

    sd_rmse = float(
        g[
            "Mean_RMSE"
        ].std(
            ddof=1
        )
    ) if len(g) > 1 else 0.0

    mean_r2 = float(
        g[
            "Mean_R2"
        ].mean()
    )

    sd_r2 = float(
        g[
            "Mean_R2"
        ].std(
            ddof=1
        )
    ) if len(g) > 1 else 0.0

    overall_group_rows.append({
        "Seed_Group":
            group_name,

        "N_Seeds":
            len(
                group_seeds
            ),

        "Mean_MAE":
            mean_mae,

        "SD_MAE":
            sd_mae,

        "CV_MAE_Percent":
            (
                100.0
                * sd_mae
                / mean_mae
            ),

        "Mean_RMSE":
            mean_rmse,

        "SD_RMSE":
            sd_rmse,

        "CV_RMSE_Percent":
            (
                100.0
                * sd_rmse
                / mean_rmse
            ),

        "Mean_R2":
            mean_r2,

        "SD_R2":
            sd_r2,

        "Min_R2":
            float(
                g[
                    "Mean_R2"
                ].min()
            ),

        "Max_R2":
            float(
                g[
                    "Mean_R2"
                ].max()
            ),
    })


CELL20_OVERALL_STABILITY_DF = (
    pd.DataFrame(
        overall_group_rows
    )
)

CELL20_OVERALL_STABILITY_DF[
    "Stability"
] = (
    CELL20_OVERALL_STABILITY_DF[
        "CV_MAE_Percent"
    ]
    .apply(
        stability_label20
    )
)


# ==================================================================================================
# 18. FINAL 10-SEED TABLE
# ==================================================================================================

CELL20_FINAL_10SEED_TASK_DF = (
    CELL20_TASK_STABILITY_DF[
        CELL20_TASK_STABILITY_DF[
            "Seed_Group"
        ].eq(
            "10 Seeds"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

CELL20_FINAL_10SEED_OVERALL_DF = (
    CELL20_OVERALL_STABILITY_DF[
        CELL20_OVERALL_STABILITY_DF[
            "Seed_Group"
        ].eq(
            "10 Seeds"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# 19. PROTOCOL
# ==================================================================================================

CELL20_PROTOCOL_DF = pd.DataFrame([
    {
        "Item":
            "Analysis",

        "Value":
            "Seed stability / reproducibility",
    },

    {
        "Item":
            "Original seed",

        "Value":
            ORIGINAL_SEED_20,
    },

    {
        "Item":
            "Evaluated seeds",

        "Value":
            "42–51",
    },

    {
        "Item":
            "Reported groups",

        "Value":
            "3 / 5 / 10 seeds",
    },

    {
        "Item":
            "Task algorithm",

        "Value":
            "Frozen from Cell 14 / Cell 16",
    },

    {
        "Item":
            "Task×Algorithm configuration",

        "Value":
            "Frozen from Cell 10",
    },

    {
        "Item":
            "RG configuration",

        "Value":
            "Frozen from Cell 16",
    },

    {
        "Item":
            "Relation topology",

        "Value":
            "Peer-only",
    },

    {
        "Item":
            "Validation/Test chronology",

        "Value":
            "Continuous",
    },

    {
        "Item":
            "Delayed target feedback",

        "Value":
            True,
    },

    {
        "Item":
            "Hyperparameter tuning",

        "Value":
            False,
    },

    {
        "Item":
            "Model re-selection",

        "Value":
            False,
    },

    {
        "Item":
            "Test-based selection",

        "Value":
            False,
    },

    {
        "Item":
            "Control A",

        "Value":
            "Exact Cell 11 local parameter replay",
    },

    {
        "Item":
            "Control B",

        "Value":
            "Exact Cell 13 FedAdam parameter replay",
    },

    {
        "Item":
            "Control C",

        "Value":
            "Exact Cell 16 final-result replay",
    },
])


# ==================================================================================================
# 20. OUTPUT
# ==================================================================================================

print("\nMandatory controls:")

display(
    pd.DataFrame([
        {
            "Control":
                "A — Cell 11 Local",

            "Passed":
                bool(
                    CELL20_CONTROL_LOCAL_DF[
                        "Reproduced"
                    ].all()
                ),
        },

        {
            "Control":
                "B — Cell 13 FedAdam",

            "Passed":
                bool(
                    CELL20_CONTROL_FED_DF[
                        "Reproduced"
                    ].all()
                ),
        },

        {
            "Control":
                "C — Cell 16 Final Model",

            "Passed":
                bool(
                    CELL20_CONTROL_FINAL_DF[
                        "Reproduced"
                    ].all()
                ),
        },
    ])
)


print("\nOverall result by seed:")

display(
    CELL20_OVERALL_BY_SEED_DF
    .round(6)
)


print("\nOverall stability — 3 / 5 / 10 seeds:")

display(
    CELL20_OVERALL_STABILITY_DF
    .round(6)
)


print("\nFinal 10-seed task stability:")

display(
    CELL20_FINAL_10SEED_TASK_DF[
        [
            "Task",
            "N_Seeds",
            "Mean_MAE",
            "SD_MAE",
            "CV_MAE_Percent",
            "Mean_RMSE",
            "SD_RMSE",
            "CV_RMSE_Percent",
            "Mean_R2",
            "SD_R2",
            "Min_R2",
            "Max_R2",
            "Stability",
        ]
    ]
    .round(6)
)


print("\nFinal 10-seed overall stability:")

display(
    CELL20_FINAL_10SEED_OVERALL_DF
    .round(6)
)


print("\nProtocol:")

display(
    CELL20_PROTOCOL_DF
)


print("\n" + "=" * 145)
print("CELL 20 COMPLETED SUCCESSFULLY")
print("=" * 145)

print(
    f"Execution time: "
    f"{time.time() - CELL20_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 21 A — STATION-LEVEL GENERALIZATION / LOSO
# FINAL CORRECTED VERSION — compatible with corrected Cell 16
# ==================================================================================================

import time
import numpy as np
import pandas as pd

CELL21_START = time.time()

print("=" * 145)
print("CELL 21 — STATION-LEVEL GENERALIZATION / LOSO")
print("=" * 145)


# ==================================================================================================
# 1. REQUIRED OBJECTS
# ==================================================================================================

required_objects = [
    "TASK_DEFINITIONS",
    "TASK_DATASETS",
    "TASK_CLIENT_DATASETS",
    "TASK_SCALERS",
    "EXPECTED_STATIONS",
    "ALGORITHM_CONFIGS",
    "CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM",
    "GLOBAL_RANDOM_SEED",
    "CENTRAL_LOCAL_EPOCHS",
    "FEDERATED_ROUNDS",
    "FEDERATED_LOCAL_EPOCHS",
    "FEDADAM_SERVER_LR",
    "FEDADAM_BETA1",
    "FEDADAM_BETA2",
    "FEDADAM_TAU",
    "GRADIENT_CLIP_NORM",
    "batch_gradient",
    "clip_grad",
    "predict_raw",
    "reg_metrics",
    "u1619_selected_algorithm",
    "u1619_run_online",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise NameError(
        "Missing required objects:\n- "
        + "\n- ".join(missing)
    )

print("✓ Required Cells 10–16 objects are available.")


# ==================================================================================================
# 2. FIXED PROTOCOL
# ==================================================================================================

TASKS_21 = [
    d["Task"]
    for d in TASK_DEFINITIONS
]

ALGORITHM_ORDER_21 = list(
    ALGORITHM_CONFIGS.keys()
)

SEED_21 = int(
    GLOBAL_RANDOM_SEED
)

SOURCE_FED_METHOD_21 = "FedAdam"


# ==================================================================================================
# 3. FROZEN CONFIG HELPER
# ==================================================================================================

def get_frozen_config21(
    task,
    algorithm,
):

    record = (
        CENTRALIZED_BEST_CONFIG_BY_TASK_ALGORITHM[
            (
                task,
                algorithm,
            )
        ]
    )

    if not isinstance(
        record,
        dict,
    ):
        raise TypeError(
            f"Invalid frozen configuration for "
            f"{task} / {algorithm}"
        )

    record = dict(
        record
    )

    for nested_key in [
        "Config",
        "config",
        "Best_Config",
        "Selected_Config",
    ]:

        if (
            nested_key in record
            and isinstance(
                record[nested_key],
                dict,
            )
        ):
            record.update(
                dict(
                    record[
                        nested_key
                    ]
                )
            )

    base = dict(
        ALGORITHM_CONFIGS[
            algorithm
        ]
    )

    aliases = {
        "Learning_Rate":
            "learning_rate",

        "Learning_Rate_Decay":
            "learning_rate_decay",

        "LR_Decay":
            "learning_rate_decay",

        "L1":
            "l1",

        "L2":
            "l2",

        "Huber_Delta":
            "huber_delta",

        "Batch_Size":
            "batch_size",
    }

    for old, new in aliases.items():

        if (
            new not in record
            and old in record
        ):
            record[new] = (
                record[
                    old
                ]
            )

    base.update(
        record
    )

    if (
        "batch_size" in base
        and pd.isna(
            base[
                "batch_size"
            ]
        )
    ):
        base[
            "batch_size"
        ] = None

    if (
        "huber_delta" in base
        and pd.isna(
            base[
                "huber_delta"
            ]
        )
    ):
        base[
            "huber_delta"
        ] = None

    return base


# ==================================================================================================
# 4. TRAIN EXACT MODEL ON PROVIDED SOURCE DATA
# ==================================================================================================

def train_exact21(
    X,
    y_raw,
    task,
    algorithm,
    initial_parameters,
    epochs,
    seed,
    scaler,
    global_epoch_offset=0,
):

    cfg = (
        get_frozen_config21(
            task,
            algorithm,
        )
    )

    X = np.asarray(
        X,
        dtype=float,
    )

    y_raw = np.asarray(
        y_raw,
        dtype=float,
    )

    y = (
        y_raw
        - float(
            scaler[
                "y_mean"
            ]
        )
    ) / float(
        scaler[
            "y_std"
        ]
    )

    parameters = (
        np.asarray(
            initial_parameters,
            dtype=float,
        )
        .copy()
    )

    n = len(
        y
    )

    batch_setting = (
        cfg.get(
            "batch_size",
            None,
        )
    )

    if batch_setting is None:
        batch_size = n
    else:
        batch_size = min(
            int(
                batch_setting
            ),
            n,
        )

    rng = np.random.default_rng(
        int(
            seed
        )
    )

    learning_rate = float(
        cfg[
            "learning_rate"
        ]
    )

    decay = float(
        cfg.get(
            "learning_rate_decay",
            1.0,
        )
    )

    l1_value = float(
        cfg.get(
            "l1",
            0.0,
        )
    )

    for epoch in range(
        int(
            epochs
        )
    ):

        global_epoch = (
            int(
                global_epoch_offset
            )
            + epoch
        )

        current_lr = (
            learning_rate
            * (
                decay
                ** global_epoch
            )
        )

        if batch_size >= n:
            order = np.arange(
                n
            )
        else:
            order = rng.permutation(
                n
            )

        for start in range(
            0,
            n,
            batch_size,
        ):

            idx = order[
                start:
                start
                + batch_size
            ]

            gradient = (
                batch_gradient(
                    X[
                        idx
                    ],
                    y[
                        idx
                    ],
                    parameters,
                    cfg,
                )
            )

            gradient = (
                clip_grad(
                    gradient,
                    GRADIENT_CLIP_NORM,
                )
            )

            parameters -= (
                current_lr
                * gradient
            )

            if l1_value > 0.0:

                w = (
                    parameters[
                        :-1
                    ]
                )

                parameters[
                    :-1
                ] = (
                    np.sign(
                        w
                    )
                    * np.maximum(
                        np.abs(
                            w
                        )
                        - current_lr
                        * l1_value,
                        0.0,
                    )
                )

    return parameters


# ==================================================================================================
# 5. SOURCE-ONLY SCALER
# ==================================================================================================

def source_scaler21(
    task,
    source_stations,
):

    y_parts = []

    for station in source_stations:

        y_parts.append(
            np.asarray(
                TASK_CLIENT_DATASETS[
                    task
                ][station][
                    "Train"
                ][
                    "y"
                ],
                dtype=float,
            )
        )

    y_all = np.concatenate(
        y_parts
    )

    y_mean = float(
        np.mean(
            y_all
        )
    )

    y_std = float(
        np.std(
            y_all
        )
    )

    if (
        not np.isfinite(
            y_std
        )
        or y_std <= 1e-12
    ):
        y_std = 1.0

    return {
        "y_mean":
            y_mean,

        "y_std":
            y_std,
    }


# ==================================================================================================
# 6. TRAIN SOURCE LOCAL MODELS
# ==================================================================================================

def train_source_local_models21(
    task,
    held_out_station,
    source_stations,
    source_scaler,
):

    algorithm = (
        u1619_selected_algorithm(
            task
        )
    )

    algorithm_position = (
        ALGORITHM_ORDER_21.index(
            algorithm
        )
        + 1
    )

    task_position = (
        TASKS_21.index(
            task
        )
        + 1
    )

    models = {}

    for source_position, station in enumerate(
        source_stations,
        start=1,
    ):

        ds = (
            TASK_CLIENT_DATASETS[
                task
            ][station][
                "Train"
            ]
        )

        p = (
            ds[
                "X"
            ].shape[1]
            + 1
        )

        seed = (
            SEED_21
            + task_position
            * 10000
            + source_position
            * 100
            + algorithm_position
        )

        models[
            station
        ] = (
            train_exact21(
                X=
                    ds[
                        "X"
                    ],

                y_raw=
                    ds[
                        "y"
                    ],

                task=
                    task,

                algorithm=
                    algorithm,

                initial_parameters=
                    np.zeros(
                        p,
                        dtype=float,
                    ),

                epochs=
                    CENTRAL_LOCAL_EPOCHS,

                seed=
                    seed,

                scaler=
                    source_scaler,

                global_epoch_offset=
                    0,
            )
        )

    return models


# ==================================================================================================
# 7. TRAIN SOURCE-ONLY FEDADAM
# ==================================================================================================

def train_source_fedadam21(
    task,
    source_stations,
    source_scaler,
):

    algorithm = (
        u1619_selected_algorithm(
            task
        )
    )

    algorithm_position = (
        ALGORITHM_ORDER_21.index(
            algorithm
        )
        + 1
    )

    task_position = (
        TASKS_21.index(
            task
        )
        + 1
    )

    sample_ds = (
        TASK_CLIENT_DATASETS[
            task
        ][
            source_stations[0]
        ][
            "Train"
        ]
    )

    p = (
        sample_ds[
            "X"
        ].shape[1]
        + 1
    )

    global_parameters = np.zeros(
        p,
        dtype=float,
    )

    m = np.zeros(
        p,
        dtype=float,
    )

    v = np.zeros(
        p,
        dtype=float,
    )

    client_sizes = {
        station:
            len(
                TASK_CLIENT_DATASETS[
                    task
                ][station][
                    "Train"
                ][
                    "y"
                ]
            )
        for station
        in source_stations
    }

    for round_index in range(
        int(
            FEDERATED_ROUNDS
        )
    ):

        local_parameters = {}

        for (
            station_position,
            station,
        ) in enumerate(
            source_stations,
            start=1,
        ):

            ds = (
                TASK_CLIENT_DATASETS[
                    task
                ][station][
                    "Train"
                ]
            )

            seed = (
                SEED_21
                + task_position
                * 1_000_000
                + algorithm_position
                * 10_000
                + 4
                * 1_000
                + round_index
                * 10
                + station_position
            )

            local_parameters[
                station
            ] = (
                train_exact21(
                    X=
                        ds[
                            "X"
                        ],

                    y_raw=
                        ds[
                            "y"
                        ],

                    task=
                        task,

                    algorithm=
                        algorithm,

                    initial_parameters=
                        global_parameters,

                    epochs=
                        FEDERATED_LOCAL_EPOCHS,

                    seed=
                        seed,

                    scaler=
                        source_scaler,

                    global_epoch_offset=
                        round_index
                        * int(
                            FEDERATED_LOCAL_EPOCHS
                        ),
                )
            )

        total_samples = sum(
            client_sizes.values()
        )

        avg = sum(
            (
                client_sizes[
                    station
                ]
                / total_samples
            )
            * local_parameters[
                station
            ]
            for station
            in source_stations
        )

        displacement = (
            avg
            - global_parameters
        )

        m = (
            FEDADAM_BETA1
            * m
            + (
                1.0
                - FEDADAM_BETA1
            )
            * displacement
        )

        v = (
            FEDADAM_BETA2
            * v
            + (
                1.0
                - FEDADAM_BETA2
            )
            * (
                displacement
                ** 2
            )
        )

        global_parameters = (
            global_parameters
            + FEDADAM_SERVER_LR
            * m
            / (
                np.sqrt(
                    v
                )
                + FEDADAM_TAU
            )
        )

    return global_parameters


# ==================================================================================================
# 8. COLD-START CONFIG
#
# Frozen structural settings. No LOSO tuning.
# ==================================================================================================

def config21(
    task,
):

    rows = (
        U1619_SELECTION_DF[
            U1619_SELECTION_DF[
                "Task"
            ].eq(
                task
            )
        ]
    )

    return {
        "prior_global":
            float(
                rows[
                    "prior_global"
                ].median()
            ),

        "expert_power":
            float(
                rows[
                    "expert_power"
                ].median()
            ),

        "lookback":
            int(
                round(
                    rows[
                        "lookback"
                    ].median()
                )
            ),

        "decay":
            float(
                rows[
                    "decay"
                ].median()
            ),

        "alpha":
            float(
                rows[
                    "alpha"
                ].median()
            ),

        "local_relation_mix":
            float(
                rows[
                    "local_relation_mix"
                ].median()
            ),
    }


# ==================================================================================================
# 9. RUN ONE LOSO FOLD
# ==================================================================================================

def run_loso_fold_21(
    held_out_station,
):

    fold_rows = []

    source_stations = [
        s
        for s in EXPECTED_STATIONS
        if s != held_out_station
    ]

    for task in TASKS_21:

        algorithm = (
            u1619_selected_algorithm(
                task
            )
        )

        meta = (
            TASK_DATASETS[
                task
            ][
                "Meta"
            ]
        )

        scaler = (
            source_scaler21(
                task,
                source_stations,
            )
        )

        source_local_models = (
            train_source_local_models21(
                task=
                    task,

                held_out_station=
                    held_out_station,

                source_stations=
                    source_stations,

                source_scaler=
                    scaler,
            )
        )

        source_fed_parameters = (
            train_source_fedadam21(
                task=
                    task,

                source_stations=
                    source_stations,

                source_scaler=
                    scaler,
            )
        )

        val_ds = (
            TASK_CLIENT_DATASETS[
                task
            ][held_out_station][
                "Validation"
            ]
        )

        test_ds = (
            TASK_CLIENT_DATASETS[
                task
            ][held_out_station][
                "Test"
            ]
        )

        val_frame = (
            val_ds[
                "Frame"
            ]
            .reset_index(
                drop=True
            )
        )

        test_frame = (
            test_ds[
                "Frame"
            ]
            .reset_index(
                drop=True
            )
        )

        # ------------------------------------------------------------------------------------------
        # Source-only FedAdam baseline
        # ------------------------------------------------------------------------------------------

        fed_test_prediction = (
            np.asarray(
                predict_raw(
                    test_ds[
                        "X"
                    ],
                    source_fed_parameters,
                    task,
                ),
                dtype=float,
            )
        )

        fed_metrics = (
            reg_metrics(
                np.asarray(
                    test_ds[
                        "y"
                    ],
                    dtype=float,
                ),
                fed_test_prediction,
            )
        )

        fold_rows.append({
            "Held_Out_Station":
                held_out_station,

            "Task":
                task,

            "Algorithm_Key":
                algorithm,

            "Model":
                "LOSO FedAdam",

            **fed_metrics,
        })

        # ------------------------------------------------------------------------------------------
        # Global predictions
        # ------------------------------------------------------------------------------------------

        val_global = (
            np.asarray(
                predict_raw(
                    val_ds[
                        "X"
                    ],
                    source_fed_parameters,
                    task,
                ),
                dtype=float,
            )
        )

        test_global = (
            np.asarray(
                predict_raw(
                    test_ds[
                        "X"
                    ],
                    source_fed_parameters,
                    task,
                ),
                dtype=float,
            )
        )

        # ------------------------------------------------------------------------------------------
        # No held-out local model exists in cold-start.
        # Self expert = global source FedAdam prediction.
        # ------------------------------------------------------------------------------------------

        val_self = (
            val_global.copy()
        )

        test_self = (
            test_global.copy()
        )

        # ------------------------------------------------------------------------------------------
        # Relation expert = uniform average of the three source-station local models.
        # ------------------------------------------------------------------------------------------

        val_peer_predictions = []

        test_peer_predictions = []

        for source_station in (
            source_stations
        ):

            source_params = (
                source_local_models[
                    source_station
                ]
            )

            val_peer_predictions.append(
                np.asarray(
                    predict_raw(
                        val_ds[
                            "X"
                        ],
                        source_params,
                        task,
                    ),
                    dtype=float,
                )
            )

            test_peer_predictions.append(
                np.asarray(
                    predict_raw(
                        test_ds[
                            "X"
                        ],
                        source_params,
                        task,
                    ),
                    dtype=float,
                )
            )

        val_relation = (
            np.mean(
                np.vstack(
                    val_peer_predictions
                ),
                axis=0,
            )
        )

        test_relation = (
            np.mean(
                np.vstack(
                    test_peer_predictions
                ),
                axis=0,
            )
        )

        # ------------------------------------------------------------------------------------------
        # Continuous Validation -> Test chronology
        # ------------------------------------------------------------------------------------------

        combined_dates = (
            pd.concat(
                [
                    val_frame[
                        "Date"
                    ],
                    test_frame[
                        "Date"
                    ],
                ],
                ignore_index=True,
            )
        )

        combined_target_dates = (
            pd.concat(
                [
                    val_frame[
                        meta[
                            "Future_Date_Column"
                        ]
                    ],
                    test_frame[
                        meta[
                            "Future_Date_Column"
                        ]
                    ],
                ],
                ignore_index=True,
            )
        )

        combined_y = (
            np.concatenate(
                [
                    np.asarray(
                        val_ds[
                            "y"
                        ],
                        dtype=float,
                    ),

                    np.asarray(
                        test_ds[
                            "y"
                        ],
                        dtype=float,
                    ),
                ]
            )
        )

        combined_global = (
            np.concatenate(
                [
                    val_global,
                    test_global,
                ]
            )
        )

        combined_self = (
            np.concatenate(
                [
                    val_self,
                    test_self,
                ]
            )
        )

        combined_relation = (
            np.concatenate(
                [
                    val_relation,
                    test_relation,
                ]
            )
        )

        cfg = (
            config21(
                task
            )
        )

        combined_prediction, _ = (
            u1619_run_online(
                dates=
                    combined_dates,

                target_dates=
                    combined_target_dates,

                y=
                    combined_y,

                global_pred=
                    combined_global,

                self_pred=
                    combined_self,

                relation_pred=
                    combined_relation,

                prior_global=
                    cfg[
                        "prior_global"
                    ],

                expert_power=
                    cfg[
                        "expert_power"
                    ],

                lookback=
                    cfg[
                        "lookback"
                    ],

                decay=
                    cfg[
                        "decay"
                    ],

                alpha=
                    cfg[
                        "alpha"
                    ],

                local_relation_mix=
                    cfg[
                        "local_relation_mix"
                    ],
            )
        )

        n_validation = len(
            val_ds[
                "y"
            ]
        )

        cold_prediction = (
            np.asarray(
                combined_prediction[
                    n_validation:
                ],
                dtype=float,
            )
        )

        cold_metrics = (
            reg_metrics(
                np.asarray(
                    test_ds[
                        "y"
                    ],
                    dtype=float,
                ),
                cold_prediction,
            )
        )

        fold_rows.append({
            "Held_Out_Station":
                held_out_station,

            "Task":
                task,

            "Algorithm_Key":
                algorithm,

            "Model":
                "LOSO UTW-RG-AdaFedResidual Cold-Start",

            **cold_metrics,
        })

    return pd.DataFrame(
        fold_rows
    )


# ==================================================================================================
# 10. RUN ALL FOUR LOSO FOLDS
# ==================================================================================================

all_fold_results = []

for fold_index, held_out_station in enumerate(
    EXPECTED_STATIONS,
    start=1,
):

    print(
        f"\nLOSO fold {fold_index}/"
        f"{len(EXPECTED_STATIONS)}: "
        f"held out = {held_out_station}"
    )

    fold_df = (
        run_loso_fold_21(
            held_out_station
        )
    )

    all_fold_results.append(
        fold_df
    )


CELL21_LOSO_RESULTS_DF = (
    pd.concat(
        all_fold_results,
        ignore_index=True,
    )
)


# ==================================================================================================
# 11. STATION-LEVEL SUMMARY
# ==================================================================================================

CELL21_STATION_SUMMARY_DF = (
    CELL21_LOSO_RESULTS_DF
    .groupby(
        [
            "Held_Out_Station",
            "Model",
        ],
        as_index=False,
    )
    .agg(
        Mean_MAE=(
            "MAE",
            "mean",
        ),

        Mean_RMSE=(
            "RMSE",
            "mean",
        ),

        Mean_R2=(
            "R2",
            "mean",
        ),

        Min_R2=(
            "R2",
            "min",
        ),

        Positive_R2_Tasks=(
            "R2",
            lambda x:
                int(
                    np.sum(
                        np.asarray(
                            x
                        )
                        > 0.0
                    )
                ),
        ),
    )
)


# ==================================================================================================
# 12. OVERALL SUMMARY
# ==================================================================================================

CELL21_OVERALL_SUMMARY_DF = (
    CELL21_LOSO_RESULTS_DF
    .groupby(
        "Model",
        as_index=False,
    )
    .agg(
        Mean_MAE=(
            "MAE",
            "mean",
        ),

        Mean_RMSE=(
            "RMSE",
            "mean",
        ),

        Mean_R2=(
            "R2",
            "mean",
        ),

        Min_R2=(
            "R2",
            "min",
        ),

        Positive_R2_TaskStation=(
            "R2",
            lambda x:
                int(
                    np.sum(
                        np.asarray(
                            x
                        )
                        > 0.0
                    )
                ),
        ),

        Total_TaskStation=(
            "R2",
            "size",
        ),
    )
)


# ==================================================================================================
# 13. TASK-STATION MAE IMPROVEMENT
# ==================================================================================================

fed_df = (
    CELL21_LOSO_RESULTS_DF[
        CELL21_LOSO_RESULTS_DF[
            "Model"
        ].eq(
            "LOSO FedAdam"
        )
    ][
        [
            "Held_Out_Station",
            "Task",
            "MAE",
        ]
    ]
    .rename(
        columns={
            "MAE":
                "FedAdam_MAE",
        }
    )
)

cold_df = (
    CELL21_LOSO_RESULTS_DF[
        CELL21_LOSO_RESULTS_DF[
            "Model"
        ].eq(
            "LOSO UTW-RG-AdaFedResidual Cold-Start"
        )
    ][
        [
            "Held_Out_Station",
            "Task",
            "MAE",
        ]
    ]
    .rename(
        columns={
            "MAE":
                "ColdStart_MAE",
        }
    )
)


CELL21_IMPROVEMENT_DF = (
    fed_df
    .merge(
        cold_df,
        on=[
            "Held_Out_Station",
            "Task",
        ],
        how="inner",
    )
)

CELL21_IMPROVEMENT_DF[
    "MAE_Improvement_Percent"
] = (
    100.0
    * (
        CELL21_IMPROVEMENT_DF[
            "FedAdam_MAE"
        ]
        - CELL21_IMPROVEMENT_DF[
            "ColdStart_MAE"
        ]
    )
    / CELL21_IMPROVEMENT_DF[
        "FedAdam_MAE"
    ]
)

CELL21_IMPROVEMENT_DF[
    "ColdStart_Better"
] = (
    CELL21_IMPROVEMENT_DF[
        "ColdStart_MAE"
    ]
    <
    CELL21_IMPROVEMENT_DF[
        "FedAdam_MAE"
    ]
)


# ==================================================================================================
# 14. PROTOCOL
# ==================================================================================================

CELL21_PROTOCOL_DF = pd.DataFrame([
    {
        "Item":
            "Evaluation",

        "Value":
            "Leave-One-Station-Out station-transfer generalization",
    },

    {
        "Item":
            "Held-out stations",

        "Value":
            ", ".join(
                EXPECTED_STATIONS
            ),
    },

    {
        "Item":
            "Source stations per fold",

        "Value":
            3,
    },

    {
        "Item":
            "Held-out station used for source training",

        "Value":
            False,
    },

    {
        "Item":
            "Held-out Test target used before prediction",

        "Value":
            False,
    },

    {
        "Item":
            "Source normalization",

        "Value":
            "Source-station Train targets only",
    },

    {
        "Item":
            "Task algorithm",

        "Value":
            "Frozen from main validation pipeline",
    },

    {
        "Item":
            "Hyperparameter tuning inside LOSO",

        "Value":
            False,
    },

    {
        "Item":
            "FedAdam rounds",

        "Value":
            FEDERATED_ROUNDS,
    },

    {
        "Item":
            "FedAdam local epochs",

        "Value":
            FEDERATED_LOCAL_EPOCHS,
    },

    {
        "Item":
            "Cold-start self expert",

        "Value":
            "Source-only FedAdam global expert",
    },

    {
        "Item":
            "Cold-start relation expert",

        "Value":
            "Uniform mean of three source local experts",
    },

    {
        "Item":
            "Final online mechanism",

        "Value":
            "Corrected u1619_run_online",
    },

    {
        "Item":
            "Validation/Test chronology",

        "Value":
            "Continuous",
    },

    {
        "Item":
            "Delayed target feedback",

        "Value":
            True,
    },
])


# ==================================================================================================
# 15. OUTPUT
# ==================================================================================================

print("\nDetailed Task × Held-Out-Station results:")

display(
    CELL21_LOSO_RESULTS_DF
    .round(6)
)


print("\nHeld-out station summary:")

display(
    CELL21_STATION_SUMMARY_DF
    .round(6)
)


print("\nOverall LOSO summary:")

display(
    CELL21_OVERALL_SUMMARY_DF
    .round(6)
)


print("\nCold-start improvement over source-only FedAdam:")

display(
    CELL21_IMPROVEMENT_DF
    .round(6)
)


print("\nProtocol:")

display(
    CELL21_PROTOCOL_DF
)


print("\n" + "=" * 145)
print("CELL 21A COMPLETED SUCCESSFULLY")
print("=" * 145)

print(
    f"Execution time: "
    f"{time.time() - CELL21_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 21B — FEW-SHOT STATION ADAPTATION
# FINAL CORRECTED VERSION
# ==================================================================================================

import time
import numpy as np
import pandas as pd

CELL21B_START = time.time()

print("=" * 145)
print("CELL 21B — FEW-SHOT STATION ADAPTATION")
print("=" * 145)


# ==================================================================================================
# 1. REQUIRED OBJECTS
# ==================================================================================================

required_objects = [
    "CELL21_LOSO_RESULTS_DF",
    "TASK_DEFINITIONS",
    "TASK_DATASETS",
    "TASK_CLIENT_DATASETS",
    "EXPECTED_STATIONS",
    "CENTRAL_LOCAL_EPOCHS",
    "GLOBAL_RANDOM_SEED",
    "u1619_selected_algorithm",
    "u1619_run_online",
    "reg_metrics",
    "predict_raw",
    "source_scaler21",
    "train_source_local_models21",
    "train_source_fedadam21",
    "train_exact21",
    "config21",
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise NameError(
        "Run corrected Cell 21 first. Missing:\n- "
        + "\n- ".join(missing)
    )

print("✓ Corrected Cell 21 objects are available.")


# ==================================================================================================
# 2. FIXED PROTOCOL
# ==================================================================================================

FEWSHOT_LEVELS_21B = [0, 5, 10, 20]

TASKS_21B = [
    d["Task"]
    for d in TASK_DEFINITIONS
]

FEWSHOT_SEED_21B = int(
    GLOBAL_RANDOM_SEED
)

FEWSHOT_EPOCHS_21B = int(
    CENTRAL_LOCAL_EPOCHS
)


# ==================================================================================================
# 3. EARLIEST CHRONOLOGICAL TRAIN SUBSET
# ==================================================================================================

def fewshot_subset21b(
    task,
    station,
    percent,
):

    train_ds = (
        TASK_CLIENT_DATASETS[
            task
        ][station]["Train"]
    )

    frame = (
        train_ds["Frame"]
        .reset_index(drop=True)
        .copy()
    )

    order = (
        pd.to_datetime(
            frame["Date"]
        )
        .sort_values()
        .index
        .to_numpy()
    )

    X = np.asarray(
        train_ds["X"],
        dtype=float,
    )[order]

    y = np.asarray(
        train_ds["y"],
        dtype=float,
    )[order]

    if percent <= 0:

        return {
            "X": X[:0],
            "y": y[:0],
            "N": 0,
            "Total_Train_N": len(y),
        }

    n = max(
        1,
        int(
            np.floor(
                len(y)
                * float(percent)
                / 100.0
            )
        ),
    )

    return {
        "X": X[:n],
        "y": y[:n],
        "N": int(n),
        "Total_Train_N": int(len(y)),
    }


# ==================================================================================================
# 4. FEW-SHOT SELF ADAPTATION
# ==================================================================================================

def adapt_self21b(
    task,
    held_out_station,
    percent,
    source_fed_parameters,
    source_scaler,
):

    algorithm = (
        u1619_selected_algorithm(
            task
        )
    )

    if percent == 0:

        return np.asarray(
            source_fed_parameters,
            dtype=float,
        ).copy()

    fewshot = (
        fewshot_subset21b(
            task,
            held_out_station,
            percent,
        )
    )

    if fewshot["N"] <= 0:

        return np.asarray(
            source_fed_parameters,
            dtype=float,
        ).copy()

    task_position = (
        TASKS_21B.index(task)
        + 1
    )

    station_position = (
        EXPECTED_STATIONS.index(
            held_out_station
        )
        + 1
    )

    adaptation_seed = (
        FEWSHOT_SEED_21B
        + task_position * 10000
        + station_position * 100
    )

    return train_exact21(
        X=
            fewshot["X"],

        y_raw=
            fewshot["y"],

        task=
            task,

        algorithm=
            algorithm,

        initial_parameters=
            np.asarray(
                source_fed_parameters,
                dtype=float,
            ),

        epochs=
            FEWSHOT_EPOCHS_21B,

        seed=
            adaptation_seed,

        scaler=
            source_scaler,

        global_epoch_offset=
            0,
    )


# ==================================================================================================
# 5. RUN ONE CASE
# ==================================================================================================

def run_fewshot_case21b(
    task,
    held_out_station,
    percent,
    source_local_models,
    source_fed_parameters,
    source_scaler,
):

    algorithm = (
        u1619_selected_algorithm(
            task
        )
    )

    meta = (
        TASK_DATASETS[
            task
        ]["Meta"]
    )

    source_stations = [
        s for s in EXPECTED_STATIONS
        if s != held_out_station
    ]

    self_parameters = (
        adapt_self21b(
            task=
                task,

            held_out_station=
                held_out_station,

            percent=
                percent,

            source_fed_parameters=
                source_fed_parameters,

            source_scaler=
                source_scaler,
        )
    )

    val_ds = (
        TASK_CLIENT_DATASETS[
            task
        ][held_out_station]["Validation"]
    )

    test_ds = (
        TASK_CLIENT_DATASETS[
            task
        ][held_out_station]["Test"]
    )

    val_frame = (
        val_ds["Frame"]
        .reset_index(drop=True)
    )

    test_frame = (
        test_ds["Frame"]
        .reset_index(drop=True)
    )

    # Global expert
    val_global = np.asarray(
        predict_raw(
            val_ds["X"],
            source_fed_parameters,
            task,
        ),
        dtype=float,
    )

    test_global = np.asarray(
        predict_raw(
            test_ds["X"],
            source_fed_parameters,
            task,
        ),
        dtype=float,
    )

    # Self expert
    val_self = np.asarray(
        predict_raw(
            val_ds["X"],
            self_parameters,
            task,
        ),
        dtype=float,
    )

    test_self = np.asarray(
        predict_raw(
            test_ds["X"],
            self_parameters,
            task,
        ),
        dtype=float,
    )

    # Relation expert
    val_peer_predictions = []
    test_peer_predictions = []

    for source_station in source_stations:

        source_parameters = (
            source_local_models[
                source_station
            ]
        )

        val_peer_predictions.append(
            np.asarray(
                predict_raw(
                    val_ds["X"],
                    source_parameters,
                    task,
                ),
                dtype=float,
            )
        )

        test_peer_predictions.append(
            np.asarray(
                predict_raw(
                    test_ds["X"],
                    source_parameters,
                    task,
                ),
                dtype=float,
            )
        )

    val_relation = np.mean(
        np.vstack(
            val_peer_predictions
        ),
        axis=0,
    )

    test_relation = np.mean(
        np.vstack(
            test_peer_predictions
        ),
        axis=0,
    )

    combined_dates = pd.concat(
        [
            val_frame["Date"],
            test_frame["Date"],
        ],
        ignore_index=True,
    )

    combined_target_dates = pd.concat(
        [
            val_frame[
                meta["Future_Date_Column"]
            ],
            test_frame[
                meta["Future_Date_Column"]
            ],
        ],
        ignore_index=True,
    )

    combined_y = np.concatenate(
        [
            np.asarray(
                val_ds["y"],
                dtype=float,
            ),
            np.asarray(
                test_ds["y"],
                dtype=float,
            ),
        ]
    )

    combined_global = np.concatenate(
        [
            val_global,
            test_global,
        ]
    )

    combined_self = np.concatenate(
        [
            val_self,
            test_self,
        ]
    )

    combined_relation = np.concatenate(
        [
            val_relation,
            test_relation,
        ]
    )

    cfg = config21(
        task
    )

    combined_prediction, _ = (
        u1619_run_online(
            dates=
                combined_dates,

            target_dates=
                combined_target_dates,

            y=
                combined_y,

            global_pred=
                combined_global,

            self_pred=
                combined_self,

            relation_pred=
                combined_relation,

            prior_global=
                cfg["prior_global"],

            expert_power=
                cfg["expert_power"],

            lookback=
                cfg["lookback"],

            decay=
                cfg["decay"],

            alpha=
                cfg["alpha"],

            local_relation_mix=
                cfg["local_relation_mix"],
        )
    )

    n_validation = len(
        val_ds["y"]
    )

    test_prediction = np.asarray(
        combined_prediction[
            n_validation:
        ],
        dtype=float,
    )

    actual = np.asarray(
        test_ds["y"],
        dtype=float,
    )

    metrics = reg_metrics(
        actual,
        test_prediction,
    )

    fewshot = (
        fewshot_subset21b(
            task,
            held_out_station,
            percent,
        )
    )

    return {
        "Held_Out_Station":
            held_out_station,

        "Task":
            task,

        "Algorithm_Key":
            algorithm,

        "FewShot_Percent":
            int(percent),

        "FewShot_Rows":
            int(
                fewshot["N"]
            ),

        "HeldOut_Train_Rows":
            int(
                fewshot[
                    "Total_Train_N"
                ]
            ),

        **metrics,
    }


# ==================================================================================================
# 6. FULL EXPERIMENT
# ==================================================================================================

result_rows = []

for fold_index, held_out_station in enumerate(
    EXPECTED_STATIONS,
    start=1,
):

    print(
        f"\nFew-shot fold "
        f"{fold_index}/"
        f"{len(EXPECTED_STATIONS)}"
        f" | Held-out station = "
        f"{held_out_station}"
    )

    source_stations = [
        s for s in EXPECTED_STATIONS
        if s != held_out_station
    ]

    for task in TASKS_21B:

        source_scaler = (
            source_scaler21(
                task,
                source_stations,
            )
        )

        source_local_models = (
            train_source_local_models21(
                task=
                    task,

                held_out_station=
                    held_out_station,

                source_stations=
                    source_stations,

                source_scaler=
                    source_scaler,
            )
        )

        source_fed_parameters = (
            train_source_fedadam21(
                task=
                    task,

                source_stations=
                    source_stations,

                source_scaler=
                    source_scaler,
            )
        )

        for percent in FEWSHOT_LEVELS_21B:

            result_rows.append(
                run_fewshot_case21b(
                    task=
                        task,

                    held_out_station=
                        held_out_station,

                    percent=
                        percent,

                    source_local_models=
                        source_local_models,

                    source_fed_parameters=
                        source_fed_parameters,

                    source_scaler=
                        source_scaler,
                )
            )


CELL21B_RESULTS_DF = pd.DataFrame(
    result_rows
)


# ==================================================================================================
# 7. ZERO-SHOT PARITY CONTROL
# ==================================================================================================

zero_current = (
    CELL21B_RESULTS_DF[
        CELL21B_RESULTS_DF[
            "FewShot_Percent"
        ].eq(0)
    ][
        [
            "Held_Out_Station",
            "Task",
            "MAE",
            "RMSE",
            "R2",
        ]
    ]
    .copy()
)

zero_reference = (
    CELL21_LOSO_RESULTS_DF[
        CELL21_LOSO_RESULTS_DF[
            "Model"
        ].eq(
            "LOSO UTW-RG-AdaFedResidual Cold-Start"
        )
    ][
        [
            "Held_Out_Station",
            "Task",
            "MAE",
            "RMSE",
            "R2",
        ]
    ]
    .copy()
)


CELL21B_ZERO_PARITY_DF = (
    zero_reference
    .merge(
        zero_current,
        on=[
            "Held_Out_Station",
            "Task",
        ],
        suffixes=(
            "_Cell21",
            "_Cell21B",
        ),
        how="inner",
    )
)


CELL21B_ZERO_PARITY_DF[
    "Max_Difference"
] = (
    CELL21B_ZERO_PARITY_DF.apply(
        lambda row: max(
            abs(
                row["MAE_Cell21"]
                - row["MAE_Cell21B"]
            ),
            abs(
                row["RMSE_Cell21"]
                - row["RMSE_Cell21B"]
            ),
            abs(
                row["R2_Cell21"]
                - row["R2_Cell21B"]
            ),
        ),
        axis=1,
    )
)


CELL21B_ZERO_PARITY_DF[
    "Reproduced"
] = (
    CELL21B_ZERO_PARITY_DF[
        "Max_Difference"
    ] < 1e-10
)


if len(
    CELL21B_ZERO_PARITY_DF
) != 24:

    raise RuntimeError(
        "Zero-shot parity control must contain 24 cases."
    )


if not (
    CELL21B_ZERO_PARITY_DF[
        "Reproduced"
    ].all()
):

    display(
        CELL21B_ZERO_PARITY_DF
    )

    raise RuntimeError(
        "CELL 21B ZERO-SHOT PARITY CONTROL FAILED."
    )


print(
    "✓ 0% exactly reproduces Cell 21 cold-start results."
)


# ==================================================================================================
# 8. STATION SUMMARY
# ==================================================================================================

CELL21B_STATION_SUMMARY_DF = (
    CELL21B_RESULTS_DF
    .groupby(
        [
            "Held_Out_Station",
            "FewShot_Percent",
        ],
        as_index=False,
    )
    .agg(
        Tasks=(
            "Task",
            "nunique",
        ),

        Mean_MAE=(
            "MAE",
            "mean",
        ),

        Mean_RMSE=(
            "RMSE",
            "mean",
        ),

        Mean_R2=(
            "R2",
            "mean",
        ),

        Min_R2=(
            "R2",
            "min",
        ),

        Positive_R2_Tasks=(
            "R2",
            lambda x:
                int(
                    np.sum(
                        np.asarray(x)
                        > 0.0
                    )
                ),
        ),
    )
)


# ==================================================================================================
# 9. OVERALL SUMMARY
# ==================================================================================================

CELL21B_OVERALL_DF = (
    CELL21B_RESULTS_DF
    .groupby(
        "FewShot_Percent",
        as_index=False,
    )
    .agg(
        TaskStation_Cases=(
            "Task",
            "size",
        ),

        Mean_MAE=(
            "MAE",
            "mean",
        ),

        Mean_RMSE=(
            "RMSE",
            "mean",
        ),

        Mean_R2=(
            "R2",
            "mean",
        ),

        Min_R2=(
            "R2",
            "min",
        ),

        Positive_R2_Cases=(
            "R2",
            lambda x:
                int(
                    np.sum(
                        np.asarray(x)
                        > 0.0
                    )
                ),
        ),
    )
)


# ==================================================================================================
# 10. IMPROVEMENT VS ZERO-SHOT
# ==================================================================================================

zero_overall = (
    CELL21B_OVERALL_DF[
        CELL21B_OVERALL_DF[
            "FewShot_Percent"
        ].eq(0)
    ]
    .iloc[0]
)


CELL21B_OVERALL_DF[
    "MAE_Improvement_vs_Zero_Percent"
] = (
    100.0
    * (
        float(
            zero_overall["Mean_MAE"]
        )
        - CELL21B_OVERALL_DF[
            "Mean_MAE"
        ]
    )
    / float(
        zero_overall["Mean_MAE"]
    )
)


CELL21B_OVERALL_DF[
    "RMSE_Improvement_vs_Zero_Percent"
] = (
    100.0
    * (
        float(
            zero_overall["Mean_RMSE"]
        )
        - CELL21B_OVERALL_DF[
            "Mean_RMSE"
        ]
    )
    / float(
        zero_overall["Mean_RMSE"]
    )
)


CELL21B_OVERALL_DF[
    "R2_Change_vs_Zero"
] = (
    CELL21B_OVERALL_DF[
        "Mean_R2"
    ]
    - float(
        zero_overall["Mean_R2"]
    )
)


# ==================================================================================================
# 11. STATION IMPROVEMENT
# ==================================================================================================

station_zero = (
    CELL21B_STATION_SUMMARY_DF[
        CELL21B_STATION_SUMMARY_DF[
            "FewShot_Percent"
        ].eq(0)
    ][
        [
            "Held_Out_Station",
            "Mean_MAE",
            "Mean_RMSE",
            "Mean_R2",
        ]
    ]
    .rename(
        columns={
            "Mean_MAE":
                "Zero_MAE",

            "Mean_RMSE":
                "Zero_RMSE",

            "Mean_R2":
                "Zero_R2",
        }
    )
)


CELL21B_STATION_SUMMARY_DF = (
    CELL21B_STATION_SUMMARY_DF
    .merge(
        station_zero,
        on="Held_Out_Station",
        how="left",
    )
)


CELL21B_STATION_SUMMARY_DF[
    "MAE_Improvement_vs_Zero_Percent"
] = (
    100.0
    * (
        CELL21B_STATION_SUMMARY_DF[
            "Zero_MAE"
        ]
        - CELL21B_STATION_SUMMARY_DF[
            "Mean_MAE"
        ]
    )
    / CELL21B_STATION_SUMMARY_DF[
        "Zero_MAE"
    ]
)


CELL21B_STATION_SUMMARY_DF[
    "R2_Change_vs_Zero"
] = (
    CELL21B_STATION_SUMMARY_DF[
        "Mean_R2"
    ]
    - CELL21B_STATION_SUMMARY_DF[
        "Zero_R2"
    ]
)


# ==================================================================================================
# 12. PAPER DISPLAY — 0 / 10 / 20%
# ==================================================================================================

CELL21B_PAPER_DISPLAY_DF = (
    CELL21B_OVERALL_DF[
        CELL21B_OVERALL_DF[
            "FewShot_Percent"
        ].isin(
            [
                0,
                10,
                20,
            ]
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# ==================================================================================================
# 13. PROTOCOL
# ==================================================================================================

CELL21B_PROTOCOL_DF = pd.DataFrame([
    {
        "Item":
            "Test",

        "Value":
            "Few-shot station adaptation after frozen-model LOSO",
    },

    {
        "Item":
            "Few-shot levels",

        "Value":
            "0% / 5% / 10% / 20%",
    },

    {
        "Item":
            "Paper display levels",

        "Value":
            "0% / 10% / 20%",
    },

    {
        "Item":
            "Adaptation data",

        "Value":
            "Held-out station TRAIN only",
    },

    {
        "Item":
            "Row selection",

        "Value":
            "Earliest chronological rows",
    },

    {
        "Item":
            "Target Validation used",

        "Value":
            False,
    },

    {
        "Item":
            "Target Test used",

        "Value":
            False,
    },

    {
        "Item":
            "Source stations",

        "Value":
            "Other three stations only",
    },

    {
        "Item":
            "Source normalization",

        "Value":
            "Source Train only",
    },

    {
        "Item":
            "Adaptation initialization",

        "Value":
            "Source-only FedAdam",
    },

    {
        "Item":
            "Algorithm/config",

        "Value":
            "Frozen",
    },

    {
        "Item":
            "Hyperparameter tuning",

        "Value":
            False,
    },

    {
        "Item":
            "Adaptation epochs",

        "Value":
            FEWSHOT_EPOCHS_21B,
    },

    {
        "Item":
            "Seed",

        "Value":
            FEWSHOT_SEED_21B,
    },

    {
        "Item":
            "Relation experts",

        "Value":
            "Three source local models",
    },

    {
        "Item":
            "Final online mechanism",

        "Value":
            "Corrected u1619_run_online",
    },

    {
        "Item":
            "0% parity with Cell 21",

        "Value":
            True,
    },
])


# ==================================================================================================
# 14. OUTPUT
# ==================================================================================================

print("\nZero-shot parity control:")

display(
    CELL21B_ZERO_PARITY_DF
    .round(12)
)


print("\nStation-level summary:")

display(
    CELL21B_STATION_SUMMARY_DF
    .round(6)
)


print("\nOverall few-shot summary:")

display(
    CELL21B_OVERALL_DF
    .round(6)
)


print("\nPaper display — 0% / 10% / 20%:")

display(
    CELL21B_PAPER_DISPLAY_DF
    .round(6)
)


print("\nProtocol:")

display(
    CELL21B_PROTOCOL_DF
)


print("\n" + "=" * 145)
print("CELL 21B COMPLETED SUCCESSFULLY")
print("=" * 145)

print(
    f"Execution time: "
    f"{time.time() - CELL21B_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 22A — AUSTIN EXTERNAL ZERO-SHOT GENERALIZATION
# ==================================================================================================
# Purpose
# -------
# External evaluation of the fully unseen Austin USGS Barton Springs station.
#
# IMPORTANT SCIENTIFIC RULE:
# Austin contains water-quality variables and future turbidity observations,
# but it does NOT contain Alum-dose or Chlorine-dose ground truth.
#
# Therefore this cell:
#   1) Loads ONLY the prepared Austin external dataset.
#   2) Performs a strict external-data integrity / chronology / W10 audit.
#   3) Verifies that no Austin data are used for training, tuning, or normalization fitting.
#   4) Characterizes the external temporal forecasting environment using observed future turbidity.
#   5) DOES NOT falsely compute Alum/Chlorine MAE/RMSE/R² when those targets do not exist.
#
# No retraining.
# No fine-tuning.
# No adaptation.
# No hyperparameter selection.
# No Austin-derived normalization.
# ==================================================================================================

import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

# --------------------------------------------------------------------------------------------------
# 1. FILE RESOLUTION
# --------------------------------------------------------------------------------------------------

AUSTIN_FILE_CANDIDATES = [
    "/content/Austin_USGS_Final_Preprocessed_W10_Input المعالج.xlsx",
    "/content/Austin_USGS_Final_Preprocessed_W10_Input.xlsx",
    "/mnt/data/Austin_USGS_Final_Preprocessed_W10_Input المعالج.xlsx",
    "/mnt/data/Austin_USGS_Final_Preprocessed_W10_Input.xlsx",
]

AUSTIN_FILE = None

for candidate in AUSTIN_FILE_CANDIDATES:
    if os.path.exists(candidate):
        AUSTIN_FILE = candidate
        break

if AUSTIN_FILE is None:
    raise FileNotFoundError(
        "Austin prepared dataset was not found.\n"
        "Expected one of:\n" +
        "\n".join(AUSTIN_FILE_CANDIDATES)
    )

print("=" * 120)
print("CELL 22A — AUSTIN EXTERNAL ZERO-SHOT GENERALIZATION")
print("=" * 120)
print(f"\nAustin file:\n{AUSTIN_FILE}")


# --------------------------------------------------------------------------------------------------
# 2. LOAD PREPARED EXTERNAL DATA
# --------------------------------------------------------------------------------------------------

excel_file = pd.ExcelFile(AUSTIN_FILE)

preferred_sheets = [
    "Model Input W10",
    "W10 Ready",
    "Model Ready Strict",
]

AUSTIN_SHEET = None

for sheet_name in preferred_sheets:
    if sheet_name in excel_file.sheet_names:
        AUSTIN_SHEET = sheet_name
        break

if AUSTIN_SHEET is None:
    raise ValueError(
        "No recognized Austin model-input sheet was found.\n"
        f"Available sheets: {excel_file.sheet_names}"
    )

AUSTIN22A_DF = pd.read_excel(
    AUSTIN_FILE,
    sheet_name=AUSTIN_SHEET,
)

AUSTIN22A_DF.columns = [
    str(c).strip()
    for c in AUSTIN22A_DF.columns
]

print(f"\nSelected sheet: {AUSTIN_SHEET}")
print(f"Loaded rows   : {len(AUSTIN22A_DF):,}")
print(f"Loaded columns: {len(AUSTIN22A_DF.columns):,}")


# --------------------------------------------------------------------------------------------------
# 3. COLUMN RESOLUTION
# --------------------------------------------------------------------------------------------------

def ext22a_find_column(frame, candidates, required=True):
    lower_map = {
        str(c).strip().lower(): c
        for c in frame.columns
    }

    for candidate in candidates:
        key = candidate.strip().lower()
        if key in lower_map:
            return lower_map[key]

    if required:
        raise KeyError(
            f"Required column not found.\n"
            f"Candidates: {candidates}\n"
            f"Available columns:\n{list(frame.columns)}"
        )

    return None


COL_DATE = ext22a_find_column(
    AUSTIN22A_DF,
    ["date", "Date"],
)

COL_STATION = ext22a_find_column(
    AUSTIN22A_DF,
    ["station", "Station"],
    required=False,
)

COL_SPLIT = ext22a_find_column(
    AUSTIN22A_DF,
    ["split", "Split"],
    required=False,
)

COL_TURB = ext22a_find_column(
    AUSTIN22A_DF,
    [
        "turbidity_fnu",
        "turbidity_fnu_source",
        "turbidity_ntu",
    ],
)

COL_PH = ext22a_find_column(
    AUSTIN22A_DF,
    ["ph", "pH"],
)

COL_TEMP = ext22a_find_column(
    AUSTIN22A_DF,
    [
        "temperature_c",
        "Water_Temperature_C",
        "temp_clean_c",
    ],
)

COL_ALL_TARGETS = ext22a_find_column(
    AUSTIN22A_DF,
    [
        "all_future_labels_observed",
        "All_Future_Labels_Available",
    ],
    required=False,
)


# --------------------------------------------------------------------------------------------------
# 4. BASIC CLEANING — NO IMPUTATION HERE
# --------------------------------------------------------------------------------------------------

AUSTIN22A_DF = AUSTIN22A_DF.copy()

AUSTIN22A_DF[COL_DATE] = pd.to_datetime(
    AUSTIN22A_DF[COL_DATE],
    errors="coerce",
)

if AUSTIN22A_DF[COL_DATE].isna().any():
    raise ValueError(
        "Austin contains invalid/unparseable dates."
    )

AUSTIN22A_DF = (
    AUSTIN22A_DF
    .sort_values(COL_DATE)
    .reset_index(drop=True)
)

if AUSTIN22A_DF[COL_DATE].duplicated().any():
    duplicated_dates = (
        AUSTIN22A_DF.loc[
            AUSTIN22A_DF[COL_DATE].duplicated(),
            COL_DATE
        ]
        .astype(str)
        .tolist()
    )

    raise ValueError(
        "Duplicate Austin dates detected:\n"
        + "\n".join(duplicated_dates[:20])
    )


# --------------------------------------------------------------------------------------------------
# 5. EXTERNAL-PROTOCOL AUDIT
# --------------------------------------------------------------------------------------------------

date_diff = (
    AUSTIN22A_DF[COL_DATE]
    .diff()
    .dt.days
)

n_consecutive_pairs = int(
    (date_diff.iloc[1:] == 1).sum()
)

n_nonconsecutive_pairs = int(
    (date_diff.iloc[1:] != 1).sum()
)

if COL_ALL_TARGETS is not None:
    target_mask = (
        AUSTIN22A_DF[COL_ALL_TARGETS]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )
else:
    target_mask = pd.Series(
        True,
        index=AUSTIN22A_DF.index,
    )

AUSTIN22A_EVAL_DF = (
    AUSTIN22A_DF.loc[target_mask]
    .copy()
    .reset_index(drop=True)
)

required_predictors = [
    COL_TURB,
    COL_PH,
    COL_TEMP,
]

predictor_complete_mask = (
    AUSTIN22A_EVAL_DF[
        required_predictors
    ]
    .notna()
    .all(axis=1)
)

AUSTIN22A_EVAL_DF = (
    AUSTIN22A_EVAL_DF.loc[
        predictor_complete_mask
    ]
    .copy()
    .reset_index(drop=True)
)

if len(AUSTIN22A_EVAL_DF) == 0:
    raise ValueError(
        "No Austin rows remain after the external-evaluation audit."
    )


# --------------------------------------------------------------------------------------------------
# 6. DETECT AVAILABLE FUTURE TURBIDITY TARGETS
# --------------------------------------------------------------------------------------------------

HORIZON_CANDIDATES = {
    1: [
        "forecast_h1_fnu",
        "forecast_turbidity_h1_fnu",
        "forecast_h1_ntu",
    ],
    3: [
        "forecast_h3_fnu",
        "forecast_turbidity_h3_fnu",
        "forecast_h3_ntu",
    ],
    6: [
        "forecast_h6_fnu",
        "forecast_turbidity_h6_fnu",
        "forecast_h6_ntu",
    ],
    7: [
        "forecast_h7_fnu",
        "forecast_turbidity_h7_fnu",
        "forecast_h7_ntu",
    ],
    10: [
        "forecast_h10_fnu",
        "forecast_turbidity_h10_fnu",
        "forecast_h10_ntu",
    ],
}

AUSTIN22A_HORIZON_COLUMNS = {}

for horizon, candidates in HORIZON_CANDIDATES.items():
    resolved = ext22a_find_column(
        AUSTIN22A_EVAL_DF,
        candidates,
        required=False,
    )

    if resolved is not None:
        AUSTIN22A_HORIZON_COLUMNS[
            horizon
        ] = resolved


# --------------------------------------------------------------------------------------------------
# 7. EXTERNAL TEMPORAL BENCHMARK
# --------------------------------------------------------------------------------------------------
# This is NOT the Paper-3 Alum/Chlorine model score.
#
# It only describes how difficult the Austin future-turbidity temporal environment is,
# using a persistence forecast:
#
#       y_hat(t+h) = turbidity(t)
#
# This gives a clean external temporal reference without retraining anything.
# --------------------------------------------------------------------------------------------------

benchmark_rows = []

for horizon, target_col in sorted(
    AUSTIN22A_HORIZON_COLUMNS.items()
):

    temp = AUSTIN22A_EVAL_DF[
        [
            COL_DATE,
            COL_TURB,
            target_col,
        ]
    ].copy()

    temp = temp.dropna().reset_index(
        drop=True
    )

    if len(temp) == 0:
        continue

    y_true = (
        temp[target_col]
        .astype(float)
        .to_numpy()
    )

    y_persistence = (
        temp[COL_TURB]
        .astype(float)
        .to_numpy()
    )

    mae = mean_absolute_error(
        y_true,
        y_persistence,
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_persistence,
        )
    )

    if (
        len(y_true) >= 2
        and np.var(y_true) > 0
    ):
        r2 = r2_score(
            y_true,
            y_persistence,
        )
    else:
        r2 = np.nan

    benchmark_rows.append({
        "Station":
            "Austin_USGS_Barton_Springs",
        "External_Test_Type":
            "Temporal environment benchmark",
        "Variable":
            "Turbidity",
        "Unit":
            "FNU",
        "Horizon_Days":
            horizon,
        "N":
            int(len(y_true)),
        "Persistence_MAE":
            float(mae),
        "Persistence_RMSE":
            float(rmse),
        "Persistence_R2":
            float(r2)
            if np.isfinite(r2)
            else np.nan,
    })


AUSTIN22A_TEMPORAL_BENCHMARK_DF = pd.DataFrame(
    benchmark_rows
)


# --------------------------------------------------------------------------------------------------
# 8. FINAL-MODEL COMPATIBILITY GATE
# --------------------------------------------------------------------------------------------------
# The accepted Paper-3 final model predicts:
#
#       Alum_H1 / Alum_H2 / Alum_H3
#       Chlorine_H1 / Chlorine_H2 / Chlorine_H3
#
# Austin does NOT provide those operational ground-truth dose targets.
#
# Therefore supervised scoring of the frozen dose model on Austin would be scientifically invalid.
# --------------------------------------------------------------------------------------------------

dose_target_candidates = [
    "alum_dose",
    "alum_percent",
    "alum_mg_l",
    "chlorine_dose",
    "chlorine_percent",
    "chlorine_mg_l",
]

present_dose_targets = [
    c
    for c in dose_target_candidates
    if c in AUSTIN22A_DF.columns
    and AUSTIN22A_DF[c].notna().any()
]

AUSTIN22A_HAS_OPERATIONAL_GROUND_TRUTH = (
    len(present_dose_targets) > 0
)

AUSTIN22A_EXACT_DOSE_MODEL_SCORABLE = False

# We intentionally keep this False unless all six required Paper-3 dose outcomes
# are actually available and semantically compatible.
#
# Austin is a monitoring station, not a DWTP operational dose log.


# --------------------------------------------------------------------------------------------------
# 9. DATASET CHARACTERIZATION
# --------------------------------------------------------------------------------------------------

def ext22a_safe_numeric_summary(
    frame,
    column,
):
    values = pd.to_numeric(
        frame[column],
        errors="coerce",
    ).dropna()

    if len(values) == 0:
        return {
            "N": 0,
            "Mean": np.nan,
            "Std": np.nan,
            "Min": np.nan,
            "Median": np.nan,
            "Max": np.nan,
        }

    return {
        "N":
            int(len(values)),
        "Mean":
            float(values.mean()),
        "Std":
            float(values.std(ddof=1))
            if len(values) > 1
            else 0.0,
        "Min":
            float(values.min()),
        "Median":
            float(values.median()),
        "Max":
            float(values.max()),
    }


characterization_rows = []

for variable_name, column_name, unit in [
    (
        "Turbidity",
        COL_TURB,
        "FNU",
    ),
    (
        "pH",
        COL_PH,
        "pH",
    ),
    (
        "Water Temperature",
        COL_TEMP,
        "°C",
    ),
]:

    summary = ext22a_safe_numeric_summary(
        AUSTIN22A_EVAL_DF,
        column_name,
    )

    characterization_rows.append({
        "Station":
            "Austin_USGS_Barton_Springs",
        "Variable":
            variable_name,
        "Unit":
            unit,
        **summary,
    })


AUSTIN22A_CHARACTERIZATION_DF = pd.DataFrame(
    characterization_rows
)


# --------------------------------------------------------------------------------------------------
# 10. PROTOCOL DECLARATIONS
# --------------------------------------------------------------------------------------------------

AUSTIN22A_PROTOCOL_DF = pd.DataFrame([
    {
        "Protocol_Item":
            "External station",
        "Confirmed":
            True,
        "Details":
            "Austin USGS Barton Springs",
    },
    {
        "Protocol_Item":
            "Austin used for training",
        "Confirmed":
            False,
        "Details":
            "No",
    },
    {
        "Protocol_Item":
            "Austin used for validation/model selection",
        "Confirmed":
            False,
        "Details":
            "No",
    },
    {
        "Protocol_Item":
            "Austin used for hyperparameter tuning",
        "Confirmed":
            False,
        "Details":
            "No",
    },
    {
        "Protocol_Item":
            "Austin used to fit normalization",
        "Confirmed":
            False,
        "Details":
            "No",
    },
    {
        "Protocol_Item":
            "External preprocessing already completed",
        "Confirmed":
            True,
        "Details":
            "W=10 prepared dataset",
    },
    {
        "Protocol_Item":
            "Future targets interpolated",
        "Confirmed":
            False,
        "Details":
            "Observed future turbidity only",
    },
    {
        "Protocol_Item":
            "Alum/Chlorine ground truth available",
        "Confirmed":
            AUSTIN22A_HAS_OPERATIONAL_GROUND_TRUTH,
        "Details":
            (
                str(present_dose_targets)
                if present_dose_targets
                else "No operational dose ground truth"
            ),
    },
    {
        "Protocol_Item":
            "Exact Paper-3 dose-model supervised scoring allowed",
        "Confirmed":
            AUSTIN22A_EXACT_DOSE_MODEL_SCORABLE,
        "Details":
            (
                "Not allowed because Austin lacks compatible "
                "Alum/Chlorine ground truth."
            ),
    },
])


# --------------------------------------------------------------------------------------------------
# 11. FINAL STATUS
# --------------------------------------------------------------------------------------------------

AUSTIN22A_STATUS = {
    "Station":
        "Austin_USGS_Barton_Springs",

    "Dataset_Type":
        "Real-world external monitoring station",

    "Evaluation_Mode":
        "External zero-shot applicability / temporal generalization audit",

    "Rows_Loaded":
        int(len(AUSTIN22A_DF)),

    "Rows_After_Audit":
        int(len(AUSTIN22A_EVAL_DF)),

    "Start_Date":
        AUSTIN22A_EVAL_DF[
            COL_DATE
        ].min(),

    "End_Date":
        AUSTIN22A_EVAL_DF[
            COL_DATE
        ].max(),

    "Available_Turbidity_Horizons":
        sorted(
            AUSTIN22A_HORIZON_COLUMNS.keys()
        ),

    "Operational_Dose_Ground_Truth":
        AUSTIN22A_HAS_OPERATIONAL_GROUND_TRUTH,

    "Exact_Final_Dose_Model_Scorable":
        AUSTIN22A_EXACT_DOSE_MODEL_SCORABLE,

    "Training_Performed":
        False,

    "Tuning_Performed":
        False,

    "Adaptation_Performed":
        False,

    "External_Normalization_Fitted":
        False,
}


# --------------------------------------------------------------------------------------------------
# 12. OUTPUT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("AUSTIN 22A — EXTERNAL DATA AUDIT")
print("=" * 120)

print(
    f"\nRows loaded                : "
    f"{len(AUSTIN22A_DF):,}"
)

print(
    f"Rows after evaluation audit: "
    f"{len(AUSTIN22A_EVAL_DF):,}"
)

print(
    f"Date range                 : "
    f"{AUSTIN22A_EVAL_DF[COL_DATE].min().date()} "
    f"to "
    f"{AUSTIN22A_EVAL_DF[COL_DATE].max().date()}"
)

print(
    f"Consecutive date pairs     : "
    f"{n_consecutive_pairs:,}"
)

print(
    f"Non-consecutive pairs      : "
    f"{n_nonconsecutive_pairs:,}"
)

print(
    "\nAvailable future turbidity horizons:",
    sorted(
        AUSTIN22A_HORIZON_COLUMNS.keys()
    ),
)

print("\nObserved-variable characterization:")
display(
    AUSTIN22A_CHARACTERIZATION_DF
    .round(6)
)

if len(
    AUSTIN22A_TEMPORAL_BENCHMARK_DF
) > 0:

    print(
        "\nAustin temporal persistence benchmark "
        "(diagnostic only — NOT Paper-3 dose-model performance):"
    )

    display(
        AUSTIN22A_TEMPORAL_BENCHMARK_DF
        .round(6)
    )


print("\nProtocol audit:")
display(
    AUSTIN22A_PROTOCOL_DF
)


print("\n" + "=" * 120)
print("CELL 22A — FINAL INTERPRETATION")
print("=" * 120)

print(
    "\nAustin is a valid real-world external station "
    "for evaluating external temporal/covariate behavior."
)

print(
    "Austin was NOT used for training, validation, tuning, "
    "adaptation, or normalization fitting."
)

print(
    "\nHowever, Austin does NOT contain the operational "
    "Alum/Chlorine dose ground truth required to calculate "
    "supervised MAE/RMSE/R² for the frozen Paper-3 "
    "dose-prediction tasks."
)

print(
    "\nTherefore Cell 22A intentionally does NOT fabricate "
    "a Paper-3 dose-model accuracy score."
)

print(
    "Its role is retained as a real-world external "
    "temporal/covariate generalization and applicability test."
)

print("\nCELL 22A STATUS: COMPLETED")
print("=" * 120)

In [ ]:
# ==================================================================================================
# CELL 22B — TONGJI EXTERNAL ZERO-SHOT GENERALIZATION
# ==================================================================================================

import os
import glob
import time
import numpy as np
import pandas as pd

CELL22B_START = time.time()

print("=" * 120)
print("CELL 22B — TONGJI EXTERNAL ZERO-SHOT GENERALIZATION")
print("=" * 120)


# ==================================================================================================
# 1. FIND FILE
# ==================================================================================================

tongji_candidates = [
    p for p in glob.glob("/content/*.xlsx")
    if "tongji" in os.path.basename(p).lower()
    and (
        "generalization" in os.path.basename(p).lower()
        or "ready" in os.path.basename(p).lower()
        or "المعالج" in os.path.basename(p)
    )
]

if not tongji_candidates:
    raise FileNotFoundError(
        "Tongji external-generalization workbook was not found in /content."
    )

tongji_candidates = sorted(
    tongji_candidates,
    key=lambda p: (
        "المعالج" not in os.path.basename(p),
        "generalization_ready" not in os.path.basename(p).lower(),
        os.path.basename(p),
    )
)

TONGJI22B_FILE = tongji_candidates[0]

print("\nTongji file:")
print(TONGJI22B_FILE)


# ==================================================================================================
# 2. LOAD MODEL INPUT
# ==================================================================================================

xls = pd.ExcelFile(
    TONGJI22B_FILE
)

preferred_sheet = "Model Input W10"

if preferred_sheet not in xls.sheet_names:
    raise KeyError(
        f"Sheet '{preferred_sheet}' not found. "
        f"Available sheets: {xls.sheet_names}"
    )

TONGJI22B_DF = pd.read_excel(
    TONGJI22B_FILE,
    sheet_name=preferred_sheet,
)

print("\nSelected sheet:", preferred_sheet)
print("Loaded rows   :", len(TONGJI22B_DF))
print("Loaded columns:", len(TONGJI22B_DF.columns))


# ==================================================================================================
# 3. REQUIRED COLUMNS
# ==================================================================================================

required_columns = [
    "station",
    "date",
    "split",
    "raw_turbidity",
    "pH",
    "temperature",
    "target_turbidity_h1",
    "target_turbidity_h2",
    "target_turbidity_h3",
    "W10_Eligible",
    "operational_ground_truth_available",
]

missing = [
    c for c in required_columns
    if c not in TONGJI22B_DF.columns
]

if missing:
    raise KeyError(
        "Tongji workbook is missing required columns:\n- "
        + "\n- ".join(missing)
    )


# ==================================================================================================
# 4. CLEAN / AUDIT
# ==================================================================================================

df = TONGJI22B_DF.copy()

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce",
)

numeric_columns = [
    "raw_turbidity",
    "pH",
    "temperature",
    "target_turbidity_h1",
    "target_turbidity_h2",
    "target_turbidity_h3",
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce",
    )

eligible = (
    df["W10_Eligible"].astype(bool)
    &
    df[
        [
            "raw_turbidity",
            "pH",
            "temperature",
            "target_turbidity_h1",
            "target_turbidity_h2",
            "target_turbidity_h3",
        ]
    ].notna().all(axis=1)
)

TONGJI22B_EVAL_DF = (
    df.loc[eligible]
    .sort_values("date")
    .reset_index(drop=True)
)

if len(TONGJI22B_EVAL_DF) == 0:
    raise RuntimeError(
        "No valid Tongji W10 external evaluation rows."
    )


# ==================================================================================================
# 5. METRIC FUNCTION
# ==================================================================================================

def external22b_metrics(
    y_true,
    y_pred,
):

    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    mask = (
        np.isfinite(y_true)
        &
        np.isfinite(y_pred)
    )

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    mae = float(
        np.mean(
            np.abs(
                y_true - y_pred
            )
        )
    )

    rmse = float(
        np.sqrt(
            np.mean(
                (y_true - y_pred) ** 2
            )
        )
    )

    denominator = float(
        np.sum(
            (
                y_true
                - np.mean(y_true)
            ) ** 2
        )
    )

    r2 = (
        np.nan
        if denominator <= 0.0
        else
        1.0
        - float(
            np.sum(
                (y_true - y_pred) ** 2
            )
        )
        / denominator
    )

    return {
        "N": len(y_true),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }


# ==================================================================================================
# 6. EXTERNAL TEMPORAL PERSISTENCE BENCHMARK
# Diagnostic environmental benchmark only
# ==================================================================================================

benchmark_rows = []

for horizon in [1, 2, 3]:

    target_col = (
        f"target_turbidity_h{horizon}"
    )

    metrics = external22b_metrics(
        TONGJI22B_EVAL_DF[
            target_col
        ],
        TONGJI22B_EVAL_DF[
            "raw_turbidity"
        ],
    )

    benchmark_rows.append({
        "Station":
            "Tongji_DWTP",

        "External_Test_Type":
            "Temporal environment benchmark",

        "Variable":
            "Raw Turbidity",

        "Horizon_Days":
            horizon,

        "Unit":
            "Source dataset unit",

        "Persistence_MAE":
            metrics["MAE"],

        "Persistence_RMSE":
            metrics["RMSE"],

        "Persistence_R2":
            metrics["R2"],

        "N":
            metrics["N"],
    })


TONGJI22B_TEMPORAL_BENCHMARK_DF = pd.DataFrame(
    benchmark_rows
)


# ==================================================================================================
# 7. OBSERVED VARIABLE CHARACTERIZATION
# ==================================================================================================

characterization_rows = []

for col, label in [
    ("raw_turbidity", "Raw Turbidity"),
    ("pH", "pH"),
    ("temperature", "Water Temperature"),
]:

    values = (
        TONGJI22B_EVAL_DF[
            col
        ]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    characterization_rows.append({
        "Station":
            "Tongji_DWTP",

        "Variable":
            label,

        "N":
            len(values),

        "Mean":
            float(
                np.mean(values)
            ),

        "Std":
            float(
                np.std(values)
            ),

        "Min":
            float(
                np.min(values)
            ),

        "Median":
            float(
                np.median(values)
            ),

        "Max":
            float(
                np.max(values)
            ),
    })


TONGJI22B_CHARACTERIZATION_DF = pd.DataFrame(
    characterization_rows
)


# ==================================================================================================
# 8. AUDIT
# ==================================================================================================

dates = (
    TONGJI22B_EVAL_DF[
        "date"
    ]
    .dropna()
    .sort_values()
)

date_diffs = (
    dates.diff()
    .dropna()
    .dt.days
)

TONGJI22B_AUDIT_DF = pd.DataFrame([
    {
        "Item": "Rows loaded",
        "Value": len(TONGJI22B_DF),
    },
    {
        "Item": "Valid W10 evaluation rows",
        "Value": len(TONGJI22B_EVAL_DF),
    },
    {
        "Item": "First date",
        "Value": dates.min(),
    },
    {
        "Item": "Last date",
        "Value": dates.max(),
    },
    {
        "Item": "Consecutive date pairs",
        "Value": int((date_diffs == 1).sum()),
    },
    {
        "Item": "Non-consecutive date pairs",
        "Value": int((date_diffs != 1).sum()),
    },
    {
        "Item": "External training performed",
        "Value": False,
    },
    {
        "Item": "External tuning performed",
        "Value": False,
    },
    {
        "Item": "External normalization fitted",
        "Value": False,
    },
    {
        "Item": "Alum/Chlorine ground truth available",
        "Value": False,
    },
])


# ==================================================================================================
# 9. PROTOCOL
# ==================================================================================================

TONGJI22B_PROTOCOL_DF = pd.DataFrame([
    {
        "Protocol_Item":
            "External station",

        "Confirmed":
            True,

        "Details":
            "Tongji DWTP",
    },
    {
        "Protocol_Item":
            "External dataset used for training",

        "Confirmed":
            False,

        "Details":
            "No",
    },
    {
        "Protocol_Item":
            "External dataset used for validation/tuning",

        "Confirmed":
            False,

        "Details":
            "No",
    },
    {
        "Protocol_Item":
            "Normalization fitted on external dataset",

        "Confirmed":
            False,

        "Details":
            "No",
    },
    {
        "Protocol_Item":
            "UTW",

        "Confirmed":
            True,

        "Details":
            "W=10",
    },
    {
        "Protocol_Item":
            "Observed external future labels",

        "Confirmed":
            True,

        "Details":
            "Raw turbidity H1/H2/H3",
    },
    {
        "Protocol_Item":
            "Alum/Chlorine dose labels",

        "Confirmed":
            False,

        "Details":
            "Not available",
    },
])


# ==================================================================================================
# 10. OUTPUT
# ==================================================================================================

print("\n" + "=" * 120)
print("TONGJI 22B — EXTERNAL DATA AUDIT")
print("=" * 120)

display(
    TONGJI22B_AUDIT_DF
)

print("\nObserved-variable characterization:")

display(
    TONGJI22B_CHARACTERIZATION_DF
    .round(6)
)

print(
    "\nTongji temporal persistence benchmark "
    "(diagnostic environmental benchmark):"
)

display(
    TONGJI22B_TEMPORAL_BENCHMARK_DF
    .round(6)
)

print("\nProtocol audit:")

display(
    TONGJI22B_PROTOCOL_DF
)

print("\n" + "=" * 120)
print("CELL 22B — FINAL INTERPRETATION")
print("=" * 120)

print(
    "Tongji is retained as a real-world external DWTP "
    "temporal/covariate generalization dataset."
)

print(
    "Tongji was not used for training, validation, tuning, "
    "adaptation, or normalization fitting."
)

print(
    "Operational Alum/Chlorine dose ground truth is not present; "
    "therefore no fabricated Paper-3 dose accuracy score is reported."
)

print("\nCELL 22B STATUS: COMPLETED")
print("=" * 120)

print(
    f"Execution time: "
    f"{time.time() - CELL22B_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 22C — HANNAN EXTERNAL ZERO-SHOT GENERALIZATION
# ==================================================================================================

import os
import glob
import time
import numpy as np
import pandas as pd

CELL22C_START = time.time()

print("=" * 120)
print("CELL 22C — HANNAN EXTERNAL ZERO-SHOT GENERALIZATION")
print("=" * 120)


# ==================================================================================================
# 1. FIND FILE
# ==================================================================================================

hannan_candidates = [
    p for p in glob.glob("/content/*.xlsx")
    if "hannan" in os.path.basename(p).lower()
    and (
        "generalization" in os.path.basename(p).lower()
        or "ready" in os.path.basename(p).lower()
        or "المعالج" in os.path.basename(p)
    )
]

if not hannan_candidates:
    raise FileNotFoundError(
        "Hannan external-generalization workbook was not found in /content."
    )

hannan_candidates = sorted(
    hannan_candidates,
    key=lambda p: (
        "المعالج" not in os.path.basename(p),
        "generalization_ready" not in os.path.basename(p).lower(),
        os.path.basename(p),
    )
)

HANNAN22C_FILE = hannan_candidates[0]

print("\nHannan file:")
print(HANNAN22C_FILE)


# ==================================================================================================
# 2. LOAD MODEL INPUT
# ==================================================================================================

xls = pd.ExcelFile(
    HANNAN22C_FILE
)

preferred_sheet = "Model Input W10"

if preferred_sheet not in xls.sheet_names:
    raise KeyError(
        f"Sheet '{preferred_sheet}' not found. "
        f"Available sheets: {xls.sheet_names}"
    )

HANNAN22C_DF = pd.read_excel(
    HANNAN22C_FILE,
    sheet_name=preferred_sheet,
)

print("\nSelected sheet:", preferred_sheet)
print("Loaded rows   :", len(HANNAN22C_DF))
print("Loaded columns:", len(HANNAN22C_DF.columns))


# ==================================================================================================
# 3. REQUIRED COLUMNS
# ==================================================================================================

required_columns = [
    "Station",
    "Date",
    "Turbidity_Observed",
    "Turbidity_Input",
    "Turbidity_Imputed",
    "pH",
    "Temperature",
    "Forecast_Turbidity_H1",
    "Forecast_Turbidity_H2",
    "Forecast_Turbidity_H3",
    "All_Targets_Observed",
    "Split",
]

missing = [
    c for c in required_columns
    if c not in HANNAN22C_DF.columns
]

if missing:
    raise KeyError(
        "Hannan workbook is missing required columns:\n- "
        + "\n- ".join(missing)
    )


# ==================================================================================================
# 4. CLEAN / EVALUATION POPULATION
# ==================================================================================================

df = HANNAN22C_DF.copy()

df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce",
)

numeric_columns = [
    "Turbidity_Observed",
    "Turbidity_Input",
    "pH",
    "Temperature",
    "Forecast_Turbidity_H1",
    "Forecast_Turbidity_H2",
    "Forecast_Turbidity_H3",
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce",
    )

eligible = (
    df["All_Targets_Observed"].astype(bool)
    &
    df[
        [
            "Turbidity_Input",
            "pH",
            "Temperature",
            "Forecast_Turbidity_H1",
            "Forecast_Turbidity_H2",
            "Forecast_Turbidity_H3",
        ]
    ].notna().all(axis=1)
)

HANNAN22C_EVAL_DF = (
    df.loc[eligible]
    .sort_values("Date")
    .reset_index(drop=True)
)

if len(HANNAN22C_EVAL_DF) == 0:
    raise RuntimeError(
        "No valid Hannan external W10 evaluation rows."
    )


# ==================================================================================================
# 5. METRICS
# ==================================================================================================

def external22c_metrics(
    y_true,
    y_pred,
):

    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    valid = (
        np.isfinite(y_true)
        &
        np.isfinite(y_pred)
    )

    y_true = y_true[valid]
    y_pred = y_pred[valid]

    mae = float(
        np.mean(
            np.abs(
                y_true - y_pred
            )
        )
    )

    rmse = float(
        np.sqrt(
            np.mean(
                (y_true - y_pred) ** 2
            )
        )
    )

    denominator = float(
        np.sum(
            (
                y_true
                - np.mean(y_true)
            ) ** 2
        )
    )

    r2 = (
        np.nan
        if denominator <= 0
        else
        1.0
        - float(
            np.sum(
                (y_true - y_pred) ** 2
            )
        )
        / denominator
    )

    return {
        "N": len(y_true),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }


# ==================================================================================================
# 6. OBSERVED-CURRENT PERSISTENCE BENCHMARK
# Only rows with genuinely observed current turbidity
# ==================================================================================================

observed_current = (
    HANNAN22C_EVAL_DF[
        "Turbidity_Observed"
    ].notna()
)

HANNAN22C_PERSISTENCE_DF = (
    HANNAN22C_EVAL_DF.loc[
        observed_current
    ]
    .copy()
    .reset_index(drop=True)
)

benchmark_rows = []

for horizon in [1, 2, 3]:

    target_col = (
        f"Forecast_Turbidity_H{horizon}"
    )

    metrics = external22c_metrics(
        HANNAN22C_PERSISTENCE_DF[
            target_col
        ],
        HANNAN22C_PERSISTENCE_DF[
            "Turbidity_Observed"
        ],
    )

    benchmark_rows.append({
        "Station":
            "Hannan_Village_Monitoring_Station",

        "External_Test_Type":
            "Observed temporal environment benchmark",

        "Variable":
            "Turbidity",

        "Horizon_Days":
            horizon,

        "N":
            metrics["N"],

        "Persistence_MAE":
            metrics["MAE"],

        "Persistence_RMSE":
            metrics["RMSE"],

        "Persistence_R2":
            metrics["R2"],
    })


HANNAN22C_TEMPORAL_BENCHMARK_DF = pd.DataFrame(
    benchmark_rows
)


# ==================================================================================================
# 7. CHARACTERIZATION
# ==================================================================================================

characterization_rows = []

for col, label in [
    ("Turbidity_Input", "Turbidity"),
    ("pH", "pH"),
    ("Temperature", "Water Temperature"),
]:

    values = (
        HANNAN22C_EVAL_DF[
            col
        ]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    characterization_rows.append({
        "Station":
            "Hannan_Village_Monitoring_Station",

        "Variable":
            label,

        "N":
            len(values),

        "Mean":
            float(
                np.mean(values)
            ),

        "Std":
            float(
                np.std(values)
            ),

        "Min":
            float(
                np.min(values)
            ),

        "Median":
            float(
                np.median(values)
            ),

        "Max":
            float(
                np.max(values)
            ),
    })


HANNAN22C_CHARACTERIZATION_DF = pd.DataFrame(
    characterization_rows
)


# ==================================================================================================
# 8. AUDIT
# ==================================================================================================

dates = (
    HANNAN22C_EVAL_DF[
        "Date"
    ]
    .dropna()
    .sort_values()
)

date_diffs = (
    dates.diff()
    .dropna()
    .dt.days
)

HANNAN22C_AUDIT_DF = pd.DataFrame([
    {
        "Item":
            "Rows loaded",

        "Value":
            len(HANNAN22C_DF),
    },
    {
        "Item":
            "Rows with all observed future H1/H2/H3 labels",

        "Value":
            len(HANNAN22C_EVAL_DF),
    },
    {
        "Item":
            "Rows usable for observed-current persistence",

        "Value":
            len(HANNAN22C_PERSISTENCE_DF),
    },
    {
        "Item":
            "First date",

        "Value":
            dates.min(),
    },
    {
        "Item":
            "Last date",

        "Value":
            dates.max(),
    },
    {
        "Item":
            "Current input rows marked imputed",

        "Value":
            int(
                HANNAN22C_EVAL_DF[
                    "Turbidity_Imputed"
                ].fillna(False).astype(bool).sum()
            ),
    },
    {
        "Item":
            "External training performed",

        "Value":
            False,
    },
    {
        "Item":
            "External tuning performed",

        "Value":
            False,
    },
    {
        "Item":
            "External normalization fitted",

        "Value":
            False,
    },
])


# ==================================================================================================
# 9. PROTOCOL
# ==================================================================================================

HANNAN22C_PROTOCOL_DF = pd.DataFrame([
    {
        "Protocol_Item":
            "External station",

        "Confirmed":
            True,

        "Details":
            "Hannan Village Monitoring Station",
    },
    {
        "Protocol_Item":
            "External data used for training",

        "Confirmed":
            False,

        "Details":
            "No",
    },
    {
        "Protocol_Item":
            "External data used for tuning",

        "Confirmed":
            False,

        "Details":
            "No",
    },
    {
        "Protocol_Item":
            "Input-only short-gap interpolation",

        "Confirmed":
            True,

        "Details":
            "Internal turbidity gaps <=3 days only",
    },
    {
        "Protocol_Item":
            "Future target interpolation",

        "Confirmed":
            False,

        "Details":
            "Never",
    },
    {
        "Protocol_Item":
            "Normalization fitted on Hannan",

        "Confirmed":
            False,

        "Details":
            "No",
    },
    {
        "Protocol_Item":
            "UTW",

        "Confirmed":
            True,

        "Details":
            "W=10",
    },
    {
        "Protocol_Item":
            "Alum/Chlorine operational labels",

        "Confirmed":
            False,

        "Details":
            "Not available",
    },
])


# ==================================================================================================
# 10. OUTPUT
# ==================================================================================================

print("\n" + "=" * 120)
print("HANNAN 22C — EXTERNAL DATA AUDIT")
print("=" * 120)

display(
    HANNAN22C_AUDIT_DF
)

print("\nObserved-variable characterization:")

display(
    HANNAN22C_CHARACTERIZATION_DF
    .round(6)
)

print(
    "\nHannan observed temporal persistence benchmark:"
)

display(
    HANNAN22C_TEMPORAL_BENCHMARK_DF
    .round(6)
)

print("\nProtocol audit:")

display(
    HANNAN22C_PROTOCOL_DF
)

print("\n" + "=" * 120)
print("CELL 22C — FINAL INTERPRETATION")
print("=" * 120)

print(
    "Hannan is retained as an independent real-world external "
    "water-quality generalization dataset."
)

print(
    "No external rows were used for model training, validation, "
    "hyperparameter tuning, or scaler fitting."
)

print(
    "Future turbidity targets remain observed-only; "
    "interpolated values are never used as future ground truth."
)

print("\nCELL 22C STATUS: COMPLETED")
print("=" * 120)

print(
    f"Execution time: "
    f"{time.time() - CELL22C_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 22D — KIRKUK EXTERNAL SUPERVISED OPERATIONAL GENERALIZATION AUDIT
# ==================================================================================================

import os
import glob
import time
import numpy as np
import pandas as pd

CELL22D_START = time.time()

print("=" * 125)
print("CELL 22D — KIRKUK EXTERNAL SUPERVISED OPERATIONAL GENERALIZATION")
print("=" * 125)


# ==================================================================================================
# 1. FIND FILE
# ==================================================================================================

kirkuk_candidates = [
    p for p in glob.glob("/content/*.xlsx")
    if "kirkuk" in os.path.basename(p).lower()
    and (
        "generalization" in os.path.basename(p).lower()
        or "ready" in os.path.basename(p).lower()
        or "المعالج" in os.path.basename(p)
    )
]

if not kirkuk_candidates:
    raise FileNotFoundError(
        "Kirkuk external-generalization workbook was not found in /content."
    )

kirkuk_candidates = sorted(
    kirkuk_candidates,
    key=lambda p: (
        "المعالج" not in os.path.basename(p),
        "generalization_ready" not in os.path.basename(p).lower(),
        os.path.basename(p),
    )
)

KIRKUK22D_FILE = kirkuk_candidates[0]

print("\nKirkuk file:")
print(KIRKUK22D_FILE)


# ==================================================================================================
# 2. LOAD SHEETS
# ==================================================================================================

xls = pd.ExcelFile(
    KIRKUK22D_FILE
)

required_sheets = [
    "Model Input W10",
    "Daily Processed",
]

missing_sheets = [
    s for s in required_sheets
    if s not in xls.sheet_names
]

if missing_sheets:
    raise KeyError(
        "Kirkuk workbook missing sheets:\n- "
        + "\n- ".join(missing_sheets)
    )

KIRKUK22D_MODEL_DF = pd.read_excel(
    KIRKUK22D_FILE,
    sheet_name="Model Input W10",
)

KIRKUK22D_DAILY_DF = pd.read_excel(
    KIRKUK22D_FILE,
    sheet_name="Daily Processed",
)

print("\nModel Input W10 rows :", len(KIRKUK22D_MODEL_DF))
print("Daily Processed rows :", len(KIRKUK22D_DAILY_DF))


# ==================================================================================================
# 3. REQUIRED MODEL COLUMNS
# ==================================================================================================

target_columns = [
    "Alum_H1_mgL",
    "Alum_H2_mgL",
    "Alum_H3_mgL",
    "Chlorine_H1_mgL",
    "Chlorine_H2_mgL",
    "Chlorine_H3_mgL",
]

required_model_columns = [
    "Station",
    "Date",
    "Split",
    "Raw_Turbidity_Lag0",
    "Raw_pH_Lag0",
    "Temperature_C_Lag0",
    "All_Targets_Observed",
] + target_columns

missing = [
    c for c in required_model_columns
    if c not in KIRKUK22D_MODEL_DF.columns
]

if missing:
    raise KeyError(
        "Kirkuk Model Input W10 missing columns:\n- "
        + "\n- ".join(missing)
    )


required_daily_columns = [
    "Date",
    "Alum_Dose_mgL",
    "Chlorine_Dose_mgL",
    "Flow_m3_hour",
]

missing_daily = [
    c for c in required_daily_columns
    if c not in KIRKUK22D_DAILY_DF.columns
]

if missing_daily:
    raise KeyError(
        "Kirkuk Daily Processed missing columns:\n- "
        + "\n- ".join(missing_daily)
    )


# ==================================================================================================
# 4. CLEAN
# ==================================================================================================

model_df = (
    KIRKUK22D_MODEL_DF
    .copy()
)

daily_df = (
    KIRKUK22D_DAILY_DF
    .copy()
)

model_df["Date"] = pd.to_datetime(
    model_df["Date"],
    errors="coerce",
)

daily_df["Date"] = pd.to_datetime(
    daily_df["Date"],
    errors="coerce",
)

for col in target_columns:
    model_df[col] = pd.to_numeric(
        model_df[col],
        errors="coerce",
    )

for col in [
    "Alum_Dose_mgL",
    "Chlorine_Dose_mgL",
    "Flow_m3_hour",
]:
    daily_df[col] = pd.to_numeric(
        daily_df[col],
        errors="coerce",
    )


# ==================================================================================================
# 5. FULLY SUPERVISED W10 POPULATION
# ==================================================================================================

eligible = (
    model_df[
        "All_Targets_Observed"
    ].astype(bool)
    &
    model_df[
        target_columns
    ].notna().all(axis=1)
)

KIRKUK22D_EVAL_DF = (
    model_df.loc[
        eligible
    ]
    .sort_values(
        "Date"
    )
    .reset_index(
        drop=True
    )
)

if len(KIRKUK22D_EVAL_DF) == 0:
    raise RuntimeError(
        "No fully supervised Kirkuk W10 rows were found."
    )


# ==================================================================================================
# 6. ADD CURRENT OBSERVED OPERATIONAL DOSES
# ==================================================================================================

current_doses = (
    daily_df[
        [
            "Date",
            "Alum_Dose_mgL",
            "Chlorine_Dose_mgL",
            "Flow_m3_hour",
        ]
    ]
    .drop_duplicates(
        subset=[
            "Date"
        ]
    )
)

KIRKUK22D_EVAL_DF = (
    KIRKUK22D_EVAL_DF
    .merge(
        current_doses,
        on="Date",
        how="left",
    )
)


# ==================================================================================================
# 7. METRICS
# ==================================================================================================

def external22d_metrics(
    y_true,
    y_pred,
):

    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    valid = (
        np.isfinite(y_true)
        &
        np.isfinite(y_pred)
    )

    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        return {
            "N": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "R2": np.nan,
            "MBE": np.nan,
        }

    residual = (
        y_pred - y_true
    )

    mae = float(
        np.mean(
            np.abs(
                residual
            )
        )
    )

    rmse = float(
        np.sqrt(
            np.mean(
                residual ** 2
            )
        )
    )

    mbe = float(
        np.mean(
            residual
        )
    )

    denominator = float(
        np.sum(
            (
                y_true
                - np.mean(y_true)
            ) ** 2
        )
    )

    r2 = (
        np.nan
        if denominator <= 0
        else
        1.0
        - float(
            np.sum(
                residual ** 2
            )
        )
        / denominator
    )

    return {
        "N": len(y_true),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MBE": mbe,
    }


# ==================================================================================================
# 8. EXTERNAL OPERATIONAL PERSISTENCE BENCHMARK
# Current observed dose y(t) -> future observed dose y(t+h)
# ==================================================================================================

benchmark_rows = []

task_specs = [
    (
        "Alum_H1",
        "Alum_Dose_mgL",
        "Alum_H1_mgL",
        1,
    ),
    (
        "Alum_H2",
        "Alum_Dose_mgL",
        "Alum_H2_mgL",
        2,
    ),
    (
        "Alum_H3",
        "Alum_Dose_mgL",
        "Alum_H3_mgL",
        3,
    ),
    (
        "Chlorine_H1",
        "Chlorine_Dose_mgL",
        "Chlorine_H1_mgL",
        1,
    ),
    (
        "Chlorine_H2",
        "Chlorine_Dose_mgL",
        "Chlorine_H2_mgL",
        2,
    ),
    (
        "Chlorine_H3",
        "Chlorine_Dose_mgL",
        "Chlorine_H3_mgL",
        3,
    ),
]

for (
    task,
    current_col,
    future_col,
    horizon,
) in task_specs:

    metrics = external22d_metrics(
        KIRKUK22D_EVAL_DF[
            future_col
        ],
        KIRKUK22D_EVAL_DF[
            current_col
        ],
    )

    benchmark_rows.append({
        "Station":
            "Kirkuk_WTP",

        "Task":
            task,

        "Horizon":
            horizon,

        "Baseline":
            "Operational persistence",

        "N":
            metrics[
                "N"
            ],

        "MAE":
            metrics[
                "MAE"
            ],

        "RMSE":
            metrics[
                "RMSE"
            ],

        "R2":
            metrics[
                "R2"
            ],

        "MBE":
            metrics[
                "MBE"
            ],
    })


KIRKUK22D_PERSISTENCE_DF = pd.DataFrame(
    benchmark_rows
)


# ==================================================================================================
# 9. TARGET CHARACTERIZATION
# ==================================================================================================

characterization_rows = []

for col, label in [
    (
        "Alum_Dose_mgL",
        "Current Alum Dose",
    ),
    (
        "Chlorine_Dose_mgL",
        "Current Chlorine Dose",
    ),
    (
        "Flow_m3_hour",
        "Plant Flow",
    ),
]:

    values = (
        KIRKUK22D_EVAL_DF[
            col
        ]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    characterization_rows.append({
        "Variable":
            label,

        "N":
            len(values),

        "Mean":
            float(
                np.mean(values)
            ) if len(values) else np.nan,

        "Std":
            float(
                np.std(values)
            ) if len(values) else np.nan,

        "Min":
            float(
                np.min(values)
            ) if len(values) else np.nan,

        "Median":
            float(
                np.median(values)
            ) if len(values) else np.nan,

        "Max":
            float(
                np.max(values)
            ) if len(values) else np.nan,
    })


KIRKUK22D_CHARACTERIZATION_DF = pd.DataFrame(
    characterization_rows
)


# ==================================================================================================
# 10. EXTERNAL DATA AUDIT
# ==================================================================================================

dates = (
    KIRKUK22D_EVAL_DF[
        "Date"
    ]
    .dropna()
    .sort_values()
)

KIRKUK22D_AUDIT_DF = pd.DataFrame([
    {
        "Item":
            "Daily processed rows",

        "Value":
            len(
                KIRKUK22D_DAILY_DF
            ),
    },
    {
        "Item":
            "Model Input W10 rows",

        "Value":
            len(
                KIRKUK22D_MODEL_DF
            ),
    },
    {
        "Item":
            "Fully supervised W10 rows",

        "Value":
            len(
                KIRKUK22D_EVAL_DF
            ),
    },
    {
        "Item":
            "First unified analytical date",

        "Value":
            dates.min(),
    },
    {
        "Item":
            "Last unified analytical date",

        "Value":
            dates.max(),
    },
    {
        "Item":
            "Six operational H1/H2/H3 targets available",

        "Value":
            bool(
                KIRKUK22D_EVAL_DF[
                    target_columns
                ].notna().all().all()
            ),
    },
    {
        "Item":
            "External model fitting performed",

        "Value":
            False,
    },
    {
        "Item":
            "External hyperparameter tuning performed",

        "Value":
            False,
    },
    {
        "Item":
            "Normalization fitted on Kirkuk",

        "Value":
            False,
    },
])


# ==================================================================================================
# 11. PROTOCOL
# ==================================================================================================

KIRKUK22D_PROTOCOL_DF = pd.DataFrame([
    {
        "Protocol_Item":
            "External site",

        "Confirmed":
            True,

        "Details":
            "Kirkuk WTP",
    },
    {
        "Protocol_Item":
            "Unified analytical chronology",

        "Confirmed":
            True,

        "Details":
            "121 consecutive analytical daily positions",
    },
    {
        "Protocol_Item":
            "Original source date conflicts retained in audit",

        "Confirmed":
            True,

        "Details":
            "Yes",
    },
    {
        "Protocol_Item":
            "UTW",

        "Confirmed":
            True,

        "Details":
            "W=10",
    },
    {
        "Protocol_Item":
            "Alum dose ground truth",

        "Confirmed":
            True,

        "Details":
            "Observed mass dose converted using observed plant flow",
    },
    {
        "Protocol_Item":
            "Chlorine dose ground truth",

        "Confirmed":
            True,

        "Details":
            "Observed mass dose converted using observed plant flow",
    },
    {
        "Protocol_Item":
            "Target interpolation",

        "Confirmed":
            False,

        "Details":
            "None",
    },
    {
        "Protocol_Item":
            "External training",

        "Confirmed":
            False,

        "Details":
            "None",
    },
    {
        "Protocol_Item":
            "External tuning",

        "Confirmed":
            False,

        "Details":
            "None",
    },
    {
        "Protocol_Item":
            "External normalization fitting",

        "Confirmed":
            False,

        "Details":
            "None",
    },
    {
        "Protocol_Item":
            "Supervised operational benchmark",

        "Confirmed":
            True,

        "Details":
            "Alum/Chlorine H1/H2/H3 ground truth available",
    },
])


# ==================================================================================================
# 12. OUTPUT
# ==================================================================================================

print("\n" + "=" * 125)
print("KIRKUK 22D — EXTERNAL SUPERVISED DATA AUDIT")
print("=" * 125)

display(
    KIRKUK22D_AUDIT_DF
)

print("\nOperational-variable characterization:")

display(
    KIRKUK22D_CHARACTERIZATION_DF
    .round(6)
)

print("\nExternal operational persistence benchmark:")

display(
    KIRKUK22D_PERSISTENCE_DF
    .round(6)
)

print("\nProtocol audit:")

display(
    KIRKUK22D_PROTOCOL_DF
)

print("\n" + "=" * 125)
print("CELL 22D — FINAL INTERPRETATION")
print("=" * 125)

print(
    "Kirkuk provides independent external operational "
    "Alum and Chlorine H1/H2/H3 ground truth."
)

print(
    "The dataset was not used for training, validation, "
    "hyperparameter tuning, adaptation, or scaler fitting."
)

print(
    "Operational persistence results are retained as the "
    "external supervised reference benchmark."
)

print("\nCELL 22D STATUS: COMPLETED")
print("=" * 125)

print(
    f"Execution time: "
    f"{time.time() - CELL22D_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 22E — DIGITAL TWIN TRANSFER TEST
# FINAL CORRECTED VERSION — REAL HEADER = EXCEL ROW 3
# ==================================================================================================

import os
import glob
import time
import numpy as np
import pandas as pd

CELL22E_START = time.time()

print("=" * 125)
print("CELL 22E — DIGITAL TWIN TRANSFER TEST")
print("=" * 125)


# ==================================================================================================
# 1. FIND DIGITAL TWIN FILE
# ==================================================================================================

digital_candidates = [
    p for p in glob.glob("/content/*.xlsx")
    if "virtual_dwtp_c" in os.path.basename(p).lower()
]

if not digital_candidates:
    raise FileNotFoundError(
        "Virtual_DWTP_C workbook was not found in /content."
    )

digital_candidates = sorted(
    digital_candidates,
    key=lambda p: (
        "(2)" not in os.path.basename(p),
        os.path.basename(p),
    )
)

DIGITAL22E_FILE = digital_candidates[0]

print("\nDigital Twin file:")
print(DIGITAL22E_FILE)


# ==================================================================================================
# 2. LOAD CORRECT SHEET WITH CORRECT HEADER
# ==================================================================================================

xls = pd.ExcelFile(
    DIGITAL22E_FILE
)

print("\nAvailable sheets:")
print(xls.sheet_names)

if "Daily_Model_Data" not in xls.sheet_names:
    raise KeyError(
        "Sheet 'Daily_Model_Data' was not found."
    )

DIGITAL22E_RAW_DF = pd.read_excel(
    DIGITAL22E_FILE,
    sheet_name="Daily_Model_Data",
    header=2,
)

DIGITAL22E_RAW_DF.columns = [
    str(c).strip()
    for c in DIGITAL22E_RAW_DF.columns
]

print("\nSelected sheet: Daily_Model_Data")
print("Loaded rows   :", len(DIGITAL22E_RAW_DF))
print("Loaded columns:", len(DIGITAL22E_RAW_DF.columns))
print("\nColumns:")
print(list(DIGITAL22E_RAW_DF.columns))


# ==================================================================================================
# 3. FLEXIBLE COLUMN RESOLUTION
# ==================================================================================================

def find_col22e(df, candidates):

    lookup = {
        str(c).strip().lower().replace("_", " "):
            c
        for c in df.columns
    }

    for candidate in candidates:

        key = (
            str(candidate)
            .strip()
            .lower()
            .replace("_", " ")
        )

        if key in lookup:
            return lookup[key]

    return None


date_col = find_col22e(
    DIGITAL22E_RAW_DF,
    [
        "Date",
        "date",
        "timestamp",
    ],
)

station_col = find_col22e(
    DIGITAL22E_RAW_DF,
    [
        "Station",
        "station",
    ],
)

temp_col = find_col22e(
    DIGITAL22E_RAW_DF,
    [
        "(°C)_raw",
        "Temperature",
        "temperature",
        "Water Temperature",
        "temperature_c",
    ],
)

turb_col = find_col22e(
    DIGITAL22E_RAW_DF,
    [
        "Turb(NTU)_raw",
        "Turbidity",
        "raw_turbidity",
        "Raw Turbidity",
    ],
)

ph_col = find_col22e(
    DIGITAL22E_RAW_DF,
    [
        "pH_raw",
        "pH",
        "ph",
    ],
)


resolved = {
    "Date": date_col,
    "Station": station_col,
    "Temperature": temp_col,
    "Turbidity": turb_col,
    "pH": ph_col,
}

missing = [
    k for k, v in resolved.items()
    if v is None and k != "Station"
]

if missing:
    raise KeyError(
        "Could not resolve required Digital Twin columns:\n- "
        + "\n- ".join(missing)
        + "\n\nAvailable columns:\n"
        + "\n".join(
            map(
                str,
                DIGITAL22E_RAW_DF.columns,
            )
        )
    )


print("\nResolved columns:")
for k, v in resolved.items():
    print(f"{k:12s}: {v}")


# ==================================================================================================
# 4. CLEAN DAILY SERIES
# ==================================================================================================

df = DIGITAL22E_RAW_DF.copy()

df["_Date"] = pd.to_datetime(
    df[date_col],
    errors="coerce",
)

df["_Turbidity"] = pd.to_numeric(
    df[turb_col],
    errors="coerce",
)

df["_pH"] = pd.to_numeric(
    df[ph_col],
    errors="coerce",
)

df["_Temperature"] = pd.to_numeric(
    df[temp_col],
    errors="coerce",
)

if station_col is not None:
    df["_Station"] = (
        df[station_col]
        .astype(str)
        .str.strip()
    )
else:
    df["_Station"] = "Virtual_DWTP_C"


df = (
    df[
        [
            "_Date",
            "_Station",
            "_Turbidity",
            "_pH",
            "_Temperature",
        ]
    ]
    .dropna(
        subset=[
            "_Date",
        ]
    )
    .sort_values(
        "_Date"
    )
    .drop_duplicates(
        subset=[
            "_Date",
        ],
        keep="first",
    )
    .reset_index(
        drop=True
    )
)

if len(df) == 0:
    raise RuntimeError(
        "Digital Twin contains no valid dated rows."
    )


# ==================================================================================================
# 5. STRICT W10 ELIGIBILITY
# ==================================================================================================

input_complete = (
    df[
        [
            "_Turbidity",
            "_pH",
            "_Temperature",
        ]
    ]
    .notna()
    .all(axis=1)
)

w10_eligible = np.zeros(
    len(df),
    dtype=bool,
)

for i in range(
    9,
    len(df),
):

    window = df.iloc[
        i - 9:
        i + 1
    ]

    complete = bool(
        input_complete.iloc[
            i - 9:
            i + 1
        ].all()
    )

    consecutive = bool(
        (
            window["_Date"]
            .diff()
            .dropna()
            .dt.days
            == 1
        ).all()
    )

    w10_eligible[i] = (
        complete
        and consecutive
    )

df["_W10_Eligible"] = (
    w10_eligible
)


# ==================================================================================================
# 6. FUTURE H1 / H2 / H3 TURBIDITY TARGETS
# ==================================================================================================

date_to_turbidity = dict(
    zip(
        df["_Date"],
        df["_Turbidity"],
    )
)

for horizon in [
    1,
    2,
    3,
]:

    df[
        f"_Target_H{horizon}"
    ] = [
        date_to_turbidity.get(
            date
            + pd.Timedelta(
                days=horizon
            ),
            np.nan,
        )
        for date in df["_Date"]
    ]


# ==================================================================================================
# 7. FINAL EVALUATION POPULATION
# ==================================================================================================

DIGITAL22E_EVAL_DF = (
    df[
        df["_W10_Eligible"]
        &
        df[
            [
                "_Target_H1",
                "_Target_H2",
                "_Target_H3",
            ]
        ]
        .notna()
        .all(axis=1)
    ]
    .copy()
    .reset_index(drop=True)
)

if len(DIGITAL22E_EVAL_DF) == 0:
    raise RuntimeError(
        "No valid Digital Twin W10 + H1/H2/H3 rows."
    )


# ==================================================================================================
# 8. METRICS
# ==================================================================================================

def metrics22e(
    y_true,
    y_pred,
):

    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    valid = (
        np.isfinite(y_true)
        &
        np.isfinite(y_pred)
    )

    y_true = y_true[valid]
    y_pred = y_pred[valid]

    residual = (
        y_pred
        - y_true
    )

    mae = float(
        np.mean(
            np.abs(residual)
        )
    )

    rmse = float(
        np.sqrt(
            np.mean(
                residual ** 2
            )
        )
    )

    denominator = float(
        np.sum(
            (
                y_true
                - np.mean(y_true)
            ) ** 2
        )
    )

    r2 = (
        np.nan
        if denominator <= 0.0
        else
        1.0
        - float(
            np.sum(
                residual ** 2
            )
        )
        / denominator
    )

    return {
        "N": len(y_true),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }


# ==================================================================================================
# 9. TEMPORAL TRANSFER / PERSISTENCE BENCHMARK
# ==================================================================================================

benchmark_rows = []

for horizon in [
    1,
    2,
    3,
]:

    metrics = metrics22e(
        DIGITAL22E_EVAL_DF[
            f"_Target_H{horizon}"
        ],
        DIGITAL22E_EVAL_DF[
            "_Turbidity"
        ],
    )

    benchmark_rows.append({
        "Station":
            "Virtual_DWTP_C",

        "Evidence_Type":
            "Digital-Twin temporal transfer",

        "Variable":
            "Turbidity",

        "Horizon_Days":
            horizon,

        "N":
            metrics["N"],

        "Persistence_MAE":
            metrics["MAE"],

        "Persistence_RMSE":
            metrics["RMSE"],

        "Persistence_R2":
            metrics["R2"],
    })


DIGITAL22E_TEMPORAL_BENCHMARK_DF = pd.DataFrame(
    benchmark_rows
)


# ==================================================================================================
# 10. COVARIATE CHARACTERIZATION
# ==================================================================================================

characterization_rows = []

for col, label in [
    (
        "_Turbidity",
        "Turbidity",
    ),
    (
        "_pH",
        "pH",
    ),
    (
        "_Temperature",
        "Water Temperature",
    ),
]:

    values = (
        DIGITAL22E_EVAL_DF[
            col
        ]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    characterization_rows.append({
        "Variable":
            label,

        "N":
            len(values),

        "Mean":
            float(
                np.mean(values)
            ),

        "Std":
            float(
                np.std(values)
            ),

        "Min":
            float(
                np.min(values)
            ),

        "Median":
            float(
                np.median(values)
            ),

        "Max":
            float(
                np.max(values)
            ),
    })


DIGITAL22E_CHARACTERIZATION_DF = pd.DataFrame(
    characterization_rows
)


# ==================================================================================================
# 11. AUDIT
# ==================================================================================================

dates = (
    df["_Date"]
    .sort_values()
)

date_diffs = (
    dates
    .diff()
    .dropna()
    .dt.days
)

DIGITAL22E_AUDIT_DF = pd.DataFrame([
    {
        "Item":
            "Rows loaded",

        "Value":
            len(
                DIGITAL22E_RAW_DF
            ),
    },
    {
        "Item":
            "Unique daily rows",

        "Value":
            len(df),
    },
    {
        "Item":
            "W10 eligible rows",

        "Value":
            int(
                df[
                    "_W10_Eligible"
                ].sum()
            ),
    },
    {
        "Item":
            "W10 + H1/H2/H3 evaluation rows",

        "Value":
            len(
                DIGITAL22E_EVAL_DF
            ),
    },
    {
        "Item":
            "First date",

        "Value":
            dates.min(),
    },
    {
        "Item":
            "Last date",

        "Value":
            dates.max(),
    },
    {
        "Item":
            "Consecutive date pairs",

        "Value":
            int(
                (
                    date_diffs == 1
                ).sum()
            ),
    },
    {
        "Item":
            "Non-consecutive date pairs",

        "Value":
            int(
                (
                    date_diffs != 1
                ).sum()
            ),
    },
    {
        "Item":
            "Missing turbidity",

        "Value":
            int(
                df[
                    "_Turbidity"
                ].isna().sum()
            ),
    },
    {
        "Item":
            "Missing pH",

        "Value":
            int(
                df[
                    "_pH"
                ].isna().sum()
            ),
    },
    {
        "Item":
            "Missing temperature",

        "Value":
            int(
                df[
                    "_Temperature"
                ].isna().sum()
            ),
    },
])


# ==================================================================================================
# 12. PROTOCOL
# ==================================================================================================

DIGITAL22E_PROTOCOL_DF = pd.DataFrame([
    {
        "Protocol_Item":
            "Dataset",

        "Confirmed":
            True,

        "Details":
            "Virtual_DWTP_C",
    },
    {
        "Protocol_Item":
            "Evidence type",

        "Confirmed":
            True,

        "Details":
            "Digital-Twin Transfer Test",
    },
    {
        "Protocol_Item":
            "Real-world external station",

        "Confirmed":
            False,

        "Details":
            "No",
    },
    {
        "Protocol_Item":
            "Included in real-world external average",

        "Confirmed":
            False,

        "Details":
            "No",
    },
    {
        "Protocol_Item":
            "UTW",

        "Confirmed":
            True,

        "Details":
            "W=10",
    },
    {
        "Protocol_Item":
            "External training",

        "Confirmed":
            False,

        "Details":
            "None",
    },
    {
        "Protocol_Item":
            "External tuning",

        "Confirmed":
            False,

        "Details":
            "None",
    },
    {
        "Protocol_Item":
            "External adaptation",

        "Confirmed":
            False,

        "Details":
            "None",
    },
    {
        "Protocol_Item":
            "External scaler fitting",

        "Confirmed":
            False,

        "Details":
            "None",
    },
    {
        "Protocol_Item":
            "Dose-model external accuracy claimed",

        "Confirmed":
            False,

        "Details":
            "No",
    },
])


# ==================================================================================================
# 13. PAPER SCOPE
# ==================================================================================================

DIGITAL22E_SUMMARY_DF = pd.DataFrame([
    {
        "Dataset":
            "Virtual_DWTP_C",

        "Evidence_Type":
            "Digital-Twin Transfer",

        "Real_World":
            False,

        "Include_In_Real_External_Average":
            False,

        "Evaluation_Rows":
            len(
                DIGITAL22E_EVAL_DF
            ),

        "Role":
            "Synthetic temporal/covariate transfer evidence",
    }
])


# ==================================================================================================
# 14. OUTPUT
# ==================================================================================================

print("\nDigital Twin audit:")

display(
    DIGITAL22E_AUDIT_DF
)


print("\nCovariate characterization:")

display(
    DIGITAL22E_CHARACTERIZATION_DF
    .round(6)
)


print("\nTemporal transfer benchmark:")

display(
    DIGITAL22E_TEMPORAL_BENCHMARK_DF
    .round(6)
)


print("\nProtocol:")

display(
    DIGITAL22E_PROTOCOL_DF
)


print("\nPaper scope:")

display(
    DIGITAL22E_SUMMARY_DF
)


print("\n" + "=" * 125)
print("CELL 22E — FINAL INTERPRETATION")
print("=" * 125)

print(
    "Virtual_DWTP_C is retained strictly as a separate Digital-Twin Transfer Test."
)

print(
    "It is not treated as independent real-world external evidence "
    "and is not included in the real-world external average."
)

print("\nCELL 22E STATUS: COMPLETED")
print("=" * 125)

print(
    f"Execution time: "
    f"{time.time() - CELL22E_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 23 — FINAL UNIFIED EXTERNAL GENERALIZATION
#
# PERSISTENCE-ANCHORED UTW-RG-AdaFedResidual
#
# Core idea:
#   Future turbidity = Persistence + learned bounded residual correction
#
# Source:
#   Kadhimiya + Karama + Resafa + Tarmyya
#
# External:
#   Austin + Tongji + Hannan + Kirkuk + Digital Twin
#
# Settings:
#   Persistence
#   ZeroShot
#   FewShot10
#   FewShot20
#
# Selection:
#   ALL model / blending choices selected from SOURCE VALIDATION only.
#
# Few-shot:
#   External calibration labels only.
#   Final external 80% Test is identical for ZeroShot / FewShot10 / FewShot20.
# ==================================================================================================

import os
import glob
import time
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

CELL23_START = time.time()

print("=" * 175)
print("CELL 23 — FINAL PERSISTENCE-ANCHORED EXTERNAL GENERALIZATION")
print("UTW + FedAdam + Relation Guidance + Domain Gate + Few-shot Residual Adapter")
print("=" * 175)


# ==================================================================================================
# 1. REQUIREMENTS
# ==================================================================================================

REQ23 = [
    "EXPECTED_STATIONS",
    "ALL_DATA_TARGETS",
    "ALGORITHM_CONFIGS",
]

missing = [
    x for x in REQ23
    if x not in globals()
]

if missing:
    raise NameError(
        "Run the previous Paper-3 cells first. Missing: "
        + str(missing)
    )

SRC23_STATIONS = list(EXPECTED_STATIONS)

EPS23 = 1e-12
SEED23 = 42

W23 = 10
HORIZONS23 = [1, 2, 3]

TRAIN_RATIO23 = 0.55
VALID_RATIO23 = 0.15

# External final 80% is fixed Test.
EXT_CONTEXT_RATIO23 = 0.20

SETTINGS23 = {
    "ZeroShot": 0.00,
    "FewShot10": 0.10,
    "FewShot20": 0.20,
}

FED_ROUNDS23 = 60
LOCAL_EPOCHS23 = 2
FINAL_LOCAL_EPOCHS23 = 120

SERVER_LR23 = 0.01
BETA1_23 = 0.90
BETA2_23 = 0.99
TAU23 = 1e-3

CLIP_NORM23 = 5.0

# Source-validation candidate grids
GLOBAL_WEIGHT_GRID23 = [
    0.50,
    0.70,
    0.85,
    1.00,
]

CORRECTION_STRENGTH_GRID23 = [
    0.00,
    0.10,
    0.25,
    0.50,
    0.75,
    1.00,
]

ADAPTER_RIDGE_GRID23 = [
    0.1,
    1.0,
    10.0,
    100.0,
]

ADAPTER_BLEND_GRID23 = [
    0.25,
    0.50,
    0.75,
    1.00,
]


# ==================================================================================================
# 2. METRICS
# ==================================================================================================

def m23_metrics(y, p, persistence=None):

    y = np.asarray(y, float).reshape(-1)
    p = np.asarray(p, float).reshape(-1)

    mask = (
        np.isfinite(y)
        &
        np.isfinite(p)
    )

    y = y[mask]
    p = p[mask]

    if len(y) == 0:
        return {
            "N": 0,
            "R": np.nan,
            "R2": np.nan,
            "MAE": np.nan,
            "RMSE": np.nan,
            "MAPE": np.nan,
            "sMAPE": np.nan,
            "NMAE": np.nan,
            "NRMSE": np.nan,
            "Skill_MAE": np.nan,
            "Skill_RMSE": np.nan,
        }

    e = p - y

    mae = float(
        np.mean(
            np.abs(e)
        )
    )

    rmse = float(
        np.sqrt(
            np.mean(
                e ** 2
            )
        )
    )

    if (
        len(y) >= 2
        and np.std(y) > EPS23
        and np.std(p) > EPS23
    ):

        r = float(
            np.corrcoef(
                y,
                p,
            )[0, 1]
        )

    else:
        r = np.nan

    sst = float(
        np.sum(
            (
                y
                - np.mean(y)
            ) ** 2
        )
    )

    r2 = (
        float(
            1.0
            -
            np.sum(
                e ** 2
            )
            / sst
        )
        if sst > EPS23
        else np.nan
    )

    nz = (
        np.abs(y) > 1e-8
    )

    mape = (
        float(
            100.0
            * np.mean(
                np.abs(
                    e[nz]
                    / y[nz]
                )
            )
        )
        if np.any(nz)
        else np.nan
    )

    den = (
        np.abs(y)
        + np.abs(p)
    )

    sm = (
        den > 1e-8
    )

    smape = (
        float(
            200.0
            * np.mean(
                np.abs(
                    p[sm] - y[sm]
                )
                / den[sm]
            )
        )
        if np.any(sm)
        else np.nan
    )

    scale = float(
        np.mean(
            np.abs(y)
        )
    )

    nmae = (
        mae / scale
        if scale > EPS23
        else np.nan
    )

    nrmse = (
        rmse / scale
        if scale > EPS23
        else np.nan
    )

    skill_mae = np.nan
    skill_rmse = np.nan

    if persistence is not None:

        per = np.asarray(
            persistence,
            float
        ).reshape(-1)

        per = per[mask]

        if len(per) == len(y):

            per_e = (
                per - y
            )

            per_mae = float(
                np.mean(
                    np.abs(
                        per_e
                    )
                )
            )

            per_rmse = float(
                np.sqrt(
                    np.mean(
                        per_e ** 2
                    )
                )
            )

            if per_mae > EPS23:
                skill_mae = float(
                    1.0
                    -
                    mae / per_mae
                )

            if per_rmse > EPS23:
                skill_rmse = float(
                    1.0
                    -
                    rmse / per_rmse
                )

    return {
        "N": int(len(y)),
        "R": r,
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "sMAPE": smape,
        "NMAE": nmae,
        "NRMSE": nrmse,
        "Skill_MAE": skill_mae,
        "Skill_RMSE": skill_rmse,
    }


# ==================================================================================================
# 3. TRANSFER-INVARIANT FEATURES
# ==================================================================================================

def m23_robust_scale(a):

    a = np.asarray(
        a,
        float,
    )

    med = float(
        np.nanmedian(a)
    )

    q75, q25 = np.nanpercentile(
        a,
        [75, 25],
    )

    scale = float(
        q75 - q25
    )

    if scale < EPS23:

        scale = float(
            np.nanstd(a)
        )

    if scale < EPS23:
        scale = 1.0

    return (
        a - med
    ) / scale


def m23_variable_features(a, log_transform=False):

    a = np.asarray(
        a,
        float,
    )

    if log_transform:

        a = np.log1p(
            np.maximum(
                a,
                0.0,
            )
        )

    current = float(
        a[0]
    )

    z = m23_robust_scale(
        a
    )

    magnitude = (
        float(
            np.mean(
                np.abs(a)
            )
        )
        + 1e-8
    )

    rel = (
        a - current
    ) / magnitude

    chronological = (
        a[::-1]
    )

    xx = np.arange(
        len(a),
        dtype=float,
    )

    xc = (
        xx - xx.mean()
    )

    yc = (
        chronological
        - chronological.mean()
    )

    slope = float(
        np.sum(
            xc * yc
        )
        /
        max(
            np.sum(
                xc ** 2
            ),
            EPS23,
        )
    )

    desc = np.array([
        float(np.mean(a)),
        float(np.std(a)),
        float(np.min(a)),
        float(np.max(a)),
        float(np.max(a) - np.min(a)),
        float(a[0] - a[-1]),
        slope,
    ])

    desc = (
        desc / magnitude
    )

    return np.r_[
        z,
        rel,
        desc,
    ]


# ==================================================================================================
# 4. GENERIC SEQUENCE BUILDER
#
# Target = future log-change.
# Persistence = current observed turbidity.
# ==================================================================================================

def m23_build_sequence(
    df,
    date_col,
    temp_col,
    turb_input_col,
    ph_col,
    turb_observed_col=None,
    mode="order",
):

    if turb_observed_col is None:
        turb_observed_col = (
            turb_input_col
        )

    x = pd.DataFrame({
        "Date":
            pd.to_datetime(
                df[date_col],
                errors="coerce",
            ),

        "Temperature":
            pd.to_numeric(
                df[temp_col],
                errors="coerce",
            ),

        "Turbidity_Input":
            pd.to_numeric(
                df[turb_input_col],
                errors="coerce",
            ),

        "Turbidity_Observed":
            pd.to_numeric(
                df[turb_observed_col],
                errors="coerce",
            ),

        "pH":
            pd.to_numeric(
                df[ph_col],
                errors="coerce",
            ),
    })

    x = (
        x
        .dropna(
            subset=["Date"]
        )
        .copy()
    )

    if mode == "calendar":

        x = (
            x
            .sort_values("Date")
            .drop_duplicates(
                subset=["Date"],
                keep="first",
            )
        )

        full_dates = pd.date_range(
            x["Date"].min(),
            x["Date"].max(),
            freq="D",
        )

        x = (
            x
            .set_index("Date")
            .reindex(full_dates)
        )

        x.index.name = "Date"

        x = (
            x
            .reset_index()
        )

    elif mode == "order":

        x = (
            x
            .reset_index(drop=True)
        )

    else:

        raise ValueError(
            "mode must be 'calendar' or 'order'"
        )

    rows = []

    for i in range(
        W23 - 1,
        len(x),
    ):

        w = x.iloc[
            i - W23 + 1:
            i + 1
        ]

        required = (
            w[
                [
                    "Temperature",
                    "Turbidity_Input",
                    "pH",
                ]
            ]
            .notna()
            .all()
            .all()
        )

        if not required:
            continue

        current_obs = x.loc[
            i,
            "Turbidity_Observed",
        ]

        if (
            not np.isfinite(
                current_obs
            )
            or current_obs < 0
        ):
            continue

        turb_arr = (
            w[
                "Turbidity_Input"
            ]
            .to_numpy(float)
            [::-1]
        )

        temp_arr = (
            w[
                "Temperature"
            ]
            .to_numpy(float)
            [::-1]
        )

        ph_arr = (
            w[
                "pH"
            ]
            .to_numpy(float)
            [::-1]
        )

        feat = np.r_[
            m23_variable_features(
                turb_arr,
                log_transform=True,
            ),

            m23_variable_features(
                temp_arr,
                log_transform=False,
            ),

            m23_variable_features(
                ph_arr,
                log_transform=False,
            ),
        ]

        log_turb = np.log1p(
            np.maximum(
                turb_arr,
                0.0,
            )
        )

        recent_trend = float(
            log_turb[0]
            - log_turb[
                min(
                    3,
                    len(log_turb) - 1,
                )
            ]
        )

        recent_vol = float(
            np.std(
                log_turb[:5]
            )
        )

        rec = {
            "Origin_Index":
                int(i),

            "Date":
                x.loc[
                    i,
                    "Date",
                ],

            "Current_Turbidity":
                float(current_obs),

            "Current_Log":
                float(
                    np.log1p(
                        current_obs
                    )
                ),

            "Recent_Trend":
                recent_trend,

            "Recent_Vol":
                recent_vol,

            "X_raw":
                np.asarray(
                    feat,
                    float,
                ),
        }

        for h in HORIZONS23:

            j = i + h

            if (
                j < len(x)
                and np.isfinite(
                    x.loc[
                        j,
                        "Turbidity_Observed",
                    ]
                )
                and x.loc[
                    j,
                    "Turbidity_Observed",
                ] >= 0
            ):

                future = float(
                    x.loc[
                        j,
                        "Turbidity_Observed",
                    ]
                )

                future_log = float(
                    np.log1p(
                        future
                    )
                )

                delta_log = float(
                    future_log
                    -
                    rec[
                        "Current_Log"
                    ]
                )

                rec[
                    f"Y_Turbidity_H{h}"
                ] = future

                rec[
                    f"Y_Log_H{h}"
                ] = future_log

                rec[
                    f"Y_DeltaLog_H{h}"
                ] = delta_log

                rec[
                    f"Target_Index_H{h}"
                ] = int(j)

                rec[
                    f"Target_Date_H{h}"
                ] = x.loc[
                    j,
                    "Date",
                ]

            else:

                rec[
                    f"Y_Turbidity_H{h}"
                ] = np.nan

                rec[
                    f"Y_Log_H{h}"
                ] = np.nan

                rec[
                    f"Y_DeltaLog_H{h}"
                ] = np.nan

                rec[
                    f"Target_Index_H{h}"
                ] = np.nan

                rec[
                    f"Target_Date_H{h}"
                ] = pd.NaT

        rows.append(rec)

    out = pd.DataFrame(
        rows
    )

    if len(out) == 0:
        raise RuntimeError(
            "No valid W10 sequences were created."
        )

    return out


# ==================================================================================================
# 5. SOURCE DATA
# ==================================================================================================

SRC23_BASE = (
    ALL_DATA_TARGETS[
        [
            "Date",
            "Station",
            "(°C)_raw",
            "Turb(NTU)_raw",
            "pH_raw",
        ]
    ]
    .copy()
)

SRC23_BASE[
    "Date"
] = pd.to_datetime(
    SRC23_BASE[
        "Date"
    ],
    errors="coerce",
)

for c in [
    "(°C)_raw",
    "Turb(NTU)_raw",
    "pH_raw",
]:

    SRC23_BASE[c] = pd.to_numeric(
        SRC23_BASE[c],
        errors="coerce",
    )


SRC23_SEQ = {}

for station in SRC23_STATIONS:

    sdf = (
        SRC23_BASE[
            SRC23_BASE[
                "Station"
            ].eq(station)
        ]
        .sort_values("Date")
        .reset_index(drop=True)
    )

    seq = m23_build_sequence(
        sdf,
        "Date",
        "(°C)_raw",
        "Turb(NTU)_raw",
        "pH_raw",
        "Turb(NTU)_raw",
        mode="order",
    )

    SRC23_SEQ[
        station
    ] = seq

    print(
        f"Source {station:10s}: "
        f"{len(seq):,} W10 origins"
    )


# ==================================================================================================
# 6. SOURCE TRAIN / VALIDATION
# ==================================================================================================

SRC23_SPLIT = {}

for station in SRC23_STATIONS:

    seq = (
        SRC23_SEQ[
            station
        ]
        .reset_index(drop=True)
    )

    n = len(seq)

    n_train = int(
        np.floor(
            TRAIN_RATIO23
            * n
        )
    )

    n_valid = int(
        np.floor(
            VALID_RATIO23
            * n
        )
    )

    SRC23_SPLIT[
        station
    ] = {
        "Train":
            seq.iloc[
                :n_train
            ].copy(),

        "Validation":
            seq.iloc[
                n_train:
                n_train + n_valid
            ].copy(),
    }


# ==================================================================================================
# 7. SOURCE FEATURE SCALER — TRAIN ONLY
# ==================================================================================================

XTRAIN23 = np.vstack([
    np.vstack(
        SRC23_SPLIT[
            s
        ][
            "Train"
        ][
            "X_raw"
        ].to_numpy()
    )
    for s in SRC23_STATIONS
])

XMEAN23 = np.mean(
    XTRAIN23,
    axis=0,
)

XSTD23 = np.std(
    XTRAIN23,
    axis=0,
)

XSTD23[
    XSTD23 < EPS23
] = 1.0


def m23_scale_X(frame):

    X = np.vstack(
        frame[
            "X_raw"
        ].to_numpy()
    )

    return (
        X - XMEAN23
    ) / XSTD23


# ==================================================================================================
# 8. CUSTOM LINEAR TRAINING
# ==================================================================================================

def m23_linear_pred(
    X,
    params,
):

    params = np.asarray(
        params,
        float,
    )

    return (
        np.asarray(
            X,
            float,
        )
        @ params[:-1]
        + params[-1]
    )


def m23_clip_grad(g):

    g = np.asarray(
        g,
        float,
    )

    norm = float(
        np.linalg.norm(g)
    )

    if (
        norm > CLIP_NORM23
        and norm > 0
    ):

        g = (
            g
            * CLIP_NORM23
            / norm
        )

    return g


def m23_train_linear(
    X,
    y,
    algorithm,
    epochs,
    seed,
    initial=None,
):

    cfg = ALGORITHM_CONFIGS[
        algorithm
    ]

    X = np.asarray(
        X,
        float,
    )

    y = np.asarray(
        y,
        float,
    )

    n, d = X.shape

    if initial is None:

        params = np.zeros(
            d + 1,
            float,
        )

    else:

        params = np.asarray(
            initial,
            float,
        ).copy()

    rng = np.random.default_rng(
        seed
    )

    for epoch in range(
        int(epochs)
    ):

        lr = (
            cfg[
                "learning_rate"
            ]
            *
            (
                cfg[
                    "learning_rate_decay"
                ]
                ** epoch
            )
        )

        bs = (
            n
            if cfg[
                "batch_size"
            ] is None
            else min(
                int(
                    cfg[
                        "batch_size"
                    ]
                ),
                n,
            )
        )

        order = (
            np.arange(n)
            if bs == n
            else rng.permutation(n)
        )

        for start in range(
            0,
            n,
            bs,
        ):

            idx = order[
                start:
                start + bs
            ]

            xb = X[idx]
            yb = y[idx]

            pred = m23_linear_pred(
                xb,
                params,
            )

            residual = (
                pred - yb
            )

            delta = cfg[
                "huber_delta"
            ]

            psi = (
                residual
                if delta is None
                else np.clip(
                    residual,
                    -delta,
                    delta,
                )
            )

            gw = (
                xb.T
                @ psi
                / max(
                    len(idx),
                    1,
                )
                +
                cfg[
                    "l2"
                ]
                * params[:-1]
            )

            gb = np.array([
                float(
                    np.mean(psi)
                )
            ])

            grad = m23_clip_grad(
                np.r_[
                    gw,
                    gb,
                ]
            )

            params -= (
                lr * grad
            )

            if cfg[
                "l1"
            ] > 0:

                z = params[:-1]

                params[:-1] = (
                    np.sign(z)
                    * np.maximum(
                        np.abs(z)
                        -
                        lr
                        * cfg[
                            "l1"
                        ],
                        0.0,
                    )
                )

    return params


# ==================================================================================================
# 9. FEDADAM
# ==================================================================================================

def m23_fedadam(
    client_data,
    algorithm,
    seed,
):

    d = next(
        iter(
            client_data.values()
        )
    )[0].shape[1]

    global_params = np.zeros(
        d + 1,
        float,
    )

    m = np.zeros_like(
        global_params
    )

    v = np.zeros_like(
        global_params
    )

    for rnd in range(
        FED_ROUNDS23
    ):

        local_params = []
        local_sizes = []

        for si, station in enumerate(
            SRC23_STATIONS
        ):

            Xc, yc = (
                client_data[
                    station
                ]
            )

            lp = m23_train_linear(
                Xc,
                yc,
                algorithm,
                LOCAL_EPOCHS23,
                seed
                + rnd * 100
                + si,
                initial=global_params,
            )

            local_params.append(
                lp
            )

            local_sizes.append(
                len(yc)
            )

        sizes = np.asarray(
            local_sizes,
            float,
        )

        weights = (
            sizes
            / sizes.sum()
        )

        avg = np.sum(
            np.stack(
                local_params,
                axis=0,
            )
            * weights[:, None],
            axis=0,
        )

        delta = (
            avg - global_params
        )

        m = (
            BETA1_23 * m
            +
            (
                1.0 - BETA1_23
            )
            * delta
        )

        v = (
            BETA2_23 * v
            +
            (
                1.0 - BETA2_23
            )
            * (
                delta ** 2
            )
        )

        global_params = (
            global_params
            +
            SERVER_LR23
            * m
            /
            (
                np.sqrt(v)
                + TAU23
            )
        )

    return global_params


# ==================================================================================================
# 10. RECONSTRUCTION
# ==================================================================================================

def m23_reconstruct(
    current,
    delta_log,
):

    current = np.asarray(
        current,
        float,
    )

    delta_log = np.asarray(
        delta_log,
        float,
    )

    future_log = (
        np.log1p(
            np.maximum(
                current,
                0.0,
            )
        )
        +
        delta_log
    )

    pred = np.expm1(
        future_log
    )

    return np.maximum(
        pred,
        0.0,
    )


# ==================================================================================================
# 11. SOURCE MODEL SELECTION
#
# Select simultaneously:
#   algorithm
#   global/relation blend
#   correction strength toward persistence
#
# ALL on SOURCE VALIDATION.
# ==================================================================================================

MODEL23 = {}
SELECTION23_ROWS = []

ALG23 = [
    "Ridge",
    "ElasticNet",
    "SGD_Linear",
    "SGD_Huber",
]


def m23_softmax_distance(
    distances,
):

    d = np.asarray(
        distances,
        float,
    )

    z = (
        -d
        -
        np.max(
            -d
        )
    )

    w = np.exp(z)

    return (
        w / w.sum()
    )


for h in HORIZONS23:

    task = (
        f"Turbidity_H{h}"
    )

    ycol = (
        f"Y_DeltaLog_H{h}"
    )

    # ----------------------------------------------------------------------
    # Target scaler from SOURCE TRAIN only
    # ----------------------------------------------------------------------

    pooled_y = np.concatenate([
        SRC23_SPLIT[
            s
        ][
            "Train"
        ][ycol]
        .dropna()
        .to_numpy(float)

        for s in SRC23_STATIONS
    ])

    y_mean = float(
        np.mean(
            pooled_y
        )
    )

    y_std = float(
        np.std(
            pooled_y
        )
    )

    if y_std < EPS23:
        y_std = 1.0

    # Robust source bounds for change prediction
    delta_low = float(
        np.quantile(
            pooled_y,
            0.005,
        )
    )

    delta_high = float(
        np.quantile(
            pooled_y,
            0.995,
        )
    )

    rows = []

    for algorithm in ALG23:

        clients = {}
        train_centroids = {}

        # --------------------------------------------------------------
        # Federated training
        # --------------------------------------------------------------

        for station in SRC23_STATIONS:

            tr = (
                SRC23_SPLIT[
                    station
                ][
                    "Train"
                ]
                .dropna(
                    subset=[ycol]
                )
                .copy()
            )

            Xtr = m23_scale_X(
                tr
            )

            ytr = (
                tr[ycol].to_numpy(float)
                - y_mean
            ) / y_std

            clients[
                station
            ] = (
                Xtr,
                ytr,
            )

            train_centroids[
                station
            ] = np.mean(
                Xtr,
                axis=0,
            )

        global_params = m23_fedadam(
            clients,
            algorithm,
            seed=(
                SEED23
                + h * 1000
            ),
        )

        # --------------------------------------------------------------
        # Train source local experts
        # --------------------------------------------------------------

        locals_alg = {}

        for si, station in enumerate(
            SRC23_STATIONS
        ):

            Xtr, ytr = (
                clients[
                    station
                ]
            )

            locals_alg[
                station
            ] = m23_train_linear(
                Xtr,
                ytr,
                algorithm,
                FINAL_LOCAL_EPOCHS23,
                seed=(
                    SEED23
                    + 5000
                    + h * 100
                    + si
                ),
            )

        # --------------------------------------------------------------
        # Validation predictions station by station
        # --------------------------------------------------------------

        cache = {}

        for station in SRC23_STATIONS:

            va = (
                SRC23_SPLIT[
                    station
                ][
                    "Validation"
                ]
                .dropna(
                    subset=[ycol]
                )
                .copy()
            )

            Xv = m23_scale_X(
                va
            )

            g = (
                m23_linear_pred(
                    Xv,
                    global_params,
                )
                * y_std
                + y_mean
            )

            other = [
                s
                for s in SRC23_STATIONS
                if s != station
            ]

            target_centroid = np.mean(
                Xv,
                axis=0,
            )

            distances = [
                float(
                    np.sqrt(
                        np.mean(
                            (
                                target_centroid
                                -
                                train_centroids[s]
                            ) ** 2
                        )
                    )
                )
                for s in other
            ]

            rw = m23_softmax_distance(
                distances
            )

            relation = np.zeros(
                len(va),
                float,
            )

            for s, w in zip(
                other,
                rw,
            ):

                lp = (
                    m23_linear_pred(
                        Xv,
                        locals_alg[s],
                    )
                    * y_std
                    + y_mean
                )

                relation += (
                    float(w)
                    * lp
                )

            cache[
                station
            ] = {
                "Frame": va,
                "Global": g,
                "Relation": relation,
            }

        # --------------------------------------------------------------
        # Select source blending parameters
        # --------------------------------------------------------------

        for global_weight in (
            GLOBAL_WEIGHT_GRID23
        ):

            for strength in (
                CORRECTION_STRENGTH_GRID23
            ):

                all_y = []
                all_p = []

                for station in SRC23_STATIONS:

                    q = cache[
                        station
                    ]

                    va = q[
                        "Frame"
                    ]

                    raw_delta = (
                        global_weight
                        * q["Global"]
                        +
                        (
                            1.0
                            - global_weight
                        )
                        * q["Relation"]
                    )

                    raw_delta = np.clip(
                        raw_delta,
                        delta_low,
                        delta_high,
                    )

                    delta_pred = (
                        strength
                        * raw_delta
                    )

                    pred = m23_reconstruct(
                        va[
                            "Current_Turbidity"
                        ].to_numpy(float),
                        delta_pred,
                    )

                    actual = va[
                        f"Y_Turbidity_H{h}"
                    ].to_numpy(float)

                    all_y.append(
                        actual
                    )

                    all_p.append(
                        pred
                    )

                met = m23_metrics(
                    np.concatenate(
                        all_y
                    ),
                    np.concatenate(
                        all_p
                    ),
                )

                rows.append({
                    "Task": task,
                    "Algorithm": algorithm,
                    "Global_Weight": global_weight,
                    "Correction_Strength": strength,
                    **met,
                })

    sel = (
        pd.DataFrame(
            rows
        )
        .sort_values(
            [
                "MAE",
                "RMSE",
                "R2",
            ],
            ascending=[
                True,
                True,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    best = sel.iloc[0]

    best_alg = str(
        best[
            "Algorithm"
        ]
    )

    best_gw = float(
        best[
            "Global_Weight"
        ]
    )

    best_strength = float(
        best[
            "Correction_Strength"
        ]
    )

    print(
        f"{task}: "
        f"{best_alg} | "
        f"Global={best_gw:.2f} | "
        f"ResidualStrength={best_strength:.2f}"
    )

    # ==================================================================
    # Final source model: Train + Validation
    # ==================================================================

    final_clients = {}
    final_centroids = {}

    tv_y_all = []

    for station in SRC23_STATIONS:

        tv = pd.concat(
            [
                SRC23_SPLIT[
                    station
                ][
                    "Train"
                ],

                SRC23_SPLIT[
                    station
                ][
                    "Validation"
                ],
            ],
            ignore_index=True,
        )

        tv = (
            tv
            .dropna(
                subset=[ycol]
            )
            .copy()
        )

        tv_y_all.append(
            tv[ycol]
            .to_numpy(float)
        )

    tv_y_all = np.concatenate(
        tv_y_all
    )

    final_y_mean = float(
        np.mean(
            tv_y_all
        )
    )

    final_y_std = float(
        np.std(
            tv_y_all
        )
    )

    if final_y_std < EPS23:
        final_y_std = 1.0

    final_low = float(
        np.quantile(
            tv_y_all,
            0.005,
        )
    )

    final_high = float(
        np.quantile(
            tv_y_all,
            0.995,
        )
    )

    for station in SRC23_STATIONS:

        tv = pd.concat(
            [
                SRC23_SPLIT[
                    station
                ][
                    "Train"
                ],

                SRC23_SPLIT[
                    station
                ][
                    "Validation"
                ],
            ],
            ignore_index=True,
        )

        tv = (
            tv
            .dropna(
                subset=[ycol]
            )
            .copy()
        )

        Xtv = m23_scale_X(
            tv
        )

        ytv = (
            tv[ycol].to_numpy(float)
            - final_y_mean
        ) / final_y_std

        final_clients[
            station
        ] = (
            Xtv,
            ytv,
        )

        final_centroids[
            station
        ] = np.mean(
            Xtv,
            axis=0,
        )

    final_global = m23_fedadam(
        final_clients,
        best_alg,
        seed=(
            SEED23
            + 12000
            + h
        ),
    )

    final_locals = {}

    for si, station in enumerate(
        SRC23_STATIONS
    ):

        Xtv, ytv = (
            final_clients[
                station
            ]
        )

        final_locals[
            station
        ] = m23_train_linear(
            Xtv,
            ytv,
            best_alg,
            FINAL_LOCAL_EPOCHS23,
            seed=(
                SEED23
                + 15000
                + h * 100
                + si
            ),
        )

    # Source-domain radius for transfer gate
    pair_distances = []

    for i, s1 in enumerate(
        SRC23_STATIONS
    ):

        for s2 in (
            SRC23_STATIONS[
                i + 1:
            ]
        ):

            pair_distances.append(
                float(
                    np.sqrt(
                        np.mean(
                            (
                                final_centroids[s1]
                                -
                                final_centroids[s2]
                            ) ** 2
                        )
                    )
                )
            )

    source_radius = float(
        np.median(
            pair_distances
        )
    )

    if source_radius < EPS23:
        source_radius = 1.0

    MODEL23[
        task
    ] = {
        "Algorithm":
            best_alg,

        "GlobalWeight":
            best_gw,

        "CorrectionStrength":
            best_strength,

        "y_mean":
            final_y_mean,

        "y_std":
            final_y_std,

        "DeltaLow":
            final_low,

        "DeltaHigh":
            final_high,

        "Global":
            final_global,

        "Locals":
            final_locals,

        "Centroids":
            final_centroids,

        "SourceRadius":
            source_radius,
    }

    SELECTION23_ROWS.append({
        "Task":
            task,

        "Algorithm":
            best_alg,

        "Global_Weight":
            best_gw,

        "Correction_Strength":
            best_strength,

        "Validation_R":
            float(
                best["R"]
            ),

        "Validation_R2":
            float(
                best["R2"]
            ),

        "Validation_MAE":
            float(
                best["MAE"]
            ),

        "Validation_RMSE":
            float(
                best["RMSE"]
            ),
    })


SELECTION23_DF = pd.DataFrame(
    SELECTION23_ROWS
)

print(
    "\nSOURCE VALIDATION SELECTION:"
)

display(
    SELECTION23_DF
    .round(6)
)


# ==================================================================================================
# 12. EXTERNAL RELATION + DOMAIN GATE
# ==================================================================================================

def m23_external_experts(
    task,
    frame_context,
    frame_prediction,
):

    model = MODEL23[
        task
    ]

    Xctx = m23_scale_X(
        frame_context
    )

    Xpred = m23_scale_X(
        frame_prediction
    )

    external_centroid = np.mean(
        Xctx,
        axis=0,
    )

    distances = []

    for station in SRC23_STATIONS:

        c = model[
            "Centroids"
        ][station]

        distances.append(
            float(
                np.sqrt(
                    np.mean(
                        (
                            external_centroid
                            - c
                        ) ** 2
                    )
                )
            )
        )

    rw = m23_softmax_distance(
        distances
    )

    relation = np.zeros(
        len(frame_prediction),
        float,
    )

    ym = model[
        "y_mean"
    ]

    ys = model[
        "y_std"
    ]

    for station, w in zip(
        SRC23_STATIONS,
        rw,
    ):

        lp = (
            m23_linear_pred(
                Xpred,
                model[
                    "Locals"
                ][station],
            )
            * ys
            + ym
        )

        relation += (
            float(w)
            * lp
        )

    global_pred = (
        m23_linear_pred(
            Xpred,
            model[
                "Global"
            ],
        )
        * ys
        + ym
    )

    nearest_distance = float(
        np.min(
            distances
        )
    )

    radius = model[
        "SourceRadius"
    ]

    # Smooth domain-similarity gate.
    # Near source domain -> close to 1.
    # Far domain         -> correction shrinks toward persistence.
    domain_gate = float(
        1.0
        /
        (
            1.0
            +
            (
                nearest_distance
                /
                max(
                    radius,
                    EPS23,
                )
            ) ** 2
        )
    )

    raw_delta = (
        model[
            "GlobalWeight"
        ]
        * global_pred
        +
        (
            1.0
            - model[
                "GlobalWeight"
            ]
        )
        * relation
    )

    raw_delta = np.clip(
        raw_delta,
        model[
            "DeltaLow"
        ],
        model[
            "DeltaHigh"
        ],
    )

    zero_delta = (
        model[
            "CorrectionStrength"
        ]
        * domain_gate
        * raw_delta
    )

    relation_weights = {
        s: float(w)
        for s, w
        in zip(
            SRC23_STATIONS,
            rw,
        )
    }

    return (
        zero_delta,
        domain_gate,
        relation_weights,
    )


# ==================================================================================================
# 13. FEW-SHOT RESIDUAL ADAPTER
#
# Model in log-space:
#
# future_log ~=
#       intercept
#     + beta_current * current_log
#     + beta_zero    * zero-shot correction
#     + beta_trend   * short trend
#     + beta_vol     * short volatility
#
# Ridge prior is centered on the ZERO-SHOT model:
#   intercept = 0
#   current    = 1
#   zero       = 1
#   trend      = 0
#   volatility = 0
#
# ==================================================================================================

def m23_adapter_design(
    frame,
    zero_delta,
):

    return np.column_stack([
        np.ones(
            len(frame)
        ),

        frame[
            "Current_Log"
        ].to_numpy(float),

        np.asarray(
            zero_delta,
            float,
        ),

        frame[
            "Recent_Trend"
        ].to_numpy(float),

        frame[
            "Recent_Vol"
        ].to_numpy(float),
    ])


ADAPTER_PRIOR23 = np.array([
    0.0,
    1.0,
    1.0,
    0.0,
    0.0,
])


def m23_prior_ridge_fit(
    X,
    y,
    ridge,
):

    X = np.asarray(
        X,
        float,
    )

    y = np.asarray(
        y,
        float,
    )

    penalty = np.eye(
        X.shape[1]
    )

    # Slightly weaker penalty on intercept
    penalty[0, 0] = 0.25

    A = (
        X.T @ X
        +
        ridge
        * penalty
    )

    b = (
        X.T @ y
        +
        ridge
        * penalty
        @ ADAPTER_PRIOR23
    )

    return np.linalg.solve(
        A,
        b,
    )


def m23_fewshot_predict(
    task,
    calibration,
    test,
    context,
    zero_delta_test,
):

    h = int(
        task.split("H")[-1]
    )

    if (
        calibration is None
        or len(calibration) < 4
    ):

        return zero_delta_test

    calibration = (
        calibration
        .dropna(
            subset=[
                f"Y_Log_H{h}",
                f"Y_DeltaLog_H{h}",
            ]
        )
        .reset_index(drop=True)
        .copy()
    )

    n = len(
        calibration
    )

    if n < 4:
        return zero_delta_test

    # Compute zero-shot delta for calibration rows
    zero_delta_cal, _, _ = (
        m23_external_experts(
            task,
            context,
            calibration,
        )
    )

    # ------------------------------------------------------------------
    # Very small calibration set:
    # only learn a strongly-shrunk mean residual.
    # ------------------------------------------------------------------

    if n < 15:

        true_delta = (
            calibration[
                f"Y_DeltaLog_H{h}"
            ]
            .to_numpy(float)
        )

        residual = (
            true_delta
            - zero_delta_cal
        )

        shrink = (
            n
            /
            (
                n + 20.0
            )
        )

        correction = (
            shrink
            * float(
                np.mean(
                    residual
                )
            )
        )

        pred_delta = (
            zero_delta_test
            + correction
        )

        model = MODEL23[
            task
        ]

        return np.clip(
            pred_delta,
            model[
                "DeltaLow"
            ],
            model[
                "DeltaHigh"
            ],
        )

    # ------------------------------------------------------------------
    # Chronological internal calibration split
    # ------------------------------------------------------------------

    nfit = max(
        8,
        int(
            np.floor(
                0.70 * n
            )
        ),
    )

    nfit = min(
        nfit,
        n - 3,
    )

    fit = calibration.iloc[
        :nfit
    ].copy()

    val = calibration.iloc[
        nfit:
    ].copy()

    zero_fit, _, _ = (
        m23_external_experts(
            task,
            context,
            fit,
        )
    )

    zero_val, _, _ = (
        m23_external_experts(
            task,
            context,
            val,
        )
    )

    Xfit = m23_adapter_design(
        fit,
        zero_fit,
    )

    yfit = (
        fit[
            f"Y_Log_H{h}"
        ]
        .to_numpy(float)
    )

    Xval = m23_adapter_design(
        val,
        zero_val,
    )

    yval = (
        val[
            f"Y_Turbidity_H{h}"
        ]
        .to_numpy(float)
    )

    current_val = (
        val[
            "Current_Turbidity"
        ]
        .to_numpy(float)
    )

    best = None

    for ridge in (
        ADAPTER_RIDGE_GRID23
    ):

        beta = m23_prior_ridge_fit(
            Xfit,
            yfit,
            ridge,
        )

        adapted_val_log = (
            Xval @ beta
        )

        base_val_log = (
            val[
                "Current_Log"
            ]
            .to_numpy(float)
            +
            zero_val
        )

        for blend in (
            ADAPTER_BLEND_GRID23
        ):

            final_val_log = (
                (
                    1.0 - blend
                )
                * base_val_log
                +
                blend
                * adapted_val_log
            )

            pred_val = np.maximum(
                np.expm1(
                    final_val_log
                ),
                0.0,
            )

            met = m23_metrics(
                yval,
                pred_val,
                persistence=current_val,
            )

            candidate = {
                "ridge":
                    ridge,

                "blend":
                    blend,

                "MAE":
                    met[
                        "MAE"
                    ],

                "RMSE":
                    met[
                        "RMSE"
                    ],

                "R2":
                    met[
                        "R2"
                    ],
            }

            if best is None:

                best = candidate

            else:

                old_key = (
                    best["MAE"],
                    best["RMSE"],
                    -(
                        best["R2"]
                        if np.isfinite(
                            best["R2"]
                        )
                        else -1e18
                    ),
                )

                new_key = (
                    candidate["MAE"],
                    candidate["RMSE"],
                    -(
                        candidate["R2"]
                        if np.isfinite(
                            candidate["R2"]
                        )
                        else -1e18
                    ),
                )

                if new_key < old_key:
                    best = candidate

    # ------------------------------------------------------------------
    # Refit adapter on ALL calibration labels
    # ------------------------------------------------------------------

    Xcal = m23_adapter_design(
        calibration,
        zero_delta_cal,
    )

    ycal = (
        calibration[
            f"Y_Log_H{h}"
        ]
        .to_numpy(float)
    )

    beta = m23_prior_ridge_fit(
        Xcal,
        ycal,
        best[
            "ridge"
        ],
    )

    Xtest = m23_adapter_design(
        test,
        zero_delta_test,
    )

    adapted_test_log = (
        Xtest @ beta
    )

    base_test_log = (
        test[
            "Current_Log"
        ]
        .to_numpy(float)
        +
        zero_delta_test
    )

    final_test_log = (
        (
            1.0
            - best[
                "blend"
            ]
        )
        * base_test_log
        +
        best[
            "blend"
        ]
        * adapted_test_log
    )

    pred_delta = (
        final_test_log
        -
        test[
            "Current_Log"
        ]
        .to_numpy(float)
    )

    # ------------------------------------------------------------------
    # Few-shot-specific bounds:
    # source bounds + observed calibration bounds.
    # ------------------------------------------------------------------

    model = MODEL23[
        task
    ]

    cal_delta = (
        calibration[
            f"Y_DeltaLog_H{h}"
        ]
        .to_numpy(float)
    )

    cal_low = float(
        np.quantile(
            cal_delta,
            0.01,
        )
    )

    cal_high = float(
        np.quantile(
            cal_delta,
            0.99,
        )
    )

    lower = min(
        model[
            "DeltaLow"
        ],
        cal_low,
    )

    upper = max(
        model[
            "DeltaHigh"
        ],
        cal_high,
    )

    pred_delta = np.clip(
        pred_delta,
        lower,
        upper,
    )

    return pred_delta


# ==================================================================================================
# 14. EXTERNAL FILE HELPERS
# ==================================================================================================

def m23_norm(c):

    return (
        str(c)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


def m23_find_col(
    df,
    candidates,
):

    lookup = {
        m23_norm(c): c
        for c in df.columns
    }

    for name in candidates:

        key = m23_norm(
            name
        )

        if key in lookup:
            return lookup[
                key
            ]

    return None


def m23_find_file(
    keywords,
    preferred=None,
):

    preferred = (
        preferred or []
    )

    for path in preferred:

        if os.path.exists(
            path
        ):
            return path

    for path in glob.glob(
        "/content/*.xlsx"
    ):

        low = os.path.basename(
            path
        ).lower()

        if all(
            k.lower() in low
            for k in keywords
        ):
            return path

    return None


def m23_scan_excel(
    path,
    date_names,
    temp_names,
    turb_input_names,
    ph_names,
    turb_observed_names=None,
):

    xls = pd.ExcelFile(
        path
    )

    best = None

    for sheet in xls.sheet_names:

        for header in [
            0,
            1,
            2,
            3,
        ]:

            try:

                df = pd.read_excel(
                    path,
                    sheet_name=sheet,
                    header=header,
                )

            except Exception:
                continue

            d = m23_find_col(
                df,
                date_names,
            )

            t = m23_find_col(
                df,
                temp_names,
            )

            ti = m23_find_col(
                df,
                turb_input_names,
            )

            p = m23_find_col(
                df,
                ph_names,
            )

            if turb_observed_names is None:

                to = ti

            else:

                to = m23_find_col(
                    df,
                    turb_observed_names,
                )

            if all(
                c is not None
                for c in [
                    d,
                    t,
                    ti,
                    p,
                    to,
                ]
            ):

                score = len(df)

                if (
                    best is None
                    or score > best[
                        "score"
                    ]
                ):

                    best = {
                        "score":
                            score,

                        "sheet":
                            sheet,

                        "header":
                            header,

                        "df":
                            df,

                        "date":
                            d,

                        "temp":
                            t,

                        "turb_input":
                            ti,

                        "ph":
                            p,

                        "turb_observed":
                            to,
                    }

    if best is None:

        raise RuntimeError(
            f"Could not resolve workbook: {path}"
        )

    return best


# ==================================================================================================
# 15. AUSTIN
# ==================================================================================================

AUSTIN23_FILE = m23_find_file(
    ["austin"],
    preferred=[
        "/content/Austin_USGS_Final_Preprocessed_W10_Input المعالج.xlsx",
        "/content/Austin_USGS_Final_Preprocessed_W10_Input.xlsx",
    ],
)

if AUSTIN23_FILE is None:
    raise FileNotFoundError(
        "Austin workbook not found."
    )

AUSTIN23_LOAD = m23_scan_excel(
    AUSTIN23_FILE,

    date_names=[
        "date",
    ],

    temp_names=[
        "temperature_c",
        "temperature",
    ],

    turb_input_names=[
        "turbidity_fnu",
        "raw_turbidity",
        "turbidity",
    ],

    ph_names=[
        "ph",
        "pH",
    ],
)

AUSTIN23_SEQ = m23_build_sequence(
    AUSTIN23_LOAD["df"],
    AUSTIN23_LOAD["date"],
    AUSTIN23_LOAD["temp"],
    AUSTIN23_LOAD["turb_input"],
    AUSTIN23_LOAD["ph"],
    AUSTIN23_LOAD["turb_observed"],
    mode="calendar",
)


# ==================================================================================================
# 16. TONGJI
# ==================================================================================================

TONGJI23_FILE = m23_find_file(
    ["tongji"],
    preferred=[
        "/content/Tongji_DWTP_Generalization_Ready_W10.xlsx",
    ],
)

if TONGJI23_FILE is None:
    raise FileNotFoundError(
        "Tongji workbook not found."
    )

TONGJI23_LOAD = m23_scan_excel(
    TONGJI23_FILE,

    date_names=[
        "date",
    ],

    temp_names=[
        "temperature",
        "temperature_c",
    ],

    turb_input_names=[
        "raw_turbidity",
        "turbidity",
        "turbidity_observed",
    ],

    ph_names=[
        "ph",
        "pH",
    ],
)

TONGJI23_SEQ = m23_build_sequence(
    TONGJI23_LOAD["df"],
    TONGJI23_LOAD["date"],
    TONGJI23_LOAD["temp"],
    TONGJI23_LOAD["turb_input"],
    TONGJI23_LOAD["ph"],
    TONGJI23_LOAD["turb_observed"],
    mode="calendar",
)


# ==================================================================================================
# 17. HANNAN
#
# INPUT  = Turbidity_Input
# TARGET = Turbidity_Observed
# ==================================================================================================

HANNAN23_FILE = m23_find_file(
    ["hannan"],
    preferred=[
        "/content/Hannan_Village_Generalization_Ready_W10.xlsx",
    ],
)

if HANNAN23_FILE is None:
    raise FileNotFoundError(
        "Hannan workbook not found."
    )

HANNAN23_LOAD = m23_scan_excel(
    HANNAN23_FILE,

    date_names=[
        "date",
    ],

    temp_names=[
        "temperature",
        "temperature_c",
    ],

    turb_input_names=[
        "turbidity_input",
        "raw_turbidity",
        "turbidity",
    ],

    ph_names=[
        "ph",
        "pH",
    ],

    turb_observed_names=[
        "turbidity_observed",
        "raw_turbidity_observed",
        "turbidity",
    ],
)

HANNAN23_SEQ = m23_build_sequence(
    HANNAN23_LOAD["df"],
    HANNAN23_LOAD["date"],
    HANNAN23_LOAD["temp"],
    HANNAN23_LOAD["turb_input"],
    HANNAN23_LOAD["ph"],
    HANNAN23_LOAD["turb_observed"],
    mode="calendar",
)


# ==================================================================================================
# 18. KIRKUK
# ==================================================================================================

KIRKUK23_FILE = m23_find_file(
    ["kirkuk"],
    preferred=[
        "/content/Kirkuk_WTP_Generalization_Ready_W10.xlsx",
    ],
)

if KIRKUK23_FILE is None:
    raise FileNotFoundError(
        "Kirkuk workbook not found."
    )

KXLS = pd.ExcelFile(
    KIRKUK23_FILE
)

if "Daily Processed" not in (
    KXLS.sheet_names
):
    raise KeyError(
        "Kirkuk: Daily Processed sheet not found."
    )

KRAW = pd.read_excel(
    KIRKUK23_FILE,
    sheet_name="Daily Processed",
)

K_DATE = m23_find_col(
    KRAW,
    ["date"],
)

K_TEMP = m23_find_col(
    KRAW,
    [
        "temperature_c",
        "temperature",
    ],
)

K_TURB = m23_find_col(
    KRAW,
    [
        "raw_turbidity",
        "turbidity",
    ],
)

K_PH = m23_find_col(
    KRAW,
    [
        "raw_ph",
        "ph",
    ],
)

if any(
    c is None
    for c in [
        K_DATE,
        K_TEMP,
        K_TURB,
        K_PH,
    ]
):
    raise KeyError(
        "Kirkuk columns unresolved."
    )

KIRKUK23_SEQ = m23_build_sequence(
    KRAW,
    K_DATE,
    K_TEMP,
    K_TURB,
    K_PH,
    K_TURB,
    mode="order",
)


# ==================================================================================================
# 19. DIGITAL TWIN
#
# IMPORTANT FIX:
# INPUT  = raw_turbidity_input when available
# TARGET = raw_turbidity
#
# target_observed is NOT used as numeric turbidity ground truth.
# ==================================================================================================

DIGITAL23_FILE = m23_find_file(
    [],
    preferred=[
        "/content/Virtual_DWTP_C_Data.xlsx",
    ],
)

if DIGITAL23_FILE is None:

    candidates = [
        p
        for p in glob.glob(
            "/content/*.xlsx"
        )
        if (
            "digital" in os.path.basename(
                p
            ).lower()
            or
            "virtual" in os.path.basename(
                p
            ).lower()
        )
    ]

    DIGITAL23_FILE = (
        candidates[0]
        if candidates
        else None
    )

if DIGITAL23_FILE is None:
    raise FileNotFoundError(
        "Digital Twin workbook not found."
    )

DIGITAL23_LOAD = m23_scan_excel(
    DIGITAL23_FILE,

    date_names=[
        "date",
    ],

    temp_names=[
        "temperature",
        "temperature_c",
    ],

    turb_input_names=[
        "raw_turbidity_input",
        "raw_turbidity",
    ],

    ph_names=[
        "ph",
        "pH",
    ],

    turb_observed_names=[
        "raw_turbidity",
    ],
)

DIGITAL23_SEQ = m23_build_sequence(
    DIGITAL23_LOAD["df"],
    DIGITAL23_LOAD["date"],
    DIGITAL23_LOAD["temp"],
    DIGITAL23_LOAD["turb_input"],
    DIGITAL23_LOAD["ph"],
    DIGITAL23_LOAD["turb_observed"],
    mode="calendar",
)


# ==================================================================================================
# 20. EXTERNAL REGISTRY
# ==================================================================================================

DATASETS23 = {
    "Austin": {
        "Sequence":
            AUSTIN23_SEQ,

        "Type":
            "Real",

        "Unit_Note":
            "FNU domain",
    },

    "Tongji": {
        "Sequence":
            TONGJI23_SEQ,

        "Type":
            "Real",

        "Unit_Note":
            "Published turbidity",
    },

    "Hannan": {
        "Sequence":
            HANNAN23_SEQ,

        "Type":
            "Real",

        "Unit_Note":
            "Observed turbidity",
    },

    "Kirkuk": {
        "Sequence":
            KIRKUK23_SEQ,

        "Type":
            "Real",

        "Unit_Note":
            "Analytical sequence",
    },

    "DigitalTwin": {
        "Sequence":
            DIGITAL23_SEQ,

        "Type":
            "Simulated",

        "Unit_Note":
            "Simulated",
    },
}


print(
    "\nExternal sequence sizes:"
)

for name, obj in (
    DATASETS23.items()
):

    print(
        f"{name:12s}: "
        f"{len(obj['Sequence']):,}"
    )


# ==================================================================================================
# 21. EVALUATION
# ==================================================================================================

RESULTS23 = []
PREDICTIONS23 = []
RELATION23 = []

for dataset_name, obj in (
    DATASETS23.items()
):

    full = (
        obj[
            "Sequence"
        ]
        .reset_index(drop=True)
    )

    print(
        "\n"
        + "-" * 160
    )

    print(
        "Evaluating:",
        dataset_name
    )

    print(
        "-" * 160
    )

    for h in HORIZONS23:

        task = (
            f"Turbidity_H{h}"
        )

        ycol = (
            f"Y_Turbidity_H{h}"
        )

        dcol = (
            f"Y_DeltaLog_H{h}"
        )

        tindex = (
            f"Target_Index_H{h}"
        )

        frame = (
            full
            .dropna(
                subset=[
                    ycol,
                    dcol,
                    tindex,
                ]
            )
            .reset_index(drop=True)
            .copy()
        )

        n = len(
            frame
        )

        if n < 25:
            raise RuntimeError(
                f"{dataset_name} {task}: "
                f"insufficient rows ({n})."
            )

        test_start_row = int(
            np.floor(
                EXT_CONTEXT_RATIO23
                * n
            )
        )

        test_start_origin = int(
            frame.loc[
                test_start_row,
                "Origin_Index",
            ]
        )

        context = (
            frame.iloc[
                :test_start_row
            ]
            .copy()
        )

        test = (
            frame.iloc[
                test_start_row:
            ]
            .reset_index(drop=True)
            .copy()
        )

        actual = (
            test[ycol]
            .to_numpy(float)
        )

        persistence = (
            test[
                "Current_Turbidity"
            ]
            .to_numpy(float)
        )

        # --------------------------------------------------------------
        # Persistence
        # --------------------------------------------------------------

        pmet = m23_metrics(
            actual,
            persistence,
        )

        RESULTS23.append({
            "Dataset":
                dataset_name,

            "Dataset_Type":
                obj[
                    "Type"
                ],

            "Task":
                task,

            "Horizon":
                h,

            "Setting":
                "Persistence",

            "Calibration_Rows":
                0,

            "Locked_Test_Rows":
                len(test),

            "Domain_Gate":
                np.nan,

            **pmet,
        })

        # --------------------------------------------------------------
        # Zero-shot source model
        # --------------------------------------------------------------

        (
            zero_delta,
            domain_gate,
            relation_weights,
        ) = m23_external_experts(
            task,
            context,
            test,
        )

        for station, weight in (
            relation_weights.items()
        ):

            RELATION23.append({
                "Dataset":
                    dataset_name,

                "Task":
                    task,

                "Source_Station":
                    station,

                "Weight":
                    weight,

                "Domain_Gate":
                    domain_gate,
            })

        # --------------------------------------------------------------
        # Settings
        # --------------------------------------------------------------

        for setting, ratio in (
            SETTINGS23.items()
        ):

            if ratio == 0:

                calibration = None
                cal_n = 0

                delta_pred = (
                    zero_delta.copy()
                )

            else:

                requested_end = int(
                    np.floor(
                        ratio * n
                    )
                )

                candidate = (
                    frame.iloc[
                        :requested_end
                    ]
                    .copy()
                )

                # Prevent target date crossing into final Test
                calibration = (
                    candidate[
                        candidate[
                            tindex
                        ]
                        < test_start_origin
                    ]
                    .reset_index(drop=True)
                    .copy()
                )

                cal_n = len(
                    calibration
                )

                delta_pred = (
                    m23_fewshot_predict(
                        task,
                        calibration,
                        test,
                        context,
                        zero_delta,
                    )
                )

            prediction = (
                m23_reconstruct(
                    persistence,
                    delta_pred,
                )
            )

            met = m23_metrics(
                actual,
                prediction,
                persistence=persistence,
            )

            RESULTS23.append({
                "Dataset":
                    dataset_name,

                "Dataset_Type":
                    obj[
                        "Type"
                    ],

                "Task":
                    task,

                "Horizon":
                    h,

                "Setting":
                    setting,

                "Calibration_Rows":
                    cal_n,

                "Locked_Test_Rows":
                    len(test),

                "Domain_Gate":
                    domain_gate,

                **met,
            })

            PREDICTIONS23.append(
                pd.DataFrame({
                    "Dataset":
                        dataset_name,

                    "Task":
                        task,

                    "Setting":
                        setting,

                    "Date":
                        test[
                            "Date"
                        ].to_numpy(),

                    "Actual":
                        actual,

                    "Persistence":
                        persistence,

                    "Prediction":
                        prediction,
                })
            )


GENERALIZATION23_RESULTS_DF = pd.DataFrame(
    RESULTS23
)

GENERALIZATION23_PREDICTIONS_DF = pd.concat(
    PREDICTIONS23,
    ignore_index=True,
)

GENERALIZATION23_RELATION_DF = pd.DataFrame(
    RELATION23
)


# ==================================================================================================
# 22. DETAILED RESULTS
# ==================================================================================================

print(
    "\n"
    + "=" * 175
)

print(
    "DETAILED EXTERNAL RESULTS"
)

print(
    "=" * 175
)

display(
    GENERALIZATION23_RESULTS_DF[
        [
            "Dataset",
            "Dataset_Type",
            "Task",
            "Setting",
            "Calibration_Rows",
            "Locked_Test_Rows",
            "Domain_Gate",
            "R",
            "R2",
            "MAE",
            "RMSE",
            "MAPE",
            "sMAPE",
            "NMAE",
            "NRMSE",
            "Skill_MAE",
            "Skill_RMSE",
        ]
    ]
    .round(6)
)


# ==================================================================================================
# 23. DATASET SUMMARY
# ==================================================================================================

MODEL_RESULTS23 = (
    GENERALIZATION23_RESULTS_DF[
        GENERALIZATION23_RESULTS_DF[
            "Setting"
        ].ne(
            "Persistence"
        )
    ]
    .copy()
)

GENERALIZATION23_DATASET_SUMMARY_DF = (
    MODEL_RESULTS23
    .groupby(
        [
            "Dataset",
            "Dataset_Type",
            "Setting",
        ],
        as_index=False,
    )
    .agg(
        Mean_R=(
            "R",
            "mean"
        ),

        Mean_R2=(
            "R2",
            "mean"
        ),

        Mean_sMAPE=(
            "sMAPE",
            "mean"
        ),

        Mean_NMAE=(
            "NMAE",
            "mean"
        ),

        Mean_NRMSE=(
            "NRMSE",
            "mean"
        ),

        Mean_Skill_MAE=(
            "Skill_MAE",
            "mean"
        ),

        Mean_Skill_RMSE=(
            "Skill_RMSE",
            "mean"
        ),

        Mean_Domain_Gate=(
            "Domain_Gate",
            "mean"
        ),
    )
)


print(
    "\n"
    + "=" * 175
)

print(
    "DATASET × SETTING SUMMARY"
)

print(
    "=" * 175
)

display(
    GENERALIZATION23_DATASET_SUMMARY_DF
    .round(6)
)


# ==================================================================================================
# 24. REAL-WORLD SUMMARY
# ==================================================================================================

REAL23 = (
    MODEL_RESULTS23[
        MODEL_RESULTS23[
            "Dataset_Type"
        ].eq(
            "Real"
        )
    ]
    .copy()
)

GENERALIZATION23_REAL_SUMMARY_DF = (
    REAL23
    .groupby(
        "Setting",
        as_index=False,
    )
    .agg(
        Mean_R=(
            "R",
            "mean"
        ),

        Mean_R2=(
            "R2",
            "mean"
        ),

        Mean_sMAPE=(
            "sMAPE",
            "mean"
        ),

        Mean_NMAE=(
            "NMAE",
            "mean"
        ),

        Mean_NRMSE=(
            "NRMSE",
            "mean"
        ),

        Mean_Skill_MAE=(
            "Skill_MAE",
            "mean"
        ),

        Mean_Skill_RMSE=(
            "Skill_RMSE",
            "mean"
        ),

        Positive_MAE_Skill=(
            "Skill_MAE",
            lambda x:
                int(
                    np.sum(
                        np.asarray(
                            x,
                            float
                        ) > 0
                    )
                ),
        ),

        Positive_RMSE_Skill=(
            "Skill_RMSE",
            lambda x:
                int(
                    np.sum(
                        np.asarray(
                            x,
                            float
                        ) > 0
                    )
                ),
        ),

        Total_Tasks=(
            "Task",
            "count"
        ),
    )
)


print(
    "\n"
    + "=" * 175
)

print(
    "REAL-WORLD GENERALIZATION SUMMARY"
)

print(
    "Austin + Tongji + Hannan + Kirkuk"
)

print(
    "Digital Twin excluded"
)

print(
    "=" * 175
)

display(
    GENERALIZATION23_REAL_SUMMARY_DF
    .round(6)
)


# ==================================================================================================
# 25. BEST RESULT PER DATASET — DESCRIPTIVE ONLY
# ==================================================================================================

BEST23_ROWS = []

for dataset in [
    "Austin",
    "Tongji",
    "Hannan",
    "Kirkuk",
]:

    q = (
        GENERALIZATION23_DATASET_SUMMARY_DF[
            GENERALIZATION23_DATASET_SUMMARY_DF[
                "Dataset"
            ].eq(dataset)
        ]
        .sort_values(
            [
                "Mean_Skill_MAE",
                "Mean_R2",
                "Mean_sMAPE",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    if len(q):

        BEST23_ROWS.append(
            q.iloc[0]
            .to_dict()
        )


GENERALIZATION23_BEST_BY_DATASET_DF = pd.DataFrame(
    BEST23_ROWS
)


print(
    "\n"
    + "=" * 175
)

print(
    "DESCRIPTIVE BEST OBSERVED SETTING PER REAL DATASET"
)

print(
    "=" * 175
)

display(
    GENERALIZATION23_BEST_BY_DATASET_DF
    .round(6)
)


# ==================================================================================================
# 26. BEST INDIVIDUAL TASKS
# ==================================================================================================

BEST_TASK23 = (
    REAL23
    .sort_values(
        [
            "R2",
            "Skill_MAE",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(15)
    [
        [
            "Dataset",
            "Task",
            "Setting",
            "R",
            "R2",
            "MAE",
            "RMSE",
            "sMAPE",
            "Skill_MAE",
            "Skill_RMSE",
        ]
    ]
)


print(
    "\n"
    + "=" * 175
)

print(
    "TOP REAL-WORLD TASK RESULTS"
)

print(
    "=" * 175
)

display(
    BEST_TASK23
    .round(6)
)


# ==================================================================================================
# 27. DIGITAL TWIN — SEPARATE
# ==================================================================================================

DIGITAL23_SUMMARY_DF = (
    GENERALIZATION23_DATASET_SUMMARY_DF[
        GENERALIZATION23_DATASET_SUMMARY_DF[
            "Dataset"
        ].eq(
            "DigitalTwin"
        )
    ]
    .copy()
)


print(
    "\n"
    + "=" * 175
)

print(
    "DIGITAL TWIN — SEPARATE SUMMARY"
)

print(
    "=" * 175
)

display(
    DIGITAL23_SUMMARY_DF
    .round(6)
)


# ==================================================================================================
# 28. PERSISTENCE BASELINE
# ==================================================================================================

PERSISTENCE23_DF = (
    GENERALIZATION23_RESULTS_DF[
        GENERALIZATION23_RESULTS_DF[
            "Setting"
        ].eq(
            "Persistence"
        )
    ]
    .copy()
)


print(
    "\n"
    + "=" * 175
)

print(
    "PERSISTENCE BASELINE"
)

print(
    "=" * 175
)

display(
    PERSISTENCE23_DF[
        [
            "Dataset",
            "Task",
            "R",
            "R2",
            "MAE",
            "RMSE",
            "sMAPE",
        ]
    ]
    .round(6)
)


# ==================================================================================================
# 29. COPY / PASTE — PER DATASET
# ==================================================================================================

print(
    "\n"
    + "=" * 175
)

print(
    "COPY / PASTE — PER DATASET"
)

print(
    "=" * 175
)

for _, row in (
    GENERALIZATION23_DATASET_SUMMARY_DF
    .sort_values(
        [
            "Dataset",
            "Setting",
        ]
    )
    .iterrows()
):

    print(
        f"{row['Dataset']} | "
        f"{row['Setting']} | "
        f"R={row['Mean_R']:.6f} | "
        f"R2={row['Mean_R2']:.6f} | "
        f"sMAPE={row['Mean_sMAPE']:.6f}% | "
        f"NMAE={row['Mean_NMAE']:.6f} | "
        f"NRMSE={row['Mean_NRMSE']:.6f} | "
        f"SkillMAE={row['Mean_Skill_MAE']:.6f} | "
        f"SkillRMSE={row['Mean_Skill_RMSE']:.6f} | "
        f"DomainGate={row['Mean_Domain_Gate']:.6f}"
    )


# ==================================================================================================
# 30. COPY / PASTE — REAL WORLD
# ==================================================================================================

print(
    "\n"
    + "=" * 175
)

print(
    "COPY / PASTE — REAL-WORLD SUMMARY"
)

print(
    "=" * 175
)

for _, row in (
    GENERALIZATION23_REAL_SUMMARY_DF
    .iterrows()
):

    print(
        f"{row['Setting']} | "
        f"R={row['Mean_R']:.6f} | "
        f"R2={row['Mean_R2']:.6f} | "
        f"sMAPE={row['Mean_sMAPE']:.6f}% | "
        f"NMAE={row['Mean_NMAE']:.6f} | "
        f"NRMSE={row['Mean_NRMSE']:.6f} | "
        f"SkillMAE={row['Mean_Skill_MAE']:.6f} | "
        f"SkillRMSE={row['Mean_Skill_RMSE']:.6f} | "
        f"PositiveMAE={int(row['Positive_MAE_Skill'])}/"
        f"{int(row['Total_Tasks'])} | "
        f"PositiveRMSE={int(row['Positive_RMSE_Skill'])}/"
        f"{int(row['Total_Tasks'])}"
    )


# ==================================================================================================
# 31. FINAL AUDIT
# ==================================================================================================

print(
    "\n"
    + "=" * 175
)

print(
    "CELL 23 COMPLETED"
)

print(
    "=" * 175
)

print(
    "Unified Cell                       : True"
)

print(
    "UTW                               : W10"
)

print(
    "Federated source training          : FedAdam"
)

print(
    "Transfer formulation               : Persistence-anchored residual forecasting"
)

print(
    "Relation guidance                  : True"
)

print(
    "Domain-similarity gate             : True"
)

print(
    "Source-only zero-shot bounds       : True"
)

print(
    "Few-shot residual adapter          : True"
)

print(
    "Zero-shot                          : True"
)

print(
    "Few-shot 10%                       : True"
)

print(
    "Few-shot 20%                       : True"
)

print(
    "Same locked final 80% Test         : True"
)

print(
    "Persistence benchmark              : True"
)

print(
    "Digital Twin target corrected      : raw_turbidity"
)

print(
    "Digital Twin reported separately   : True"
)

print(
    "Raw MAE pooled across stations     : False"
)

print(
    f"Execution time: "
    f"{time.time() - CELL23_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 24 — EXTERNAL GENERALIZATION ANALYSIS
#
# Computes:
#   1) Zero-shot -> Few-shot improvement for every Station × Horizon
#   2) Best setting for every Station × Horizon
#   3) Generalization gain versus Persistence
#   4) Station ranking for H1 / H2 / H3 separately
#   5) Final publication-ready external generalization table
#
# IMPORTANT:
#   - Uses Cell 23 results only.
#   - No model retraining.
#   - No averaging across different stations for scientific ranking.
#   - Digital Twin remains separated as simulated data.
# ==================================================================================================

import time
import numpy as np
import pandas as pd

CELL24_START = time.time()

print("=" * 170)
print("CELL 24 — EXTERNAL GENERALIZATION ANALYSIS")
print("=" * 170)


# ==================================================================================================
# 1. REQUIREMENTS
# ==================================================================================================

if "GENERALIZATION23_RESULTS_DF" not in globals():
    raise NameError(
        "GENERALIZATION23_RESULTS_DF was not found. "
        "Run Cell 23 first."
    )

G24 = GENERALIZATION23_RESULTS_DF.copy()

required_cols = [
    "Dataset",
    "Dataset_Type",
    "Task",
    "Horizon",
    "Setting",
    "R",
    "R2",
    "MAE",
    "RMSE",
    "Skill_MAE",
    "Skill_RMSE",
]

missing_cols = [
    c for c in required_cols
    if c not in G24.columns
]

if missing_cols:
    raise KeyError(
        "Missing columns in GENERALIZATION23_RESULTS_DF: "
        + str(missing_cols)
    )

G24["Horizon_Label"] = (
    "H"
    + G24["Horizon"].astype(int).astype(str)
)

REAL_STATIONS24 = [
    "Austin",
    "Tongji",
    "Hannan",
    "Kirkuk",
]

MODEL_SETTINGS24 = [
    "ZeroShot",
    "FewShot10",
    "FewShot20",
]


# ==================================================================================================
# 2. BASIC HELPERS
# ==================================================================================================

def c24_pct_improvement(old_value, new_value):
    """
    Positive value means reduction/improvement when lower is better.
    """
    if (
        not np.isfinite(old_value)
        or not np.isfinite(new_value)
        or abs(old_value) < 1e-12
    ):
        return np.nan

    return float(
        100.0
        * (
            old_value - new_value
        )
        / abs(old_value)
    )


def c24_r2_gain(old_value, new_value):
    """
    Absolute R2 gain.
    """
    if (
        not np.isfinite(old_value)
        or not np.isfinite(new_value)
    ):
        return np.nan

    return float(
        new_value - old_value
    )


# ==================================================================================================
# 3. POINT 1
# ZERO-SHOT -> FEW-SHOT IMPROVEMENT
# ==================================================================================================

improvement_rows = []

for dataset in G24["Dataset"].unique():

    for h in sorted(
        G24[
            G24["Dataset"].eq(dataset)
        ]["Horizon"].unique()
    ):

        q = G24[
            G24["Dataset"].eq(dataset)
            &
            G24["Horizon"].eq(h)
        ].copy()

        zero = q[
            q["Setting"].eq("ZeroShot")
        ]

        if len(zero) != 1:
            continue

        zero = zero.iloc[0]

        for setting in [
            "FewShot10",
            "FewShot20",
        ]:

            fs = q[
                q["Setting"].eq(setting)
            ]

            if len(fs) != 1:
                continue

            fs = fs.iloc[0]

            improvement_rows.append({
                "Dataset":
                    dataset,

                "Dataset_Type":
                    zero["Dataset_Type"],

                "Horizon":
                    int(h),

                "Horizon_Label":
                    f"H{int(h)}",

                "From":
                    "ZeroShot",

                "To":
                    setting,

                "Zero_R":
                    float(zero["R"]),

                "Few_R":
                    float(fs["R"]),

                "Delta_R":
                    float(fs["R"] - zero["R"]),

                "Zero_R2":
                    float(zero["R2"]),

                "Few_R2":
                    float(fs["R2"]),

                "Delta_R2":
                    c24_r2_gain(
                        zero["R2"],
                        fs["R2"],
                    ),

                "Zero_MAE":
                    float(zero["MAE"]),

                "Few_MAE":
                    float(fs["MAE"]),

                "MAE_Reduction":
                    float(
                        zero["MAE"]
                        - fs["MAE"]
                    ),

                "MAE_Improvement_%":
                    c24_pct_improvement(
                        zero["MAE"],
                        fs["MAE"],
                    ),

                "Zero_RMSE":
                    float(zero["RMSE"]),

                "Few_RMSE":
                    float(fs["RMSE"]),

                "RMSE_Reduction":
                    float(
                        zero["RMSE"]
                        - fs["RMSE"]
                    ),

                "RMSE_Improvement_%":
                    c24_pct_improvement(
                        zero["RMSE"],
                        fs["RMSE"],
                    ),
            })


GENERALIZATION24_IMPROVEMENT_DF = pd.DataFrame(
    improvement_rows
)


print("\n" + "=" * 170)
print("1) ZERO-SHOT -> FEW-SHOT IMPROVEMENT")
print("=" * 170)

display(
    GENERALIZATION24_IMPROVEMENT_DF[
        [
            "Dataset",
            "Horizon_Label",
            "To",
            "Delta_R",
            "Delta_R2",
            "MAE_Improvement_%",
            "RMSE_Improvement_%",
        ]
    ]
    .round(6)
)


# ==================================================================================================
# 4. POINT 2
# BEST SETTING FOR EACH STATION × HORIZON
#
# Ranking priority:
#   1) Higher R2
#   2) Lower RMSE
#   3) Lower MAE
# ==================================================================================================

best_rows = []

for dataset in G24["Dataset"].unique():

    for h in sorted(
        G24[
            G24["Dataset"].eq(dataset)
        ]["Horizon"].unique()
    ):

        q = G24[
            G24["Dataset"].eq(dataset)
            &
            G24["Horizon"].eq(h)
            &
            G24["Setting"].isin(
                MODEL_SETTINGS24
            )
        ].copy()

        if len(q) == 0:
            continue

        q = (
            q
            .sort_values(
                [
                    "R2",
                    "RMSE",
                    "MAE",
                ],
                ascending=[
                    False,
                    True,
                    True,
                ],
            )
            .reset_index(drop=True)
        )

        best = q.iloc[0]

        best_rows.append({
            "Dataset":
                dataset,

            "Dataset_Type":
                best["Dataset_Type"],

            "Horizon":
                int(h),

            "Horizon_Label":
                f"H{int(h)}",

            "Best_Setting":
                best["Setting"],

            "R":
                float(best["R"]),

            "R2":
                float(best["R2"]),

            "MAE":
                float(best["MAE"]),

            "RMSE":
                float(best["RMSE"]),

            "Skill_MAE":
                float(best["Skill_MAE"]),

            "Skill_RMSE":
                float(best["Skill_RMSE"]),
        })


GENERALIZATION24_BEST_SETTING_DF = pd.DataFrame(
    best_rows
)


print("\n" + "=" * 170)
print("2) BEST SETTING FOR EACH STATION × HORIZON")
print("=" * 170)

display(
    GENERALIZATION24_BEST_SETTING_DF
    .sort_values(
        [
            "Dataset",
            "Horizon",
        ]
    )
    [
        [
            "Dataset",
            "Horizon_Label",
            "Best_Setting",
            "R",
            "R2",
            "MAE",
            "RMSE",
            "Skill_MAE",
            "Skill_RMSE",
        ]
    ]
    .round(6)
)


# ==================================================================================================
# 5. POINT 3
# GENERALIZATION GAIN VERSUS PERSISTENCE
#
# Uses Skill directly from Cell 23.
# Also classifies whether model improved MAE / RMSE.
# ==================================================================================================

skill_rows = []

for _, row in G24[
    G24["Setting"].isin(
        MODEL_SETTINGS24
    )
].iterrows():

    skill_mae = float(
        row["Skill_MAE"]
    )

    skill_rmse = float(
        row["Skill_RMSE"]
    )

    skill_rows.append({
        "Dataset":
            row["Dataset"],

        "Dataset_Type":
            row["Dataset_Type"],

        "Horizon":
            int(row["Horizon"]),

        "Horizon_Label":
            f"H{int(row['Horizon'])}",

        "Setting":
            row["Setting"],

        "R":
            float(row["R"]),

        "R2":
            float(row["R2"]),

        "MAE":
            float(row["MAE"]),

        "RMSE":
            float(row["RMSE"]),

        "Skill_MAE":
            skill_mae,

        "Skill_MAE_%":
            100.0 * skill_mae,

        "Skill_RMSE":
            skill_rmse,

        "Skill_RMSE_%":
            100.0 * skill_rmse,

        "Better_MAE_than_Persistence":
            bool(
                np.isfinite(skill_mae)
                and skill_mae > 0
            ),

        "Better_RMSE_than_Persistence":
            bool(
                np.isfinite(skill_rmse)
                and skill_rmse > 0
            ),
    })


GENERALIZATION24_SKILL_DF = pd.DataFrame(
    skill_rows
)


print("\n" + "=" * 170)
print("3) GENERALIZATION GAIN VERSUS PERSISTENCE")
print("=" * 170)

display(
    GENERALIZATION24_SKILL_DF[
        [
            "Dataset",
            "Horizon_Label",
            "Setting",
            "Skill_MAE_%",
            "Skill_RMSE_%",
            "Better_MAE_than_Persistence",
            "Better_RMSE_than_Persistence",
        ]
    ]
    .round(6)
)


# ==================================================================================================
# 6. POINT 4
# STATION RANKING — H1 / H2 / H3 SEPARATELY
#
# REAL STATIONS ONLY.
#
# Two rankings are produced:
#   A) Zero-shot ranking
#   B) Best-setting ranking
#
# Main ranking criterion = R2
# ==================================================================================================

ranking_rows = []

for h in [1, 2, 3]:

    # ----------------------------------------------------------------------------------------------
    # A. Zero-shot ranking
    # ----------------------------------------------------------------------------------------------

    q_zero = G24[
        G24["Dataset"].isin(
            REAL_STATIONS24
        )
        &
        G24["Horizon"].eq(h)
        &
        G24["Setting"].eq(
            "ZeroShot"
        )
    ].copy()

    q_zero = (
        q_zero
        .sort_values(
            [
                "R2",
                "R",
                "RMSE",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    for rank, (_, row) in enumerate(
        q_zero.iterrows(),
        start=1,
    ):

        ranking_rows.append({
            "Ranking_Type":
                "ZeroShot",

            "Horizon":
                h,

            "Horizon_Label":
                f"H{h}",

            "Rank":
                rank,

            "Dataset":
                row["Dataset"],

            "Setting":
                "ZeroShot",

            "R":
                float(row["R"]),

            "R2":
                float(row["R2"]),

            "MAE":
                float(row["MAE"]),

            "RMSE":
                float(row["RMSE"]),
        })

    # ----------------------------------------------------------------------------------------------
    # B. Best-setting ranking
    # ----------------------------------------------------------------------------------------------

    q_best = GENERALIZATION24_BEST_SETTING_DF[
        GENERALIZATION24_BEST_SETTING_DF[
            "Dataset"
        ].isin(
            REAL_STATIONS24
        )
        &
        GENERALIZATION24_BEST_SETTING_DF[
            "Horizon"
        ].eq(h)
    ].copy()

    q_best = (
        q_best
        .sort_values(
            [
                "R2",
                "R",
                "RMSE",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    for rank, (_, row) in enumerate(
        q_best.iterrows(),
        start=1,
    ):

        ranking_rows.append({
            "Ranking_Type":
                "Best_Setting",

            "Horizon":
                h,

            "Horizon_Label":
                f"H{h}",

            "Rank":
                rank,

            "Dataset":
                row["Dataset"],

            "Setting":
                row["Best_Setting"],

            "R":
                float(row["R"]),

            "R2":
                float(row["R2"]),

            "MAE":
                float(row["MAE"]),

            "RMSE":
                float(row["RMSE"]),
        })


GENERALIZATION24_RANKING_DF = pd.DataFrame(
    ranking_rows
)


print("\n" + "=" * 170)
print("4A) REAL-STATION ZERO-SHOT RANKING — EACH HORIZON SEPARATELY")
print("=" * 170)

display(
    GENERALIZATION24_RANKING_DF[
        GENERALIZATION24_RANKING_DF[
            "Ranking_Type"
        ].eq(
            "ZeroShot"
        )
    ]
    [
        [
            "Horizon_Label",
            "Rank",
            "Dataset",
            "R",
            "R2",
            "MAE",
            "RMSE",
        ]
    ]
    .round(6)
)


print("\n" + "=" * 170)
print("4B) REAL-STATION BEST-SETTING RANKING — EACH HORIZON SEPARATELY")
print("=" * 170)

display(
    GENERALIZATION24_RANKING_DF[
        GENERALIZATION24_RANKING_DF[
            "Ranking_Type"
        ].eq(
            "Best_Setting"
        )
    ]
    [
        [
            "Horizon_Label",
            "Rank",
            "Dataset",
            "Setting",
            "R",
            "R2",
            "MAE",
            "RMSE",
        ]
    ]
    .round(6)
)


# ==================================================================================================
# 7. POINT 5
# FINAL PUBLICATION-READY TABLE
#
# No station-level averaging.
# Every Station × Horizon × Setting reported independently.
# ==================================================================================================

GENERALIZATION24_FINAL_TABLE_DF = (
    G24[
        G24["Setting"].isin(
            MODEL_SETTINGS24
        )
    ]
    [
        [
            "Dataset",
            "Dataset_Type",
            "Horizon_Label",
            "Horizon",
            "Setting",
            "R",
            "R2",
            "MAE",
            "RMSE",
            "Skill_MAE",
            "Skill_RMSE",
        ]
    ]
    .copy()
)

setting_order = pd.CategoricalDtype(
    categories=[
        "ZeroShot",
        "FewShot10",
        "FewShot20",
    ],
    ordered=True,
)

GENERALIZATION24_FINAL_TABLE_DF[
    "Setting"
] = GENERALIZATION24_FINAL_TABLE_DF[
    "Setting"
].astype(
    setting_order
)

dataset_order = [
    "Hannan",
    "Tongji",
    "Austin",
    "Kirkuk",
    "DigitalTwin",
]

GENERALIZATION24_FINAL_TABLE_DF[
    "Dataset"
] = pd.Categorical(
    GENERALIZATION24_FINAL_TABLE_DF[
        "Dataset"
    ],
    categories=dataset_order,
    ordered=True,
)

GENERALIZATION24_FINAL_TABLE_DF = (
    GENERALIZATION24_FINAL_TABLE_DF
    .sort_values(
        [
            "Dataset",
            "Horizon",
            "Setting",
        ]
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 170)
print("5) FINAL PUBLICATION-READY EXTERNAL GENERALIZATION TABLE")
print("=" * 170)

display(
    GENERALIZATION24_FINAL_TABLE_DF[
        [
            "Dataset",
            "Dataset_Type",
            "Horizon_Label",
            "Setting",
            "R",
            "R2",
            "MAE",
            "RMSE",
            "Skill_MAE",
            "Skill_RMSE",
        ]
    ]
    .round(6)
)


# ==================================================================================================
# 8. ZERO-SHOT H1 RANKING — SHORT PAPER SUMMARY
# ==================================================================================================

GENERALIZATION24_ZEROSHOT_H1_DF = (
    GENERALIZATION24_RANKING_DF[
        GENERALIZATION24_RANKING_DF[
            "Ranking_Type"
        ].eq(
            "ZeroShot"
        )
        &
        GENERALIZATION24_RANKING_DF[
            "Horizon"
        ].eq(1)
    ]
    .copy()
)


print("\n" + "=" * 170)
print("ZERO-SHOT H1 — REAL STATIONS FROM STRONGEST TO WEAKEST")
print("=" * 170)

display(
    GENERALIZATION24_ZEROSHOT_H1_DF[
        [
            "Rank",
            "Dataset",
            "R",
            "R2",
            "MAE",
            "RMSE",
        ]
    ]
    .round(6)
)


# ==================================================================================================
# 9. DIGITAL TWIN — SEPARATE TABLE
# ==================================================================================================

GENERALIZATION24_DIGITAL_TWIN_DF = (
    GENERALIZATION24_FINAL_TABLE_DF[
        GENERALIZATION24_FINAL_TABLE_DF[
            "Dataset"
        ].astype(str).eq(
            "DigitalTwin"
        )
    ]
    .copy()
)


print("\n" + "=" * 170)
print("DIGITAL TWIN — SIMULATED DATA REPORTED SEPARATELY")
print("=" * 170)

display(
    GENERALIZATION24_DIGITAL_TWIN_DF[
        [
            "Horizon_Label",
            "Setting",
            "R",
            "R2",
            "MAE",
            "RMSE",
            "Skill_MAE",
            "Skill_RMSE",
        ]
    ]
    .round(6)
)


# ==================================================================================================
# 10. COMPACT COPY / PASTE OUTPUT
# ==================================================================================================

print("\n" + "=" * 170)
print("COPY / PASTE — ZERO-SHOT H1 RANKING")
print("=" * 170)

for _, row in (
    GENERALIZATION24_ZEROSHOT_H1_DF
    .iterrows()
):

    print(
        f"#{int(row['Rank'])} "
        f"{row['Dataset']} | "
        f"R={row['R']:.6f} | "
        f"R2={row['R2']:.6f} | "
        f"MAE={row['MAE']:.6f} | "
        f"RMSE={row['RMSE']:.6f}"
    )


print("\n" + "=" * 170)
print("COPY / PASTE — BEST SETTING PER STATION × HORIZON")
print("=" * 170)

for _, row in (
    GENERALIZATION24_BEST_SETTING_DF
    .sort_values(
        [
            "Dataset",
            "Horizon",
        ]
    )
    .iterrows()
):

    print(
        f"{row['Dataset']} | "
        f"{row['Horizon_Label']} | "
        f"{row['Best_Setting']} | "
        f"R={row['R']:.6f} | "
        f"R2={row['R2']:.6f} | "
        f"MAE={row['MAE']:.6f} | "
        f"RMSE={row['RMSE']:.6f} | "
        f"SkillMAE={row['Skill_MAE']:.6f} | "
        f"SkillRMSE={row['Skill_RMSE']:.6f}"
    )


print("\n" + "=" * 170)
print("CELL 24 COMPLETED SUCCESSFULLY")
print("=" * 170)

print("Zero -> Few-shot improvement      : DONE")
print("Best setting per Station × H      : DONE")
print("Persistence Skill analysis        : DONE")
print("Station ranking per Horizon       : DONE")
print("Publication-ready table           : DONE")
print("Cross-station averaging           : NOT USED")
print("Digital Twin separated            : YES")
print(
    f"Execution time: "
    f"{time.time() - CELL24_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 25 A — EXTERNAL GENERALIZATION STATISTICAL SIGNIFICANCE
#
# FINAL VERSION
#
# Primary significance framework:
#   1) Paired Bootstrap effect estimate
#   2) Bootstrap 95% CI determines effect direction
#   3) Holm-adjusted Bootstrap p-value determines multiplicity-adjusted significance
#
# FINAL DECISION:
#
#   Few-shot significant improvement if:
#       MAE_Gain > 0
#       AND MAE_CI_Low > 0
#       AND Holm-adjusted Bootstrap p < 0.05
#
#   Zero-shot significant advantage if:
#       MAE_Gain < 0
#       AND MAE_CI_High < 0
#       AND Holm-adjusted Bootstrap p < 0.05
#
#   Otherwise:
#       No Significant Difference
#
# Wilcoxon:
#   Supporting test only.
#
# IMPORTANT:
#   - Uses Cell 23 predictions only
#   - No retraining
#   - No model-selection changes
#   - No change to numerical performance metrics
#   - Real stations analyzed independently
#   - Digital Twin reported separately
# ==================================================================================================

import time
import numpy as np
import pandas as pd

from scipy.stats import wilcoxon

CELL25_START = time.time()

print("=" * 175)
print("CELL 25 — EXTERNAL GENERALIZATION STATISTICAL SIGNIFICANCE")
print("PRIMARY = PAIRED BOOTSTRAP + HOLM MULTIPLE-COMPARISON CONTROL")
print("=" * 175)


# ==================================================================================================
# 1. REQUIREMENTS
# ==================================================================================================

if "GENERALIZATION23_PREDICTIONS_DF" not in globals():
    raise NameError(
        "GENERALIZATION23_PREDICTIONS_DF was not found. "
        "Run Cell 23 first."
    )

P25 = GENERALIZATION23_PREDICTIONS_DF.copy()

required_cols = [
    "Dataset",
    "Task",
    "Setting",
    "Date",
    "Actual",
    "Persistence",
    "Prediction",
]

missing_cols = [
    c for c in required_cols
    if c not in P25.columns
]

if missing_cols:
    raise KeyError(
        "Missing columns in GENERALIZATION23_PREDICTIONS_DF: "
        + str(missing_cols)
    )


# ==================================================================================================
# 2. SETTINGS
# ==================================================================================================

SEED25 = 42
BOOTSTRAP_REPS25 = 5000
CONFIDENCE25 = 0.95
ALPHA25 = 0.05

REAL_DATASETS25 = [
    "Austin",
    "Tongji",
    "Hannan",
    "Kirkuk",
]

SIMULATED_DATASETS25 = [
    "DigitalTwin",
]

COMPARISONS25 = [
    ("ZeroShot", "FewShot10"),
    ("ZeroShot", "FewShot20"),
]


# ==================================================================================================
# 3. HELPERS
# ==================================================================================================

def c25_parse_horizon(task):
    return int(
        str(task).split("H")[-1]
    )


def c25_holm_adjust(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    adjusted = np.full(
        len(p_values),
        np.nan,
        dtype=float,
    )

    valid_mask = np.isfinite(
        p_values
    )

    valid_indices = np.where(
        valid_mask
    )[0]

    if len(valid_indices) == 0:
        return adjusted

    valid_p = p_values[
        valid_indices
    ]

    order = np.argsort(
        valid_p
    )

    sorted_p = valid_p[
        order
    ]

    m = len(
        sorted_p
    )

    sorted_adjusted = np.empty(
        m,
        dtype=float,
    )

    running_max = 0.0

    for i, p in enumerate(
        sorted_p
    ):

        value = (
            (m - i)
            * p
        )

        running_max = max(
            running_max,
            value,
        )

        sorted_adjusted[i] = min(
            running_max,
            1.0,
        )

    inverse_order = np.empty(
        m,
        dtype=int,
    )

    inverse_order[
        order
    ] = np.arange(m)

    adjusted_valid = sorted_adjusted[
        inverse_order
    ]

    adjusted[
        valid_indices
    ] = adjusted_valid

    return adjusted


def c25_bootstrap_pvalue(samples):

    samples = np.asarray(
        samples,
        dtype=float,
    )

    samples = samples[
        np.isfinite(samples)
    ]

    if len(samples) == 0:
        return np.nan

    left = float(
        np.mean(
            samples <= 0.0
        )
    )

    right = float(
        np.mean(
            samples >= 0.0
        )
    )

    # +1 correction prevents an exact Monte-Carlo zero.
    count_tail = min(
        np.sum(samples <= 0.0),
        np.sum(samples >= 0.0),
    )

    p = (
        2.0
        * (
            count_tail + 1.0
        )
        / (
            len(samples) + 1.0
        )
    )

    return float(
        min(
            p,
            1.0,
        )
    )


def c25_paired_bootstrap(
    actual,
    pred_zero,
    pred_few,
    reps=5000,
    seed=42,
):

    actual = np.asarray(
        actual,
        dtype=float,
    )

    pred_zero = np.asarray(
        pred_zero,
        dtype=float,
    )

    pred_few = np.asarray(
        pred_few,
        dtype=float,
    )

    valid = (
        np.isfinite(actual)
        &
        np.isfinite(pred_zero)
        &
        np.isfinite(pred_few)
    )

    actual = actual[
        valid
    ]

    pred_zero = pred_zero[
        valid
    ]

    pred_few = pred_few[
        valid
    ]

    n = len(
        actual
    )

    if n < 5:
        return {
            "N": n,

            "MAE_Gain":
                np.nan,

            "MAE_CI_Low":
                np.nan,

            "MAE_CI_High":
                np.nan,

            "MAE_Bootstrap_p":
                np.nan,

            "RMSE_Gain":
                np.nan,

            "RMSE_CI_Low":
                np.nan,

            "RMSE_CI_High":
                np.nan,

            "RMSE_Bootstrap_p":
                np.nan,
        }

    error_zero = (
        pred_zero
        - actual
    )

    error_few = (
        pred_few
        - actual
    )

    abs_zero = np.abs(
        error_zero
    )

    abs_few = np.abs(
        error_few
    )

    sq_zero = (
        error_zero ** 2
    )

    sq_few = (
        error_few ** 2
    )

    mae_gain = float(
        np.mean(
            abs_zero
            - abs_few
        )
    )

    rmse_zero = float(
        np.sqrt(
            np.mean(
                sq_zero
            )
        )
    )

    rmse_few = float(
        np.sqrt(
            np.mean(
                sq_few
            )
        )
    )

    rmse_gain = float(
        rmse_zero
        - rmse_few
    )

    rng = np.random.default_rng(
        seed
    )

    bootstrap_mae_gain = np.empty(
        reps,
        dtype=float,
    )

    bootstrap_rmse_gain = np.empty(
        reps,
        dtype=float,
    )

    for b in range(
        reps
    ):

        index = rng.integers(
            0,
            n,
            size=n,
        )

        bootstrap_mae_gain[b] = float(
            np.mean(
                abs_zero[index]
                - abs_few[index]
            )
        )

        bootstrap_zero_rmse = float(
            np.sqrt(
                np.mean(
                    sq_zero[index]
                )
            )
        )

        bootstrap_few_rmse = float(
            np.sqrt(
                np.mean(
                    sq_few[index]
                )
            )
        )

        bootstrap_rmse_gain[b] = float(
            bootstrap_zero_rmse
            - bootstrap_few_rmse
        )

    alpha = (
        1.0
        - CONFIDENCE25
    )

    mae_ci_low = float(
        np.quantile(
            bootstrap_mae_gain,
            alpha / 2.0,
        )
    )

    mae_ci_high = float(
        np.quantile(
            bootstrap_mae_gain,
            1.0 - alpha / 2.0,
        )
    )

    rmse_ci_low = float(
        np.quantile(
            bootstrap_rmse_gain,
            alpha / 2.0,
        )
    )

    rmse_ci_high = float(
        np.quantile(
            bootstrap_rmse_gain,
            1.0 - alpha / 2.0,
        )
    )

    mae_p = c25_bootstrap_pvalue(
        bootstrap_mae_gain
    )

    rmse_p = c25_bootstrap_pvalue(
        bootstrap_rmse_gain
    )

    return {
        "N":
            int(n),

        "MAE_Gain":
            mae_gain,

        "MAE_CI_Low":
            mae_ci_low,

        "MAE_CI_High":
            mae_ci_high,

        "MAE_Bootstrap_p":
            mae_p,

        "RMSE_Gain":
            rmse_gain,

        "RMSE_CI_Low":
            rmse_ci_low,

        "RMSE_CI_High":
            rmse_ci_high,

        "RMSE_Bootstrap_p":
            rmse_p,
    }


def c25_wilcoxon(
    actual,
    pred_zero,
    pred_few,
):

    actual = np.asarray(
        actual,
        dtype=float,
    )

    pred_zero = np.asarray(
        pred_zero,
        dtype=float,
    )

    pred_few = np.asarray(
        pred_few,
        dtype=float,
    )

    valid = (
        np.isfinite(actual)
        &
        np.isfinite(pred_zero)
        &
        np.isfinite(pred_few)
    )

    actual = actual[
        valid
    ]

    pred_zero = pred_zero[
        valid
    ]

    pred_few = pred_few[
        valid
    ]

    abs_zero = np.abs(
        pred_zero
        - actual
    )

    abs_few = np.abs(
        pred_few
        - actual
    )

    difference = (
        abs_zero
        - abs_few
    )

    nonzero_count = int(
        np.sum(
            np.abs(
                difference
            ) > 1e-12
        )
    )

    if nonzero_count < 5:

        return {
            "Wilcoxon_Statistic":
                np.nan,

            "Wilcoxon_p":
                np.nan,
        }

    try:

        statistic, p_value = wilcoxon(
            abs_zero,
            abs_few,
            alternative="two-sided",
            zero_method="wilcox",
        )

        return {
            "Wilcoxon_Statistic":
                float(statistic),

            "Wilcoxon_p":
                float(p_value),
        }

    except Exception:

        return {
            "Wilcoxon_Statistic":
                np.nan,

            "Wilcoxon_p":
                np.nan,
        }


# ==================================================================================================
# 4. BUILD PAIRED TEST-SET COMPARISONS
# ==================================================================================================

rows25 = []

datasets25 = list(
    P25["Dataset"].dropna().unique()
)

tasks25 = sorted(
    P25["Task"].dropna().unique()
)

for dataset_index, dataset in enumerate(
    datasets25
):

    for task in tasks25:

        task_data = P25[
            P25["Dataset"].eq(
                dataset
            )
            &
            P25["Task"].eq(
                task
            )
        ].copy()

        if len(
            task_data
        ) == 0:
            continue

        for (
            base_setting,
            few_setting,
        ) in COMPARISONS25:

            zero_data = (
                task_data[
                    task_data[
                        "Setting"
                    ].eq(
                        base_setting
                    )
                ]
                [
                    [
                        "Date",
                        "Actual",
                        "Prediction",
                    ]
                ]
                .rename(
                    columns={
                        "Prediction":
                            "Prediction_Zero"
                    }
                )
            )

            few_data = (
                task_data[
                    task_data[
                        "Setting"
                    ].eq(
                        few_setting
                    )
                ]
                [
                    [
                        "Date",
                        "Actual",
                        "Prediction",
                    ]
                ]
                .rename(
                    columns={
                        "Prediction":
                            "Prediction_Few"
                    }
                )
            )

            paired = pd.merge(
                zero_data,
                few_data,
                on=[
                    "Date",
                    "Actual",
                ],
                how="inner",
            )

            if len(
                paired
            ) == 0:
                continue

            actual = paired[
                "Actual"
            ].to_numpy(
                dtype=float
            )

            prediction_zero = paired[
                "Prediction_Zero"
            ].to_numpy(
                dtype=float
            )

            prediction_few = paired[
                "Prediction_Few"
            ].to_numpy(
                dtype=float
            )

            horizon = c25_parse_horizon(
                task
            )

            bootstrap_seed = (
                SEED25
                + 1000
                * dataset_index
                + 100
                * horizon
                + (
                    10
                    if few_setting == "FewShot10"
                    else 20
                )
            )

            bootstrap_result = c25_paired_bootstrap(
                actual=actual,
                pred_zero=prediction_zero,
                pred_few=prediction_few,
                reps=BOOTSTRAP_REPS25,
                seed=bootstrap_seed,
            )

            wilcoxon_result = c25_wilcoxon(
                actual=actual,
                pred_zero=prediction_zero,
                pred_few=prediction_few,
            )

            zero_mae = float(
                np.mean(
                    np.abs(
                        prediction_zero
                        - actual
                    )
                )
            )

            few_mae = float(
                np.mean(
                    np.abs(
                        prediction_few
                        - actual
                    )
                )
            )

            zero_rmse = float(
                np.sqrt(
                    np.mean(
                        (
                            prediction_zero
                            - actual
                        ) ** 2
                    )
                )
            )

            few_rmse = float(
                np.sqrt(
                    np.mean(
                        (
                            prediction_few
                            - actual
                        ) ** 2
                    )
                )
            )

            mae_gain_percentage = (
                100.0
                * (
                    zero_mae
                    - few_mae
                )
                / zero_mae
                if zero_mae > 1e-12
                else np.nan
            )

            rmse_gain_percentage = (
                100.0
                * (
                    zero_rmse
                    - few_rmse
                )
                / zero_rmse
                if zero_rmse > 1e-12
                else np.nan
            )

            dataset_type = (
                "Simulated"
                if dataset
                in SIMULATED_DATASETS25
                else "Real"
            )

            rows25.append(
                {
                    "Dataset":
                        dataset,

                    "Dataset_Type":
                        dataset_type,

                    "Task":
                        task,

                    "Horizon":
                        horizon,

                    "Comparison":
                        f"{few_setting} vs ZeroShot",

                    "Few_Setting":
                        few_setting,

                    "N":
                        int(
                            bootstrap_result[
                                "N"
                            ]
                        ),

                    "Zero_MAE":
                        zero_mae,

                    "Few_MAE":
                        few_mae,

                    "MAE_Gain":
                        bootstrap_result[
                            "MAE_Gain"
                        ],

                    "MAE_Gain_%":
                        mae_gain_percentage,

                    "MAE_CI_Low":
                        bootstrap_result[
                            "MAE_CI_Low"
                        ],

                    "MAE_CI_High":
                        bootstrap_result[
                            "MAE_CI_High"
                        ],

                    "MAE_Bootstrap_p":
                        bootstrap_result[
                            "MAE_Bootstrap_p"
                        ],

                    "Zero_RMSE":
                        zero_rmse,

                    "Few_RMSE":
                        few_rmse,

                    "RMSE_Gain":
                        bootstrap_result[
                            "RMSE_Gain"
                        ],

                    "RMSE_Gain_%":
                        rmse_gain_percentage,

                    "RMSE_CI_Low":
                        bootstrap_result[
                            "RMSE_CI_Low"
                        ],

                    "RMSE_CI_High":
                        bootstrap_result[
                            "RMSE_CI_High"
                        ],

                    "RMSE_Bootstrap_p":
                        bootstrap_result[
                            "RMSE_Bootstrap_p"
                        ],

                    "Wilcoxon_Statistic":
                        wilcoxon_result[
                            "Wilcoxon_Statistic"
                        ],

                    "Wilcoxon_p":
                        wilcoxon_result[
                            "Wilcoxon_p"
                        ],
                }
            )


GENERALIZATION25_SIGNIFICANCE_DF = pd.DataFrame(
    rows25
)


# ==================================================================================================
# 5. HOLM CORRECTION — REAL STATIONS ONLY
# ==================================================================================================

for column in [
    "MAE_Bootstrap_p_Holm",
    "RMSE_Bootstrap_p_Holm",
    "Wilcoxon_p_Holm",
]:
    GENERALIZATION25_SIGNIFICANCE_DF[
        column
    ] = np.nan


real_mask25 = (
    GENERALIZATION25_SIGNIFICANCE_DF[
        "Dataset_Type"
    ].eq(
        "Real"
    )
)

real_indices25 = (
    GENERALIZATION25_SIGNIFICANCE_DF[
        real_mask25
    ].index
)


GENERALIZATION25_SIGNIFICANCE_DF.loc[
    real_indices25,
    "MAE_Bootstrap_p_Holm",
] = c25_holm_adjust(
    GENERALIZATION25_SIGNIFICANCE_DF.loc[
        real_indices25,
        "MAE_Bootstrap_p",
    ].to_numpy(
        dtype=float
    )
)


GENERALIZATION25_SIGNIFICANCE_DF.loc[
    real_indices25,
    "RMSE_Bootstrap_p_Holm",
] = c25_holm_adjust(
    GENERALIZATION25_SIGNIFICANCE_DF.loc[
        real_indices25,
        "RMSE_Bootstrap_p",
    ].to_numpy(
        dtype=float
    )
)


GENERALIZATION25_SIGNIFICANCE_DF.loc[
    real_indices25,
    "Wilcoxon_p_Holm",
] = c25_holm_adjust(
    GENERALIZATION25_SIGNIFICANCE_DF.loc[
        real_indices25,
        "Wilcoxon_p",
    ].to_numpy(
        dtype=float
    )
)


# ==================================================================================================
# 6. FINAL PRIMARY SIGNIFICANCE RULE
# ==================================================================================================

def c25_final_conclusion(row):

    mae_gain = row[
        "MAE_Gain"
    ]

    ci_low = row[
        "MAE_CI_Low"
    ]

    ci_high = row[
        "MAE_CI_High"
    ]

    holm_p = row[
        "MAE_Bootstrap_p_Holm"
    ]

    if not (
        np.isfinite(
            mae_gain
        )
        and np.isfinite(
            ci_low
        )
        and np.isfinite(
            ci_high
        )
        and np.isfinite(
            holm_p
        )
    ):
        return (
            "Insufficient Statistical Evidence"
        )

    if (
        mae_gain > 0.0
        and ci_low > 0.0
        and holm_p < ALPHA25
    ):
        return (
            "Significant Few-shot Improvement"
        )

    if (
        mae_gain < 0.0
        and ci_high < 0.0
        and holm_p < ALPHA25
    ):
        return (
            "Significant Zero-shot Advantage"
        )

    return (
        "No Significant Difference"
    )


GENERALIZATION25_SIGNIFICANCE_DF[
    "Primary_Conclusion"
] = GENERALIZATION25_SIGNIFICANCE_DF.apply(
    c25_final_conclusion,
    axis=1,
)


# ==================================================================================================
# 7. SUPPORTING WILCOXON INTERPRETATION
# ==================================================================================================

def c25_supporting_status(row):

    primary = row[
        "Primary_Conclusion"
    ]

    wilcoxon_p = row[
        "Wilcoxon_p_Holm"
    ]

    wilcoxon_significant = (
        np.isfinite(
            wilcoxon_p
        )
        and wilcoxon_p < ALPHA25
    )

    if (
        primary
        ==
        "Significant Few-shot Improvement"
    ):

        if wilcoxon_significant:
            return (
                "Primary + Wilcoxon agree"
            )

        return (
            "Primary significant; Wilcoxon not significant"
        )

    if (
        primary
        ==
        "Significant Zero-shot Advantage"
    ):

        if wilcoxon_significant:
            return (
                "Primary + Wilcoxon agree"
            )

        return (
            "Primary significant; Wilcoxon not significant"
        )

    if wilcoxon_significant:
        return (
            "Wilcoxon-only signal"
        )

    return (
        "No supporting significance"
    )


GENERALIZATION25_SIGNIFICANCE_DF[
    "Supporting_Status"
] = GENERALIZATION25_SIGNIFICANCE_DF.apply(
    c25_supporting_status,
    axis=1,
)


# ==================================================================================================
# 8. REAL-STATION TABLE
# ==================================================================================================

GENERALIZATION25_REAL_DF = (
    GENERALIZATION25_SIGNIFICANCE_DF[
        GENERALIZATION25_SIGNIFICANCE_DF[
            "Dataset_Type"
        ].eq(
            "Real"
        )
    ]
    .copy()
    .sort_values(
        [
            "Dataset",
            "Horizon",
            "Few_Setting",
        ]
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 175)
print("REAL STATIONS — FINAL MULTIPLICITY-ADJUSTED SIGNIFICANCE")
print("=" * 175)

display(
    GENERALIZATION25_REAL_DF[
        [
            "Dataset",
            "Task",
            "Comparison",
            "N",

            "Zero_MAE",
            "Few_MAE",
            "MAE_Gain_%",
            "MAE_CI_Low",
            "MAE_CI_High",
            "MAE_Bootstrap_p_Holm",

            "Zero_RMSE",
            "Few_RMSE",
            "RMSE_Gain_%",
            "RMSE_CI_Low",
            "RMSE_CI_High",
            "RMSE_Bootstrap_p_Holm",

            "Wilcoxon_p_Holm",
            "Primary_Conclusion",
            "Supporting_Status",
        ]
    ].round(
        6
    )
)


# ==================================================================================================
# 9. SIGNIFICANT FEW-SHOT IMPROVEMENTS
# ==================================================================================================

GENERALIZATION25_SIGNIFICANT_IMPROVEMENTS_DF = (
    GENERALIZATION25_REAL_DF[
        GENERALIZATION25_REAL_DF[
            "Primary_Conclusion"
        ].eq(
            "Significant Few-shot Improvement"
        )
    ]
    .copy()
)


print("\n" + "=" * 175)
print("SIGNIFICANT FEW-SHOT IMPROVEMENTS")
print("=" * 175)

if len(
    GENERALIZATION25_SIGNIFICANT_IMPROVEMENTS_DF
) > 0:

    display(
        GENERALIZATION25_SIGNIFICANT_IMPROVEMENTS_DF[
            [
                "Dataset",
                "Task",
                "Few_Setting",

                "Zero_MAE",
                "Few_MAE",
                "MAE_Gain_%",

                "Zero_RMSE",
                "Few_RMSE",
                "RMSE_Gain_%",

                "MAE_CI_Low",
                "MAE_CI_High",
                "MAE_Bootstrap_p_Holm",

                "Supporting_Status",
            ]
        ].round(
            6
        )
    )

else:

    print(
        "No significant Few-shot improvements."
    )


# ==================================================================================================
# 10. SIGNIFICANT ZERO-SHOT ADVANTAGE
# ==================================================================================================

GENERALIZATION25_ZEROSHOT_BETTER_DF = (
    GENERALIZATION25_REAL_DF[
        GENERALIZATION25_REAL_DF[
            "Primary_Conclusion"
        ].eq(
            "Significant Zero-shot Advantage"
        )
    ]
    .copy()
)


print("\n" + "=" * 175)
print("SIGNIFICANT ZERO-SHOT ADVANTAGE")
print("=" * 175)

if len(
    GENERALIZATION25_ZEROSHOT_BETTER_DF
) > 0:

    display(
        GENERALIZATION25_ZEROSHOT_BETTER_DF[
            [
                "Dataset",
                "Task",
                "Few_Setting",

                "Zero_MAE",
                "Few_MAE",
                "MAE_Gain_%",

                "Zero_RMSE",
                "Few_RMSE",
                "RMSE_Gain_%",

                "MAE_CI_Low",
                "MAE_CI_High",
                "MAE_Bootstrap_p_Holm",

                "Supporting_Status",
            ]
        ].round(
            6
        )
    )

else:

    print(
        "No significant Zero-shot advantages."
    )


# ==================================================================================================
# 11. NON-SIGNIFICANT COMPARISONS
# ==================================================================================================

GENERALIZATION25_NONSIGNIFICANT_DF = (
    GENERALIZATION25_REAL_DF[
        GENERALIZATION25_REAL_DF[
            "Primary_Conclusion"
        ].eq(
            "No Significant Difference"
        )
    ]
    .copy()
)


print("\n" + "=" * 175)
print("NO SIGNIFICANT DIFFERENCE")
print("=" * 175)

display(
    GENERALIZATION25_NONSIGNIFICANT_DF[
        [
            "Dataset",
            "Task",
            "Few_Setting",
            "MAE_Gain_%",
            "RMSE_Gain_%",
            "MAE_CI_Low",
            "MAE_CI_High",
            "MAE_Bootstrap_p_Holm",
            "Wilcoxon_p_Holm",
            "Supporting_Status",
        ]
    ].round(
        6
    )
)


# ==================================================================================================
# 12. DIGITAL TWIN — SEPARATE SIMULATED ANALYSIS
#
# No Holm mixing with real stations.
# Report raw Bootstrap CI + raw p-value.
# ==================================================================================================

GENERALIZATION25_DIGITAL_TWIN_DF = (
    GENERALIZATION25_SIGNIFICANCE_DF[
        GENERALIZATION25_SIGNIFICANCE_DF[
            "Dataset_Type"
        ].eq(
            "Simulated"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


def c25_simulated_conclusion(row):

    if (
        row["MAE_Gain"] > 0
        and row["MAE_CI_Low"] > 0
        and row["MAE_Bootstrap_p"] < ALPHA25
    ):
        return (
            "Significant Few-shot Improvement"
        )

    if (
        row["MAE_Gain"] < 0
        and row["MAE_CI_High"] < 0
        and row["MAE_Bootstrap_p"] < ALPHA25
    ):
        return (
            "Significant Zero-shot Advantage"
        )

    return (
        "No Significant Difference"
    )


if len(
    GENERALIZATION25_DIGITAL_TWIN_DF
) > 0:

    GENERALIZATION25_DIGITAL_TWIN_DF[
        "Simulated_Conclusion"
    ] = GENERALIZATION25_DIGITAL_TWIN_DF.apply(
        c25_simulated_conclusion,
        axis=1,
    )


print("\n" + "=" * 175)
print("DIGITAL TWIN — SIMULATED DATA REPORTED SEPARATELY")
print("=" * 175)

if len(
    GENERALIZATION25_DIGITAL_TWIN_DF
) > 0:

    display(
        GENERALIZATION25_DIGITAL_TWIN_DF[
            [
                "Task",
                "Comparison",
                "N",

                "Zero_MAE",
                "Few_MAE",
                "MAE_Gain_%",

                "Zero_RMSE",
                "Few_RMSE",
                "RMSE_Gain_%",

                "MAE_CI_Low",
                "MAE_CI_High",
                "MAE_Bootstrap_p",

                "Wilcoxon_p",
                "Simulated_Conclusion",
            ]
        ].round(
            6
        )
    )


# ==================================================================================================
# 13. PUBLICATION-READY SUMMARY TABLE
# ==================================================================================================

GENERALIZATION25_SUMMARY_DF = (
    GENERALIZATION25_REAL_DF[
        [
            "Dataset",
            "Task",
            "Comparison",

            "Zero_MAE",
            "Few_MAE",
            "MAE_Gain_%",

            "Zero_RMSE",
            "Few_RMSE",
            "RMSE_Gain_%",

            "MAE_CI_Low",
            "MAE_CI_High",

            "MAE_Bootstrap_p_Holm",
            "Wilcoxon_p_Holm",

            "Primary_Conclusion",
            "Supporting_Status",
        ]
    ]
    .copy()
)


print("\n" + "=" * 175)
print("FINAL PUBLICATION-READY STATISTICAL SUMMARY")
print("=" * 175)

display(
    GENERALIZATION25_SUMMARY_DF.round(
        6
    )
)


# ==================================================================================================
# 14. FINAL COUNTS
# ==================================================================================================

n_fewshot_significant = int(
    np.sum(
        GENERALIZATION25_SUMMARY_DF[
            "Primary_Conclusion"
        ].eq(
            "Significant Few-shot Improvement"
        )
    )
)

n_zeroshot_significant = int(
    np.sum(
        GENERALIZATION25_SUMMARY_DF[
            "Primary_Conclusion"
        ].eq(
            "Significant Zero-shot Advantage"
        )
    )
)

n_no_difference = int(
    np.sum(
        GENERALIZATION25_SUMMARY_DF[
            "Primary_Conclusion"
        ].eq(
            "No Significant Difference"
        )
    )
)


print("\n" + "=" * 175)
print("FINAL PRIMARY STATISTICAL COUNTS")
print("=" * 175)

print(
    f"Significant Few-shot Improvement : "
    f"{n_fewshot_significant}"
)

print(
    f"Significant Zero-shot Advantage  : "
    f"{n_zeroshot_significant}"
)

print(
    f"No Significant Difference        : "
    f"{n_no_difference}"
)


# ==================================================================================================
# 15. STRONGEST NUMERICAL FEW-SHOT GAINS
# ==================================================================================================

strongest25 = (
    GENERALIZATION25_REAL_DF[
        GENERALIZATION25_REAL_DF[
            "MAE_Gain_%"
        ] > 0
    ]
    .copy()
    .sort_values(
        [
            "MAE_Gain_%",
            "RMSE_Gain_%",
        ],
        ascending=[
            False,
            False,
        ],
    )
)


print("\n" + "=" * 175)
print("STRONGEST NUMERICAL FEW-SHOT GAINS")
print("=" * 175)

display(
    strongest25[
        [
            "Dataset",
            "Task",
            "Few_Setting",

            "MAE_Gain_%",
            "RMSE_Gain_%",

            "MAE_CI_Low",
            "MAE_CI_High",

            "MAE_Bootstrap_p_Holm",
            "Primary_Conclusion",
        ]
    ].round(
        6
    )
)


# ==================================================================================================
# 16. COPY / PASTE RESULTS
# ==================================================================================================

print("\n" + "=" * 175)
print("COPY / PASTE — FINAL REAL-STATION SIGNIFICANCE")
print("=" * 175)

for _, row in (
    GENERALIZATION25_SUMMARY_DF
    .iterrows()
):

    print(
        f"{row['Dataset']} | "
        f"{row['Task']} | "
        f"{row['Comparison']} | "
        f"Zero MAE={row['Zero_MAE']:.6f} | "
        f"Few MAE={row['Few_MAE']:.6f} | "
        f"MAE Gain={row['MAE_Gain_%']:.3f}% | "
        f"Zero RMSE={row['Zero_RMSE']:.6f} | "
        f"Few RMSE={row['Few_RMSE']:.6f} | "
        f"RMSE Gain={row['RMSE_Gain_%']:.3f}% | "
        f"MAE CI=[{row['MAE_CI_Low']:.6f}, "
        f"{row['MAE_CI_High']:.6f}] | "
        f"Holm Bootstrap p="
        f"{row['MAE_Bootstrap_p_Holm']:.6f} | "
        f"Holm Wilcoxon p="
        f"{row['Wilcoxon_p_Holm']:.6f} | "
        f"{row['Primary_Conclusion']}"
    )


# ==================================================================================================
# 17. FINAL STATUS
# ==================================================================================================

print("\n" + "=" * 175)
print("CELL 25A COMPLETED SUCCESSFULLY")
print("=" * 175)

print(
    "Primary effect estimator          : "
    "Paired Bootstrap"
)

print(
    "Bootstrap repetitions             :",
    BOOTSTRAP_REPS25,
)

print(
    "Confidence level                  :",
    CONFIDENCE25,
)

print(
    "Multiplicity control              : "
    "Holm"
)

print(
    "Final significance threshold      :",
    ALPHA25,
)

print(
    "Wilcoxon role                     : "
    "Supporting only"
)

print(
    "Real stations analyzed separately : YES"
)

print(
    "Digital Twin separated            : YES"
)

print(
    "Model retraining                  : NO"
)

print(
    f"Execution time: "
    f"{time.time() - CELL25_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 25B — ZERO-SHOT H1 EXTERNAL STATISTICAL VALIDATION
# FINAL SCIENTIFIC VERSION
#
# PRIMARY OBJECTIVE:
#   Evaluate the numerical strength and statistical stability of ZERO-SHOT H1 external generalization.
#
# PRIMARY RESULTS:
#   R, R², MAE, RMSE + 95% Moving-Block-Bootstrap confidence intervals.
#
# SECONDARY ANALYSIS:
#   Zero-shot H1 versus Persistence.
#
# WHY MOVING BLOCK BOOTSTRAP?
#   External observations are time-series data. Ordinary iid bootstrap breaks temporal dependence.
#   Circular moving-block bootstrap preserves local temporal dependence and gives more defensible CIs.
#
# IMPORTANT:
#   - ZERO-SHOT only
#   - H1 only
#   - No retraining
#   - No model-selection changes
#   - No cross-station averaging
#   - Digital Twin included and explicitly identified as Simulated
#   - Persistence is a supporting baseline, NOT the main Zero-shot quality criterion
# ==================================================================================================

import time
import numpy as np
import pandas as pd

from scipy.stats import wilcoxon

CELL25B_START = time.time()

print("=" * 175)
print("CELL 25B — ZERO-SHOT H1 EXTERNAL STATISTICAL VALIDATION")
print("FINAL VERSION — MOVING BLOCK BOOTSTRAP")
print("=" * 175)


# ==================================================================================================
# 1. REQUIREMENTS
# ==================================================================================================

if "GENERALIZATION23_PREDICTIONS_DF" not in globals():
    raise NameError(
        "GENERALIZATION23_PREDICTIONS_DF was not found. "
        "Run Cell 23 first."
    )

P25B = GENERALIZATION23_PREDICTIONS_DF.copy()

required_cols = [
    "Dataset",
    "Task",
    "Setting",
    "Date",
    "Actual",
    "Persistence",
    "Prediction",
]

missing_cols = [
    c for c in required_cols
    if c not in P25B.columns
]

if missing_cols:
    raise KeyError(
        "Missing columns in GENERALIZATION23_PREDICTIONS_DF: "
        + str(missing_cols)
    )


# ==================================================================================================
# 2. FIXED STATISTICAL PROTOCOL
# ==================================================================================================

SEED25B = 42

BOOTSTRAP_REPS25B = 5000
CONFIDENCE25B = 0.95
ALPHA25B = 0.05

TARGET_TASK25B = "Turbidity_H1"
TARGET_SETTING25B = "ZeroShot"

SIMULATED_DATASETS25B = [
    "DigitalTwin"
]

# Numerical-equivalence tolerance.
# Only prevents floating-point noise from being interpreted as a real performance difference.
ABS_TOL25B = 1e-12
REL_TOL25B = 1e-9


# ==================================================================================================
# 3. FILTER ZERO-SHOT H1 ONLY
# ==================================================================================================

Z25B = P25B[
    P25B["Task"].eq(TARGET_TASK25B)
    &
    P25B["Setting"].eq(TARGET_SETTING25B)
].copy()

if len(Z25B) == 0:
    raise ValueError(
        "No ZeroShot H1 rows were found."
    )

Z25B["Date"] = pd.to_datetime(
    Z25B["Date"],
    errors="coerce"
)

for c in [
    "Actual",
    "Prediction",
    "Persistence",
]:
    Z25B[c] = pd.to_numeric(
        Z25B[c],
        errors="coerce"
    )

Z25B = Z25B[
    Z25B[
        [
            "Actual",
            "Prediction",
            "Persistence",
        ]
    ].notna().all(axis=1)
].copy()

print(
    f"Rows retained for Zero-shot H1: {len(Z25B)}"
)

print(
    "Datasets:",
    sorted(
        Z25B["Dataset"].dropna().unique()
    )
)


# ==================================================================================================
# 4. METRIC HELPERS
# ==================================================================================================

def c25b_r(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    mask = (
        np.isfinite(y_true)
        &
        np.isfinite(y_pred)
    )

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) < 3:
        return np.nan

    if (
        np.std(y_true) <= 1e-15
        or
        np.std(y_pred) <= 1e-15
    ):
        return np.nan

    return float(
        np.corrcoef(
            y_true,
            y_pred
        )[0, 1]
    )


def c25b_r2(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    mask = (
        np.isfinite(y_true)
        &
        np.isfinite(y_pred)
    )

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) < 2:
        return np.nan

    ss_res = float(
        np.sum(
            (y_true - y_pred) ** 2
        )
    )

    ss_tot = float(
        np.sum(
            (
                y_true
                - np.mean(y_true)
            ) ** 2
        )
    )

    if ss_tot <= 1e-15:
        return np.nan

    return float(
        1.0
        - ss_res / ss_tot
    )


def c25b_mae(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    return float(
        np.mean(
            np.abs(
                y_true - y_pred
            )
        )
    )


def c25b_rmse(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    return float(
        np.sqrt(
            np.mean(
                (
                    y_true - y_pred
                ) ** 2
            )
        )
    )


# ==================================================================================================
# 5. NUMERICAL EQUIVALENCE
# ==================================================================================================

def c25b_tolerance(reference):

    reference = abs(
        float(reference)
    )

    return max(
        ABS_TOL25B,
        REL_TOL25B * max(
            reference,
            1.0
        )
    )


def c25b_effectively_equal(a, b):

    tol = c25b_tolerance(
        max(
            abs(float(a)),
            abs(float(b)),
        )
    )

    return bool(
        abs(
            float(a) - float(b)
        ) <= tol
    )


# ==================================================================================================
# 6. CIRCULAR MOVING-BLOCK BOOTSTRAP INDICES
#
# Fixed non-tuned block rule:
#   block length = round(n^(1/3))
#
# This is a deterministic statistical protocol choice, not model tuning.
# ==================================================================================================

def c25b_block_length(n):

    return int(
        max(
            5,
            min(
                n,
                round(
                    n ** (1.0 / 3.0)
                )
            )
        )
    )


def c25b_circular_block_indices(
    n,
    block_length,
    rng
):

    n_blocks = int(
        np.ceil(
            n / block_length
        )
    )

    starts = rng.integers(
        0,
        n,
        size=n_blocks
    )

    indices = []

    offsets = np.arange(
        block_length
    )

    for start in starts:

        block = (
            start + offsets
        ) % n

        indices.extend(
            block.tolist()
        )

    return np.asarray(
        indices[:n],
        dtype=int
    )


# ==================================================================================================
# 7. EMPIRICAL TWO-SIDED BOOTSTRAP P-VALUE
# ==================================================================================================

def c25b_bootstrap_pvalue(samples):

    samples = np.asarray(
        samples,
        dtype=float
    )

    samples = samples[
        np.isfinite(samples)
    ]

    if len(samples) == 0:
        return np.nan

    negative_or_zero = int(
        np.sum(
            samples <= 0.0
        )
    )

    positive_or_zero = int(
        np.sum(
            samples >= 0.0
        )
    )

    smaller_tail = min(
        negative_or_zero,
        positive_or_zero
    )

    # +1 finite Monte-Carlo correction
    p = (
        2.0
        * (
            smaller_tail + 1
        )
        / (
            len(samples) + 1
        )
    )

    return float(
        min(
            p,
            1.0
        )
    )


# ==================================================================================================
# 8. HOLM MULTIPLE-COMPARISON CORRECTION
# ==================================================================================================

def c25b_holm_adjust(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    adjusted = np.full(
        len(p_values),
        np.nan,
        dtype=float
    )

    valid = np.isfinite(
        p_values
    )

    valid_idx = np.where(
        valid
    )[0]

    if len(valid_idx) == 0:
        return adjusted

    pv = p_values[
        valid_idx
    ]

    order = np.argsort(
        pv
    )

    sorted_p = pv[
        order
    ]

    m = len(
        sorted_p
    )

    sorted_adj = np.empty(
        m,
        dtype=float
    )

    running_max = 0.0

    for i, p in enumerate(
        sorted_p
    ):

        adjusted_p = (
            (m - i)
            * p
        )

        running_max = max(
            running_max,
            adjusted_p
        )

        sorted_adj[i] = min(
            running_max,
            1.0
        )

    inverse = np.empty(
        m,
        dtype=int
    )

    inverse[
        order
    ] = np.arange(
        m
    )

    adjusted[
        valid_idx
    ] = sorted_adj[
        inverse
    ]

    return adjusted


# ==================================================================================================
# 9. MOVING-BLOCK BOOTSTRAP ANALYSIS
# ==================================================================================================

def c25b_analyze_dataset(
    actual,
    prediction,
    persistence,
    reps,
    seed
):

    actual = np.asarray(
        actual,
        dtype=float
    )

    prediction = np.asarray(
        prediction,
        dtype=float
    )

    persistence = np.asarray(
        persistence,
        dtype=float
    )

    valid = (
        np.isfinite(actual)
        &
        np.isfinite(prediction)
        &
        np.isfinite(persistence)
    )

    actual = actual[valid]
    prediction = prediction[valid]
    persistence = persistence[valid]

    n = len(actual)

    if n < 10:
        raise ValueError(
            "Too few observations for Cell 25B."
        )

    block_length = c25b_block_length(
        n
    )

    # ----------------------------------------------------------------------------------------------
    # Original Zero-shot metrics
    # ----------------------------------------------------------------------------------------------

    r = c25b_r(
        actual,
        prediction
    )

    r2 = c25b_r2(
        actual,
        prediction
    )

    mae = c25b_mae(
        actual,
        prediction
    )

    rmse = c25b_rmse(
        actual,
        prediction
    )

    # ----------------------------------------------------------------------------------------------
    # Persistence metrics
    # ----------------------------------------------------------------------------------------------

    persistence_mae = c25b_mae(
        actual,
        persistence
    )

    persistence_rmse = c25b_rmse(
        actual,
        persistence
    )

    mae_gain = float(
        persistence_mae
        - mae
    )

    rmse_gain = float(
        persistence_rmse
        - rmse
    )

    skill_mae = (
        1.0
        - mae / persistence_mae
        if persistence_mae > 1e-15
        else np.nan
    )

    skill_rmse = (
        1.0
        - rmse / persistence_rmse
        if persistence_rmse > 1e-15
        else np.nan
    )

    # ----------------------------------------------------------------------------------------------
    # Exact/effective equivalence check
    # ----------------------------------------------------------------------------------------------

    mae_equivalent = c25b_effectively_equal(
        mae,
        persistence_mae
    )

    rmse_equivalent = c25b_effectively_equal(
        rmse,
        persistence_rmse
    )

    predictions_equivalent = bool(
        np.allclose(
            prediction,
            persistence,
            rtol=REL_TOL25B,
            atol=ABS_TOL25B,
            equal_nan=True
        )
    )

    baseline_equivalent = bool(
        mae_equivalent
        and rmse_equivalent
        and predictions_equivalent
    )

    # ----------------------------------------------------------------------------------------------
    # Bootstrap arrays
    # ----------------------------------------------------------------------------------------------

    rng = np.random.default_rng(
        seed
    )

    boot_r = np.full(
        reps,
        np.nan,
        dtype=float
    )

    boot_r2 = np.full(
        reps,
        np.nan,
        dtype=float
    )

    boot_mae = np.full(
        reps,
        np.nan,
        dtype=float
    )

    boot_rmse = np.full(
        reps,
        np.nan,
        dtype=float
    )

    boot_skill_mae = np.full(
        reps,
        np.nan,
        dtype=float
    )

    boot_skill_rmse = np.full(
        reps,
        np.nan,
        dtype=float
    )

    boot_mae_gain = np.full(
        reps,
        np.nan,
        dtype=float
    )

    boot_rmse_gain = np.full(
        reps,
        np.nan,
        dtype=float
    )

    for b in range(
        reps
    ):

        idx = c25b_circular_block_indices(
            n=n,
            block_length=block_length,
            rng=rng
        )

        yb = actual[idx]
        pb = prediction[idx]
        persb = persistence[idx]

        boot_r[b] = c25b_r(
            yb,
            pb
        )

        boot_r2[b] = c25b_r2(
            yb,
            pb
        )

        mae_b = c25b_mae(
            yb,
            pb
        )

        rmse_b = c25b_rmse(
            yb,
            pb
        )

        persistence_mae_b = c25b_mae(
            yb,
            persb
        )

        persistence_rmse_b = c25b_rmse(
            yb,
            persb
        )

        boot_mae[b] = mae_b
        boot_rmse[b] = rmse_b

        boot_mae_gain[b] = (
            persistence_mae_b
            - mae_b
        )

        boot_rmse_gain[b] = (
            persistence_rmse_b
            - rmse_b
        )

        if persistence_mae_b > 1e-15:

            boot_skill_mae[b] = (
                1.0
                - mae_b
                / persistence_mae_b
            )

        if persistence_rmse_b > 1e-15:

            boot_skill_rmse[b] = (
                1.0
                - rmse_b
                / persistence_rmse_b
            )

    # ----------------------------------------------------------------------------------------------
    # Confidence interval helper
    # ----------------------------------------------------------------------------------------------

    alpha = (
        1.0
        - CONFIDENCE25B
    )

    def ci(values):

        values = np.asarray(
            values,
            dtype=float
        )

        values = values[
            np.isfinite(values)
        ]

        if len(values) == 0:
            return (
                np.nan,
                np.nan
            )

        return (
            float(
                np.quantile(
                    values,
                    alpha / 2.0
                )
            ),
            float(
                np.quantile(
                    values,
                    1.0 - alpha / 2.0
                )
            )
        )

    r_low, r_high = ci(
        boot_r
    )

    r2_low, r2_high = ci(
        boot_r2
    )

    mae_low, mae_high = ci(
        boot_mae
    )

    rmse_low, rmse_high = ci(
        boot_rmse
    )

    skill_mae_low, skill_mae_high = ci(
        boot_skill_mae
    )

    skill_rmse_low, skill_rmse_high = ci(
        boot_skill_rmse
    )

    mae_gain_low, mae_gain_high = ci(
        boot_mae_gain
    )

    rmse_gain_low, rmse_gain_high = ci(
        boot_rmse_gain
    )

    # ----------------------------------------------------------------------------------------------
    # Baseline p-values
    #
    # If the predictions are numerically equivalent to Persistence,
    # the scientifically correct interpretation is equivalence/no detectable difference.
    # ----------------------------------------------------------------------------------------------

    if baseline_equivalent:

        mae_gain_p = 1.0
        rmse_gain_p = 1.0

    else:

        mae_gain_p = c25b_bootstrap_pvalue(
            boot_mae_gain
        )

        rmse_gain_p = c25b_bootstrap_pvalue(
            boot_rmse_gain
        )

    # ----------------------------------------------------------------------------------------------
    # Wilcoxon supporting test
    # ----------------------------------------------------------------------------------------------

    abs_zero = np.abs(
        prediction
        - actual
    )

    abs_persistence = np.abs(
        persistence
        - actual
    )

    diff = (
        abs_persistence
        - abs_zero
    )

    tol = c25b_tolerance(
        np.mean(
            abs_persistence
        )
    )

    if np.all(
        np.abs(diff) <= tol
    ):

        wilcoxon_stat = np.nan
        wilcoxon_p = 1.0

    else:

        try:

            wilcoxon_stat, wilcoxon_p = wilcoxon(
                abs_persistence,
                abs_zero,
                alternative="two-sided",
                zero_method="wilcox"
            )

            wilcoxon_stat = float(
                wilcoxon_stat
            )

            wilcoxon_p = float(
                wilcoxon_p
            )

        except Exception:

            wilcoxon_stat = np.nan
            wilcoxon_p = np.nan

    return {
        "N":
            int(n),

        "Block_Length":
            int(block_length),

        "R":
            r,

        "R_CI_Low":
            r_low,

        "R_CI_High":
            r_high,

        "R2":
            r2,

        "R2_CI_Low":
            r2_low,

        "R2_CI_High":
            r2_high,

        "MAE":
            mae,

        "MAE_CI_Low":
            mae_low,

        "MAE_CI_High":
            mae_high,

        "RMSE":
            rmse,

        "RMSE_CI_Low":
            rmse_low,

        "RMSE_CI_High":
            rmse_high,

        "Persistence_MAE":
            persistence_mae,

        "Persistence_RMSE":
            persistence_rmse,

        "MAE_Gain_vs_Persistence":
            mae_gain,

        "RMSE_Gain_vs_Persistence":
            rmse_gain,

        "Skill_MAE":
            skill_mae,

        "Skill_RMSE":
            skill_rmse,

        "Skill_MAE_CI_Low":
            skill_mae_low,

        "Skill_MAE_CI_High":
            skill_mae_high,

        "Skill_RMSE_CI_Low":
            skill_rmse_low,

        "Skill_RMSE_CI_High":
            skill_rmse_high,

        "MAE_Gain_CI_Low":
            mae_gain_low,

        "MAE_Gain_CI_High":
            mae_gain_high,

        "RMSE_Gain_CI_Low":
            rmse_gain_low,

        "RMSE_Gain_CI_High":
            rmse_gain_high,

        "MAE_Gain_p":
            mae_gain_p,

        "RMSE_Gain_p":
            rmse_gain_p,

        "Wilcoxon_Statistic":
            wilcoxon_stat,

        "Wilcoxon_p":
            wilcoxon_p,

        "Baseline_Equivalent":
            baseline_equivalent,
    }


# ==================================================================================================
# 10. RUN DATASET-BY-DATASET
# ==================================================================================================

rows25b = []

datasets25b = sorted(
    Z25B[
        "Dataset"
    ].dropna().unique()
)

for dataset_index, dataset in enumerate(
    datasets25b
):

    q = Z25B[
        Z25B[
            "Dataset"
        ].eq(
            dataset
        )
    ].copy()

    # Chronological ordering is essential for moving-block bootstrap.
    q = q.sort_values(
        "Date",
        kind="mergesort"
    ).reset_index(
        drop=True
    )

    actual = q[
        "Actual"
    ].to_numpy(
        dtype=float
    )

    prediction = q[
        "Prediction"
    ].to_numpy(
        dtype=float
    )

    persistence = q[
        "Persistence"
    ].to_numpy(
        dtype=float
    )

    result = c25b_analyze_dataset(
        actual=actual,
        prediction=prediction,
        persistence=persistence,
        reps=BOOTSTRAP_REPS25B,
        seed=(
            SEED25B
            + dataset_index * 1000
        )
    )

    dataset_type = (
        "Simulated"
        if dataset
        in SIMULATED_DATASETS25B
        else "Real"
    )

    rows25b.append(
        {
            "Dataset":
                dataset,

            "Dataset_Type":
                dataset_type,

            **result,
        }
    )


GENERALIZATION25B_ZEROSHOT_H1_DF = pd.DataFrame(
    rows25b
)


# ==================================================================================================
# 11. HOLM CORRECTION FOR REAL DATASETS
#
# Digital Twin stays separate because it is simulated.
# ==================================================================================================

GENERALIZATION25B_ZEROSHOT_H1_DF[
    "MAE_Gain_p_Holm"
] = np.nan

GENERALIZATION25B_ZEROSHOT_H1_DF[
    "RMSE_Gain_p_Holm"
] = np.nan

GENERALIZATION25B_ZEROSHOT_H1_DF[
    "Wilcoxon_p_Holm"
] = np.nan


real_mask25b = (
    GENERALIZATION25B_ZEROSHOT_H1_DF[
        "Dataset_Type"
    ].eq(
        "Real"
    )
)

real_idx25b = (
    GENERALIZATION25B_ZEROSHOT_H1_DF[
        real_mask25b
    ].index
)


GENERALIZATION25B_ZEROSHOT_H1_DF.loc[
    real_idx25b,
    "MAE_Gain_p_Holm"
] = c25b_holm_adjust(
    GENERALIZATION25B_ZEROSHOT_H1_DF.loc[
        real_idx25b,
        "MAE_Gain_p"
    ].to_numpy(
        dtype=float
    )
)


GENERALIZATION25B_ZEROSHOT_H1_DF.loc[
    real_idx25b,
    "RMSE_Gain_p_Holm"
] = c25b_holm_adjust(
    GENERALIZATION25B_ZEROSHOT_H1_DF.loc[
        real_idx25b,
        "RMSE_Gain_p"
    ].to_numpy(
        dtype=float
    )
)


GENERALIZATION25B_ZEROSHOT_H1_DF.loc[
    real_idx25b,
    "Wilcoxon_p_Holm"
] = c25b_holm_adjust(
    GENERALIZATION25B_ZEROSHOT_H1_DF.loc[
        real_idx25b,
        "Wilcoxon_p"
    ].to_numpy(
        dtype=float
    )
)


# ==================================================================================================
# 12. SUPPORTING PERSISTENCE CONCLUSION
#
# This conclusion is SECONDARY.
# The primary conclusion of Cell 25B is the numerical Zero-shot H1 performance itself.
# ==================================================================================================

def c25b_persistence_conclusion(row):

    if bool(
        row[
            "Baseline_Equivalent"
        ]
    ):
        return (
            "Equivalent to Persistence"
        )

    if row[
        "Dataset_Type"
    ] == "Real":

        p_value = row[
            "MAE_Gain_p_Holm"
        ]

    else:

        p_value = row[
            "MAE_Gain_p"
        ]

    if not np.isfinite(
        p_value
    ):
        return (
            "No Significant Difference"
        )

    tol = c25b_tolerance(
        row[
            "Persistence_MAE"
        ]
    )

    gain = row[
        "MAE_Gain_vs_Persistence"
    ]

    ci_low = row[
        "MAE_Gain_CI_Low"
    ]

    ci_high = row[
        "MAE_Gain_CI_High"
    ]

    if (
        gain > tol
        and ci_low > 0.0
        and p_value < ALPHA25B
    ):
        return (
            "Zero-shot Significantly Better"
        )

    if (
        gain < -tol
        and ci_high < 0.0
        and p_value < ALPHA25B
    ):
        return (
            "Persistence Significantly Better"
        )

    return (
        "No Significant Difference"
    )


GENERALIZATION25B_ZEROSHOT_H1_DF[
    "Persistence_Conclusion"
] = (
    GENERALIZATION25B_ZEROSHOT_H1_DF.apply(
        c25b_persistence_conclusion,
        axis=1
    )
)


# ==================================================================================================
# 13. PERCENT SKILLS
# ==================================================================================================

GENERALIZATION25B_ZEROSHOT_H1_DF[
    "Skill_MAE_%"
] = (
    100.0
    * GENERALIZATION25B_ZEROSHOT_H1_DF[
        "Skill_MAE"
    ]
)

GENERALIZATION25B_ZEROSHOT_H1_DF[
    "Skill_RMSE_%"
] = (
    100.0
    * GENERALIZATION25B_ZEROSHOT_H1_DF[
        "Skill_RMSE"
    ]
)


# ==================================================================================================
# 14. PRIMARY ZERO-SHOT H1 RANKING
#
# Main criterion = R²
# Supporting criteria = R then RMSE
# ==================================================================================================

GENERALIZATION25B_RANKING_DF = (
    GENERALIZATION25B_ZEROSHOT_H1_DF
    .sort_values(
        [
            "R2",
            "R",
            "RMSE",
        ],
        ascending=[
            False,
            False,
            True,
        ]
    )
    .reset_index(
        drop=True
    )
)

GENERALIZATION25B_RANKING_DF[
    "Rank"
] = np.arange(
    1,
    len(
        GENERALIZATION25B_RANKING_DF
    ) + 1
)


# ==================================================================================================
# 15. PRIMARY TABLE — ZERO-SHOT H1 NUMERICAL STRENGTH
# ==================================================================================================

print("\n" + "=" * 175)
print("PRIMARY RESULT — ZERO-SHOT H1 EXTERNAL GENERALIZATION")
print("=" * 175)

display(
    GENERALIZATION25B_RANKING_DF[
        [
            "Rank",
            "Dataset",
            "Dataset_Type",
            "N",
            "Block_Length",
            "R",
            "R2",
            "MAE",
            "RMSE",
        ]
    ].round(
        6
    )
)


# ==================================================================================================
# 16. PRIMARY STATISTICAL STABILITY TABLE
# ==================================================================================================

print("\n" + "=" * 175)
print("ZERO-SHOT H1 — MOVING-BLOCK-BOOTSTRAP 95% CONFIDENCE INTERVALS")
print("=" * 175)

display(
    GENERALIZATION25B_RANKING_DF[
        [
            "Dataset",
            "Dataset_Type",

            "R",
            "R_CI_Low",
            "R_CI_High",

            "R2",
            "R2_CI_Low",
            "R2_CI_High",

            "MAE",
            "MAE_CI_Low",
            "MAE_CI_High",

            "RMSE",
            "RMSE_CI_Low",
            "RMSE_CI_High",
        ]
    ].round(
        6
    )
)


# ==================================================================================================
# 17. COMPACT PUBLICATION TABLE
# ==================================================================================================

GENERALIZATION25B_PUBLICATION_DF = (
    GENERALIZATION25B_RANKING_DF[
        [
            "Rank",
            "Dataset",
            "Dataset_Type",
            "N",

            "R",
            "R_CI_Low",
            "R_CI_High",

            "R2",
            "R2_CI_Low",
            "R2_CI_High",

            "MAE",
            "RMSE",
        ]
    ]
    .copy()
)


print("\n" + "=" * 175)
print("PUBLICATION-READY — ZERO-SHOT H1 PRIMARY TABLE")
print("=" * 175)

display(
    GENERALIZATION25B_PUBLICATION_DF.round(
        6
    )
)


# ==================================================================================================
# 18. SECONDARY PERSISTENCE ANALYSIS
# ==================================================================================================

print("\n" + "=" * 175)
print("SECONDARY ANALYSIS — ZERO-SHOT H1 VERSUS PERSISTENCE")
print("=" * 175)

display(
    GENERALIZATION25B_RANKING_DF[
        [
            "Dataset",
            "Dataset_Type",

            "MAE",
            "Persistence_MAE",
            "Skill_MAE_%",

            "RMSE",
            "Persistence_RMSE",
            "Skill_RMSE_%",

            "MAE_Gain_CI_Low",
            "MAE_Gain_CI_High",

            "MAE_Gain_p",
            "MAE_Gain_p_Holm",

            "Wilcoxon_p",
            "Wilcoxon_p_Holm",

            "Baseline_Equivalent",
            "Persistence_Conclusion",
        ]
    ].round(
        6
    )
)


# ==================================================================================================
# 19. REAL-WORLD ZERO-SHOT H1 RANKING
# ==================================================================================================

GENERALIZATION25B_REAL_DF = (
    GENERALIZATION25B_RANKING_DF[
        GENERALIZATION25B_RANKING_DF[
            "Dataset_Type"
        ].eq(
            "Real"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

GENERALIZATION25B_REAL_DF[
    "Real_Rank"
] = np.arange(
    1,
    len(
        GENERALIZATION25B_REAL_DF
    ) + 1
)


print("\n" + "=" * 175)
print("REAL EXTERNAL DATASETS — ZERO-SHOT H1 RANKING")
print("=" * 175)

display(
    GENERALIZATION25B_REAL_DF[
        [
            "Real_Rank",
            "Dataset",

            "R",
            "R_CI_Low",
            "R_CI_High",

            "R2",
            "R2_CI_Low",
            "R2_CI_High",

            "MAE",
            "RMSE",
        ]
    ].round(
        6
    )
)


# ==================================================================================================
# 20. DIGITAL TWIN — INCLUDED, BUT CLEARLY IDENTIFIED
# ==================================================================================================

GENERALIZATION25B_DIGITAL_TWIN_DF = (
    GENERALIZATION25B_RANKING_DF[
        GENERALIZATION25B_RANKING_DF[
            "Dataset"
        ].eq(
            "DigitalTwin"
        )
    ]
    .copy()
)


print("\n" + "=" * 175)
print("DIGITAL TWIN — SIMULATED ZERO-SHOT H1 VALIDATION")
print("=" * 175)

if len(
    GENERALIZATION25B_DIGITAL_TWIN_DF
) > 0:

    display(
        GENERALIZATION25B_DIGITAL_TWIN_DF[
            [
                "Dataset",

                "R",
                "R_CI_Low",
                "R_CI_High",

                "R2",
                "R2_CI_Low",
                "R2_CI_High",

                "MAE",
                "RMSE",

                "Persistence_Conclusion",
            ]
        ].round(
            6
        )
    )


# ==================================================================================================
# 21. STRONGEST ZERO-SHOT H1 NUMERICAL RESULTS
# ==================================================================================================

print("\n" + "=" * 175)
print("STRONGEST ZERO-SHOT H1 NUMERICAL RESULTS")
print("=" * 175)

for _, row in (
    GENERALIZATION25B_RANKING_DF
    .iterrows()
):

    note = (
        " [SIMULATED]"
        if row["Dataset_Type"] == "Simulated"
        else ""
    )

    print(
        f"#{int(row['Rank'])} "
        f"{row['Dataset']}{note} | "
        f"R={row['R']:.6f} "
        f"[{row['R_CI_Low']:.6f}, {row['R_CI_High']:.6f}] | "
        f"R2={row['R2']:.6f} "
        f"[{row['R2_CI_Low']:.6f}, {row['R2_CI_High']:.6f}] | "
        f"MAE={row['MAE']:.6f} | "
        f"RMSE={row['RMSE']:.6f}"
    )


# ==================================================================================================
# 22. PERSISTENCE COUNTS — SECONDARY ONLY
# ==================================================================================================

n_zero_better = int(
    np.sum(
        GENERALIZATION25B_RANKING_DF[
            "Persistence_Conclusion"
        ].eq(
            "Zero-shot Significantly Better"
        )
    )
)

n_persistence_better = int(
    np.sum(
        GENERALIZATION25B_RANKING_DF[
            "Persistence_Conclusion"
        ].eq(
            "Persistence Significantly Better"
        )
    )
)

n_equivalent = int(
    np.sum(
        GENERALIZATION25B_RANKING_DF[
            "Persistence_Conclusion"
        ].eq(
            "Equivalent to Persistence"
        )
    )
)

n_no_difference = int(
    np.sum(
        GENERALIZATION25B_RANKING_DF[
            "Persistence_Conclusion"
        ].eq(
            "No Significant Difference"
        )
    )
)


print("\n" + "=" * 175)
print("SECONDARY PERSISTENCE COMPARISON COUNTS")
print("=" * 175)

print(
    f"Zero-shot Significantly Better   : {n_zero_better}"
)

print(
    f"Persistence Significantly Better : {n_persistence_better}"
)

print(
    f"Equivalent                       : {n_equivalent}"
)

print(
    f"No Significant Difference        : {n_no_difference}"
)


# ==================================================================================================
# 23. COPY / PASTE — PRIMARY PAPER RESULTS
# ==================================================================================================

print("\n" + "=" * 175)
print("COPY / PASTE — PRIMARY ZERO-SHOT H1 RESULTS")
print("=" * 175)

for _, row in (
    GENERALIZATION25B_RANKING_DF
    .iterrows()
):

    data_label = (
        "Simulated"
        if row[
            "Dataset_Type"
        ] == "Simulated"
        else "Real"
    )

    print(
        f"{row['Dataset']} | "
        f"{data_label} | "
        f"N={int(row['N'])} | "
        f"R={row['R']:.6f} "
        f"[{row['R_CI_Low']:.6f}, {row['R_CI_High']:.6f}] | "
        f"R2={row['R2']:.6f} "
        f"[{row['R2_CI_Low']:.6f}, {row['R2_CI_High']:.6f}] | "
        f"MAE={row['MAE']:.6f} | "
        f"RMSE={row['RMSE']:.6f}"
    )


# ==================================================================================================
# 24. FINAL STATUS
# ==================================================================================================

print("\n" + "=" * 175)
print("CELL 25B COMPLETED SUCCESSFULLY")
print("=" * 175)

print("Primary focus                     : Zero-shot H1")
print("Primary ranking criterion         : R²")
print("Temporal statistical method       : Circular Moving Block Bootstrap")
print("Bootstrap repetitions             :", BOOTSTRAP_REPS25B)
print("Confidence level                  :", CONFIDENCE25B)
print("Persistence role                  : Secondary baseline")
print("Holm correction                   : Real datasets, persistence tests")
print("Numerical equivalence protection  : YES")
print("Cross-station averaging           : NO")
print("Digital Twin included             : YES")
print("Digital Twin labeled              : Simulated")
print("Model retraining                  : NO")

print(
    f"Execution time: "
    f"{time.time() - CELL25B_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 26A — SECURITY LAYER
# REAL LOCAL TLS TRANSPORT + PAIRWISE-MASK SECURE AGGREGATION
#
# PURPOSE
#   Test a real security layer for the federated workflow inside Google Colab.
#
# COMPONENTS
#   1) Real TLS/HTTPS connection between local FL clients and aggregation server
#   2) Self-signed X.509 certificate generated inside Colab
#   3) TLS certificate verification by the clients
#   4) SHA-256 payload-integrity verification
#   5) Pairwise-mask Secure Aggregation
#   6) Server receives MASKED updates only
#   7) Verification that secure aggregation gives the same aggregate as ordinary aggregation
#
# IMPORTANT
#   - TLS transport is REAL inside the Colab runtime.
#   - The experiment is a local deployment prototype, not a geographically distributed network.
#   - Secure Aggregation is implemented as a pairwise-mask protocol prototype.
#   - No model retraining is performed.
#   - No model accuracy is changed.
# ==================================================================================================

import os
import ssl
import time
import json
import socket
import hashlib
import secrets
import tempfile
import threading
import subprocess
import urllib.request

import numpy as np
import pandas as pd

from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer


CELL26_START = time.time()

print("=" * 175)
print("CELL 26 — SECURITY LAYER")
print("REAL TLS TRANSPORT + SECURE AGGREGATION")
print("=" * 175)


# ==================================================================================================
# 1. FIXED SECURITY SETTINGS
# ==================================================================================================

SECURITY_SEED_26 = 42

TLS_MINIMUM_VERSION_26 = ssl.TLSVersion.TLSv1_2

SECURE_AGG_ATOL_26 = 1e-9
SECURE_AGG_RTOL_26 = 1e-9

HOST_26 = "127.0.0.1"
TLS_HOSTNAME_26 = "localhost"


# ==================================================================================================
# 2. LOAD CLIENT UPDATES IF AVAILABLE
#
# The cell attempts to use update vectors already produced by the federated workflow.
#
# If no explicit stored client updates exist, it creates reproducible protocol-test vectors.
# These fallback vectors are used ONLY to test the security protocol.
# ==================================================================================================

candidate_update_names_26 = [
    "CLIENT_UPDATES",
    "FED_CLIENT_UPDATES",
    "LOCAL_MODEL_UPDATES",
    "FEDERATED_CLIENT_UPDATES",
    "client_updates",
    "federated_client_updates",
]

CLIENT_UPDATES_26 = None
CLIENT_UPDATE_SOURCE_26 = None

for variable_name in candidate_update_names_26:

    if variable_name in globals():

        candidate = globals()[variable_name]

        if isinstance(candidate, dict) and len(candidate) >= 2:

            try:

                test_arrays = {
                    str(k): np.asarray(v, dtype=float).reshape(-1)
                    for k, v in candidate.items()
                }

                dimensions = {
                    len(v)
                    for v in test_arrays.values()
                }

                if (
                    len(dimensions) == 1
                    and all(
                        np.all(np.isfinite(v))
                        for v in test_arrays.values()
                    )
                ):

                    CLIENT_UPDATES_26 = test_arrays
                    CLIENT_UPDATE_SOURCE_26 = variable_name

                    break

            except Exception:
                pass


if CLIENT_UPDATES_26 is None:

    print(
        "NOTE: Stored FL client-update vectors were not found."
    )

    print(
        "A reproducible four-client protocol test will therefore be used."
    )

    demo_rng_26 = np.random.default_rng(
        SECURITY_SEED_26
    )

    client_names_demo_26 = [
        "Kadhimiya",
        "Karama",
        "Resafa",
        "Tarmyya",
    ]

    DEMO_VECTOR_DIM_26 = 512

    CLIENT_UPDATES_26 = {
        client:
            demo_rng_26.normal(
                loc=0.0,
                scale=0.01,
                size=DEMO_VECTOR_DIM_26,
            ).astype(np.float64)

        for client in client_names_demo_26
    }

    CLIENT_UPDATE_SOURCE_26 = (
        "Reproducible security-protocol demonstration vectors"
    )


CLIENT_UPDATES_26 = {
    str(client):
        np.asarray(
            update,
            dtype=np.float64
        ).reshape(-1)

    for client, update in CLIENT_UPDATES_26.items()
}


CLIENTS_26 = sorted(
    CLIENT_UPDATES_26.keys()
)

if len(CLIENTS_26) < 2:

    raise ValueError(
        "Secure Aggregation requires at least two clients."
    )


UPDATE_DIMENSION_26 = len(
    CLIENT_UPDATES_26[
        CLIENTS_26[0]
    ]
)


for client in CLIENTS_26:

    vector = CLIENT_UPDATES_26[
        client
    ]

    if len(vector) != UPDATE_DIMENSION_26:

        raise ValueError(
            "All client-update vectors must have the same dimension."
        )

    if not np.all(
        np.isfinite(vector)
    ):

        raise ValueError(
            f"Non-finite values found for client {client}."
        )


print(
    "\nClient update source :",
    CLIENT_UPDATE_SOURCE_26
)

print(
    "Clients              :",
    CLIENTS_26
)

print(
    "Update dimension     :",
    UPDATE_DIMENSION_26
)


# ==================================================================================================
# 3. CLIENT AGGREGATION WEIGHTS
#
# Use existing FL weights when available.
# Otherwise equal weights are used for the protocol validation.
# ==================================================================================================

candidate_weight_names_26 = [
    "CLIENT_WEIGHTS",
    "FED_CLIENT_WEIGHTS",
    "FEDERATED_CLIENT_WEIGHTS",
    "client_weights",
]

CLIENT_WEIGHTS_26 = None
CLIENT_WEIGHT_SOURCE_26 = None

for variable_name in candidate_weight_names_26:

    if variable_name in globals():

        candidate = globals()[variable_name]

        if isinstance(candidate, dict):

            if all(
                client in candidate
                for client in CLIENTS_26
            ):

                try:

                    CLIENT_WEIGHTS_26 = {
                        client:
                            float(
                                candidate[
                                    client
                                ]
                            )

                        for client in CLIENTS_26
                    }

                    CLIENT_WEIGHT_SOURCE_26 = variable_name

                    break

                except Exception:
                    pass


if CLIENT_WEIGHTS_26 is None:

    CLIENT_WEIGHTS_26 = {
        client: 1.0
        for client in CLIENTS_26
    }

    CLIENT_WEIGHT_SOURCE_26 = "Equal protocol-test weights"


weight_sum_26 = float(
    sum(
        CLIENT_WEIGHTS_26.values()
    )
)

if weight_sum_26 <= 0.0:

    raise ValueError(
        "Invalid aggregation weights."
    )


CLIENT_WEIGHTS_26 = {
    client:
        CLIENT_WEIGHTS_26[
            client
        ]
        / weight_sum_26

    for client in CLIENTS_26
}


print(
    "Weight source         :",
    CLIENT_WEIGHT_SOURCE_26
)


# ==================================================================================================
# 4. ORDINARY WEIGHTED FEDERATED AGGREGATE
#
# This is retained locally ONLY as the numerical reference.
# ==================================================================================================

PLAIN_AGGREGATE_26 = np.zeros(
    UPDATE_DIMENSION_26,
    dtype=np.float64
)


for client in CLIENTS_26:

    PLAIN_AGGREGATE_26 += (
        CLIENT_WEIGHTS_26[
            client
        ]
        *
        CLIENT_UPDATES_26[
            client
        ]
    )


# ==================================================================================================
# 5. CRYPTOGRAPHIC PAIRWISE MASK GENERATOR
#
# Every pair of clients shares a random 256-bit secret.
#
# Client i adds mask Mij.
# Client j subtracts the identical mask Mij.
#
# Therefore:
#
#       Σ masks = 0
#
# and the aggregation server receives:
#
#       Σ(w_i * update_i + mask_i)
#
# which reduces to:
#
#       Σ(w_i * update_i)
#
# without requiring the server to receive the plain individual updates.
# ==================================================================================================

PAIRWISE_SECRETS_26 = {}

for i in range(
    len(CLIENTS_26)
):

    for j in range(
        i + 1,
        len(CLIENTS_26)
    ):

        client_i = CLIENTS_26[i]
        client_j = CLIENTS_26[j]

        PAIRWISE_SECRETS_26[
            (
                client_i,
                client_j,
            )
        ] = secrets.token_bytes(
            32
        )


def c26_secret_to_seed(
    secret,
    client_a,
    client_b,
):

    context = (
        f"{client_a}|{client_b}|"
        f"{UPDATE_DIMENSION_26}"
    ).encode(
        "utf-8"
    )

    digest = hashlib.sha256(
        secret + context
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="big",
        signed=False,
    )


PAIRWISE_MASK_SUM_26 = {
    client:
        np.zeros(
            UPDATE_DIMENSION_26,
            dtype=np.float64
        )

    for client in CLIENTS_26
}


for i in range(
    len(CLIENTS_26)
):

    for j in range(
        i + 1,
        len(CLIENTS_26)
    ):

        client_i = CLIENTS_26[i]
        client_j = CLIENTS_26[j]

        shared_secret = PAIRWISE_SECRETS_26[
            (
                client_i,
                client_j,
            )
        ]

        pair_seed = c26_secret_to_seed(
            shared_secret,
            client_i,
            client_j,
        )

        pair_rng = np.random.default_rng(
            pair_seed
        )

        mask = pair_rng.normal(
            loc=0.0,
            scale=1.0,
            size=UPDATE_DIMENSION_26,
        ).astype(
            np.float64
        )

        PAIRWISE_MASK_SUM_26[
            client_i
        ] += mask

        PAIRWISE_MASK_SUM_26[
            client_j
        ] -= mask


# ==================================================================================================
# 6. PREPARE MASKED CLIENT PAYLOADS
#
# Server will receive these vectors, NOT the plain updates.
# ==================================================================================================

MASKED_UPDATES_26 = {}

for client in CLIENTS_26:

    weighted_update = (
        CLIENT_WEIGHTS_26[
            client
        ]
        *
        CLIENT_UPDATES_26[
            client
        ]
    )

    MASKED_UPDATES_26[
        client
    ] = (
        weighted_update
        +
        PAIRWISE_MASK_SUM_26[
            client
        ]
    )


# ==================================================================================================
# 7. CHECK MASK CANCELLATION LOCALLY
# ==================================================================================================

TOTAL_MASK_26 = np.zeros(
    UPDATE_DIMENSION_26,
    dtype=np.float64
)


for client in CLIENTS_26:

    TOTAL_MASK_26 += (
        PAIRWISE_MASK_SUM_26[
            client
        ]
    )


MASK_RESIDUAL_MAX_26 = float(
    np.max(
        np.abs(
            TOTAL_MASK_26
        )
    )
)


# ==================================================================================================
# 8. GENERATE REAL X.509 CERTIFICATE FOR TLS
# ==================================================================================================

TLS_DIRECTORY_26 = tempfile.mkdtemp(
    prefix="paper3_tls_"
)

CERT_FILE_26 = os.path.join(
    TLS_DIRECTORY_26,
    "server_cert.pem"
)

KEY_FILE_26 = os.path.join(
    TLS_DIRECTORY_26,
    "server_key.pem"
)


openssl_command_26 = [
    "openssl",
    "req",
    "-x509",
    "-newkey",
    "rsa:2048",
    "-sha256",
    "-nodes",
    "-days",
    "1",
    "-keyout",
    KEY_FILE_26,
    "-out",
    CERT_FILE_26,
    "-subj",
    "/CN=localhost",
    "-addext",
    "subjectAltName=DNS:localhost,IP:127.0.0.1",
]


openssl_result_26 = subprocess.run(
    openssl_command_26,
    capture_output=True,
    text=True,
)


if openssl_result_26.returncode != 0:

    raise RuntimeError(
        "OpenSSL certificate generation failed:\n"
        + openssl_result_26.stderr
    )


if not (
    os.path.exists(
        CERT_FILE_26
    )
    and
    os.path.exists(
        KEY_FILE_26
    )
):

    raise RuntimeError(
        "TLS certificate/key files were not created."
    )


# ==================================================================================================
# 9. SERVER-SIDE STORAGE
#
# IMPORTANT:
#   The server stores only MASKED transmitted updates.
# ==================================================================================================

SERVER_RECEIVED_MASKED_26 = {}
SERVER_TLS_RECORDS_26 = []
SERVER_INTEGRITY_RECORDS_26 = []


# ==================================================================================================
# 10. TLS HTTPS AGGREGATION SERVER
# ==================================================================================================

class C26SecureAggregationHandler(
    BaseHTTPRequestHandler
):

    def log_message(
        self,
        format,
        *args
    ):
        return


    def do_GET(
        self
    ):

        if self.path != "/health":

            self.send_response(
                404
            )

            self.end_headers()

            return


        tls_version = (
            self.connection.version()
            if hasattr(
                self.connection,
                "version"
            )
            else None
        )

        cipher_info = (
            self.connection.cipher()
            if hasattr(
                self.connection,
                "cipher"
            )
            else None
        )

        response = json.dumps(
            {
                "status":
                    "secure-server-ready",

                "tls_version":
                    tls_version,

                "cipher":
                    cipher_info[0]
                    if cipher_info
                    else None,
            }
        ).encode(
            "utf-8"
        )

        self.send_response(
            200
        )

        self.send_header(
            "Content-Type",
            "application/json"
        )

        self.send_header(
            "Content-Length",
            str(
                len(response)
            )
        )

        self.end_headers()

        self.wfile.write(
            response
        )


    def do_POST(
        self
    ):

        if self.path != "/secure-update":

            self.send_response(
                404
            )

            self.end_headers()

            return


        client_name = self.headers.get(
            "X-FL-Client"
        )

        expected_hash = self.headers.get(
            "X-Payload-SHA256"
        )

        content_length = int(
            self.headers.get(
                "Content-Length",
                "0"
            )
        )

        payload = self.rfile.read(
            content_length
        )

        actual_hash = hashlib.sha256(
            payload
        ).hexdigest()

        integrity_ok = (
            actual_hash
            ==
            expected_hash
        )


        tls_version = (
            self.connection.version()
            if hasattr(
                self.connection,
                "version"
            )
            else None
        )

        cipher_info = (
            self.connection.cipher()
            if hasattr(
                self.connection,
                "cipher"
            )
            else None
        )


        SERVER_TLS_RECORDS_26.append(
            {
                "Client":
                    client_name,

                "TLS_Version":
                    tls_version,

                "Cipher":
                    (
                        cipher_info[0]
                        if cipher_info
                        else None
                    ),

                "Cipher_Bits":
                    (
                        cipher_info[2]
                        if cipher_info
                        else None
                    ),
            }
        )


        SERVER_INTEGRITY_RECORDS_26.append(
            {
                "Client":
                    client_name,

                "Payload_Bytes":
                    content_length,

                "Expected_SHA256":
                    expected_hash,

                "Received_SHA256":
                    actual_hash,

                "Integrity_OK":
                    integrity_ok,
            }
        )


        if not integrity_ok:

            self.send_response(
                400
            )

            self.end_headers()

            return


        vector = np.frombuffer(
            payload,
            dtype=np.float64
        ).copy()


        if len(vector) != UPDATE_DIMENSION_26:

            self.send_response(
                400
            )

            self.end_headers()

            return


        SERVER_RECEIVED_MASKED_26[
            client_name
        ] = vector


        response = json.dumps(
            {
                "status":
                    "accepted",

                "client":
                    client_name,

                "tls":
                    tls_version,

                "integrity":
                    True,
            }
        ).encode(
            "utf-8"
        )


        self.send_response(
            200
        )

        self.send_header(
            "Content-Type",
            "application/json"
        )

        self.send_header(
            "Content-Length",
            str(
                len(response)
            )
        )

        self.end_headers()

        self.wfile.write(
            response
        )


# ==================================================================================================
# 11. START HTTPS SERVER ON AN AVAILABLE LOCAL PORT
# ==================================================================================================

SERVER_26 = ThreadingHTTPServer(
    (
        HOST_26,
        0,
    ),
    C26SecureAggregationHandler,
)

PORT_26 = SERVER_26.server_address[1]


server_ssl_context_26 = ssl.SSLContext(
    ssl.PROTOCOL_TLS_SERVER
)

server_ssl_context_26.minimum_version = (
    TLS_MINIMUM_VERSION_26
)

server_ssl_context_26.load_cert_chain(
    certfile=CERT_FILE_26,
    keyfile=KEY_FILE_26,
)


SERVER_26.socket = (
    server_ssl_context_26.wrap_socket(
        SERVER_26.socket,
        server_side=True,
    )
)


SERVER_THREAD_26 = threading.Thread(
    target=SERVER_26.serve_forever,
    daemon=True,
)

SERVER_THREAD_26.start()


time.sleep(
    0.3
)


# ==================================================================================================
# 12. VERIFIED TLS CLIENT CONTEXT
#
# The client explicitly trusts the generated experiment certificate.
# Hostname verification remains enabled.
# ==================================================================================================

client_ssl_context_26 = ssl.create_default_context(
    cafile=CERT_FILE_26
)

client_ssl_context_26.minimum_version = (
    TLS_MINIMUM_VERSION_26
)

client_ssl_context_26.check_hostname = True

client_ssl_context_26.verify_mode = (
    ssl.CERT_REQUIRED
)


# ==================================================================================================
# 13. TLS HANDSHAKE / SERVER HEALTH TEST
# ==================================================================================================

health_url_26 = (
    f"https://{TLS_HOSTNAME_26}:"
    f"{PORT_26}/health"
)

health_request_26 = urllib.request.Request(
    health_url_26,
    method="GET",
)

with urllib.request.urlopen(
    health_request_26,
    context=client_ssl_context_26,
    timeout=10,
) as response:

    HEALTH_RESPONSE_26 = json.loads(
        response.read().decode(
            "utf-8"
        )
    )


TLS_HANDSHAKE_OK_26 = (
    HEALTH_RESPONSE_26.get(
        "status"
    )
    ==
    "secure-server-ready"
)


# ==================================================================================================
# 14. SEND MASKED UPDATES THROUGH REAL TLS
# ==================================================================================================

CLIENT_TRANSMISSION_RECORDS_26 = []


for client in CLIENTS_26:

    masked_vector = (
        MASKED_UPDATES_26[
            client
        ]
        .astype(
            np.float64
        )
    )

    payload = masked_vector.tobytes()

    payload_hash = hashlib.sha256(
        payload
    ).hexdigest()


    secure_url = (
        f"https://{TLS_HOSTNAME_26}:"
        f"{PORT_26}/secure-update"
    )


    request = urllib.request.Request(
        secure_url,
        data=payload,
        method="POST",
        headers={
            "Content-Type":
                "application/octet-stream",

            "X-FL-Client":
                client,

            "X-Payload-SHA256":
                payload_hash,
        },
    )


    with urllib.request.urlopen(
        request,
        context=client_ssl_context_26,
        timeout=20,
    ) as response:

        response_data = json.loads(
            response.read().decode(
                "utf-8"
            )
        )


    CLIENT_TRANSMISSION_RECORDS_26.append(
        {
            "Client":
                client,

            "HTTPS_Status":
                response_data.get(
                    "status"
                ),

            "TLS_Version":
                response_data.get(
                    "tls"
                ),

            "Payload_Bytes":
                len(payload),

            "SHA256":
                payload_hash,

            "Server_Integrity_Accepted":
                response_data.get(
                    "integrity"
                ),
        }
    )


time.sleep(
    0.2
)


# ==================================================================================================
# 15. SHUT DOWN SERVER CLEANLY
# ==================================================================================================

SERVER_26.shutdown()
SERVER_26.server_close()

SERVER_THREAD_26.join(
    timeout=5
)


# ==================================================================================================
# 16. VERIFY ALL CLIENT UPDATES ARRIVED
# ==================================================================================================

ALL_CLIENTS_RECEIVED_26 = (
    set(
        SERVER_RECEIVED_MASKED_26.keys()
    )
    ==
    set(
        CLIENTS_26
    )
)


if not ALL_CLIENTS_RECEIVED_26:

    raise RuntimeError(
        "Not all client updates reached the secure aggregation server."
    )


# ==================================================================================================
# 17. SERVER SECURE AGGREGATION
#
# Server aggregates ONLY the transmitted masked vectors.
# ==================================================================================================

SECURE_AGGREGATE_26 = np.zeros(
    UPDATE_DIMENSION_26,
    dtype=np.float64
)


for client in CLIENTS_26:

    SECURE_AGGREGATE_26 += (
        SERVER_RECEIVED_MASKED_26[
            client
        ]
    )


# ==================================================================================================
# 18. VERIFY SECURE AGGREGATION FIDELITY
# ==================================================================================================

AGGREGATION_DIFFERENCE_26 = (
    SECURE_AGGREGATE_26
    -
    PLAIN_AGGREGATE_26
)


MAX_AGGREGATION_ERROR_26 = float(
    np.max(
        np.abs(
            AGGREGATION_DIFFERENCE_26
        )
    )
)


MEAN_AGGREGATION_ERROR_26 = float(
    np.mean(
        np.abs(
            AGGREGATION_DIFFERENCE_26
        )
    )
)


SECURE_AGGREGATION_EQUIVALENT_26 = bool(
    np.allclose(
        SECURE_AGGREGATE_26,
        PLAIN_AGGREGATE_26,
        atol=SECURE_AGG_ATOL_26,
        rtol=SECURE_AGG_RTOL_26,
    )
)


# ==================================================================================================
# 19. VERIFY THAT SERVER DID NOT RECEIVE PLAIN CLIENT UPDATES
# ==================================================================================================

privacy_rows_26 = []


for client in CLIENTS_26:

    plain_weighted = (
        CLIENT_WEIGHTS_26[
            client
        ]
        *
        CLIENT_UPDATES_26[
            client
        ]
    )

    transmitted = (
        SERVER_RECEIVED_MASKED_26[
            client
        ]
    )

    masking_distance = float(
        np.linalg.norm(
            transmitted
            -
            plain_weighted
        )
    )


    transmitted_equals_plain = bool(
        np.allclose(
            transmitted,
            plain_weighted,
            atol=SECURE_AGG_ATOL_26,
            rtol=SECURE_AGG_RTOL_26,
        )
    )


    privacy_rows_26.append(
        {
            "Client":
                client,

            "Weight":
                CLIENT_WEIGHTS_26[
                    client
                ],

            "Update_Dimension":
                UPDATE_DIMENSION_26,

            "Masking_Distance_L2":
                masking_distance,

            "Server_Received_Plain_Update":
                transmitted_equals_plain,

            "Update_Masked":
                not transmitted_equals_plain,
        }
    )


SECURITY26_PRIVACY_DF = pd.DataFrame(
    privacy_rows_26
)


# ==================================================================================================
# 20. CREATE TLS / TRANSPORT TABLES
# ==================================================================================================

SECURITY26_CLIENT_TRANSPORT_DF = pd.DataFrame(
    CLIENT_TRANSMISSION_RECORDS_26
)


SECURITY26_SERVER_TLS_DF = pd.DataFrame(
    SERVER_TLS_RECORDS_26
)


SECURITY26_INTEGRITY_DF = pd.DataFrame(
    SERVER_INTEGRITY_RECORDS_26
)


TLS_VERSION_OK_26 = bool(
    SECURITY26_SERVER_TLS_DF[
        "TLS_Version"
    ].astype(str).isin(
        [
            "TLSv1.2",
            "TLSv1.3",
        ]
    ).all()
)


INTEGRITY_OK_26 = bool(
    SECURITY26_INTEGRITY_DF[
        "Integrity_OK"
    ].all()
)


MASKING_OK_26 = bool(
    SECURITY26_PRIVACY_DF[
        "Update_Masked"
    ].all()
)


MASK_CANCELLATION_OK_26 = bool(
    MASK_RESIDUAL_MAX_26
    <= SECURE_AGG_ATOL_26
)


# ==================================================================================================
# 21. CERTIFICATE INFORMATION
# ==================================================================================================

certificate_info_command_26 = [
    "openssl",
    "x509",
    "-in",
    CERT_FILE_26,
    "-noout",
    "-subject",
    "-issuer",
    "-serial",
    "-dates",
    "-fingerprint",
    "-sha256",
]


certificate_result_26 = subprocess.run(
    certificate_info_command_26,
    capture_output=True,
    text=True,
)


CERTIFICATE_INFO_26 = (
    certificate_result_26.stdout.strip()
)


# ==================================================================================================
# 22. FINAL SECURITY SUMMARY
# ==================================================================================================

SECURITY26_SUMMARY_DF = pd.DataFrame(
    [
        {
            "Security_Test":
                "TLS handshake",

            "Mechanism":
                "HTTPS / X.509",

            "Result":
                (
                    "PASS"
                    if TLS_HANDSHAKE_OK_26
                    else "FAIL"
                ),
        },

        {
            "Security_Test":
                "TLS protocol version",

            "Mechanism":
                "TLS >= 1.2",

            "Result":
                (
                    "PASS"
                    if TLS_VERSION_OK_26
                    else "FAIL"
                ),
        },

        {
            "Security_Test":
                "Certificate verification",

            "Mechanism":
                "X.509 hostname + CA verification",

            "Result":
                (
                    "PASS"
                    if TLS_HANDSHAKE_OK_26
                    else "FAIL"
                ),
        },

        {
            "Security_Test":
                "Payload integrity",

            "Mechanism":
                "SHA-256 verification",

            "Result":
                (
                    "PASS"
                    if INTEGRITY_OK_26
                    else "FAIL"
                ),
        },

        {
            "Security_Test":
                "Individual update confidentiality",

            "Mechanism":
                "Pairwise additive masks",

            "Result":
                (
                    "PASS"
                    if MASKING_OK_26
                    else "FAIL"
                ),
        },

        {
            "Security_Test":
                "Pairwise mask cancellation",

            "Mechanism":
                "Zero-sum secure masks",

            "Result":
                (
                    "PASS"
                    if MASK_CANCELLATION_OK_26
                    else "FAIL"
                ),
        },

        {
            "Security_Test":
                "Aggregation fidelity",

            "Mechanism":
                "Secure vs ordinary weighted aggregate",

            "Result":
                (
                    "PASS"
                    if SECURE_AGGREGATION_EQUIVALENT_26
                    else "FAIL"
                ),
        },
    ]
)


SECURITY_LAYER_PASS_26 = bool(
    SECURITY26_SUMMARY_DF[
        "Result"
    ].eq(
        "PASS"
    ).all()
)


# ==================================================================================================
# 23. DISPLAY — TLS CERTIFICATE
# ==================================================================================================

print("\n" + "=" * 175)
print("TLS CERTIFICATE")
print("=" * 175)

print(
    CERTIFICATE_INFO_26
)


# ==================================================================================================
# 24. DISPLAY — TLS TRANSPORT
# ==================================================================================================

print("\n" + "=" * 175)
print("REAL TLS CLIENT-TO-SERVER TRANSPORT")
print("=" * 175)

display(
    SECURITY26_SERVER_TLS_DF
)


# ==================================================================================================
# 25. DISPLAY — INTEGRITY
# ==================================================================================================

print("\n" + "=" * 175)
print("PAYLOAD INTEGRITY VERIFICATION")
print("=" * 175)

display(
    SECURITY26_INTEGRITY_DF[
        [
            "Client",
            "Payload_Bytes",
            "Integrity_OK",
        ]
    ]
)


# ==================================================================================================
# 26. DISPLAY — SECURE AGGREGATION PRIVACY
# ==================================================================================================

print("\n" + "=" * 175)
print("SECURE AGGREGATION — INDIVIDUAL UPDATE MASKING")
print("=" * 175)

display(
    SECURITY26_PRIVACY_DF.round(
        8
    )
)


# ==================================================================================================
# 27. DISPLAY — NUMERICAL FIDELITY
# ==================================================================================================

print("\n" + "=" * 175)
print("SECURE AGGREGATION — NUMERICAL FIDELITY")
print("=" * 175)

print(
    f"Number of FL clients             : {len(CLIENTS_26)}"
)

print(
    f"Update dimension                 : {UPDATE_DIMENSION_26}"
)

print(
    f"Maximum pairwise-mask residual   : {MASK_RESIDUAL_MAX_26:.12e}"
)

print(
    f"Maximum aggregation difference   : {MAX_AGGREGATION_ERROR_26:.12e}"
)

print(
    f"Mean aggregation difference      : {MEAN_AGGREGATION_ERROR_26:.12e}"
)

print(
    f"Secure aggregate == plain result : {SECURE_AGGREGATION_EQUIVALENT_26}"
)


# ==================================================================================================
# 28. DISPLAY — FINAL SECURITY TABLE
# ==================================================================================================

print("\n" + "=" * 175)
print("CELL 26A — FINAL SECURITY VALIDATION")
print("=" * 175)

display(
    SECURITY26_SUMMARY_DF
)


# ==================================================================================================
# 29. PAPER-READY SECURITY SUMMARY
# ==================================================================================================

print("\n" + "=" * 175)
print("COPY / PASTE — SECURITY SUMMARY")
print("=" * 175)

print(
    f"TLS handshake successful            : {TLS_HANDSHAKE_OK_26}"
)

print(
    f"TLS >= 1.2 verified                 : {TLS_VERSION_OK_26}"
)

print(
    f"Certificate verification successful : {TLS_HANDSHAKE_OK_26}"
)

print(
    f"Payload integrity verified           : {INTEGRITY_OK_26}"
)

print(
    f"All individual updates masked        : {MASKING_OK_26}"
)

print(
    f"Pairwise masks cancelled             : {MASK_CANCELLATION_OK_26}"
)

print(
    f"Secure aggregation fidelity          : {SECURE_AGGREGATION_EQUIVALENT_26}"
)

print(
    f"Maximum aggregation error            : {MAX_AGGREGATION_ERROR_26:.12e}"
)


# ==================================================================================================
# 30. FINAL STATUS
# ==================================================================================================

print("\n" + "=" * 175)

if SECURITY_LAYER_PASS_26:

    print(
        "CELL 26 COMPLETED SUCCESSFULLY — SECURITY LAYER VALIDATED"
    )

else:

    print(
        "CELL 26 COMPLETED — SECURITY VALIDATION REQUIRES REVIEW"
    )

print("=" * 175)

print(
    "Transport encryption               : REAL TLS inside Colab"
)

print(
    "Certificate type                   : Self-signed X.509 experimental certificate"
)

print(
    "Client certificate verification    : ENABLED"
)

print(
    "Minimum TLS version                : TLS 1.2"
)

print(
    "Update protection                  : Pairwise-mask Secure Aggregation"
)

print(
    "Server receives plain updates      : NO"
)

print(
    "Raw local datasets transmitted     : NO"
)

print(
    "Model retraining                   : NO"
)

print(
    "Model predictions changed          : NO"
)

print(
    "Deployment interpretation          : Local security prototype"
)

print(
    f"Execution time                     : {time.time() - CELL26_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 26B — REAL CELL-13 FEDERATED SECURITY INTEGRATION — FINAL
#
# SOURCE OF CLIENT UPDATES:
#   Existing Cell-13 FEDERATED_RUNS state
#
# SOURCE OF AGGREGATION WEIGHTS:
#   REAL_CELL13_CLIENT_WEIGHTS produced explicitly by Cell 13B
#
# SECURITY:
#   - Verified local TLS / HTTPS transport
#   - X.509 certificate validation
#   - SHA-256 payload integrity
#   - Pairwise-mask Secure Aggregation
#
# IMPORTANT:
#   - NO synthetic updates
#   - NO fallback weights
#   - NO CLIENT_WEIGHTS_26
#   - NO automatic equal weighting
#   - NO retraining
#   - NO modification of Cell 13
#   - NO modification of Cell 16
#   - NO modification of predictions/results
# ==================================================================================================

import os
import ssl
import json
import time
import hashlib
import secrets
import tempfile
import threading
import subprocess
import urllib.request

import numpy as np
import pandas as pd

from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer


CELL26B_START = time.time()

print("=" * 180)
print("CELL 26B — FINAL REAL CELL-13 FEDERATED SECURITY INTEGRATION")
print("REAL CELL-13 STATE + VERIFIED CELL-13B WEIGHTS + TLS + SECURE AGGREGATION")
print("=" * 180)


# ==================================================================================================
# 1. FIXED CONFIGURATION
# ==================================================================================================

FL_STATIONS_26B = [
    "Kadhimiya",
    "Karama",
    "Resafa",
    "Tarmyya",
]

HOST_26B = "127.0.0.1"
TLS_HOSTNAME_26B = "localhost"

TLS_MINIMUM_VERSION_26B = ssl.TLSVersion.TLSv1_2

SECURE_ATOL_26B = 1e-9
SECURE_RTOL_26B = 1e-9

MAX_RECURSION_DEPTH_26B = 8


# ==================================================================================================
# 2. REQUIRE CELL 13
# ==================================================================================================

if "FEDERATED_RUNS" not in globals():

    raise RuntimeError(
        "\nCELL 26B STOPPED.\n\n"
        "FEDERATED_RUNS was not found.\n"
        "Run Cell 13 first."
    )


# ==================================================================================================
# 3. REQUIRE CELL 13B VERIFIED WEIGHTS
# ==================================================================================================

if "REAL_CELL13_CLIENT_WEIGHTS_BY_TASK" not in globals():

    raise RuntimeError(
        "\nCELL 26B STOPPED.\n\n"
        "REAL_CELL13_CLIENT_WEIGHTS_BY_TASK was not found.\n"
        "Run Cell 13B before Cell 26B."
    )


if not isinstance(
    REAL_CELL13_CLIENT_WEIGHTS_BY_TASK,
    dict
):

    raise RuntimeError(
        "REAL_CELL13_CLIENT_WEIGHTS_BY_TASK must be a dictionary."
    )


for task_key_26b, weights_26b in REAL_CELL13_CLIENT_WEIGHTS_BY_TASK.items():

    if not isinstance(weights_26b, dict):

        raise RuntimeError(
            f"Invalid weight dictionary for {task_key_26b}."
        )

    if not all(
        station in weights_26b
        for station in FL_STATIONS_26B
    ):

        raise RuntimeError(
            f"Missing station weight for task {task_key_26b}."
        )

    weight_sum_26b = float(
        sum(
            float(weights_26b[station])
            for station in FL_STATIONS_26B
        )
    )

    if not np.isclose(
        weight_sum_26b,
        1.0,
        atol=1e-12,
        rtol=1e-12,
    ):

        raise RuntimeError(
            f"Cell-13B weights for {task_key_26b} "
            f"do not sum to 1.0: {weight_sum_26b}"
        )


WEIGHT_SOURCE_26B = (
    "REAL_CELL13_CLIENT_WEIGHTS_BY_TASK from Cell 13B"
)


def c26b_task_from_source_path(source_path):

    source_text = str(source_path)

    matches = [
        str(task_key)
        for task_key
        in REAL_CELL13_CLIENT_WEIGHTS_BY_TASK.keys()
        if str(task_key) in source_text
    ]

    if len(matches) == 1:
        return matches[0]

    raise RuntimeError(
        f"Could not uniquely map Cell-13 bundle to task from Source_Path: "
        f"{source_path}"
    )


print("\nVerified aggregation-weight source:")
print(" ", WEIGHT_SOURCE_26B)

# ==================================================================================================
# 4. OPTIONAL SAMPLE COUNTS FROM CELL 13B
# ==================================================================================================

CELL13_SAMPLE_COUNTS_BY_TASK_26B = (
    REAL_CELL13_CLIENT_SAMPLE_COUNTS_BY_TASK
    if (
        "REAL_CELL13_CLIENT_SAMPLE_COUNTS_BY_TASK"
        in globals()
    )
    else None
)


# ==================================================================================================
# 5. STATION NORMALIZATION
# ==================================================================================================

def c26b_normalize_station_name(name):

    s = str(name).strip().lower()

    aliases = {
        "kadhimiya": "Kadhimiya",
        "kadhimiyah": "Kadhimiya",
        "kadhimiya station": "Kadhimiya",

        "karama": "Karama",
        "al karama": "Karama",

        "resafa": "Resafa",
        "rusafa": "Resafa",
        "al resafa": "Resafa",

        "tarmyya": "Tarmyya",
        "tarmiya": "Tarmyya",
        "tarmiyah": "Tarmyya",
    }

    return aliases.get(
        s,
        str(name)
    )


# ==================================================================================================
# 6. FLATTEN A STORED CLIENT UPDATE
# ==================================================================================================

def c26b_flatten_update(obj):

    if isinstance(
        obj,
        np.ndarray
    ):

        try:

            arr = np.asarray(
                obj,
                dtype=np.float64
            ).reshape(-1)

            if (
                arr.size > 0
                and
                np.all(
                    np.isfinite(arr)
                )
            ):

                return arr

        except Exception:

            return None


    if isinstance(
        obj,
        (list, tuple)
    ):

        pieces = []

        for item in obj:

            part = c26b_flatten_update(
                item
            )

            if part is None:

                return None

            pieces.append(
                part
            )


        if not pieces:

            return None


        return np.concatenate(
            pieces
        ).astype(
            np.float64
        )


    if isinstance(
        obj,
        dict
    ):

        pieces = []


        for key in sorted(
            obj.keys(),
            key=lambda x: str(x)
        ):

            part = c26b_flatten_update(
                obj[key]
            )

            if part is None:

                return None

            pieces.append(
                part
            )


        if not pieces:

            return None


        return np.concatenate(
            pieces
        ).astype(
            np.float64
        )


    if np.isscalar(
        obj
    ):

        try:

            value = float(
                obj
            )

            if np.isfinite(
                value
            ):

                return np.asarray(
                    [value],
                    dtype=np.float64
                )

        except Exception:

            return None


    return None


# ==================================================================================================
# 7. DETECT STATION -> UPDATE BUNDLE
# ==================================================================================================

def c26b_extract_station_bundle(obj):

    if not isinstance(
        obj,
        dict
    ):

        return None


    normalized_keys = {}


    for key in obj.keys():

        station = c26b_normalize_station_name(
            key
        )

        if station in FL_STATIONS_26B:

            normalized_keys[
                station
            ] = key


    if not all(
        station in normalized_keys
        for station in FL_STATIONS_26B
    ):

        return None


    vectors = {}

    dimension = None


    for station in FL_STATIONS_26B:

        original_key = normalized_keys[
            station
        ]


        vector = c26b_flatten_update(
            obj[
                original_key
            ]
        )


        if vector is None:

            return None


        if vector.size < 2:

            return None


        if dimension is None:

            dimension = vector.size

        elif vector.size != dimension:

            return None


        vectors[
            station
        ] = vector


    return vectors


# ==================================================================================================
# 8. SEARCH ONLY INSIDE FEDERATED_RUNS
#
# This avoids accidentally using Cell 26 / 26B security variables.
# ==================================================================================================

DISCOVERED_BUNDLES_26B = []

visited26b = set()


def c26b_recursive_scan(
    obj,
    path,
    depth
):

    if depth > MAX_RECURSION_DEPTH_26B:

        return


    oid = id(
        obj
    )

    if oid in visited26b:

        return


    visited26b.add(
        oid
    )


    bundle = c26b_extract_station_bundle(
        obj
    )


    if bundle is not None:

        dimension = len(
            next(
                iter(
                    bundle.values()
                )
            )
        )


        DISCOVERED_BUNDLES_26B.append(
            {
                "Path":
                    tuple(path),

                "Path_String":
                    "/".join(
                        map(
                            str,
                            path
                        )
                    ),

                "Updates":
                    bundle,

                "Dimension":
                    dimension,
            }
        )


    if isinstance(
        obj,
        dict
    ):

        for key, value in obj.items():

            if isinstance(
                value,
                (
                    dict,
                    list,
                    tuple,
                )
            ):

                c26b_recursive_scan(
                    value,
                    path + [key],
                    depth + 1
                )


    elif isinstance(
        obj,
        (list, tuple)
    ):

        max_items = min(
            len(obj),
            500
        )


        for idx in range(
            max_items
        ):

            value = obj[
                idx
            ]


            if isinstance(
                value,
                (
                    dict,
                    list,
                    tuple,
                )
            ):

                c26b_recursive_scan(
                    value,
                    path + [idx],
                    depth + 1
                )


c26b_recursive_scan(
    FEDERATED_RUNS,
    ["FEDERATED_RUNS"],
    0
)


# ==================================================================================================
# 9. REMOVE DUPLICATE UPDATE BUNDLES
# ==================================================================================================

unique_bundles26b = {}


for item in DISCOVERED_BUNDLES_26B:

    fingerprint = hashlib.sha256()


    for station in FL_STATIONS_26B:

        fingerprint.update(
            item[
                "Updates"
            ][station].astype(
                np.float64
            ).tobytes()
        )


    digest = fingerprint.hexdigest()


    if digest not in unique_bundles26b:

        unique_bundles26b[
            digest
        ] = item


DISCOVERED_BUNDLES_26B = list(
    unique_bundles26b.values()
)


DISCOVERED_BUNDLES_26B = sorted(
    DISCOVERED_BUNDLES_26B,
    key=lambda x: x[
        "Path_String"
    ]
)


if len(
    DISCOVERED_BUNDLES_26B
) == 0:

    raise RuntimeError(
        "\nCELL 26B STOPPED SAFELY.\n\n"
        "No reusable per-station update bundles were found "
        "inside FEDERATED_RUNS.\n\n"
        "No synthetic updates were substituted."
    )


print(
    "\nReal Cell-13 FL bundles discovered:",
    len(
        DISCOVERED_BUNDLES_26B
    )
)


# ==================================================================================================
# 10. DISCOVERY TABLE
# ==================================================================================================

SECURITY26B_DISCOVERY_DF = pd.DataFrame(
    [
        {
            "Bundle_ID":
                idx,

            "Source_Path":
                item[
                    "Path_String"
                ],

            "Update_Dimension":
                item[
                    "Dimension"
                ],
        }

        for idx, item in enumerate(
            DISCOVERED_BUNDLES_26B,
            start=1
        )
    ]
)


print("\n" + "=" * 180)
print("REAL CELL-13 UPDATE BUNDLES")
print("=" * 180)

display(
    SECURITY26B_DISCOVERY_DF
)


# ==================================================================================================
# 11. CELL-13B WEIGHT TABLE
# ==================================================================================================

weight_rows26b = []

for task_key_26b, weights_26b in sorted(
    REAL_CELL13_CLIENT_WEIGHTS_BY_TASK.items()
):

    for station in FL_STATIONS_26B:

        sample_count_26b = np.nan

        if (
            CELL13_SAMPLE_COUNTS_BY_TASK_26B is not None
            and
            task_key_26b in CELL13_SAMPLE_COUNTS_BY_TASK_26B
        ):
            sample_count_26b = (
                CELL13_SAMPLE_COUNTS_BY_TASK_26B[
                    task_key_26b
                ][station]
            )

        weight_rows26b.append(
            {
                "Task": task_key_26b,
                "Station": station,
                "Sample_Count": sample_count_26b,
                "Verified_Cell13_Weight": float(
                    weights_26b[station]
                ),
            }
        )


SECURITY26B_WEIGHT_DF = pd.DataFrame(
    weight_rows26b
)


print("\n" + "=" * 180)
print("VERIFIED CELL-13B TASK-SPECIFIC AGGREGATION WEIGHTS")
print("=" * 180)

display(
    SECURITY26B_WEIGHT_DF.round(12)
)


# ==================================================================================================
# 12. GENERATE EXPERIMENTAL X.509 CERTIFICATE
# ==================================================================================================

TLS_DIR_26B = tempfile.mkdtemp(
    prefix="paper3_real_cell13_tls_"
)


CERT_FILE_26B = os.path.join(
    TLS_DIR_26B,
    "server_cert.pem"
)


KEY_FILE_26B = os.path.join(
    TLS_DIR_26B,
    "server_key.pem"
)


openssl_command26b = [
    "openssl",
    "req",
    "-x509",
    "-newkey",
    "rsa:2048",
    "-sha256",
    "-nodes",
    "-days",
    "1",
    "-keyout",
    KEY_FILE_26B,
    "-out",
    CERT_FILE_26B,
    "-subj",
    "/CN=localhost",
    "-addext",
    "subjectAltName=DNS:localhost,IP:127.0.0.1",
]


openssl_result26b = subprocess.run(
    openssl_command26b,
    capture_output=True,
    text=True
)


if openssl_result26b.returncode != 0:

    raise RuntimeError(
        "TLS certificate generation failed:\n"
        +
        openssl_result26b.stderr
    )


# ==================================================================================================
# 13. TLS SERVER STORAGE
# ==================================================================================================

SERVER_PAYLOADS_26B = {}

SERVER_TLS_LOG_26B = []

SERVER_INTEGRITY_LOG_26B = []


# ==================================================================================================
# 14. HTTPS SERVER
# ==================================================================================================

class C26BHandler(
    BaseHTTPRequestHandler
):

    def log_message(
        self,
        format,
        *args
    ):

        return


    def do_GET(
        self
    ):

        if self.path != "/health":

            self.send_response(
                404
            )

            self.end_headers()

            return


        tls_version = self.connection.version()

        cipher = self.connection.cipher()


        body = json.dumps(
            {
                "status":
                    "ready",

                "tls_version":
                    tls_version,

                "cipher":
                    (
                        cipher[0]
                        if cipher
                        else None
                    ),
            }
        ).encode(
            "utf-8"
        )


        self.send_response(
            200
        )

        self.send_header(
            "Content-Type",
            "application/json"
        )

        self.send_header(
            "Content-Length",
            str(
                len(body)
            )
        )

        self.end_headers()

        self.wfile.write(
            body
        )


    def do_POST(
        self
    ):

        if self.path != "/secure-cell13-update":

            self.send_response(
                404
            )

            self.end_headers()

            return


        bundle_id = self.headers.get(
            "X-Bundle-ID"
        )

        client = self.headers.get(
            "X-FL-Client"
        )

        expected_sha = self.headers.get(
            "X-Payload-SHA256"
        )


        content_length = int(
            self.headers.get(
                "Content-Length",
                "0"
            )
        )


        payload = self.rfile.read(
            content_length
        )


        actual_sha = hashlib.sha256(
            payload
        ).hexdigest()


        integrity_ok = (
            expected_sha
            ==
            actual_sha
        )


        tls_version = self.connection.version()

        cipher = self.connection.cipher()


        SERVER_TLS_LOG_26B.append(
            {
                "Bundle_ID":
                    bundle_id,

                "Client":
                    client,

                "TLS_Version":
                    tls_version,

                "Cipher":
                    (
                        cipher[0]
                        if cipher
                        else None
                    ),

                "Cipher_Bits":
                    (
                        cipher[2]
                        if cipher
                        else None
                    ),
            }
        )


        SERVER_INTEGRITY_LOG_26B.append(
            {
                "Bundle_ID":
                    bundle_id,

                "Client":
                    client,

                "Payload_Bytes":
                    content_length,

                "Integrity_OK":
                    integrity_ok,
            }
        )


        if not integrity_ok:

            self.send_response(
                400
            )

            self.end_headers()

            return


        vector = np.frombuffer(
            payload,
            dtype=np.float64
        ).copy()


        SERVER_PAYLOADS_26B[
            (
                str(bundle_id),
                str(client),
            )
        ] = vector


        response_body = json.dumps(
            {
                "status":
                    "accepted",

                "client":
                    client,

                "bundle":
                    bundle_id,

                "tls":
                    tls_version,
            }
        ).encode(
            "utf-8"
        )


        self.send_response(
            200
        )

        self.send_header(
            "Content-Type",
            "application/json"
        )

        self.send_header(
            "Content-Length",
            str(
                len(response_body)
            )
        )

        self.end_headers()

        self.wfile.write(
            response_body
        )


# ==================================================================================================
# 15. START TLS SERVER
# ==================================================================================================

SERVER_26B = ThreadingHTTPServer(
    (
        HOST_26B,
        0,
    ),
    C26BHandler,
)


PORT_26B = SERVER_26B.server_address[
    1
]


server_context26b = ssl.SSLContext(
    ssl.PROTOCOL_TLS_SERVER
)


server_context26b.minimum_version = (
    TLS_MINIMUM_VERSION_26B
)


server_context26b.load_cert_chain(
    certfile=CERT_FILE_26B,
    keyfile=KEY_FILE_26B
)


SERVER_26B.socket = (
    server_context26b.wrap_socket(
        SERVER_26B.socket,
        server_side=True
    )
)


SERVER_THREAD_26B = threading.Thread(
    target=SERVER_26B.serve_forever,
    daemon=True
)


SERVER_THREAD_26B.start()


time.sleep(
    0.25
)


# ==================================================================================================
# 16. VERIFIED TLS CLIENT CONTEXT
# ==================================================================================================

client_context26b = ssl.create_default_context(
    cafile=CERT_FILE_26B
)


client_context26b.minimum_version = (
    TLS_MINIMUM_VERSION_26B
)


client_context26b.check_hostname = True


client_context26b.verify_mode = (
    ssl.CERT_REQUIRED
)


# ==================================================================================================
# 17. TLS HANDSHAKE
# ==================================================================================================

health_url26b = (
    f"https://{TLS_HOSTNAME_26B}:"
    f"{PORT_26B}/health"
)


with urllib.request.urlopen(
    health_url26b,
    context=client_context26b,
    timeout=10
) as response:

    health_info26b = json.loads(
        response.read().decode(
            "utf-8"
        )
    )


TLS_HANDSHAKE_OK_26B = (
    health_info26b.get(
        "status"
    )
    ==
    "ready"
)


# ==================================================================================================
# 18. SECURE AGGREGATION OF EACH REAL CELL-13 BUNDLE
# ==================================================================================================

BUNDLE_RESULTS_26B = []

CLIENT_PRIVACY_RESULTS_26B = []


for bundle_index, bundle_info in enumerate(
    DISCOVERED_BUNDLES_26B,
    start=1
):

    bundle_id = str(
        bundle_index
    )


    source_path = bundle_info[
        "Path_String"
    ]

    task_key_26b = c26b_task_from_source_path(
        source_path
    )

    bundle_weights_26b = {
        station: float(
            REAL_CELL13_CLIENT_WEIGHTS_BY_TASK[
                task_key_26b
            ][station]
        )
        for station in FL_STATIONS_26B
    }


    updates = bundle_info[
        "Updates"
    ]


    dimension = bundle_info[
        "Dimension"
    ]


    # ----------------------------------------------------------------------------------------------
    # Reference weighted aggregate using VERIFIED Cell-13B weights
    # ----------------------------------------------------------------------------------------------

    plain_aggregate = np.zeros(
        dimension,
        dtype=np.float64
    )


    for station in FL_STATIONS_26B:

        plain_aggregate += (
            bundle_weights_26b[
                station
            ]
            *
            updates[
                station
            ]
        )


    # ----------------------------------------------------------------------------------------------
    # Pairwise secrets
    # ----------------------------------------------------------------------------------------------

    pairwise_secrets = {}


    for i in range(
        len(FL_STATIONS_26B)
    ):

        for j in range(
            i + 1,
            len(FL_STATIONS_26B)
        ):

            station_i = FL_STATIONS_26B[
                i
            ]

            station_j = FL_STATIONS_26B[
                j
            ]


            pairwise_secrets[
                (
                    station_i,
                    station_j,
                )
            ] = secrets.token_bytes(
                32
            )


    # ----------------------------------------------------------------------------------------------
    # Create pairwise masks
    # ----------------------------------------------------------------------------------------------

    masks = {
        station:
            np.zeros(
                dimension,
                dtype=np.float64
            )

        for station in FL_STATIONS_26B
    }


    for i in range(
        len(FL_STATIONS_26B)
    ):

        for j in range(
            i + 1,
            len(FL_STATIONS_26B)
        ):

            station_i = FL_STATIONS_26B[
                i
            ]

            station_j = FL_STATIONS_26B[
                j
            ]


            secret = pairwise_secrets[
                (
                    station_i,
                    station_j,
                )
            ]


            context = (
                f"Paper3|RealCell13|"
                f"Bundle={bundle_id}|"
                f"{station_i}|{station_j}|"
                f"Dim={dimension}"
            ).encode(
                "utf-8"
            )


            digest = hashlib.sha256(
                secret
                +
                context
            ).digest()


            seed = int.from_bytes(
                digest[:8],
                byteorder="big",
                signed=False
            )


            rng_pair = np.random.default_rng(
                seed
            )


            pair_mask = rng_pair.normal(
                loc=0.0,
                scale=1.0,
                size=dimension
            ).astype(
                np.float64
            )


            masks[
                station_i
            ] += pair_mask


            masks[
                station_j
            ] -= pair_mask


    # ----------------------------------------------------------------------------------------------
    # Validate zero-sum masks
    # ----------------------------------------------------------------------------------------------

    total_mask = np.zeros(
        dimension,
        dtype=np.float64
    )


    for station in FL_STATIONS_26B:

        total_mask += masks[
            station
        ]


    mask_residual = float(
        np.max(
            np.abs(
                total_mask
            )
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Mask REAL weighted client update and transmit through TLS
    # ----------------------------------------------------------------------------------------------

    for station in FL_STATIONS_26B:

        weighted_plain = (
            bundle_weights_26b[
                station
            ]
            *
            updates[
                station
            ]
        )


        masked_update = (
            weighted_plain
            +
            masks[
                station
            ]
        )


        plain_exposed = bool(
            np.allclose(
                masked_update,
                weighted_plain,
                atol=SECURE_ATOL_26B,
                rtol=SECURE_RTOL_26B
            )
        )


        masking_distance = float(
            np.linalg.norm(
                masked_update
                -
                weighted_plain
            )
        )


        CLIENT_PRIVACY_RESULTS_26B.append(
            {
                "Bundle_ID":
                    bundle_id,

                "Source_Path":
                    source_path,

                "Task":
                    task_key_26b,

                "Client":
                    station,

                "Update_Dimension":
                    dimension,

                "Verified_Cell13_Weight":
                    bundle_weights_26b[
                        station
                    ],

                "Masking_Distance_L2":
                    masking_distance,

                "Plain_Update_Exposed":
                    plain_exposed,
            }
        )


        payload = masked_update.astype(
            np.float64
        ).tobytes()


        payload_hash = hashlib.sha256(
            payload
        ).hexdigest()


        secure_url = (
            f"https://{TLS_HOSTNAME_26B}:"
            f"{PORT_26B}/secure-cell13-update"
        )


        request = urllib.request.Request(
            secure_url,
            data=payload,
            method="POST",
            headers={
                "Content-Type":
                    "application/octet-stream",

                "X-Bundle-ID":
                    bundle_id,

                "X-FL-Client":
                    station,

                "X-Payload-SHA256":
                    payload_hash,
            }
        )


        with urllib.request.urlopen(
            request,
            context=client_context26b,
            timeout=30
        ) as response:

            response_info = json.loads(
                response.read().decode(
                    "utf-8"
                )
            )


        if response_info.get(
            "status"
        ) != "accepted":

            raise RuntimeError(
                f"TLS transmission failed for "
                f"{source_path} / {station}"
            )


    # ----------------------------------------------------------------------------------------------
    # Server-side secure aggregate
    # ----------------------------------------------------------------------------------------------

    secure_aggregate = np.zeros(
        dimension,
        dtype=np.float64
    )


    for station in FL_STATIONS_26B:

        secure_aggregate += (
            SERVER_PAYLOADS_26B[
                (
                    bundle_id,
                    station,
                )
            ]
        )


    difference = (
        secure_aggregate
        -
        plain_aggregate
    )


    max_difference = float(
        np.max(
            np.abs(
                difference
            )
        )
    )


    mean_difference = float(
        np.mean(
            np.abs(
                difference
            )
        )
    )


    equivalent = bool(
        np.allclose(
            secure_aggregate,
            plain_aggregate,
            atol=SECURE_ATOL_26B,
            rtol=SECURE_RTOL_26B
        )
    )


    BUNDLE_RESULTS_26B.append(
        {
            "Bundle_ID":
                bundle_id,

            "Source_Path":
                source_path,

            "Task":
                task_key_26b,

            "Update_Dimension":
                dimension,

            "Mask_Residual_Max":
                mask_residual,

            "Max_Aggregation_Error":
                max_difference,

            "Mean_Aggregation_Error":
                mean_difference,

            "Secure_Equals_Plain":
                equivalent,
        }
    )


# ==================================================================================================
# 19. SHUT DOWN TLS SERVER
# ==================================================================================================

SERVER_26B.shutdown()

SERVER_26B.server_close()

SERVER_THREAD_26B.join(
    timeout=5
)


# ==================================================================================================
# 20. RESULT TABLES
# ==================================================================================================

SECURITY26B_BUNDLE_RESULTS_DF = pd.DataFrame(
    BUNDLE_RESULTS_26B
)


SECURITY26B_CLIENT_PRIVACY_DF = pd.DataFrame(
    CLIENT_PRIVACY_RESULTS_26B
)


SECURITY26B_TLS_DF = pd.DataFrame(
    SERVER_TLS_LOG_26B
)


SECURITY26B_INTEGRITY_DF = pd.DataFrame(
    SERVER_INTEGRITY_LOG_26B
)


# ==================================================================================================
# 21. VALIDATION FLAGS
# ==================================================================================================

TLS_OK_26B = bool(
    TLS_HANDSHAKE_OK_26B
    and
    SECURITY26B_TLS_DF[
        "TLS_Version"
    ].astype(
        str
    ).isin(
        [
            "TLSv1.2",
            "TLSv1.3",
        ]
    ).all()
)


INTEGRITY_OK_26B = bool(
    SECURITY26B_INTEGRITY_DF[
        "Integrity_OK"
    ].all()
)


PRIVACY_OK_26B = bool(
    (
        SECURITY26B_CLIENT_PRIVACY_DF[
            "Plain_Update_Exposed"
        ]
        ==
        False
    ).all()
)


MASKS_OK_26B = bool(
    (
        SECURITY26B_BUNDLE_RESULTS_DF[
            "Mask_Residual_Max"
        ]
        <=
        SECURE_ATOL_26B
    ).all()
)


FIDELITY_OK_26B = bool(
    SECURITY26B_BUNDLE_RESULTS_DF[
        "Secure_Equals_Plain"
    ].all()
)


WEIGHTS_OK_26B = bool(
    all(
        np.isclose(
            sum(
                float(weights[station])
                for station in FL_STATIONS_26B
            ),
            1.0,
            atol=1e-12,
            rtol=1e-12,
        )
        for weights
        in REAL_CELL13_CLIENT_WEIGHTS_BY_TASK.values()
    )
)


# ==================================================================================================
# 22. FINAL SECURITY SUMMARY
# ==================================================================================================

SECURITY26B_SUMMARY_DF = pd.DataFrame(
    [
        {
            "Security_Test":
                "Real Cell-13 FL state discovered",

            "Result":
                "PASS",
        },

        {
            "Security_Test":
                "Cell-13B verified aggregation weights used",

            "Result":
                (
                    "PASS"
                    if WEIGHTS_OK_26B
                    else "FAIL"
                ),
        },

        {
            "Security_Test":
                "Verified TLS transport",

            "Result":
                (
                    "PASS"
                    if TLS_OK_26B
                    else "FAIL"
                ),
        },

        {
            "Security_Test":
                "SHA-256 payload integrity",

            "Result":
                (
                    "PASS"
                    if INTEGRITY_OK_26B
                    else "FAIL"
                ),
        },

        {
            "Security_Test":
                "Individual real client payloads masked",

            "Result":
                (
                    "PASS"
                    if PRIVACY_OK_26B
                    else "FAIL"
                ),
        },

        {
            "Security_Test":
                "Pairwise masks cancelled",

            "Result":
                (
                    "PASS"
                    if MASKS_OK_26B
                    else "FAIL"
                ),
        },

        {
            "Security_Test":
                "Secure aggregate equals verified weighted aggregate",

            "Result":
                (
                    "PASS"
                    if FIDELITY_OK_26B
                    else "FAIL"
                ),
        },
    ]
)


CELL26B_PASS = bool(
    SECURITY26B_SUMMARY_DF[
        "Result"
    ].eq(
        "PASS"
    ).all()
)


# ==================================================================================================
# 23. DISPLAY — TLS
# ==================================================================================================

print("\n" + "=" * 180)
print("TLS TRANSPORT")
print("=" * 180)

display(
    SECURITY26B_TLS_DF
)


# ==================================================================================================
# 24. DISPLAY — PAYLOAD INTEGRITY
# ==================================================================================================

print("\n" + "=" * 180)
print("PAYLOAD INTEGRITY")
print("=" * 180)

display(
    SECURITY26B_INTEGRITY_DF
)


# ==================================================================================================
# 25. DISPLAY — UPDATE MASKING
# ==================================================================================================

print("\n" + "=" * 180)
print("REAL CELL-13 UPDATE MASKING")
print("=" * 180)

display(
    SECURITY26B_CLIENT_PRIVACY_DF[
        [
            "Bundle_ID",
            "Source_Path",
            "Task",
            "Client",
            "Update_Dimension",
            "Verified_Cell13_Weight",
            "Masking_Distance_L2",
            "Plain_Update_Exposed",
        ]
    ].round(
        12
    )
)


# ==================================================================================================
# 26. DISPLAY — SECURE AGGREGATION FIDELITY
# ==================================================================================================

print("\n" + "=" * 180)
print("SECURE AGGREGATION FIDELITY")
print("=" * 180)

display(
    SECURITY26B_BUNDLE_RESULTS_DF.round(
        15
    )
)


# ==================================================================================================
# 27. FINAL VALIDATION TABLE
# ==================================================================================================

print("\n" + "=" * 180)
print("CELL 26B — FINAL SECURITY VALIDATION")
print("=" * 180)

display(
    SECURITY26B_SUMMARY_DF
)


# ==================================================================================================
# 28. PAPER-READY SUMMARY
# ==================================================================================================

print("\n" + "=" * 180)
print("COPY / PASTE — FINAL REAL FEDERATED SECURITY RESULTS")
print("=" * 180)


print(
    f"FL update bundles secured             : "
    f"{len(SECURITY26B_BUNDLE_RESULTS_DF)}"
)


print(
    f"Participating stations                : "
    f"{len(FL_STATIONS_26B)}"
)


print(
    f"Stations                              : "
    f"{', '.join(FL_STATIONS_26B)}"
)


print(
    f"Aggregation-weight source             : "
    f"{WEIGHT_SOURCE_26B}"
)


for station in FL_STATIONS_26B:

    print(
        f"{station:<36}: "
        f"{REAL_CELL13_CLIENT_WEIGHTS_BY_TASK[next(iter(REAL_CELL13_CLIENT_WEIGHTS_BY_TASK))][station]:.12f}"
    )


print(
    f"Weight sum                            : "
    f"{sum(REAL_CELL13_CLIENT_WEIGHTS_BY_TASK[next(iter(REAL_CELL13_CLIENT_WEIGHTS_BY_TASK))].values()):.12f}"
)


print(
    f"TLS transport validated               : "
    f"{TLS_OK_26B}"
)


print(
    f"Payload integrity validated           : "
    f"{INTEGRITY_OK_26B}"
)


print(
    f"Individual real updates masked        : "
    f"{PRIVACY_OK_26B}"
)


print(
    f"Pairwise masks cancelled              : "
    f"{MASKS_OK_26B}"
)


print(
    f"Secure aggregation fidelity           : "
    f"{FIDELITY_OK_26B}"
)


print(
    "Maximum secure-aggregation error      : "
    f"{SECURITY26B_BUNDLE_RESULTS_DF['Max_Aggregation_Error'].max():.12e}"
)


# ==================================================================================================
# 29. FINAL STATUS
# ==================================================================================================

print("\n" + "=" * 180)


if CELL26B_PASS:

    print(
        "CELL 26B COMPLETED SUCCESSFULLY — "
        "REAL CELL-13 FEDERATED SECURITY VALIDATED "
        "USING VERIFIED CELL-13B WEIGHTS"
    )

else:

    print(
        "CELL 26B COMPLETED — "
        "SECURITY VALIDATION REQUIRES REVIEW"
    )


print("=" * 180)


print(
    "Source of FL state               : FEDERATED_RUNS from Cell 13"
)


print(
    "Source of aggregation weights    : REAL_CELL13_CLIENT_WEIGHTS from Cell 13B"
)


print(
    "CLIENT_WEIGHTS_26 used           : NO"
)


print(
    "Automatic equal weights used     : NO"
)


print(
    "Synthetic/demo updates used      : NO"
)


print(
    "Transport                        : Verified local TLS / X.509"
)


print(
    "Minimum TLS version              : TLS 1.2"
)


print(
    "Update protection                : Pairwise-mask Secure Aggregation"
)


print(
    "Raw local datasets transmitted   : NO"
)


print(
    "Server receives plain payloads   : NO"
)


print(
    "Cell 13 retrained                : NO"
)


print(
    "Cell 16 modified                 : NO"
)


print(
    "Predictions changed              : NO"
)


print(
    "Previous result cells changed    : NO"
)


print(
    f"Execution time                   : "
    f"{time.time() - CELL26B_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 27 — FINAL COMPUTATIONAL / COMMUNICATION / STORAGE / SECURITY COST ANALYSIS
# UNIT-AWARE PUBLICATION VERSION
#
# PAPER 3 — UTW-RG-AdaFedResidual
#
# PURPOSE
#   - Computational cost
#   - Communication cost
#   - Storage / memory
#   - Secure Aggregation overhead
#   - Network-transfer estimates
#   - Theoretical complexity
#
# IMPORTANT
#   - 143 active features
#   - 144 trainable parameters = 143 weights + intercept
#   - 7-D FEDERATED_RUNS records are NOT treated as model parameters
#   - Generic scalar n_train is NOT used
#   - NO retraining
#   - NO tuning
#   - NO prediction changes
# ==================================================================================================

import time
import numpy as np
import pandas as pd


CELL27_START = time.time()

print("=" * 180)
print("CELL 27 — FINAL COMPUTATIONAL / COMMUNICATION / STORAGE / SECURITY COST ANALYSIS")
print("=" * 180)


# ==================================================================================================
# 1. FEDERATED PROTOCOL
# ==================================================================================================

CLIENTS_27 = [
    "Kadhimiya",
    "Karama",
    "Resafa",
    "Tarmyya",
]

TASKS_27 = [
    "Alum_H1",
    "Alum_H2",
    "Alum_H3",
    "Chlorine_H1",
    "Chlorine_H2",
    "Chlorine_H3",
]

N_CLIENTS_27 = len(CLIENTS_27)
N_TASKS_27 = len(TASKS_27)

FL_ROUNDS_27 = 60
LOCAL_EPOCHS_27 = 2
CLIENT_FRACTION_27 = 1.0

ACTIVE_CLIENTS_PER_ROUND_27 = int(
    round(
        N_CLIENTS_27
        *
        CLIENT_FRACTION_27
    )
)

FLOAT64_BYTES_27 = 8
FLOAT32_BYTES_27 = 4


# ==================================================================================================
# 2. ACTIVE FEATURES
# ==================================================================================================

if "ACTIVE_FEATURE_COLUMNS" in globals():

    ACTIVE_FEATURES_27 = int(
        len(ACTIVE_FEATURE_COLUMNS)
    )

    FEATURE_SOURCE_27 = "ACTIVE_FEATURE_COLUMNS"

elif "active_feature_columns" in globals():

    ACTIVE_FEATURES_27 = int(
        len(active_feature_columns)
    )

    FEATURE_SOURCE_27 = "active_feature_columns"

else:

    ACTIVE_FEATURES_27 = 143

    FEATURE_SOURCE_27 = (
        "Verified Paper-3 Cell-8 result"
    )


# ==================================================================================================
# 3. TRAINABLE MODEL PARAMETERS
# ==================================================================================================

MODEL_PARAMETERS_27 = (
    ACTIVE_FEATURES_27
    +
    1
)

MODEL_SIZE_F64_BYTES_27 = (
    MODEL_PARAMETERS_27
    *
    FLOAT64_BYTES_27
)

MODEL_SIZE_F32_BYTES_27 = (
    MODEL_PARAMETERS_27
    *
    FLOAT32_BYTES_27
)


# ==================================================================================================
# 4. FEDERATED_RUNS RECORD DIMENSION
# ==================================================================================================

FEDERATED_RUN_RECORD_DIMENSION_27 = np.nan


if "SECURITY26B_DISCOVERY_DF" in globals():

    try:

        dims27 = sorted(
            set(
                SECURITY26B_DISCOVERY_DF[
                    "Update_Dimension"
                ]
                .dropna()
                .astype(int)
                .tolist()
            )
        )

        if len(dims27) == 1:

            FEDERATED_RUN_RECORD_DIMENSION_27 = int(
                dims27[0]
            )

    except Exception:

        pass


RECORD_DIMENSION_IS_MODEL_DIMENSION_27 = bool(
    np.isfinite(
        FEDERATED_RUN_RECORD_DIMENSION_27
    )
    and
    int(
        FEDERATED_RUN_RECORD_DIMENSION_27
    )
    ==
    MODEL_PARAMETERS_27
)


# ==================================================================================================
# 5. BYTE HELPERS
# ==================================================================================================

def c27_kb(x):
    return float(x) / 1024.0


def c27_mb(x):
    return float(x) / (1024.0 ** 2)


def c27_gb(x):
    return float(x) / (1024.0 ** 3)


# ==================================================================================================
# 6. FIND GENUINE GLOBAL TRAIN MATRIX
# ==================================================================================================

TRAIN_MATRIX_CANDIDATES_27 = []


for variable_name, variable_value in list(
    globals().items()
):

    name_lower = str(
        variable_name
    ).lower()

    if not (
        "train" in name_lower
        or
        name_lower.startswith("x_")
        or
        name_lower.startswith("xtrain")
    ):
        continue

    try:

        arr = np.asarray(
            variable_value
        )

        if (
            arr.ndim == 2
            and
            arr.shape[1] == ACTIVE_FEATURES_27
            and
            arr.shape[0] > 100
        ):

            TRAIN_MATRIX_CANDIDATES_27.append(
                {
                    "Variable":
                        variable_name,

                    "Rows":
                        int(arr.shape[0]),

                    "Columns":
                        int(arr.shape[1]),
                }
            )

    except Exception:

        pass


if len(
    TRAIN_MATRIX_CANDIDATES_27
) > 0:

    TRAIN_MATRIX_CANDIDATES_27 = sorted(
        TRAIN_MATRIX_CANDIDATES_27,
        key=lambda x: -x["Rows"]
    )

    selected_train_matrix_27 = (
        TRAIN_MATRIX_CANDIDATES_27[0]
    )

    TRAIN_ROWS_TOTAL_PER_TASK_27 = int(
        selected_train_matrix_27[
            "Rows"
        ]
    )

    TRAIN_ROWS_SOURCE_27 = (
        selected_train_matrix_27[
            "Variable"
        ]
    )

    TRAIN_ROWS_SOURCE_TYPE_27 = (
        "Detected global 143-feature training matrix"
    )

else:

    TRAIN_ROWS_TOTAL_PER_TASK_27 = 2620

    TRAIN_ROWS_SOURCE_27 = (
        "Paper-3 Cell-8 verified approximate total"
    )

    TRAIN_ROWS_SOURCE_TYPE_27 = (
        "Verified experiment-level fallback"
    )


AVG_LOCAL_TRAIN_ROWS_27 = (
    TRAIN_ROWS_TOTAL_PER_TASK_27
    /
    N_CLIENTS_27
)


# ==================================================================================================
# 7. REAL CELL-13B FEDERATED WEIGHTS
# ==================================================================================================

if (
    "REAL_CELL13_CLIENT_WEIGHTS_BY_TASK" in globals()
    and
    isinstance(
        REAL_CELL13_CLIENT_WEIGHTS_BY_TASK,
        dict
    )
    and
    len(
        REAL_CELL13_CLIENT_WEIGHTS_BY_TASK
    ) > 0
):

    # For the efficiency summary only, report the mean exact Cell-13 weight
    # across the six tasks. Communication/computation formulas below do not
    # depend on these weights.
    CLIENT_WEIGHTS_27 = {
        client:
            float(
                np.mean(
                    [
                        REAL_CELL13_CLIENT_WEIGHTS_BY_TASK[
                            task_key
                        ][client]
                        for task_key
                        in REAL_CELL13_CLIENT_WEIGHTS_BY_TASK
                    ]
                )
            )

        for client in CLIENTS_27
    }

    WEIGHT_SOURCE_27 = (
        "Mean of exact task-specific Cell-13B aggregation weights"
    )

elif (
    "REAL_CELL13_CLIENT_WEIGHTS" in globals()
    and
    isinstance(
        REAL_CELL13_CLIENT_WEIGHTS,
        dict
    )
):

    CLIENT_WEIGHTS_27 = {
        client:
            float(
                REAL_CELL13_CLIENT_WEIGHTS[
                    client
                ]
            )

        for client in CLIENTS_27
    }

    WEIGHT_SOURCE_27 = (
        "REAL_CELL13_CLIENT_WEIGHTS from Cell 13B"
    )

else:

    raise RuntimeError(
        "Cell 27 requires Cell 13B aggregation weights."
    )


ESTIMATED_LOCAL_ROWS_27 = {
    client:
        TRAIN_ROWS_TOTAL_PER_TASK_27
        *
        CLIENT_WEIGHTS_27[
            client
        ]

    for client in CLIENTS_27
}


# ==================================================================================================
# 8. COMMUNICATION COUNTS
# ==================================================================================================

UPLINK_MESSAGES_PER_TASK_27 = (
    FL_ROUNDS_27
    *
    ACTIVE_CLIENTS_PER_ROUND_27
)

DOWNLINK_MESSAGES_PER_TASK_27 = (
    FL_ROUNDS_27
    *
    ACTIVE_CLIENTS_PER_ROUND_27
)

TOTAL_MESSAGES_PER_TASK_27 = (
    UPLINK_MESSAGES_PER_TASK_27
    +
    DOWNLINK_MESSAGES_PER_TASK_27
)


# ==================================================================================================
# 9. COMMUNICATION VOLUME
# ==================================================================================================

UPLINK_BYTES_PER_TASK_27 = (
    UPLINK_MESSAGES_PER_TASK_27
    *
    MODEL_SIZE_F64_BYTES_27
)

DOWNLINK_BYTES_PER_TASK_27 = (
    DOWNLINK_MESSAGES_PER_TASK_27
    *
    MODEL_SIZE_F64_BYTES_27
)

TOTAL_BYTES_PER_TASK_27 = (
    UPLINK_BYTES_PER_TASK_27
    +
    DOWNLINK_BYTES_PER_TASK_27
)

FINAL_SIX_TASK_UPLINK_BYTES_27 = (
    UPLINK_BYTES_PER_TASK_27
    *
    N_TASKS_27
)

FINAL_SIX_TASK_DOWNLINK_BYTES_27 = (
    DOWNLINK_BYTES_PER_TASK_27
    *
    N_TASKS_27
)

FINAL_SIX_TASK_TOTAL_BYTES_27 = (
    TOTAL_BYTES_PER_TASK_27
    *
    N_TASKS_27
)

FINAL_SIX_TASK_TOTAL_F32_BYTES_27 = (
    FINAL_SIX_TASK_TOTAL_BYTES_27
    /
    2.0
)


# ==================================================================================================
# 10. CELL-13 FULL EXPERIMENTAL SEARCH
# ==================================================================================================

if (
    "FEDERATED_RUNS" in globals()
    and
    isinstance(
        FEDERATED_RUNS,
        dict
    )
):

    CELL13_EXPERIMENT_RUNS_27 = int(
        len(
            FEDERATED_RUNS
        )
    )

else:

    CELL13_EXPERIMENT_RUNS_27 = np.nan


if np.isfinite(
    CELL13_EXPERIMENT_RUNS_27
):

    CELL13_SEARCH_COMM_BYTES_27 = (
        CELL13_EXPERIMENT_RUNS_27
        *
        TOTAL_BYTES_PER_TASK_27
    )

else:

    CELL13_SEARCH_COMM_BYTES_27 = np.nan


# ==================================================================================================
# 11. LOCAL TRAINING COMPUTATIONAL SCALE
#
# O(R × E × N_train × P)
# ==================================================================================================

LOCAL_TRAIN_OPS_PER_TASK_27 = (
    FL_ROUNDS_27
    *
    LOCAL_EPOCHS_27
    *
    TRAIN_ROWS_TOTAL_PER_TASK_27
    *
    MODEL_PARAMETERS_27
)

LOCAL_TRAIN_OPS_ALL_TASKS_27 = (
    LOCAL_TRAIN_OPS_PER_TASK_27
    *
    N_TASKS_27
)


if np.isfinite(
    CELL13_EXPERIMENT_RUNS_27
):

    CELL13_SEARCH_OPS_27 = (
        LOCAL_TRAIN_OPS_PER_TASK_27
        *
        CELL13_EXPERIMENT_RUNS_27
    )

else:

    CELL13_SEARCH_OPS_27 = np.nan


# ==================================================================================================
# 12. SERVER AGGREGATION
# ==================================================================================================

SERVER_AGG_OPS_PER_TASK_27 = (
    FL_ROUNDS_27
    *
    ACTIVE_CLIENTS_PER_ROUND_27
    *
    MODEL_PARAMETERS_27
)


# ==================================================================================================
# 13. STORAGE
# ==================================================================================================

SINGLE_MODEL_BYTES_27 = (
    MODEL_SIZE_F64_BYTES_27
)

CLIENT_MODELS_BYTES_27 = (
    N_CLIENTS_27
    *
    SINGLE_MODEL_BYTES_27
)

FEDADAM_SERVER_STATE_BYTES_27 = (
    3
    *
    SINGLE_MODEL_BYTES_27
)

ACTIVE_FL_PARAMETER_STATE_BYTES_27 = (
    CLIENT_MODELS_BYTES_27
    +
    FEDADAM_SERVER_STATE_BYTES_27
)


# ==================================================================================================
# 14. SECURE AGGREGATION COMPLEXITY
# ==================================================================================================

CLIENT_PAIRS_27 = (
    N_CLIENTS_27
    *
    (
        N_CLIENTS_27 - 1
    )
    //
    2
)

MASK_VALUES_PER_ROUND_27 = (
    CLIENT_PAIRS_27
    *
    MODEL_PARAMETERS_27
)

MASK_VALUES_PER_TASK_27 = (
    MASK_VALUES_PER_ROUND_27
    *
    FL_ROUNDS_27
)


# ==================================================================================================
# 15. ROBUST SECURE-AGGREGATION MICROBENCHMARK
# ==================================================================================================

BENCHMARK_BATCHES_27 = 25
BENCHMARK_REPS_PER_BATCH_27 = 1000

rng27 = np.random.default_rng(
    42
)


benchmark_updates_27 = {
    client:
        rng27.normal(
            size=MODEL_PARAMETERS_27
        ).astype(
            np.float64
        )

    for client in CLIENTS_27
}


plain_times_27 = []
secure_times_27 = []


# --------------------------------------------------------------------------------------------------
# Plain aggregation
# --------------------------------------------------------------------------------------------------

for _batch in range(
    BENCHMARK_BATCHES_27
):

    t0 = time.perf_counter()

    for _ in range(
        BENCHMARK_REPS_PER_BATCH_27
    ):

        result_plain_27 = np.zeros(
            MODEL_PARAMETERS_27,
            dtype=np.float64
        )

        for client in CLIENTS_27:

            result_plain_27 += (
                CLIENT_WEIGHTS_27[
                    client
                ]
                *
                benchmark_updates_27[
                    client
                ]
            )

    elapsed = (
        time.perf_counter()
        -
        t0
    )

    plain_times_27.append(
        elapsed
        /
        BENCHMARK_REPS_PER_BATCH_27
        *
        1e6
    )


# --------------------------------------------------------------------------------------------------
# Secure aggregation
# --------------------------------------------------------------------------------------------------

for _batch in range(
    BENCHMARK_BATCHES_27
):

    t0 = time.perf_counter()

    for _ in range(
        BENCHMARK_REPS_PER_BATCH_27
    ):

        masks27 = {
            client:
                np.zeros(
                    MODEL_PARAMETERS_27,
                    dtype=np.float64
                )

            for client in CLIENTS_27
        }

        for i in range(
            N_CLIENTS_27
        ):

            for j in range(
                i + 1,
                N_CLIENTS_27
            ):

                pair_mask = rng27.normal(
                    size=MODEL_PARAMETERS_27
                )

                masks27[
                    CLIENTS_27[i]
                ] += pair_mask

                masks27[
                    CLIENTS_27[j]
                ] -= pair_mask

        result_secure_27 = np.zeros(
            MODEL_PARAMETERS_27,
            dtype=np.float64
        )

        for client in CLIENTS_27:

            result_secure_27 += (
                CLIENT_WEIGHTS_27[
                    client
                ]
                *
                benchmark_updates_27[
                    client
                ]
                +
                masks27[
                    client
                ]
            )

    elapsed = (
        time.perf_counter()
        -
        t0
    )

    secure_times_27.append(
        elapsed
        /
        BENCHMARK_REPS_PER_BATCH_27
        *
        1e6
    )


PLAIN_AGG_MEDIAN_US_27 = float(
    np.median(
        plain_times_27
    )
)

SECURE_AGG_MEDIAN_US_27 = float(
    np.median(
        secure_times_27
    )
)

PLAIN_AGG_IQR_US_27 = float(
    np.percentile(
        plain_times_27,
        75
    )
    -
    np.percentile(
        plain_times_27,
        25
    )
)

SECURE_AGG_IQR_US_27 = float(
    np.percentile(
        secure_times_27,
        75
    )
    -
    np.percentile(
        secure_times_27,
        25
    )
)

SECURE_ADDED_LATENCY_US_27 = (
    SECURE_AGG_MEDIAN_US_27
    -
    PLAIN_AGG_MEDIAN_US_27
)

SECURE_MULTIPLIER_27 = (
    SECURE_AGG_MEDIAN_US_27
    /
    PLAIN_AGG_MEDIAN_US_27
)

SECURE_RELATIVE_OVERHEAD_PERCENT_27 = (
    (
        SECURE_MULTIPLIER_27
        -
        1.0
    )
    *
    100.0
)

SECURITY_ADDED_MS_PER_TASK_27 = (
    SECURE_ADDED_LATENCY_US_27
    *
    FL_ROUNDS_27
    /
    1000.0
)

SECURITY_ADDED_MS_ALL_TASKS_27 = (
    SECURITY_ADDED_MS_PER_TASK_27
    *
    N_TASKS_27
)


# ==================================================================================================
# 16. NUMERICAL EQUIVALENCE
# ==================================================================================================

rng_eq27 = np.random.default_rng(
    12345
)

test_masks_27 = {
    client:
        np.zeros(
            MODEL_PARAMETERS_27,
            dtype=np.float64
        )

    for client in CLIENTS_27
}


for i in range(
    N_CLIENTS_27
):

    for j in range(
        i + 1,
        N_CLIENTS_27
    ):

        pair_mask = rng_eq27.normal(
            size=MODEL_PARAMETERS_27
        )

        test_masks_27[
            CLIENTS_27[i]
        ] += pair_mask

        test_masks_27[
            CLIENTS_27[j]
        ] -= pair_mask


plain_eq27 = np.zeros(
    MODEL_PARAMETERS_27,
    dtype=np.float64
)

secure_eq27 = np.zeros(
    MODEL_PARAMETERS_27,
    dtype=np.float64
)


for client in CLIENTS_27:

    weighted_update = (
        CLIENT_WEIGHTS_27[
            client
        ]
        *
        benchmark_updates_27[
            client
        ]
    )

    plain_eq27 += (
        weighted_update
    )

    secure_eq27 += (
        weighted_update
        +
        test_masks_27[
            client
        ]
    )


SECURE_EQUIVALENT_27 = bool(
    np.allclose(
        plain_eq27,
        secure_eq27,
        atol=1e-9,
        rtol=1e-9
    )
)

SECURE_MAX_ERROR_27 = float(
    np.max(
        np.abs(
            plain_eq27
            -
            secure_eq27
        )
    )
)


# ==================================================================================================
# 17. PER-STATION LOAD TABLE — UNIT-AWARE
# ==================================================================================================

CELL27_CLIENT_LOAD_DF = pd.DataFrame(
    [
        {
            "Station":
                client,

            "Aggregation Weight":
                f"{CLIENT_WEIGHTS_27[client] * 100:.3f} %",

            "Estimated Training Rows":
                f"{ESTIMATED_LOCAL_ROWS_27[client]:.1f} rows",

            "Computational Scale":
                (
                    f"{(
                        FL_ROUNDS_27
                        *
                        LOCAL_EPOCHS_27
                        *
                        ESTIMATED_LOCAL_ROWS_27[client]
                        *
                        MODEL_PARAMETERS_27
                    ) / 1e6:.3f} million parameter-sample ops"
                ),
        }

        for client in CLIENTS_27
    ]
)


# ==================================================================================================
# 18. PER-TASK COMMUNICATION TABLE — UNIT-AWARE
# ==================================================================================================

CELL27_COMMUNICATION_DF = pd.DataFrame(
    [
        {
            "Task":
                task,

            "FL Rounds":
                f"{FL_ROUNDS_27} rounds",

            "Clients":
                f"{ACTIVE_CLIENTS_PER_ROUND_27} stations",

            "Uplink":
                f"{UPLINK_MESSAGES_PER_TASK_27} messages",

            "Downlink":
                f"{DOWNLINK_MESSAGES_PER_TASK_27} messages",

            "Total Messages":
                f"{TOTAL_MESSAGES_PER_TASK_27} messages/task",

            "Model Parameters":
                f"{MODEL_PARAMETERS_27} parameters",

            "Model Size":
                f"{c27_kb(MODEL_SIZE_F64_BYTES_27):.3f} KB",

            "Communication":
                f"{c27_mb(TOTAL_BYTES_PER_TASK_27):.6f} MB/task",
        }

        for task in TASKS_27
    ]
)


# ==================================================================================================
# 19. STORAGE TABLE — UNIT-AWARE
# ==================================================================================================

CELL27_STORAGE_DF = pd.DataFrame(
    [
        {
            "Component":
                "Single float64 model",

            "Result":
                f"{c27_kb(SINGLE_MODEL_BYTES_27):.3f} KB",
        },

        {
            "Component":
                "Four client model copies",

            "Result":
                f"{c27_kb(CLIENT_MODELS_BYTES_27):.3f} KB",
        },

        {
            "Component":
                "FedAdam server state (model + m + v)",

            "Result":
                f"{c27_kb(FEDADAM_SERVER_STATE_BYTES_27):.3f} KB",
        },

        {
            "Component":
                "Approx. active FL parameter state",

            "Result":
                f"{c27_kb(ACTIVE_FL_PARAMETER_STATE_BYTES_27):.3f} KB",
        },
    ]
)


# ==================================================================================================
# 20. SECURITY TABLE — UNIT-AWARE
# ==================================================================================================

CELL27_SECURITY_DF = pd.DataFrame(
    [
        {
            "Metric":
                "Model parameters protected",

            "Result":
                f"{MODEL_PARAMETERS_27} parameters",
        },

        {
            "Metric":
                "Secure-Aggregation client pairs",

            "Result":
                f"{CLIENT_PAIRS_27} pairs",
        },

        {
            "Metric":
                "Pairwise mask values per round",

            "Result":
                f"{MASK_VALUES_PER_ROUND_27} values/round",
        },

        {
            "Metric":
                "Plain aggregation median latency",

            "Result":
                f"{PLAIN_AGG_MEDIAN_US_27:.3f} µs/aggregation",
        },

        {
            "Metric":
                "Plain aggregation IQR",

            "Result":
                f"{PLAIN_AGG_IQR_US_27:.3f} µs",
        },

        {
            "Metric":
                "Secure aggregation median latency",

            "Result":
                f"{SECURE_AGG_MEDIAN_US_27:.3f} µs/aggregation",
        },

        {
            "Metric":
                "Secure aggregation IQR",

            "Result":
                f"{SECURE_AGG_IQR_US_27:.3f} µs",
        },

        {
            "Metric":
                "Added secure-aggregation latency",

            "Result":
                f"{SECURE_ADDED_LATENCY_US_27:.3f} µs/round",
        },

        {
            "Metric":
                "Secure/plain latency multiplier",

            "Result":
                f"{SECURE_MULTIPLIER_27:.3f} ×",
        },

        {
            "Metric":
                "Relative secure-aggregation overhead",

            "Result":
                f"{SECURE_RELATIVE_OVERHEAD_PERCENT_27:.3f} %",
        },

        {
            "Metric":
                "Added security CPU time per task",

            "Result":
                f"{SECURITY_ADDED_MS_PER_TASK_27:.3f} ms/task",
        },

        {
            "Metric":
                "Added security CPU time — six tasks",

            "Result":
                f"{SECURITY_ADDED_MS_ALL_TASKS_27:.3f} ms",
        },

        {
            "Metric":
                "Secure aggregation numerical error",

            "Result":
                f"{SECURE_MAX_ERROR_27:.3e} absolute error",
        },
    ]
)


# ==================================================================================================
# 21. NETWORK TABLE — UNIT-AWARE
# ==================================================================================================

network_rows_27 = []


for bandwidth_mbps in [
    1,
    10,
    100,
]:

    bits_per_second = (
        bandwidth_mbps
        *
        1_000_000.0
    )

    per_task_seconds = (
        TOTAL_BYTES_PER_TASK_27
        *
        8.0
        /
        bits_per_second
    )

    six_tasks_seconds = (
        FINAL_SIX_TASK_TOTAL_BYTES_27
        *
        8.0
        /
        bits_per_second
    )

    if np.isfinite(
        CELL13_SEARCH_COMM_BYTES_27
    ):

        full_search_seconds = (
            CELL13_SEARCH_COMM_BYTES_27
            *
            8.0
            /
            bits_per_second
        )

        full_search_text = (
            f"{full_search_seconds:.6f} s"
        )

    else:

        full_search_text = "N/A"

    network_rows_27.append(
        {
            "Bandwidth":
                f"{bandwidth_mbps} Mbps",

            "One Task":
                f"{per_task_seconds:.6f} s",

            "Six Tasks":
                f"{six_tasks_seconds:.6f} s",

            "Full Cell-13 Search":
                full_search_text,
        }
    )


CELL27_NETWORK_DF = pd.DataFrame(
    network_rows_27
)


# ==================================================================================================
# 22. THEORETICAL COMPLEXITY TABLE
# ==================================================================================================

CELL27_COMPLEXITY_DF = pd.DataFrame(
    [
        {
            "Component":
                "Local training",

            "Complexity":
                "O(R × E × N_train × P)",

            "Numerical Scale":
                f"{LOCAL_TRAIN_OPS_PER_TASK_27 / 1e6:.3f} million parameter-sample ops/task",
        },

        {
            "Component":
                "Server aggregation",

            "Complexity":
                "O(R × K × P)",

            "Numerical Scale":
                f"{SERVER_AGG_OPS_PER_TASK_27:,} parameter operations/task",
        },

        {
            "Component":
                "Secure Aggregation",

            "Complexity":
                "O(R × K² × P)",

            "Numerical Scale":
                f"{MASK_VALUES_PER_TASK_27:,} mask values/task",
        },

        {
            "Component":
                "Inference",

            "Complexity":
                "O(P)",

            "Numerical Scale":
                f"{MODEL_PARAMETERS_27} parameter operations/prediction",
        },

        {
            "Component":
                "Model storage",

            "Complexity":
                "O(P)",

            "Numerical Scale":
                f"{c27_kb(SINGLE_MODEL_BYTES_27):.3f} KB/model",
        },
    ]
)


# ==================================================================================================
# 23. MAIN PUBLICATION-READY SUMMARY — RESULT + UNIT IN SAME COLUMN
# ==================================================================================================

summary_rows_27 = [
    {
        "Metric":
            "Active features",

        "Result":
            f"{ACTIVE_FEATURES_27} features",

        "Meaning":
            "Effective model inputs after feature filtering",
    },

    {
        "Metric":
            "Trainable parameters",

        "Result":
            f"{MODEL_PARAMETERS_27} parameters",

        "Meaning":
            "143 feature weights + 1 intercept",
    },

    {
        "Metric":
            "Float64 model size",

        "Result":
            f"{c27_kb(MODEL_SIZE_F64_BYTES_27):.3f} KB/model",

        "Meaning":
            "Memory required for one linear model",
    },

    {
        "Metric":
            "Float32 model size",

        "Result":
            f"{c27_kb(MODEL_SIZE_F32_BYTES_27):.3f} KB/model",

        "Meaning":
            "Reference size using 32-bit precision",
    },

    {
        "Metric":
            "Training data",

        "Result":
            f"{TRAIN_ROWS_TOTAL_PER_TASK_27:,} rows/task",

        "Meaning":
            f"Detected from {TRAIN_ROWS_SOURCE_27}",
    },

    {
        "Metric":
            "Federated clients",

        "Result":
            f"{N_CLIENTS_27} stations",

        "Meaning":
            "Four participating water-treatment stations",
    },

    {
        "Metric":
            "Federated rounds",

        "Result":
            f"{FL_ROUNDS_27} rounds/task",

        "Meaning":
            "Global communication rounds",
    },

    {
        "Metric":
            "Local epochs",

        "Result":
            f"{LOCAL_EPOCHS_27} epochs/round",

        "Meaning":
            "Local training passes before each aggregation",
    },

    {
        "Metric":
            "Client participation",

        "Result":
            f"{CLIENT_FRACTION_27 * 100:.1f} % clients/round",

        "Meaning":
            "All four stations participate in every round",
    },

    {
        "Metric":
            "Messages per task",

        "Result":
            f"{TOTAL_MESSAGES_PER_TASK_27} messages/task",

        "Meaning":
            f"{UPLINK_MESSAGES_PER_TASK_27} uplink + {DOWNLINK_MESSAGES_PER_TASK_27} downlink",
    },

    {
        "Metric":
            "Communication per task",

        "Result":
            f"{c27_mb(TOTAL_BYTES_PER_TASK_27):.6f} MB/task",

        "Meaning":
            "Total model/update payload exchange",
    },

    {
        "Metric":
            "Final six-task communication",

        "Result":
            f"{c27_mb(FINAL_SIX_TASK_TOTAL_BYTES_27):.6f} MB",

        "Meaning":
            "Total communication for the selected six-task system",
    },

    {
        "Metric":
            "Float32 six-task reference",

        "Result":
            f"{c27_mb(FINAL_SIX_TASK_TOTAL_F32_BYTES_27):.6f} MB",

        "Meaning":
            "Reference communication if 32-bit parameters are transmitted",
    },

    {
        "Metric":
            "Computation per selected task",

        "Result":
            f"{LOCAL_TRAIN_OPS_PER_TASK_27 / 1e6:.3f} million parameter-sample ops/task",

        "Meaning":
            "Approximate local-training computational scale",
    },

    {
        "Metric":
            "Computation for six selected tasks",

        "Result":
            f"{LOCAL_TRAIN_OPS_ALL_TASKS_27 / 1e6:.3f} million parameter-sample ops",

        "Meaning":
            "Approximate training scale across all six tasks",
    },

    {
        "Metric":
            "FedAdam server-state memory",

        "Result":
            f"{c27_kb(FEDADAM_SERVER_STATE_BYTES_27):.3f} KB/server",

        "Meaning":
            "Global model + first moment + second moment",
    },

    {
        "Metric":
            "Secure-Aggregation client pairs",

        "Result":
            f"{CLIENT_PAIRS_27} pairs",

        "Meaning":
            "Pairwise masking relationships among four stations",
    },

    {
        "Metric":
            "Plain aggregation latency",

        "Result":
            f"{PLAIN_AGG_MEDIAN_US_27:.3f} µs/aggregation",

        "Meaning":
            "Median aggregation time without secure masking",
    },

    {
        "Metric":
            "Secure aggregation latency",

        "Result":
            f"{SECURE_AGG_MEDIAN_US_27:.3f} µs/aggregation",

        "Meaning":
            "Median aggregation time with pairwise masking",
    },

    {
        "Metric":
            "Added security latency",

        "Result":
            f"{SECURE_ADDED_LATENCY_US_27:.3f} µs/round",

        "Meaning":
            "Absolute computational overhead introduced by Secure Aggregation",
    },

    {
        "Metric":
            "Secure/plain latency ratio",

        "Result":
            f"{SECURE_MULTIPLIER_27:.3f} ×",

        "Meaning":
            "Secure aggregation time relative to plain aggregation",
    },

    {
        "Metric":
            "Relative security overhead",

        "Result":
            f"{SECURE_RELATIVE_OVERHEAD_PERCENT_27:.3f} %",

        "Meaning":
            "Percentage increase relative to the very small plain aggregation time",
    },

    {
        "Metric":
            "Security CPU overhead per task",

        "Result":
            f"{SECURITY_ADDED_MS_PER_TASK_27:.3f} ms/task",

        "Meaning":
            "Added aggregation CPU time across 60 rounds",
    },

    {
        "Metric":
            "Security CPU overhead — six tasks",

        "Result":
            f"{SECURITY_ADDED_MS_ALL_TASKS_27:.3f} ms",

        "Meaning":
            "Added aggregation CPU time across the final six tasks",
    },

    {
        "Metric":
            "Secure-Aggregation numerical error",

        "Result":
            f"{SECURE_MAX_ERROR_27:.3e} absolute error",

        "Meaning":
            "Numerical difference between secure and ordinary aggregation",
    },
]


if np.isfinite(
    CELL13_EXPERIMENT_RUNS_27
):

    summary_rows_27.extend(
        [
            {
                "Metric":
                    "Cell-13 experimental configurations",

                "Result":
                    f"{int(CELL13_EXPERIMENT_RUNS_27)} runs",

                "Meaning":
                    "Full federated model-selection/search workload",
            },

            {
                "Metric":
                    "Full Cell-13 search communication",

                "Result":
                    f"{c27_mb(CELL13_SEARCH_COMM_BYTES_27):.3f} MB",

                "Meaning":
                    "Communication across all Cell-13 experimental runs",
            },

            {
                "Metric":
                    "Full Cell-13 search computation",

                "Result":
                    f"{CELL13_SEARCH_OPS_27 / 1e9:.3f} billion parameter-sample ops",

                "Meaning":
                    "Approximate computation across all Cell-13 experimental runs",
            },
        ]
    )


CELL27_SUMMARY_DF = pd.DataFrame(
    summary_rows_27
)


# ==================================================================================================
# 24. INTERNAL VALIDATION
# ==================================================================================================

CELL27_VALIDATION_DF = pd.DataFrame(
    [
        {
            "Check":
                "Active feature dimension",

            "Result":
                f"{ACTIVE_FEATURES_27} features",

            "Status":
                (
                    "PASS"
                    if ACTIVE_FEATURES_27 == 143
                    else "REVIEW"
                ),
        },

        {
            "Check":
                "Trainable model dimension",

            "Result":
                f"{MODEL_PARAMETERS_27} parameters",

            "Status":
                (
                    "PASS"
                    if MODEL_PARAMETERS_27 == 144
                    else "REVIEW"
                ),
        },

        {
            "Check":
                "7-D FEDERATED_RUNS record treated as model",

            "Result":
                str(
                    RECORD_DIMENSION_IS_MODEL_DIMENSION_27
                ),

            "Status":
                (
                    "PASS"
                    if not RECORD_DIMENSION_IS_MODEL_DIMENSION_27
                    else "REVIEW"
                ),
        },

        {
            "Check":
                "Generic n_train scalar used",

            "Result":
                "NO",

            "Status":
                "PASS",
        },

        {
            "Check":
                "Secure aggregation equivalence",

            "Result":
                (
                    f"{SECURE_MAX_ERROR_27:.3e} absolute error"
                ),

            "Status":
                (
                    "PASS"
                    if SECURE_EQUIVALENT_27
                    else "FAIL"
                ),
        },
    ]
)


# ==================================================================================================
# 25. DISPLAY
# ==================================================================================================

print("\n" + "=" * 180)
print("MODEL VERIFICATION")
print("=" * 180)

print(
    f"Feature source                     : {FEATURE_SOURCE_27}"
)

print(
    f"Active features                    : {ACTIVE_FEATURES_27} features"
)

print(
    f"Trainable parameters               : {MODEL_PARAMETERS_27} parameters"
)

print(
    f"FEDERATED_RUNS stored dimension    : {FEDERATED_RUN_RECORD_DIMENSION_27} record values"
)

print(
    f"Float64 model size                 : {c27_kb(MODEL_SIZE_F64_BYTES_27):.3f} KB/model"
)

print(
    f"Training rows                      : {TRAIN_ROWS_TOTAL_PER_TASK_27:,} rows/task"
)


print("\n" + "=" * 180)
print("PER-STATION TRAINING LOAD")
print("=" * 180)

display(
    CELL27_CLIENT_LOAD_DF
)


print("\n" + "=" * 180)
print("COMMUNICATION COST — PER TASK")
print("=" * 180)

display(
    CELL27_COMMUNICATION_DF
)


print("\n" + "=" * 180)
print("MODEL / OPTIMIZER STORAGE")
print("=" * 180)

display(
    CELL27_STORAGE_DF
)


print("\n" + "=" * 180)
print("SECURE AGGREGATION COST")
print("=" * 180)

display(
    CELL27_SECURITY_DF
)


print("\n" + "=" * 180)
print("THEORETICAL NETWORK TRANSFER TIME")
print("=" * 180)

display(
    CELL27_NETWORK_DF
)


print("\n" + "=" * 180)
print("THEORETICAL COMPLEXITY")
print("=" * 180)

display(
    CELL27_COMPLEXITY_DF
)


print("\n" + "=" * 180)
print("CELL 27 — PUBLICATION-READY UNIT-AWARE SUMMARY")
print("=" * 180)

display(
    CELL27_SUMMARY_DF
)


print("\n" + "=" * 180)
print("CELL 27 — INTERNAL VALIDATION")
print("=" * 180)

display(
    CELL27_VALIDATION_DF
)


# ==================================================================================================
# 26. FINAL STATUS
# ==================================================================================================

CELL27_PASS = bool(
    (
        CELL27_VALIDATION_DF[
            "Status"
        ]
        ==
        "PASS"
    ).all()
)


print("\n" + "=" * 180)

if CELL27_PASS:

    print(
        "CELL 27 COMPLETED SUCCESSFULLY — "
        "UNIT-AWARE EFFICIENCY AND COST ANALYSIS VALIDATED"
    )

else:

    print(
        "CELL 27 COMPLETED — REVIEW VALIDATION TABLE"
    )

print("=" * 180)

print(
    "All reported values include explicit units          : YES"
)

print(
    "7-D FEDERATED_RUNS records used as model size       : NO"
)

print(
    "Generic n_train scalar used                         : NO"
)

print(
    "Model retraining                                   : NO"
)

print(
    "Hyperparameter tuning                              : NO"
)

print(
    "Predictions changed                                : NO"
)

print(
    "Cell 13 modified                                   : NO"
)

print(
    "Cell 16 modified                                   : NO"
)

print(
    f"Execution time                                     : "
    f"{time.time() - CELL27_START:.2f} seconds"
)

In [ ]:
# ==================================================================================================
# CELL 28 — LOCAL-EPOCH SENSITIVITY ANALYSIS — FINAL VERIFIED
#
# VALIDATION ONLY
# Local Epochs tested: [1, 2, 5, 10]
#
# Uses:
#   - the final Validation-selected algorithm for each task from Cell 14
#   - the final selected federated method from Cell 14
#   - the exact 1-based task/algorithm/method positions created by Cell 13
#
# 6 tasks × 4 settings = 24 runs
#
# NO Test
# NO overwrite of FEDERATED_RUNS
# NO modification of Cells 14–27
# ==================================================================================================

import time
import random
import numpy as np
import pandas as pd

CELL28_START = time.time()

print("=" * 180)
print("CELL 28 — LOCAL-EPOCH SENSITIVITY ANALYSIS")
print("VALIDATION ONLY — LOCAL EPOCHS = [1, 2, 5, 10]")
print("=" * 180)

# ==================================================================================================
# 1. REQUIRED STATE
# ==================================================================================================

required_28 = [
    "train_federated_method",
    "task_keys",
    "algorithm_keys",
    "method_keys",
    "FEDERATED_SELECTED_ALGORITHM_DF",
    "SELECTED_FEDERATED_METHOD",
]

missing_28 = [
    name for name in required_28
    if name not in globals()
]

if missing_28:
    raise RuntimeError(
        "Cell 28 requires: "
        + ", ".join(missing_28)
        + ". Run Cells 13 and 14 first."
    )

# ==================================================================================================
# 2. SETTINGS
# ==================================================================================================

LOCAL_EPOCH_CANDIDATES_28 = [1, 2, 5, 10]
ROUNDS_28 = int(globals().get("FEDERATED_ROUNDS", 60))

# ==================================================================================================
# 3. BUILD EXACT FINAL CONFIGURATIONS FROM CELLS 13–14
# ==================================================================================================

selected_28 = (
    FEDERATED_SELECTED_ALGORITHM_DF[
        [
            "Task",
            "Algorithm_Key",
            "Federated_Method",
        ]
    ]
    .copy()
)

if len(selected_28) != len(task_keys):
    raise RuntimeError(
        "Expected exactly one final selected algorithm per task."
    )

CELL28_CONFIGS = []

for _, row in selected_28.iterrows():

    task_key_28 = str(row["Task"])
    algorithm_key_28 = str(row["Algorithm_Key"])
    method_key_28 = str(row["Federated_Method"])

    if method_key_28 != str(SELECTED_FEDERATED_METHOD):
        raise RuntimeError(
            f"Unexpected selected method for {task_key_28}: "
            f"{method_key_28} != {SELECTED_FEDERATED_METHOD}"
        )

    if task_key_28 not in task_keys:
        raise RuntimeError(
            f"Task not found in Cell-13 task order: {task_key_28}"
        )

    if algorithm_key_28 not in algorithm_keys:
        raise RuntimeError(
            f"Algorithm not found in Cell-13 algorithm order: "
            f"{algorithm_key_28}"
        )

    if method_key_28 not in method_keys:
        raise RuntimeError(
            f"Method not found in Cell-13 method order: {method_key_28}"
        )

    # Cell 13 enumerates all three position indices with start=1.
    CELL28_CONFIGS.append(
        {
            "Task": task_key_28,
            "Task_Position": task_keys.index(task_key_28) + 1,
            "Algorithm": algorithm_key_28,
            "Algorithm_Position": algorithm_keys.index(algorithm_key_28) + 1,
            "Method": method_key_28,
            "Method_Position": method_keys.index(method_key_28) + 1,
        }
    )

CELL28_CONFIG_DF = pd.DataFrame(CELL28_CONFIGS)

print("\nConfigurations used:")
display(CELL28_CONFIG_DF)

# ==================================================================================================
# 4. PRESERVE RANDOM STATES
# ==================================================================================================

numpy_state_28 = np.random.get_state()
python_state_28 = random.getstate()

# ==================================================================================================
# 5. RUN ONE EXPERIMENT
# ==================================================================================================

def c28_get_metric(validation_dict, names):

    for name in names:
        if name in validation_dict:
            return float(validation_dict[name])

    raise KeyError(
        f"Could not find any of {names} "
        f"in Validation keys: {list(validation_dict.keys())}"
    )


def c28_run_one(config, local_epochs):

    np.random.seed(42)
    random.seed(42)

    output = train_federated_method(
        config["Task"],
        config["Algorithm"],
        config["Method"],
        config["Task_Position"],
        config["Algorithm_Position"],
        config["Method_Position"],
        rounds=ROUNDS_28,
        local_epochs=int(local_epochs),
    )

    if "Validation" not in output:
        raise RuntimeError(
            f"No Validation result returned for "
            f"{config['Task']} / E={local_epochs}"
        )

    validation = output["Validation"]

    return {
        "Validation_MAE": c28_get_metric(
            validation,
            ["MAE", "mae"],
        ),
        "Validation_RMSE": c28_get_metric(
            validation,
            ["RMSE", "rmse"],
        ),
        "Validation_R2": c28_get_metric(
            validation,
            ["R2", "R²", "r2"],
        ),
        "Training_Time_Seconds": float(
            output.get("Training_Time_Seconds", np.nan)
        ),
        "Transmitted_Parameters": int(
            output.get("Transmitted_Parameters", 0)
        ),
        "Test_Used": bool(
            output.get("Test_Used", False)
        ),
    }

# ==================================================================================================
# 6. RUN 24 VALIDATION EXPERIMENTS
# ==================================================================================================

results_28 = []
total_runs_28 = len(CELL28_CONFIGS) * len(LOCAL_EPOCH_CANDIDATES_28)
run_counter_28 = 0

try:

    for config in CELL28_CONFIGS:

        print("\n" + "-" * 180)
        print(
            f"{config['Task']} | "
            f"{config['Algorithm']} | "
            f"{config['Method']}"
        )
        print("-" * 180)

        for local_epochs in LOCAL_EPOCH_CANDIDATES_28:

            run_counter_28 += 1

            print(
                f"[{run_counter_28:02d}/{total_runs_28}] "
                f"{config['Task']} | Local Epochs = {local_epochs}"
            )

            result = c28_run_one(
                config=config,
                local_epochs=local_epochs,
            )

            results_28.append(
                {
                    "Task": config["Task"],
                    "Algorithm": config["Algorithm"],
                    "Method": config["Method"],
                    "Local_Epochs": int(local_epochs),
                    "Validation_MAE": result["Validation_MAE"],
                    "Validation_RMSE": result["Validation_RMSE"],
                    "Validation_R2": result["Validation_R2"],
                    "Training_Time_Seconds": result["Training_Time_Seconds"],
                    "Transmitted_Parameters": result["Transmitted_Parameters"],
                    "Test_Used": result["Test_Used"],
                }
            )

finally:

    np.random.set_state(numpy_state_28)
    random.setstate(python_state_28)

CELL28_RESULTS_DF = pd.DataFrame(results_28)

if len(CELL28_RESULTS_DF) != 24:
    raise RuntimeError(
        f"Expected 24 runs, obtained {len(CELL28_RESULTS_DF)}."
    )

if CELL28_RESULTS_DF["Test_Used"].any():
    raise RuntimeError(
        "Unexpected Test usage detected in Cell 28."
    )

# ==================================================================================================
# 7. TASK-WISE RANKING
# ==================================================================================================

ranked_groups_28 = []

for task, group in CELL28_RESULTS_DF.groupby(
    "Task",
    sort=False,
):

    group = (
        group
        .copy()
        .sort_values(
            [
                "Validation_MAE",
                "Validation_RMSE",
                "Validation_R2",
                "Local_Epochs",
            ],
            ascending=[
                True,
                True,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    group["Task_Rank"] = np.arange(
        1,
        len(group) + 1,
    )

    best_mae_28 = float(
        group.iloc[0]["Validation_MAE"]
    )

    group["MAE_Gap_From_Best_Percent"] = (
        (
            group["Validation_MAE"]
            - best_mae_28
        )
        / max(abs(best_mae_28), 1e-12)
        * 100.0
    )

    e2_mae_28 = float(
        group.loc[
            group["Local_Epochs"].eq(2),
            "Validation_MAE",
        ].iloc[0]
    )

    group["MAE_Change_vs_E2_Percent"] = (
        (
            group["Validation_MAE"]
            - e2_mae_28
        )
        / max(abs(e2_mae_28), 1e-12)
        * 100.0
    )

    ranked_groups_28.append(group)

CELL28_RESULTS_DF = pd.concat(
    ranked_groups_28,
    ignore_index=True,
)

# ==================================================================================================
# 8. BEST EPOCH PER TASK
# ==================================================================================================

CELL28_TASK_BEST_DF = (
    CELL28_RESULTS_DF[
        CELL28_RESULTS_DF["Task_Rank"].eq(1)
    ]
    .sort_values("Task")
    .reset_index(drop=True)
)

# ==================================================================================================
# 9. OVERALL SUMMARY
#
# Raw Alum and Chlorine MAEs are NOT averaged together.
# Cross-task comparison uses rank and relative MAE gap.
# ==================================================================================================

summary_rows_28 = []

for local_epochs in LOCAL_EPOCH_CANDIDATES_28:

    group = CELL28_RESULTS_DF[
        CELL28_RESULTS_DF["Local_Epochs"].eq(local_epochs)
    ]

    summary_rows_28.append(
        {
            "Local_Epochs": int(local_epochs),
            "Task_Wins": int(
                group["Task_Rank"].eq(1).sum()
            ),
            "Mean_Task_Rank": float(
                group["Task_Rank"].mean()
            ),
            "Median_Task_Rank": float(
                group["Task_Rank"].median()
            ),
            "Mean_MAE_Gap_From_Best_Percent": float(
                group[
                    "MAE_Gap_From_Best_Percent"
                ].mean()
            ),
            "Mean_MAE_Change_vs_E2_Percent": float(
                group[
                    "MAE_Change_vs_E2_Percent"
                ].mean()
            ),
            "Mean_Validation_R2": float(
                group["Validation_R2"].mean()
            ),
            "Total_Training_Time_Seconds": float(
                group["Training_Time_Seconds"].sum()
            ),
        }
    )

CELL28_SUMMARY_DF = pd.DataFrame(summary_rows_28)

CELL28_SELECTION_DF = (
    CELL28_SUMMARY_DF
    .sort_values(
        [
            "Mean_Task_Rank",
            "Task_Wins",
            "Mean_MAE_Gap_From_Best_Percent",
            "Mean_Validation_R2",
            "Local_Epochs",
        ],
        ascending=[
            True,
            False,
            True,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

CELL28_SELECTION_DF["Overall_Rank"] = np.arange(
    1,
    len(CELL28_SELECTION_DF) + 1,
)

CELL28_RECOMMENDED_LOCAL_EPOCHS = int(
    CELL28_SELECTION_DF.iloc[0]["Local_Epochs"]
)

# Compatibility alias used by earlier notes/export cells.
CELL13C_RECOMMENDED_LOCAL_EPOCHS = (
    CELL28_RECOMMENDED_LOCAL_EPOCHS
)

# ==================================================================================================
# 10. DISPLAY
# ==================================================================================================

print("\n" + "=" * 180)
print("ALL 24 VALIDATION RUNS")
print("=" * 180)

display(
    CELL28_RESULTS_DF[
        [
            "Task",
            "Algorithm",
            "Method",
            "Local_Epochs",
            "Validation_MAE",
            "Validation_RMSE",
            "Validation_R2",
            "Task_Rank",
            "MAE_Gap_From_Best_Percent",
            "MAE_Change_vs_E2_Percent",
            "Training_Time_Seconds",
        ]
    ]
    .sort_values(
        ["Task", "Local_Epochs"]
    )
    .reset_index(drop=True)
    .round(8)
)

print("\n" + "=" * 180)
print("BEST LOCAL EPOCH — EACH TASK")
print("=" * 180)

display(
    CELL28_TASK_BEST_DF[
        [
            "Task",
            "Algorithm",
            "Method",
            "Local_Epochs",
            "Validation_MAE",
            "Validation_RMSE",
            "Validation_R2",
        ]
    ].round(8)
)

print("\n" + "=" * 180)
print("FINAL VALIDATION-BASED RANKING")
print("=" * 180)

display(
    CELL28_SELECTION_DF[
        [
            "Overall_Rank",
            "Local_Epochs",
            "Task_Wins",
            "Mean_Task_Rank",
            "Median_Task_Rank",
            "Mean_MAE_Gap_From_Best_Percent",
            "Mean_MAE_Change_vs_E2_Percent",
            "Mean_Validation_R2",
            "Total_Training_Time_Seconds",
        ]
    ].round(8)
)

epoch2_rank_28 = int(
    CELL28_SELECTION_DF.loc[
        CELL28_SELECTION_DF["Local_Epochs"].eq(2),
        "Overall_Rank",
    ].iloc[0]
)

print("\n" + "=" * 180)
print("COPY / PASTE — LOCAL-EPOCH SENSITIVITY RESULT")
print("=" * 180)

print("Epochs tested                  : 1, 2, 5, 10 epochs/round")
print(f"Federated rounds               : {ROUNDS_28} rounds/run")
print("Tasks                          : 6 tasks")
print("Total experiments              : 24 validation runs")
print("Test data used                 : NO")
print(
    f"Recommended Local Epochs        : "
    f"{CELL28_RECOMMENDED_LOCAL_EPOCHS} epochs/round"
)
print(
    f"Original E=2 overall rank       : "
    f"{epoch2_rank_28} / 4"
)

print("\nBest epoch by task:")

for row in CELL28_TASK_BEST_DF.itertuples():
    print(
        f"  {row.Task:<15} "
        f"E={int(row.Local_Epochs):2d} | "
        f"MAE={row.Validation_MAE:.8f} | "
        f"RMSE={row.Validation_RMSE:.8f} | "
        f"R²={row.Validation_R2:.8f}"
    )

print("\n" + "=" * 180)
print("CELL 28 COMPLETED SUCCESSFULLY")
print("=" * 180)
print("Cell-13 exact 1-based positions used : YES")
print("Final Cell-14 selections used        : YES")
print("FEDERATED_RUNS overwritten           : NO")
print("Test data used                       : NO")
print("Predictions changed                  : NO")
print(
    f"Execution time                       : "
    f"{time.time() - CELL28_START:.2f} seconds"
)


In [ ]:
# ==================================================================================================
# CELL 29 — FINAL RESULTS EXPORT — ONE EXCEL FILE
#
# FINAL NOTEBOOK CELL
#
# Exports:
#   - all result / summary / validation / test / audit DataFrames currently in memory
#   - one FINAL_SUMMARY sheet
#   - total wall-clock time from Cell 1 start to Cell 29
#
# Output:
#   PAPER3_UTW_RG_AdaFedResidual_ALL_RESULTS.xlsx
#
# No retraining.
# No predictions changed.
# ==================================================================================================

import time
import datetime
import numpy as np
import pandas as pd

CELL29_START = time.time()

print("=" * 160)
print("CELL 29 — FINAL RESULTS EXPORT")
print("=" * 160)

OUTPUT_XLSX_29 = (
    "PAPER3_UTW_RG_AdaFedResidual_ALL_RESULTS.xlsx"
)

# ==================================================================================================
# 1. TOTAL NOTEBOOK EXECUTION TIME
# ==================================================================================================

if "NOTEBOOK_RUN_START_TIME" in globals():

    TOTAL_EXECUTION_SECONDS_29 = (
        time.time()
        - float(NOTEBOOK_RUN_START_TIME)
    )

    TOTAL_TIME_SOURCE_29 = (
        "NOTEBOOK_RUN_START_TIME from Cell 1"
    )

else:

    TOTAL_EXECUTION_SECONDS_29 = np.nan
    TOTAL_TIME_SOURCE_29 = (
        "Unavailable — run Cell 1 before Cell 29"
    )


def format_time_29(seconds):

    if not np.isfinite(seconds):
        return "Unavailable"

    seconds = float(seconds)

    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60

    return (
        f"{hours:02d}:"
        f"{minutes:02d}:"
        f"{secs:06.3f}"
    )


TOTAL_EXECUTION_TIME_29 = format_time_29(
    TOTAL_EXECUTION_SECONDS_29
)

# ==================================================================================================
# 2. COLLECT RESULT TABLES ONLY
# ==================================================================================================

RESULT_KEYWORDS_29 = [
    "RESULT",
    "SUMMARY",
    "VALIDATION",
    "TEST",
    "BEST",
    "SELECT",
    "RANK",
    "PROTOCOL",
    "METRIC",
    "COMPARISON",
    "ABLATION",
    "ROBUST",
    "SIGNIFIC",
    "STABILITY",
    "LOSO",
    "FEW",
    "ZERO",
    "GENERALIZATION",
    "AUDIT",
    "BENCHMARK",
    "CHARACTERIZATION",
    "SECURITY",
    "EFFICIENCY",
    "COST",
    "WEIGHT",
    "CHECK",
    "SENSITIVITY",
    "EXTERNAL",
    "PERFORMANCE",
    "TABLE",
    "B16_",
    "S30_",
]

RESULT_DATAFRAMES_29 = {}

for name, value in list(globals().items()):

    if not isinstance(value, pd.DataFrame):
        continue

    upper_name = str(name).upper()

    if str(name).startswith("_"):
        continue

    if any(
        keyword in upper_name
        for keyword in RESULT_KEYWORDS_29
    ):
        RESULT_DATAFRAMES_29[name] = value.copy()


print(
    f"Result DataFrames found : "
    f"{len(RESULT_DATAFRAMES_29)}"
)

# ==================================================================================================
# 3. SAFE UNIQUE EXCEL SHEET NAMES
# ==================================================================================================

used_sheet_names_29 = set()


def safe_sheet_name_29(name):

    name = str(name)

    for char in [
        "[", "]", ":", "*", "?", "/", "\\"
    ]:
        name = name.replace(char, "_")

    base = name[:31]
    candidate = base
    counter = 1

    while candidate in used_sheet_names_29:

        suffix = f"_{counter}"

        candidate = (
            base[
                :
                31 - len(suffix)
            ]
            + suffix
        )

        counter += 1

    used_sheet_names_29.add(candidate)

    return candidate

# ==================================================================================================
# 4. FINAL SUMMARY
# ==================================================================================================

summary_rows_29 = [
    {
        "Item": "Project",
        "Result": "Paper 3 — UTW-RG-AdaFedResidual",
    },
    {
        "Item": "Export Time",
        "Result": datetime.datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    },
    {
        "Item": "Total Execution Time",
        "Result": TOTAL_EXECUTION_TIME_29,
    },
    {
        "Item": "Total Execution Seconds",
        "Result": (
            round(TOTAL_EXECUTION_SECONDS_29, 3)
            if np.isfinite(TOTAL_EXECUTION_SECONDS_29)
            else "Unavailable"
        ),
    },
    {
        "Item": "Timing Source",
        "Result": TOTAL_TIME_SOURCE_29,
    },
    {
        "Item": "Result Tables Exported",
        "Result": len(RESULT_DATAFRAMES_29),
    },
]

important_values_29 = [
    (
        "Federated Rounds",
        "FEDERATED_ROUNDS",
        "rounds",
    ),
    (
        "Primary Local Epochs",
        "FEDERATED_LOCAL_EPOCHS",
        "epochs/round",
    ),
    (
        "Random Seed",
        "GLOBAL_RANDOM_SEED",
        "",
    ),
    (
        "Fixed Alpha",
        "FIXED_ALPHA",
        "",
    ),
    (
        "Cell 28 Recommended Local Epochs",
        "CELL28_RECOMMENDED_LOCAL_EPOCHS",
        "epochs/round",
    ),
]

for label, variable_name, unit in important_values_29:

    if variable_name not in globals():
        continue

    value = globals()[variable_name]

    if unit:
        value = f"{value} {unit}"

    summary_rows_29.append(
        {
            "Item": label,
            "Result": value,
        }
    )

FINAL_SUMMARY_DF_29 = pd.DataFrame(
    summary_rows_29
)

RESULT_TABLE_MANIFEST_DF_29 = pd.DataFrame(
    [
        {
            "Variable": name,
            "Rows": int(df.shape[0]),
            "Columns": int(df.shape[1]),
        }
        for name, df
        in sorted(
            RESULT_DATAFRAMES_29.items()
        )
    ]
)

# ==================================================================================================
# 5. WRITE ONE EXCEL FILE
# ==================================================================================================

with pd.ExcelWriter(
    OUTPUT_XLSX_29,
    engine="openpyxl",
) as writer:

    FINAL_SUMMARY_DF_29.to_excel(
        writer,
        sheet_name="FINAL_SUMMARY",
        index=False,
    )

    RESULT_TABLE_MANIFEST_DF_29.to_excel(
        writer,
        sheet_name="RESULT_TABLES",
        index=False,
    )

    for variable_name, dataframe in sorted(
        RESULT_DATAFRAMES_29.items()
    ):

        sheet_name = safe_sheet_name_29(
            variable_name
        )

        export_df = dataframe.copy()

        if isinstance(export_df.columns, pd.MultiIndex):
            export_df.columns = [
                " | ".join(map(str, col))
                for col in export_df.columns.to_flat_index()
            ]

        export_df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False,
        )
# ==================================================================================================
# 6. FINAL STATUS
# ==================================================================================================

CELL29_EXECUTION_SECONDS = (
    time.time()
    - CELL29_START
)

print("\n" + "=" * 160)
print("FINAL EXPORT SUMMARY")
print("=" * 160)

display(
    pd.DataFrame(
        [
            {
                "Item": "Excel file",
                "Result": OUTPUT_XLSX_29,
            },
            {
                "Item": "Result tables exported",
                "Result": len(
                    RESULT_DATAFRAMES_29
                ),
            },
            {
                "Item": "Total notebook execution time",
                "Result": TOTAL_EXECUTION_TIME_29,
            },
            {
                "Item": "Cell 29 execution time",
                "Result": (
                    f"{CELL29_EXECUTION_SECONDS:.3f} seconds"
                ),
            },
        ]
    )
)

print("\n" + "=" * 160)
print("CELL 29 COMPLETED SUCCESSFULLY")
print("=" * 160)

# ==================================================================================================
# 7. DOWNLOAD
# ==================================================================================================

try:
    from google.colab import files

    files.download(
        OUTPUT_XLSX_29
    )

except Exception as exc:

    print(
        f"\nExcel file saved as: "
        f"{OUTPUT_XLSX_29}"
    )

    print(
        f"Automatic download unavailable: "
        f"{exc}"
    )


In [ ]:
# ==================================================================================================
# 30 FINAL COMPLETION ALERT — BEEP + DESKTOP NOTIFICATION
# ==================================================================================================

from IPython.display import display, Javascript
import time

display(
    Javascript(
        """
        (async () => {

            // ----------------------------------------------------------
            // 1. Ask permission for desktop notifications
            // ----------------------------------------------------------
            if ("Notification" in window) {

                if (Notification.permission === "default") {
                    await Notification.requestPermission();
                }

                if (Notification.permission === "granted") {

                    new Notification(
                        "Paper 3 — Execution Completed",
                        {
                            body:
                                "All notebook cells finished successfully.",
                            requireInteraction: true
                        }
                    );
                }
            }


            // ----------------------------------------------------------
            // 2. Audible alert
            // ----------------------------------------------------------
            try {

                const ctx =
                    new (window.AudioContext ||
                         window.webkitAudioContext)();

                const beep = (frequency, start, duration) => {

                    const osc = ctx.createOscillator();
                    const gain = ctx.createGain();

                    osc.connect(gain);
                    gain.connect(ctx.destination);

                    osc.frequency.value = frequency;
                    osc.type = "sine";

                    gain.gain.setValueAtTime(
                        0.8,
                        ctx.currentTime + start
                    );

                    gain.gain.exponentialRampToValueAtTime(
                        0.01,
                        ctx.currentTime + start + duration
                    );

                    osc.start(
                        ctx.currentTime + start
                    );

                    osc.stop(
                        ctx.currentTime + start + duration
                    );
                };


                // Strong completion pattern
                beep(880, 0.0, 0.45);
                beep(1100, 0.55, 0.45);
                beep(1320, 1.10, 0.70);

            }

            catch (e) {

                console.log(
                    "Audio alert unavailable:",
                    e
                );
            }

        })();
        """
    )
)

print("=" * 100)
print("🔔 NOTEBOOK EXECUTION COMPLETED")
print("=" * 100)
print(
    "Desktop notification + audible completion alert requested."
)

In [ ]:
import re
import time

source = next(
    (s for s in reversed(In[:-1])
     if re.search(r"(?m)^def u1619_run_online\(", s)),
    None
)
if source is None:
    raise RuntimeError("شغّل الخلية 16 أولًا في هذه الجلسة.")

start = source.index(
    "prediction_rows = []",
    source.index("FINAL CONTINUOUS VALIDATION")
)
end = source.index("# 12. FINAL NOVEL TEST RESULTS", start)

t0 = time.perf_counter()
exec(source[start:end], globals())
elapsed = time.perf_counter() - t0

print(f"Novel forecasting time: {elapsed:.3f} seconds")
print(f"Test forecasts: {len(U1619_TEST_PREDICTIONS_DF)}")

In [ ]:

# 31 FINAL COMPUTATIONAL TIME

import time
elapsed = time.time() - NOTEBOOK_RUN_START_TIME
print(f"Total execution time: {elapsed/60:.2f} minutes")